In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from pathlib import Path
import sys
import random

import numpy as np
import pandas as pd


# --------------------------------------------------
# Project directories
# --------------------------------------------------

BASE_DIR = Path(
    PROJECT_ROOT
)

PYTHON_DIR = (
    BASE_DIR
    / "03_Python_Optimization"
)

RESULTS_DIR = (
    BASE_DIR
    / "05_Results"
)

FIGURES_DIR = (
    BASE_DIR
    / "06_Figures"
)

TABLES_DIR = (
    BASE_DIR
    / "07_Tables"
)


# --------------------------------------------------
# Make local MST module importable
# --------------------------------------------------

python_path = str(PYTHON_DIR)

if python_path not in sys.path:
    sys.path.insert(
        0,
        python_path,
    )

import mst_core


# --------------------------------------------------
# Reproducibility
# --------------------------------------------------

GLOBAL_SEED = 2026

random.seed(
    GLOBAL_SEED
)

np.random.seed(
    GLOBAL_SEED
)


print("MST experimental environment initialized.")
print("Random seed:", GLOBAL_SEED)
print("MST module:", mst_core.__file__)
print("Results folder:", RESULTS_DIR)
print("Figures folder:", FIGURES_DIR)
print("Tables folder:", TABLES_DIR)

In [ ]:
# --------------------------------------------------
# Computational experiment design
# --------------------------------------------------

# Candidate-set sizes where Exact-MST remains practical.
EXACT_BENCHMARK_SIZES = [
    8,
    10,
    12,
    14,
    16,
]

# Larger candidate sets used later for scalability analysis.
SCALABILITY_SIZES = [
    16,
    32,
    64,
    128,
    256,
]

# Sufficiency requirements spanning moderate to stringent conditions.
TAU_LEVELS = [
    0.70,
    0.80,
    0.90,
    0.95,
]

# Objective-weight configurations.
#
# Each configuration sums to 1 so that the relative influence
# of privacy, interpretation cost, and parsimony is transparent.
WEIGHT_PROFILES = {
    "balanced": {
        "lambda_privacy": 1 / 3,
        "lambda_cognitive": 1 / 3,
        "lambda_cardinality": 1 / 3,
    },

    "privacy_priority": {
        "lambda_privacy": 0.60,
        "lambda_cognitive": 0.20,
        "lambda_cardinality": 0.20,
    },

    "interpretation_priority": {
        "lambda_privacy": 0.20,
        "lambda_cognitive": 0.60,
        "lambda_cardinality": 0.20,
    },

    "parsimony_priority": {
        "lambda_privacy": 0.20,
        "lambda_cognitive": 0.20,
        "lambda_cardinality": 0.60,
    },
}

# Structural regimes controlling the relationship between
# information utility and disclosure burden.
INSTANCE_REGIMES = [
    "independent",
    "utility_burden_conflict",
    "utility_burden_alignment",
]

print("Exact benchmark sizes:", EXACT_BENCHMARK_SIZES)
print("Scalability sizes:", SCALABILITY_SIZES)
print("Tau levels:", TAU_LEVELS)
print("Weight profiles:", list(WEIGHT_PROFILES))
print("Instance regimes:", INSTANCE_REGIMES)

In [ ]:
# --------------------------------------------------
# Synthetic MST benchmark-instance generator
# --------------------------------------------------

REGIME_STRENGTH = 0.75


def generate_mst_instance(
    n_items: int,
    regime: str,
    seed: int,
    regime_strength: float = REGIME_STRENGTH,
) -> list[mst_core.InformationItem]:
    """
    Generate one reproducible synthetic MST benchmark instance.

    Parameters
    ----------
    n_items
        Number of privacy-admissible candidate information items.

    regime
        Structural relationship between modeled utility and burden:

        "independent"
            Utility, privacy exposure, and interpretation cost
            are generated independently.

        "utility_burden_conflict"
            Higher-utility items tend to have higher privacy
            exposure and interpretation cost.

        "utility_burden_alignment"
            Higher-utility items tend to have lower privacy
            exposure and interpretation cost.

    seed
        Random seed for reproducibility.

    regime_strength
        Strength of the structural relationship in [0, 1].
        A value of 0 gives independence, while larger values
        produce stronger alignment or conflict.

    Returns
    -------
    list[mst_core.InformationItem]
        Generated candidate information items.
    """

    if n_items <= 0:
        raise ValueError(
            "n_items must be greater than zero."
        )

    if regime not in INSTANCE_REGIMES:
        raise ValueError(
            f"Unknown regime: {regime}"
        )

    if not 0.0 <= regime_strength <= 1.0:
        raise ValueError(
            "regime_strength must be in [0, 1]."
        )

    rng = np.random.default_rng(seed)

    # --------------------------------------------------
    # Latent modeled utility
    # --------------------------------------------------
    #
    # Beta(2, 2) provides heterogeneous but non-extreme
    # candidate values.
    #
    # The transformation restricts individual utility
    # contributions to [0.08, 0.60], preventing most
    # single items from trivially satisfying stringent
    # thresholds while retaining feasible combinations.
    # --------------------------------------------------

    utility_latent = rng.beta(
        a=2.0,
        b=2.0,
        size=n_items,
    )

    utility_values = (
        0.08
        + 0.52 * utility_latent
    )


    # --------------------------------------------------
    # Independent latent burden components
    # --------------------------------------------------

    privacy_noise = rng.beta(
        a=2.0,
        b=2.0,
        size=n_items,
    )

    cognitive_noise = rng.beta(
        a=2.0,
        b=2.0,
        size=n_items,
    )


    # --------------------------------------------------
    # Structural regime
    # --------------------------------------------------

    if regime == "independent":

        privacy_latent = privacy_noise

        cognitive_latent = cognitive_noise


    elif regime == "utility_burden_conflict":

        # Higher modeled utility tends to imply
        # greater disclosure burden.

        privacy_latent = (
            regime_strength
            * utility_latent
            +
            (1.0 - regime_strength)
            * privacy_noise
        )

        cognitive_latent = (
            regime_strength
            * utility_latent
            +
            (1.0 - regime_strength)
            * cognitive_noise
        )


    elif regime == "utility_burden_alignment":

        # Higher modeled utility tends to imply
        # lower disclosure burden.

        inverse_utility = (
            1.0 - utility_latent
        )

        privacy_latent = (
            regime_strength
            * inverse_utility
            +
            (1.0 - regime_strength)
            * privacy_noise
        )

        cognitive_latent = (
            regime_strength
            * inverse_utility
            +
            (1.0 - regime_strength)
            * cognitive_noise
        )


    # --------------------------------------------------
    # Convert latent burdens to computational scales
    # --------------------------------------------------
    #
    # Per-item privacy values are intentionally bounded
    # below 0.25 to reduce premature saturation of the
    # cumulative bounded privacy function.
    #
    # Cognitive costs are positive raw quantities that
    # are subsequently normalized by MST relative to
    # the complete admissible candidate set.
    # --------------------------------------------------

    privacy_values = (
        0.01
        + 0.24 * privacy_latent
    )

    cognitive_values = (
        0.10
        + 0.90 * cognitive_latent
    )


    # --------------------------------------------------
    # Construct MST information items
    # --------------------------------------------------

    items = []

    for index in range(n_items):

        items.append(
            mst_core.InformationItem(
                item_id=f"z{index + 1:03d}",
                name=(
                    f"Synthetic information item "
                    f"{index + 1}"
                ),
                utility=float(
                    utility_values[index]
                ),
                privacy=float(
                    privacy_values[index]
                ),
                cognitive_cost=float(
                    cognitive_values[index]
                ),
            )
        )

    return items

In [ ]:
test_synthetic_instance = generate_mst_instance(
    n_items=8,
    regime="independent",
    seed=GLOBAL_SEED,
)

for item in test_synthetic_instance:
    print(
        item.item_id,
        f"U={item.utility:.4f}",
        f"P={item.privacy:.4f}",
        f"C={item.cognitive_cost:.4f}",
    )

print(
    "\nFull-set utility:",
    f"{mst_core.mst_utility(test_synthetic_instance):.4f}",
)

In [ ]:
# --------------------------------------------------
# Validate synthetic benchmark regimes
# --------------------------------------------------

N_VALIDATION_INSTANCES = 200
VALIDATION_INSTANCE_SIZE = 32


def rank_correlation(x, y):
    """
    Compute Spearman-style rank correlation without
    requiring an additional statistics package.
    """
    x_rank = pd.Series(x).rank()
    y_rank = pd.Series(y).rank()

    return x_rank.corr(y_rank)


regime_validation_rows = []

for regime_index, regime in enumerate(INSTANCE_REGIMES):

    pooled_utility = []
    pooled_privacy = []
    pooled_cognitive = []

    for replicate in range(N_VALIDATION_INSTANCES):

        instance_seed = (
            GLOBAL_SEED
            + regime_index * 100_000
            + replicate
        )

        generated_items = generate_mst_instance(
            n_items=VALIDATION_INSTANCE_SIZE,
            regime=regime,
            seed=instance_seed,
        )

        pooled_utility.extend(
            item.utility
            for item in generated_items
        )

        pooled_privacy.extend(
            item.privacy
            for item in generated_items
        )

        pooled_cognitive.extend(
            item.cognitive_cost
            for item in generated_items
        )

    regime_validation_rows.append(
        {
            "regime": regime,

            "utility_privacy_pearson":
                pd.Series(pooled_utility).corr(
                    pd.Series(pooled_privacy)
                ),

            "utility_cognitive_pearson":
                pd.Series(pooled_utility).corr(
                    pd.Series(pooled_cognitive)
                ),

            "utility_privacy_rank":
                rank_correlation(
                    pooled_utility,
                    pooled_privacy,
                ),

            "utility_cognitive_rank":
                rank_correlation(
                    pooled_utility,
                    pooled_cognitive,
                ),
        }
    )


regime_validation_df = pd.DataFrame(
    regime_validation_rows
)

print(
    regime_validation_df.to_string(
        index=False
    )
)

In [ ]:
regime_validation_output = (
    RESULTS_DIR
    / "synthetic_regime_validation.csv"
)

regime_validation_df.to_csv(
    regime_validation_output,
    index=False,
)

print("Synthetic regime validation saved successfully:")
print(regime_validation_output)

In [ ]:
import time


def run_exact_greedy_benchmark(
    n_items: int,
    regime: str,
    tau: float,
    weight_profile: str,
    seed: int,
) -> dict:
    """
    Generate one synthetic MST instance and compare
    Exact-MST with Greedy-MST.

    Returns one structured experimental result row
    including solution quality, runtime, optimality gap,
    and selected-subset overlap.
    """

    if weight_profile not in WEIGHT_PROFILES:
        raise ValueError(
            f"Unknown weight profile: {weight_profile}"
        )

    weights = WEIGHT_PROFILES[
        weight_profile
    ]


    # --------------------------------------------------
    # Generate reproducible benchmark instance
    # --------------------------------------------------

    items = generate_mst_instance(
        n_items=n_items,
        regime=regime,
        seed=seed,
    )

    full_set_utility = (
        mst_core.mst_utility(items)
    )


    # --------------------------------------------------
    # Exact-MST
    # --------------------------------------------------

    exact_start = time.perf_counter()

    exact_result = mst_core.exact_mst(
        items=items,
        tau=tau,
        **weights,
    )

    exact_runtime = (
        time.perf_counter()
        - exact_start
    )


    # --------------------------------------------------
    # Greedy-MST
    # --------------------------------------------------

    greedy_start = time.perf_counter()

    greedy_result = mst_core.greedy_mst(
        items=items,
        tau=tau,
        **weights,
    )

    greedy_runtime = (
        time.perf_counter()
        - greedy_start
    )


    # --------------------------------------------------
    # Exact-vs-Greedy comparison
    # --------------------------------------------------

    if (
        exact_result["status"] == "OPTIMAL"
        and greedy_result["status"] == "FEASIBLE"
    ):

        optimality_gap = (
            mst_core.compute_optimality_gap(
                exact_result=exact_result,
                approximate_result=greedy_result,
            )
        )

        exact_set = set(
            exact_result[
                "selected_items"
            ]
        )

        greedy_set = set(
            greedy_result[
                "selected_items"
            ]
        )

        intersection = (
            exact_set
            & greedy_set
        )

        union = (
            exact_set
            | greedy_set
        )

        same_selected_subset = (
            exact_set
            == greedy_set
        )

        intersection_count = (
            len(intersection)
        )

        union_count = (
            len(union)
        )

        if union_count == 0:
            jaccard_similarity = 1.0
        else:
            jaccard_similarity = (
                intersection_count
                / union_count
            )

        shared_items = "|".join(
            sorted(intersection)
        )

        exact_only_items = "|".join(
            sorted(
                exact_set
                - greedy_set
            )
        )

        greedy_only_items = "|".join(
            sorted(
                greedy_set
                - exact_set
            )
        )

    else:

        optimality_gap = np.nan
        same_selected_subset = np.nan

        intersection_count = np.nan
        union_count = np.nan
        jaccard_similarity = np.nan

        shared_items = ""
        exact_only_items = ""
        greedy_only_items = ""


    # --------------------------------------------------
    # Return one experimental observation
    # --------------------------------------------------

    return {

        # Experimental condition
        "seed": seed,
        "n_items": n_items,
        "regime": regime,
        "tau": tau,
        "weight_profile": weight_profile,

        # Objective weights
        "lambda_privacy":
            weights[
                "lambda_privacy"
            ],

        "lambda_cognitive":
            weights[
                "lambda_cognitive"
            ],

        "lambda_cardinality":
            weights[
                "lambda_cardinality"
            ],

        # Candidate-set information
        "full_set_utility":
            full_set_utility,


        # ----------------------------------------------
        # Exact-MST results
        # ----------------------------------------------

        "exact_status":
            exact_result[
                "status"
            ],

        "exact_selected_count":
            len(
                exact_result[
                    "selected_items"
                ]
            ),

        "exact_selected_items":
            "|".join(
                exact_result[
                    "selected_items"
                ]
            ),

        "exact_utility":
            exact_result[
                "utility"
            ],

        "exact_privacy":
            exact_result[
                "privacy"
            ],

        "exact_normalized_cognitive_cost":
            exact_result[
                "normalized_cognitive_cost"
            ],

        "exact_disclosure_cost":
            exact_result[
                "disclosure_cost"
            ],

        "exact_runtime_seconds":
            exact_runtime,


        # ----------------------------------------------
        # Greedy-MST results
        # ----------------------------------------------

        "greedy_status":
            greedy_result[
                "status"
            ],

        "greedy_selected_count":
            len(
                greedy_result[
                    "selected_items"
                ]
            ),

        "greedy_selected_items":
            "|".join(
                greedy_result[
                    "selected_items"
                ]
            ),

        "greedy_utility":
            greedy_result[
                "utility"
            ],

        "greedy_privacy":
            greedy_result[
                "privacy"
            ],

        "greedy_normalized_cognitive_cost":
            greedy_result[
                "normalized_cognitive_cost"
            ],

        "greedy_disclosure_cost":
            greedy_result[
                "disclosure_cost"
            ],

        "greedy_runtime_seconds":
            greedy_runtime,


        # ----------------------------------------------
        # Solution-comparison metrics
        # ----------------------------------------------

        "optimality_gap":
            optimality_gap,

        "optimality_gap_percent":
            (
                100.0
                * optimality_gap
                if not np.isnan(
                    optimality_gap
                )
                else np.nan
            ),

        "same_selected_subset":
            same_selected_subset,

        "intersection_count":
            intersection_count,

        "union_count":
            union_count,

        "jaccard_similarity":
            jaccard_similarity,

        "shared_items":
            shared_items,

        "exact_only_items":
            exact_only_items,

        "greedy_only_items":
            greedy_only_items,
    }

In [ ]:
single_benchmark = (
    run_exact_greedy_benchmark(
        n_items=8,
        regime="utility_burden_conflict",
        tau=0.90,
        weight_profile="balanced",
        seed=GLOBAL_SEED,
    )
)

print(
    "Optimality gap:",
    f"{single_benchmark['optimality_gap_percent']:.2f}%"
)

print(
    "Jaccard similarity:",
    f"{single_benchmark['jaccard_similarity']:.4f}"
)

print(
    "Shared items:",
    single_benchmark["shared_items"]
)

print(
    "Exact-only items:",
    single_benchmark["exact_only_items"]
)

print(
    "Greedy-only items:",
    single_benchmark["greedy_only_items"]
)

In [ ]:
# --------------------------------------------------
# Small pilot benchmark
# --------------------------------------------------

PILOT_SIZES = [
    8,
    10,
]

PILOT_TAU_LEVELS = [
    0.80,
    0.90,
]

PILOT_REPLICATES = 5

PILOT_WEIGHT_PROFILE = "balanced"


pilot_rows = []

pilot_run_number = 0

total_pilot_runs = (
    len(PILOT_SIZES)
    * len(INSTANCE_REGIMES)
    * len(PILOT_TAU_LEVELS)
    * PILOT_REPLICATES
)


for n_items in PILOT_SIZES:

    for regime_index, regime in enumerate(
        INSTANCE_REGIMES
    ):

        for tau_index, tau in enumerate(
            PILOT_TAU_LEVELS
        ):

            for replicate in range(
                PILOT_REPLICATES
            ):

                pilot_run_number += 1

                instance_seed = (
                    GLOBAL_SEED
                    + n_items * 100_000
                    + regime_index * 10_000
                    + tau_index * 1_000
                    + replicate
                )

                result = (
                    run_exact_greedy_benchmark(
                        n_items=n_items,
                        regime=regime,
                        tau=tau,
                        weight_profile=
                            PILOT_WEIGHT_PROFILE,
                        seed=instance_seed,
                    )
                )

                result[
                    "replicate"
                ] = replicate

                pilot_rows.append(
                    result
                )

                print(
                    f"Completed "
                    f"{pilot_run_number}/"
                    f"{total_pilot_runs}",
                    end="\r",
                )


pilot_df = pd.DataFrame(
    pilot_rows
)

print(
    f"\nPilot benchmark completed: "
    f"{len(pilot_df)} runs"
)

In [ ]:
# --------------------------------------------------
# Pilot integrity checks
# --------------------------------------------------

feasible_rows = pilot_df[
    (pilot_df["exact_status"] == "OPTIMAL")
    &
    (pilot_df["greedy_status"] == "FEASIBLE")
].copy()


negative_gap_count = (
    feasible_rows[
        "optimality_gap"
    ]
    < -1e-12
).sum()


exact_threshold_failures = (
    feasible_rows[
        "exact_utility"
    ]
    <
    feasible_rows[
        "tau"
    ]
).sum()


greedy_threshold_failures = (
    feasible_rows[
        "greedy_utility"
    ]
    <
    feasible_rows[
        "tau"
    ]
).sum()


print(
    "Total pilot runs:",
    len(pilot_df)
)

print(
    "Comparable feasible runs:",
    len(feasible_rows)
)

print(
    "Negative optimality gaps:",
    negative_gap_count
)

print(
    "Exact threshold failures:",
    exact_threshold_failures
)

print(
    "Greedy threshold failures:",
    greedy_threshold_failures
)

print(
    "Mean optimality gap (%):",
    f"{feasible_rows['optimality_gap_percent'].mean():.4f}"
)

print(
    "Maximum optimality gap (%):",
    f"{feasible_rows['optimality_gap_percent'].max():.4f}"
)

print(
    "Mean Jaccard similarity:",
    f"{feasible_rows['jaccard_similarity'].mean():.4f}"
)

In [ ]:
# --------------------------------------------------
# Inspect the largest Greedy-MST optimality gaps
# --------------------------------------------------

worst_pilot_cases = (
    feasible_rows
    .sort_values(
        by="optimality_gap_percent",
        ascending=False,
    )
    .head(10)
    [
        [
            "seed",
            "n_items",
            "regime",
            "tau",
            "weight_profile",
            "exact_selected_count",
            "greedy_selected_count",
            "exact_disclosure_cost",
            "greedy_disclosure_cost",
            "optimality_gap_percent",
            "jaccard_similarity",
        ]
    ]
)

print(
    worst_pilot_cases.to_string(
        index=False
    )
)

In [ ]:
# --------------------------------------------------
# Detailed inspection of the worst pilot instance
# --------------------------------------------------

WORST_SEED = 802029
WORST_N_ITEMS = 8
WORST_REGIME = "independent"
WORST_TAU = 0.80
WORST_WEIGHT_PROFILE = "balanced"


worst_items = generate_mst_instance(
    n_items=WORST_N_ITEMS,
    regime=WORST_REGIME,
    seed=WORST_SEED,
)

worst_weights = WEIGHT_PROFILES[
    WORST_WEIGHT_PROFILE
]


worst_exact = mst_core.exact_mst(
    items=worst_items,
    tau=WORST_TAU,
    **worst_weights,
)

worst_greedy = mst_core.greedy_mst(
    items=worst_items,
    tau=WORST_TAU,
    **worst_weights,
)


exact_selected_set = set(
    worst_exact["selected_items"]
)

greedy_selected_set = set(
    worst_greedy["selected_items"]
)


worst_item_rows = []

for item in worst_items:

    worst_item_rows.append(
        {
            "item_id": item.item_id,
            "utility": item.utility,
            "privacy": item.privacy,
            "cognitive_cost": item.cognitive_cost,
            "selected_exact":
                item.item_id
                in exact_selected_set,
            "selected_greedy":
                item.item_id
                in greedy_selected_set,
        }
    )


worst_item_df = pd.DataFrame(
    worst_item_rows
)


print("ITEM-LEVEL DATA")
print(
    worst_item_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)


print("\nEXACT-MST")
print(
    "Selected:",
    worst_exact["selected_items"]
)
print(
    "Count:",
    len(worst_exact["selected_items"])
)
print(
    "Utility:",
    f"{worst_exact['utility']:.4f}"
)
print(
    "Privacy:",
    f"{worst_exact['privacy']:.4f}"
)
print(
    "Normalized cognitive cost:",
    f"{worst_exact['normalized_cognitive_cost']:.4f}"
)
print(
    "Disclosure cost:",
    f"{worst_exact['disclosure_cost']:.4f}"
)


print("\nGREEDY-MST")
print(
    "Selected:",
    worst_greedy["selected_items"]
)
print(
    "Count:",
    len(worst_greedy["selected_items"])
)
print(
    "Utility:",
    f"{worst_greedy['utility']:.4f}"
)
print(
    "Privacy:",
    f"{worst_greedy['privacy']:.4f}"
)
print(
    "Normalized cognitive cost:",
    f"{worst_greedy['normalized_cognitive_cost']:.4f}"
)
print(
    "Disclosure cost:",
    f"{worst_greedy['disclosure_cost']:.4f}"
)


worst_gap = mst_core.compute_optimality_gap(
    exact_result=worst_exact,
    approximate_result=worst_greedy,
)

print(
    "\nOptimality gap:",
    f"{100 * worst_gap:.2f}%"
)

In [ ]:
# --------------------------------------------------
# Trace Greedy-MST selection on the worst instance
# --------------------------------------------------

GREEDY_EPSILON = 1e-12


def evaluate_subset_for_trace(
    selected_items,
    all_items,
    weights,
):
    """
    Evaluate a subset using the official experimental
    MST utility and normalized disclosure-cost formulation.
    """

    # Modeled utility
    utility = (
        1.0
        - np.prod(
            [
                1.0 - item.utility
                for item in selected_items
            ]
        )
    ) if selected_items else 0.0

    # Relative privacy exposure
    privacy = (
        1.0
        - np.prod(
            [
                1.0 - item.privacy
                for item in selected_items
            ]
        )
    ) if selected_items else 0.0

    # Normalized interpretation cost
    total_cognitive_cost = sum(
        item.cognitive_cost
        for item in all_items
    )

    selected_cognitive_cost = sum(
        item.cognitive_cost
        for item in selected_items
    )

    if total_cognitive_cost > 0:
        normalized_cognitive_cost = (
            selected_cognitive_cost
            / total_cognitive_cost
        )
    else:
        normalized_cognitive_cost = 0.0

    # Normalized cardinality
    normalized_cardinality = (
        len(selected_items)
        / len(all_items)
    )

    # Official MST disclosure objective
    disclosure_cost = (
        weights["lambda_privacy"]
        * privacy
        +
        weights["lambda_cognitive"]
        * normalized_cognitive_cost
        +
        weights["lambda_cardinality"]
        * normalized_cardinality
    )

    return {
        "utility": utility,
        "disclosure_cost": disclosure_cost,
    }


selected = []
remaining = worst_items.copy()

trace_rows = []

step = 1


while True:

    current_metrics = (
        evaluate_subset_for_trace(
            selected_items=selected,
            all_items=worst_items,
            weights=worst_weights,
        )
    )

    if (
        current_metrics["utility"]
        >= WORST_TAU
    ):
        break

    candidate_rows = []

    for candidate in remaining:

        trial_subset = (
            selected
            + [candidate]
        )

        trial_metrics = (
            evaluate_subset_for_trace(
                selected_items=trial_subset,
                all_items=worst_items,
                weights=worst_weights,
            )
        )

        marginal_utility = (
            trial_metrics["utility"]
            - current_metrics["utility"]
        )

        marginal_cost = (
            trial_metrics[
                "disclosure_cost"
            ]
            - current_metrics[
                "disclosure_cost"
            ]
        )

        ratio = (
            marginal_utility
            /
            (
                marginal_cost
                + GREEDY_EPSILON
            )
        )

        candidate_rows.append(
            {
                "step": step,
                "candidate": candidate.item_id,
                "current_utility":
                    current_metrics[
                        "utility"
                    ],
                "marginal_utility":
                    marginal_utility,
                "marginal_cost":
                    marginal_cost,
                "ratio":
                    ratio,
                "resulting_utility":
                    trial_metrics[
                        "utility"
                    ],
            }
        )

    candidate_df = (
        pd.DataFrame(
            candidate_rows
        )
        .sort_values(
            by="ratio",
            ascending=False,
        )
        .reset_index(
            drop=True
        )
    )

    chosen_item_id = (
        candidate_df
        .iloc[0][
            "candidate"
        ]
    )

    candidate_df[
        "chosen"
    ] = (
        candidate_df[
            "candidate"
        ]
        == chosen_item_id
    )

    trace_rows.extend(
        candidate_df.to_dict(
            "records"
        )
    )

    chosen_item = next(
        item
        for item in remaining
        if item.item_id
        == chosen_item_id
    )

    selected.append(
        chosen_item
    )

    remaining.remove(
        chosen_item
    )

    step += 1


greedy_trace_df = pd.DataFrame(
    trace_rows
)


for step_number in sorted(
    greedy_trace_df["step"].unique()
):

    print(
        f"\nGREEDY STEP {step_number}"
    )

    step_df = (
        greedy_trace_df[
            greedy_trace_df[
                "step"
            ]
            == step_number
        ]
    )

    print(
        step_df[
            [
                "candidate",
                "marginal_utility",
                "marginal_cost",
                "ratio",
                "resulting_utility",
                "chosen",
            ]
        ].to_string(
            index=False,
            float_format=
                lambda x: f"{x:.4f}",
        )
    )


print(
    "\nFinal traced selection:",
    tuple(
        item.item_id
        for item in selected
    )
)

print(
    "Original Greedy selection:",
    worst_greedy[
        "selected_items"
    ]
)

In [ ]:
# --------------------------------------------------
# Pilot benchmark summary by experimental condition
# --------------------------------------------------

pilot_condition_summary = (
    feasible_rows
    .groupby(
        [
            "n_items",
            "regime",
            "tau",
        ],
        as_index=False,
    )
    .agg(
        number_of_runs=(
            "optimality_gap_percent",
            "count",
        ),

        mean_optimality_gap_percent=(
            "optimality_gap_percent",
            "mean",
        ),

        median_optimality_gap_percent=(
            "optimality_gap_percent",
            "median",
        ),

        max_optimality_gap_percent=(
            "optimality_gap_percent",
            "max",
        ),

        mean_jaccard_similarity=(
            "jaccard_similarity",
            "mean",
        ),

        exact_subset_match_rate=(
            "same_selected_subset",
            "mean",
        ),

        mean_exact_selected_count=(
            "exact_selected_count",
            "mean",
        ),

        mean_greedy_selected_count=(
            "greedy_selected_count",
            "mean",
        ),

        mean_exact_runtime_seconds=(
            "exact_runtime_seconds",
            "mean",
        ),

        mean_greedy_runtime_seconds=(
            "greedy_runtime_seconds",
            "mean",
        ),
    )
)


# Convert exact subset match rate to percentage
pilot_condition_summary[
    "exact_subset_match_percent"
] = (
    100.0
    * pilot_condition_summary[
        "exact_subset_match_rate"
    ]
)


# Compute average runtime speedup:
# Exact runtime divided by Greedy runtime
pilot_condition_summary[
    "greedy_runtime_speedup"
] = (
    pilot_condition_summary[
        "mean_exact_runtime_seconds"
    ]
    /
    pilot_condition_summary[
        "mean_greedy_runtime_seconds"
    ]
)


display_columns = [
    "n_items",
    "regime",
    "tau",
    "number_of_runs",
    "mean_optimality_gap_percent",
    "max_optimality_gap_percent",
    "mean_jaccard_similarity",
    "exact_subset_match_percent",
    "mean_exact_selected_count",
    "mean_greedy_selected_count",
    "greedy_runtime_speedup",
]


print(
    pilot_condition_summary[
        display_columns
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)

In [ ]:
# --------------------------------------------------
# Exact-MST runtime scaling pilot
# --------------------------------------------------

RUNTIME_PILOT_SIZES = [
    12,
    14,
    16,
]

RUNTIME_PILOT_REPLICATES = 3

RUNTIME_PILOT_TAU = 0.90

RUNTIME_PILOT_WEIGHT_PROFILE = (
    "balanced"
)


runtime_pilot_rows = []

runtime_run_number = 0

total_runtime_runs = (
    len(RUNTIME_PILOT_SIZES)
    * len(INSTANCE_REGIMES)
    * RUNTIME_PILOT_REPLICATES
)


for n_items in RUNTIME_PILOT_SIZES:

    for regime_index, regime in enumerate(
        INSTANCE_REGIMES
    ):

        for replicate in range(
            RUNTIME_PILOT_REPLICATES
        ):

            runtime_run_number += 1

            instance_seed = (
                GLOBAL_SEED
                + 5_000_000
                + n_items * 100_000
                + regime_index * 10_000
                + replicate
            )

            result = (
                run_exact_greedy_benchmark(
                    n_items=n_items,
                    regime=regime,
                    tau=RUNTIME_PILOT_TAU,
                    weight_profile=
                        RUNTIME_PILOT_WEIGHT_PROFILE,
                    seed=instance_seed,
                )
            )

            result[
                "replicate"
            ] = replicate

            runtime_pilot_rows.append(
                result
            )

            print(
                f"Completed "
                f"{runtime_run_number}/"
                f"{total_runtime_runs}",
                end="\r",
            )


runtime_pilot_df = pd.DataFrame(
    runtime_pilot_rows
)


runtime_scaling_summary = (
    runtime_pilot_df
    .groupby(
        "n_items",
        as_index=False,
    )
    .agg(
        number_of_runs=(
            "exact_runtime_seconds",
            "count",
        ),

        mean_exact_runtime_seconds=(
            "exact_runtime_seconds",
            "mean",
        ),

        median_exact_runtime_seconds=(
            "exact_runtime_seconds",
            "median",
        ),

        max_exact_runtime_seconds=(
            "exact_runtime_seconds",
            "max",
        ),

        mean_greedy_runtime_seconds=(
            "greedy_runtime_seconds",
            "mean",
        ),

        mean_optimality_gap_percent=(
            "optimality_gap_percent",
            "mean",
        ),

        max_optimality_gap_percent=(
            "optimality_gap_percent",
            "max",
        ),
    )
)


print(
    "\nExact-MST runtime scaling pilot"
)

print(
    runtime_scaling_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

In [ ]:
# --------------------------------------------------
# Reusable exact subset-space precomputation
# --------------------------------------------------

from itertools import combinations


def precompute_subset_space(items):
    """
    Enumerate every subset of one MST candidate set once
    and store all weight-independent metrics.

    This allows the same synthetic instance to be evaluated
    efficiently across multiple tau levels and objective
    weight profiles without repeating subset enumeration.
    """

    n_items = len(items)

    if n_items == 0:
        raise ValueError(
            "The candidate set must contain at least one item."
        )


    # Total raw interpretation cost of the admissible set
    total_cognitive_cost = sum(
        item.cognitive_cost
        for item in items
    )


    subset_rows = []


    # Enumerate every subset, including the empty set
    for subset_size in range(
        n_items + 1
    ):

        for subset_indices in combinations(
            range(n_items),
            subset_size,
        ):

            selected_items = [
                items[index]
                for index in subset_indices
            ]


            # ------------------------------------------
            # Modeled utility
            # ------------------------------------------

            if selected_items:

                utility = (
                    1.0
                    - np.prod(
                        [
                            1.0
                            - item.utility
                            for item
                            in selected_items
                        ]
                    )
                )

            else:

                utility = 0.0


            # ------------------------------------------
            # Relative privacy exposure
            # ------------------------------------------

            if selected_items:

                privacy = (
                    1.0
                    - np.prod(
                        [
                            1.0
                            - item.privacy
                            for item
                            in selected_items
                        ]
                    )
                )

            else:

                privacy = 0.0


            # ------------------------------------------
            # Normalized interpretation cost
            # ------------------------------------------

            selected_cognitive_cost = sum(
                item.cognitive_cost
                for item
                in selected_items
            )


            if total_cognitive_cost > 0:

                normalized_cognitive_cost = (
                    selected_cognitive_cost
                    / total_cognitive_cost
                )

            else:

                normalized_cognitive_cost = 0.0


            # ------------------------------------------
            # Normalized cardinality
            # ------------------------------------------

            normalized_cardinality = (
                subset_size
                / n_items
            )


            # ------------------------------------------
            # Selected item identifiers
            # ------------------------------------------

            selected_item_ids = tuple(
                item.item_id
                for item
                in selected_items
            )


            subset_rows.append(
                {
                    "selected_items":
                        selected_item_ids,

                    "selected_count":
                        subset_size,

                    "utility":
                        float(utility),

                    "privacy":
                        float(privacy),

                    "normalized_cognitive_cost":
                        float(
                            normalized_cognitive_cost
                        ),

                    "normalized_cardinality":
                        float(
                            normalized_cardinality
                        ),
                }
            )


    return pd.DataFrame(
        subset_rows
    )



def solve_exact_from_precomputed(
    subset_space,
    tau,
    weights,
):
    """
    Recover the exact MST solution from an already
    enumerated subset space.
    """

    feasible = subset_space[
        subset_space[
            "utility"
        ]
        >= tau
    ].copy()


    if feasible.empty:

        return {
            "status":
                "INFEASIBLE",

            "selected_items":
                tuple(),

            "selected_count":
                0,

            "utility":
                np.nan,

            "privacy":
                np.nan,

            "normalized_cognitive_cost":
                np.nan,

            "disclosure_cost":
                np.nan,
        }


    # Official normalized MST objective
    feasible[
        "disclosure_cost"
    ] = (

        weights[
            "lambda_privacy"
        ]
        * feasible[
            "privacy"
        ]

        +

        weights[
            "lambda_cognitive"
        ]
        * feasible[
            "normalized_cognitive_cost"
        ]

        +

        weights[
            "lambda_cardinality"
        ]
        * feasible[
            "normalized_cardinality"
        ]
    )


    # Identify the minimum objective value
    minimum_cost = feasible[
        "disclosure_cost"
    ].min()


    minimum_cost_candidates = feasible[
        feasible[
            "disclosure_cost"
        ]
        == minimum_cost
    ]


    # Apply the same deterministic tie-breaking logic
    # used by the official Exact-MST implementation.
    best_row = min(

        minimum_cost_candidates.to_dict(
            "records"
        ),

        key=lambda row: (

            row[
                "privacy"
            ],

            row[
                "normalized_cognitive_cost"
            ],

            row[
                "selected_count"
            ],

            row[
                "selected_items"
            ],
        ),
    )


    return {
        "status":
            "OPTIMAL",

        "selected_items":
            best_row[
                "selected_items"
            ],

        "selected_count":
            int(
                best_row[
                    "selected_count"
                ]
            ),

        "utility":
            best_row[
                "utility"
            ],

        "privacy":
            best_row[
                "privacy"
            ],

        "normalized_cognitive_cost":
            best_row[
                "normalized_cognitive_cost"
            ],

        "disclosure_cost":
            best_row[
                "disclosure_cost"
            ],
    }

In [ ]:
# --------------------------------------------------
# Validate precomputed Exact-MST against official solver
# --------------------------------------------------

precompute_start = time.perf_counter()

worst_subset_space = (
    precompute_subset_space(
        worst_items
    )
)

precompute_runtime = (
    time.perf_counter()
    - precompute_start
)


cached_exact = (
    solve_exact_from_precomputed(
        subset_space=
            worst_subset_space,

        tau=
            WORST_TAU,

        weights=
            worst_weights,
    )
)


print(
    "Number of enumerated subsets:",
    len(worst_subset_space)
)

print(
    "Expected subsets:",
    2 ** WORST_N_ITEMS
)

print(
    "Precomputation runtime:",
    f"{precompute_runtime:.6f} seconds"
)


print(
    "\nOfficial Exact selection:",
    worst_exact[
        "selected_items"
    ]
)

print(
    "Precomputed Exact selection:",
    cached_exact[
        "selected_items"
    ]
)


print(
    "\nOfficial Exact cost:",
    f"{worst_exact['disclosure_cost']:.10f}"
)

print(
    "Precomputed Exact cost:",
    f"{cached_exact['disclosure_cost']:.10f}"
)


print(
    "\nSelection match:",
    cached_exact[
        "selected_items"
    ]
    ==
    worst_exact[
        "selected_items"
    ]
)

print(
    "Cost match:",
    np.isclose(
        cached_exact[
            "disclosure_cost"
        ],
        worst_exact[
            "disclosure_cost"
        ],
        atol=1e-12,
    )
)

In [ ]:
# --------------------------------------------------
# Cached Exact-MST efficiency test at n = 16
# --------------------------------------------------

CACHE_TEST_N_ITEMS = 16
CACHE_TEST_REGIME = "independent"
CACHE_TEST_SEED = GLOBAL_SEED + 9_000_000


# Generate one fixed benchmark instance
cache_test_items = generate_mst_instance(
    n_items=CACHE_TEST_N_ITEMS,
    regime=CACHE_TEST_REGIME,
    seed=CACHE_TEST_SEED,
)


# --------------------------------------------------
# Enumerate subset space once
# --------------------------------------------------

precompute_start = time.perf_counter()

cache_test_subset_space = (
    precompute_subset_space(
        cache_test_items
    )
)

precompute_runtime = (
    time.perf_counter()
    - precompute_start
)


# --------------------------------------------------
# Solve all tau × weight conditions
# --------------------------------------------------

cached_solve_rows = []

cached_solve_start = time.perf_counter()


for tau in TAU_LEVELS:

    for weight_profile, weights in (
        WEIGHT_PROFILES.items()
    ):

        solve_start = time.perf_counter()

        exact_solution = (
            solve_exact_from_precomputed(
                subset_space=
                    cache_test_subset_space,
                tau=tau,
                weights=weights,
            )
        )

        solve_runtime = (
            time.perf_counter()
            - solve_start
        )

        cached_solve_rows.append(
            {
                "tau":
                    tau,

                "weight_profile":
                    weight_profile,

                "status":
                    exact_solution[
                        "status"
                    ],

                "selected_count":
                    exact_solution[
                        "selected_count"
                    ],

                "disclosure_cost":
                    exact_solution[
                        "disclosure_cost"
                    ],

                "solve_runtime_seconds":
                    solve_runtime,
            }
        )


total_cached_solve_runtime = (
    time.perf_counter()
    - cached_solve_start
)


cached_solve_df = pd.DataFrame(
    cached_solve_rows
)


# --------------------------------------------------
# Summary
# --------------------------------------------------

print(
    "Number of enumerated subsets:",
    len(cache_test_subset_space)
)

print(
    "Expected subsets:",
    2 ** CACHE_TEST_N_ITEMS
)

print(
    "Subset precomputation runtime:",
    f"{precompute_runtime:.4f} seconds"
)

print(
    "Total runtime for all 16 cached solves:",
    f"{total_cached_solve_runtime:.4f} seconds"
)

print(
    "Mean cached solve runtime:",
    f"{cached_solve_df['solve_runtime_seconds'].mean():.6f} seconds"
)

print(
    "Maximum cached solve runtime:",
    f"{cached_solve_df['solve_runtime_seconds'].max():.6f} seconds"
)

print(
    "\nNumber of evaluated conditions:",
    len(cached_solve_df)
)

print(
    "Feasible conditions:",
    (
        cached_solve_df[
            "status"
        ]
        == "OPTIMAL"
    ).sum()
)

In [ ]:
# --------------------------------------------------
# Cached Exact-vs-Greedy benchmark for one instance
# --------------------------------------------------

def run_cached_instance_benchmark(
    n_items: int,
    regime: str,
    seed: int,
) -> list[dict]:
    """
    Generate one synthetic MST instance, enumerate its
    complete subset space once, and evaluate all combinations
    of tau levels and objective-weight profiles.

    Exact-MST uses the reusable precomputed subset space.
    Greedy-MST is executed separately for each condition.

    Returns
    -------
    list[dict]
        One result row for every tau × weight-profile condition.
    """

    # --------------------------------------------------
    # Generate one reproducible synthetic instance
    # --------------------------------------------------

    items = generate_mst_instance(
        n_items=n_items,
        regime=regime,
        seed=seed,
    )


    full_set_utility = (
        mst_core.mst_utility(
            items
        )
    )


    # --------------------------------------------------
    # Precompute complete exact subset space once
    # --------------------------------------------------

    precompute_start = (
        time.perf_counter()
    )

    subset_space = (
        precompute_subset_space(
            items
        )
    )

    precompute_runtime = (
        time.perf_counter()
        - precompute_start
    )


    result_rows = []


    # --------------------------------------------------
    # Evaluate all experimental conditions
    # --------------------------------------------------

    for tau in TAU_LEVELS:

        for (
            weight_profile,
            weights,
        ) in WEIGHT_PROFILES.items():


            # ------------------------------------------
            # Cached Exact-MST solution
            # ------------------------------------------

            exact_solve_start = (
                time.perf_counter()
            )

            exact_result = (
                solve_exact_from_precomputed(
                    subset_space=
                        subset_space,

                    tau=
                        tau,

                    weights=
                        weights,
                )
            )

            exact_solve_runtime = (
                time.perf_counter()
                - exact_solve_start
            )


            # ------------------------------------------
            # Greedy-MST solution
            # ------------------------------------------

            greedy_start = (
                time.perf_counter()
            )

            greedy_result = (
                mst_core.greedy_mst(
                    items=items,
                    tau=tau,
                    **weights,
                )
            )

            greedy_runtime = (
                time.perf_counter()
                - greedy_start
            )


            # ------------------------------------------
            # Compare solutions
            # ------------------------------------------

            if (
                exact_result["status"]
                == "OPTIMAL"
                and
                greedy_result["status"]
                == "FEASIBLE"
            ):

                exact_set = set(
                    exact_result[
                        "selected_items"
                    ]
                )

                greedy_set = set(
                    greedy_result[
                        "selected_items"
                    ]
                )


                # Relative optimality gap
                exact_cost = (
                    exact_result[
                        "disclosure_cost"
                    ]
                )

                greedy_cost = (
                    greedy_result[
                        "disclosure_cost"
                    ]
                )


                if exact_cost > 0:

                    optimality_gap = (
                        greedy_cost
                        - exact_cost
                    ) / exact_cost

                else:

                    optimality_gap = (
                        0.0
                        if np.isclose(
                            greedy_cost,
                            exact_cost,
                        )
                        else np.inf
                    )


                # Subset overlap
                intersection = (
                    exact_set
                    & greedy_set
                )

                union = (
                    exact_set
                    | greedy_set
                )

                intersection_count = (
                    len(intersection)
                )

                union_count = (
                    len(union)
                )


                if union_count > 0:

                    jaccard_similarity = (
                        intersection_count
                        / union_count
                    )

                else:

                    jaccard_similarity = 1.0


                same_selected_subset = (
                    exact_set
                    == greedy_set
                )


                shared_items = "|".join(
                    sorted(
                        intersection
                    )
                )

                exact_only_items = "|".join(
                    sorted(
                        exact_set
                        - greedy_set
                    )
                )

                greedy_only_items = "|".join(
                    sorted(
                        greedy_set
                        - exact_set
                    )
                )


            else:

                optimality_gap = np.nan

                intersection_count = np.nan
                union_count = np.nan
                jaccard_similarity = np.nan
                same_selected_subset = np.nan

                shared_items = ""
                exact_only_items = ""
                greedy_only_items = ""


            # ------------------------------------------
            # Store experimental observation
            # ------------------------------------------

            result_rows.append(
                {
                    # Instance identity
                    "seed":
                        seed,

                    "n_items":
                        n_items,

                    "regime":
                        regime,


                    # Experimental condition
                    "tau":
                        tau,

                    "weight_profile":
                        weight_profile,

                    "lambda_privacy":
                        weights[
                            "lambda_privacy"
                        ],

                    "lambda_cognitive":
                        weights[
                            "lambda_cognitive"
                        ],

                    "lambda_cardinality":
                        weights[
                            "lambda_cardinality"
                        ],


                    # Candidate-set information
                    "full_set_utility":
                        full_set_utility,

                    "subset_space_size":
                        len(
                            subset_space
                        ),

                    "subset_precomputation_runtime_seconds":
                        precompute_runtime,


                    # Exact-MST
                    "exact_status":
                        exact_result[
                            "status"
                        ],

                    "exact_selected_count":
                        exact_result[
                            "selected_count"
                        ],

                    "exact_selected_items":
                        "|".join(
                            exact_result[
                                "selected_items"
                            ]
                        ),

                    "exact_utility":
                        exact_result[
                            "utility"
                        ],

                    "exact_privacy":
                        exact_result[
                            "privacy"
                        ],

                    "exact_normalized_cognitive_cost":
                        exact_result[
                            "normalized_cognitive_cost"
                        ],

                    "exact_disclosure_cost":
                        exact_result[
                            "disclosure_cost"
                        ],

                    "exact_cached_solve_runtime_seconds":
                        exact_solve_runtime,


                    # Greedy-MST
                    "greedy_status":
                        greedy_result[
                            "status"
                        ],

                    "greedy_selected_count":
                        len(
                            greedy_result[
                                "selected_items"
                            ]
                        ),

                    "greedy_selected_items":
                        "|".join(
                            greedy_result[
                                "selected_items"
                            ]
                        ),

                    "greedy_utility":
                        greedy_result[
                            "utility"
                        ],

                    "greedy_privacy":
                        greedy_result[
                            "privacy"
                        ],

                    "greedy_normalized_cognitive_cost":
                        greedy_result[
                            "normalized_cognitive_cost"
                        ],

                    "greedy_disclosure_cost":
                        greedy_result[
                            "disclosure_cost"
                        ],

                    "greedy_runtime_seconds":
                        greedy_runtime,


                    # Comparison
                    "optimality_gap":
                        optimality_gap,

                    "optimality_gap_percent":
                        (
                            100.0
                            * optimality_gap
                            if np.isfinite(
                                optimality_gap
                            )
                            else optimality_gap
                        ),

                    "same_selected_subset":
                        same_selected_subset,

                    "intersection_count":
                        intersection_count,

                    "union_count":
                        union_count,

                    "jaccard_similarity":
                        jaccard_similarity,

                    "shared_items":
                        shared_items,

                    "exact_only_items":
                        exact_only_items,

                    "greedy_only_items":
                        greedy_only_items,
                }
            )


    return result_rows

In [ ]:
cached_instance_test = (
    run_cached_instance_benchmark(
        n_items=8,
        regime="independent",
        seed=GLOBAL_SEED,
    )
)

cached_instance_test_df = pd.DataFrame(
    cached_instance_test
)


print(
    "Number of result rows:",
    len(cached_instance_test_df)
)

print(
    "Expected rows:",
    len(TAU_LEVELS)
    * len(WEIGHT_PROFILES)
)

print(
    "Unique instance seeds:",
    cached_instance_test_df[
        "seed"
    ].nunique()
)

print(
    "Negative optimality gaps:",
    (
        cached_instance_test_df[
            "optimality_gap"
        ]
        < -1e-12
    ).sum()
)

print(
    "Exact threshold failures:",
    (
        cached_instance_test_df[
            "exact_utility"
        ]
        <
        cached_instance_test_df[
            "tau"
        ]
    ).sum()
)

print(
    "Greedy threshold failures:",
    (
        cached_instance_test_df[
            "greedy_utility"
        ]
        <
        cached_instance_test_df[
            "tau"
        ]
    ).sum()
)

In [ ]:
# --------------------------------------------------
# Cross-validate cached Exact-MST against official solver
# across all tau × weight-profile conditions
# --------------------------------------------------

validation_items = generate_mst_instance(
    n_items=8,
    regime="independent",
    seed=GLOBAL_SEED,
)

validation_subset_space = (
    precompute_subset_space(
        validation_items
    )
)


cross_validation_rows = []


for tau in TAU_LEVELS:

    for (
        weight_profile,
        weights,
    ) in WEIGHT_PROFILES.items():


        # ----------------------------------------------
        # Official Exact-MST
        # ----------------------------------------------

        official_result = (
            mst_core.exact_mst(
                items=validation_items,
                tau=tau,
                **weights,
            )
        )


        # ----------------------------------------------
        # Cached Exact-MST
        # ----------------------------------------------

        cached_result = (
            solve_exact_from_precomputed(
                subset_space=
                    validation_subset_space,

                tau=
                    tau,

                weights=
                    weights,
            )
        )


        # ----------------------------------------------
        # Compare results
        # ----------------------------------------------

        status_match = (
            official_result["status"]
            ==
            cached_result["status"]
        )


        if (
            official_result["status"]
            == "OPTIMAL"
            and
            cached_result["status"]
            == "OPTIMAL"
        ):

            selection_match = (
                tuple(
                    official_result[
                        "selected_items"
                    ]
                )
                ==
                tuple(
                    cached_result[
                        "selected_items"
                    ]
                )
            )

            utility_match = np.isclose(
                official_result[
                    "utility"
                ],
                cached_result[
                    "utility"
                ],
                atol=1e-12,
            )

            privacy_match = np.isclose(
                official_result[
                    "privacy"
                ],
                cached_result[
                    "privacy"
                ],
                atol=1e-12,
            )

            cognitive_match = np.isclose(
                official_result[
                    "normalized_cognitive_cost"
                ],
                cached_result[
                    "normalized_cognitive_cost"
                ],
                atol=1e-12,
            )

            cost_match = np.isclose(
                official_result[
                    "disclosure_cost"
                ],
                cached_result[
                    "disclosure_cost"
                ],
                atol=1e-12,
            )

        else:

            selection_match = (
                status_match
            )

            utility_match = (
                status_match
            )

            privacy_match = (
                status_match
            )

            cognitive_match = (
                status_match
            )

            cost_match = (
                status_match
            )


        all_match = all(
            [
                status_match,
                selection_match,
                utility_match,
                privacy_match,
                cognitive_match,
                cost_match,
            ]
        )


        cross_validation_rows.append(
            {
                "tau":
                    tau,

                "weight_profile":
                    weight_profile,

                "status_match":
                    status_match,

                "selection_match":
                    selection_match,

                "utility_match":
                    utility_match,

                "privacy_match":
                    privacy_match,

                "cognitive_match":
                    cognitive_match,

                "cost_match":
                    cost_match,

                "all_match":
                    all_match,
            }
        )


cross_validation_df = pd.DataFrame(
    cross_validation_rows
)


print(
    "Conditions checked:",
    len(cross_validation_df)
)

print(
    "Complete matches:",
    cross_validation_df[
        "all_match"
    ].sum()
)

print(
    "Selection mismatches:",
    (
        ~cross_validation_df[
            "selection_match"
        ]
    ).sum()
)

print(
    "Cost mismatches:",
    (
        ~cross_validation_df[
            "cost_match"
        ]
    ).sum()
)

print(
    "\nAll cached Exact-MST results match official solver:",
    cross_validation_df[
        "all_match"
    ].all()
)

In [ ]:
# --------------------------------------------------
# Main Exact-vs-Greedy computational benchmark
# --------------------------------------------------

import time


MAIN_REPLICATES = 100


MAIN_CHECKPOINT_PATH = (
    RESULTS_DIR
    / "exact_vs_greedy_main_benchmark_checkpoint.csv"
)

MAIN_FINAL_PATH = (
    RESULTS_DIR
    / "exact_vs_greedy_main_benchmark.csv"
)


# --------------------------------------------------
# Load existing checkpoint if available
# --------------------------------------------------

if MAIN_CHECKPOINT_PATH.exists():

    print(
        "Existing checkpoint found."
    )

    existing_main_df = pd.read_csv(
        MAIN_CHECKPOINT_PATH
    )

    main_rows = (
        existing_main_df
        .to_dict(
            "records"
        )
    )

    completed_instance_ids = set(
        existing_main_df[
            "instance_id"
        ].unique()
    )

    print(
        "Completed instances already available:",
        len(
            completed_instance_ids
        )
    )

else:

    main_rows = []

    completed_instance_ids = set()

    print(
        "No existing checkpoint found."
    )

    print(
        "Starting new main benchmark."
    )


# --------------------------------------------------
# Experimental size
# --------------------------------------------------

total_instances = (
    len(
        EXACT_BENCHMARK_SIZES
    )
    *
    len(
        INSTANCE_REGIMES
    )
    *
    MAIN_REPLICATES
)


total_expected_rows = (
    total_instances
    *
    len(
        TAU_LEVELS
    )
    *
    len(
        WEIGHT_PROFILES
    )
)


print(
    "\nTotal independent instances:",
    total_instances
)

print(
    "Expected experimental rows:",
    total_expected_rows
)


# --------------------------------------------------
# Main experiment
# --------------------------------------------------

benchmark_start_time = (
    time.perf_counter()
)


for size_index, n_items in enumerate(
    EXACT_BENCHMARK_SIZES
):

    for regime_index, regime in enumerate(
        INSTANCE_REGIMES
    ):

        print(
            "\nStarting condition:",
            f"n_items={n_items},",
            f"regime={regime}"
        )


        for replicate in range(
            MAIN_REPLICATES
        ):


            # ------------------------------------------
            # Unique instance identifier
            # ------------------------------------------

            instance_id = (
                f"M{n_items}"
                f"_{regime}"
                f"_R{replicate:03d}"
            )


            # ------------------------------------------
            # Skip already completed instances
            # when resuming from checkpoint
            # ------------------------------------------

            if (
                instance_id
                in completed_instance_ids
            ):

                continue


            # ------------------------------------------
            # Deterministic unique seed
            # ------------------------------------------

            instance_seed = (

                GLOBAL_SEED

                + 10_000_000

                + size_index
                * 1_000_000

                + regime_index
                * 100_000

                + replicate
            )


            # ------------------------------------------
            # Run all 16 conditions for this instance
            # ------------------------------------------

            instance_result_rows = (
                run_cached_instance_benchmark(
                    n_items=n_items,
                    regime=regime,
                    seed=instance_seed,
                )
            )


            # ------------------------------------------
            # Add instance metadata
            # ------------------------------------------

            for row in (
                instance_result_rows
            ):

                row[
                    "instance_id"
                ] = instance_id

                row[
                    "replicate"
                ] = replicate


            # ------------------------------------------
            # Store completed results
            # ------------------------------------------

            main_rows.extend(
                instance_result_rows
            )

            completed_instance_ids.add(
                instance_id
            )


            # ------------------------------------------
            # Save checkpoint every 5 instances
            # ------------------------------------------

            completed_count = len(
                completed_instance_ids
            )


            if (
                completed_count % 5
                == 0
            ):

                checkpoint_df = (
                    pd.DataFrame(
                        main_rows
                    )
                )

                checkpoint_df.to_csv(
                    MAIN_CHECKPOINT_PATH,
                    index=False,
                )


                elapsed_seconds = (
                    time.perf_counter()
                    - benchmark_start_time
                )


                print(
                    f"Completed "
                    f"{completed_count}/"
                    f"{total_instances} instances "
                    f"| rows="
                    f"{len(checkpoint_df)} "
                    f"| elapsed="
                    f"{elapsed_seconds / 60:.2f} min"
                )


        # ----------------------------------------------
        # Save after every size-regime block
        # ----------------------------------------------

        block_checkpoint_df = (
            pd.DataFrame(
                main_rows
            )
        )

        block_checkpoint_df.to_csv(
            MAIN_CHECKPOINT_PATH,
            index=False,
        )


# --------------------------------------------------
# Create final benchmark dataset
# --------------------------------------------------

main_benchmark_df = (
    pd.DataFrame(
        main_rows
    )
)


# Sort for reproducible organization
main_benchmark_df = (
    main_benchmark_df
    .sort_values(
        by=[
            "n_items",
            "regime",
            "replicate",
            "tau",
            "weight_profile",
        ]
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------
# Save final benchmark
# --------------------------------------------------

main_benchmark_df.to_csv(
    MAIN_FINAL_PATH,
    index=False,
)


# --------------------------------------------------
# Final integrity summary
# --------------------------------------------------

total_runtime_seconds = (
    time.perf_counter()
    - benchmark_start_time
)


print(
    "\n======================================"
)

print(
    "MAIN BENCHMARK COMPLETED"
)

print(
    "======================================"
)


print(
    "Independent instances:",
    main_benchmark_df[
        "instance_id"
    ].nunique()
)


print(
    "Total experimental rows:",
    len(
        main_benchmark_df
    )
)


print(
    "Expected experimental rows:",
    total_expected_rows
)


print(
    "Negative optimality gaps:",
    (
        main_benchmark_df[
            "optimality_gap"
        ]
        < -1e-12
    ).sum()
)


print(
    "Exact threshold failures:",
    (
        (
            main_benchmark_df[
                "exact_status"
            ]
            == "OPTIMAL"
        )
        &
        (
            main_benchmark_df[
                "exact_utility"
            ]
            <
            main_benchmark_df[
                "tau"
            ]
        )
    ).sum()
)


print(
    "Greedy threshold failures:",
    (
        (
            main_benchmark_df[
                "greedy_status"
            ]
            == "FEASIBLE"
        )
        &
        (
            main_benchmark_df[
                "greedy_utility"
            ]
            <
            main_benchmark_df[
                "tau"
            ]
        )
    ).sum()
)


print(
    "Total runtime:",
    f"{total_runtime_seconds / 60:.2f} minutes"
)


print(
    "\nFinal results saved to:"
)

print(
    MAIN_FINAL_PATH
)

In [ ]:
# --------------------------------------------------
# Final benchmark dataset integrity audit
# --------------------------------------------------

BENCHMARK_KEY_COLUMNS = [
    "instance_id",
    "tau",
    "weight_profile",
]

COMPARISON_METRIC_COLUMNS = [
    "exact_utility",
    "exact_privacy",
    "exact_normalized_cognitive_cost",
    "exact_disclosure_cost",
    "greedy_utility",
    "greedy_privacy",
    "greedy_normalized_cognitive_cost",
    "greedy_disclosure_cost",
    "optimality_gap",
    "optimality_gap_percent",
    "jaccard_similarity",
]


# --------------------------------------------------
# Basic dimensional checks
# --------------------------------------------------

expected_conditions_per_instance = (
    len(TAU_LEVELS)
    * len(WEIGHT_PROFILES)
)

rows_per_instance = (
    main_benchmark_df
    .groupby("instance_id")
    .size()
)

duplicate_condition_rows = (
    main_benchmark_df
    .duplicated(
        subset=BENCHMARK_KEY_COLUMNS,
        keep=False,
    )
    .sum()
)


# --------------------------------------------------
# Experimental coverage
# --------------------------------------------------

coverage_summary = (
    main_benchmark_df
    .groupby(
        [
            "n_items",
            "regime",
        ],
        as_index=False,
    )
    .agg(
        independent_instances=(
            "instance_id",
            "nunique",
        ),
        experimental_rows=(
            "instance_id",
            "size",
        ),
        unique_seeds=(
            "seed",
            "nunique",
        ),
    )
)


# --------------------------------------------------
# Status checks
# --------------------------------------------------

exact_status_counts = (
    main_benchmark_df[
        "exact_status"
    ]
    .value_counts(
        dropna=False
    )
)

greedy_status_counts = (
    main_benchmark_df[
        "greedy_status"
    ]
    .value_counts(
        dropna=False
    )
)

comparable_mask = (
    (
        main_benchmark_df[
            "exact_status"
        ]
        == "OPTIMAL"
    )
    &
    (
        main_benchmark_df[
            "greedy_status"
        ]
        == "FEASIBLE"
    )
)

comparable_df = (
    main_benchmark_df[
        comparable_mask
    ].copy()
)


# --------------------------------------------------
# Missing and invalid-value checks
# --------------------------------------------------

missing_comparison_values = (
    comparable_df[
        COMPARISON_METRIC_COLUMNS
    ]
    .isna()
    .sum()
    .sum()
)

nonfinite_comparison_values = (
    ~np.isfinite(
        comparable_df[
            COMPARISON_METRIC_COLUMNS
        ].to_numpy(
            dtype=float
        )
    )
).sum()

invalid_jaccard_values = (
    (
        comparable_df[
            "jaccard_similarity"
        ]
        < 0.0
    )
    |
    (
        comparable_df[
            "jaccard_similarity"
        ]
        > 1.0
    )
).sum()

negative_gap_values = (
    comparable_df[
        "optimality_gap"
    ]
    < -1e-12
).sum()


# --------------------------------------------------
# Threshold checks
# --------------------------------------------------

exact_threshold_failures = (
    comparable_df[
        "exact_utility"
    ]
    <
    comparable_df[
        "tau"
    ]
).sum()

greedy_threshold_failures = (
    comparable_df[
        "greedy_utility"
    ]
    <
    comparable_df[
        "tau"
    ]
).sum()


# --------------------------------------------------
# Timing reconstruction
# --------------------------------------------------
#
# Subset precomputation time is repeated across the
# 16 condition rows belonging to one instance.
# Therefore, count it once per instance.
# --------------------------------------------------

instance_timing_df = (
    main_benchmark_df
    .groupby(
        "instance_id",
        as_index=False,
    )
    .agg(
        subset_precomputation_runtime_seconds=(
            "subset_precomputation_runtime_seconds",
            "first",
        ),
    )
)

total_precomputation_seconds = (
    instance_timing_df[
        "subset_precomputation_runtime_seconds"
    ].sum()
)

total_cached_exact_solve_seconds = (
    main_benchmark_df[
        "exact_cached_solve_runtime_seconds"
    ].sum()
)

total_greedy_seconds = (
    main_benchmark_df[
        "greedy_runtime_seconds"
    ].sum()
)

total_recorded_algorithm_seconds = (
    total_precomputation_seconds
    + total_cached_exact_solve_seconds
    + total_greedy_seconds
)


# --------------------------------------------------
# Print audit report
# --------------------------------------------------

print(
    "======================================"
)

print(
    "FINAL BENCHMARK INTEGRITY AUDIT"
)

print(
    "======================================"
)

print(
    "Total rows:",
    len(main_benchmark_df)
)

print(
    "Unique instances:",
    main_benchmark_df[
        "instance_id"
    ].nunique()
)

print(
    "Expected conditions per instance:",
    expected_conditions_per_instance
)

print(
    "Minimum rows per instance:",
    rows_per_instance.min()
)

print(
    "Maximum rows per instance:",
    rows_per_instance.max()
)

print(
    "Duplicate condition rows:",
    duplicate_condition_rows
)

print(
    "Comparable Exact–Greedy rows:",
    len(comparable_df)
)

print(
    "Missing comparison values:",
    missing_comparison_values
)

print(
    "Non-finite comparison values:",
    nonfinite_comparison_values
)

print(
    "Invalid Jaccard values:",
    invalid_jaccard_values
)

print(
    "Negative optimality gaps:",
    negative_gap_values
)

print(
    "Exact threshold failures:",
    exact_threshold_failures
)

print(
    "Greedy threshold failures:",
    greedy_threshold_failures
)


print(
    "\nExact status counts:"
)

print(
    exact_status_counts.to_string()
)


print(
    "\nGreedy status counts:"
)

print(
    greedy_status_counts.to_string()
)


print(
    "\nCoverage by size and regime:"
)

print(
    coverage_summary.to_string(
        index=False
    )
)


print(
    "\nRecorded computational timing:"
)

print(
    "Subset precomputation:",
    f"{total_precomputation_seconds / 60:.2f} minutes"
)

print(
    "Cached Exact solves:",
    f"{total_cached_exact_solve_seconds / 60:.2f} minutes"
)

print(
    "Greedy solves:",
    f"{total_greedy_seconds / 60:.2f} minutes"
)

print(
    "Total recorded algorithm time:",
    f"{total_recorded_algorithm_seconds / 60:.2f} minutes"
)

In [ ]:
# --------------------------------------------------
# Concise final benchmark integrity checks
# --------------------------------------------------

expected_rows = (
    len(EXACT_BENCHMARK_SIZES)
    * len(INSTANCE_REGIMES)
    * MAIN_REPLICATES
    * len(TAU_LEVELS)
    * len(WEIGHT_PROFILES)
)

expected_instances = (
    len(EXACT_BENCHMARK_SIZES)
    * len(INSTANCE_REGIMES)
    * MAIN_REPLICATES
)

key_columns = [
    "instance_id",
    "tau",
    "weight_profile",
]

rows_per_instance = (
    main_benchmark_df
    .groupby("instance_id")
    .size()
)

duplicate_rows = (
    main_benchmark_df
    .duplicated(
        subset=key_columns,
        keep=False,
    )
    .sum()
)

comparable_mask = (
    (
        main_benchmark_df["exact_status"]
        == "OPTIMAL"
    )
    &
    (
        main_benchmark_df["greedy_status"]
        == "FEASIBLE"
    )
)

comparable_df = (
    main_benchmark_df[
        comparable_mask
    ].copy()
)

metric_columns = [
    "exact_utility",
    "exact_privacy",
    "exact_normalized_cognitive_cost",
    "exact_disclosure_cost",
    "greedy_utility",
    "greedy_privacy",
    "greedy_normalized_cognitive_cost",
    "greedy_disclosure_cost",
    "optimality_gap",
    "optimality_gap_percent",
    "jaccard_similarity",
]

missing_values = (
    comparable_df[
        metric_columns
    ]
    .isna()
    .sum()
    .sum()
)

nonfinite_values = (
    ~np.isfinite(
        comparable_df[
            metric_columns
        ].to_numpy(
            dtype=float
        )
    )
).sum()

invalid_jaccard = (
    (
        comparable_df["jaccard_similarity"]
        < 0.0
    )
    |
    (
        comparable_df["jaccard_similarity"]
        > 1.0
    )
).sum()

negative_gaps = (
    comparable_df["optimality_gap"]
    < -1e-12
).sum()

exact_threshold_failures = (
    comparable_df["exact_utility"]
    <
    comparable_df["tau"]
).sum()

greedy_threshold_failures = (
    comparable_df["greedy_utility"]
    <
    comparable_df["tau"]
).sum()

coverage_df = (
    main_benchmark_df
    .groupby(
        [
            "n_items",
            "regime",
        ],
        as_index=False,
    )
    .agg(
        instances=(
            "instance_id",
            "nunique",
        ),
        rows=(
            "instance_id",
            "size",
        ),
        seeds=(
            "seed",
            "nunique",
        ),
    )
)

print("======================================")
print("FINAL BENCHMARK INTEGRITY SUMMARY")
print("======================================")

print("Total rows:", len(main_benchmark_df))
print("Expected rows:", expected_rows)

print(
    "Unique instances:",
    main_benchmark_df["instance_id"].nunique()
)

print(
    "Expected instances:",
    expected_instances
)

print(
    "Minimum rows per instance:",
    rows_per_instance.min()
)

print(
    "Maximum rows per instance:",
    rows_per_instance.max()
)

print(
    "Duplicate condition rows:",
    duplicate_rows
)

print(
    "Comparable Exact–Greedy rows:",
    len(comparable_df)
)

print(
    "Missing comparison values:",
    missing_values
)

print(
    "Non-finite comparison values:",
    nonfinite_values
)

print(
    "Invalid Jaccard values:",
    invalid_jaccard
)

print(
    "Negative optimality gaps:",
    negative_gaps
)

print(
    "Exact threshold failures:",
    exact_threshold_failures
)

print(
    "Greedy threshold failures:",
    greedy_threshold_failures
)

print("\nExact status counts:")
print(
    main_benchmark_df[
        "exact_status"
    ].value_counts(
        dropna=False
    ).to_string()
)

print("\nGreedy status counts:")
print(
    main_benchmark_df[
        "greedy_status"
    ].value_counts(
        dropna=False
    ).to_string()
)

print("\nCoverage by size and regime:")
print(
    coverage_df.to_string(
        index=False
    )
)

In [ ]:
# --------------------------------------------------
# Analyze infeasible Exact-vs-Greedy benchmark cases
# --------------------------------------------------

# Collapse the four weight profiles because feasibility
# depends only on the candidate set and tau, not on weights.
instance_tau_df = (
    main_benchmark_df
    .groupby(
        [
            "instance_id",
            "n_items",
            "regime",
            "tau",
        ],
        as_index=False,
    )
    .agg(
        full_set_utility=(
            "full_set_utility",
            "first",
        ),

        exact_status_count=(
            "exact_status",
            "nunique",
        ),

        greedy_status_count=(
            "greedy_status",
            "nunique",
        ),

        exact_status=(
            "exact_status",
            "first",
        ),

        greedy_status=(
            "greedy_status",
            "first",
        ),

        number_of_weight_profiles=(
            "weight_profile",
            "nunique",
        ),
    )
)


# --------------------------------------------------
# Integrity checks for feasibility status
# --------------------------------------------------

status_disagreements = (
    instance_tau_df[
        "exact_status"
    ]
    !=
    instance_tau_df[
        "greedy_status"
    ]
).sum()

weight_dependent_exact_status = (
    instance_tau_df[
        "exact_status_count"
    ]
    > 1
).sum()

weight_dependent_greedy_status = (
    instance_tau_df[
        "greedy_status_count"
    ]
    > 1
).sum()


# --------------------------------------------------
# Identify unique infeasible instance–tau cases
# --------------------------------------------------

infeasible_instance_tau_df = (
    instance_tau_df[
        instance_tau_df[
            "exact_status"
        ]
        == "INFEASIBLE"
    ].copy()
)


# Mathematical consistency:
# infeasibility should occur exactly when U(D) < tau.
incorrect_infeasibility_cases = (
    infeasible_instance_tau_df[
        "full_set_utility"
    ]
    >=
    infeasible_instance_tau_df[
        "tau"
    ]
).sum()


# --------------------------------------------------
# Summarize feasibility by size, regime, and tau
# --------------------------------------------------

feasibility_summary = (
    instance_tau_df
    .groupby(
        [
            "n_items",
            "regime",
            "tau",
        ],
        as_index=False,
    )
    .agg(
        total_instances=(
            "instance_id",
            "nunique",
        ),

        infeasible_instances=(
            "exact_status",
            lambda values:
                (values == "INFEASIBLE").sum(),
        ),

        mean_full_set_utility=(
            "full_set_utility",
            "mean",
        ),

        minimum_full_set_utility=(
            "full_set_utility",
            "min",
        ),
    )
)


feasibility_summary[
    "feasibility_rate_percent"
] = (
    100.0
    *
    (
        feasibility_summary[
            "total_instances"
        ]
        -
        feasibility_summary[
            "infeasible_instances"
        ]
    )
    /
    feasibility_summary[
        "total_instances"
    ]
)


infeasible_condition_summary = (
    feasibility_summary[
        feasibility_summary[
            "infeasible_instances"
        ]
        > 0
    ]
    .copy()
)


# --------------------------------------------------
# Print results
# --------------------------------------------------

print(
    "======================================"
)

print(
    "INFEASIBILITY ANALYSIS"
)

print(
    "======================================"
)

print(
    "Unique instance–tau cases:",
    len(instance_tau_df)
)

print(
    "Unique infeasible instance–tau cases:",
    len(infeasible_instance_tau_df)
)

print(
    "Corresponding benchmark rows:",
    len(infeasible_instance_tau_df)
    * len(WEIGHT_PROFILES)
)

print(
    "Exact–Greedy status disagreements:",
    status_disagreements
)

print(
    "Exact status changed across weights:",
    weight_dependent_exact_status
)

print(
    "Greedy status changed across weights:",
    weight_dependent_greedy_status
)

print(
    "Incorrect infeasibility cases:",
    incorrect_infeasibility_cases
)


print(
    "\nConditions containing infeasible instances:"
)

print(
    infeasible_condition_summary[
        [
            "n_items",
            "regime",
            "tau",
            "total_instances",
            "infeasible_instances",
            "feasibility_rate_percent",
            "mean_full_set_utility",
            "minimum_full_set_utility",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)

In [ ]:
# --------------------------------------------------
# Correct Exact-vs-Greedy feasibility comparison
# --------------------------------------------------

corrected_instance_tau_df = (
    instance_tau_df.copy()
)


# Convert method-specific status labels into the same
# Boolean feasibility representation.
corrected_instance_tau_df[
    "exact_is_feasible"
] = (
    corrected_instance_tau_df[
        "exact_status"
    ]
    == "OPTIMAL"
)

corrected_instance_tau_df[
    "greedy_is_feasible"
] = (
    corrected_instance_tau_df[
        "greedy_status"
    ]
    == "FEASIBLE"
)


# A genuine disagreement occurs only when one method
# identifies the instance as feasible and the other does not.
corrected_status_disagreements = (
    corrected_instance_tau_df[
        "exact_is_feasible"
    ]
    !=
    corrected_instance_tau_df[
        "greedy_is_feasible"
    ]
).sum()


# Check for unexpected status labels.
valid_exact_statuses = {
    "OPTIMAL",
    "INFEASIBLE",
}

valid_greedy_statuses = {
    "FEASIBLE",
    "INFEASIBLE",
}

unexpected_exact_statuses = set(
    corrected_instance_tau_df[
        "exact_status"
    ].unique()
) - valid_exact_statuses

unexpected_greedy_statuses = set(
    corrected_instance_tau_df[
        "greedy_status"
    ].unique()
) - valid_greedy_statuses


# Count feasible and infeasible instance–tau cases.
exact_feasible_cases = (
    corrected_instance_tau_df[
        "exact_is_feasible"
    ].sum()
)

greedy_feasible_cases = (
    corrected_instance_tau_df[
        "greedy_is_feasible"
    ].sum()
)

exact_infeasible_cases = (
    ~corrected_instance_tau_df[
        "exact_is_feasible"
    ]
).sum()

greedy_infeasible_cases = (
    ~corrected_instance_tau_df[
        "greedy_is_feasible"
    ]
).sum()


# Validate infeasibility against the full-set utility.
expected_feasibility = (
    corrected_instance_tau_df[
        "full_set_utility"
    ]
    >=
    corrected_instance_tau_df[
        "tau"
    ]
)

exact_feasibility_errors = (
    corrected_instance_tau_df[
        "exact_is_feasible"
    ]
    != expected_feasibility
).sum()

greedy_feasibility_errors = (
    corrected_instance_tau_df[
        "greedy_is_feasible"
    ]
    != expected_feasibility
).sum()


print("======================================")
print("CORRECTED FEASIBILITY CONSISTENCY CHECK")
print("======================================")

print(
    "Unique instance–tau cases:",
    len(corrected_instance_tau_df)
)

print(
    "Exact feasible cases:",
    exact_feasible_cases
)

print(
    "Greedy feasible cases:",
    greedy_feasible_cases
)

print(
    "Exact infeasible cases:",
    exact_infeasible_cases
)

print(
    "Greedy infeasible cases:",
    greedy_infeasible_cases
)

print(
    "Corrected Exact–Greedy disagreements:",
    corrected_status_disagreements
)

print(
    "Exact feasibility errors:",
    exact_feasibility_errors
)

print(
    "Greedy feasibility errors:",
    greedy_feasibility_errors
)

print(
    "Unexpected Exact statuses:",
    unexpected_exact_statuses
)

print(
    "Unexpected Greedy statuses:",
    unexpected_greedy_statuses
)

In [ ]:
# --------------------------------------------------
# Overall Exact-vs-Greedy solution-quality summary
# --------------------------------------------------

quality_df = (
    main_benchmark_df[
        (
            main_benchmark_df["exact_status"]
            == "OPTIMAL"
        )
        &
        (
            main_benchmark_df["greedy_status"]
            == "FEASIBLE"
        )
    ]
    .copy()
)


# Additional comparison quantities
quality_df[
    "selected_count_difference"
] = (
    quality_df[
        "greedy_selected_count"
    ]
    -
    quality_df[
        "exact_selected_count"
    ]
)

quality_df[
    "utility_difference"
] = (
    quality_df[
        "greedy_utility"
    ]
    -
    quality_df[
        "exact_utility"
    ]
)

quality_df[
    "privacy_difference"
] = (
    quality_df[
        "greedy_privacy"
    ]
    -
    quality_df[
        "exact_privacy"
    ]
)

quality_df[
    "cognitive_cost_difference"
] = (
    quality_df[
        "greedy_normalized_cognitive_cost"
    ]
    -
    quality_df[
        "exact_normalized_cognitive_cost"
    ]
)

quality_df[
    "absolute_disclosure_cost_difference"
] = (
    quality_df[
        "greedy_disclosure_cost"
    ]
    -
    quality_df[
        "exact_disclosure_cost"
    ]
)


# --------------------------------------------------
# Overall descriptive summary
# --------------------------------------------------

overall_quality_summary = pd.DataFrame(
    [
        {
            "comparable_rows":
                len(quality_df),

            "unique_instances":
                quality_df[
                    "instance_id"
                ].nunique(),

            "mean_optimality_gap_percent":
                quality_df[
                    "optimality_gap_percent"
                ].mean(),

            "median_optimality_gap_percent":
                quality_df[
                    "optimality_gap_percent"
                ].median(),

            "p90_optimality_gap_percent":
                quality_df[
                    "optimality_gap_percent"
                ].quantile(0.90),

            "p95_optimality_gap_percent":
                quality_df[
                    "optimality_gap_percent"
                ].quantile(0.95),

            "maximum_optimality_gap_percent":
                quality_df[
                    "optimality_gap_percent"
                ].max(),

            "exact_subset_match_percent":
                100.0
                * quality_df[
                    "same_selected_subset"
                ].mean(),

            "mean_jaccard_similarity":
                quality_df[
                    "jaccard_similarity"
                ].mean(),

            "mean_exact_selected_count":
                quality_df[
                    "exact_selected_count"
                ].mean(),

            "mean_greedy_selected_count":
                quality_df[
                    "greedy_selected_count"
                ].mean(),

            "mean_selected_count_difference":
                quality_df[
                    "selected_count_difference"
                ].mean(),

            "mean_utility_difference":
                quality_df[
                    "utility_difference"
                ].mean(),

            "mean_privacy_difference":
                quality_df[
                    "privacy_difference"
                ].mean(),

            "mean_cognitive_cost_difference":
                quality_df[
                    "cognitive_cost_difference"
                ].mean(),

            "mean_absolute_cost_difference":
                quality_df[
                    "absolute_disclosure_cost_difference"
                ].mean(),
        }
    ]
)


# --------------------------------------------------
# Greedy quality-threshold frequencies
# --------------------------------------------------

gap_threshold_summary = pd.DataFrame(
    [
        {
            "gap_threshold_percent":
                threshold,

            "rows_within_threshold":
                (
                    quality_df[
                        "optimality_gap_percent"
                    ]
                    <= threshold
                ).sum(),

            "percent_within_threshold":
                100.0
                * (
                    quality_df[
                        "optimality_gap_percent"
                    ]
                    <= threshold
                ).mean(),
        }

        for threshold in [
            0.0,
            1.0,
            5.0,
            10.0,
        ]
    ]
)


print(
    "======================================"
)

print(
    "OVERALL SOLUTION-QUALITY SUMMARY"
)

print(
    "======================================"
)

print(
    overall_quality_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)


print(
    "\nGREEDY OPTIMALITY-GAP THRESHOLDS"
)

print(
    gap_threshold_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)

In [ ]:
# --------------------------------------------------
# Numerical consistency audit for optimality gaps
# --------------------------------------------------

same_subset_df = (
    quality_df[
        quality_df[
            "same_selected_subset"
        ]
        == True
    ]
    .copy()
)

different_subset_df = (
    quality_df[
        quality_df[
            "same_selected_subset"
        ]
        == False
    ]
    .copy()
)


# --------------------------------------------------
# Absolute differences for identical selected subsets
# --------------------------------------------------

same_subset_df[
    "absolute_gap"
] = (
    same_subset_df[
        "optimality_gap"
    ].abs()
)

same_subset_df[
    "absolute_gap_percent"
] = (
    same_subset_df[
        "optimality_gap_percent"
    ].abs()
)

same_subset_df[
    "absolute_utility_difference"
] = (
    same_subset_df[
        "greedy_utility"
    ]
    -
    same_subset_df[
        "exact_utility"
    ]
).abs()

same_subset_df[
    "absolute_privacy_difference"
] = (
    same_subset_df[
        "greedy_privacy"
    ]
    -
    same_subset_df[
        "exact_privacy"
    ]
).abs()

same_subset_df[
    "absolute_cognitive_difference"
] = (
    same_subset_df[
        "greedy_normalized_cognitive_cost"
    ]
    -
    same_subset_df[
        "exact_normalized_cognitive_cost"
    ]
).abs()

same_subset_df[
    "absolute_disclosure_cost_difference"
] = (
    same_subset_df[
        "greedy_disclosure_cost"
    ]
    -
    same_subset_df[
        "exact_disclosure_cost"
    ]
).abs()


# --------------------------------------------------
# Tolerance-based optimal-cost matching
# --------------------------------------------------

GAP_TOLERANCES = [
    0.0,
    1e-12,
    1e-10,
    1e-8,
    1e-6,
]

tolerance_rows = []

for tolerance in GAP_TOLERANCES:

    tolerance_rows.append(
        {
            "gap_tolerance":
                tolerance,

            "all_rows_within_tolerance":
                (
                    quality_df[
                        "optimality_gap"
                    ].abs()
                    <= tolerance
                ).sum(),

            "all_rows_within_tolerance_percent":
                100.0
                * (
                    quality_df[
                        "optimality_gap"
                    ].abs()
                    <= tolerance
                ).mean(),

            "same_subset_rows_above_tolerance":
                (
                    same_subset_df[
                        "absolute_gap"
                    ]
                    > tolerance
                ).sum(),

            "different_subset_rows_within_tolerance":
                (
                    different_subset_df[
                        "optimality_gap"
                    ].abs()
                    <= tolerance
                ).sum(),
        }
    )

tolerance_audit_df = pd.DataFrame(
    tolerance_rows
)


# --------------------------------------------------
# Print numerical audit
# --------------------------------------------------

print(
    "======================================"
)

print(
    "NUMERICAL GAP CONSISTENCY AUDIT"
)

print(
    "======================================"
)

print(
    "Comparable rows:",
    len(quality_df)
)

print(
    "Identical selected-subset rows:",
    len(same_subset_df)
)

print(
    "Different selected-subset rows:",
    len(different_subset_df)
)


print(
    "\nMaximum differences when subsets are identical:"
)

print(
    "Utility:",
    f"{same_subset_df['absolute_utility_difference'].max():.16e}"
)

print(
    "Privacy:",
    f"{same_subset_df['absolute_privacy_difference'].max():.16e}"
)

print(
    "Normalized cognitive cost:",
    f"{same_subset_df['absolute_cognitive_difference'].max():.16e}"
)

print(
    "Disclosure cost:",
    f"{same_subset_df['absolute_disclosure_cost_difference'].max():.16e}"
)

print(
    "Optimality gap:",
    f"{same_subset_df['absolute_gap'].max():.16e}"
)

print(
    "Optimality gap percent:",
    f"{same_subset_df['absolute_gap_percent'].max():.16e}"
)


print(
    "\nTolerance-based audit:"
)

print(
    tolerance_audit_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.12e}",
    )
)

In [ ]:
# --------------------------------------------------
# Create numerically robust analysis-ready dataset
# --------------------------------------------------

OPTIMALITY_TOLERANCE = 1e-12

ANALYSIS_READY_PATH = (
    RESULTS_DIR
    / "exact_vs_greedy_main_benchmark_analysis_ready.csv"
)


analysis_benchmark_df = (
    main_benchmark_df.copy()
)


# --------------------------------------------------
# Identify comparable feasible conditions
# --------------------------------------------------

analysis_benchmark_df[
    "is_comparable"
] = (
    (
        analysis_benchmark_df[
            "exact_status"
        ]
        == "OPTIMAL"
    )
    &
    (
        analysis_benchmark_df[
            "greedy_status"
        ]
        == "FEASIBLE"
    )
)


# --------------------------------------------------
# Numerically robust optimal-cost indicator
# --------------------------------------------------

analysis_benchmark_df[
    "greedy_optimal_cost"
] = pd.Series(
    pd.NA,
    index=analysis_benchmark_df.index,
    dtype="boolean",
)

analysis_benchmark_df.loc[
    analysis_benchmark_df[
        "is_comparable"
    ],
    "greedy_optimal_cost",
] = (
    analysis_benchmark_df.loc[
        analysis_benchmark_df[
            "is_comparable"
        ],
        "optimality_gap",
    ].abs()
    <= OPTIMALITY_TOLERANCE
)


# --------------------------------------------------
# Derived comparison metrics
# --------------------------------------------------

comparable_mask = (
    analysis_benchmark_df[
        "is_comparable"
    ]
)


analysis_benchmark_df[
    "selected_count_difference"
] = np.nan

analysis_benchmark_df.loc[
    comparable_mask,
    "selected_count_difference",
] = (
    analysis_benchmark_df.loc[
        comparable_mask,
        "greedy_selected_count",
    ]
    -
    analysis_benchmark_df.loc[
        comparable_mask,
        "exact_selected_count",
    ]
)


analysis_benchmark_df[
    "utility_difference"
] = np.nan

analysis_benchmark_df.loc[
    comparable_mask,
    "utility_difference",
] = (
    analysis_benchmark_df.loc[
        comparable_mask,
        "greedy_utility",
    ]
    -
    analysis_benchmark_df.loc[
        comparable_mask,
        "exact_utility",
    ]
)


analysis_benchmark_df[
    "privacy_difference"
] = np.nan

analysis_benchmark_df.loc[
    comparable_mask,
    "privacy_difference",
] = (
    analysis_benchmark_df.loc[
        comparable_mask,
        "greedy_privacy",
    ]
    -
    analysis_benchmark_df.loc[
        comparable_mask,
        "exact_privacy",
    ]
)


analysis_benchmark_df[
    "cognitive_cost_difference"
] = np.nan

analysis_benchmark_df.loc[
    comparable_mask,
    "cognitive_cost_difference",
] = (
    analysis_benchmark_df.loc[
        comparable_mask,
        "greedy_normalized_cognitive_cost",
    ]
    -
    analysis_benchmark_df.loc[
        comparable_mask,
        "exact_normalized_cognitive_cost",
    ]
)


analysis_benchmark_df[
    "absolute_disclosure_cost_difference"
] = np.nan

analysis_benchmark_df.loc[
    comparable_mask,
    "absolute_disclosure_cost_difference",
] = (
    analysis_benchmark_df.loc[
        comparable_mask,
        "greedy_disclosure_cost",
    ]
    -
    analysis_benchmark_df.loc[
        comparable_mask,
        "exact_disclosure_cost",
    ]
)


# --------------------------------------------------
# Create interpretable optimality-gap bands
# --------------------------------------------------

analysis_benchmark_df[
    "optimality_gap_band"
] = "not_comparable"


analysis_benchmark_df.loc[
    comparable_mask
    &
    (
        analysis_benchmark_df[
            "optimality_gap"
        ].abs()
        <= OPTIMALITY_TOLERANCE
    ),
    "optimality_gap_band",
] = "optimal"


analysis_benchmark_df.loc[
    comparable_mask
    &
    (
        analysis_benchmark_df[
            "optimality_gap_percent"
        ]
        > 0.0
    )
    &
    (
        analysis_benchmark_df[
            "optimality_gap_percent"
        ]
        <= 1.0
    ),
    "optimality_gap_band",
] = "greater_than_0_to_1_percent"


analysis_benchmark_df.loc[
    comparable_mask
    &
    (
        analysis_benchmark_df[
            "optimality_gap_percent"
        ]
        > 1.0
    )
    &
    (
        analysis_benchmark_df[
            "optimality_gap_percent"
        ]
        <= 5.0
    ),
    "optimality_gap_band",
] = "greater_than_1_to_5_percent"


analysis_benchmark_df.loc[
    comparable_mask
    &
    (
        analysis_benchmark_df[
            "optimality_gap_percent"
        ]
        > 5.0
    )
    &
    (
        analysis_benchmark_df[
            "optimality_gap_percent"
        ]
        <= 10.0
    ),
    "optimality_gap_band",
] = "greater_than_5_to_10_percent"


analysis_benchmark_df.loc[
    comparable_mask
    &
    (
        analysis_benchmark_df[
            "optimality_gap_percent"
        ]
        > 10.0
    ),
    "optimality_gap_band",
] = "greater_than_10_percent"


# --------------------------------------------------
# Save analysis-ready dataset
# --------------------------------------------------

analysis_benchmark_df.to_csv(
    ANALYSIS_READY_PATH,
    index=False,
)


# --------------------------------------------------
# Validation summary
# --------------------------------------------------

comparable_analysis_df = (
    analysis_benchmark_df[
        analysis_benchmark_df[
            "is_comparable"
        ]
    ].copy()
)


optimal_cost_rows = (
    comparable_analysis_df[
        "greedy_optimal_cost"
    ].sum()
)

same_subset_rows = (
    comparable_analysis_df[
        "same_selected_subset"
    ]
    == True
).sum()

different_subset_optimal_rows = (
    (
        comparable_analysis_df[
            "greedy_optimal_cost"
        ]
        == True
    )
    &
    (
        comparable_analysis_df[
            "same_selected_subset"
        ]
        == False
    )
).sum()


gap_band_summary = (
    comparable_analysis_df[
        "optimality_gap_band"
    ]
    .value_counts()
    .reindex(
        [
            "optimal",
            "greater_than_0_to_1_percent",
            "greater_than_1_to_5_percent",
            "greater_than_5_to_10_percent",
            "greater_than_10_percent",
        ],
        fill_value=0,
    )
    .rename_axis(
        "optimality_gap_band"
    )
    .reset_index(
        name="number_of_rows"
    )
)

gap_band_summary[
    "percent_of_comparable_rows"
] = (
    100.0
    * gap_band_summary[
        "number_of_rows"
    ]
    / len(comparable_analysis_df)
)


print("======================================")
print("ANALYSIS-READY DATASET CREATED")
print("======================================")

print(
    "Total rows:",
    len(analysis_benchmark_df)
)

print(
    "Comparable rows:",
    len(comparable_analysis_df)
)

print(
    "Greedy optimal-cost rows:",
    int(optimal_cost_rows)
)

print(
    "Greedy optimal-cost percent:",
    f"{100.0 * optimal_cost_rows / len(comparable_analysis_df):.4f}"
)

print(
    "Identical-subset rows:",
    int(same_subset_rows)
)

print(
    "Different-subset optimal-cost rows:",
    int(different_subset_optimal_rows)
)

print(
    "\nOptimality-gap bands:"
)

print(
    gap_band_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)

print(
    "\nAnalysis-ready dataset saved to:"
)

print(
    ANALYSIS_READY_PATH
)

In [ ]:
# --------------------------------------------------
# Correct tolerance-aware optimality-gap bands
# --------------------------------------------------

OPTIMALITY_TOLERANCE = 1e-12

ANALYSIS_READY_PATH = (
    RESULTS_DIR
    / "exact_vs_greedy_main_benchmark_analysis_ready.csv"
)


# Start from the existing analysis-ready dataframe
corrected_analysis_df = (
    analysis_benchmark_df.copy()
)


# --------------------------------------------------
# Comparable feasible conditions
# --------------------------------------------------

comparable_mask = (
    corrected_analysis_df[
        "is_comparable"
    ]
    == True
)


# --------------------------------------------------
# Numerically robust optimal-cost indicator
# --------------------------------------------------

corrected_analysis_df[
    "greedy_optimal_cost"
] = pd.Series(
    pd.NA,
    index=corrected_analysis_df.index,
    dtype="boolean",
)

corrected_analysis_df.loc[
    comparable_mask,
    "greedy_optimal_cost",
] = (
    corrected_analysis_df.loc[
        comparable_mask,
        "optimality_gap",
    ].abs()
    <= OPTIMALITY_TOLERANCE
)


# --------------------------------------------------
# Initialize gap bands
# --------------------------------------------------

corrected_analysis_df[
    "optimality_gap_band"
] = "not_comparable"


optimal_mask = (
    comparable_mask
    &
    (
        corrected_analysis_df[
            "optimality_gap"
        ].abs()
        <= OPTIMALITY_TOLERANCE
    )
)


nonoptimal_mask = (
    comparable_mask
    &
    (
        corrected_analysis_df[
            "optimality_gap"
        ]
        > OPTIMALITY_TOLERANCE
    )
)


# --------------------------------------------------
# Apply mutually exclusive classifications
# --------------------------------------------------

corrected_analysis_df.loc[
    optimal_mask,
    "optimality_gap_band",
] = "optimal"


corrected_analysis_df.loc[
    nonoptimal_mask
    &
    (
        corrected_analysis_df[
            "optimality_gap_percent"
        ]
        <= 1.0
    ),
    "optimality_gap_band",
] = "greater_than_0_to_1_percent"


corrected_analysis_df.loc[
    nonoptimal_mask
    &
    (
        corrected_analysis_df[
            "optimality_gap_percent"
        ]
        > 1.0
    )
    &
    (
        corrected_analysis_df[
            "optimality_gap_percent"
        ]
        <= 5.0
    ),
    "optimality_gap_band",
] = "greater_than_1_to_5_percent"


corrected_analysis_df.loc[
    nonoptimal_mask
    &
    (
        corrected_analysis_df[
            "optimality_gap_percent"
        ]
        > 5.0
    )
    &
    (
        corrected_analysis_df[
            "optimality_gap_percent"
        ]
        <= 10.0
    ),
    "optimality_gap_band",
] = "greater_than_5_to_10_percent"


corrected_analysis_df.loc[
    nonoptimal_mask
    &
    (
        corrected_analysis_df[
            "optimality_gap_percent"
        ]
        > 10.0
    ),
    "optimality_gap_band",
] = "greater_than_10_percent"


# --------------------------------------------------
# Validation
# --------------------------------------------------

corrected_comparable_df = (
    corrected_analysis_df[
        comparable_mask
    ].copy()
)


optimal_cost_count = int(
    corrected_comparable_df[
        "greedy_optimal_cost"
    ].sum()
)


optimal_band_count = int(
    (
        corrected_comparable_df[
            "optimality_gap_band"
        ]
        == "optimal"
    ).sum()
)


unclassified_comparable_rows = int(
    (
        corrected_comparable_df[
            "optimality_gap_band"
        ]
        == "not_comparable"
    ).sum()
)


gap_band_order = [
    "optimal",
    "greater_than_0_to_1_percent",
    "greater_than_1_to_5_percent",
    "greater_than_5_to_10_percent",
    "greater_than_10_percent",
]


corrected_gap_band_summary = (
    corrected_comparable_df[
        "optimality_gap_band"
    ]
    .value_counts()
    .reindex(
        gap_band_order,
        fill_value=0,
    )
    .rename_axis(
        "optimality_gap_band"
    )
    .reset_index(
        name="number_of_rows"
    )
)


corrected_gap_band_summary[
    "percent_of_comparable_rows"
] = (
    100.0
    * corrected_gap_band_summary[
        "number_of_rows"
    ]
    / len(corrected_comparable_df)
)


# --------------------------------------------------
# Integrity assertions
# --------------------------------------------------

assert (
    optimal_cost_count
    == optimal_band_count
)

assert (
    unclassified_comparable_rows
    == 0
)

assert (
    corrected_gap_band_summary[
        "number_of_rows"
    ].sum()
    ==
    len(corrected_comparable_df)
)


# --------------------------------------------------
# Replace active analysis dataframe and save
# --------------------------------------------------

analysis_benchmark_df = (
    corrected_analysis_df
)

analysis_benchmark_df.to_csv(
    ANALYSIS_READY_PATH,
    index=False,
)


# --------------------------------------------------
# Print corrected results
# --------------------------------------------------

print(
    "======================================"
)

print(
    "CORRECTED OPTIMALITY-GAP CLASSIFICATION"
)

print(
    "======================================"
)

print(
    "Comparable rows:",
    len(corrected_comparable_df)
)

print(
    "Greedy optimal-cost rows:",
    optimal_cost_count
)

print(
    "Rows classified as optimal:",
    optimal_band_count
)

print(
    "Unclassified comparable rows:",
    unclassified_comparable_rows
)

print(
    "\nCorrected optimality-gap bands:"
)

print(
    corrected_gap_band_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)

print(
    "\nCorrected analysis-ready dataset saved to:"
)

print(
    ANALYSIS_READY_PATH
)

In [ ]:
# --------------------------------------------------
# Solution quality by structural benchmark regime
# --------------------------------------------------

regime_quality_df = (
    analysis_benchmark_df[
        analysis_benchmark_df[
            "is_comparable"
        ]
        == True
    ]
    .copy()
)


# Convert nullable Boolean indicator to numeric form
# for aggregation.
regime_quality_df[
    "greedy_optimal_cost_numeric"
] = (
    regime_quality_df[
        "greedy_optimal_cost"
    ]
    .astype(float)
)


# --------------------------------------------------
# Approximation-quality summary
# --------------------------------------------------

regime_quality_summary = (
    regime_quality_df
    .groupby(
        "regime",
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "count",
        ),

        unique_instances=(
            "instance_id",
            "nunique",
        ),

        mean_optimality_gap_percent=(
            "optimality_gap_percent",
            "mean",
        ),

        median_optimality_gap_percent=(
            "optimality_gap_percent",
            "median",
        ),

        p90_optimality_gap_percent=(
            "optimality_gap_percent",
            lambda values:
                values.quantile(0.90),
        ),

        p95_optimality_gap_percent=(
            "optimality_gap_percent",
            lambda values:
                values.quantile(0.95),
        ),

        maximum_optimality_gap_percent=(
            "optimality_gap_percent",
            "max",
        ),

        greedy_optimal_cost_rate=(
            "greedy_optimal_cost_numeric",
            "mean",
        ),

        mean_jaccard_similarity=(
            "jaccard_similarity",
            "mean",
        ),

        mean_exact_selected_count=(
            "exact_selected_count",
            "mean",
        ),

        mean_greedy_selected_count=(
            "greedy_selected_count",
            "mean",
        ),

        mean_selected_count_difference=(
            "selected_count_difference",
            "mean",
        ),

        mean_absolute_cost_difference=(
            "absolute_disclosure_cost_difference",
            "mean",
        ),
    )
)


regime_quality_summary[
    "greedy_optimal_cost_percent"
] = (
    100.0
    * regime_quality_summary[
        "greedy_optimal_cost_rate"
    ]
)


# --------------------------------------------------
# Feasibility by regime
# Use unique instance–tau cases rather than the four
# repeated weight-profile rows.
# --------------------------------------------------

regime_feasibility_summary = (
    corrected_instance_tau_df
    .groupby(
        "regime",
        as_index=False,
    )
    .agg(
        total_instance_tau_cases=(
            "instance_id",
            "size",
        ),

        feasible_instance_tau_cases=(
            "exact_is_feasible",
            "sum",
        ),
    )
)


regime_feasibility_summary[
    "feasibility_rate_percent"
] = (
    100.0
    * regime_feasibility_summary[
        "feasible_instance_tau_cases"
    ]
    / regime_feasibility_summary[
        "total_instance_tau_cases"
    ]
)


# --------------------------------------------------
# Merge feasibility and approximation results
# --------------------------------------------------

regime_summary = (
    regime_quality_summary
    .merge(
        regime_feasibility_summary,
        on="regime",
        how="left",
    )
)


display_columns = [
    "regime",
    "total_instance_tau_cases",
    "feasibility_rate_percent",
    "comparable_rows",
    "mean_optimality_gap_percent",
    "median_optimality_gap_percent",
    "p90_optimality_gap_percent",
    "p95_optimality_gap_percent",
    "maximum_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "mean_exact_selected_count",
    "mean_greedy_selected_count",
    "mean_selected_count_difference",
]


print(
    "======================================"
)

print(
    "SOLUTION QUALITY BY STRUCTURAL REGIME"
)

print(
    "======================================"
)

print(
    regime_summary[
        display_columns
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)

In [ ]:
# --------------------------------------------------
# Inspect rare large gaps in the alignment regime
# --------------------------------------------------

alignment_quality_df = (
    analysis_benchmark_df[
        (
            analysis_benchmark_df[
                "is_comparable"
            ]
            == True
        )
        &
        (
            analysis_benchmark_df[
                "regime"
            ]
            == "utility_burden_alignment"
        )
    ]
    .copy()
)


alignment_quality_df[
    "relative_gap_denominator"
] = (
    alignment_quality_df[
        "exact_disclosure_cost"
    ]
)


alignment_quality_df[
    "recomputed_absolute_cost_difference"
] = (
    alignment_quality_df[
        "greedy_disclosure_cost"
    ]
    -
    alignment_quality_df[
        "exact_disclosure_cost"
    ]
)


largest_alignment_gaps = (
    alignment_quality_df
    .sort_values(
        by="optimality_gap_percent",
        ascending=False,
    )
    .head(15)
    [
        [
            "instance_id",
            "seed",
            "n_items",
            "tau",
            "weight_profile",
            "lambda_privacy",
            "lambda_cognitive",
            "lambda_cardinality",
            "exact_selected_count",
            "greedy_selected_count",
            "exact_disclosure_cost",
            "greedy_disclosure_cost",
            "recomputed_absolute_cost_difference",
            "optimality_gap_percent",
            "jaccard_similarity",
            "exact_selected_items",
            "greedy_selected_items",
        ]
    ]
)


print(
    "======================================"
)

print(
    "LARGEST ALIGNMENT-REGIME GAPS"
)

print(
    "======================================"
)

print(
    largest_alignment_gaps.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)


print(
    "\nAlignment-regime gap distribution:"
)

print(
    alignment_quality_df[
        "optimality_gap_percent"
    ]
    .quantile(
        [
            0.50,
            0.90,
            0.95,
            0.99,
            0.995,
            1.00,
        ]
    )
    .to_string(
        float_format=lambda x: f"{x:.6f}"
    )
)

In [ ]:
# --------------------------------------------------
# Trace the largest alignment-regime outlier
# --------------------------------------------------

ALIGNMENT_OUTLIER_SEED = 13202090
ALIGNMENT_OUTLIER_N_ITEMS = 14
ALIGNMENT_OUTLIER_REGIME = (
    "utility_burden_alignment"
)
ALIGNMENT_OUTLIER_TAU = 0.80
ALIGNMENT_OUTLIER_WEIGHT_PROFILE = (
    "privacy_priority"
)

GREEDY_TRACE_EPSILON = 1e-12


# --------------------------------------------------
# Reconstruct the benchmark instance
# --------------------------------------------------

alignment_outlier_items = (
    generate_mst_instance(
        n_items=ALIGNMENT_OUTLIER_N_ITEMS,
        regime=ALIGNMENT_OUTLIER_REGIME,
        seed=ALIGNMENT_OUTLIER_SEED,
    )
)

alignment_outlier_weights = (
    WEIGHT_PROFILES[
        ALIGNMENT_OUTLIER_WEIGHT_PROFILE
    ]
)


# --------------------------------------------------
# Solve with official Exact and Greedy methods
# --------------------------------------------------

alignment_outlier_exact = (
    mst_core.exact_mst(
        items=alignment_outlier_items,
        tau=ALIGNMENT_OUTLIER_TAU,
        **alignment_outlier_weights,
    )
)

alignment_outlier_greedy = (
    mst_core.greedy_mst(
        items=alignment_outlier_items,
        tau=ALIGNMENT_OUTLIER_TAU,
        **alignment_outlier_weights,
    )
)


exact_item_set = set(
    alignment_outlier_exact[
        "selected_items"
    ]
)

greedy_item_set = set(
    alignment_outlier_greedy[
        "selected_items"
    ]
)


# --------------------------------------------------
# Print item-level data
# --------------------------------------------------

alignment_item_rows = []

for item in alignment_outlier_items:

    alignment_item_rows.append(
        {
            "item_id":
                item.item_id,

            "utility":
                item.utility,

            "privacy":
                item.privacy,

            "cognitive_cost":
                item.cognitive_cost,

            "selected_exact":
                item.item_id
                in exact_item_set,

            "selected_greedy":
                item.item_id
                in greedy_item_set,
        }
    )


alignment_item_df = pd.DataFrame(
    alignment_item_rows
)


print(
    "======================================"
)

print(
    "ALIGNMENT OUTLIER: ITEM-LEVEL DATA"
)

print(
    "======================================"
)

print(
    alignment_item_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)


# --------------------------------------------------
# Print final solutions
# --------------------------------------------------

print(
    "\nEXACT-MST SOLUTION"
)

print(
    "Selected:",
    alignment_outlier_exact[
        "selected_items"
    ]
)

print(
    "Count:",
    len(
        alignment_outlier_exact[
            "selected_items"
        ]
    )
)

print(
    "Utility:",
    f"{alignment_outlier_exact['utility']:.6f}"
)

print(
    "Privacy:",
    f"{alignment_outlier_exact['privacy']:.6f}"
)

print(
    "Normalized cognitive cost:",
    f"{alignment_outlier_exact['normalized_cognitive_cost']:.6f}"
)

print(
    "Disclosure cost:",
    f"{alignment_outlier_exact['disclosure_cost']:.6f}"
)


print(
    "\nGREEDY-MST SOLUTION"
)

print(
    "Selected:",
    alignment_outlier_greedy[
        "selected_items"
    ]
)

print(
    "Count:",
    len(
        alignment_outlier_greedy[
            "selected_items"
        ]
    )
)

print(
    "Utility:",
    f"{alignment_outlier_greedy['utility']:.6f}"
)

print(
    "Privacy:",
    f"{alignment_outlier_greedy['privacy']:.6f}"
)

print(
    "Normalized cognitive cost:",
    f"{alignment_outlier_greedy['normalized_cognitive_cost']:.6f}"
)

print(
    "Disclosure cost:",
    f"{alignment_outlier_greedy['disclosure_cost']:.6f}"
)


alignment_outlier_gap = (
    mst_core.compute_optimality_gap(
        exact_result=
            alignment_outlier_exact,

        approximate_result=
            alignment_outlier_greedy,
    )
)

print(
    "\nOptimality gap:",
    f"{100.0 * alignment_outlier_gap:.4f}%"
)


# --------------------------------------------------
# Trace every Greedy selection step
# --------------------------------------------------

selected_items = []
remaining_items = (
    alignment_outlier_items.copy()
)

alignment_trace_rows = []

step_number = 1


while True:

    current_metrics = (
        evaluate_subset_for_trace(
            selected_items=selected_items,
            all_items=alignment_outlier_items,
            weights=alignment_outlier_weights,
        )
    )

    if (
        current_metrics["utility"]
        >= ALIGNMENT_OUTLIER_TAU
    ):
        break


    step_candidate_rows = []


    for candidate in remaining_items:

        trial_subset = (
            selected_items
            + [candidate]
        )

        trial_metrics = (
            evaluate_subset_for_trace(
                selected_items=trial_subset,
                all_items=
                    alignment_outlier_items,
                weights=
                    alignment_outlier_weights,
            )
        )

        marginal_utility = (
            trial_metrics["utility"]
            -
            current_metrics["utility"]
        )

        marginal_cost = (
            trial_metrics[
                "disclosure_cost"
            ]
            -
            current_metrics[
                "disclosure_cost"
            ]
        )

        ratio = (
            marginal_utility
            /
            (
                marginal_cost
                + GREEDY_TRACE_EPSILON
            )
        )

        step_candidate_rows.append(
            {
                "step":
                    step_number,

                "candidate":
                    candidate.item_id,

                "current_utility":
                    current_metrics[
                        "utility"
                    ],

                "marginal_utility":
                    marginal_utility,

                "marginal_cost":
                    marginal_cost,

                "ratio":
                    ratio,

                "resulting_utility":
                    trial_metrics[
                        "utility"
                    ],

                "reaches_threshold":
                    trial_metrics[
                        "utility"
                    ]
                    >= ALIGNMENT_OUTLIER_TAU,

                "part_of_exact_solution":
                    candidate.item_id
                    in exact_item_set,
            }
        )


    step_candidate_df = (
        pd.DataFrame(
            step_candidate_rows
        )
        .sort_values(
            by="ratio",
            ascending=False,
        )
        .reset_index(
            drop=True
        )
    )


    chosen_item_id = (
        step_candidate_df
        .iloc[0][
            "candidate"
        ]
    )


    step_candidate_df[
        "chosen"
    ] = (
        step_candidate_df[
            "candidate"
        ]
        == chosen_item_id
    )


    alignment_trace_rows.extend(
        step_candidate_df.to_dict(
            "records"
        )
    )


    chosen_item = next(
        item
        for item in remaining_items
        if item.item_id
        == chosen_item_id
    )

    selected_items.append(
        chosen_item
    )

    remaining_items.remove(
        chosen_item
    )

    step_number += 1


alignment_trace_df = pd.DataFrame(
    alignment_trace_rows
)


# --------------------------------------------------
# Display each Greedy step
# --------------------------------------------------

for current_step in sorted(
    alignment_trace_df[
        "step"
    ].unique()
):

    print(
        f"\nGREEDY STEP {current_step}"
    )

    current_step_df = (
        alignment_trace_df[
            alignment_trace_df[
                "step"
            ]
            == current_step
        ]
    )

    print(
        current_step_df[
            [
                "candidate",
                "marginal_utility",
                "marginal_cost",
                "ratio",
                "resulting_utility",
                "reaches_threshold",
                "part_of_exact_solution",
                "chosen",
            ]
        ].to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}",
        )
    )


print(
    "\nFinal traced Greedy selection:",
    tuple(
        item.item_id
        for item in selected_items
    )
)

print(
    "Official Greedy selection:",
    alignment_outlier_greedy[
        "selected_items"
    ]
)

print(
    "Trace matches official Greedy:",
    tuple(
        item.item_id
        for item in selected_items
    )
    ==
    alignment_outlier_greedy[
        "selected_items"
    ]
)

In [ ]:
# --------------------------------------------------
# Relationship between selection-count differences
# and Greedy-MST optimality gaps
# --------------------------------------------------

count_effect_df = (
    analysis_benchmark_df[
        analysis_benchmark_df[
            "is_comparable"
        ]
        == True
    ]
    .copy()
)


# Ensure the count difference is represented as an integer.
count_effect_df[
    "selected_count_difference"
] = (
    count_effect_df[
        "selected_count_difference"
    ]
    .astype(int)
)


# --------------------------------------------------
# Summary by Greedy-minus-Exact item count
# --------------------------------------------------

count_difference_summary = (
    count_effect_df
    .groupby(
        "selected_count_difference",
        as_index=False,
    )
    .agg(
        number_of_rows=(
            "optimality_gap_percent",
            "count",
        ),

        unique_instances=(
            "instance_id",
            "nunique",
        ),

        mean_optimality_gap_percent=(
            "optimality_gap_percent",
            "mean",
        ),

        median_optimality_gap_percent=(
            "optimality_gap_percent",
            "median",
        ),

        p90_optimality_gap_percent=(
            "optimality_gap_percent",
            lambda values:
                values.quantile(0.90),
        ),

        maximum_optimality_gap_percent=(
            "optimality_gap_percent",
            "max",
        ),

        mean_jaccard_similarity=(
            "jaccard_similarity",
            "mean",
        ),

        mean_absolute_cost_difference=(
            "absolute_disclosure_cost_difference",
            "mean",
        ),

        mean_utility_difference=(
            "utility_difference",
            "mean",
        ),

        mean_privacy_difference=(
            "privacy_difference",
            "mean",
        ),

        mean_cognitive_cost_difference=(
            "cognitive_cost_difference",
            "mean",
        ),
    )
)


count_difference_summary[
    "percent_of_comparable_rows"
] = (
    100.0
    * count_difference_summary[
        "number_of_rows"
    ]
    / len(count_effect_df)
)


# --------------------------------------------------
# Overall count-based frequencies
# --------------------------------------------------

same_count_rows = (
    count_effect_df[
        "selected_count_difference"
    ]
    == 0
).sum()

greedy_more_rows = (
    count_effect_df[
        "selected_count_difference"
    ]
    > 0
).sum()

greedy_fewer_rows = (
    count_effect_df[
        "selected_count_difference"
    ]
    < 0
).sum()


greedy_one_extra_rows = (
    count_effect_df[
        "selected_count_difference"
    ]
    == 1
).sum()

greedy_two_or_more_extra_rows = (
    count_effect_df[
        "selected_count_difference"
    ]
    >= 2
).sum()


# --------------------------------------------------
# Large-gap cases associated with extra disclosures
# --------------------------------------------------

large_gap_mask = (
    count_effect_df[
        "optimality_gap_percent"
    ]
    > 10.0
)

large_gap_rows = (
    large_gap_mask.sum()
)

large_gap_with_extra_items = (
    large_gap_mask
    &
    (
        count_effect_df[
            "selected_count_difference"
        ]
        > 0
    )
).sum()


print(
    "======================================"
)

print(
    "SELECTION-COUNT EFFECT"
)

print(
    "======================================"
)

print(
    "Comparable rows:",
    len(count_effect_df)
)

print(
    "Same number of selected items:",
    same_count_rows,
    f"({100.0 * same_count_rows / len(count_effect_df):.4f}%)"
)

print(
    "Greedy selected more items:",
    greedy_more_rows,
    f"({100.0 * greedy_more_rows / len(count_effect_df):.4f}%)"
)

print(
    "Greedy selected fewer items:",
    greedy_fewer_rows,
    f"({100.0 * greedy_fewer_rows / len(count_effect_df):.4f}%)"
)

print(
    "Greedy selected exactly one extra item:",
    greedy_one_extra_rows,
    f"({100.0 * greedy_one_extra_rows / len(count_effect_df):.4f}%)"
)

print(
    "Greedy selected two or more extra items:",
    greedy_two_or_more_extra_rows,
    f"({100.0 * greedy_two_or_more_extra_rows / len(count_effect_df):.4f}%)"
)

print(
    "Rows with gap greater than 10%:",
    large_gap_rows
)

print(
    "Large-gap rows with extra Greedy items:",
    large_gap_with_extra_items,
    (
        f"({100.0 * large_gap_with_extra_items / large_gap_rows:.4f}%"
        if large_gap_rows > 0
        else "(not applicable"
    )
    + ")"
)


print(
    "\nSummary by Greedy-minus-Exact selected-item count:"
)

print(
    count_difference_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)

In [ ]:
# --------------------------------------------------
# Inspect cases where Greedy selects fewer items
# than the Exact-MST optimum
# --------------------------------------------------

greedy_fewer_cases = (
    count_effect_df[
        count_effect_df[
            "selected_count_difference"
        ]
        < 0
    ]
    .copy()
)


# --------------------------------------------------
# Objective-component contributions
# --------------------------------------------------

greedy_fewer_cases[
    "exact_privacy_contribution"
] = (
    greedy_fewer_cases[
        "lambda_privacy"
    ]
    *
    greedy_fewer_cases[
        "exact_privacy"
    ]
)

greedy_fewer_cases[
    "exact_cognitive_contribution"
] = (
    greedy_fewer_cases[
        "lambda_cognitive"
    ]
    *
    greedy_fewer_cases[
        "exact_normalized_cognitive_cost"
    ]
)

greedy_fewer_cases[
    "exact_cardinality_contribution"
] = (
    greedy_fewer_cases[
        "lambda_cardinality"
    ]
    *
    (
        greedy_fewer_cases[
            "exact_selected_count"
        ]
        /
        greedy_fewer_cases[
            "n_items"
        ]
    )
)


greedy_fewer_cases[
    "greedy_privacy_contribution"
] = (
    greedy_fewer_cases[
        "lambda_privacy"
    ]
    *
    greedy_fewer_cases[
        "greedy_privacy"
    ]
)

greedy_fewer_cases[
    "greedy_cognitive_contribution"
] = (
    greedy_fewer_cases[
        "lambda_cognitive"
    ]
    *
    greedy_fewer_cases[
        "greedy_normalized_cognitive_cost"
    ]
)

greedy_fewer_cases[
    "greedy_cardinality_contribution"
] = (
    greedy_fewer_cases[
        "lambda_cardinality"
    ]
    *
    (
        greedy_fewer_cases[
            "greedy_selected_count"
        ]
        /
        greedy_fewer_cases[
            "n_items"
        ]
    )
)


# --------------------------------------------------
# Verify that component sums reproduce J(S)
# --------------------------------------------------

greedy_fewer_cases[
    "exact_recomputed_cost"
] = (
    greedy_fewer_cases[
        "exact_privacy_contribution"
    ]
    +
    greedy_fewer_cases[
        "exact_cognitive_contribution"
    ]
    +
    greedy_fewer_cases[
        "exact_cardinality_contribution"
    ]
)

greedy_fewer_cases[
    "greedy_recomputed_cost"
] = (
    greedy_fewer_cases[
        "greedy_privacy_contribution"
    ]
    +
    greedy_fewer_cases[
        "greedy_cognitive_contribution"
    ]
    +
    greedy_fewer_cases[
        "greedy_cardinality_contribution"
    ]
)


# --------------------------------------------------
# Display relevant details
# --------------------------------------------------

display_columns = [
    "instance_id",
    "seed",
    "n_items",
    "regime",
    "tau",
    "weight_profile",

    "exact_selected_count",
    "greedy_selected_count",

    "exact_selected_items",
    "greedy_selected_items",

    "exact_utility",
    "greedy_utility",

    "exact_privacy_contribution",
    "greedy_privacy_contribution",

    "exact_cognitive_contribution",
    "greedy_cognitive_contribution",

    "exact_cardinality_contribution",
    "greedy_cardinality_contribution",

    "exact_disclosure_cost",
    "greedy_disclosure_cost",

    "optimality_gap_percent",
    "jaccard_similarity",
]


print(
    "======================================"
)

print(
    "GREEDY FEWER-ITEM CASES"
)

print(
    "======================================"
)

print(
    "Number of cases:",
    len(greedy_fewer_cases)
)

print(
    greedy_fewer_cases[
        display_columns
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)


print(
    "\nMaximum Exact cost-reconstruction error:",
    f"{(
        greedy_fewer_cases['exact_recomputed_cost']
        -
        greedy_fewer_cases['exact_disclosure_cost']
    ).abs().max():.16e}"
)

print(
    "Maximum Greedy cost-reconstruction error:",
    f"{(
        greedy_fewer_cases['greedy_recomputed_cost']
        -
        greedy_fewer_cases['greedy_disclosure_cost']
    ).abs().max():.16e}"
)

In [ ]:
# --------------------------------------------------
# Decompose Greedy-MST suboptimality mechanisms
# --------------------------------------------------

mechanism_df = (
    analysis_benchmark_df[
        analysis_benchmark_df[
            "is_comparable"
        ]
        == True
    ]
    .copy()
)


# --------------------------------------------------
# Classify each Exact-vs-Greedy comparison
# --------------------------------------------------

mechanism_df[
    "solution_mechanism"
] = np.select(
    condlist=[
        mechanism_df[
            "same_selected_subset"
        ]
        == True,

        (
            mechanism_df[
                "same_selected_subset"
            ]
            == False
        )
        &
        (
            mechanism_df[
                "selected_count_difference"
            ]
            == 0
        ),

        mechanism_df[
            "selected_count_difference"
        ]
        > 0,

        mechanism_df[
            "selected_count_difference"
        ]
        < 0,
    ],

    choicelist=[
        "identical_subset",
        "same_count_different_items",
        "greedy_more_items",
        "greedy_fewer_items",
    ],

    default="unclassified",
)


# --------------------------------------------------
# Decompose the objective difference
# --------------------------------------------------

mechanism_df[
    "privacy_component_difference"
] = (
    mechanism_df[
        "lambda_privacy"
    ]
    *
    (
        mechanism_df[
            "greedy_privacy"
        ]
        -
        mechanism_df[
            "exact_privacy"
        ]
    )
)


mechanism_df[
    "cognitive_component_difference"
] = (
    mechanism_df[
        "lambda_cognitive"
    ]
    *
    (
        mechanism_df[
            "greedy_normalized_cognitive_cost"
        ]
        -
        mechanism_df[
            "exact_normalized_cognitive_cost"
        ]
    )
)


mechanism_df[
    "cardinality_component_difference"
] = (
    mechanism_df[
        "lambda_cardinality"
    ]
    *
    (
        mechanism_df[
            "greedy_selected_count"
        ]
        -
        mechanism_df[
            "exact_selected_count"
        ]
    )
    /
    mechanism_df[
        "n_items"
    ]
)


mechanism_df[
    "reconstructed_cost_difference"
] = (
    mechanism_df[
        "privacy_component_difference"
    ]
    +
    mechanism_df[
        "cognitive_component_difference"
    ]
    +
    mechanism_df[
        "cardinality_component_difference"
    ]
)


mechanism_df[
    "large_gap"
] = (
    mechanism_df[
        "optimality_gap_percent"
    ]
    > 10.0
).astype(int)


# --------------------------------------------------
# Validate objective decomposition
# --------------------------------------------------

maximum_decomposition_error = (
    mechanism_df[
        "reconstructed_cost_difference"
    ]
    -
    mechanism_df[
        "absolute_disclosure_cost_difference"
    ]
).abs().max()


# --------------------------------------------------
# Aggregate by solution mechanism
# --------------------------------------------------

mechanism_summary = (
    mechanism_df
    .groupby(
        "solution_mechanism",
        as_index=False,
    )
    .agg(
        number_of_rows=(
            "optimality_gap_percent",
            "count",
        ),

        unique_instances=(
            "instance_id",
            "nunique",
        ),

        mean_optimality_gap_percent=(
            "optimality_gap_percent",
            "mean",
        ),

        median_optimality_gap_percent=(
            "optimality_gap_percent",
            "median",
        ),

        p90_optimality_gap_percent=(
            "optimality_gap_percent",
            lambda values:
                values.quantile(0.90),
        ),

        maximum_optimality_gap_percent=(
            "optimality_gap_percent",
            "max",
        ),

        large_gap_rate=(
            "large_gap",
            "mean",
        ),

        mean_jaccard_similarity=(
            "jaccard_similarity",
            "mean",
        ),

        mean_utility_difference=(
            "utility_difference",
            "mean",
        ),

        mean_privacy_component_difference=(
            "privacy_component_difference",
            "mean",
        ),

        mean_cognitive_component_difference=(
            "cognitive_component_difference",
            "mean",
        ),

        mean_cardinality_component_difference=(
            "cardinality_component_difference",
            "mean",
        ),

        mean_total_cost_difference=(
            "absolute_disclosure_cost_difference",
            "mean",
        ),
    )
)


mechanism_summary[
    "percent_of_comparable_rows"
] = (
    100.0
    * mechanism_summary[
        "number_of_rows"
    ]
    / len(mechanism_df)
)


mechanism_summary[
    "large_gap_percent"
] = (
    100.0
    * mechanism_summary[
        "large_gap_rate"
    ]
)


# --------------------------------------------------
# Organize mechanism order
# --------------------------------------------------

mechanism_order = [
    "identical_subset",
    "same_count_different_items",
    "greedy_more_items",
    "greedy_fewer_items",
]


mechanism_summary[
    "solution_mechanism"
] = pd.Categorical(
    mechanism_summary[
        "solution_mechanism"
    ],
    categories=mechanism_order,
    ordered=True,
)


mechanism_summary = (
    mechanism_summary
    .sort_values(
        "solution_mechanism"
    )
    .reset_index(
        drop=True
    )
)


display_columns = [
    "solution_mechanism",
    "number_of_rows",
    "percent_of_comparable_rows",
    "unique_instances",
    "mean_optimality_gap_percent",
    "median_optimality_gap_percent",
    "p90_optimality_gap_percent",
    "maximum_optimality_gap_percent",
    "large_gap_percent",
    "mean_jaccard_similarity",
    "mean_privacy_component_difference",
    "mean_cognitive_component_difference",
    "mean_cardinality_component_difference",
    "mean_total_cost_difference",
]


print(
    "======================================"
)

print(
    "GREEDY SUBOPTIMALITY MECHANISMS"
)

print(
    "======================================"
)

print(
    "Comparable rows:",
    len(mechanism_df)
)

print(
    "Unclassified rows:",
    (
        mechanism_df[
            "solution_mechanism"
        ]
        == "unclassified"
    ).sum()
)

print(
    "Maximum objective-decomposition error:",
    f"{maximum_decomposition_error:.16e}"
)


print(
    "\nMechanism summary:"
)

print(
    mechanism_summary[
        display_columns
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)

In [ ]:
# --------------------------------------------------
# Exact-vs-Greedy quality by sufficiency threshold
# --------------------------------------------------

threshold_quality_df = (
    analysis_benchmark_df[
        analysis_benchmark_df[
            "is_comparable"
        ]
        == True
    ]
    .copy()
)


threshold_quality_df[
    "greedy_optimal_cost_numeric"
] = (
    threshold_quality_df[
        "greedy_optimal_cost"
    ]
    .astype(float)
)


threshold_quality_df[
    "large_gap"
] = (
    threshold_quality_df[
        "optimality_gap_percent"
    ]
    > 10.0
).astype(int)


threshold_quality_df[
    "greedy_more_items"
] = (
    threshold_quality_df[
        "selected_count_difference"
    ]
    > 0
).astype(int)


# --------------------------------------------------
# Approximation-quality summary by tau
# --------------------------------------------------

threshold_quality_summary = (
    threshold_quality_df
    .groupby(
        "tau",
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "count",
        ),

        unique_instances=(
            "instance_id",
            "nunique",
        ),

        mean_optimality_gap_percent=(
            "optimality_gap_percent",
            "mean",
        ),

        median_optimality_gap_percent=(
            "optimality_gap_percent",
            "median",
        ),

        p90_optimality_gap_percent=(
            "optimality_gap_percent",
            lambda values:
                values.quantile(0.90),
        ),

        p95_optimality_gap_percent=(
            "optimality_gap_percent",
            lambda values:
                values.quantile(0.95),
        ),

        maximum_optimality_gap_percent=(
            "optimality_gap_percent",
            "max",
        ),

        greedy_optimal_cost_rate=(
            "greedy_optimal_cost_numeric",
            "mean",
        ),

        large_gap_rate=(
            "large_gap",
            "mean",
        ),

        greedy_more_items_rate=(
            "greedy_more_items",
            "mean",
        ),

        mean_jaccard_similarity=(
            "jaccard_similarity",
            "mean",
        ),

        mean_exact_selected_count=(
            "exact_selected_count",
            "mean",
        ),

        mean_greedy_selected_count=(
            "greedy_selected_count",
            "mean",
        ),

        mean_selected_count_difference=(
            "selected_count_difference",
            "mean",
        ),

        mean_exact_utility=(
            "exact_utility",
            "mean",
        ),

        mean_greedy_utility=(
            "greedy_utility",
            "mean",
        ),

        mean_utility_difference=(
            "utility_difference",
            "mean",
        ),

        mean_absolute_cost_difference=(
            "absolute_disclosure_cost_difference",
            "mean",
        ),
    )
)


threshold_quality_summary[
    "greedy_optimal_cost_percent"
] = (
    100.0
    * threshold_quality_summary[
        "greedy_optimal_cost_rate"
    ]
)


threshold_quality_summary[
    "large_gap_percent"
] = (
    100.0
    * threshold_quality_summary[
        "large_gap_rate"
    ]
)


threshold_quality_summary[
    "greedy_more_items_percent"
] = (
    100.0
    * threshold_quality_summary[
        "greedy_more_items_rate"
    ]
)


# --------------------------------------------------
# Feasibility summary by tau
#
# Use one row per instance–tau case because the same
# feasibility outcome is repeated across weight profiles.
# --------------------------------------------------

threshold_feasibility_summary = (
    corrected_instance_tau_df
    .groupby(
        "tau",
        as_index=False,
    )
    .agg(
        total_instance_tau_cases=(
            "instance_id",
            "size",
        ),

        feasible_instance_tau_cases=(
            "exact_is_feasible",
            "sum",
        ),

        mean_full_set_utility=(
            "full_set_utility",
            "mean",
        ),

        minimum_full_set_utility=(
            "full_set_utility",
            "min",
        ),
    )
)


threshold_feasibility_summary[
    "infeasible_instance_tau_cases"
] = (
    threshold_feasibility_summary[
        "total_instance_tau_cases"
    ]
    -
    threshold_feasibility_summary[
        "feasible_instance_tau_cases"
    ]
)


threshold_feasibility_summary[
    "feasibility_rate_percent"
] = (
    100.0
    * threshold_feasibility_summary[
        "feasible_instance_tau_cases"
    ]
    / threshold_feasibility_summary[
        "total_instance_tau_cases"
    ]
)


# --------------------------------------------------
# Merge feasibility and solution-quality summaries
# --------------------------------------------------

threshold_summary = (
    threshold_quality_summary
    .merge(
        threshold_feasibility_summary,
        on="tau",
        how="left",
    )
)


display_columns = [
    "tau",
    "total_instance_tau_cases",
    "feasible_instance_tau_cases",
    "infeasible_instance_tau_cases",
    "feasibility_rate_percent",
    "comparable_rows",
    "mean_optimality_gap_percent",
    "median_optimality_gap_percent",
    "p90_optimality_gap_percent",
    "p95_optimality_gap_percent",
    "maximum_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "large_gap_percent",
    "greedy_more_items_percent",
    "mean_jaccard_similarity",
    "mean_exact_selected_count",
    "mean_greedy_selected_count",
    "mean_selected_count_difference",
    "mean_utility_difference",
]


print(
    "======================================"
)

print(
    "SOLUTION QUALITY BY SUFFICIENCY THRESHOLD"
)

print(
    "======================================"
)

print(
    threshold_summary[
        display_columns
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)

In [ ]:
# --------------------------------------------------
# Exact-vs-Greedy quality by objective-weight profile
# --------------------------------------------------

weight_quality_df = (
    analysis_benchmark_df[
        analysis_benchmark_df[
            "is_comparable"
        ]
        == True
    ]
    .copy()
)


weight_quality_df[
    "greedy_optimal_cost_numeric"
] = (
    weight_quality_df[
        "greedy_optimal_cost"
    ]
    .astype(float)
)


weight_quality_df[
    "large_gap"
] = (
    weight_quality_df[
        "optimality_gap_percent"
    ]
    > 10.0
).astype(int)


weight_quality_df[
    "greedy_more_items"
] = (
    weight_quality_df[
        "selected_count_difference"
    ]
    > 0
).astype(int)


# --------------------------------------------------
# Summary by weight profile
# --------------------------------------------------

weight_profile_summary = (
    weight_quality_df
    .groupby(
        [
            "weight_profile",
            "lambda_privacy",
            "lambda_cognitive",
            "lambda_cardinality",
        ],
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "count",
        ),

        unique_instances=(
            "instance_id",
            "nunique",
        ),

        mean_optimality_gap_percent=(
            "optimality_gap_percent",
            "mean",
        ),

        median_optimality_gap_percent=(
            "optimality_gap_percent",
            "median",
        ),

        p90_optimality_gap_percent=(
            "optimality_gap_percent",
            lambda values:
                values.quantile(0.90),
        ),

        p95_optimality_gap_percent=(
            "optimality_gap_percent",
            lambda values:
                values.quantile(0.95),
        ),

        maximum_optimality_gap_percent=(
            "optimality_gap_percent",
            "max",
        ),

        greedy_optimal_cost_rate=(
            "greedy_optimal_cost_numeric",
            "mean",
        ),

        large_gap_rate=(
            "large_gap",
            "mean",
        ),

        greedy_more_items_rate=(
            "greedy_more_items",
            "mean",
        ),

        mean_jaccard_similarity=(
            "jaccard_similarity",
            "mean",
        ),

        mean_exact_selected_count=(
            "exact_selected_count",
            "mean",
        ),

        mean_greedy_selected_count=(
            "greedy_selected_count",
            "mean",
        ),

        mean_selected_count_difference=(
            "selected_count_difference",
            "mean",
        ),

        mean_utility_difference=(
            "utility_difference",
            "mean",
        ),

        mean_privacy_difference=(
            "privacy_difference",
            "mean",
        ),

        mean_cognitive_cost_difference=(
            "cognitive_cost_difference",
            "mean",
        ),

        mean_absolute_cost_difference=(
            "absolute_disclosure_cost_difference",
            "mean",
        ),
    )
)


# --------------------------------------------------
# Convert rates to percentages
# --------------------------------------------------

weight_profile_summary[
    "greedy_optimal_cost_percent"
] = (
    100.0
    * weight_profile_summary[
        "greedy_optimal_cost_rate"
    ]
)


weight_profile_summary[
    "large_gap_percent"
] = (
    100.0
    * weight_profile_summary[
        "large_gap_rate"
    ]
)


weight_profile_summary[
    "greedy_more_items_percent"
] = (
    100.0
    * weight_profile_summary[
        "greedy_more_items_rate"
    ]
)


# --------------------------------------------------
# Organize profile order
# --------------------------------------------------

weight_profile_order = [
    "balanced",
    "privacy_priority",
    "interpretation_priority",
    "parsimony_priority",
]


weight_profile_summary[
    "weight_profile"
] = pd.Categorical(
    weight_profile_summary[
        "weight_profile"
    ],
    categories=weight_profile_order,
    ordered=True,
)


weight_profile_summary = (
    weight_profile_summary
    .sort_values(
        "weight_profile"
    )
    .reset_index(
        drop=True
    )
)


display_columns = [
    "weight_profile",
    "lambda_privacy",
    "lambda_cognitive",
    "lambda_cardinality",
    "comparable_rows",
    "mean_optimality_gap_percent",
    "median_optimality_gap_percent",
    "p90_optimality_gap_percent",
    "p95_optimality_gap_percent",
    "maximum_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "large_gap_percent",
    "greedy_more_items_percent",
    "mean_jaccard_similarity",
    "mean_exact_selected_count",
    "mean_greedy_selected_count",
    "mean_selected_count_difference",
    "mean_privacy_difference",
    "mean_cognitive_cost_difference",
    "mean_absolute_cost_difference",
]


print(
    "======================================"
)

print(
    "SOLUTION QUALITY BY WEIGHT PROFILE"
)

print(
    "======================================"
)

print(
    weight_profile_summary[
        display_columns
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)

In [ ]:
# --------------------------------------------------
# Exact-vs-Greedy quality by candidate-set size
# --------------------------------------------------

size_quality_df = (
    analysis_benchmark_df[
        analysis_benchmark_df[
            "is_comparable"
        ]
        == True
    ]
    .copy()
)


size_quality_df[
    "greedy_optimal_cost_numeric"
] = (
    size_quality_df[
        "greedy_optimal_cost"
    ]
    .astype(float)
)


size_quality_df[
    "large_gap"
] = (
    size_quality_df[
        "optimality_gap_percent"
    ]
    > 10.0
).astype(int)


size_quality_df[
    "greedy_more_items"
] = (
    size_quality_df[
        "selected_count_difference"
    ]
    > 0
).astype(int)


# --------------------------------------------------
# Approximation quality by candidate-set size
# --------------------------------------------------

size_quality_summary = (
    size_quality_df
    .groupby(
        "n_items",
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "count",
        ),

        unique_instances=(
            "instance_id",
            "nunique",
        ),

        mean_optimality_gap_percent=(
            "optimality_gap_percent",
            "mean",
        ),

        median_optimality_gap_percent=(
            "optimality_gap_percent",
            "median",
        ),

        p90_optimality_gap_percent=(
            "optimality_gap_percent",
            lambda values:
                values.quantile(0.90),
        ),

        p95_optimality_gap_percent=(
            "optimality_gap_percent",
            lambda values:
                values.quantile(0.95),
        ),

        maximum_optimality_gap_percent=(
            "optimality_gap_percent",
            "max",
        ),

        greedy_optimal_cost_rate=(
            "greedy_optimal_cost_numeric",
            "mean",
        ),

        large_gap_rate=(
            "large_gap",
            "mean",
        ),

        greedy_more_items_rate=(
            "greedy_more_items",
            "mean",
        ),

        mean_jaccard_similarity=(
            "jaccard_similarity",
            "mean",
        ),

        mean_exact_selected_count=(
            "exact_selected_count",
            "mean",
        ),

        mean_greedy_selected_count=(
            "greedy_selected_count",
            "mean",
        ),

        mean_selected_count_difference=(
            "selected_count_difference",
            "mean",
        ),

        mean_utility_difference=(
            "utility_difference",
            "mean",
        ),

        mean_privacy_difference=(
            "privacy_difference",
            "mean",
        ),

        mean_cognitive_cost_difference=(
            "cognitive_cost_difference",
            "mean",
        ),

        mean_absolute_cost_difference=(
            "absolute_disclosure_cost_difference",
            "mean",
        ),

        mean_subset_precomputation_runtime_seconds=(
            "subset_precomputation_runtime_seconds",
            "mean",
        ),

        mean_cached_exact_solve_runtime_seconds=(
            "exact_cached_solve_runtime_seconds",
            "mean",
        ),

        mean_greedy_runtime_seconds=(
            "greedy_runtime_seconds",
            "mean",
        ),
    )
)


# --------------------------------------------------
# Convert rates to percentages
# --------------------------------------------------

size_quality_summary[
    "greedy_optimal_cost_percent"
] = (
    100.0
    * size_quality_summary[
        "greedy_optimal_cost_rate"
    ]
)


size_quality_summary[
    "large_gap_percent"
] = (
    100.0
    * size_quality_summary[
        "large_gap_rate"
    ]
)


size_quality_summary[
    "greedy_more_items_percent"
] = (
    100.0
    * size_quality_summary[
        "greedy_more_items_rate"
    ]
)


# --------------------------------------------------
# Feasibility by candidate-set size
#
# One row per instance–tau case is used because
# feasibility is repeated across weight profiles.
# --------------------------------------------------

size_feasibility_summary = (
    corrected_instance_tau_df
    .groupby(
        "n_items",
        as_index=False,
    )
    .agg(
        total_instance_tau_cases=(
            "instance_id",
            "size",
        ),

        feasible_instance_tau_cases=(
            "exact_is_feasible",
            "sum",
        ),

        mean_full_set_utility=(
            "full_set_utility",
            "mean",
        ),

        minimum_full_set_utility=(
            "full_set_utility",
            "min",
        ),
    )
)


size_feasibility_summary[
    "infeasible_instance_tau_cases"
] = (
    size_feasibility_summary[
        "total_instance_tau_cases"
    ]
    -
    size_feasibility_summary[
        "feasible_instance_tau_cases"
    ]
)


size_feasibility_summary[
    "feasibility_rate_percent"
] = (
    100.0
    * size_feasibility_summary[
        "feasible_instance_tau_cases"
    ]
    / size_feasibility_summary[
        "total_instance_tau_cases"
    ]
)


# --------------------------------------------------
# Correct precomputation timing
#
# The same precomputation runtime is repeated in all
# 16 condition rows of an instance. Compute its mean
# using one timing value per independent instance.
# --------------------------------------------------

instance_size_timing_df = (
    analysis_benchmark_df
    .groupby(
        [
            "instance_id",
            "n_items",
        ],
        as_index=False,
    )
    .agg(
        subset_precomputation_runtime_seconds=(
            "subset_precomputation_runtime_seconds",
            "first",
        ),
    )
)


size_precomputation_summary = (
    instance_size_timing_df
    .groupby(
        "n_items",
        as_index=False,
    )
    .agg(
        mean_precomputation_runtime_seconds=(
            "subset_precomputation_runtime_seconds",
            "mean",
        ),

        median_precomputation_runtime_seconds=(
            "subset_precomputation_runtime_seconds",
            "median",
        ),

        maximum_precomputation_runtime_seconds=(
            "subset_precomputation_runtime_seconds",
            "max",
        ),
    )
)


# --------------------------------------------------
# Merge all size summaries
# --------------------------------------------------

size_summary = (
    size_quality_summary
    .drop(
        columns=[
            "mean_subset_precomputation_runtime_seconds",
        ]
    )
    .merge(
        size_feasibility_summary,
        on="n_items",
        how="left",
    )
    .merge(
        size_precomputation_summary,
        on="n_items",
        how="left",
    )
)


display_columns = [
    "n_items",
    "unique_instances",
    "total_instance_tau_cases",
    "infeasible_instance_tau_cases",
    "feasibility_rate_percent",
    "comparable_rows",
    "mean_optimality_gap_percent",
    "median_optimality_gap_percent",
    "p90_optimality_gap_percent",
    "p95_optimality_gap_percent",
    "maximum_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "large_gap_percent",
    "greedy_more_items_percent",
    "mean_jaccard_similarity",
    "mean_exact_selected_count",
    "mean_greedy_selected_count",
    "mean_selected_count_difference",
    "mean_precomputation_runtime_seconds",
    "mean_cached_exact_solve_runtime_seconds",
    "mean_greedy_runtime_seconds",
]


print(
    "======================================"
)

print(
    "SOLUTION QUALITY BY CANDIDATE-SET SIZE"
)

print(
    "======================================"
)

print(
    size_summary[
        display_columns
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

In [ ]:
# --------------------------------------------------
# Create one analysis record per independent instance
# --------------------------------------------------

INSTANCE_LEVEL_PATH = (
    RESULTS_DIR
    / "exact_vs_greedy_instance_level_summary.csv"
)


# --------------------------------------------------
# Comparable-condition quality data
# --------------------------------------------------

instance_quality_source_df = (
    analysis_benchmark_df[
        analysis_benchmark_df[
            "is_comparable"
        ]
        == True
    ]
    .copy()
)


instance_quality_source_df[
    "greedy_optimal_cost_numeric"
] = (
    instance_quality_source_df[
        "greedy_optimal_cost"
    ]
    .astype(float)
)


instance_quality_source_df[
    "large_gap"
] = (
    instance_quality_source_df[
        "optimality_gap_percent"
    ]
    > 10.0
).astype(int)


instance_quality_source_df[
    "greedy_more_items"
] = (
    instance_quality_source_df[
        "selected_count_difference"
    ]
    > 0
).astype(int)


# --------------------------------------------------
# Aggregate quality metrics within each independent
# synthetic candidate set
# --------------------------------------------------

instance_quality_summary = (
    instance_quality_source_df
    .groupby(
        [
            "instance_id",
            "seed",
            "n_items",
            "regime",
            "replicate",
        ],
        as_index=False,
    )
    .agg(
        comparable_condition_rows=(
            "optimality_gap_percent",
            "count",
        ),

        mean_optimality_gap_percent=(
            "optimality_gap_percent",
            "mean",
        ),

        median_optimality_gap_percent=(
            "optimality_gap_percent",
            "median",
        ),

        p90_optimality_gap_percent=(
            "optimality_gap_percent",
            lambda values:
                values.quantile(0.90),
        ),

        maximum_optimality_gap_percent=(
            "optimality_gap_percent",
            "max",
        ),

        greedy_optimal_cost_rate=(
            "greedy_optimal_cost_numeric",
            "mean",
        ),

        large_gap_rate=(
            "large_gap",
            "mean",
        ),

        greedy_more_items_rate=(
            "greedy_more_items",
            "mean",
        ),

        mean_jaccard_similarity=(
            "jaccard_similarity",
            "mean",
        ),

        mean_exact_selected_count=(
            "exact_selected_count",
            "mean",
        ),

        mean_greedy_selected_count=(
            "greedy_selected_count",
            "mean",
        ),

        mean_selected_count_difference=(
            "selected_count_difference",
            "mean",
        ),

        mean_utility_difference=(
            "utility_difference",
            "mean",
        ),

        mean_privacy_difference=(
            "privacy_difference",
            "mean",
        ),

        mean_cognitive_cost_difference=(
            "cognitive_cost_difference",
            "mean",
        ),

        mean_absolute_cost_difference=(
            "absolute_disclosure_cost_difference",
            "mean",
        ),
    )
)


# --------------------------------------------------
# Feasibility summary
#
# corrected_instance_tau_df contains one record per
# instance and sufficiency threshold, avoiding the four
# repeated objective-weight rows.
# --------------------------------------------------

instance_feasibility_summary = (
    corrected_instance_tau_df
    .groupby(
        [
            "instance_id",
            "n_items",
            "regime",
        ],
        as_index=False,
    )
    .agg(
        total_tau_cases=(
            "tau",
            "count",
        ),

        feasible_tau_cases=(
            "exact_is_feasible",
            "sum",
        ),

        mean_full_set_utility=(
            "full_set_utility",
            "mean",
        ),

        minimum_full_set_utility=(
            "full_set_utility",
            "min",
        ),
    )
)


instance_feasibility_summary[
    "infeasible_tau_cases"
] = (
    instance_feasibility_summary[
        "total_tau_cases"
    ]
    -
    instance_feasibility_summary[
        "feasible_tau_cases"
    ]
)


instance_feasibility_summary[
    "feasibility_rate"
] = (
    instance_feasibility_summary[
        "feasible_tau_cases"
    ]
    /
    instance_feasibility_summary[
        "total_tau_cases"
    ]
)


# --------------------------------------------------
# Timing summary
#
# Precomputation is counted once per instance.
# Cached Exact and Greedy times are summed across the
# 16 evaluated conditions.
# --------------------------------------------------

instance_timing_summary = (
    analysis_benchmark_df
    .groupby(
        [
            "instance_id",
            "n_items",
            "regime",
        ],
        as_index=False,
    )
    .agg(
        subset_precomputation_runtime_seconds=(
            "subset_precomputation_runtime_seconds",
            "first",
        ),

        total_cached_exact_solve_runtime_seconds=(
            "exact_cached_solve_runtime_seconds",
            "sum",
        ),

        total_greedy_runtime_seconds=(
            "greedy_runtime_seconds",
            "sum",
        ),
    )
)


# --------------------------------------------------
# Merge the instance-level components
# --------------------------------------------------

instance_level_df = (
    instance_quality_summary
    .merge(
        instance_feasibility_summary,
        on=[
            "instance_id",
            "n_items",
            "regime",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        instance_timing_summary,
        on=[
            "instance_id",
            "n_items",
            "regime",
        ],
        how="left",
        validate="one_to_one",
    )
)


# Convert rates to percentages for convenient reporting.
instance_level_df[
    "greedy_optimal_cost_percent"
] = (
    100.0
    * instance_level_df[
        "greedy_optimal_cost_rate"
    ]
)

instance_level_df[
    "large_gap_percent"
] = (
    100.0
    * instance_level_df[
        "large_gap_rate"
    ]
)

instance_level_df[
    "greedy_more_items_percent"
] = (
    100.0
    * instance_level_df[
        "greedy_more_items_rate"
    ]
)

instance_level_df[
    "feasibility_rate_percent"
] = (
    100.0
    * instance_level_df[
        "feasibility_rate"
    ]
)


# --------------------------------------------------
# Integrity checks
# --------------------------------------------------

expected_comparable_rows = (
    instance_level_df[
        "feasible_tau_cases"
    ]
    * len(WEIGHT_PROFILES)
)

comparable_row_mismatches = (
    instance_level_df[
        "comparable_condition_rows"
    ]
    != expected_comparable_rows
).sum()

duplicate_instance_rows = (
    instance_level_df[
        "instance_id"
    ]
    .duplicated()
    .sum()
)

missing_instance_values = (
    instance_level_df
    .isna()
    .sum()
    .sum()
)


assert len(instance_level_df) == 1500
assert duplicate_instance_rows == 0
assert comparable_row_mismatches == 0
assert missing_instance_values == 0


# --------------------------------------------------
# Save the independent-instance dataset
# --------------------------------------------------

instance_level_df = (
    instance_level_df
    .sort_values(
        by=[
            "n_items",
            "regime",
            "replicate",
        ]
    )
    .reset_index(
        drop=True
    )
)

instance_level_df.to_csv(
    INSTANCE_LEVEL_PATH,
    index=False,
)


# --------------------------------------------------
# Print validation summary
# --------------------------------------------------

print(
    "======================================"
)

print(
    "INSTANCE-LEVEL DATASET CREATED"
)

print(
    "======================================"
)

print(
    "Independent instance rows:",
    len(instance_level_df)
)

print(
    "Unique instance IDs:",
    instance_level_df[
        "instance_id"
    ].nunique()
)

print(
    "Duplicate instance rows:",
    duplicate_instance_rows
)

print(
    "Comparable-row mismatches:",
    comparable_row_mismatches
)

print(
    "Missing values:",
    missing_instance_values
)

print(
    "Minimum comparable conditions per instance:",
    instance_level_df[
        "comparable_condition_rows"
    ].min()
)

print(
    "Maximum comparable conditions per instance:",
    instance_level_df[
        "comparable_condition_rows"
    ].max()
)

print(
    "\nInstances by size and regime:"
)

print(
    instance_level_df
    .groupby(
        [
            "n_items",
            "regime",
        ]
    )
    .size()
    .rename(
        "number_of_instances"
    )
    .reset_index()
    .to_string(
        index=False
    )
)

print(
    "\nInstance-level dataset saved to:"
)

print(
    INSTANCE_LEVEL_PATH
)

In [ ]:
# --------------------------------------------------
# Stratified instance-cluster bootstrap confidence intervals
# --------------------------------------------------

BOOTSTRAP_REPLICATES = 5000
BOOTSTRAP_SEED = GLOBAL_SEED + 83

BOOTSTRAP_RESULTS_PATH = (
    RESULTS_DIR
    / "overall_quality_cluster_bootstrap_replicates.csv"
)

BOOTSTRAP_SUMMARY_PATH = (
    TABLES_DIR
    / "overall_quality_cluster_bootstrap_ci.csv"
)


# --------------------------------------------------
# Prepare comparable condition-level observations
# --------------------------------------------------

bootstrap_source_df = (
    analysis_benchmark_df[
        analysis_benchmark_df[
            "is_comparable"
        ]
        == True
    ]
    .copy()
)


bootstrap_source_df[
    "greedy_optimal_cost_numeric"
] = (
    bootstrap_source_df[
        "greedy_optimal_cost"
    ]
    .astype(int)
)


bootstrap_source_df[
    "large_gap_numeric"
] = (
    bootstrap_source_df[
        "optimality_gap_percent"
    ]
    > 10.0
).astype(int)


bootstrap_source_df[
    "greedy_more_items_numeric"
] = (
    bootstrap_source_df[
        "selected_count_difference"
    ]
    > 0
).astype(int)


# --------------------------------------------------
# Create sufficient statistics for each independent
# synthetic candidate-set instance
# --------------------------------------------------

cluster_summary_df = (
    bootstrap_source_df
    .groupby(
        [
            "instance_id",
            "n_items",
            "regime",
        ],
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "size",
        ),

        gap_sum=(
            "optimality_gap_percent",
            "sum",
        ),

        optimal_cost_sum=(
            "greedy_optimal_cost_numeric",
            "sum",
        ),

        jaccard_sum=(
            "jaccard_similarity",
            "sum",
        ),

        large_gap_sum=(
            "large_gap_numeric",
            "sum",
        ),

        greedy_more_items_sum=(
            "greedy_more_items_numeric",
            "sum",
        ),

        selected_count_difference_sum=(
            "selected_count_difference",
            "sum",
        ),
    )
)


# --------------------------------------------------
# Validate the stratified experimental structure
# --------------------------------------------------

stratum_counts_df = (
    cluster_summary_df
    .groupby(
        [
            "n_items",
            "regime",
        ],
        as_index=False,
    )
    .agg(
        number_of_instances=(
            "instance_id",
            "nunique",
        )
    )
)


assert len(cluster_summary_df) == 1500

assert (
    stratum_counts_df[
        "number_of_instances"
    ]
    == 100
).all()


# --------------------------------------------------
# Columns needed for efficient resampling
# --------------------------------------------------

aggregate_columns = [
    "comparable_rows",
    "gap_sum",
    "optimal_cost_sum",
    "jaccard_sum",
    "large_gap_sum",
    "greedy_more_items_sum",
    "selected_count_difference_sum",
]


# Store one numerical matrix for each size–regime stratum.
stratum_arrays = []

for _, stratum_df in (
    cluster_summary_df
    .groupby(
        [
            "n_items",
            "regime",
        ],
        sort=True,
    )
):

    stratum_arrays.append(
        stratum_df[
            aggregate_columns
        ].to_numpy(
            dtype=float
        )
    )


# --------------------------------------------------
# Convert aggregated totals into reporting metrics
# --------------------------------------------------

def calculate_bootstrap_metrics(
    totals: np.ndarray,
) -> np.ndarray:
    """
    Calculate row-weighted benchmark metrics from
    cluster-level aggregate totals.
    """

    comparable_rows = totals[0]

    return np.array(
        [
            # Mean relative optimality gap
            totals[1]
            / comparable_rows,

            # Percentage attaining optimal cost
            100.0
            * totals[2]
            / comparable_rows,

            # Mean selected-subset overlap
            totals[3]
            / comparable_rows,

            # Percentage with gap greater than 10%
            100.0
            * totals[4]
            / comparable_rows,

            # Percentage where Greedy selected more items
            100.0
            * totals[5]
            / comparable_rows,

            # Mean Greedy-minus-Exact item count
            totals[6]
            / comparable_rows,
        ],
        dtype=float,
    )


metric_names = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "greedy_more_items_percent",
    "mean_selected_count_difference",
]


# --------------------------------------------------
# Point estimates from the complete benchmark
# --------------------------------------------------

observed_totals = (
    cluster_summary_df[
        aggregate_columns
    ]
    .sum()
    .to_numpy(
        dtype=float
    )
)

point_estimates = (
    calculate_bootstrap_metrics(
        observed_totals
    )
)


# --------------------------------------------------
# Stratified cluster bootstrap
#
# Within every size–regime stratum, resample the
# 100 independent instances with replacement.
# --------------------------------------------------

bootstrap_rng = np.random.default_rng(
    BOOTSTRAP_SEED
)

bootstrap_estimates = np.empty(
    (
        BOOTSTRAP_REPLICATES,
        len(metric_names),
    ),
    dtype=float,
)


for bootstrap_index in range(
    BOOTSTRAP_REPLICATES
):

    resampled_totals = np.zeros(
        len(aggregate_columns),
        dtype=float,
    )

    for stratum_array in stratum_arrays:

        number_of_instances = (
            stratum_array.shape[0]
        )

        sampled_indices = (
            bootstrap_rng.integers(
                low=0,
                high=number_of_instances,
                size=number_of_instances,
            )
        )

        resampled_totals += (
            stratum_array[
                sampled_indices
            ]
            .sum(
                axis=0
            )
        )


    bootstrap_estimates[
        bootstrap_index
    ] = calculate_bootstrap_metrics(
        resampled_totals
    )


    if (
        bootstrap_index + 1
    ) % 1000 == 0:

        print(
            "Completed bootstrap replicates:",
            bootstrap_index + 1,
        )


# --------------------------------------------------
# Bootstrap uncertainty summary
# --------------------------------------------------

bootstrap_standard_errors = (
    bootstrap_estimates.std(
        axis=0,
        ddof=1,
    )
)

confidence_lower = np.quantile(
    bootstrap_estimates,
    0.025,
    axis=0,
)

confidence_upper = np.quantile(
    bootstrap_estimates,
    0.975,
    axis=0,
)


bootstrap_summary_df = pd.DataFrame(
    {
        "metric":
            metric_names,

        "estimate":
            point_estimates,

        "bootstrap_standard_error":
            bootstrap_standard_errors,

        "ci_95_lower":
            confidence_lower,

        "ci_95_upper":
            confidence_upper,

        "bootstrap_replicates":
            BOOTSTRAP_REPLICATES,

        "resampling_unit":
            "independent_instance",

        "resampling_design":
            "stratified_by_size_and_regime",
    }
)


# --------------------------------------------------
# Save bootstrap results
# --------------------------------------------------

bootstrap_replicates_df = pd.DataFrame(
    bootstrap_estimates,
    columns=metric_names,
)

bootstrap_replicates_df.insert(
    0,
    "bootstrap_replicate",
    np.arange(
        1,
        BOOTSTRAP_REPLICATES + 1,
    ),
)


bootstrap_replicates_df.to_csv(
    BOOTSTRAP_RESULTS_PATH,
    index=False,
)

bootstrap_summary_df.to_csv(
    BOOTSTRAP_SUMMARY_PATH,
    index=False,
)


# --------------------------------------------------
# Print results
# --------------------------------------------------

print(
    "\n======================================"
)

print(
    "STRATIFIED CLUSTER BOOTSTRAP RESULTS"
)

print(
    "======================================"
)

print(
    "Independent instances:",
    len(cluster_summary_df)
)

print(
    "Experimental strata:",
    len(stratum_arrays)
)

print(
    "Instances per stratum:",
    stratum_counts_df[
        "number_of_instances"
    ].unique()
)

print(
    "Bootstrap replicates:",
    BOOTSTRAP_REPLICATES
)


print(
    "\nOverall estimates and 95% confidence intervals:"
)

print(
    bootstrap_summary_df[
        [
            "metric",
            "estimate",
            "bootstrap_standard_error",
            "ci_95_lower",
            "ci_95_upper",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)


print(
    "\nBootstrap replicates saved to:"
)

print(
    BOOTSTRAP_RESULTS_PATH
)

print(
    "\nBootstrap summary saved to:"
)

print(
    BOOTSTRAP_SUMMARY_PATH
)

In [ ]:
# --------------------------------------------------
# Regime-specific stratified cluster bootstrap
# --------------------------------------------------

REGIME_BOOTSTRAP_REPLICATES = 5000
REGIME_BOOTSTRAP_SEED = GLOBAL_SEED + 84

REGIME_BOOTSTRAP_SUMMARY_PATH = (
    TABLES_DIR
    / "regime_quality_cluster_bootstrap_ci.csv"
)


regime_metric_names = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "greedy_more_items_percent",
    "mean_selected_count_difference",
]


def calculate_regime_metrics(
    totals: np.ndarray,
) -> np.ndarray:
    """
    Convert cluster-level sufficient-statistic totals
    into regime-level quality metrics.
    """

    comparable_rows = totals[0]

    return np.array(
        [
            totals[1]
            / comparable_rows,

            100.0
            * totals[2]
            / comparable_rows,

            totals[3]
            / comparable_rows,

            100.0
            * totals[4]
            / comparable_rows,

            100.0
            * totals[5]
            / comparable_rows,

            totals[6]
            / comparable_rows,
        ],
        dtype=float,
    )


regime_bootstrap_rng = (
    np.random.default_rng(
        REGIME_BOOTSTRAP_SEED
    )
)


regime_bootstrap_summary_rows = []


for regime_index, regime in enumerate(
    INSTANCE_REGIMES
):

    regime_cluster_df = (
        cluster_summary_df[
            cluster_summary_df[
                "regime"
            ]
            == regime
        ]
        .copy()
    )


    # ----------------------------------------------
    # Preserve candidate-set-size balance
    # ----------------------------------------------

    regime_size_arrays = []

    for n_items, size_df in (
        regime_cluster_df
        .groupby(
            "n_items",
            sort=True,
        )
    ):

        size_array = (
            size_df[
                aggregate_columns
            ]
            .to_numpy(
                dtype=float
            )
        )

        assert (
            size_array.shape[0]
            == 100
        )

        regime_size_arrays.append(
            size_array
        )


    # ----------------------------------------------
    # Observed regime-level estimates
    # ----------------------------------------------

    observed_regime_totals = (
        regime_cluster_df[
            aggregate_columns
        ]
        .sum()
        .to_numpy(
            dtype=float
        )
    )

    observed_regime_metrics = (
        calculate_regime_metrics(
            observed_regime_totals
        )
    )


    # ----------------------------------------------
    # Bootstrap independent instances within size
    # ----------------------------------------------

    regime_bootstrap_estimates = np.empty(
        (
            REGIME_BOOTSTRAP_REPLICATES,
            len(regime_metric_names),
        ),
        dtype=float,
    )


    for bootstrap_index in range(
        REGIME_BOOTSTRAP_REPLICATES
    ):

        resampled_totals = np.zeros(
            len(aggregate_columns),
            dtype=float,
        )


        for size_array in regime_size_arrays:

            number_of_instances = (
                size_array.shape[0]
            )

            sampled_indices = (
                regime_bootstrap_rng.integers(
                    low=0,
                    high=number_of_instances,
                    size=number_of_instances,
                )
            )

            resampled_totals += (
                size_array[
                    sampled_indices
                ]
                .sum(
                    axis=0
                )
            )


        regime_bootstrap_estimates[
            bootstrap_index
        ] = calculate_regime_metrics(
            resampled_totals
        )


    # ----------------------------------------------
    # Confidence intervals
    # ----------------------------------------------

    regime_standard_errors = (
        regime_bootstrap_estimates.std(
            axis=0,
            ddof=1,
        )
    )

    regime_ci_lower = np.quantile(
        regime_bootstrap_estimates,
        0.025,
        axis=0,
    )

    regime_ci_upper = np.quantile(
        regime_bootstrap_estimates,
        0.975,
        axis=0,
    )


    for metric_index, metric_name in enumerate(
        regime_metric_names
    ):

        regime_bootstrap_summary_rows.append(
            {
                "regime":
                    regime,

                "metric":
                    metric_name,

                "estimate":
                    observed_regime_metrics[
                        metric_index
                    ],

                "bootstrap_standard_error":
                    regime_standard_errors[
                        metric_index
                    ],

                "ci_95_lower":
                    regime_ci_lower[
                        metric_index
                    ],

                "ci_95_upper":
                    regime_ci_upper[
                        metric_index
                    ],

                "independent_instances":
                    len(
                        regime_cluster_df
                    ),

                "size_strata":
                    len(
                        regime_size_arrays
                    ),

                "bootstrap_replicates":
                    REGIME_BOOTSTRAP_REPLICATES,
            }
        )


    print(
        "Completed regime:",
        regime
    )


regime_bootstrap_summary_df = pd.DataFrame(
    regime_bootstrap_summary_rows
)


# --------------------------------------------------
# Save full regime-bootstrap table
# --------------------------------------------------

regime_bootstrap_summary_df.to_csv(
    REGIME_BOOTSTRAP_SUMMARY_PATH,
    index=False,
)


# --------------------------------------------------
# Create compact display for key metrics
# --------------------------------------------------

key_regime_metrics = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
]


regime_bootstrap_display = (
    regime_bootstrap_summary_df[
        regime_bootstrap_summary_df[
            "metric"
        ].isin(
            key_regime_metrics
        )
    ]
    .copy()
)


regime_bootstrap_display[
    "estimate_and_ci"
] = (
    regime_bootstrap_display[
        "estimate"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    regime_bootstrap_display[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    regime_bootstrap_display[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


regime_bootstrap_pivot = (
    regime_bootstrap_display
    .pivot(
        index="regime",
        columns="metric",
        values="estimate_and_ci",
    )
    .reset_index()
)


print(
    "\n======================================"
)

print(
    "REGIME-SPECIFIC CLUSTERED 95% CIs"
)

print(
    "======================================"
)

print(
    regime_bootstrap_pivot.to_string(
        index=False
    )
)


print(
    "\nFull regime-bootstrap table saved to:"
)

print(
    REGIME_BOOTSTRAP_SUMMARY_PATH
)

In [ ]:
# --------------------------------------------------
# Pairwise regime contrasts using stratified
# independent-instance bootstrap
# --------------------------------------------------

PAIRWISE_BOOTSTRAP_REPLICATES = 5000
PAIRWISE_BOOTSTRAP_SEED = GLOBAL_SEED + 85

PAIRWISE_REGIME_PATH = (
    TABLES_DIR
    / "pairwise_regime_bootstrap_contrasts.csv"
)


PAIRWISE_METRICS = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
]


PAIRWISE_REGIME_COMPARISONS = [
    (
        "utility_burden_conflict",
        "utility_burden_alignment",
    ),
    (
        "independent",
        "utility_burden_alignment",
    ),
    (
        "utility_burden_conflict",
        "independent",
    ),
]


# --------------------------------------------------
# Prepare one array for each regime × size stratum
# --------------------------------------------------

regime_size_arrays = {}

for regime in INSTANCE_REGIMES:

    regime_size_arrays[
        regime
    ] = {}

    for n_items in EXACT_BENCHMARK_SIZES:

        stratum_df = (
            cluster_summary_df[
                (
                    cluster_summary_df[
                        "regime"
                    ]
                    == regime
                )
                &
                (
                    cluster_summary_df[
                        "n_items"
                    ]
                    == n_items
                )
            ]
            .copy()
        )

        stratum_array = (
            stratum_df[
                aggregate_columns
            ]
            .to_numpy(
                dtype=float
            )
        )

        assert (
            stratum_array.shape[0]
            == 100
        )

        regime_size_arrays[
            regime
        ][
            n_items
        ] = stratum_array


# --------------------------------------------------
# Metric helper
# --------------------------------------------------

def calculate_pairwise_metrics(
    totals: np.ndarray,
) -> np.ndarray:
    """
    Return the four metrics used in the direct
    pairwise regime comparisons.
    """

    comparable_rows = totals[0]

    return np.array(
        [
            # Mean optimality gap percentage
            totals[1]
            / comparable_rows,

            # Greedy optimal-cost recovery percentage
            100.0
            * totals[2]
            / comparable_rows,

            # Mean Jaccard similarity
            totals[3]
            / comparable_rows,

            # Percentage of gaps greater than 10%
            100.0
            * totals[4]
            / comparable_rows,
        ],
        dtype=float,
    )


# --------------------------------------------------
# Observed regime-level point estimates
# --------------------------------------------------

observed_regime_metrics = {}

for regime in INSTANCE_REGIMES:

    regime_totals = (
        cluster_summary_df[
            cluster_summary_df[
                "regime"
            ]
            == regime
        ][
            aggregate_columns
        ]
        .sum()
        .to_numpy(
            dtype=float
        )
    )

    observed_regime_metrics[
        regime
    ] = calculate_pairwise_metrics(
        regime_totals
    )


# --------------------------------------------------
# Bootstrap pairwise differences
#
# Difference definition:
# first regime minus second regime.
# --------------------------------------------------

pairwise_rng = np.random.default_rng(
    PAIRWISE_BOOTSTRAP_SEED
)

pairwise_summary_rows = []


for first_regime, second_regime in (
    PAIRWISE_REGIME_COMPARISONS
):

    bootstrap_differences = np.empty(
        (
            PAIRWISE_BOOTSTRAP_REPLICATES,
            len(PAIRWISE_METRICS),
        ),
        dtype=float,
    )


    for bootstrap_index in range(
        PAIRWISE_BOOTSTRAP_REPLICATES
    ):

        first_totals = np.zeros(
            len(aggregate_columns),
            dtype=float,
        )

        second_totals = np.zeros(
            len(aggregate_columns),
            dtype=float,
        )


        # Preserve the five candidate-set-size strata.
        for n_items in EXACT_BENCHMARK_SIZES:

            first_array = (
                regime_size_arrays[
                    first_regime
                ][
                    n_items
                ]
            )

            second_array = (
                regime_size_arrays[
                    second_regime
                ][
                    n_items
                ]
            )


            first_indices = (
                pairwise_rng.integers(
                    low=0,
                    high=first_array.shape[0],
                    size=first_array.shape[0],
                )
            )

            second_indices = (
                pairwise_rng.integers(
                    low=0,
                    high=second_array.shape[0],
                    size=second_array.shape[0],
                )
            )


            first_totals += (
                first_array[
                    first_indices
                ]
                .sum(
                    axis=0
                )
            )

            second_totals += (
                second_array[
                    second_indices
                ]
                .sum(
                    axis=0
                )
            )


        first_metrics = (
            calculate_pairwise_metrics(
                first_totals
            )
        )

        second_metrics = (
            calculate_pairwise_metrics(
                second_totals
            )
        )

        bootstrap_differences[
            bootstrap_index
        ] = (
            first_metrics
            -
            second_metrics
        )


    # --------------------------------------------------
    # Point estimates and confidence intervals
    # --------------------------------------------------

    observed_difference = (
        observed_regime_metrics[
            first_regime
        ]
        -
        observed_regime_metrics[
            second_regime
        ]
    )

    standard_errors = (
        bootstrap_differences.std(
            axis=0,
            ddof=1,
        )
    )

    ci_lower = np.quantile(
        bootstrap_differences,
        0.025,
        axis=0,
    )

    ci_upper = np.quantile(
        bootstrap_differences,
        0.975,
        axis=0,
    )


    for metric_index, metric_name in enumerate(
        PAIRWISE_METRICS
    ):

        pairwise_summary_rows.append(
            {
                "first_regime":
                    first_regime,

                "second_regime":
                    second_regime,

                "contrast":
                    (
                        f"{first_regime}"
                        f" minus "
                        f"{second_regime}"
                    ),

                "metric":
                    metric_name,

                "estimated_difference":
                    observed_difference[
                        metric_index
                    ],

                "bootstrap_standard_error":
                    standard_errors[
                        metric_index
                    ],

                "ci_95_lower":
                    ci_lower[
                        metric_index
                    ],

                "ci_95_upper":
                    ci_upper[
                        metric_index
                    ],

                "ci_excludes_zero":
                    (
                        ci_lower[
                            metric_index
                        ]
                        > 0.0
                    )
                    or
                    (
                        ci_upper[
                            metric_index
                        ]
                        < 0.0
                    ),

                "bootstrap_replicates":
                    PAIRWISE_BOOTSTRAP_REPLICATES,
            }
        )


    print(
        "Completed contrast:",
        first_regime,
        "minus",
        second_regime,
    )


# --------------------------------------------------
# Save and display
# --------------------------------------------------

pairwise_regime_df = pd.DataFrame(
    pairwise_summary_rows
)

pairwise_regime_df.to_csv(
    PAIRWISE_REGIME_PATH,
    index=False,
)


pairwise_display_df = (
    pairwise_regime_df.copy()
)


pairwise_display_df[
    "difference_and_ci"
] = (
    pairwise_display_df[
        "estimated_difference"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    pairwise_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    pairwise_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


print(
    "\n======================================"
)

print(
    "PAIRWISE REGIME BOOTSTRAP CONTRASTS"
)

print(
    "======================================"
)

print(
    pairwise_display_df[
        [
            "contrast",
            "metric",
            "difference_and_ci",
            "ci_excludes_zero",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\nPairwise regime contrasts saved to:"
)

print(
    PAIRWISE_REGIME_PATH
)

In [ ]:
# --------------------------------------------------
# Candidate-set-size-specific stratified cluster bootstrap
# --------------------------------------------------

SIZE_BOOTSTRAP_REPLICATES = 5000
SIZE_BOOTSTRAP_SEED = GLOBAL_SEED + 86

SIZE_BOOTSTRAP_SUMMARY_PATH = (
    TABLES_DIR
    / "candidate_size_quality_cluster_bootstrap_ci.csv"
)


size_metric_names = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "greedy_more_items_percent",
    "mean_selected_count_difference",
]


def calculate_size_metrics(
    totals: np.ndarray,
) -> np.ndarray:
    """
    Convert cluster-level sufficient-statistic totals
    into candidate-set-size-specific quality metrics.
    """

    comparable_rows = totals[0]

    return np.array(
        [
            # Mean relative optimality gap
            totals[1]
            / comparable_rows,

            # Greedy optimal-cost recovery percentage
            100.0
            * totals[2]
            / comparable_rows,

            # Mean Exact–Greedy subset overlap
            totals[3]
            / comparable_rows,

            # Percentage with gap above 10%
            100.0
            * totals[4]
            / comparable_rows,

            # Percentage where Greedy selected more items
            100.0
            * totals[5]
            / comparable_rows,

            # Mean Greedy-minus-Exact item count
            totals[6]
            / comparable_rows,
        ],
        dtype=float,
    )


size_bootstrap_rng = np.random.default_rng(
    SIZE_BOOTSTRAP_SEED
)

size_bootstrap_summary_rows = []


for n_items in EXACT_BENCHMARK_SIZES:

    size_cluster_df = (
        cluster_summary_df[
            cluster_summary_df[
                "n_items"
            ]
            == n_items
        ]
        .copy()
    )


    # ----------------------------------------------
    # Preserve the three structural-regime strata
    # ----------------------------------------------

    size_regime_arrays = []

    for regime, regime_df in (
        size_cluster_df
        .groupby(
            "regime",
            sort=True,
        )
    ):

        regime_array = (
            regime_df[
                aggregate_columns
            ]
            .to_numpy(
                dtype=float
            )
        )

        assert (
            regime_array.shape[0]
            == 100
        )

        size_regime_arrays.append(
            regime_array
        )


    # ----------------------------------------------
    # Observed size-specific estimates
    # ----------------------------------------------

    observed_size_totals = (
        size_cluster_df[
            aggregate_columns
        ]
        .sum()
        .to_numpy(
            dtype=float
        )
    )

    observed_size_metrics = (
        calculate_size_metrics(
            observed_size_totals
        )
    )


    # ----------------------------------------------
    # Bootstrap within each regime stratum
    # ----------------------------------------------

    size_bootstrap_estimates = np.empty(
        (
            SIZE_BOOTSTRAP_REPLICATES,
            len(size_metric_names),
        ),
        dtype=float,
    )


    for bootstrap_index in range(
        SIZE_BOOTSTRAP_REPLICATES
    ):

        resampled_totals = np.zeros(
            len(aggregate_columns),
            dtype=float,
        )


        for regime_array in size_regime_arrays:

            number_of_instances = (
                regime_array.shape[0]
            )

            sampled_indices = (
                size_bootstrap_rng.integers(
                    low=0,
                    high=number_of_instances,
                    size=number_of_instances,
                )
            )

            resampled_totals += (
                regime_array[
                    sampled_indices
                ]
                .sum(
                    axis=0
                )
            )


        size_bootstrap_estimates[
            bootstrap_index
        ] = calculate_size_metrics(
            resampled_totals
        )


    # ----------------------------------------------
    # Confidence intervals
    # ----------------------------------------------

    size_standard_errors = (
        size_bootstrap_estimates.std(
            axis=0,
            ddof=1,
        )
    )

    size_ci_lower = np.quantile(
        size_bootstrap_estimates,
        0.025,
        axis=0,
    )

    size_ci_upper = np.quantile(
        size_bootstrap_estimates,
        0.975,
        axis=0,
    )


    for metric_index, metric_name in enumerate(
        size_metric_names
    ):

        size_bootstrap_summary_rows.append(
            {
                "n_items":
                    n_items,

                "metric":
                    metric_name,

                "estimate":
                    observed_size_metrics[
                        metric_index
                    ],

                "bootstrap_standard_error":
                    size_standard_errors[
                        metric_index
                    ],

                "ci_95_lower":
                    size_ci_lower[
                        metric_index
                    ],

                "ci_95_upper":
                    size_ci_upper[
                        metric_index
                    ],

                "independent_instances":
                    len(
                        size_cluster_df
                    ),

                "regime_strata":
                    len(
                        size_regime_arrays
                    ),

                "bootstrap_replicates":
                    SIZE_BOOTSTRAP_REPLICATES,
            }
        )


    print(
        "Completed candidate-set size:",
        n_items
    )


# --------------------------------------------------
# Save complete size-bootstrap table
# --------------------------------------------------

size_bootstrap_summary_df = pd.DataFrame(
    size_bootstrap_summary_rows
)

size_bootstrap_summary_df.to_csv(
    SIZE_BOOTSTRAP_SUMMARY_PATH,
    index=False,
)


# --------------------------------------------------
# Compact display for key metrics
# --------------------------------------------------

key_size_metrics = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
]


size_bootstrap_display = (
    size_bootstrap_summary_df[
        size_bootstrap_summary_df[
            "metric"
        ].isin(
            key_size_metrics
        )
    ]
    .copy()
)


size_bootstrap_display[
    "estimate_and_ci"
] = (
    size_bootstrap_display[
        "estimate"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    size_bootstrap_display[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    size_bootstrap_display[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


size_bootstrap_pivot = (
    size_bootstrap_display
    .pivot(
        index="n_items",
        columns="metric",
        values="estimate_and_ci",
    )
    .reset_index()
    .sort_values(
        "n_items"
    )
)


print(
    "\n======================================"
)

print(
    "CANDIDATE-SIZE-SPECIFIC CLUSTERED 95% CIs"
)

print(
    "======================================"
)

print(
    size_bootstrap_pivot.to_string(
        index=False
    )
)


print(
    "\nFull candidate-size bootstrap table saved to:"
)

print(
    SIZE_BOOTSTRAP_SUMMARY_PATH
)

In [ ]:
# --------------------------------------------------
# Pairwise candidate-set-size bootstrap contrasts
# --------------------------------------------------

SIZE_CONTRAST_BOOTSTRAP_REPLICATES = 5000
SIZE_CONTRAST_BOOTSTRAP_SEED = GLOBAL_SEED + 87

SIZE_CONTRAST_PATH = (
    TABLES_DIR
    / "pairwise_candidate_size_bootstrap_contrasts.csv"
)


# Adjacent-size comparisons plus the full M=16 versus M=8 contrast.
# Each contrast is defined as:
#
# first candidate-set size minus second candidate-set size.
SIZE_COMPARISONS = [
    (10, 8),
    (12, 10),
    (14, 12),
    (16, 14),
    (16, 8),
]


SIZE_CONTRAST_METRICS = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
]


# --------------------------------------------------
# Convert aggregate totals into reporting metrics
# --------------------------------------------------

def calculate_size_contrast_metrics(
    totals: np.ndarray,
) -> np.ndarray:
    """
    Calculate candidate-set-size quality metrics from
    cluster-level sufficient-statistic totals.
    """

    comparable_rows = totals[0]

    return np.array(
        [
            # Mean relative optimality gap percentage
            totals[1]
            / comparable_rows,

            # Greedy optimal-cost recovery percentage
            100.0
            * totals[2]
            / comparable_rows,

            # Mean Exact–Greedy Jaccard similarity
            totals[3]
            / comparable_rows,

            # Percentage of gaps greater than 10%
            100.0
            * totals[4]
            / comparable_rows,
        ],
        dtype=float,
    )


# --------------------------------------------------
# Prepare one independent-instance array for every
# candidate-set-size × structural-regime stratum
# --------------------------------------------------

size_regime_cluster_arrays = {}


for n_items in EXACT_BENCHMARK_SIZES:

    size_regime_cluster_arrays[
        n_items
    ] = {}


    for regime in INSTANCE_REGIMES:

        stratum_df = (
            cluster_summary_df[
                (
                    cluster_summary_df[
                        "n_items"
                    ]
                    == n_items
                )
                &
                (
                    cluster_summary_df[
                        "regime"
                    ]
                    == regime
                )
            ]
            .copy()
        )


        stratum_array = (
            stratum_df[
                aggregate_columns
            ]
            .to_numpy(
                dtype=float
            )
        )


        # The benchmark contains 100 independent
        # instances in every size–regime stratum.
        assert (
            stratum_array.shape[0]
            == 100
        )


        size_regime_cluster_arrays[
            n_items
        ][
            regime
        ] = stratum_array


# --------------------------------------------------
# Observed point estimates for each candidate-set size
# --------------------------------------------------

observed_size_metrics = {}


for n_items in EXACT_BENCHMARK_SIZES:

    observed_totals = np.zeros(
        len(aggregate_columns),
        dtype=float,
    )


    for regime in INSTANCE_REGIMES:

        observed_totals += (
            size_regime_cluster_arrays[
                n_items
            ][
                regime
            ]
            .sum(
                axis=0
            )
        )


    observed_size_metrics[
        n_items
    ] = (
        calculate_size_contrast_metrics(
            observed_totals
        )
    )


# --------------------------------------------------
# Bootstrap direct size contrasts
#
# Instances are resampled separately within every
# candidate-set-size × regime stratum.
# --------------------------------------------------

size_contrast_rng = np.random.default_rng(
    SIZE_CONTRAST_BOOTSTRAP_SEED
)


size_contrast_rows = []


for first_size, second_size in SIZE_COMPARISONS:

    bootstrap_differences = np.empty(
        (
            SIZE_CONTRAST_BOOTSTRAP_REPLICATES,
            len(SIZE_CONTRAST_METRICS),
        ),
        dtype=float,
    )


    for bootstrap_index in range(
        SIZE_CONTRAST_BOOTSTRAP_REPLICATES
    ):

        first_size_totals = np.zeros(
            len(aggregate_columns),
            dtype=float,
        )

        second_size_totals = np.zeros(
            len(aggregate_columns),
            dtype=float,
        )


        # Preserve the three structural-regime strata.
        for regime in INSTANCE_REGIMES:

            first_array = (
                size_regime_cluster_arrays[
                    first_size
                ][
                    regime
                ]
            )

            second_array = (
                size_regime_cluster_arrays[
                    second_size
                ][
                    regime
                ]
            )


            first_sample_indices = (
                size_contrast_rng.integers(
                    low=0,
                    high=first_array.shape[0],
                    size=first_array.shape[0],
                )
            )

            second_sample_indices = (
                size_contrast_rng.integers(
                    low=0,
                    high=second_array.shape[0],
                    size=second_array.shape[0],
                )
            )


            first_size_totals += (
                first_array[
                    first_sample_indices
                ]
                .sum(
                    axis=0
                )
            )


            second_size_totals += (
                second_array[
                    second_sample_indices
                ]
                .sum(
                    axis=0
                )
            )


        first_metrics = (
            calculate_size_contrast_metrics(
                first_size_totals
            )
        )

        second_metrics = (
            calculate_size_contrast_metrics(
                second_size_totals
            )
        )


        bootstrap_differences[
            bootstrap_index
        ] = (
            first_metrics
            -
            second_metrics
        )


    # --------------------------------------------------
    # Observed differences and bootstrap intervals
    # --------------------------------------------------

    observed_difference = (
        observed_size_metrics[
            first_size
        ]
        -
        observed_size_metrics[
            second_size
        ]
    )


    bootstrap_standard_errors = (
        bootstrap_differences.std(
            axis=0,
            ddof=1,
        )
    )


    confidence_lower = np.quantile(
        bootstrap_differences,
        0.025,
        axis=0,
    )


    confidence_upper = np.quantile(
        bootstrap_differences,
        0.975,
        axis=0,
    )


    for metric_index, metric_name in enumerate(
        SIZE_CONTRAST_METRICS
    ):

        size_contrast_rows.append(
            {
                "first_size":
                    first_size,

                "second_size":
                    second_size,

                "contrast":
                    (
                        f"M={first_size}"
                        f" minus "
                        f"M={second_size}"
                    ),

                "metric":
                    metric_name,

                "estimated_difference":
                    observed_difference[
                        metric_index
                    ],

                "bootstrap_standard_error":
                    bootstrap_standard_errors[
                        metric_index
                    ],

                "ci_95_lower":
                    confidence_lower[
                        metric_index
                    ],

                "ci_95_upper":
                    confidence_upper[
                        metric_index
                    ],

                "ci_excludes_zero":
                    (
                        confidence_lower[
                            metric_index
                        ]
                        > 0.0
                    )
                    or
                    (
                        confidence_upper[
                            metric_index
                        ]
                        < 0.0
                    ),

                "bootstrap_replicates":
                    SIZE_CONTRAST_BOOTSTRAP_REPLICATES,

                "resampling_unit":
                    "independent_instance",

                "resampling_strata":
                    "structural_regime",

                "multiple_comparison_adjustment":
                    "none",
            }
        )


    print(
        "Completed size contrast:",
        f"M={first_size}",
        "minus",
        f"M={second_size}",
    )


# --------------------------------------------------
# Save complete contrast results
# --------------------------------------------------

size_contrast_df = pd.DataFrame(
    size_contrast_rows
)


size_contrast_df.to_csv(
    SIZE_CONTRAST_PATH,
    index=False,
)


# --------------------------------------------------
# Create readable display table
# --------------------------------------------------

size_contrast_display_df = (
    size_contrast_df.copy()
)


size_contrast_display_df[
    "difference_and_ci"
] = (
    size_contrast_display_df[
        "estimated_difference"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    size_contrast_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    size_contrast_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


print(
    "\n======================================"
)

print(
    "PAIRWISE CANDIDATE-SIZE CONTRASTS"
)

print(
    "======================================"
)


print(
    size_contrast_display_df[
        [
            "contrast",
            "metric",
            "difference_and_ci",
            "ci_excludes_zero",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\nCandidate-size contrasts saved to:"
)

print(
    SIZE_CONTRAST_PATH
)

In [ ]:
# --------------------------------------------------
# Weight-profile-specific stratified cluster bootstrap
# --------------------------------------------------

WEIGHT_BOOTSTRAP_REPLICATES = 5000
WEIGHT_BOOTSTRAP_SEED = GLOBAL_SEED + 88

WEIGHT_BOOTSTRAP_SUMMARY_PATH = (
    TABLES_DIR
    / "weight_profile_quality_cluster_bootstrap_ci.csv"
)


# --------------------------------------------------
# Prepare comparable condition-level observations
# --------------------------------------------------

weight_bootstrap_source_df = (
    analysis_benchmark_df[
        analysis_benchmark_df[
            "is_comparable"
        ]
        == True
    ]
    .copy()
)


weight_bootstrap_source_df[
    "greedy_optimal_cost_numeric"
] = (
    weight_bootstrap_source_df[
        "greedy_optimal_cost"
    ]
    .astype(int)
)


weight_bootstrap_source_df[
    "large_gap_numeric"
] = (
    weight_bootstrap_source_df[
        "optimality_gap_percent"
    ]
    > 10.0
).astype(int)


weight_bootstrap_source_df[
    "greedy_more_items_numeric"
] = (
    weight_bootstrap_source_df[
        "selected_count_difference"
    ]
    > 0
).astype(int)


# --------------------------------------------------
# Create sufficient statistics for each independent
# instance and objective-weight profile
# --------------------------------------------------

weight_cluster_df = (
    weight_bootstrap_source_df
    .groupby(
        [
            "instance_id",
            "n_items",
            "regime",
            "weight_profile",
        ],
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "size",
        ),

        gap_sum=(
            "optimality_gap_percent",
            "sum",
        ),

        optimal_cost_sum=(
            "greedy_optimal_cost_numeric",
            "sum",
        ),

        jaccard_sum=(
            "jaccard_similarity",
            "sum",
        ),

        large_gap_sum=(
            "large_gap_numeric",
            "sum",
        ),

        greedy_more_items_sum=(
            "greedy_more_items_numeric",
            "sum",
        ),

        selected_count_difference_sum=(
            "selected_count_difference",
            "sum",
        ),
    )
)


weight_aggregate_columns = [
    "comparable_rows",
    "gap_sum",
    "optimal_cost_sum",
    "jaccard_sum",
    "large_gap_sum",
    "greedy_more_items_sum",
    "selected_count_difference_sum",
]


weight_metric_names = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "greedy_more_items_percent",
    "mean_selected_count_difference",
]


# --------------------------------------------------
# Convert sufficient-statistic totals into metrics
# --------------------------------------------------

def calculate_weight_profile_metrics(
    totals: np.ndarray,
) -> np.ndarray:
    """
    Calculate objective-weight-profile quality metrics
    from independent-instance sufficient statistics.
    """

    comparable_rows = totals[0]

    return np.array(
        [
            totals[1]
            / comparable_rows,

            100.0
            * totals[2]
            / comparable_rows,

            totals[3]
            / comparable_rows,

            100.0
            * totals[4]
            / comparable_rows,

            100.0
            * totals[5]
            / comparable_rows,

            totals[6]
            / comparable_rows,
        ],
        dtype=float,
    )


# --------------------------------------------------
# Validate cluster coverage
# --------------------------------------------------

weight_cluster_counts = (
    weight_cluster_df
    .groupby(
        [
            "weight_profile",
            "n_items",
            "regime",
        ],
        as_index=False,
    )
    .agg(
        independent_instances=(
            "instance_id",
            "nunique",
        )
    )
)


assert (
    weight_cluster_counts[
        "independent_instances"
    ]
    == 100
).all()


# --------------------------------------------------
# Bootstrap separately for each weight profile
# --------------------------------------------------

weight_bootstrap_rng = np.random.default_rng(
    WEIGHT_BOOTSTRAP_SEED
)

weight_bootstrap_summary_rows = []


for weight_profile in WEIGHT_PROFILES:

    profile_cluster_df = (
        weight_cluster_df[
            weight_cluster_df[
                "weight_profile"
            ]
            == weight_profile
        ]
        .copy()
    )


    # ----------------------------------------------
    # Preserve the 15 size × regime strata
    # ----------------------------------------------

    profile_stratum_arrays = []


    for (
        n_items,
        regime,
    ), stratum_df in (
        profile_cluster_df
        .groupby(
            [
                "n_items",
                "regime",
            ],
            sort=True,
        )
    ):

        stratum_array = (
            stratum_df[
                weight_aggregate_columns
            ]
            .to_numpy(
                dtype=float
            )
        )


        assert (
            stratum_array.shape[0]
            == 100
        )


        profile_stratum_arrays.append(
            stratum_array
        )


    # ----------------------------------------------
    # Observed profile estimates
    # ----------------------------------------------

    observed_profile_totals = (
        profile_cluster_df[
            weight_aggregate_columns
        ]
        .sum()
        .to_numpy(
            dtype=float
        )
    )


    observed_profile_metrics = (
        calculate_weight_profile_metrics(
            observed_profile_totals
        )
    )


    # ----------------------------------------------
    # Stratified independent-instance bootstrap
    # ----------------------------------------------

    profile_bootstrap_estimates = np.empty(
        (
            WEIGHT_BOOTSTRAP_REPLICATES,
            len(weight_metric_names),
        ),
        dtype=float,
    )


    for bootstrap_index in range(
        WEIGHT_BOOTSTRAP_REPLICATES
    ):

        resampled_totals = np.zeros(
            len(weight_aggregate_columns),
            dtype=float,
        )


        for stratum_array in profile_stratum_arrays:

            number_of_instances = (
                stratum_array.shape[0]
            )


            sampled_indices = (
                weight_bootstrap_rng.integers(
                    low=0,
                    high=number_of_instances,
                    size=number_of_instances,
                )
            )


            resampled_totals += (
                stratum_array[
                    sampled_indices
                ]
                .sum(
                    axis=0
                )
            )


        profile_bootstrap_estimates[
            bootstrap_index
        ] = calculate_weight_profile_metrics(
            resampled_totals
        )


    # ----------------------------------------------
    # Bootstrap uncertainty
    # ----------------------------------------------

    profile_standard_errors = (
        profile_bootstrap_estimates.std(
            axis=0,
            ddof=1,
        )
    )


    profile_ci_lower = np.quantile(
        profile_bootstrap_estimates,
        0.025,
        axis=0,
    )


    profile_ci_upper = np.quantile(
        profile_bootstrap_estimates,
        0.975,
        axis=0,
    )


    for metric_index, metric_name in enumerate(
        weight_metric_names
    ):

        weight_bootstrap_summary_rows.append(
            {
                "weight_profile":
                    weight_profile,

                "metric":
                    metric_name,

                "estimate":
                    observed_profile_metrics[
                        metric_index
                    ],

                "bootstrap_standard_error":
                    profile_standard_errors[
                        metric_index
                    ],

                "ci_95_lower":
                    profile_ci_lower[
                        metric_index
                    ],

                "ci_95_upper":
                    profile_ci_upper[
                        metric_index
                    ],

                "independent_instances":
                    len(
                        profile_cluster_df
                    ),

                "experimental_strata":
                    len(
                        profile_stratum_arrays
                    ),

                "bootstrap_replicates":
                    WEIGHT_BOOTSTRAP_REPLICATES,

                "resampling_unit":
                    "independent_instance",

                "resampling_design":
                    "stratified_by_size_and_regime",
            }
        )


    print(
        "Completed weight profile:",
        weight_profile
    )


# --------------------------------------------------
# Save complete bootstrap summary
# --------------------------------------------------

weight_bootstrap_summary_df = pd.DataFrame(
    weight_bootstrap_summary_rows
)


weight_bootstrap_summary_df.to_csv(
    WEIGHT_BOOTSTRAP_SUMMARY_PATH,
    index=False,
)


# --------------------------------------------------
# Create compact display
# --------------------------------------------------

key_weight_metrics = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
]


weight_bootstrap_display_df = (
    weight_bootstrap_summary_df[
        weight_bootstrap_summary_df[
            "metric"
        ].isin(
            key_weight_metrics
        )
    ]
    .copy()
)


weight_bootstrap_display_df[
    "estimate_and_ci"
] = (
    weight_bootstrap_display_df[
        "estimate"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    weight_bootstrap_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    weight_bootstrap_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


weight_bootstrap_pivot = (
    weight_bootstrap_display_df
    .pivot(
        index="weight_profile",
        columns="metric",
        values="estimate_and_ci",
    )
    .reset_index()
)


weight_profile_order = [
    "balanced",
    "privacy_priority",
    "interpretation_priority",
    "parsimony_priority",
]


weight_bootstrap_pivot[
    "weight_profile"
] = pd.Categorical(
    weight_bootstrap_pivot[
        "weight_profile"
    ],
    categories=weight_profile_order,
    ordered=True,
)


weight_bootstrap_pivot = (
    weight_bootstrap_pivot
    .sort_values(
        "weight_profile"
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n======================================"
)

print(
    "WEIGHT-PROFILE-SPECIFIC CLUSTERED 95% CIs"
)

print(
    "======================================"
)


print(
    weight_bootstrap_pivot.to_string(
        index=False
    )
)


print(
    "\nFull weight-profile bootstrap table saved to:"
)

print(
    WEIGHT_BOOTSTRAP_SUMMARY_PATH
)

In [ ]:
# --------------------------------------------------
# Weight-profile-specific stratified cluster bootstrap
# --------------------------------------------------

WEIGHT_BOOTSTRAP_REPLICATES = 5000
WEIGHT_BOOTSTRAP_SEED = GLOBAL_SEED + 88

WEIGHT_BOOTSTRAP_SUMMARY_PATH = (
    TABLES_DIR
    / "weight_profile_quality_cluster_bootstrap_ci.csv"
)


# --------------------------------------------------
# Prepare comparable condition-level observations
# --------------------------------------------------

weight_bootstrap_source_df = (
    analysis_benchmark_df[
        analysis_benchmark_df[
            "is_comparable"
        ]
        == True
    ]
    .copy()
)


weight_bootstrap_source_df[
    "greedy_optimal_cost_numeric"
] = (
    weight_bootstrap_source_df[
        "greedy_optimal_cost"
    ]
    .astype(int)
)


weight_bootstrap_source_df[
    "large_gap_numeric"
] = (
    weight_bootstrap_source_df[
        "optimality_gap_percent"
    ]
    > 10.0
).astype(int)


weight_bootstrap_source_df[
    "greedy_more_items_numeric"
] = (
    weight_bootstrap_source_df[
        "selected_count_difference"
    ]
    > 0
).astype(int)


# --------------------------------------------------
# Create sufficient statistics for each independent
# instance and objective-weight profile
# --------------------------------------------------

weight_cluster_df = (
    weight_bootstrap_source_df
    .groupby(
        [
            "instance_id",
            "n_items",
            "regime",
            "weight_profile",
        ],
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "size",
        ),

        gap_sum=(
            "optimality_gap_percent",
            "sum",
        ),

        optimal_cost_sum=(
            "greedy_optimal_cost_numeric",
            "sum",
        ),

        jaccard_sum=(
            "jaccard_similarity",
            "sum",
        ),

        large_gap_sum=(
            "large_gap_numeric",
            "sum",
        ),

        greedy_more_items_sum=(
            "greedy_more_items_numeric",
            "sum",
        ),

        selected_count_difference_sum=(
            "selected_count_difference",
            "sum",
        ),
    )
)


weight_aggregate_columns = [
    "comparable_rows",
    "gap_sum",
    "optimal_cost_sum",
    "jaccard_sum",
    "large_gap_sum",
    "greedy_more_items_sum",
    "selected_count_difference_sum",
]


weight_metric_names = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "greedy_more_items_percent",
    "mean_selected_count_difference",
]


# --------------------------------------------------
# Convert sufficient-statistic totals into metrics
# --------------------------------------------------

def calculate_weight_profile_metrics(
    totals: np.ndarray,
) -> np.ndarray:
    """
    Calculate objective-weight-profile quality metrics
    from independent-instance sufficient statistics.
    """

    comparable_rows = totals[0]

    return np.array(
        [
            totals[1]
            / comparable_rows,

            100.0
            * totals[2]
            / comparable_rows,

            totals[3]
            / comparable_rows,

            100.0
            * totals[4]
            / comparable_rows,

            100.0
            * totals[5]
            / comparable_rows,

            totals[6]
            / comparable_rows,
        ],
        dtype=float,
    )


# --------------------------------------------------
# Validate cluster coverage
# --------------------------------------------------

weight_cluster_counts = (
    weight_cluster_df
    .groupby(
        [
            "weight_profile",
            "n_items",
            "regime",
        ],
        as_index=False,
    )
    .agg(
        independent_instances=(
            "instance_id",
            "nunique",
        )
    )
)


assert (
    weight_cluster_counts[
        "independent_instances"
    ]
    == 100
).all()


# --------------------------------------------------
# Bootstrap separately for each weight profile
# --------------------------------------------------

weight_bootstrap_rng = np.random.default_rng(
    WEIGHT_BOOTSTRAP_SEED
)

weight_bootstrap_summary_rows = []


for weight_profile in WEIGHT_PROFILES:

    profile_cluster_df = (
        weight_cluster_df[
            weight_cluster_df[
                "weight_profile"
            ]
            == weight_profile
        ]
        .copy()
    )


    # ----------------------------------------------
    # Preserve the 15 size × regime strata
    # ----------------------------------------------

    profile_stratum_arrays = []


    for (
        n_items,
        regime,
    ), stratum_df in (
        profile_cluster_df
        .groupby(
            [
                "n_items",
                "regime",
            ],
            sort=True,
        )
    ):

        stratum_array = (
            stratum_df[
                weight_aggregate_columns
            ]
            .to_numpy(
                dtype=float
            )
        )


        assert (
            stratum_array.shape[0]
            == 100
        )


        profile_stratum_arrays.append(
            stratum_array
        )


    # ----------------------------------------------
    # Observed profile estimates
    # ----------------------------------------------

    observed_profile_totals = (
        profile_cluster_df[
            weight_aggregate_columns
        ]
        .sum()
        .to_numpy(
            dtype=float
        )
    )


    observed_profile_metrics = (
        calculate_weight_profile_metrics(
            observed_profile_totals
        )
    )


    # ----------------------------------------------
    # Stratified independent-instance bootstrap
    # ----------------------------------------------

    profile_bootstrap_estimates = np.empty(
        (
            WEIGHT_BOOTSTRAP_REPLICATES,
            len(weight_metric_names),
        ),
        dtype=float,
    )


    for bootstrap_index in range(
        WEIGHT_BOOTSTRAP_REPLICATES
    ):

        resampled_totals = np.zeros(
            len(weight_aggregate_columns),
            dtype=float,
        )


        for stratum_array in profile_stratum_arrays:

            number_of_instances = (
                stratum_array.shape[0]
            )


            sampled_indices = (
                weight_bootstrap_rng.integers(
                    low=0,
                    high=number_of_instances,
                    size=number_of_instances,
                )
            )


            resampled_totals += (
                stratum_array[
                    sampled_indices
                ]
                .sum(
                    axis=0
                )
            )


        profile_bootstrap_estimates[
            bootstrap_index
        ] = calculate_weight_profile_metrics(
            resampled_totals
        )


    # ----------------------------------------------
    # Bootstrap uncertainty
    # ----------------------------------------------

    profile_standard_errors = (
        profile_bootstrap_estimates.std(
            axis=0,
            ddof=1,
        )
    )


    profile_ci_lower = np.quantile(
        profile_bootstrap_estimates,
        0.025,
        axis=0,
    )


    profile_ci_upper = np.quantile(
        profile_bootstrap_estimates,
        0.975,
        axis=0,
    )


    for metric_index, metric_name in enumerate(
        weight_metric_names
    ):

        weight_bootstrap_summary_rows.append(
            {
                "weight_profile":
                    weight_profile,

                "metric":
                    metric_name,

                "estimate":
                    observed_profile_metrics[
                        metric_index
                    ],

                "bootstrap_standard_error":
                    profile_standard_errors[
                        metric_index
                    ],

                "ci_95_lower":
                    profile_ci_lower[
                        metric_index
                    ],

                "ci_95_upper":
                    profile_ci_upper[
                        metric_index
                    ],

                "independent_instances":
                    len(
                        profile_cluster_df
                    ),

                "experimental_strata":
                    len(
                        profile_stratum_arrays
                    ),

                "bootstrap_replicates":
                    WEIGHT_BOOTSTRAP_REPLICATES,

                "resampling_unit":
                    "independent_instance",

                "resampling_design":
                    "stratified_by_size_and_regime",
            }
        )


    print(
        "Completed weight profile:",
        weight_profile
    )


# --------------------------------------------------
# Save complete bootstrap summary
# --------------------------------------------------

weight_bootstrap_summary_df = pd.DataFrame(
    weight_bootstrap_summary_rows
)


weight_bootstrap_summary_df.to_csv(
    WEIGHT_BOOTSTRAP_SUMMARY_PATH,
    index=False,
)


# --------------------------------------------------
# Create compact display
# --------------------------------------------------

key_weight_metrics = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
]


weight_bootstrap_display_df = (
    weight_bootstrap_summary_df[
        weight_bootstrap_summary_df[
            "metric"
        ].isin(
            key_weight_metrics
        )
    ]
    .copy()
)


weight_bootstrap_display_df[
    "estimate_and_ci"
] = (
    weight_bootstrap_display_df[
        "estimate"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    weight_bootstrap_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    weight_bootstrap_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


weight_bootstrap_pivot = (
    weight_bootstrap_display_df
    .pivot(
        index="weight_profile",
        columns="metric",
        values="estimate_and_ci",
    )
    .reset_index()
)


weight_profile_order = [
    "balanced",
    "privacy_priority",
    "interpretation_priority",
    "parsimony_priority",
]


weight_bootstrap_pivot[
    "weight_profile"
] = pd.Categorical(
    weight_bootstrap_pivot[
        "weight_profile"
    ],
    categories=weight_profile_order,
    ordered=True,
)


weight_bootstrap_pivot = (
    weight_bootstrap_pivot
    .sort_values(
        "weight_profile"
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n======================================"
)

print(
    "WEIGHT-PROFILE-SPECIFIC CLUSTERED 95% CIs"
)

print(
    "======================================"
)


print(
    weight_bootstrap_pivot.to_string(
        index=False
    )
)


print(
    "\nFull weight-profile bootstrap table saved to:"
)

print(
    WEIGHT_BOOTSTRAP_SUMMARY_PATH
)

In [ ]:
# --------------------------------------------------
# Paired objective-weight-profile bootstrap contrasts
# --------------------------------------------------

WEIGHT_CONTRAST_BOOTSTRAP_REPLICATES = 5000
WEIGHT_CONTRAST_BOOTSTRAP_SEED = GLOBAL_SEED + 89

WEIGHT_CONTRAST_PATH = (
    TABLES_DIR
    / "pairwise_weight_profile_bootstrap_contrasts.csv"
)


# Each contrast is defined as:
# first profile minus second profile.
WEIGHT_PROFILE_COMPARISONS = [
    (
        "privacy_priority",
        "balanced",
    ),
    (
        "interpretation_priority",
        "balanced",
    ),
    (
        "parsimony_priority",
        "balanced",
    ),
    (
        "privacy_priority",
        "interpretation_priority",
    ),
    (
        "privacy_priority",
        "parsimony_priority",
    ),
    (
        "interpretation_priority",
        "parsimony_priority",
    ),
]


WEIGHT_CONTRAST_METRICS = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "greedy_more_items_percent",
]


# --------------------------------------------------
# Convert sufficient-statistic totals into metrics
# --------------------------------------------------

def calculate_weight_contrast_metrics(
    totals: np.ndarray,
) -> np.ndarray:
    """
    Calculate weight-profile performance metrics from
    independent-instance sufficient-statistic totals.
    """

    comparable_rows = totals[0]

    return np.array(
        [
            totals[1]
            / comparable_rows,

            100.0
            * totals[2]
            / comparable_rows,

            totals[3]
            / comparable_rows,

            100.0
            * totals[4]
            / comparable_rows,

            100.0
            * totals[5]
            / comparable_rows,
        ],
        dtype=float,
    )


# --------------------------------------------------
# Validate paired profile coverage
# --------------------------------------------------

profile_instance_counts = (
    weight_cluster_df
    .groupby(
        "weight_profile"
    )[
        "instance_id"
    ]
    .nunique()
)


assert (
    profile_instance_counts
    == 1500
).all()


# --------------------------------------------------
# Observed profile-level metrics
# --------------------------------------------------

observed_weight_metrics = {}


for weight_profile in WEIGHT_PROFILES:

    profile_totals = (
        weight_cluster_df[
            weight_cluster_df[
                "weight_profile"
            ]
            == weight_profile
        ][
            weight_aggregate_columns
        ]
        .sum()
        .to_numpy(
            dtype=float
        )
    )

    observed_weight_metrics[
        weight_profile
    ] = calculate_weight_contrast_metrics(
        profile_totals
    )


# --------------------------------------------------
# Paired stratified cluster bootstrap
#
# The same sampled instance indices are applied to
# both profiles within every size × regime stratum.
# --------------------------------------------------

weight_contrast_rng = np.random.default_rng(
    WEIGHT_CONTRAST_BOOTSTRAP_SEED
)

weight_contrast_rows = []


for first_profile, second_profile in (
    WEIGHT_PROFILE_COMPARISONS
):

    bootstrap_differences = np.empty(
        (
            WEIGHT_CONTRAST_BOOTSTRAP_REPLICATES,
            len(WEIGHT_CONTRAST_METRICS),
        ),
        dtype=float,
    )


    # Store paired arrays for all 15 experimental strata.
    paired_stratum_arrays = []


    for n_items in EXACT_BENCHMARK_SIZES:

        for regime in INSTANCE_REGIMES:

            first_stratum_df = (
                weight_cluster_df[
                    (
                        weight_cluster_df[
                            "weight_profile"
                        ]
                        == first_profile
                    )
                    &
                    (
                        weight_cluster_df[
                            "n_items"
                        ]
                        == n_items
                    )
                    &
                    (
                        weight_cluster_df[
                            "regime"
                        ]
                        == regime
                    )
                ]
                .sort_values(
                    "instance_id"
                )
                .reset_index(
                    drop=True
                )
            )


            second_stratum_df = (
                weight_cluster_df[
                    (
                        weight_cluster_df[
                            "weight_profile"
                        ]
                        == second_profile
                    )
                    &
                    (
                        weight_cluster_df[
                            "n_items"
                        ]
                        == n_items
                    )
                    &
                    (
                        weight_cluster_df[
                            "regime"
                        ]
                        == regime
                    )
                ]
                .sort_values(
                    "instance_id"
                )
                .reset_index(
                    drop=True
                )
            )


            # Confirm exact instance-level pairing.
            assert len(first_stratum_df) == 100
            assert len(second_stratum_df) == 100

            assert (
                first_stratum_df[
                    "instance_id"
                ].tolist()
                ==
                second_stratum_df[
                    "instance_id"
                ].tolist()
            )


            first_array = (
                first_stratum_df[
                    weight_aggregate_columns
                ]
                .to_numpy(
                    dtype=float
                )
            )

            second_array = (
                second_stratum_df[
                    weight_aggregate_columns
                ]
                .to_numpy(
                    dtype=float
                )
            )


            paired_stratum_arrays.append(
                (
                    first_array,
                    second_array,
                )
            )


    # ----------------------------------------------
    # Bootstrap paired instance clusters
    # ----------------------------------------------

    for bootstrap_index in range(
        WEIGHT_CONTRAST_BOOTSTRAP_REPLICATES
    ):

        first_totals = np.zeros(
            len(weight_aggregate_columns),
            dtype=float,
        )

        second_totals = np.zeros(
            len(weight_aggregate_columns),
            dtype=float,
        )


        for (
            first_array,
            second_array,
        ) in paired_stratum_arrays:

            number_of_instances = (
                first_array.shape[0]
            )


            # One shared sample preserves within-instance
            # pairing between the two weight profiles.
            sampled_indices = (
                weight_contrast_rng.integers(
                    low=0,
                    high=number_of_instances,
                    size=number_of_instances,
                )
            )


            first_totals += (
                first_array[
                    sampled_indices
                ]
                .sum(
                    axis=0
                )
            )

            second_totals += (
                second_array[
                    sampled_indices
                ]
                .sum(
                    axis=0
                )
            )


        first_metrics = (
            calculate_weight_contrast_metrics(
                first_totals
            )
        )

        second_metrics = (
            calculate_weight_contrast_metrics(
                second_totals
            )
        )


        bootstrap_differences[
            bootstrap_index
        ] = (
            first_metrics
            -
            second_metrics
        )


    # ----------------------------------------------
    # Observed paired contrasts and uncertainty
    # ----------------------------------------------

    observed_difference = (
        observed_weight_metrics[
            first_profile
        ]
        -
        observed_weight_metrics[
            second_profile
        ]
    )


    bootstrap_standard_errors = (
        bootstrap_differences.std(
            axis=0,
            ddof=1,
        )
    )


    confidence_lower = np.quantile(
        bootstrap_differences,
        0.025,
        axis=0,
    )


    confidence_upper = np.quantile(
        bootstrap_differences,
        0.975,
        axis=0,
    )


    for metric_index, metric_name in enumerate(
        WEIGHT_CONTRAST_METRICS
    ):

        weight_contrast_rows.append(
            {
                "first_profile":
                    first_profile,

                "second_profile":
                    second_profile,

                "contrast":
                    (
                        f"{first_profile}"
                        f" minus "
                        f"{second_profile}"
                    ),

                "metric":
                    metric_name,

                "estimated_difference":
                    observed_difference[
                        metric_index
                    ],

                "bootstrap_standard_error":
                    bootstrap_standard_errors[
                        metric_index
                    ],

                "ci_95_lower":
                    confidence_lower[
                        metric_index
                    ],

                "ci_95_upper":
                    confidence_upper[
                        metric_index
                    ],

                "ci_excludes_zero":
                    (
                        confidence_lower[
                            metric_index
                        ]
                        > 0.0
                    )
                    or
                    (
                        confidence_upper[
                            metric_index
                        ]
                        < 0.0
                    ),

                "bootstrap_replicates":
                    WEIGHT_CONTRAST_BOOTSTRAP_REPLICATES,

                "resampling_unit":
                    "independent_instance",

                "resampling_design":
                    "paired_within_size_and_regime",

                "multiple_comparison_adjustment":
                    "none",
            }
        )


    print(
        "Completed paired contrast:",
        first_profile,
        "minus",
        second_profile,
    )


# --------------------------------------------------
# Save complete contrast results
# --------------------------------------------------

weight_contrast_df = pd.DataFrame(
    weight_contrast_rows
)


weight_contrast_df.to_csv(
    WEIGHT_CONTRAST_PATH,
    index=False,
)


# --------------------------------------------------
# Create readable display table
# --------------------------------------------------

weight_contrast_display_df = (
    weight_contrast_df.copy()
)


weight_contrast_display_df[
    "difference_and_ci"
] = (
    weight_contrast_display_df[
        "estimated_difference"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    weight_contrast_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    weight_contrast_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


print(
    "\n======================================"
)

print(
    "PAIRED WEIGHT-PROFILE CONTRASTS"
)

print(
    "======================================"
)


print(
    weight_contrast_display_df[
        [
            "contrast",
            "metric",
            "difference_and_ci",
            "ci_excludes_zero",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\nPaired weight-profile contrasts saved to:"
)

print(
    WEIGHT_CONTRAST_PATH
)

In [ ]:
# --------------------------------------------------
# Threshold-specific stratified cluster bootstrap
# --------------------------------------------------

THRESHOLD_BOOTSTRAP_REPLICATES = 5000
THRESHOLD_BOOTSTRAP_SEED = GLOBAL_SEED + 90

THRESHOLD_BOOTSTRAP_SUMMARY_PATH = (
    TABLES_DIR
    / "threshold_quality_cluster_bootstrap_ci.csv"
)


# --------------------------------------------------
# Prepare comparable condition-level observations
# --------------------------------------------------

threshold_bootstrap_source_df = (
    analysis_benchmark_df[
        analysis_benchmark_df[
            "is_comparable"
        ]
        == True
    ]
    .copy()
)


threshold_bootstrap_source_df[
    "greedy_optimal_cost_numeric"
] = (
    threshold_bootstrap_source_df[
        "greedy_optimal_cost"
    ]
    .astype(int)
)


threshold_bootstrap_source_df[
    "large_gap_numeric"
] = (
    threshold_bootstrap_source_df[
        "optimality_gap_percent"
    ]
    > 10.0
).astype(int)


threshold_bootstrap_source_df[
    "greedy_more_items_numeric"
] = (
    threshold_bootstrap_source_df[
        "selected_count_difference"
    ]
    > 0
).astype(int)


# --------------------------------------------------
# Create sufficient statistics for each independent
# instance at each feasible sufficiency threshold
# --------------------------------------------------

threshold_cluster_df = (
    threshold_bootstrap_source_df
    .groupby(
        [
            "instance_id",
            "n_items",
            "regime",
            "tau",
        ],
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "size",
        ),

        gap_sum=(
            "optimality_gap_percent",
            "sum",
        ),

        optimal_cost_sum=(
            "greedy_optimal_cost_numeric",
            "sum",
        ),

        jaccard_sum=(
            "jaccard_similarity",
            "sum",
        ),

        large_gap_sum=(
            "large_gap_numeric",
            "sum",
        ),

        greedy_more_items_sum=(
            "greedy_more_items_numeric",
            "sum",
        ),

        selected_count_difference_sum=(
            "selected_count_difference",
            "sum",
        ),
    )
)


threshold_aggregate_columns = [
    "comparable_rows",
    "gap_sum",
    "optimal_cost_sum",
    "jaccard_sum",
    "large_gap_sum",
    "greedy_more_items_sum",
    "selected_count_difference_sum",
]


threshold_metric_names = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "greedy_more_items_percent",
    "mean_selected_count_difference",
]


# --------------------------------------------------
# Convert sufficient-statistic totals into metrics
# --------------------------------------------------

def calculate_threshold_metrics(
    totals: np.ndarray,
) -> np.ndarray:
    """
    Calculate quality metrics among feasible cases at
    one sufficiency threshold.
    """

    comparable_rows = totals[0]

    return np.array(
        [
            totals[1]
            / comparable_rows,

            100.0
            * totals[2]
            / comparable_rows,

            totals[3]
            / comparable_rows,

            100.0
            * totals[4]
            / comparable_rows,

            100.0
            * totals[5]
            / comparable_rows,

            totals[6]
            / comparable_rows,
        ],
        dtype=float,
    )


# --------------------------------------------------
# Bootstrap separately for each threshold
# --------------------------------------------------

threshold_bootstrap_rng = (
    np.random.default_rng(
        THRESHOLD_BOOTSTRAP_SEED
    )
)


threshold_bootstrap_summary_rows = []


for tau in TAU_LEVELS:

    tau_cluster_df = (
        threshold_cluster_df[
            threshold_cluster_df[
                "tau"
            ]
            == tau
        ]
        .copy()
    )


    # ----------------------------------------------
    # Preserve candidate-size × regime strata
    #
    # At high thresholds, some strata contain fewer
    # than 100 feasible instances. Each stratum is
    # resampled using its observed feasible count.
    # ----------------------------------------------

    tau_stratum_arrays = []
    tau_stratum_sizes = []


    for (
        n_items,
        regime,
    ), stratum_df in (
        tau_cluster_df
        .groupby(
            [
                "n_items",
                "regime",
            ],
            sort=True,
        )
    ):

        stratum_array = (
            stratum_df[
                threshold_aggregate_columns
            ]
            .to_numpy(
                dtype=float
            )
        )


        if stratum_array.shape[0] == 0:
            continue


        tau_stratum_arrays.append(
            stratum_array
        )

        tau_stratum_sizes.append(
            {
                "n_items":
                    n_items,

                "regime":
                    regime,

                "feasible_instances":
                    stratum_array.shape[0],
            }
        )


    # ----------------------------------------------
    # Observed threshold-specific estimates
    # ----------------------------------------------

    observed_tau_totals = (
        tau_cluster_df[
            threshold_aggregate_columns
        ]
        .sum()
        .to_numpy(
            dtype=float
        )
    )


    observed_tau_metrics = (
        calculate_threshold_metrics(
            observed_tau_totals
        )
    )


    # ----------------------------------------------
    # Stratified independent-instance bootstrap
    # ----------------------------------------------

    tau_bootstrap_estimates = np.empty(
        (
            THRESHOLD_BOOTSTRAP_REPLICATES,
            len(threshold_metric_names),
        ),
        dtype=float,
    )


    for bootstrap_index in range(
        THRESHOLD_BOOTSTRAP_REPLICATES
    ):

        resampled_totals = np.zeros(
            len(
                threshold_aggregate_columns
            ),
            dtype=float,
        )


        for stratum_array in tau_stratum_arrays:

            number_of_feasible_instances = (
                stratum_array.shape[0]
            )


            sampled_indices = (
                threshold_bootstrap_rng.integers(
                    low=0,
                    high=number_of_feasible_instances,
                    size=number_of_feasible_instances,
                )
            )


            resampled_totals += (
                stratum_array[
                    sampled_indices
                ]
                .sum(
                    axis=0
                )
            )


        tau_bootstrap_estimates[
            bootstrap_index
        ] = calculate_threshold_metrics(
            resampled_totals
        )


    # ----------------------------------------------
    # Bootstrap uncertainty
    # ----------------------------------------------

    tau_standard_errors = (
        tau_bootstrap_estimates.std(
            axis=0,
            ddof=1,
        )
    )


    tau_ci_lower = np.quantile(
        tau_bootstrap_estimates,
        0.025,
        axis=0,
    )


    tau_ci_upper = np.quantile(
        tau_bootstrap_estimates,
        0.975,
        axis=0,
    )


    for metric_index, metric_name in enumerate(
        threshold_metric_names
    ):

        threshold_bootstrap_summary_rows.append(
            {
                "tau":
                    tau,

                "metric":
                    metric_name,

                "estimate":
                    observed_tau_metrics[
                        metric_index
                    ],

                "bootstrap_standard_error":
                    tau_standard_errors[
                        metric_index
                    ],

                "ci_95_lower":
                    tau_ci_lower[
                        metric_index
                    ],

                "ci_95_upper":
                    tau_ci_upper[
                        metric_index
                    ],

                "feasible_independent_instances":
                    len(
                        tau_cluster_df
                    ),

                "experimental_strata":
                    len(
                        tau_stratum_arrays
                    ),

                "minimum_feasible_instances_per_stratum":
                    min(
                        item[
                            "feasible_instances"
                        ]
                        for item
                        in tau_stratum_sizes
                    ),

                "maximum_feasible_instances_per_stratum":
                    max(
                        item[
                            "feasible_instances"
                        ]
                        for item
                        in tau_stratum_sizes
                    ),

                "bootstrap_replicates":
                    THRESHOLD_BOOTSTRAP_REPLICATES,

                "resampling_unit":
                    "feasible_independent_instance",

                "estimand":
                    "quality_conditional_on_feasibility",
            }
        )


    print(
        "Completed threshold:",
        tau
    )


# --------------------------------------------------
# Save complete threshold-bootstrap summary
# --------------------------------------------------

threshold_bootstrap_summary_df = (
    pd.DataFrame(
        threshold_bootstrap_summary_rows
    )
)


threshold_bootstrap_summary_df.to_csv(
    THRESHOLD_BOOTSTRAP_SUMMARY_PATH,
    index=False,
)


# --------------------------------------------------
# Create compact display for key metrics
# --------------------------------------------------

key_threshold_metrics = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
]


threshold_bootstrap_display_df = (
    threshold_bootstrap_summary_df[
        threshold_bootstrap_summary_df[
            "metric"
        ].isin(
            key_threshold_metrics
        )
    ]
    .copy()
)


threshold_bootstrap_display_df[
    "estimate_and_ci"
] = (
    threshold_bootstrap_display_df[
        "estimate"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    threshold_bootstrap_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    threshold_bootstrap_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


threshold_bootstrap_pivot = (
    threshold_bootstrap_display_df
    .pivot(
        index="tau",
        columns="metric",
        values="estimate_and_ci",
    )
    .reset_index()
    .sort_values(
        "tau"
    )
)


print(
    "\n======================================"
)

print(
    "THRESHOLD-SPECIFIC CLUSTERED 95% CIs"
)

print(
    "======================================"
)


print(
    threshold_bootstrap_pivot.to_string(
        index=False
    )
)


print(
    "\nFeasible independent instances by threshold:"
)


print(
    threshold_bootstrap_summary_df[
        [
            "tau",
            "feasible_independent_instances",
            "minimum_feasible_instances_per_stratum",
            "maximum_feasible_instances_per_stratum",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "tau"
    )
    .to_string(
        index=False
    )
)


print(
    "\nFull threshold-bootstrap table saved to:"
)

print(
    THRESHOLD_BOOTSTRAP_SUMMARY_PATH
)

In [ ]:
# --------------------------------------------------
# Paired threshold contrasts on common feasible support
# --------------------------------------------------

THRESHOLD_CONTRAST_BOOTSTRAP_REPLICATES = 5000
THRESHOLD_CONTRAST_BOOTSTRAP_SEED = GLOBAL_SEED + 91

THRESHOLD_CONTRAST_PATH = (
    TABLES_DIR
    / "pairwise_threshold_common_support_bootstrap_contrasts.csv"
)


# Each contrast is defined as:
# higher threshold minus lower threshold.
THRESHOLD_COMPARISONS = [
    (0.80, 0.70),
    (0.90, 0.80),
    (0.95, 0.90),
    (0.95, 0.70),
]


THRESHOLD_CONTRAST_METRICS = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "greedy_more_items_percent",
    "mean_selected_count_difference",
]


# --------------------------------------------------
# Build one sufficient-statistics row per feasible
# instance and threshold
# --------------------------------------------------

threshold_contrast_source_df = (
    analysis_benchmark_df[
        analysis_benchmark_df[
            "is_comparable"
        ]
        == True
    ]
    .copy()
)


threshold_contrast_source_df[
    "greedy_optimal_cost_numeric"
] = (
    threshold_contrast_source_df[
        "greedy_optimal_cost"
    ]
    .astype(int)
)


threshold_contrast_source_df[
    "large_gap_numeric"
] = (
    threshold_contrast_source_df[
        "optimality_gap_percent"
    ]
    > 10.0
).astype(int)


threshold_contrast_source_df[
    "greedy_more_items_numeric"
] = (
    threshold_contrast_source_df[
        "selected_count_difference"
    ]
    > 0
).astype(int)


threshold_contrast_cluster_df = (
    threshold_contrast_source_df
    .groupby(
        [
            "instance_id",
            "n_items",
            "regime",
            "tau",
        ],
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "size",
        ),

        gap_sum=(
            "optimality_gap_percent",
            "sum",
        ),

        optimal_cost_sum=(
            "greedy_optimal_cost_numeric",
            "sum",
        ),

        jaccard_sum=(
            "jaccard_similarity",
            "sum",
        ),

        large_gap_sum=(
            "large_gap_numeric",
            "sum",
        ),

        greedy_more_items_sum=(
            "greedy_more_items_numeric",
            "sum",
        ),

        selected_count_difference_sum=(
            "selected_count_difference",
            "sum",
        ),
    )
)


threshold_contrast_aggregate_columns = [
    "comparable_rows",
    "gap_sum",
    "optimal_cost_sum",
    "jaccard_sum",
    "large_gap_sum",
    "greedy_more_items_sum",
    "selected_count_difference_sum",
]


# --------------------------------------------------
# Convert aggregated totals into reporting metrics
# --------------------------------------------------

def calculate_threshold_contrast_metrics(
    totals: np.ndarray,
) -> np.ndarray:
    """
    Calculate threshold-specific quality metrics from
    common-support independent-instance totals.
    """

    comparable_rows = totals[0]

    return np.array(
        [
            totals[1]
            / comparable_rows,

            100.0
            * totals[2]
            / comparable_rows,

            totals[3]
            / comparable_rows,

            100.0
            * totals[4]
            / comparable_rows,

            100.0
            * totals[5]
            / comparable_rows,

            totals[6]
            / comparable_rows,
        ],
        dtype=float,
    )


# --------------------------------------------------
# Paired common-support bootstrap
# --------------------------------------------------

threshold_contrast_rng = np.random.default_rng(
    THRESHOLD_CONTRAST_BOOTSTRAP_SEED
)

threshold_contrast_rows = []


for higher_tau, lower_tau in THRESHOLD_COMPARISONS:

    paired_strata = []

    total_common_instances = 0
    common_stratum_counts = []


    # ----------------------------------------------
    # Align identical feasible instances within every
    # candidate-size × regime stratum
    # ----------------------------------------------

    for n_items in EXACT_BENCHMARK_SIZES:

        for regime in INSTANCE_REGIMES:

            higher_df = (
                threshold_contrast_cluster_df[
                    (
                        threshold_contrast_cluster_df[
                            "tau"
                        ]
                        == higher_tau
                    )
                    &
                    (
                        threshold_contrast_cluster_df[
                            "n_items"
                        ]
                        == n_items
                    )
                    &
                    (
                        threshold_contrast_cluster_df[
                            "regime"
                        ]
                        == regime
                    )
                ]
                .copy()
            )


            lower_df = (
                threshold_contrast_cluster_df[
                    (
                        threshold_contrast_cluster_df[
                            "tau"
                        ]
                        == lower_tau
                    )
                    &
                    (
                        threshold_contrast_cluster_df[
                            "n_items"
                        ]
                        == n_items
                    )
                    &
                    (
                        threshold_contrast_cluster_df[
                            "regime"
                        ]
                        == regime
                    )
                ]
                .copy()
            )


            common_instance_ids = sorted(
                set(
                    higher_df[
                        "instance_id"
                    ]
                )
                &
                set(
                    lower_df[
                        "instance_id"
                    ]
                )
            )


            if len(common_instance_ids) == 0:
                continue


            higher_common_df = (
                higher_df[
                    higher_df[
                        "instance_id"
                    ].isin(
                        common_instance_ids
                    )
                ]
                .sort_values(
                    "instance_id"
                )
                .reset_index(
                    drop=True
                )
            )


            lower_common_df = (
                lower_df[
                    lower_df[
                        "instance_id"
                    ].isin(
                        common_instance_ids
                    )
                ]
                .sort_values(
                    "instance_id"
                )
                .reset_index(
                    drop=True
                )
            )


            assert (
                higher_common_df[
                    "instance_id"
                ].tolist()
                ==
                lower_common_df[
                    "instance_id"
                ].tolist()
            )


            higher_array = (
                higher_common_df[
                    threshold_contrast_aggregate_columns
                ]
                .to_numpy(
                    dtype=float
                )
            )


            lower_array = (
                lower_common_df[
                    threshold_contrast_aggregate_columns
                ]
                .to_numpy(
                    dtype=float
                )
            )


            paired_strata.append(
                (
                    higher_array,
                    lower_array,
                )
            )


            stratum_count = len(
                common_instance_ids
            )

            total_common_instances += (
                stratum_count
            )

            common_stratum_counts.append(
                stratum_count
            )


    # ----------------------------------------------
    # Observed common-support contrast
    # ----------------------------------------------

    observed_higher_totals = np.zeros(
        len(
            threshold_contrast_aggregate_columns
        ),
        dtype=float,
    )

    observed_lower_totals = np.zeros(
        len(
            threshold_contrast_aggregate_columns
        ),
        dtype=float,
    )


    for higher_array, lower_array in paired_strata:

        observed_higher_totals += (
            higher_array.sum(
                axis=0
            )
        )

        observed_lower_totals += (
            lower_array.sum(
                axis=0
            )
        )


    observed_higher_metrics = (
        calculate_threshold_contrast_metrics(
            observed_higher_totals
        )
    )


    observed_lower_metrics = (
        calculate_threshold_contrast_metrics(
            observed_lower_totals
        )
    )


    observed_difference = (
        observed_higher_metrics
        -
        observed_lower_metrics
    )


    # ----------------------------------------------
    # Paired bootstrap
    # ----------------------------------------------

    bootstrap_differences = np.empty(
        (
            THRESHOLD_CONTRAST_BOOTSTRAP_REPLICATES,
            len(
                THRESHOLD_CONTRAST_METRICS
            ),
        ),
        dtype=float,
    )


    for bootstrap_index in range(
        THRESHOLD_CONTRAST_BOOTSTRAP_REPLICATES
    ):

        resampled_higher_totals = np.zeros(
            len(
                threshold_contrast_aggregate_columns
            ),
            dtype=float,
        )

        resampled_lower_totals = np.zeros(
            len(
                threshold_contrast_aggregate_columns
            ),
            dtype=float,
        )


        for higher_array, lower_array in paired_strata:

            number_of_instances = (
                higher_array.shape[0]
            )


            # The same sampled indices preserve the
            # within-instance threshold pairing.
            sampled_indices = (
                threshold_contrast_rng.integers(
                    low=0,
                    high=number_of_instances,
                    size=number_of_instances,
                )
            )


            resampled_higher_totals += (
                higher_array[
                    sampled_indices
                ]
                .sum(
                    axis=0
                )
            )


            resampled_lower_totals += (
                lower_array[
                    sampled_indices
                ]
                .sum(
                    axis=0
                )
            )


        higher_metrics = (
            calculate_threshold_contrast_metrics(
                resampled_higher_totals
            )
        )


        lower_metrics = (
            calculate_threshold_contrast_metrics(
                resampled_lower_totals
            )
        )


        bootstrap_differences[
            bootstrap_index
        ] = (
            higher_metrics
            -
            lower_metrics
        )


    # ----------------------------------------------
    # Confidence intervals
    # ----------------------------------------------

    bootstrap_standard_errors = (
        bootstrap_differences.std(
            axis=0,
            ddof=1,
        )
    )


    confidence_lower = np.quantile(
        bootstrap_differences,
        0.025,
        axis=0,
    )


    confidence_upper = np.quantile(
        bootstrap_differences,
        0.975,
        axis=0,
    )


    for metric_index, metric_name in enumerate(
        THRESHOLD_CONTRAST_METRICS
    ):

        threshold_contrast_rows.append(
            {
                "higher_tau":
                    higher_tau,

                "lower_tau":
                    lower_tau,

                "contrast":
                    (
                        f"tau={higher_tau:.2f}"
                        f" minus "
                        f"tau={lower_tau:.2f}"
                    ),

                "metric":
                    metric_name,

                "estimated_difference":
                    observed_difference[
                        metric_index
                    ],

                "bootstrap_standard_error":
                    bootstrap_standard_errors[
                        metric_index
                    ],

                "ci_95_lower":
                    confidence_lower[
                        metric_index
                    ],

                "ci_95_upper":
                    confidence_upper[
                        metric_index
                    ],

                "ci_excludes_zero":
                    (
                        confidence_lower[
                            metric_index
                        ]
                        > 0.0
                    )
                    or
                    (
                        confidence_upper[
                            metric_index
                        ]
                        < 0.0
                    ),

                "common_feasible_instances":
                    total_common_instances,

                "minimum_common_instances_per_stratum":
                    min(
                        common_stratum_counts
                    ),

                "maximum_common_instances_per_stratum":
                    max(
                        common_stratum_counts
                    ),

                "bootstrap_replicates":
                    THRESHOLD_CONTRAST_BOOTSTRAP_REPLICATES,

                "resampling_unit":
                    "common_feasible_independent_instance",

                "resampling_design":
                    "paired_within_size_and_regime",

                "multiple_comparison_adjustment":
                    "none",
            }
        )


    print(
        "Completed threshold contrast:",
        f"tau={higher_tau:.2f}",
        "minus",
        f"tau={lower_tau:.2f}",
        "| common instances:",
        total_common_instances,
    )


# --------------------------------------------------
# Save complete contrast table
# --------------------------------------------------

threshold_contrast_df = pd.DataFrame(
    threshold_contrast_rows
)


threshold_contrast_df.to_csv(
    THRESHOLD_CONTRAST_PATH,
    index=False,
)


# --------------------------------------------------
# Readable display
# --------------------------------------------------

threshold_contrast_display_df = (
    threshold_contrast_df.copy()
)


threshold_contrast_display_df[
    "difference_and_ci"
] = (
    threshold_contrast_display_df[
        "estimated_difference"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    threshold_contrast_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    threshold_contrast_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


print(
    "\n======================================"
)

print(
    "PAIRED THRESHOLD CONTRASTS"
)

print(
    "======================================"
)


print(
    threshold_contrast_display_df[
        [
            "contrast",
            "metric",
            "difference_and_ci",
            "ci_excludes_zero",
            "common_feasible_instances",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\nThreshold contrasts saved to:"
)

print(
    THRESHOLD_CONTRAST_PATH
)

In [ ]:
# --------------------------------------------------
# Marginal-preserving synthetic MST generator
# --------------------------------------------------

REGIME_STRENGTH = 0.75


# Preserve the previous generator for development
# reproducibility if this cell is run for the first time.
if (
    "generate_mst_instance_legacy"
    not in globals()
):
    generate_mst_instance_legacy = (
        generate_mst_instance
    )


def _assign_burden_preserving_marginal(
    base_values: np.ndarray,
    utility_values: np.ndarray,
    regime: str,
    rng: np.random.Generator,
    regime_strength: float,
) -> np.ndarray:
    """
    Reassign an existing set of burden values while
    preserving its exact marginal distribution.

    The returned array contains exactly the same burden
    values as base_values, but their association with
    utility changes according to the requested regime.

    regime_strength controls a noisy rank-association
    mechanism. It is not assumed to equal the resulting
    Pearson or Spearman correlation exactly.
    """

    base_values = np.asarray(
        base_values,
        dtype=float,
    )

    utility_values = np.asarray(
        utility_values,
        dtype=float,
    )

    if (
        base_values.shape
        != utility_values.shape
    ):
        raise ValueError(
            "base_values and utility_values "
            "must have the same shape."
        )

    number_of_items = len(
        base_values
    )

    if number_of_items <= 1:
        return base_values.copy()


    # ----------------------------------------------
    # Independent regime
    # ----------------------------------------------
    #
    # base_values were generated independently of
    # utility_values, so their original assignment
    # already represents the independent condition.
    # ----------------------------------------------

    if regime == "independent":

        return base_values.copy()


    # ----------------------------------------------
    # Construct standardized utility ranks
    # ----------------------------------------------

    utility_order = np.argsort(
        utility_values,
        kind="mergesort",
    )

    utility_ranks = np.empty(
        number_of_items,
        dtype=float,
    )

    utility_ranks[
        utility_order
    ] = np.arange(
        number_of_items,
        dtype=float,
    )

    utility_rank_std = (
        utility_ranks.std(
            ddof=0
        )
    )

    if utility_rank_std > 0:

        standardized_utility_ranks = (
            utility_ranks
            -
            utility_ranks.mean()
        ) / utility_rank_std

    else:

        standardized_utility_ranks = (
            np.zeros(
                number_of_items,
                dtype=float,
            )
        )


    # ----------------------------------------------
    # Create a noisy ordering score
    # ----------------------------------------------

    independent_noise = rng.normal(
        loc=0.0,
        scale=1.0,
        size=number_of_items,
    )


    residual_strength = np.sqrt(
        max(
            0.0,
            1.0
            -
            regime_strength ** 2,
        )
    )


    if (
        regime
        == "utility_burden_conflict"
    ):

        direction = 1.0

    elif (
        regime
        == "utility_burden_alignment"
    ):

        direction = -1.0

    else:

        raise ValueError(
            f"Unknown regime: {regime}"
        )


    ordering_score = (
        direction
        * regime_strength
        * standardized_utility_ranks
        +
        residual_strength
        * independent_noise
    )


    # ----------------------------------------------
    # Rank-assign the unchanged burden values
    # ----------------------------------------------
    #
    # Sorting and reassigning preserves the complete
    # empirical marginal distribution exactly.
    # ----------------------------------------------

    score_order = np.argsort(
        ordering_score,
        kind="mergesort",
    )

    sorted_burden_values = np.sort(
        base_values
    )

    assigned_values = np.empty_like(
        base_values,
        dtype=float,
    )

    assigned_values[
        score_order
    ] = sorted_burden_values


    return assigned_values



def generate_mst_instance(
    n_items: int,
    regime: str,
    seed: int,
    regime_strength: float = REGIME_STRENGTH,
) -> list[mst_core.InformationItem]:
    """
    Generate one reproducible synthetic MST benchmark
    instance with regime-invariant marginal distributions.

    The three regimes use identical marginal distributions
    for utility, privacy exposure, and interpretation cost.
    They differ only in the association between utility and
    disclosure burden.

    Parameters
    ----------
    n_items
        Number of privacy-admissible candidate items.

    regime
        One of:
        - independent
        - utility_burden_conflict
        - utility_burden_alignment

    seed
        Random seed for reproducibility.

    regime_strength
        Strength of the noisy rank association in [0, 1].
        Larger values create stronger positive or negative
        utility–burden relationships.

    Returns
    -------
    list[mst_core.InformationItem]
        Generated privacy-admissible candidate items.
    """

    if n_items <= 0:
        raise ValueError(
            "n_items must be greater than zero."
        )

    if regime not in INSTANCE_REGIMES:
        raise ValueError(
            f"Unknown regime: {regime}"
        )

    if not 0.0 <= regime_strength <= 1.0:
        raise ValueError(
            "regime_strength must be in [0, 1]."
        )


    rng = np.random.default_rng(
        seed
    )


    # ----------------------------------------------
    # Modeled item utility
    # ----------------------------------------------

    utility_latent = rng.beta(
        a=2.0,
        b=2.0,
        size=n_items,
    )

    utility_values = (
        0.08
        +
        0.52
        * utility_latent
    )


    # ----------------------------------------------
    # Regime-invariant burden marginals
    # ----------------------------------------------

    base_privacy_latent = rng.beta(
        a=2.0,
        b=2.0,
        size=n_items,
    )

    base_cognitive_latent = rng.beta(
        a=2.0,
        b=2.0,
        size=n_items,
    )


    # ----------------------------------------------
    # Change association without changing marginals
    # ----------------------------------------------

    privacy_latent = (
        _assign_burden_preserving_marginal(
            base_values=
                base_privacy_latent,

            utility_values=
                utility_latent,

            regime=
                regime,

            rng=
                rng,

            regime_strength=
                regime_strength,
        )
    )


    cognitive_latent = (
        _assign_burden_preserving_marginal(
            base_values=
                base_cognitive_latent,

            utility_values=
                utility_latent,

            regime=
                regime,

            rng=
                rng,

            regime_strength=
                regime_strength,
        )
    )


    # ----------------------------------------------
    # Convert to benchmark scales
    # ----------------------------------------------

    privacy_values = (
        0.01
        +
        0.24
        * privacy_latent
    )

    cognitive_values = (
        0.10
        +
        0.90
        * cognitive_latent
    )


    # ----------------------------------------------
    # Construct information items
    # ----------------------------------------------

    items = []

    for index in range(
        n_items
    ):

        items.append(
            mst_core.InformationItem(
                item_id=
                    f"z{index + 1:03d}",

                name=(
                    "Synthetic information item "
                    f"{index + 1}"
                ),

                utility=float(
                    utility_values[
                        index
                    ]
                ),

                privacy=float(
                    privacy_values[
                        index
                    ]
                ),

                cognitive_cost=float(
                    cognitive_values[
                        index
                    ]
                ),
            )
        )


    return items


print(
    "Marginal-preserving MST generator defined."
)

print(
    "Default regime strength:",
    REGIME_STRENGTH
)

In [ ]:
# --------------------------------------------------
# Validate corrected marginal-preserving generator
# --------------------------------------------------

CORRECTED_AUDIT_INSTANCES = 500
CORRECTED_AUDIT_SIZE = 64


corrected_audit_rows = []

maximum_sorted_privacy_difference = 0.0
maximum_sorted_cognitive_difference = 0.0


for replicate in range(
    CORRECTED_AUDIT_INSTANCES
):

    instance_seed = (
        GLOBAL_SEED
        + 30_000_000
        + replicate
    )


    # Generate all regimes using the same seed.
    # Utility and burden marginal values should therefore
    # be identical across regimes; only their assignment
    # and association should change.
    generated_by_regime = {
        regime: generate_mst_instance(
            n_items=CORRECTED_AUDIT_SIZE,
            regime=regime,
            seed=instance_seed,
        )
        for regime in INSTANCE_REGIMES
    }


    independent_items = (
        generated_by_regime[
            "independent"
        ]
    )


    independent_privacy = np.array(
        [
            item.privacy
            for item in independent_items
        ],
        dtype=float,
    )

    independent_cognitive = np.array(
        [
            item.cognitive_cost
            for item in independent_items
        ],
        dtype=float,
    )


    for regime, items in (
        generated_by_regime.items()
    ):

        utility_values = np.array(
            [
                item.utility
                for item in items
            ],
            dtype=float,
        )

        privacy_values = np.array(
            [
                item.privacy
                for item in items
            ],
            dtype=float,
        )

        cognitive_values = np.array(
            [
                item.cognitive_cost
                for item in items
            ],
            dtype=float,
        )


        corrected_audit_rows.append(
            {
                "replicate":
                    replicate,

                "regime":
                    regime,

                "utility_privacy_pearson":
                    pd.Series(
                        utility_values
                    ).corr(
                        pd.Series(
                            privacy_values
                        )
                    ),

                "utility_cognitive_pearson":
                    pd.Series(
                        utility_values
                    ).corr(
                        pd.Series(
                            cognitive_values
                        )
                    ),

                "utility_privacy_rank":
                    pd.Series(
                        utility_values
                    ).corr(
                        pd.Series(
                            privacy_values
                        ),
                        method="spearman",
                    ),

                "utility_cognitive_rank":
                    pd.Series(
                        utility_values
                    ).corr(
                        pd.Series(
                            cognitive_values
                        ),
                        method="spearman",
                    ),

                "privacy_mean":
                    privacy_values.mean(),

                "privacy_sd":
                    privacy_values.std(
                        ddof=1
                    ),

                "cognitive_mean":
                    cognitive_values.mean(),

                "cognitive_sd":
                    cognitive_values.std(
                        ddof=1
                    ),
            }
        )


        # Exact marginal-preservation checks.
        maximum_sorted_privacy_difference = max(
            maximum_sorted_privacy_difference,
            float(
                np.max(
                    np.abs(
                        np.sort(
                            privacy_values
                        )
                        -
                        np.sort(
                            independent_privacy
                        )
                    )
                )
            ),
        )


        maximum_sorted_cognitive_difference = max(
            maximum_sorted_cognitive_difference,
            float(
                np.max(
                    np.abs(
                        np.sort(
                            cognitive_values
                        )
                        -
                        np.sort(
                            independent_cognitive
                        )
                    )
                )
            ),
        )


corrected_audit_df = pd.DataFrame(
    corrected_audit_rows
)


corrected_regime_summary = (
    corrected_audit_df
    .groupby(
        "regime",
        as_index=False,
    )
    .agg(
        mean_utility_privacy_pearson=(
            "utility_privacy_pearson",
            "mean",
        ),

        mean_utility_cognitive_pearson=(
            "utility_cognitive_pearson",
            "mean",
        ),

        mean_utility_privacy_rank=(
            "utility_privacy_rank",
            "mean",
        ),

        mean_utility_cognitive_rank=(
            "utility_cognitive_rank",
            "mean",
        ),

        mean_privacy=(
            "privacy_mean",
            "mean",
        ),

        mean_privacy_sd=(
            "privacy_sd",
            "mean",
        ),

        mean_cognitive=(
            "cognitive_mean",
            "mean",
        ),

        mean_cognitive_sd=(
            "cognitive_sd",
            "mean",
        ),
    )
)


print(
    "======================================"
)

print(
    "CORRECTED GENERATOR VALIDATION"
)

print(
    "======================================"
)


print(
    corrected_regime_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)


print(
    "\nMaximum sorted privacy difference:",
    f"{maximum_sorted_privacy_difference:.16e}"
)

print(
    "Maximum sorted cognitive-cost difference:",
    f"{maximum_sorted_cognitive_difference:.16e}"
)

In [ ]:
# --------------------------------------------------
# Save corrected-generator validation results
# --------------------------------------------------

GENERATOR_VERSION = (
    "marginal_preserving_v2"
)

CORRECTED_AUDIT_RAW_PATH = (
    RESULTS_DIR
    / "corrected_generator_validation_raw.csv"
)

CORRECTED_AUDIT_SUMMARY_PATH = (
    RESULTS_DIR
    / "corrected_generator_validation_summary.csv"
)

CORRECTED_GENERATOR_METADATA_PATH = (
    RESULTS_DIR
    / "corrected_generator_metadata.csv"
)


# --------------------------------------------------
# Save raw replicate-level validation results
# --------------------------------------------------

corrected_audit_df.to_csv(
    CORRECTED_AUDIT_RAW_PATH,
    index=False,
)


# --------------------------------------------------
# Save regime-level validation summary
# --------------------------------------------------

corrected_regime_summary.to_csv(
    CORRECTED_AUDIT_SUMMARY_PATH,
    index=False,
)


# --------------------------------------------------
# Save generator configuration and audit metadata
# --------------------------------------------------

corrected_generator_metadata_df = pd.DataFrame(
    [
        {
            "generator_version":
                GENERATOR_VERSION,

            "regime_strength":
                REGIME_STRENGTH,

            "audit_instances":
                CORRECTED_AUDIT_INSTANCES,

            "items_per_audit_instance":
                CORRECTED_AUDIT_SIZE,

            "utility_distribution":
                "Beta(2,2) scaled to [0.08,0.60]",

            "privacy_distribution":
                "Beta(2,2) scaled to [0.01,0.25]",

            "cognitive_cost_distribution":
                "Beta(2,2) scaled to [0.10,1.00]",

            "association_method":
                "noisy rank reassignment",

            "marginal_preservation":
                "exact within matched seed",

            "maximum_sorted_privacy_difference":
                maximum_sorted_privacy_difference,

            "maximum_sorted_cognitive_difference":
                maximum_sorted_cognitive_difference,

            "intended_interpretation":
                (
                    "Controlled computational benchmark; "
                    "parameters are not empirical human estimates."
                ),
        }
    ]
)


corrected_generator_metadata_df.to_csv(
    CORRECTED_GENERATOR_METADATA_PATH,
    index=False,
)


print(
    "======================================"
)

print(
    "CORRECTED GENERATOR VALIDATION SAVED"
)

print(
    "======================================"
)

print(
    "Generator version:",
    GENERATOR_VERSION
)

print(
    "\nRaw audit results:"
)

print(
    CORRECTED_AUDIT_RAW_PATH
)

print(
    "\nValidation summary:"
)

print(
    CORRECTED_AUDIT_SUMMARY_PATH
)

print(
    "\nGenerator metadata:"
)

print(
    CORRECTED_GENERATOR_METADATA_PATH
)

In [ ]:
# --------------------------------------------------
# Validate matched-regime benchmark construction
# --------------------------------------------------

MATCHED_TEST_SIZE = 12
MATCHED_TEST_REPLICATE = 0

MATCHED_TEST_SEED = (
    GLOBAL_SEED
    + 40_000_000
    + EXACT_BENCHMARK_SIZES.index(
        MATCHED_TEST_SIZE
    )
    * 1_000_000
    + MATCHED_TEST_REPLICATE
)


# --------------------------------------------------
# Generate the same latent instance under all regimes
# --------------------------------------------------

matched_test_instances = {
    regime: generate_mst_instance(
        n_items=MATCHED_TEST_SIZE,
        regime=regime,
        seed=MATCHED_TEST_SEED,
    )
    for regime in INSTANCE_REGIMES
}


def extract_instance_arrays(items):
    """
    Convert an information-item collection into arrays
    for matched-regime validation.
    """

    return {
        "utility": np.array(
            [
                item.utility
                for item in items
            ],
            dtype=float,
        ),

        "privacy": np.array(
            [
                item.privacy
                for item in items
            ],
            dtype=float,
        ),

        "cognitive_cost": np.array(
            [
                item.cognitive_cost
                for item in items
            ],
            dtype=float,
        ),
    }


matched_arrays = {
    regime: extract_instance_arrays(
        items
    )
    for regime, items
    in matched_test_instances.items()
}


reference_regime = "independent"

reference_arrays = matched_arrays[
    reference_regime
]


validation_rows = []


for regime in INSTANCE_REGIMES:

    current_arrays = matched_arrays[
        regime
    ]

    utility_match = np.allclose(
        current_arrays[
            "utility"
        ],
        reference_arrays[
            "utility"
        ],
        atol=0.0,
        rtol=0.0,
    )

    privacy_marginal_match = np.allclose(
        np.sort(
            current_arrays[
                "privacy"
            ]
        ),
        np.sort(
            reference_arrays[
                "privacy"
            ]
        ),
        atol=0.0,
        rtol=0.0,
    )

    cognitive_marginal_match = np.allclose(
        np.sort(
            current_arrays[
                "cognitive_cost"
            ]
        ),
        np.sort(
            reference_arrays[
                "cognitive_cost"
            ]
        ),
        atol=0.0,
        rtol=0.0,
    )


    utility_privacy_rank_correlation = (
        pd.Series(
            current_arrays[
                "utility"
            ]
        )
        .corr(
            pd.Series(
                current_arrays[
                    "privacy"
                ]
            ),
            method="spearman",
        )
    )


    utility_cognitive_rank_correlation = (
        pd.Series(
            current_arrays[
                "utility"
            ]
        )
        .corr(
            pd.Series(
                current_arrays[
                    "cognitive_cost"
                ]
            ),
            method="spearman",
        )
    )


    validation_rows.append(
        {
            "regime":
                regime,

            "seed":
                MATCHED_TEST_SEED,

            "utility_values_match":
                utility_match,

            "privacy_marginal_match":
                privacy_marginal_match,

            "cognitive_marginal_match":
                cognitive_marginal_match,

            "utility_privacy_rank_correlation":
                utility_privacy_rank_correlation,

            "utility_cognitive_rank_correlation":
                utility_cognitive_rank_correlation,
        }
    )


matched_regime_validation_df = pd.DataFrame(
    validation_rows
)


# --------------------------------------------------
# Run all 16 benchmark conditions for each regime
# --------------------------------------------------

matched_benchmark_rows = []


for regime in INSTANCE_REGIMES:

    regime_rows = (
        run_cached_instance_benchmark(
            n_items=MATCHED_TEST_SIZE,
            regime=regime,
            seed=MATCHED_TEST_SEED,
        )
    )


    for row in regime_rows:

        row[
            "matched_test_replicate"
        ] = MATCHED_TEST_REPLICATE


    matched_benchmark_rows.extend(
        regime_rows
    )


matched_benchmark_test_df = pd.DataFrame(
    matched_benchmark_rows
)


# --------------------------------------------------
# Integrity checks
# --------------------------------------------------

assert (
    matched_regime_validation_df[
        "utility_values_match"
    ]
    .all()
)

assert (
    matched_regime_validation_df[
        "privacy_marginal_match"
    ]
    .all()
)

assert (
    matched_regime_validation_df[
        "cognitive_marginal_match"
    ]
    .all()
)

assert len(
    matched_benchmark_test_df
) == (
    len(INSTANCE_REGIMES)
    *
    len(TAU_LEVELS)
    *
    len(WEIGHT_PROFILES)
)

assert (
    matched_benchmark_test_df[
        "seed"
    ].nunique()
    == 1
)

assert (
    matched_benchmark_test_df[
        "optimality_gap"
    ]
    .dropna()
    >= -1e-12
).all()


# --------------------------------------------------
# Print validation results
# --------------------------------------------------

print(
    "======================================"
)

print(
    "MATCHED-REGIME BENCHMARK VALIDATION"
)

print(
    "======================================"
)

print(
    "Matched seed:",
    MATCHED_TEST_SEED
)

print(
    "Candidate-set size:",
    MATCHED_TEST_SIZE
)

print(
    "\nMarginal and association checks:"
)

print(
    matched_regime_validation_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

print(
    "\nBenchmark rows:",
    len(
        matched_benchmark_test_df
    )
)

print(
    "Expected benchmark rows:",
    (
        len(INSTANCE_REGIMES)
        *
        len(TAU_LEVELS)
        *
        len(WEIGHT_PROFILES)
    )
)

print(
    "Unique seeds across regimes:",
    matched_benchmark_test_df[
        "seed"
    ].nunique()
)

print(
    "Negative optimality gaps:",
    (
        matched_benchmark_test_df[
            "optimality_gap"
        ]
        .dropna()
        < -1e-12
    ).sum()
)

print(
    "Exact threshold failures:",
    (
        (
            matched_benchmark_test_df[
                "exact_status"
            ]
            == "OPTIMAL"
        )
        &
        (
            matched_benchmark_test_df[
                "exact_utility"
            ]
            <
            matched_benchmark_test_df[
                "tau"
            ]
        )
    ).sum()
)

print(
    "Greedy threshold failures:",
    (
        (
            matched_benchmark_test_df[
                "greedy_status"
            ]
            == "FEASIBLE"
        )
        &
        (
            matched_benchmark_test_df[
                "greedy_utility"
            ]
            <
            matched_benchmark_test_df[
                "tau"
            ]
        )
    ).sum()
)

In [ ]:
# --------------------------------------------------
# Corrected marginal-preserving Exact-vs-Greedy benchmark
# --------------------------------------------------

import time


CORRECTED_MAIN_REPLICATES = 100

CORRECTED_GENERATOR_VERSION = (
    "marginal_preserving_v2"
)


CORRECTED_CHECKPOINT_PATH = (
    RESULTS_DIR
    / (
        "exact_vs_greedy_main_benchmark_"
        "v2_marginal_preserving_checkpoint.csv"
    )
)

CORRECTED_FINAL_PATH = (
    RESULTS_DIR
    / (
        "exact_vs_greedy_main_benchmark_"
        "v2_marginal_preserving.csv"
    )
)


# --------------------------------------------------
# Expected experimental dimensions
# --------------------------------------------------

EXPECTED_CONDITIONS_PER_INSTANCE = (
    len(TAU_LEVELS)
    * len(WEIGHT_PROFILES)
)

EXPECTED_REGIMES_PER_MATCHED_GROUP = (
    len(INSTANCE_REGIMES)
)

EXPECTED_ROWS_PER_MATCHED_GROUP = (
    EXPECTED_CONDITIONS_PER_INSTANCE
    * EXPECTED_REGIMES_PER_MATCHED_GROUP
)

TOTAL_CORRECTED_MATCHED_GROUPS = (
    len(EXACT_BENCHMARK_SIZES)
    * CORRECTED_MAIN_REPLICATES
)

TOTAL_CORRECTED_INSTANCES = (
    TOTAL_CORRECTED_MATCHED_GROUPS
    * len(INSTANCE_REGIMES)
)

TOTAL_CORRECTED_ROWS = (
    TOTAL_CORRECTED_INSTANCES
    * EXPECTED_CONDITIONS_PER_INSTANCE
)


# --------------------------------------------------
# Load an existing final file or checkpoint
# --------------------------------------------------

if CORRECTED_FINAL_PATH.exists():

    loading_path = (
        CORRECTED_FINAL_PATH
    )

    print(
        "Existing corrected final dataset found."
    )

elif CORRECTED_CHECKPOINT_PATH.exists():

    loading_path = (
        CORRECTED_CHECKPOINT_PATH
    )

    print(
        "Existing corrected checkpoint found."
    )

else:

    loading_path = None

    print(
        "No corrected checkpoint found."
    )

    print(
        "Starting a new corrected benchmark."
    )


if loading_path is not None:

    loaded_corrected_df = pd.read_csv(
        loading_path
    )


    # Retain only complete 16-condition instances.
    loaded_instance_counts = (
        loaded_corrected_df
        .groupby(
            "instance_id"
        )
        .size()
    )


    complete_loaded_instance_ids = set(
        loaded_instance_counts[
            loaded_instance_counts
            == EXPECTED_CONDITIONS_PER_INSTANCE
        ].index
    )


    incomplete_loaded_instance_ids = set(
        loaded_instance_counts[
            loaded_instance_counts
            != EXPECTED_CONDITIONS_PER_INSTANCE
        ].index
    )


    corrected_main_df = (
        loaded_corrected_df[
            loaded_corrected_df[
                "instance_id"
            ].isin(
                complete_loaded_instance_ids
            )
        ]
        .copy()
    )


    corrected_main_rows = (
        corrected_main_df.to_dict(
            "records"
        )
    )


    completed_instance_ids = (
        complete_loaded_instance_ids
    )


    print(
        "Complete instances loaded:",
        len(
            completed_instance_ids
        )
    )

    print(
        "Incomplete instances discarded:",
        len(
            incomplete_loaded_instance_ids
        )
    )

else:

    corrected_main_rows = []

    completed_instance_ids = set()


# --------------------------------------------------
# Experiment information
# --------------------------------------------------

print(
    "\nGenerator version:",
    CORRECTED_GENERATOR_VERSION
)

print(
    "Regime strength:",
    REGIME_STRENGTH
)

print(
    "Matched size–replicate groups:",
    TOTAL_CORRECTED_MATCHED_GROUPS
)

print(
    "Regime-specific instances:",
    TOTAL_CORRECTED_INSTANCES
)

print(
    "Expected experimental rows:",
    TOTAL_CORRECTED_ROWS
)


# --------------------------------------------------
# Corrected matched-regime experiment
# --------------------------------------------------

corrected_session_start = (
    time.perf_counter()
)


for size_index, n_items in enumerate(
    EXACT_BENCHMARK_SIZES
):

    print(
        "\nStarting candidate-set size:",
        n_items
    )


    for replicate in range(
        CORRECTED_MAIN_REPLICATES
    ):


        # ------------------------------------------
        # One matched latent seed is used for every
        # structural regime in this size–replicate pair.
        # ------------------------------------------

        matched_seed = (
            GLOBAL_SEED
            + 50_000_000
            + size_index
            * 1_000_000
            + replicate
        )


        matched_group_id = (
            f"V2_M{n_items}"
            f"_R{replicate:03d}"
        )


        # ------------------------------------------
        # Evaluate all three matched regimes
        # ------------------------------------------

        for regime in INSTANCE_REGIMES:

            instance_id = (
                f"{matched_group_id}"
                f"_{regime}"
            )


            # Resume safely from completed instances.
            if (
                instance_id
                in completed_instance_ids
            ):

                continue


            instance_result_rows = (
                run_cached_instance_benchmark(
                    n_items=n_items,
                    regime=regime,
                    seed=matched_seed,
                )
            )


            for row in instance_result_rows:

                row[
                    "generator_version"
                ] = (
                    CORRECTED_GENERATOR_VERSION
                )

                row[
                    "regime_strength"
                ] = (
                    REGIME_STRENGTH
                )

                row[
                    "matched_group_id"
                ] = (
                    matched_group_id
                )

                row[
                    "instance_id"
                ] = (
                    instance_id
                )

                row[
                    "replicate"
                ] = (
                    replicate
                )

                row[
                    "matched_seed"
                ] = (
                    matched_seed
                )


            corrected_main_rows.extend(
                instance_result_rows
            )

            completed_instance_ids.add(
                instance_id
            )


        # ------------------------------------------
        # Save after every completed matched group
        # ------------------------------------------

        corrected_checkpoint_df = (
            pd.DataFrame(
                corrected_main_rows
            )
        )


        corrected_checkpoint_df.to_csv(
            CORRECTED_CHECKPOINT_PATH,
            index=False,
        )


        completed_group_count = (
            corrected_checkpoint_df
            .groupby(
                "matched_group_id"
            )
            .filter(
                lambda group:
                    group[
                        "instance_id"
                    ].nunique()
                    ==
                    EXPECTED_REGIMES_PER_MATCHED_GROUP
            )[
                "matched_group_id"
            ]
            .nunique()
        )


        if (
            completed_group_count % 5
            == 0
        ):

            elapsed_seconds = (
                time.perf_counter()
                - corrected_session_start
            )


            print(
                f"Completed matched groups: "
                f"{completed_group_count}/"
                f"{TOTAL_CORRECTED_MATCHED_GROUPS}"
                f" | instances="
                f"{len(completed_instance_ids)}"
                f" | rows="
                f"{len(corrected_checkpoint_df)}"
                f" | session elapsed="
                f"{elapsed_seconds / 60:.2f} min"
            )


# --------------------------------------------------
# Create final corrected dataset
# --------------------------------------------------

corrected_main_benchmark_df = (
    pd.DataFrame(
        corrected_main_rows
    )
)


corrected_main_benchmark_df = (
    corrected_main_benchmark_df
    .drop_duplicates(
        subset=[
            "instance_id",
            "tau",
            "weight_profile",
        ],
        keep="last",
    )
    .sort_values(
        by=[
            "n_items",
            "replicate",
            "regime",
            "tau",
            "weight_profile",
        ]
    )
    .reset_index(
        drop=True
    )
)


corrected_main_benchmark_df.to_csv(
    CORRECTED_FINAL_PATH,
    index=False,
)


# --------------------------------------------------
# Final structural checks
# --------------------------------------------------

corrected_rows_per_instance = (
    corrected_main_benchmark_df
    .groupby(
        "instance_id"
    )
    .size()
)


corrected_rows_per_matched_group = (
    corrected_main_benchmark_df
    .groupby(
        "matched_group_id"
    )
    .size()
)


corrected_seeds_per_matched_group = (
    corrected_main_benchmark_df
    .groupby(
        "matched_group_id"
    )[
        "seed"
    ]
    .nunique()
)


corrected_regimes_per_matched_group = (
    corrected_main_benchmark_df
    .groupby(
        "matched_group_id"
    )[
        "regime"
    ]
    .nunique()
)


negative_optimality_gaps = (
    corrected_main_benchmark_df[
        "optimality_gap"
    ]
    .dropna()
    < -1e-12
).sum()


exact_threshold_failures = (
    (
        corrected_main_benchmark_df[
            "exact_status"
        ]
        == "OPTIMAL"
    )
    &
    (
        corrected_main_benchmark_df[
            "exact_utility"
        ]
        <
        corrected_main_benchmark_df[
            "tau"
        ]
    )
).sum()


greedy_threshold_failures = (
    (
        corrected_main_benchmark_df[
            "greedy_status"
        ]
        == "FEASIBLE"
    )
    &
    (
        corrected_main_benchmark_df[
            "greedy_utility"
        ]
        <
        corrected_main_benchmark_df[
            "tau"
        ]
    )
).sum()


duplicate_condition_rows = (
    corrected_main_benchmark_df
    .duplicated(
        subset=[
            "instance_id",
            "tau",
            "weight_profile",
        ],
        keep=False,
    )
    .sum()
)


# --------------------------------------------------
# Reconstruct recorded computational timing
# --------------------------------------------------

corrected_instance_timing_df = (
    corrected_main_benchmark_df
    .groupby(
        "instance_id",
        as_index=False,
    )
    .agg(
        subset_precomputation_runtime_seconds=(
            "subset_precomputation_runtime_seconds",
            "first",
        )
    )
)


corrected_total_precomputation_seconds = (
    corrected_instance_timing_df[
        "subset_precomputation_runtime_seconds"
    ].sum()
)


corrected_total_cached_exact_seconds = (
    corrected_main_benchmark_df[
        "exact_cached_solve_runtime_seconds"
    ].sum()
)


corrected_total_greedy_seconds = (
    corrected_main_benchmark_df[
        "greedy_runtime_seconds"
    ].sum()
)


corrected_total_recorded_seconds = (
    corrected_total_precomputation_seconds
    +
    corrected_total_cached_exact_seconds
    +
    corrected_total_greedy_seconds
)


corrected_current_session_seconds = (
    time.perf_counter()
    - corrected_session_start
)


# --------------------------------------------------
# Assertions
# --------------------------------------------------

assert (
    len(
        corrected_main_benchmark_df
    )
    ==
    TOTAL_CORRECTED_ROWS
)

assert (
    corrected_main_benchmark_df[
        "instance_id"
    ].nunique()
    ==
    TOTAL_CORRECTED_INSTANCES
)

assert (
    corrected_main_benchmark_df[
        "matched_group_id"
    ].nunique()
    ==
    TOTAL_CORRECTED_MATCHED_GROUPS
)

assert (
    corrected_rows_per_instance
    ==
    EXPECTED_CONDITIONS_PER_INSTANCE
).all()

assert (
    corrected_rows_per_matched_group
    ==
    EXPECTED_ROWS_PER_MATCHED_GROUP
).all()

assert (
    corrected_seeds_per_matched_group
    == 1
).all()

assert (
    corrected_regimes_per_matched_group
    ==
    EXPECTED_REGIMES_PER_MATCHED_GROUP
).all()

assert duplicate_condition_rows == 0
assert negative_optimality_gaps == 0
assert exact_threshold_failures == 0
assert greedy_threshold_failures == 0


# --------------------------------------------------
# Final report
# --------------------------------------------------

print(
    "\n======================================"
)

print(
    "CORRECTED MAIN BENCHMARK COMPLETED"
)

print(
    "======================================"
)

print(
    "Generator version:",
    CORRECTED_GENERATOR_VERSION
)

print(
    "Matched groups:",
    corrected_main_benchmark_df[
        "matched_group_id"
    ].nunique()
)

print(
    "Regime-specific instances:",
    corrected_main_benchmark_df[
        "instance_id"
    ].nunique()
)

print(
    "Total experimental rows:",
    len(
        corrected_main_benchmark_df
    )
)

print(
    "Expected experimental rows:",
    TOTAL_CORRECTED_ROWS
)

print(
    "Minimum rows per instance:",
    corrected_rows_per_instance.min()
)

print(
    "Maximum rows per instance:",
    corrected_rows_per_instance.max()
)

print(
    "Minimum rows per matched group:",
    corrected_rows_per_matched_group.min()
)

print(
    "Maximum rows per matched group:",
    corrected_rows_per_matched_group.max()
)

print(
    "Matched groups with multiple seeds:",
    (
        corrected_seeds_per_matched_group
        != 1
    ).sum()
)

print(
    "Matched groups missing regimes:",
    (
        corrected_regimes_per_matched_group
        !=
        EXPECTED_REGIMES_PER_MATCHED_GROUP
    ).sum()
)

print(
    "Duplicate condition rows:",
    duplicate_condition_rows
)

print(
    "Negative optimality gaps:",
    negative_optimality_gaps
)

print(
    "Exact threshold failures:",
    exact_threshold_failures
)

print(
    "Greedy threshold failures:",
    greedy_threshold_failures
)


print(
    "\nRecorded computational timing:"
)

print(
    "Subset precomputation:",
    f"{corrected_total_precomputation_seconds / 60:.2f} minutes"
)

print(
    "Cached Exact solves:",
    f"{corrected_total_cached_exact_seconds / 60:.2f} minutes"
)

print(
    "Greedy solves:",
    f"{corrected_total_greedy_seconds / 60:.2f} minutes"
)

print(
    "Total recorded algorithm time:",
    f"{corrected_total_recorded_seconds / 60:.2f} minutes"
)

print(
    "Current execution-session time:",
    f"{corrected_current_session_seconds / 60:.2f} minutes"
)


print(
    "\nCorrected final results saved to:"
)

print(
    CORRECTED_FINAL_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Corrected v2 benchmark integrity and feasibility audit
# --------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd


# --------------------------------------------------
# Locate and load the corrected benchmark
# --------------------------------------------------

if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


CORRECTED_FINAL_PATH = (
    RESULTS_DIR
    / "exact_vs_greedy_main_benchmark_v2_marginal_preserving.csv"
)

CORRECTED_INTEGRITY_PATH = (
    TABLES_DIR
    / "corrected_v2_benchmark_integrity_audit.csv"
)


corrected_main_benchmark_df = pd.read_csv(
    CORRECTED_FINAL_PATH
)


# --------------------------------------------------
# Expected benchmark dimensions
# --------------------------------------------------

EXPECTED_CORRECTED_ROWS = 24000
EXPECTED_CORRECTED_INSTANCES = 1500
EXPECTED_MATCHED_GROUPS = 500
EXPECTED_ROWS_PER_INSTANCE = 16
EXPECTED_ROWS_PER_MATCHED_GROUP = 48
EXPECTED_REGIMES_PER_MATCHED_GROUP = 3


# --------------------------------------------------
# Basic dimensions
# --------------------------------------------------

actual_rows = len(
    corrected_main_benchmark_df
)

actual_instances = (
    corrected_main_benchmark_df[
        "instance_id"
    ]
    .nunique()
)

actual_matched_groups = (
    corrected_main_benchmark_df[
        "matched_group_id"
    ]
    .nunique()
)


rows_per_instance = (
    corrected_main_benchmark_df
    .groupby(
        "instance_id"
    )
    .size()
)


rows_per_matched_group = (
    corrected_main_benchmark_df
    .groupby(
        "matched_group_id"
    )
    .size()
)


regimes_per_matched_group = (
    corrected_main_benchmark_df
    .groupby(
        "matched_group_id"
    )[
        "regime"
    ]
    .nunique()
)


seed_column = (
    "matched_seed"
    if "matched_seed"
    in corrected_main_benchmark_df.columns
    else "seed"
)


seeds_per_matched_group = (
    corrected_main_benchmark_df
    .groupby(
        "matched_group_id"
    )[
        seed_column
    ]
    .nunique()
)


duplicate_condition_rows = (
    corrected_main_benchmark_df
    .duplicated(
        subset=[
            "instance_id",
            "tau",
            "weight_profile",
        ],
        keep=False,
    )
    .sum()
)


# --------------------------------------------------
# Feasibility and solver-status checks
# --------------------------------------------------

corrected_main_benchmark_df[
    "status_pair"
] = (
    corrected_main_benchmark_df[
        "exact_status"
    ].astype(str)
    +
    " / "
    +
    corrected_main_benchmark_df[
        "greedy_status"
    ].astype(str)
)


status_pair_counts = (
    corrected_main_benchmark_df[
        "status_pair"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "status_pair"
    )
    .reset_index(
        name="rows"
    )
)


comparable_mask = (
    (
        corrected_main_benchmark_df[
            "exact_status"
        ]
        == "OPTIMAL"
    )
    &
    (
        corrected_main_benchmark_df[
            "greedy_status"
        ]
        == "FEASIBLE"
    )
)


both_infeasible_mask = (
    (
        corrected_main_benchmark_df[
            "exact_status"
        ]
        == "INFEASIBLE"
    )
    &
    (
        corrected_main_benchmark_df[
            "greedy_status"
        ]
        == "INFEASIBLE"
    )
)


feasibility_disagreement_mask = (
    ~comparable_mask
    &
    ~both_infeasible_mask
)


comparable_rows = int(
    comparable_mask.sum()
)

infeasible_rows = int(
    both_infeasible_mask.sum()
)

feasibility_disagreement_rows = int(
    feasibility_disagreement_mask.sum()
)


# --------------------------------------------------
# Numerical validity checks
# --------------------------------------------------

if (
    "optimality_gap"
    in corrected_main_benchmark_df.columns
):

    negative_gap_rows = int(
        (
            corrected_main_benchmark_df.loc[
                comparable_mask,
                "optimality_gap",
            ]
            < -1e-12
        ).sum()
    )

elif (
    "optimality_gap_percent"
    in corrected_main_benchmark_df.columns
):

    negative_gap_rows = int(
        (
            corrected_main_benchmark_df.loc[
                comparable_mask,
                "optimality_gap_percent",
            ]
            < -1e-10
        ).sum()
    )

else:

    negative_gap_rows = np.nan


exact_threshold_failures = int(
    (
        (
            corrected_main_benchmark_df[
                "exact_status"
            ]
            == "OPTIMAL"
        )
        &
        (
            corrected_main_benchmark_df[
                "exact_utility"
            ]
            + 1e-12
            <
            corrected_main_benchmark_df[
                "tau"
            ]
        )
    ).sum()
)


greedy_threshold_failures = int(
    (
        (
            corrected_main_benchmark_df[
                "greedy_status"
            ]
            == "FEASIBLE"
        )
        &
        (
            corrected_main_benchmark_df[
                "greedy_utility"
            ]
            + 1e-12
            <
            corrected_main_benchmark_df[
                "tau"
            ]
        )
    ).sum()
)


# --------------------------------------------------
# Construct and save integrity table
# --------------------------------------------------

integrity_rows = [
    {
        "check":
            "total_experimental_rows",

        "observed":
            actual_rows,

        "expected":
            EXPECTED_CORRECTED_ROWS,

        "passed":
            actual_rows
            == EXPECTED_CORRECTED_ROWS,
    },
    {
        "check":
            "regime_specific_instances",

        "observed":
            actual_instances,

        "expected":
            EXPECTED_CORRECTED_INSTANCES,

        "passed":
            actual_instances
            == EXPECTED_CORRECTED_INSTANCES,
    },
    {
        "check":
            "matched_groups",

        "observed":
            actual_matched_groups,

        "expected":
            EXPECTED_MATCHED_GROUPS,

        "passed":
            actual_matched_groups
            == EXPECTED_MATCHED_GROUPS,
    },
    {
        "check":
            "minimum_rows_per_instance",

        "observed":
            int(
                rows_per_instance.min()
            ),

        "expected":
            EXPECTED_ROWS_PER_INSTANCE,

        "passed":
            rows_per_instance.min()
            == EXPECTED_ROWS_PER_INSTANCE,
    },
    {
        "check":
            "maximum_rows_per_instance",

        "observed":
            int(
                rows_per_instance.max()
            ),

        "expected":
            EXPECTED_ROWS_PER_INSTANCE,

        "passed":
            rows_per_instance.max()
            == EXPECTED_ROWS_PER_INSTANCE,
    },
    {
        "check":
            "minimum_rows_per_matched_group",

        "observed":
            int(
                rows_per_matched_group.min()
            ),

        "expected":
            EXPECTED_ROWS_PER_MATCHED_GROUP,

        "passed":
            rows_per_matched_group.min()
            == EXPECTED_ROWS_PER_MATCHED_GROUP,
    },
    {
        "check":
            "maximum_rows_per_matched_group",

        "observed":
            int(
                rows_per_matched_group.max()
            ),

        "expected":
            EXPECTED_ROWS_PER_MATCHED_GROUP,

        "passed":
            rows_per_matched_group.max()
            == EXPECTED_ROWS_PER_MATCHED_GROUP,
    },
    {
        "check":
            "matched_groups_with_incorrect_regime_count",

        "observed":
            int(
                (
                    regimes_per_matched_group
                    !=
                    EXPECTED_REGIMES_PER_MATCHED_GROUP
                ).sum()
            ),

        "expected":
            0,

        "passed":
            (
                regimes_per_matched_group
                ==
                EXPECTED_REGIMES_PER_MATCHED_GROUP
            ).all(),
    },
    {
        "check":
            "matched_groups_with_multiple_seeds",

        "observed":
            int(
                (
                    seeds_per_matched_group
                    != 1
                ).sum()
            ),

        "expected":
            0,

        "passed":
            (
                seeds_per_matched_group
                == 1
            ).all(),
    },
    {
        "check":
            "duplicate_condition_rows",

        "observed":
            int(
                duplicate_condition_rows
            ),

        "expected":
            0,

        "passed":
            duplicate_condition_rows
            == 0,
    },
    {
        "check":
            "feasibility_disagreement_rows",

        "observed":
            feasibility_disagreement_rows,

        "expected":
            0,

        "passed":
            feasibility_disagreement_rows
            == 0,
    },
    {
        "check":
            "negative_optimality_gap_rows",

        "observed":
            negative_gap_rows,

        "expected":
            0,

        "passed":
            negative_gap_rows
            == 0,
    },
    {
        "check":
            "exact_threshold_failures",

        "observed":
            exact_threshold_failures,

        "expected":
            0,

        "passed":
            exact_threshold_failures
            == 0,
    },
    {
        "check":
            "greedy_threshold_failures",

        "observed":
            greedy_threshold_failures,

        "expected":
            0,

        "passed":
            greedy_threshold_failures
            == 0,
    },
]


corrected_integrity_df = pd.DataFrame(
    integrity_rows
)


corrected_integrity_df.to_csv(
    CORRECTED_INTEGRITY_PATH,
    index=False,
)


# --------------------------------------------------
# Assertions
# --------------------------------------------------

assert (
    corrected_integrity_df[
        "passed"
    ]
    .all()
)


# --------------------------------------------------
# Display audit
# --------------------------------------------------

print(
    "======================================"
)

print(
    "CORRECTED V2 BENCHMARK INTEGRITY AUDIT"
)

print(
    "======================================"
)


print(
    corrected_integrity_df.to_string(
        index=False
    )
)


print(
    "\nSolver-status combinations:"
)

print(
    status_pair_counts.to_string(
        index=False
    )
)


print(
    "\nComparable feasible rows:",
    comparable_rows
)

print(
    "Both-infeasible rows:",
    infeasible_rows
)

print(
    "Feasibility-disagreement rows:",
    feasibility_disagreement_rows
)


if (
    "generator_version"
    in corrected_main_benchmark_df.columns
):

    print(
        "\nGenerator versions:",
        corrected_main_benchmark_df[
            "generator_version"
        ]
        .value_counts()
        .to_dict()
    )


print(
    "\nIntegrity audit saved to:"
)

print(
    CORRECTED_INTEGRITY_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Create corrected v2 analysis-ready benchmark dataset
# --------------------------------------------------

from pathlib import Path
import ast
import numpy as np
import pandas as pd


# --------------------------------------------------
# File locations
# --------------------------------------------------

if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


CORRECTED_RAW_PATH = (
    RESULTS_DIR
    / "exact_vs_greedy_main_benchmark_v2_marginal_preserving.csv"
)


CORRECTED_ANALYSIS_READY_PATH = (
    RESULTS_DIR
    / (
        "exact_vs_greedy_main_benchmark_"
        "v2_marginal_preserving_analysis_ready.csv"
    )
)


CORRECTED_GAP_BAND_PATH = (
    TABLES_DIR
    / "corrected_v2_optimality_gap_band_counts.csv"
)


# --------------------------------------------------
# Load without chunk-based dtype inference
# --------------------------------------------------

corrected_analysis_df = pd.read_csv(
    CORRECTED_RAW_PATH,
    low_memory=False,
)


# --------------------------------------------------
# Utility functions
# --------------------------------------------------

def first_existing_column(
    dataframe,
    candidate_names,
):
    """
    Return the first available column from a list of
    possible names, or None if none exists.
    """

    for column_name in candidate_names:

        if column_name in dataframe.columns:

            return column_name

    return None



def parse_selected_subset(value):
    """
    Convert a stored selected-subset representation into
    a Python set.

    Supports lists, tuples, sets, comma-separated strings,
    and missing values.
    """

    if pd.isna(value):

        return set()


    if isinstance(
        value,
        (
            list,
            tuple,
            set,
            np.ndarray,
        ),
    ):

        return {
            str(item)
            for item in value
        }


    text = str(value).strip()


    if text in {
        "",
        "[]",
        "()",
        "{}",
        "nan",
        "None",
    }:

        return set()


    try:

        parsed_value = ast.literal_eval(
            text
        )


        if isinstance(
            parsed_value,
            (
                list,
                tuple,
                set,
            ),
        ):

            return {
                str(item)
                for item in parsed_value
            }


        return {
            str(parsed_value)
        }


    except (
        ValueError,
        SyntaxError,
    ):

        cleaned_text = (
            text
            .replace("[", "")
            .replace("]", "")
            .replace("(", "")
            .replace(")", "")
            .replace("{", "")
            .replace("}", "")
            .replace("'", "")
            .replace('"', "")
        )


        return {
            token.strip()
            for token
            in cleaned_text.split(",")
            if token.strip()
        }



def calculate_jaccard(
    first_set,
    second_set,
):
    """
    Calculate Jaccard similarity between two subsets.
    """

    union = (
        first_set
        |
        second_set
    )


    if len(union) == 0:

        return 1.0


    return (
        len(
            first_set
            &
            second_set
        )
        /
        len(union)
    )


# --------------------------------------------------
# Standardize known numerical columns
# --------------------------------------------------

numeric_candidate_columns = [
    "n_items",
    "replicate",
    "seed",
    "matched_seed",
    "regime_strength",
    "tau",
    "lambda_privacy",
    "lambda_cognitive",
    "lambda_cardinality",
    "exact_selected_count",
    "greedy_selected_count",
    "exact_utility",
    "greedy_utility",
    "exact_privacy",
    "greedy_privacy",
    "exact_cognitive_cost",
    "greedy_cognitive_cost",
    "exact_normalized_cognitive_cost",
    "greedy_normalized_cognitive_cost",
    "exact_J",
    "greedy_J",
    "exact_cost",
    "greedy_cost",
    "exact_disclosure_cost",
    "greedy_disclosure_cost",
    "optimality_gap",
    "optimality_gap_percent",
    "jaccard_similarity",
    "subset_precomputation_runtime_seconds",
    "exact_cached_solve_runtime_seconds",
    "greedy_runtime_seconds",
]


converted_numeric_columns = []


for column_name in numeric_candidate_columns:

    if column_name in corrected_analysis_df.columns:

        corrected_analysis_df[
            column_name
        ] = pd.to_numeric(
            corrected_analysis_df[
                column_name
            ],
            errors="coerce",
        )

        converted_numeric_columns.append(
            column_name
        )


# --------------------------------------------------
# Feasibility indicators
# --------------------------------------------------

corrected_analysis_df[
    "is_comparable"
] = (
    (
        corrected_analysis_df[
            "exact_status"
        ]
        == "OPTIMAL"
    )
    &
    (
        corrected_analysis_df[
            "greedy_status"
        ]
        == "FEASIBLE"
    )
)


corrected_analysis_df[
    "is_both_infeasible"
] = (
    (
        corrected_analysis_df[
            "exact_status"
        ]
        == "INFEASIBLE"
    )
    &
    (
        corrected_analysis_df[
            "greedy_status"
        ]
        == "INFEASIBLE"
    )
)


corrected_analysis_df[
    "has_feasibility_disagreement"
] = (
    ~corrected_analysis_df[
        "is_comparable"
    ]
    &
    ~corrected_analysis_df[
        "is_both_infeasible"
    ]
)


# --------------------------------------------------
# Standardize optimality gap as a percentage
# --------------------------------------------------

gap_scale_description = None


if (
    "optimality_gap_percent"
    in corrected_analysis_df.columns
):

    standardized_gap_percent = (
        corrected_analysis_df[
            "optimality_gap_percent"
        ]
        .copy()
    )

    gap_scale_description = (
        "existing optimality_gap_percent column"
    )


elif (
    "optimality_gap"
    in corrected_analysis_df.columns
):

    raw_gap_values = (
        corrected_analysis_df.loc[
            corrected_analysis_df[
                "is_comparable"
            ],
            "optimality_gap",
        ]
        .dropna()
        .abs()
    )


    if (
        len(raw_gap_values) > 0
        and
        raw_gap_values.max() <= 1.5
    ):

        standardized_gap_percent = (
            100.0
            *
            corrected_analysis_df[
                "optimality_gap"
            ]
        )

        gap_scale_description = (
            "optimality_gap interpreted as proportion "
            "and multiplied by 100"
        )


    else:

        standardized_gap_percent = (
            corrected_analysis_df[
                "optimality_gap"
            ]
            .copy()
        )

        gap_scale_description = (
            "optimality_gap interpreted as percentage"
        )


else:

    exact_cost_column = first_existing_column(
        corrected_analysis_df,
        [
            "exact_J",
            "exact_cost",
            "exact_disclosure_cost",
        ],
    )


    greedy_cost_column = first_existing_column(
        corrected_analysis_df,
        [
            "greedy_J",
            "greedy_cost",
            "greedy_disclosure_cost",
        ],
    )


    if (
        exact_cost_column is None
        or
        greedy_cost_column is None
    ):

        raise KeyError(
            "No optimality-gap or compatible objective-cost "
            "columns were found."
        )


    standardized_gap_percent = (
        100.0
        *
        (
            corrected_analysis_df[
                greedy_cost_column
            ]
            -
            corrected_analysis_df[
                exact_cost_column
            ]
        )
        /
        corrected_analysis_df[
            exact_cost_column
        ]
    )


    gap_scale_description = (
        f"recomputed from {exact_cost_column} "
        f"and {greedy_cost_column}"
    )


corrected_analysis_df[
    "optimality_gap_percent"
] = standardized_gap_percent


# Remove machine-level negative noise.
corrected_analysis_df.loc[
    (
        corrected_analysis_df[
            "is_comparable"
        ]
        &
        (
            corrected_analysis_df[
                "optimality_gap_percent"
            ]
            < 0.0
        )
        &
        (
            corrected_analysis_df[
                "optimality_gap_percent"
            ]
            > -1e-10
        )
    ),
    "optimality_gap_percent",
] = 0.0


corrected_analysis_df.loc[
    ~corrected_analysis_df[
        "is_comparable"
    ],
    "optimality_gap_percent",
] = np.nan


# --------------------------------------------------
# Exact optimal-cost recovery indicator
# --------------------------------------------------

OPTIMALITY_TOLERANCE_PERCENT = 1e-10


corrected_analysis_df[
    "greedy_optimal_cost"
] = pd.Series(
    pd.NA,
    index=corrected_analysis_df.index,
    dtype="boolean",
)


corrected_analysis_df.loc[
    corrected_analysis_df[
        "is_comparable"
    ],
    "greedy_optimal_cost",
] = (
    corrected_analysis_df.loc[
        corrected_analysis_df[
            "is_comparable"
        ],
        "optimality_gap_percent",
    ]
    .abs()
    <=
    OPTIMALITY_TOLERANCE_PERCENT
)


# --------------------------------------------------
# Selected-count difference
# --------------------------------------------------

exact_count_column = first_existing_column(
    corrected_analysis_df,
    [
        "exact_selected_count",
        "exact_count",
    ],
)


greedy_count_column = first_existing_column(
    corrected_analysis_df,
    [
        "greedy_selected_count",
        "greedy_count",
    ],
)


if (
    exact_count_column is not None
    and
    greedy_count_column is not None
):

    corrected_analysis_df[
        "selected_count_difference"
    ] = (
        corrected_analysis_df[
            greedy_count_column
        ]
        -
        corrected_analysis_df[
            exact_count_column
        ]
    )


else:

    corrected_analysis_df[
        "selected_count_difference"
    ] = np.nan


# --------------------------------------------------
# Standardize Jaccard similarity
# --------------------------------------------------

exact_subset_column = first_existing_column(
    corrected_analysis_df,
    [
        "exact_selected_ids",
        "exact_selected_items",
        "exact_selected",
        "exact_subset",
    ],
)


greedy_subset_column = first_existing_column(
    corrected_analysis_df,
    [
        "greedy_selected_ids",
        "greedy_selected_items",
        "greedy_selected",
        "greedy_subset",
    ],
)


if (
    "jaccard_similarity"
    not in corrected_analysis_df.columns
    or
    corrected_analysis_df[
        "jaccard_similarity"
    ].isna().all()
):

    if (
        exact_subset_column is None
        or
        greedy_subset_column is None
    ):

        raise KeyError(
            "Jaccard similarity is unavailable and selected-subset "
            "columns could not be identified."
        )


    exact_subset_sets = (
        corrected_analysis_df[
            exact_subset_column
        ]
        .apply(
            parse_selected_subset
        )
    )


    greedy_subset_sets = (
        corrected_analysis_df[
            greedy_subset_column
        ]
        .apply(
            parse_selected_subset
        )
    )


    corrected_analysis_df[
        "jaccard_similarity"
    ] = [
        calculate_jaccard(
            exact_set,
            greedy_set,
        )
        if is_comparable
        else np.nan

        for (
            exact_set,
            greedy_set,
            is_comparable,
        )
        in zip(
            exact_subset_sets,
            greedy_subset_sets,
            corrected_analysis_df[
                "is_comparable"
            ],
        )
    ]


else:

    corrected_analysis_df[
        "jaccard_similarity"
    ] = pd.to_numeric(
        corrected_analysis_df[
            "jaccard_similarity"
        ],
        errors="coerce",
    )


    corrected_analysis_df.loc[
        ~corrected_analysis_df[
            "is_comparable"
        ],
        "jaccard_similarity",
    ] = np.nan


# --------------------------------------------------
# Optimality-gap reporting bands
# --------------------------------------------------

corrected_analysis_df[
    "optimality_gap_band"
] = "infeasible"


comparable_gap = (
    corrected_analysis_df[
        "is_comparable"
    ]
)


corrected_analysis_df.loc[
    comparable_gap
    &
    (
        corrected_analysis_df[
            "optimality_gap_percent"
        ]
        <=
        OPTIMALITY_TOLERANCE_PERCENT
    ),
    "optimality_gap_band",
] = "optimal"


corrected_analysis_df.loc[
    comparable_gap
    &
    (
        corrected_analysis_df[
            "optimality_gap_percent"
        ]
        >
        OPTIMALITY_TOLERANCE_PERCENT
    )
    &
    (
        corrected_analysis_df[
            "optimality_gap_percent"
        ]
        <= 1.0
    ),
    "optimality_gap_band",
] = ">0 to 1%"


corrected_analysis_df.loc[
    comparable_gap
    &
    (
        corrected_analysis_df[
            "optimality_gap_percent"
        ]
        > 1.0
    )
    &
    (
        corrected_analysis_df[
            "optimality_gap_percent"
        ]
        <= 5.0
    ),
    "optimality_gap_band",
] = ">1 to 5%"


corrected_analysis_df.loc[
    comparable_gap
    &
    (
        corrected_analysis_df[
            "optimality_gap_percent"
        ]
        > 5.0
    )
    &
    (
        corrected_analysis_df[
            "optimality_gap_percent"
        ]
        <= 10.0
    ),
    "optimality_gap_band",
] = ">5 to 10%"


corrected_analysis_df.loc[
    comparable_gap
    &
    (
        corrected_analysis_df[
            "optimality_gap_percent"
        ]
        > 10.0
    ),
    "optimality_gap_band",
] = ">10%"


gap_band_order = [
    "optimal",
    ">0 to 1%",
    ">1 to 5%",
    ">5 to 10%",
    ">10%",
    "infeasible",
]


corrected_analysis_df[
    "optimality_gap_band"
] = pd.Categorical(
    corrected_analysis_df[
        "optimality_gap_band"
    ],
    categories=gap_band_order,
    ordered=True,
)


# --------------------------------------------------
# Gap-band counts and percentages
# --------------------------------------------------

corrected_gap_band_df = (
    corrected_analysis_df[
        "optimality_gap_band"
    ]
    .value_counts(
        sort=False
    )
    .rename_axis(
        "optimality_gap_band"
    )
    .reset_index(
        name="rows"
    )
)


corrected_gap_band_df[
    "percent_of_all_rows"
] = (
    100.0
    *
    corrected_gap_band_df[
        "rows"
    ]
    /
    len(
        corrected_analysis_df
    )
)


corrected_gap_band_df[
    "percent_of_comparable_rows"
] = np.where(
    corrected_gap_band_df[
        "optimality_gap_band"
    ].astype(str)
    != "infeasible",

    100.0
    *
    corrected_gap_band_df[
        "rows"
    ]
    /
    corrected_analysis_df[
        "is_comparable"
    ].sum(),

    np.nan,
)


# --------------------------------------------------
# Floating-point and subset-agreement audit
# --------------------------------------------------

comparable_df = (
    corrected_analysis_df[
        corrected_analysis_df[
            "is_comparable"
        ]
    ]
)


identical_subset_mask = (
    comparable_df[
        "jaccard_similarity"
    ]
    >=
    1.0 - 1e-12
)


different_subset_mask = (
    comparable_df[
        "jaccard_similarity"
    ]
    <
    1.0 - 1e-12
)


identical_subset_rows = int(
    identical_subset_mask.sum()
)


maximum_gap_identical_subset = (
    comparable_df.loc[
        identical_subset_mask,
        "optimality_gap_percent",
    ]
    .abs()
    .max()
)


different_subset_equal_cost_rows = int(
    (
        different_subset_mask
        &
        (
            comparable_df[
                "optimality_gap_percent"
            ]
            .abs()
            <=
            OPTIMALITY_TOLERANCE_PERCENT
        )
    ).sum()
)


# --------------------------------------------------
# Infeasible instance–threshold cases
# --------------------------------------------------

infeasible_instance_tau_cases = (
    corrected_analysis_df[
        corrected_analysis_df[
            "is_both_infeasible"
        ]
    ][
        [
            "instance_id",
            "tau",
        ]
    ]
    .drop_duplicates()
)


# --------------------------------------------------
# Final validation
# --------------------------------------------------

assert len(
    corrected_analysis_df
) == 24000


assert (
    corrected_analysis_df[
        "is_comparable"
    ].sum()
    == 23784
)


assert (
    corrected_analysis_df[
        "is_both_infeasible"
    ].sum()
    == 216
)


assert (
    corrected_analysis_df[
        "has_feasibility_disagreement"
    ].sum()
    == 0
)


assert (
    corrected_analysis_df.loc[
        corrected_analysis_df[
            "is_comparable"
        ],
        "optimality_gap_percent",
    ]
    >= -1e-10
).all()


assert (
    corrected_analysis_df.loc[
        corrected_analysis_df[
            "is_comparable"
        ],
        "jaccard_similarity",
    ]
    .between(
        0.0,
        1.0,
        inclusive="both",
    )
    .all()
)


# --------------------------------------------------
# Save analysis-ready outputs
# --------------------------------------------------

corrected_analysis_df.to_csv(
    CORRECTED_ANALYSIS_READY_PATH,
    index=False,
)


corrected_gap_band_df.to_csv(
    CORRECTED_GAP_BAND_PATH,
    index=False,
)


# --------------------------------------------------
# Display summary
# --------------------------------------------------

print(
    "======================================"
)

print(
    "CORRECTED V2 ANALYSIS-READY DATASET"
)

print(
    "======================================"
)


print(
    "Optimality-gap source:",
    gap_scale_description
)

print(
    "Numerical columns standardized:",
    len(
        converted_numeric_columns
    )
)

print(
    "\nTotal rows:",
    len(
        corrected_analysis_df
    )
)

print(
    "Comparable feasible rows:",
    int(
        corrected_analysis_df[
            "is_comparable"
        ].sum()
    )
)

print(
    "Both-infeasible rows:",
    int(
        corrected_analysis_df[
            "is_both_infeasible"
        ].sum()
    )
)

print(
    "Unique infeasible instance–tau cases:",
    len(
        infeasible_instance_tau_cases
    )
)

print(
    "Feasibility disagreements:",
    int(
        corrected_analysis_df[
            "has_feasibility_disagreement"
        ].sum()
    )
)


print(
    "\nOptimality-gap bands:"
)

print(
    corrected_gap_band_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.4f}",
    )
)


print(
    "\nFloating-point and subset audit:"
)

print(
    "Identical selected-subset rows:",
    identical_subset_rows
)

print(
    "Maximum gap among identical subsets:",
    f"{maximum_gap_identical_subset:.16e}"
)

print(
    "Different-subset equal-cost rows:",
    different_subset_equal_cost_rows
)


print(
    "\nAnalysis-ready dataset saved to:"
)

print(
    CORRECTED_ANALYSIS_READY_PATH
)

print(
    "\nGap-band table saved to:"
)

print(
    CORRECTED_GAP_BAND_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Corrected v2 overall results with matched-group
# stratified cluster bootstrap
# --------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd


OVERALL_V2_BOOTSTRAP_REPLICATES = 5000
OVERALL_V2_BOOTSTRAP_SEED = GLOBAL_SEED + 100


if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


CORRECTED_ANALYSIS_READY_PATH = (
    RESULTS_DIR
    / (
        "exact_vs_greedy_main_benchmark_"
        "v2_marginal_preserving_analysis_ready.csv"
    )
)


OVERALL_V2_BOOTSTRAP_REPLICATES_PATH = (
    RESULTS_DIR
    / "corrected_v2_overall_quality_matched_group_bootstrap_replicates.csv"
)


OVERALL_V2_BOOTSTRAP_SUMMARY_PATH = (
    TABLES_DIR
    / "corrected_v2_overall_quality_matched_group_bootstrap_ci.csv"
)


OVERALL_V2_DESCRIPTIVE_PATH = (
    TABLES_DIR
    / "corrected_v2_overall_descriptive_summary.csv"
)


# --------------------------------------------------
# Load analysis-ready dataset
# --------------------------------------------------

corrected_v2_df = pd.read_csv(
    CORRECTED_ANALYSIS_READY_PATH,
    low_memory=False,
)


# --------------------------------------------------
# Reconstruct key indicators robustly
# --------------------------------------------------

corrected_v2_df[
    "is_comparable"
] = (
    (
        corrected_v2_df[
            "exact_status"
        ]
        == "OPTIMAL"
    )
    &
    (
        corrected_v2_df[
            "greedy_status"
        ]
        == "FEASIBLE"
    )
)


corrected_v2_comparable_df = (
    corrected_v2_df[
        corrected_v2_df[
            "is_comparable"
        ]
    ]
    .copy()
)


corrected_v2_comparable_df[
    "optimality_gap_percent"
] = pd.to_numeric(
    corrected_v2_comparable_df[
        "optimality_gap_percent"
    ],
    errors="coerce",
)


corrected_v2_comparable_df[
    "jaccard_similarity"
] = pd.to_numeric(
    corrected_v2_comparable_df[
        "jaccard_similarity"
    ],
    errors="coerce",
)


corrected_v2_comparable_df[
    "selected_count_difference"
] = pd.to_numeric(
    corrected_v2_comparable_df[
        "selected_count_difference"
    ],
    errors="coerce",
)


OPTIMAL_COST_TOLERANCE_PERCENT = 1e-10


corrected_v2_comparable_df[
    "greedy_optimal_cost_numeric"
] = (
    corrected_v2_comparable_df[
        "optimality_gap_percent"
    ]
    .abs()
    <= OPTIMAL_COST_TOLERANCE_PERCENT
).astype(int)


corrected_v2_comparable_df[
    "large_gap_numeric"
] = (
    corrected_v2_comparable_df[
        "optimality_gap_percent"
    ]
    > 10.0
).astype(int)


corrected_v2_comparable_df[
    "greedy_more_items_numeric"
] = (
    corrected_v2_comparable_df[
        "selected_count_difference"
    ]
    > 0
).astype(int)


corrected_v2_comparable_df[
    "within_1_percent_numeric"
] = (
    corrected_v2_comparable_df[
        "optimality_gap_percent"
    ]
    <= 1.0
).astype(int)


corrected_v2_comparable_df[
    "within_5_percent_numeric"
] = (
    corrected_v2_comparable_df[
        "optimality_gap_percent"
    ]
    <= 5.0
).astype(int)


corrected_v2_comparable_df[
    "within_10_percent_numeric"
] = (
    corrected_v2_comparable_df[
        "optimality_gap_percent"
    ]
    <= 10.0
).astype(int)


# --------------------------------------------------
# Overall condition-level descriptive summary
# --------------------------------------------------

gap_values = (
    corrected_v2_comparable_df[
        "optimality_gap_percent"
    ]
    .to_numpy(
        dtype=float
    )
)


overall_v2_descriptive_row = {
    "total_rows":
        len(
            corrected_v2_df
        ),

    "comparable_feasible_rows":
        len(
            corrected_v2_comparable_df
        ),

    "matched_groups":
        corrected_v2_df[
            "matched_group_id"
        ]
        .nunique(),

    "regime_specific_instances":
        corrected_v2_df[
            "instance_id"
        ]
        .nunique(),

    "mean_optimality_gap_percent":
        gap_values.mean(),

    "median_optimality_gap_percent":
        np.median(
            gap_values
        ),

    "p90_optimality_gap_percent":
        np.quantile(
            gap_values,
            0.90,
        ),

    "p95_optimality_gap_percent":
        np.quantile(
            gap_values,
            0.95,
        ),

    "maximum_optimality_gap_percent":
        gap_values.max(),

    "greedy_optimal_cost_percent":
        100.0
        *
        corrected_v2_comparable_df[
            "greedy_optimal_cost_numeric"
        ].mean(),

    "mean_jaccard_similarity":
        corrected_v2_comparable_df[
            "jaccard_similarity"
        ].mean(),

    "large_gap_percent":
        100.0
        *
        corrected_v2_comparable_df[
            "large_gap_numeric"
        ].mean(),

    "greedy_more_items_percent":
        100.0
        *
        corrected_v2_comparable_df[
            "greedy_more_items_numeric"
        ].mean(),

    "mean_selected_count_difference":
        corrected_v2_comparable_df[
            "selected_count_difference"
        ].mean(),

    "within_1_percent":
        100.0
        *
        corrected_v2_comparable_df[
            "within_1_percent_numeric"
        ].mean(),

    "within_5_percent":
        100.0
        *
        corrected_v2_comparable_df[
            "within_5_percent_numeric"
        ].mean(),

    "within_10_percent":
        100.0
        *
        corrected_v2_comparable_df[
            "within_10_percent_numeric"
        ].mean(),
}


for column_name in [
    "exact_selected_count",
    "greedy_selected_count",
    "exact_utility",
    "greedy_utility",
    "exact_privacy",
    "greedy_privacy",
    "exact_cognitive_cost",
    "greedy_cognitive_cost",
]:

    if column_name in corrected_v2_comparable_df.columns:

        corrected_v2_comparable_df[
            column_name
        ] = pd.to_numeric(
            corrected_v2_comparable_df[
                column_name
            ],
            errors="coerce",
        )


if (
    "exact_selected_count"
    in corrected_v2_comparable_df.columns
):

    overall_v2_descriptive_row[
        "mean_exact_selected_count"
    ] = (
        corrected_v2_comparable_df[
            "exact_selected_count"
        ].mean()
    )


if (
    "greedy_selected_count"
    in corrected_v2_comparable_df.columns
):

    overall_v2_descriptive_row[
        "mean_greedy_selected_count"
    ] = (
        corrected_v2_comparable_df[
            "greedy_selected_count"
        ].mean()
    )


if (
    "exact_utility"
    in corrected_v2_comparable_df.columns
    and
    "greedy_utility"
    in corrected_v2_comparable_df.columns
):

    overall_v2_descriptive_row[
        "mean_utility_difference"
    ] = (
        corrected_v2_comparable_df[
            "greedy_utility"
        ]
        -
        corrected_v2_comparable_df[
            "exact_utility"
        ]
    ).mean()


if (
    "exact_privacy"
    in corrected_v2_comparable_df.columns
    and
    "greedy_privacy"
    in corrected_v2_comparable_df.columns
):

    overall_v2_descriptive_row[
        "mean_privacy_difference"
    ] = (
        corrected_v2_comparable_df[
            "greedy_privacy"
        ]
        -
        corrected_v2_comparable_df[
            "exact_privacy"
        ]
    ).mean()


if (
    "exact_cognitive_cost"
    in corrected_v2_comparable_df.columns
    and
    "greedy_cognitive_cost"
    in corrected_v2_comparable_df.columns
):

    overall_v2_descriptive_row[
        "mean_cognitive_cost_difference"
    ] = (
        corrected_v2_comparable_df[
            "greedy_cognitive_cost"
        ]
        -
        corrected_v2_comparable_df[
            "exact_cognitive_cost"
        ]
    ).mean()


overall_v2_descriptive_df = pd.DataFrame(
    [
        overall_v2_descriptive_row
    ]
)


overall_v2_descriptive_df.to_csv(
    OVERALL_V2_DESCRIPTIVE_PATH,
    index=False,
)


# --------------------------------------------------
# Create matched-group sufficient statistics
# --------------------------------------------------

overall_v2_cluster_df = (
    corrected_v2_comparable_df
    .groupby(
        [
            "matched_group_id",
            "n_items",
        ],
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "size",
        ),

        gap_sum=(
            "optimality_gap_percent",
            "sum",
        ),

        optimal_cost_sum=(
            "greedy_optimal_cost_numeric",
            "sum",
        ),

        jaccard_sum=(
            "jaccard_similarity",
            "sum",
        ),

        large_gap_sum=(
            "large_gap_numeric",
            "sum",
        ),

        greedy_more_items_sum=(
            "greedy_more_items_numeric",
            "sum",
        ),

        selected_count_difference_sum=(
            "selected_count_difference",
            "sum",
        ),

        within_1_percent_sum=(
            "within_1_percent_numeric",
            "sum",
        ),

        within_5_percent_sum=(
            "within_5_percent_numeric",
            "sum",
        ),

        within_10_percent_sum=(
            "within_10_percent_numeric",
            "sum",
        ),
    )
)


overall_v2_aggregate_columns = [
    "comparable_rows",
    "gap_sum",
    "optimal_cost_sum",
    "jaccard_sum",
    "large_gap_sum",
    "greedy_more_items_sum",
    "selected_count_difference_sum",
    "within_1_percent_sum",
    "within_5_percent_sum",
    "within_10_percent_sum",
]


overall_v2_metric_names = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "greedy_more_items_percent",
    "mean_selected_count_difference",
    "within_1_percent",
    "within_5_percent",
    "within_10_percent",
]


# --------------------------------------------------
# Validate matched-group cluster structure
# --------------------------------------------------

assert (
    overall_v2_cluster_df[
        "matched_group_id"
    ].nunique()
    == 500
)


matched_groups_per_size = (
    overall_v2_cluster_df
    .groupby(
        "n_items"
    )[
        "matched_group_id"
    ]
    .nunique()
)


assert (
    matched_groups_per_size
    == 100
).all()


# --------------------------------------------------
# Convert aggregate totals into reporting metrics
# --------------------------------------------------

def calculate_overall_v2_metrics(
    totals: np.ndarray,
) -> np.ndarray:
    """
    Convert matched-group sufficient-statistic totals
    into overall benchmark quality metrics.
    """

    comparable_rows = totals[0]

    return np.array(
        [
            totals[1]
            / comparable_rows,

            100.0
            * totals[2]
            / comparable_rows,

            totals[3]
            / comparable_rows,

            100.0
            * totals[4]
            / comparable_rows,

            100.0
            * totals[5]
            / comparable_rows,

            totals[6]
            / comparable_rows,

            100.0
            * totals[7]
            / comparable_rows,

            100.0
            * totals[8]
            / comparable_rows,

            100.0
            * totals[9]
            / comparable_rows,
        ],
        dtype=float,
    )


# --------------------------------------------------
# Observed estimates
# --------------------------------------------------

observed_overall_v2_totals = (
    overall_v2_cluster_df[
        overall_v2_aggregate_columns
    ]
    .sum()
    .to_numpy(
        dtype=float
    )
)


observed_overall_v2_metrics = (
    calculate_overall_v2_metrics(
        observed_overall_v2_totals
    )
)


# --------------------------------------------------
# Preserve the five candidate-size strata
# --------------------------------------------------

overall_v2_stratum_arrays = []


for n_items, size_df in (
    overall_v2_cluster_df
    .groupby(
        "n_items",
        sort=True,
    )
):

    size_array = (
        size_df[
            overall_v2_aggregate_columns
        ]
        .to_numpy(
            dtype=float
        )
    )


    assert (
        size_array.shape[0]
        == 100
    )


    overall_v2_stratum_arrays.append(
        size_array
    )


# --------------------------------------------------
# Stratified matched-group bootstrap
# --------------------------------------------------

overall_v2_rng = np.random.default_rng(
    OVERALL_V2_BOOTSTRAP_SEED
)


overall_v2_bootstrap_estimates = np.empty(
    (
        OVERALL_V2_BOOTSTRAP_REPLICATES,
        len(
            overall_v2_metric_names
        ),
    ),
    dtype=float,
)


for bootstrap_index in range(
    OVERALL_V2_BOOTSTRAP_REPLICATES
):

    resampled_totals = np.zeros(
        len(
            overall_v2_aggregate_columns
        ),
        dtype=float,
    )


    for size_array in overall_v2_stratum_arrays:

        number_of_matched_groups = (
            size_array.shape[0]
        )


        sampled_indices = (
            overall_v2_rng.integers(
                low=0,
                high=number_of_matched_groups,
                size=number_of_matched_groups,
            )
        )


        resampled_totals += (
            size_array[
                sampled_indices
            ]
            .sum(
                axis=0
            )
        )


    overall_v2_bootstrap_estimates[
        bootstrap_index
    ] = calculate_overall_v2_metrics(
        resampled_totals
    )


# --------------------------------------------------
# Bootstrap summary
# --------------------------------------------------

overall_v2_standard_errors = (
    overall_v2_bootstrap_estimates.std(
        axis=0,
        ddof=1,
    )
)


overall_v2_ci_lower = np.quantile(
    overall_v2_bootstrap_estimates,
    0.025,
    axis=0,
)


overall_v2_ci_upper = np.quantile(
    overall_v2_bootstrap_estimates,
    0.975,
    axis=0,
)


overall_v2_bootstrap_summary_rows = []


for metric_index, metric_name in enumerate(
    overall_v2_metric_names
):

    overall_v2_bootstrap_summary_rows.append(
        {
            "metric":
                metric_name,

            "estimate":
                observed_overall_v2_metrics[
                    metric_index
                ],

            "bootstrap_standard_error":
                overall_v2_standard_errors[
                    metric_index
                ],

            "ci_95_lower":
                overall_v2_ci_lower[
                    metric_index
                ],

            "ci_95_upper":
                overall_v2_ci_upper[
                    metric_index
                ],

            "matched_groups":
                500,

            "candidate_size_strata":
                len(
                    EXACT_BENCHMARK_SIZES
                ),

            "bootstrap_replicates":
                OVERALL_V2_BOOTSTRAP_REPLICATES,

            "resampling_unit":
                "matched_size_replicate_group",

            "resampling_design":
                "stratified_by_candidate_set_size",
        }
    )


overall_v2_bootstrap_summary_df = pd.DataFrame(
    overall_v2_bootstrap_summary_rows
)


overall_v2_bootstrap_replicates_df = pd.DataFrame(
    overall_v2_bootstrap_estimates,
    columns=overall_v2_metric_names,
)


overall_v2_bootstrap_summary_df.to_csv(
    OVERALL_V2_BOOTSTRAP_SUMMARY_PATH,
    index=False,
)


overall_v2_bootstrap_replicates_df.to_csv(
    OVERALL_V2_BOOTSTRAP_REPLICATES_PATH,
    index=False,
)


# --------------------------------------------------
# Readable display tables
# --------------------------------------------------

descriptive_display_columns = [
    "mean_optimality_gap_percent",
    "median_optimality_gap_percent",
    "p90_optimality_gap_percent",
    "p95_optimality_gap_percent",
    "maximum_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "within_1_percent",
    "within_5_percent",
    "within_10_percent",
]


descriptive_display_df = (
    overall_v2_descriptive_df[
        descriptive_display_columns
    ]
    .T
    .reset_index()
)


descriptive_display_df.columns = [
    "metric",
    "estimate",
]


bootstrap_display_df = (
    overall_v2_bootstrap_summary_df
    .copy()
)


bootstrap_display_df[
    "estimate_and_ci"
] = (
    bootstrap_display_df[
        "estimate"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    bootstrap_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    bootstrap_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


print(
    "======================================"
)

print(
    "CORRECTED V2 OVERALL DESCRIPTIVE RESULTS"
)

print(
    "======================================"
)


print(
    descriptive_display_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.4f}",
    )
)


print(
    "\n======================================"
)

print(
    "CORRECTED V2 MATCHED-GROUP CLUSTERED 95% CIs"
)

print(
    "======================================"
)


print(
    bootstrap_display_df[
        [
            "metric",
            "estimate_and_ci",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\nOverall descriptive summary saved to:"
)

print(
    OVERALL_V2_DESCRIPTIVE_PATH
)


print(
    "\nBootstrap confidence intervals saved to:"
)

print(
    OVERALL_V2_BOOTSTRAP_SUMMARY_PATH
)


print(
    "\nBootstrap replicates saved to:"
)

print(
    OVERALL_V2_BOOTSTRAP_REPLICATES_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Corrected v2 regime-specific descriptive results
# and matched-group clustered confidence intervals
# --------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd


REGIME_V2_BOOTSTRAP_REPLICATES = 5000
REGIME_V2_BOOTSTRAP_SEED = GLOBAL_SEED + 101


if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


CORRECTED_ANALYSIS_READY_PATH = (
    RESULTS_DIR
    / (
        "exact_vs_greedy_main_benchmark_"
        "v2_marginal_preserving_analysis_ready.csv"
    )
)


REGIME_V2_DESCRIPTIVE_PATH = (
    TABLES_DIR
    / "corrected_v2_regime_descriptive_summary.csv"
)


REGIME_V2_BOOTSTRAP_SUMMARY_PATH = (
    TABLES_DIR
    / "corrected_v2_regime_matched_group_bootstrap_ci.csv"
)


REGIME_V2_BOOTSTRAP_REPLICATES_PATH = (
    RESULTS_DIR
    / "corrected_v2_regime_matched_group_bootstrap_replicates.csv"
)


# --------------------------------------------------
# Load corrected analysis-ready benchmark
# --------------------------------------------------

regime_v2_df = pd.read_csv(
    CORRECTED_ANALYSIS_READY_PATH,
    low_memory=False,
)


regime_v2_df[
    "is_comparable"
] = (
    (
        regime_v2_df[
            "exact_status"
        ]
        == "OPTIMAL"
    )
    &
    (
        regime_v2_df[
            "greedy_status"
        ]
        == "FEASIBLE"
    )
)


regime_v2_comparable_df = (
    regime_v2_df[
        regime_v2_df[
            "is_comparable"
        ]
    ]
    .copy()
)


# --------------------------------------------------
# Standardize numerical columns
# --------------------------------------------------

for column_name in [
    "n_items",
    "tau",
    "optimality_gap_percent",
    "jaccard_similarity",
    "selected_count_difference",
]:

    regime_v2_comparable_df[
        column_name
    ] = pd.to_numeric(
        regime_v2_comparable_df[
            column_name
        ],
        errors="coerce",
    )


OPTIMAL_COST_TOLERANCE_PERCENT = 1e-10


regime_v2_comparable_df[
    "greedy_optimal_cost_numeric"
] = (
    regime_v2_comparable_df[
        "optimality_gap_percent"
    ]
    .abs()
    <=
    OPTIMAL_COST_TOLERANCE_PERCENT
).astype(int)


regime_v2_comparable_df[
    "large_gap_numeric"
] = (
    regime_v2_comparable_df[
        "optimality_gap_percent"
    ]
    > 10.0
).astype(int)


regime_v2_comparable_df[
    "greedy_more_items_numeric"
] = (
    regime_v2_comparable_df[
        "selected_count_difference"
    ]
    > 0
).astype(int)


regime_v2_comparable_df[
    "within_1_percent_numeric"
] = (
    regime_v2_comparable_df[
        "optimality_gap_percent"
    ]
    <= 1.0
).astype(int)


regime_v2_comparable_df[
    "within_5_percent_numeric"
] = (
    regime_v2_comparable_df[
        "optimality_gap_percent"
    ]
    <= 5.0
).astype(int)


regime_v2_comparable_df[
    "within_10_percent_numeric"
] = (
    regime_v2_comparable_df[
        "optimality_gap_percent"
    ]
    <= 10.0
).astype(int)


# --------------------------------------------------
# Regime-specific descriptive summaries
# --------------------------------------------------

regime_descriptive_rows = []


for regime in INSTANCE_REGIMES:

    regime_df = (
        regime_v2_comparable_df[
            regime_v2_comparable_df[
                "regime"
            ]
            == regime
        ]
    )


    regime_gap_values = (
        regime_df[
            "optimality_gap_percent"
        ]
        .to_numpy(
            dtype=float
        )
    )


    regime_descriptive_rows.append(
        {
            "regime":
                regime,

            "matched_groups":
                regime_df[
                    "matched_group_id"
                ]
                .nunique(),

            "comparable_rows":
                len(
                    regime_df
                ),

            "feasibility_percent":
                100.0
                *
                len(
                    regime_df
                )
                /
                (
                    regime_v2_df[
                        regime_v2_df[
                            "regime"
                        ]
                        == regime
                    ]
                    .shape[0]
                ),

            "mean_optimality_gap_percent":
                regime_gap_values.mean(),

            "median_optimality_gap_percent":
                np.median(
                    regime_gap_values
                ),

            "p90_optimality_gap_percent":
                np.quantile(
                    regime_gap_values,
                    0.90,
                ),

            "p95_optimality_gap_percent":
                np.quantile(
                    regime_gap_values,
                    0.95,
                ),

            "maximum_optimality_gap_percent":
                regime_gap_values.max(),

            "greedy_optimal_cost_percent":
                100.0
                *
                regime_df[
                    "greedy_optimal_cost_numeric"
                ].mean(),

            "mean_jaccard_similarity":
                regime_df[
                    "jaccard_similarity"
                ].mean(),

            "large_gap_percent":
                100.0
                *
                regime_df[
                    "large_gap_numeric"
                ].mean(),

            "greedy_more_items_percent":
                100.0
                *
                regime_df[
                    "greedy_more_items_numeric"
                ].mean(),

            "mean_selected_count_difference":
                regime_df[
                    "selected_count_difference"
                ].mean(),

            "within_1_percent":
                100.0
                *
                regime_df[
                    "within_1_percent_numeric"
                ].mean(),

            "within_5_percent":
                100.0
                *
                regime_df[
                    "within_5_percent_numeric"
                ].mean(),

            "within_10_percent":
                100.0
                *
                regime_df[
                    "within_10_percent_numeric"
                ].mean(),
        }
    )


regime_v2_descriptive_df = pd.DataFrame(
    regime_descriptive_rows
)


regime_v2_descriptive_df.to_csv(
    REGIME_V2_DESCRIPTIVE_PATH,
    index=False,
)


# --------------------------------------------------
# Build one sufficient-statistics row per
# matched group and regime
# --------------------------------------------------

regime_v2_cluster_df = (
    regime_v2_comparable_df
    .groupby(
        [
            "matched_group_id",
            "n_items",
            "regime",
        ],
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "size",
        ),

        gap_sum=(
            "optimality_gap_percent",
            "sum",
        ),

        optimal_cost_sum=(
            "greedy_optimal_cost_numeric",
            "sum",
        ),

        jaccard_sum=(
            "jaccard_similarity",
            "sum",
        ),

        large_gap_sum=(
            "large_gap_numeric",
            "sum",
        ),

        greedy_more_items_sum=(
            "greedy_more_items_numeric",
            "sum",
        ),

        selected_count_difference_sum=(
            "selected_count_difference",
            "sum",
        ),

        within_1_percent_sum=(
            "within_1_percent_numeric",
            "sum",
        ),

        within_5_percent_sum=(
            "within_5_percent_numeric",
            "sum",
        ),

        within_10_percent_sum=(
            "within_10_percent_numeric",
            "sum",
        ),
    )
)


regime_v2_aggregate_columns = [
    "comparable_rows",
    "gap_sum",
    "optimal_cost_sum",
    "jaccard_sum",
    "large_gap_sum",
    "greedy_more_items_sum",
    "selected_count_difference_sum",
    "within_1_percent_sum",
    "within_5_percent_sum",
    "within_10_percent_sum",
]


regime_v2_metric_names = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "greedy_more_items_percent",
    "mean_selected_count_difference",
    "within_1_percent",
    "within_5_percent",
    "within_10_percent",
]


def calculate_regime_v2_metrics(
    totals: np.ndarray,
) -> np.ndarray:
    """
    Convert matched-group sufficient-statistic totals
    into regime-specific quality metrics.
    """

    comparable_rows = totals[0]

    return np.array(
        [
            totals[1]
            / comparable_rows,

            100.0
            * totals[2]
            / comparable_rows,

            totals[3]
            / comparable_rows,

            100.0
            * totals[4]
            / comparable_rows,

            100.0
            * totals[5]
            / comparable_rows,

            totals[6]
            / comparable_rows,

            100.0
            * totals[7]
            / comparable_rows,

            100.0
            * totals[8]
            / comparable_rows,

            100.0
            * totals[9]
            / comparable_rows,
        ],
        dtype=float,
    )


# --------------------------------------------------
# Validate cluster structure
# --------------------------------------------------

cluster_counts_by_regime_size = (
    regime_v2_cluster_df
    .groupby(
        [
            "regime",
            "n_items",
        ]
    )[
        "matched_group_id"
    ]
    .nunique()
)


assert (
    cluster_counts_by_regime_size
    == 100
).all()


# --------------------------------------------------
# Bootstrap separately for each regime
# --------------------------------------------------

regime_v2_rng = np.random.default_rng(
    REGIME_V2_BOOTSTRAP_SEED
)


regime_v2_bootstrap_summary_rows = []
regime_v2_bootstrap_replicate_rows = []


for regime in INSTANCE_REGIMES:

    current_regime_cluster_df = (
        regime_v2_cluster_df[
            regime_v2_cluster_df[
                "regime"
            ]
            == regime
        ]
        .copy()
    )


    observed_totals = (
        current_regime_cluster_df[
            regime_v2_aggregate_columns
        ]
        .sum()
        .to_numpy(
            dtype=float
        )
    )


    observed_metrics = (
        calculate_regime_v2_metrics(
            observed_totals
        )
    )


    # Preserve the five candidate-size strata.
    size_stratum_arrays = []


    for n_items, size_df in (
        current_regime_cluster_df
        .groupby(
            "n_items",
            sort=True,
        )
    ):

        size_array = (
            size_df[
                regime_v2_aggregate_columns
            ]
            .to_numpy(
                dtype=float
            )
        )


        assert (
            size_array.shape[0]
            == 100
        )


        size_stratum_arrays.append(
            size_array
        )


    bootstrap_estimates = np.empty(
        (
            REGIME_V2_BOOTSTRAP_REPLICATES,
            len(
                regime_v2_metric_names
            ),
        ),
        dtype=float,
    )


    for bootstrap_index in range(
        REGIME_V2_BOOTSTRAP_REPLICATES
    ):

        resampled_totals = np.zeros(
            len(
                regime_v2_aggregate_columns
            ),
            dtype=float,
        )


        for size_array in size_stratum_arrays:

            number_of_groups = (
                size_array.shape[0]
            )


            sampled_indices = (
                regime_v2_rng.integers(
                    low=0,
                    high=number_of_groups,
                    size=number_of_groups,
                )
            )


            resampled_totals += (
                size_array[
                    sampled_indices
                ]
                .sum(
                    axis=0
                )
            )


        bootstrap_estimates[
            bootstrap_index
        ] = calculate_regime_v2_metrics(
            resampled_totals
        )


    standard_errors = (
        bootstrap_estimates.std(
            axis=0,
            ddof=1,
        )
    )


    ci_lower = np.quantile(
        bootstrap_estimates,
        0.025,
        axis=0,
    )


    ci_upper = np.quantile(
        bootstrap_estimates,
        0.975,
        axis=0,
    )


    for metric_index, metric_name in enumerate(
        regime_v2_metric_names
    ):

        regime_v2_bootstrap_summary_rows.append(
            {
                "regime":
                    regime,

                "metric":
                    metric_name,

                "estimate":
                    observed_metrics[
                        metric_index
                    ],

                "bootstrap_standard_error":
                    standard_errors[
                        metric_index
                    ],

                "ci_95_lower":
                    ci_lower[
                        metric_index
                    ],

                "ci_95_upper":
                    ci_upper[
                        metric_index
                    ],

                "matched_groups":
                    500,

                "candidate_size_strata":
                    5,

                "bootstrap_replicates":
                    REGIME_V2_BOOTSTRAP_REPLICATES,

                "resampling_unit":
                    "matched_size_replicate_group",

                "resampling_design":
                    "stratified_by_candidate_set_size",
            }
        )


    replicate_df = pd.DataFrame(
        bootstrap_estimates,
        columns=regime_v2_metric_names,
    )


    replicate_df.insert(
        0,
        "bootstrap_replicate",
        np.arange(
            REGIME_V2_BOOTSTRAP_REPLICATES
        ),
    )


    replicate_df.insert(
        0,
        "regime",
        regime,
    )


    regime_v2_bootstrap_replicate_rows.append(
        replicate_df
    )


    print(
        "Completed corrected regime:",
        regime
    )


# --------------------------------------------------
# Save bootstrap outputs
# --------------------------------------------------

regime_v2_bootstrap_summary_df = pd.DataFrame(
    regime_v2_bootstrap_summary_rows
)


regime_v2_bootstrap_replicates_df = pd.concat(
    regime_v2_bootstrap_replicate_rows,
    ignore_index=True,
)


regime_v2_bootstrap_summary_df.to_csv(
    REGIME_V2_BOOTSTRAP_SUMMARY_PATH,
    index=False,
)


regime_v2_bootstrap_replicates_df.to_csv(
    REGIME_V2_BOOTSTRAP_REPLICATES_PATH,
    index=False,
)


# --------------------------------------------------
# Compact display
# --------------------------------------------------

key_regime_metrics = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "greedy_more_items_percent",
]


regime_v2_display_df = (
    regime_v2_bootstrap_summary_df[
        regime_v2_bootstrap_summary_df[
            "metric"
        ].isin(
            key_regime_metrics
        )
    ]
    .copy()
)


regime_v2_display_df[
    "estimate_and_ci"
] = (
    regime_v2_display_df[
        "estimate"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    regime_v2_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    regime_v2_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


regime_v2_pivot_df = (
    regime_v2_display_df
    .pivot(
        index="regime",
        columns="metric",
        values="estimate_and_ci",
    )
    .reset_index()
)


regime_order = [
    "independent",
    "utility_burden_alignment",
    "utility_burden_conflict",
]


regime_v2_pivot_df[
    "regime"
] = pd.Categorical(
    regime_v2_pivot_df[
        "regime"
    ],
    categories=regime_order,
    ordered=True,
)


regime_v2_pivot_df = (
    regime_v2_pivot_df
    .sort_values(
        "regime"
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n======================================"
)

print(
    "CORRECTED V2 REGIME DESCRIPTIVE RESULTS"
)

print(
    "======================================"
)


print(
    regime_v2_descriptive_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.4f}",
    )
)


print(
    "\n======================================"
)

print(
    "CORRECTED V2 REGIME MATCHED-GROUP 95% CIs"
)

print(
    "======================================"
)


print(
    regime_v2_pivot_df.to_string(
        index=False
    )
)


print(
    "\nRegime descriptive summary saved to:"
)

print(
    REGIME_V2_DESCRIPTIVE_PATH
)


print(
    "\nRegime bootstrap confidence intervals saved to:"
)

print(
    REGIME_V2_BOOTSTRAP_SUMMARY_PATH
)


print(
    "\nRegime bootstrap replicates saved to:"
)

print(
    REGIME_V2_BOOTSTRAP_REPLICATES_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Corrected v2 paired regime contrasts using matched
# size–replicate groups
# --------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd


REGIME_CONTRAST_BOOTSTRAP_REPLICATES = 5000
REGIME_CONTRAST_BOOTSTRAP_SEED = GLOBAL_SEED + 102


if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


CORRECTED_ANALYSIS_READY_PATH = (
    RESULTS_DIR
    / (
        "exact_vs_greedy_main_benchmark_"
        "v2_marginal_preserving_analysis_ready.csv"
    )
)


REGIME_CONTRAST_PATH = (
    TABLES_DIR
    / "corrected_v2_paired_regime_bootstrap_contrasts.csv"
)


REGIME_CONTRAST_REPLICATES_PATH = (
    RESULTS_DIR
    / "corrected_v2_paired_regime_bootstrap_contrast_replicates.csv"
)


# Each contrast is:
# first regime minus second regime.
REGIME_CONTRAST_COMPARISONS = [
    (
        "utility_burden_conflict",
        "independent",
    ),
    (
        "utility_burden_alignment",
        "independent",
    ),
    (
        "utility_burden_conflict",
        "utility_burden_alignment",
    ),
]


REGIME_CONTRAST_METRICS = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "greedy_more_items_percent",
    "mean_selected_count_difference",
    "within_1_percent",
    "within_5_percent",
    "within_10_percent",
]


# --------------------------------------------------
# Load and prepare corrected benchmark
# --------------------------------------------------

regime_contrast_source_df = pd.read_csv(
    CORRECTED_ANALYSIS_READY_PATH,
    low_memory=False,
)


regime_contrast_source_df[
    "is_comparable"
] = (
    (
        regime_contrast_source_df[
            "exact_status"
        ]
        == "OPTIMAL"
    )
    &
    (
        regime_contrast_source_df[
            "greedy_status"
        ]
        == "FEASIBLE"
    )
)


regime_contrast_source_df = (
    regime_contrast_source_df[
        regime_contrast_source_df[
            "is_comparable"
        ]
    ]
    .copy()
)


for column_name in [
    "n_items",
    "optimality_gap_percent",
    "jaccard_similarity",
    "selected_count_difference",
]:

    regime_contrast_source_df[
        column_name
    ] = pd.to_numeric(
        regime_contrast_source_df[
            column_name
        ],
        errors="coerce",
    )


OPTIMAL_COST_TOLERANCE_PERCENT = 1e-10


regime_contrast_source_df[
    "greedy_optimal_cost_numeric"
] = (
    regime_contrast_source_df[
        "optimality_gap_percent"
    ]
    .abs()
    <= OPTIMAL_COST_TOLERANCE_PERCENT
).astype(int)


regime_contrast_source_df[
    "large_gap_numeric"
] = (
    regime_contrast_source_df[
        "optimality_gap_percent"
    ]
    > 10.0
).astype(int)


regime_contrast_source_df[
    "greedy_more_items_numeric"
] = (
    regime_contrast_source_df[
        "selected_count_difference"
    ]
    > 0
).astype(int)


regime_contrast_source_df[
    "within_1_percent_numeric"
] = (
    regime_contrast_source_df[
        "optimality_gap_percent"
    ]
    <= 1.0
).astype(int)


regime_contrast_source_df[
    "within_5_percent_numeric"
] = (
    regime_contrast_source_df[
        "optimality_gap_percent"
    ]
    <= 5.0
).astype(int)


regime_contrast_source_df[
    "within_10_percent_numeric"
] = (
    regime_contrast_source_df[
        "optimality_gap_percent"
    ]
    <= 10.0
).astype(int)


# --------------------------------------------------
# One sufficient-statistics row per matched group
# and regime
# --------------------------------------------------

regime_contrast_cluster_df = (
    regime_contrast_source_df
    .groupby(
        [
            "matched_group_id",
            "n_items",
            "regime",
        ],
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "size",
        ),

        gap_sum=(
            "optimality_gap_percent",
            "sum",
        ),

        optimal_cost_sum=(
            "greedy_optimal_cost_numeric",
            "sum",
        ),

        jaccard_sum=(
            "jaccard_similarity",
            "sum",
        ),

        large_gap_sum=(
            "large_gap_numeric",
            "sum",
        ),

        greedy_more_items_sum=(
            "greedy_more_items_numeric",
            "sum",
        ),

        selected_count_difference_sum=(
            "selected_count_difference",
            "sum",
        ),

        within_1_percent_sum=(
            "within_1_percent_numeric",
            "sum",
        ),

        within_5_percent_sum=(
            "within_5_percent_numeric",
            "sum",
        ),

        within_10_percent_sum=(
            "within_10_percent_numeric",
            "sum",
        ),
    )
)


REGIME_AGGREGATE_COLUMNS = [
    "comparable_rows",
    "gap_sum",
    "optimal_cost_sum",
    "jaccard_sum",
    "large_gap_sum",
    "greedy_more_items_sum",
    "selected_count_difference_sum",
    "within_1_percent_sum",
    "within_5_percent_sum",
    "within_10_percent_sum",
]


def calculate_regime_contrast_metrics(
    totals: np.ndarray,
) -> np.ndarray:
    """
    Convert regime sufficient-statistic totals into
    benchmark quality metrics.
    """

    comparable_rows = totals[0]

    return np.array(
        [
            totals[1]
            / comparable_rows,

            100.0
            * totals[2]
            / comparable_rows,

            totals[3]
            / comparable_rows,

            100.0
            * totals[4]
            / comparable_rows,

            100.0
            * totals[5]
            / comparable_rows,

            totals[6]
            / comparable_rows,

            100.0
            * totals[7]
            / comparable_rows,

            100.0
            * totals[8]
            / comparable_rows,

            100.0
            * totals[9]
            / comparable_rows,
        ],
        dtype=float,
    )


# --------------------------------------------------
# Validate complete matched pairing
# --------------------------------------------------

expected_regimes = set(
    INSTANCE_REGIMES
)


regimes_per_group = (
    regime_contrast_cluster_df
    .groupby(
        "matched_group_id"
    )[
        "regime"
    ]
    .apply(set)
)


assert (
    regimes_per_group
    .apply(
        lambda value:
            value == expected_regimes
    )
    .all()
)


assert (
    regime_contrast_cluster_df[
        "matched_group_id"
    ]
    .nunique()
    == 500
)


# Feasible condition counts should be identical
# across matched regimes.
comparable_count_pivot = (
    regime_contrast_cluster_df
    .pivot(
        index="matched_group_id",
        columns="regime",
        values="comparable_rows",
    )
)


assert (
    comparable_count_pivot.nunique(
        axis=1
    )
    == 1
).all()


# --------------------------------------------------
# Construct matched arrays for each size stratum
# --------------------------------------------------

paired_size_strata = []


for n_items in EXACT_BENCHMARK_SIZES:

    regime_arrays = {}
    reference_group_ids = None


    for regime in INSTANCE_REGIMES:

        current_regime_df = (
            regime_contrast_cluster_df[
                (
                    regime_contrast_cluster_df[
                        "n_items"
                    ]
                    == n_items
                )
                &
                (
                    regime_contrast_cluster_df[
                        "regime"
                    ]
                    == regime
                )
            ]
            .sort_values(
                "matched_group_id"
            )
            .reset_index(
                drop=True
            )
        )


        assert len(
            current_regime_df
        ) == 100


        current_group_ids = (
            current_regime_df[
                "matched_group_id"
            ]
            .tolist()
        )


        if reference_group_ids is None:

            reference_group_ids = (
                current_group_ids
            )

        else:

            assert (
                current_group_ids
                ==
                reference_group_ids
            )


        regime_arrays[
            regime
        ] = (
            current_regime_df[
                REGIME_AGGREGATE_COLUMNS
            ]
            .to_numpy(
                dtype=float
            )
        )


    paired_size_strata.append(
        {
            "n_items":
                n_items,

            "group_ids":
                reference_group_ids,

            "arrays":
                regime_arrays,
        }
    )


# --------------------------------------------------
# Observed regime-level metrics
# --------------------------------------------------

observed_regime_metrics = {}


for regime in INSTANCE_REGIMES:

    observed_totals = (
        regime_contrast_cluster_df[
            regime_contrast_cluster_df[
                "regime"
            ]
            == regime
        ][
            REGIME_AGGREGATE_COLUMNS
        ]
        .sum()
        .to_numpy(
            dtype=float
        )
    )


    observed_regime_metrics[
        regime
    ] = calculate_regime_contrast_metrics(
        observed_totals
    )


# --------------------------------------------------
# Shared paired matched-group bootstrap
# --------------------------------------------------

regime_contrast_rng = np.random.default_rng(
    REGIME_CONTRAST_BOOTSTRAP_SEED
)


bootstrap_regime_metrics = {
    regime: np.empty(
        (
            REGIME_CONTRAST_BOOTSTRAP_REPLICATES,
            len(
                REGIME_CONTRAST_METRICS
            ),
        ),
        dtype=float,
    )
    for regime in INSTANCE_REGIMES
}


for bootstrap_index in range(
    REGIME_CONTRAST_BOOTSTRAP_REPLICATES
):

    resampled_totals_by_regime = {
        regime: np.zeros(
            len(
                REGIME_AGGREGATE_COLUMNS
            ),
            dtype=float,
        )
        for regime in INSTANCE_REGIMES
    }


    for size_stratum in paired_size_strata:

        number_of_groups = len(
            size_stratum[
                "group_ids"
            ]
        )


        # The same indices are used for every regime,
        # preserving matched-group pairing.
        sampled_indices = (
            regime_contrast_rng.integers(
                low=0,
                high=number_of_groups,
                size=number_of_groups,
            )
        )


        for regime in INSTANCE_REGIMES:

            regime_array = (
                size_stratum[
                    "arrays"
                ][
                    regime
                ]
            )


            resampled_totals_by_regime[
                regime
            ] += (
                regime_array[
                    sampled_indices
                ]
                .sum(
                    axis=0
                )
            )


    for regime in INSTANCE_REGIMES:

        bootstrap_regime_metrics[
            regime
        ][
            bootstrap_index
        ] = calculate_regime_contrast_metrics(
            resampled_totals_by_regime[
                regime
            ]
        )


# --------------------------------------------------
# Calculate paired contrasts
# --------------------------------------------------

regime_contrast_rows = []
regime_contrast_replicate_frames = []


for first_regime, second_regime in (
    REGIME_CONTRAST_COMPARISONS
):

    contrast_name = (
        f"{first_regime}"
        f" minus "
        f"{second_regime}"
    )


    observed_difference = (
        observed_regime_metrics[
            first_regime
        ]
        -
        observed_regime_metrics[
            second_regime
        ]
    )


    bootstrap_differences = (
        bootstrap_regime_metrics[
            first_regime
        ]
        -
        bootstrap_regime_metrics[
            second_regime
        ]
    )


    bootstrap_standard_errors = (
        bootstrap_differences.std(
            axis=0,
            ddof=1,
        )
    )


    confidence_lower = np.quantile(
        bootstrap_differences,
        0.025,
        axis=0,
    )


    confidence_upper = np.quantile(
        bootstrap_differences,
        0.975,
        axis=0,
    )


    for metric_index, metric_name in enumerate(
        REGIME_CONTRAST_METRICS
    ):

        regime_contrast_rows.append(
            {
                "first_regime":
                    first_regime,

                "second_regime":
                    second_regime,

                "contrast":
                    contrast_name,

                "metric":
                    metric_name,

                "estimated_difference":
                    observed_difference[
                        metric_index
                    ],

                "bootstrap_standard_error":
                    bootstrap_standard_errors[
                        metric_index
                    ],

                "ci_95_lower":
                    confidence_lower[
                        metric_index
                    ],

                "ci_95_upper":
                    confidence_upper[
                        metric_index
                    ],

                "ci_excludes_zero":
                    (
                        confidence_lower[
                            metric_index
                        ]
                        > 0.0
                    )
                    or
                    (
                        confidence_upper[
                            metric_index
                        ]
                        < 0.0
                    ),

                "matched_groups":
                    500,

                "candidate_size_strata":
                    5,

                "bootstrap_replicates":
                    REGIME_CONTRAST_BOOTSTRAP_REPLICATES,

                "resampling_unit":
                    "matched_size_replicate_group",

                "resampling_design":
                    (
                        "paired_across_regimes_"
                        "stratified_by_candidate_set_size"
                    ),

                "multiple_comparison_adjustment":
                    "none",
            }
        )


    contrast_replicate_df = pd.DataFrame(
        bootstrap_differences,
        columns=REGIME_CONTRAST_METRICS,
    )


    contrast_replicate_df.insert(
        0,
        "bootstrap_replicate",
        np.arange(
            REGIME_CONTRAST_BOOTSTRAP_REPLICATES
        ),
    )


    contrast_replicate_df.insert(
        0,
        "contrast",
        contrast_name,
    )


    regime_contrast_replicate_frames.append(
        contrast_replicate_df
    )


    print(
        "Completed paired corrected-v2 contrast:",
        contrast_name
    )


# --------------------------------------------------
# Save outputs
# --------------------------------------------------

regime_contrast_df = pd.DataFrame(
    regime_contrast_rows
)


regime_contrast_replicates_df = pd.concat(
    regime_contrast_replicate_frames,
    ignore_index=True,
)


regime_contrast_df.to_csv(
    REGIME_CONTRAST_PATH,
    index=False,
)


regime_contrast_replicates_df.to_csv(
    REGIME_CONTRAST_REPLICATES_PATH,
    index=False,
)


# --------------------------------------------------
# Readable display
# --------------------------------------------------

key_contrast_metrics = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "greedy_more_items_percent",
    "mean_selected_count_difference",
]


regime_contrast_display_df = (
    regime_contrast_df[
        regime_contrast_df[
            "metric"
        ].isin(
            key_contrast_metrics
        )
    ]
    .copy()
)


regime_contrast_display_df[
    "difference_and_ci"
] = (
    regime_contrast_display_df[
        "estimated_difference"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    regime_contrast_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    regime_contrast_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


print(
    "\n======================================"
)

print(
    "CORRECTED V2 PAIRED REGIME CONTRASTS"
)

print(
    "======================================"
)


print(
    regime_contrast_display_df[
        [
            "contrast",
            "metric",
            "difference_and_ci",
            "ci_excludes_zero",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\nPaired regime contrasts saved to:"
)

print(
    REGIME_CONTRAST_PATH
)


print(
    "\nPaired contrast replicates saved to:"
)

print(
    REGIME_CONTRAST_REPLICATES_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Corrected v2 candidate-set-size descriptive results
# and matched-group clustered confidence intervals
# --------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd


SIZE_V2_BOOTSTRAP_REPLICATES = 5000
SIZE_V2_BOOTSTRAP_SEED = GLOBAL_SEED + 103


if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


CORRECTED_ANALYSIS_READY_PATH = (
    RESULTS_DIR
    / (
        "exact_vs_greedy_main_benchmark_"
        "v2_marginal_preserving_analysis_ready.csv"
    )
)


SIZE_V2_DESCRIPTIVE_PATH = (
    TABLES_DIR
    / "corrected_v2_candidate_size_descriptive_summary.csv"
)


SIZE_V2_BOOTSTRAP_SUMMARY_PATH = (
    TABLES_DIR
    / "corrected_v2_candidate_size_matched_group_bootstrap_ci.csv"
)


SIZE_V2_BOOTSTRAP_REPLICATES_PATH = (
    RESULTS_DIR
    / "corrected_v2_candidate_size_matched_group_bootstrap_replicates.csv"
)


# --------------------------------------------------
# Load corrected analysis-ready benchmark
# --------------------------------------------------

size_v2_df = pd.read_csv(
    CORRECTED_ANALYSIS_READY_PATH,
    low_memory=False,
)


size_v2_df[
    "is_comparable"
] = (
    (
        size_v2_df[
            "exact_status"
        ]
        == "OPTIMAL"
    )
    &
    (
        size_v2_df[
            "greedy_status"
        ]
        == "FEASIBLE"
    )
)


for column_name in [
    "n_items",
    "optimality_gap_percent",
    "jaccard_similarity",
    "selected_count_difference",
]:

    size_v2_df[
        column_name
    ] = pd.to_numeric(
        size_v2_df[
            column_name
        ],
        errors="coerce",
    )


size_v2_comparable_df = (
    size_v2_df[
        size_v2_df[
            "is_comparable"
        ]
    ]
    .copy()
)


# --------------------------------------------------
# Construct reporting indicators
# --------------------------------------------------

OPTIMAL_COST_TOLERANCE_PERCENT = 1e-10


size_v2_comparable_df[
    "greedy_optimal_cost_numeric"
] = (
    size_v2_comparable_df[
        "optimality_gap_percent"
    ]
    .abs()
    <=
    OPTIMAL_COST_TOLERANCE_PERCENT
).astype(int)


size_v2_comparable_df[
    "large_gap_numeric"
] = (
    size_v2_comparable_df[
        "optimality_gap_percent"
    ]
    > 10.0
).astype(int)


size_v2_comparable_df[
    "greedy_more_items_numeric"
] = (
    size_v2_comparable_df[
        "selected_count_difference"
    ]
    > 0
).astype(int)


size_v2_comparable_df[
    "within_1_percent_numeric"
] = (
    size_v2_comparable_df[
        "optimality_gap_percent"
    ]
    <= 1.0
).astype(int)


size_v2_comparable_df[
    "within_5_percent_numeric"
] = (
    size_v2_comparable_df[
        "optimality_gap_percent"
    ]
    <= 5.0
).astype(int)


size_v2_comparable_df[
    "within_10_percent_numeric"
] = (
    size_v2_comparable_df[
        "optimality_gap_percent"
    ]
    <= 10.0
).astype(int)


# --------------------------------------------------
# Candidate-size descriptive summaries
# --------------------------------------------------

size_descriptive_rows = []


for n_items in EXACT_BENCHMARK_SIZES:

    all_size_df = (
        size_v2_df[
            size_v2_df[
                "n_items"
            ]
            == n_items
        ]
    )


    comparable_size_df = (
        size_v2_comparable_df[
            size_v2_comparable_df[
                "n_items"
            ]
            == n_items
        ]
    )


    gap_values = (
        comparable_size_df[
            "optimality_gap_percent"
        ]
        .to_numpy(
            dtype=float
        )
    )


    size_descriptive_rows.append(
        {
            "n_items":
                n_items,

            "matched_groups":
                all_size_df[
                    "matched_group_id"
                ]
                .nunique(),

            "regime_specific_instances":
                all_size_df[
                    "instance_id"
                ]
                .nunique(),

            "total_rows":
                len(
                    all_size_df
                ),

            "comparable_rows":
                len(
                    comparable_size_df
                ),

            "feasibility_percent":
                100.0
                *
                len(
                    comparable_size_df
                )
                /
                len(
                    all_size_df
                ),

            "mean_optimality_gap_percent":
                gap_values.mean(),

            "median_optimality_gap_percent":
                np.median(
                    gap_values
                ),

            "p90_optimality_gap_percent":
                np.quantile(
                    gap_values,
                    0.90,
                ),

            "p95_optimality_gap_percent":
                np.quantile(
                    gap_values,
                    0.95,
                ),

            "maximum_optimality_gap_percent":
                gap_values.max(),

            "greedy_optimal_cost_percent":
                100.0
                *
                comparable_size_df[
                    "greedy_optimal_cost_numeric"
                ].mean(),

            "mean_jaccard_similarity":
                comparable_size_df[
                    "jaccard_similarity"
                ].mean(),

            "large_gap_percent":
                100.0
                *
                comparable_size_df[
                    "large_gap_numeric"
                ].mean(),

            "greedy_more_items_percent":
                100.0
                *
                comparable_size_df[
                    "greedy_more_items_numeric"
                ].mean(),

            "mean_selected_count_difference":
                comparable_size_df[
                    "selected_count_difference"
                ].mean(),

            "within_1_percent":
                100.0
                *
                comparable_size_df[
                    "within_1_percent_numeric"
                ].mean(),

            "within_5_percent":
                100.0
                *
                comparable_size_df[
                    "within_5_percent_numeric"
                ].mean(),

            "within_10_percent":
                100.0
                *
                comparable_size_df[
                    "within_10_percent_numeric"
                ].mean(),
        }
    )


size_v2_descriptive_df = pd.DataFrame(
    size_descriptive_rows
)


size_v2_descriptive_df.to_csv(
    SIZE_V2_DESCRIPTIVE_PATH,
    index=False,
)


# --------------------------------------------------
# Build one sufficient-statistics row per matched
# size–replicate group
# --------------------------------------------------

size_v2_cluster_df = (
    size_v2_comparable_df
    .groupby(
        [
            "matched_group_id",
            "n_items",
        ],
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "size",
        ),

        gap_sum=(
            "optimality_gap_percent",
            "sum",
        ),

        optimal_cost_sum=(
            "greedy_optimal_cost_numeric",
            "sum",
        ),

        jaccard_sum=(
            "jaccard_similarity",
            "sum",
        ),

        large_gap_sum=(
            "large_gap_numeric",
            "sum",
        ),

        greedy_more_items_sum=(
            "greedy_more_items_numeric",
            "sum",
        ),

        selected_count_difference_sum=(
            "selected_count_difference",
            "sum",
        ),

        within_1_percent_sum=(
            "within_1_percent_numeric",
            "sum",
        ),

        within_5_percent_sum=(
            "within_5_percent_numeric",
            "sum",
        ),

        within_10_percent_sum=(
            "within_10_percent_numeric",
            "sum",
        ),
    )
)


SIZE_V2_AGGREGATE_COLUMNS = [
    "comparable_rows",
    "gap_sum",
    "optimal_cost_sum",
    "jaccard_sum",
    "large_gap_sum",
    "greedy_more_items_sum",
    "selected_count_difference_sum",
    "within_1_percent_sum",
    "within_5_percent_sum",
    "within_10_percent_sum",
]


SIZE_V2_METRIC_NAMES = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "greedy_more_items_percent",
    "mean_selected_count_difference",
    "within_1_percent",
    "within_5_percent",
    "within_10_percent",
]


def calculate_size_v2_metrics(
    totals: np.ndarray,
) -> np.ndarray:
    """
    Convert matched-group sufficient-statistic totals
    into candidate-size quality metrics.
    """

    comparable_rows = totals[0]

    return np.array(
        [
            totals[1]
            / comparable_rows,

            100.0
            * totals[2]
            / comparable_rows,

            totals[3]
            / comparable_rows,

            100.0
            * totals[4]
            / comparable_rows,

            100.0
            * totals[5]
            / comparable_rows,

            totals[6]
            / comparable_rows,

            100.0
            * totals[7]
            / comparable_rows,

            100.0
            * totals[8]
            / comparable_rows,

            100.0
            * totals[9]
            / comparable_rows,
        ],
        dtype=float,
    )


# --------------------------------------------------
# Validate the matched-group structure
# --------------------------------------------------

groups_per_size = (
    size_v2_cluster_df
    .groupby(
        "n_items"
    )[
        "matched_group_id"
    ]
    .nunique()
)


assert (
    groups_per_size
    == 100
).all()


# --------------------------------------------------
# Bootstrap separately within each candidate size
# --------------------------------------------------

size_v2_rng = np.random.default_rng(
    SIZE_V2_BOOTSTRAP_SEED
)


size_v2_bootstrap_summary_rows = []
size_v2_bootstrap_replicate_frames = []


for n_items in EXACT_BENCHMARK_SIZES:

    current_size_cluster_df = (
        size_v2_cluster_df[
            size_v2_cluster_df[
                "n_items"
            ]
            == n_items
        ]
        .sort_values(
            "matched_group_id"
        )
        .reset_index(
            drop=True
        )
    )


    size_array = (
        current_size_cluster_df[
            SIZE_V2_AGGREGATE_COLUMNS
        ]
        .to_numpy(
            dtype=float
        )
    )


    assert (
        size_array.shape[0]
        == 100
    )


    observed_totals = (
        size_array.sum(
            axis=0
        )
    )


    observed_metrics = (
        calculate_size_v2_metrics(
            observed_totals
        )
    )


    bootstrap_estimates = np.empty(
        (
            SIZE_V2_BOOTSTRAP_REPLICATES,
            len(
                SIZE_V2_METRIC_NAMES
            ),
        ),
        dtype=float,
    )


    for bootstrap_index in range(
        SIZE_V2_BOOTSTRAP_REPLICATES
    ):

        sampled_indices = (
            size_v2_rng.integers(
                low=0,
                high=len(
                    size_array
                ),
                size=len(
                    size_array
                ),
            )
        )


        resampled_totals = (
            size_array[
                sampled_indices
            ]
            .sum(
                axis=0
            )
        )


        bootstrap_estimates[
            bootstrap_index
        ] = calculate_size_v2_metrics(
            resampled_totals
        )


    standard_errors = (
        bootstrap_estimates.std(
            axis=0,
            ddof=1,
        )
    )


    confidence_lower = np.quantile(
        bootstrap_estimates,
        0.025,
        axis=0,
    )


    confidence_upper = np.quantile(
        bootstrap_estimates,
        0.975,
        axis=0,
    )


    for metric_index, metric_name in enumerate(
        SIZE_V2_METRIC_NAMES
    ):

        size_v2_bootstrap_summary_rows.append(
            {
                "n_items":
                    n_items,

                "metric":
                    metric_name,

                "estimate":
                    observed_metrics[
                        metric_index
                    ],

                "bootstrap_standard_error":
                    standard_errors[
                        metric_index
                    ],

                "ci_95_lower":
                    confidence_lower[
                        metric_index
                    ],

                "ci_95_upper":
                    confidence_upper[
                        metric_index
                    ],

                "matched_groups":
                    100,

                "bootstrap_replicates":
                    SIZE_V2_BOOTSTRAP_REPLICATES,

                "resampling_unit":
                    "matched_size_replicate_group",

                "resampling_design":
                    "within_candidate_set_size",
            }
        )


    replicate_df = pd.DataFrame(
        bootstrap_estimates,
        columns=SIZE_V2_METRIC_NAMES,
    )


    replicate_df.insert(
        0,
        "bootstrap_replicate",
        np.arange(
            SIZE_V2_BOOTSTRAP_REPLICATES
        ),
    )


    replicate_df.insert(
        0,
        "n_items",
        n_items,
    )


    size_v2_bootstrap_replicate_frames.append(
        replicate_df
    )


    print(
        "Completed corrected candidate size:",
        n_items
    )


# --------------------------------------------------
# Save bootstrap outputs
# --------------------------------------------------

size_v2_bootstrap_summary_df = pd.DataFrame(
    size_v2_bootstrap_summary_rows
)


size_v2_bootstrap_replicates_df = pd.concat(
    size_v2_bootstrap_replicate_frames,
    ignore_index=True,
)


size_v2_bootstrap_summary_df.to_csv(
    SIZE_V2_BOOTSTRAP_SUMMARY_PATH,
    index=False,
)


size_v2_bootstrap_replicates_df.to_csv(
    SIZE_V2_BOOTSTRAP_REPLICATES_PATH,
    index=False,
)


# --------------------------------------------------
# Create compact confidence-interval display
# --------------------------------------------------

key_size_metrics = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "greedy_more_items_percent",
]


size_v2_display_df = (
    size_v2_bootstrap_summary_df[
        size_v2_bootstrap_summary_df[
            "metric"
        ].isin(
            key_size_metrics
        )
    ]
    .copy()
)


size_v2_display_df[
    "estimate_and_ci"
] = (
    size_v2_display_df[
        "estimate"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    size_v2_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    size_v2_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


size_v2_pivot_df = (
    size_v2_display_df
    .pivot(
        index="n_items",
        columns="metric",
        values="estimate_and_ci",
    )
    .reset_index()
    .sort_values(
        "n_items"
    )
)


print(
    "\n======================================"
)

print(
    "CORRECTED V2 CANDIDATE-SIZE DESCRIPTIVE RESULTS"
)

print(
    "======================================"
)


print(
    size_v2_descriptive_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.4f}",
    )
)


print(
    "\n======================================"
)

print(
    "CORRECTED V2 CANDIDATE-SIZE MATCHED-GROUP 95% CIs"
)

print(
    "======================================"
)


print(
    size_v2_pivot_df.to_string(
        index=False
    )
)


print(
    "\nCandidate-size descriptive summary saved to:"
)

print(
    SIZE_V2_DESCRIPTIVE_PATH
)


print(
    "\nCandidate-size confidence intervals saved to:"
)

print(
    SIZE_V2_BOOTSTRAP_SUMMARY_PATH
)


print(
    "\nCandidate-size bootstrap replicates saved to:"
)

print(
    SIZE_V2_BOOTSTRAP_REPLICATES_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Corrected v2 candidate-size bootstrap contrasts
# --------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd


SIZE_CONTRAST_BOOTSTRAP_REPLICATES = 5000
SIZE_CONTRAST_BOOTSTRAP_SEED = GLOBAL_SEED + 104


if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


CORRECTED_ANALYSIS_READY_PATH = (
    RESULTS_DIR
    / (
        "exact_vs_greedy_main_benchmark_"
        "v2_marginal_preserving_analysis_ready.csv"
    )
)


SIZE_CONTRAST_PATH = (
    TABLES_DIR
    / "corrected_v2_pairwise_candidate_size_bootstrap_contrasts.csv"
)


SIZE_CONTRAST_REPLICATES_PATH = (
    RESULTS_DIR
    / "corrected_v2_pairwise_candidate_size_bootstrap_contrast_replicates.csv"
)


EXACT_SIZE_LEVELS = [
    8,
    10,
    12,
    14,
    16,
]


# Each contrast is:
# larger candidate set minus smaller candidate set.
SIZE_CONTRAST_COMPARISONS = [
    (10, 8),
    (12, 10),
    (14, 12),
    (16, 14),
    (16, 8),
]


SIZE_CONTRAST_METRICS = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "greedy_more_items_percent",
    "mean_selected_count_difference",
    "within_1_percent",
    "within_5_percent",
    "within_10_percent",
]


# --------------------------------------------------
# Load corrected benchmark
# --------------------------------------------------

size_contrast_source_df = pd.read_csv(
    CORRECTED_ANALYSIS_READY_PATH,
    low_memory=False,
)


size_contrast_source_df[
    "is_comparable"
] = (
    (
        size_contrast_source_df[
            "exact_status"
        ]
        == "OPTIMAL"
    )
    &
    (
        size_contrast_source_df[
            "greedy_status"
        ]
        == "FEASIBLE"
    )
)


size_contrast_source_df = (
    size_contrast_source_df[
        size_contrast_source_df[
            "is_comparable"
        ]
    ]
    .copy()
)


for column_name in [
    "n_items",
    "optimality_gap_percent",
    "jaccard_similarity",
    "selected_count_difference",
]:

    size_contrast_source_df[
        column_name
    ] = pd.to_numeric(
        size_contrast_source_df[
            column_name
        ],
        errors="coerce",
    )


# --------------------------------------------------
# Reporting indicators
# --------------------------------------------------

OPTIMAL_COST_TOLERANCE_PERCENT = 1e-10


size_contrast_source_df[
    "greedy_optimal_cost_numeric"
] = (
    size_contrast_source_df[
        "optimality_gap_percent"
    ]
    .abs()
    <= OPTIMAL_COST_TOLERANCE_PERCENT
).astype(int)


size_contrast_source_df[
    "large_gap_numeric"
] = (
    size_contrast_source_df[
        "optimality_gap_percent"
    ]
    > 10.0
).astype(int)


size_contrast_source_df[
    "greedy_more_items_numeric"
] = (
    size_contrast_source_df[
        "selected_count_difference"
    ]
    > 0
).astype(int)


size_contrast_source_df[
    "within_1_percent_numeric"
] = (
    size_contrast_source_df[
        "optimality_gap_percent"
    ]
    <= 1.0
).astype(int)


size_contrast_source_df[
    "within_5_percent_numeric"
] = (
    size_contrast_source_df[
        "optimality_gap_percent"
    ]
    <= 5.0
).astype(int)


size_contrast_source_df[
    "within_10_percent_numeric"
] = (
    size_contrast_source_df[
        "optimality_gap_percent"
    ]
    <= 10.0
).astype(int)


# --------------------------------------------------
# One sufficient-statistics row per matched-regime
# group within each candidate-set size
# --------------------------------------------------

size_contrast_cluster_df = (
    size_contrast_source_df
    .groupby(
        [
            "matched_group_id",
            "n_items",
        ],
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "size",
        ),

        gap_sum=(
            "optimality_gap_percent",
            "sum",
        ),

        optimal_cost_sum=(
            "greedy_optimal_cost_numeric",
            "sum",
        ),

        jaccard_sum=(
            "jaccard_similarity",
            "sum",
        ),

        large_gap_sum=(
            "large_gap_numeric",
            "sum",
        ),

        greedy_more_items_sum=(
            "greedy_more_items_numeric",
            "sum",
        ),

        selected_count_difference_sum=(
            "selected_count_difference",
            "sum",
        ),

        within_1_percent_sum=(
            "within_1_percent_numeric",
            "sum",
        ),

        within_5_percent_sum=(
            "within_5_percent_numeric",
            "sum",
        ),

        within_10_percent_sum=(
            "within_10_percent_numeric",
            "sum",
        ),
    )
)


SIZE_CONTRAST_AGGREGATE_COLUMNS = [
    "comparable_rows",
    "gap_sum",
    "optimal_cost_sum",
    "jaccard_sum",
    "large_gap_sum",
    "greedy_more_items_sum",
    "selected_count_difference_sum",
    "within_1_percent_sum",
    "within_5_percent_sum",
    "within_10_percent_sum",
]


def calculate_size_contrast_metrics(
    totals: np.ndarray,
) -> np.ndarray:
    """
    Convert candidate-size sufficient-statistic totals
    into benchmark quality metrics.
    """

    comparable_rows = totals[0]

    return np.array(
        [
            totals[1]
            / comparable_rows,

            100.0
            * totals[2]
            / comparable_rows,

            totals[3]
            / comparable_rows,

            100.0
            * totals[4]
            / comparable_rows,

            100.0
            * totals[5]
            / comparable_rows,

            totals[6]
            / comparable_rows,

            100.0
            * totals[7]
            / comparable_rows,

            100.0
            * totals[8]
            / comparable_rows,

            100.0
            * totals[9]
            / comparable_rows,
        ],
        dtype=float,
    )


# --------------------------------------------------
# Validate and store candidate-size arrays
# --------------------------------------------------

size_arrays = {}
observed_size_metrics = {}


for n_items in EXACT_SIZE_LEVELS:

    current_size_df = (
        size_contrast_cluster_df[
            size_contrast_cluster_df[
                "n_items"
            ]
            == n_items
        ]
        .sort_values(
            "matched_group_id"
        )
        .reset_index(
            drop=True
        )
    )


    assert len(
        current_size_df
    ) == 100


    current_size_array = (
        current_size_df[
            SIZE_CONTRAST_AGGREGATE_COLUMNS
        ]
        .to_numpy(
            dtype=float
        )
    )


    size_arrays[
        n_items
    ] = current_size_array


    observed_size_metrics[
        n_items
    ] = calculate_size_contrast_metrics(
        current_size_array.sum(
            axis=0
        )
    )


# --------------------------------------------------
# Independent bootstrap contrasts across sizes
# --------------------------------------------------

size_contrast_rng = np.random.default_rng(
    SIZE_CONTRAST_BOOTSTRAP_SEED
)


size_contrast_rows = []
size_contrast_replicate_frames = []


for larger_size, smaller_size in (
    SIZE_CONTRAST_COMPARISONS
):

    larger_array = (
        size_arrays[
            larger_size
        ]
    )

    smaller_array = (
        size_arrays[
            smaller_size
        ]
    )


    observed_difference = (
        observed_size_metrics[
            larger_size
        ]
        -
        observed_size_metrics[
            smaller_size
        ]
    )


    bootstrap_differences = np.empty(
        (
            SIZE_CONTRAST_BOOTSTRAP_REPLICATES,
            len(
                SIZE_CONTRAST_METRICS
            ),
        ),
        dtype=float,
    )


    for bootstrap_index in range(
        SIZE_CONTRAST_BOOTSTRAP_REPLICATES
    ):

        # Candidate sizes use different generated
        # instances, so each size is resampled
        # independently.
        larger_sample_indices = (
            size_contrast_rng.integers(
                low=0,
                high=len(
                    larger_array
                ),
                size=len(
                    larger_array
                ),
            )
        )


        smaller_sample_indices = (
            size_contrast_rng.integers(
                low=0,
                high=len(
                    smaller_array
                ),
                size=len(
                    smaller_array
                ),
            )
        )


        larger_totals = (
            larger_array[
                larger_sample_indices
            ]
            .sum(
                axis=0
            )
        )


        smaller_totals = (
            smaller_array[
                smaller_sample_indices
            ]
            .sum(
                axis=0
            )
        )


        bootstrap_differences[
            bootstrap_index
        ] = (
            calculate_size_contrast_metrics(
                larger_totals
            )
            -
            calculate_size_contrast_metrics(
                smaller_totals
            )
        )


    bootstrap_standard_errors = (
        bootstrap_differences.std(
            axis=0,
            ddof=1,
        )
    )


    confidence_lower = np.quantile(
        bootstrap_differences,
        0.025,
        axis=0,
    )


    confidence_upper = np.quantile(
        bootstrap_differences,
        0.975,
        axis=0,
    )


    contrast_name = (
        f"M={larger_size}"
        f" minus "
        f"M={smaller_size}"
    )


    for metric_index, metric_name in enumerate(
        SIZE_CONTRAST_METRICS
    ):

        size_contrast_rows.append(
            {
                "larger_size":
                    larger_size,

                "smaller_size":
                    smaller_size,

                "contrast":
                    contrast_name,

                "metric":
                    metric_name,

                "estimated_difference":
                    observed_difference[
                        metric_index
                    ],

                "bootstrap_standard_error":
                    bootstrap_standard_errors[
                        metric_index
                    ],

                "ci_95_lower":
                    confidence_lower[
                        metric_index
                    ],

                "ci_95_upper":
                    confidence_upper[
                        metric_index
                    ],

                "ci_excludes_zero":
                    (
                        confidence_lower[
                            metric_index
                        ]
                        > 0.0
                    )
                    or
                    (
                        confidence_upper[
                            metric_index
                        ]
                        < 0.0
                    ),

                "matched_groups_per_size":
                    100,

                "bootstrap_replicates":
                    SIZE_CONTRAST_BOOTSTRAP_REPLICATES,

                "resampling_unit":
                    "matched_regime_group",

                "resampling_design":
                    (
                        "independent_resampling_"
                        "within_each_candidate_size"
                    ),

                "estimand":
                    "quality_conditional_on_feasibility",

                "multiple_comparison_adjustment":
                    "none",
            }
        )


    replicate_df = pd.DataFrame(
        bootstrap_differences,
        columns=SIZE_CONTRAST_METRICS,
    )


    replicate_df.insert(
        0,
        "bootstrap_replicate",
        np.arange(
            SIZE_CONTRAST_BOOTSTRAP_REPLICATES
        ),
    )


    replicate_df.insert(
        0,
        "contrast",
        contrast_name,
    )


    size_contrast_replicate_frames.append(
        replicate_df
    )


    print(
        "Completed corrected size contrast:",
        contrast_name
    )


# --------------------------------------------------
# Save contrast outputs
# --------------------------------------------------

size_contrast_df = pd.DataFrame(
    size_contrast_rows
)


size_contrast_replicates_df = pd.concat(
    size_contrast_replicate_frames,
    ignore_index=True,
)


size_contrast_df.to_csv(
    SIZE_CONTRAST_PATH,
    index=False,
)


size_contrast_replicates_df.to_csv(
    SIZE_CONTRAST_REPLICATES_PATH,
    index=False,
)


# --------------------------------------------------
# Readable display
# --------------------------------------------------

key_size_contrast_metrics = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "greedy_more_items_percent",
    "mean_selected_count_difference",
]


size_contrast_display_df = (
    size_contrast_df[
        size_contrast_df[
            "metric"
        ].isin(
            key_size_contrast_metrics
        )
    ]
    .copy()
)


size_contrast_display_df[
    "difference_and_ci"
] = (
    size_contrast_display_df[
        "estimated_difference"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    size_contrast_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    size_contrast_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


print(
    "\n======================================"
)

print(
    "CORRECTED V2 CANDIDATE-SIZE CONTRASTS"
)

print(
    "======================================"
)


print(
    size_contrast_display_df[
        [
            "contrast",
            "metric",
            "difference_and_ci",
            "ci_excludes_zero",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\nCandidate-size contrasts saved to:"
)

print(
    SIZE_CONTRAST_PATH
)


print(
    "\nCandidate-size contrast replicates saved to:"
)

print(
    SIZE_CONTRAST_REPLICATES_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Corrected v2 threshold-specific descriptive results
# and matched-group clustered confidence intervals
# --------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd


THRESHOLD_V2_BOOTSTRAP_REPLICATES = 5000
THRESHOLD_V2_BOOTSTRAP_SEED = GLOBAL_SEED + 105


if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


CORRECTED_ANALYSIS_READY_PATH = (
    RESULTS_DIR
    / (
        "exact_vs_greedy_main_benchmark_"
        "v2_marginal_preserving_analysis_ready.csv"
    )
)


THRESHOLD_V2_DESCRIPTIVE_PATH = (
    TABLES_DIR
    / "corrected_v2_threshold_descriptive_summary.csv"
)


THRESHOLD_V2_FEASIBILITY_PATH = (
    TABLES_DIR
    / "corrected_v2_threshold_feasible_matched_group_counts.csv"
)


THRESHOLD_V2_BOOTSTRAP_SUMMARY_PATH = (
    TABLES_DIR
    / "corrected_v2_threshold_matched_group_bootstrap_ci.csv"
)


THRESHOLD_V2_BOOTSTRAP_REPLICATES_PATH = (
    RESULTS_DIR
    / "corrected_v2_threshold_matched_group_bootstrap_replicates.csv"
)


# --------------------------------------------------
# Load corrected analysis-ready benchmark
# --------------------------------------------------

threshold_v2_df = pd.read_csv(
    CORRECTED_ANALYSIS_READY_PATH,
    low_memory=False,
)


threshold_v2_df[
    "is_comparable"
] = (
    (
        threshold_v2_df[
            "exact_status"
        ]
        == "OPTIMAL"
    )
    &
    (
        threshold_v2_df[
            "greedy_status"
        ]
        == "FEASIBLE"
    )
)


threshold_v2_df[
    "is_both_infeasible"
] = (
    (
        threshold_v2_df[
            "exact_status"
        ]
        == "INFEASIBLE"
    )
    &
    (
        threshold_v2_df[
            "greedy_status"
        ]
        == "INFEASIBLE"
    )
)


for column_name in [
    "n_items",
    "tau",
    "optimality_gap_percent",
    "jaccard_similarity",
    "selected_count_difference",
]:

    threshold_v2_df[
        column_name
    ] = pd.to_numeric(
        threshold_v2_df[
            column_name
        ],
        errors="coerce",
    )


threshold_v2_comparable_df = (
    threshold_v2_df[
        threshold_v2_df[
            "is_comparable"
        ]
    ]
    .copy()
)


# --------------------------------------------------
# Reporting indicators
# --------------------------------------------------

OPTIMAL_COST_TOLERANCE_PERCENT = 1e-10


threshold_v2_comparable_df[
    "greedy_optimal_cost_numeric"
] = (
    threshold_v2_comparable_df[
        "optimality_gap_percent"
    ]
    .abs()
    <= OPTIMAL_COST_TOLERANCE_PERCENT
).astype(int)


threshold_v2_comparable_df[
    "large_gap_numeric"
] = (
    threshold_v2_comparable_df[
        "optimality_gap_percent"
    ]
    > 10.0
).astype(int)


threshold_v2_comparable_df[
    "greedy_more_items_numeric"
] = (
    threshold_v2_comparable_df[
        "selected_count_difference"
    ]
    > 0
).astype(int)


threshold_v2_comparable_df[
    "within_1_percent_numeric"
] = (
    threshold_v2_comparable_df[
        "optimality_gap_percent"
    ]
    <= 1.0
).astype(int)


threshold_v2_comparable_df[
    "within_5_percent_numeric"
] = (
    threshold_v2_comparable_df[
        "optimality_gap_percent"
    ]
    <= 5.0
).astype(int)


threshold_v2_comparable_df[
    "within_10_percent_numeric"
] = (
    threshold_v2_comparable_df[
        "optimality_gap_percent"
    ]
    <= 10.0
).astype(int)


# --------------------------------------------------
# Feasible matched-group counts
#
# Each feasible matched group contributes:
# 3 regimes × 4 weight profiles = 12 rows per tau.
# --------------------------------------------------

feasible_group_counts_df = (
    threshold_v2_comparable_df[
        [
            "matched_group_id",
            "n_items",
            "tau",
        ]
    ]
    .drop_duplicates()
    .groupby(
        [
            "tau",
            "n_items",
        ],
        as_index=False,
    )
    .agg(
        feasible_matched_groups=(
            "matched_group_id",
            "nunique",
        )
    )
)


complete_threshold_size_grid = pd.MultiIndex.from_product(
    [
        sorted(
            threshold_v2_df[
                "tau"
            ]
            .dropna()
            .unique()
        ),
        sorted(
            threshold_v2_df[
                "n_items"
            ]
            .dropna()
            .unique()
        ),
    ],
    names=[
        "tau",
        "n_items",
    ],
).to_frame(
    index=False
)


feasible_group_counts_df = (
    complete_threshold_size_grid
    .merge(
        feasible_group_counts_df,
        on=[
            "tau",
            "n_items",
        ],
        how="left",
    )
)


feasible_group_counts_df[
    "feasible_matched_groups"
] = (
    feasible_group_counts_df[
        "feasible_matched_groups"
    ]
    .fillna(0)
    .astype(int)
)


feasible_group_counts_df[
    "total_matched_groups"
] = 100


feasible_group_counts_df[
    "feasibility_percent"
] = (
    100.0
    *
    feasible_group_counts_df[
        "feasible_matched_groups"
    ]
    /
    feasible_group_counts_df[
        "total_matched_groups"
    ]
)


feasible_group_counts_df.to_csv(
    THRESHOLD_V2_FEASIBILITY_PATH,
    index=False,
)


# --------------------------------------------------
# Threshold-specific descriptive summaries
# --------------------------------------------------

threshold_descriptive_rows = []


for tau in sorted(
    threshold_v2_df[
        "tau"
    ]
    .dropna()
    .unique()
):

    all_tau_df = (
        threshold_v2_df[
            threshold_v2_df[
                "tau"
            ]
            == tau
        ]
    )


    comparable_tau_df = (
        threshold_v2_comparable_df[
            threshold_v2_comparable_df[
                "tau"
            ]
            == tau
        ]
    )


    gap_values = (
        comparable_tau_df[
            "optimality_gap_percent"
        ]
        .to_numpy(
            dtype=float
        )
    )


    feasible_matched_groups = (
        comparable_tau_df[
            "matched_group_id"
        ]
        .nunique()
    )


    threshold_descriptive_rows.append(
        {
            "tau":
                tau,

            "total_matched_groups":
                all_tau_df[
                    "matched_group_id"
                ]
                .nunique(),

            "feasible_matched_groups":
                feasible_matched_groups,

            "total_rows":
                len(
                    all_tau_df
                ),

            "comparable_rows":
                len(
                    comparable_tau_df
                ),

            "infeasible_rows":
                int(
                    all_tau_df[
                        "is_both_infeasible"
                    ].sum()
                ),

            "feasibility_percent":
                100.0
                *
                len(
                    comparable_tau_df
                )
                /
                len(
                    all_tau_df
                ),

            "mean_optimality_gap_percent":
                gap_values.mean(),

            "median_optimality_gap_percent":
                np.median(
                    gap_values
                ),

            "p90_optimality_gap_percent":
                np.quantile(
                    gap_values,
                    0.90,
                ),

            "p95_optimality_gap_percent":
                np.quantile(
                    gap_values,
                    0.95,
                ),

            "maximum_optimality_gap_percent":
                gap_values.max(),

            "greedy_optimal_cost_percent":
                100.0
                *
                comparable_tau_df[
                    "greedy_optimal_cost_numeric"
                ].mean(),

            "mean_jaccard_similarity":
                comparable_tau_df[
                    "jaccard_similarity"
                ].mean(),

            "large_gap_percent":
                100.0
                *
                comparable_tau_df[
                    "large_gap_numeric"
                ].mean(),

            "greedy_more_items_percent":
                100.0
                *
                comparable_tau_df[
                    "greedy_more_items_numeric"
                ].mean(),

            "mean_selected_count_difference":
                comparable_tau_df[
                    "selected_count_difference"
                ].mean(),

            "within_1_percent":
                100.0
                *
                comparable_tau_df[
                    "within_1_percent_numeric"
                ].mean(),

            "within_5_percent":
                100.0
                *
                comparable_tau_df[
                    "within_5_percent_numeric"
                ].mean(),

            "within_10_percent":
                100.0
                *
                comparable_tau_df[
                    "within_10_percent_numeric"
                ].mean(),
        }
    )


threshold_v2_descriptive_df = pd.DataFrame(
    threshold_descriptive_rows
)


threshold_v2_descriptive_df.to_csv(
    THRESHOLD_V2_DESCRIPTIVE_PATH,
    index=False,
)


# --------------------------------------------------
# One sufficient-statistics row per feasible matched
# group at each threshold
# --------------------------------------------------

threshold_v2_cluster_df = (
    threshold_v2_comparable_df
    .groupby(
        [
            "matched_group_id",
            "n_items",
            "tau",
        ],
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "size",
        ),

        gap_sum=(
            "optimality_gap_percent",
            "sum",
        ),

        optimal_cost_sum=(
            "greedy_optimal_cost_numeric",
            "sum",
        ),

        jaccard_sum=(
            "jaccard_similarity",
            "sum",
        ),

        large_gap_sum=(
            "large_gap_numeric",
            "sum",
        ),

        greedy_more_items_sum=(
            "greedy_more_items_numeric",
            "sum",
        ),

        selected_count_difference_sum=(
            "selected_count_difference",
            "sum",
        ),

        within_1_percent_sum=(
            "within_1_percent_numeric",
            "sum",
        ),

        within_5_percent_sum=(
            "within_5_percent_numeric",
            "sum",
        ),

        within_10_percent_sum=(
            "within_10_percent_numeric",
            "sum",
        ),
    )
)


# Every feasible group should contain exactly
# 3 regimes × 4 weight profiles = 12 observations.
assert (
    threshold_v2_cluster_df[
        "comparable_rows"
    ]
    == 12
).all()


THRESHOLD_V2_AGGREGATE_COLUMNS = [
    "comparable_rows",
    "gap_sum",
    "optimal_cost_sum",
    "jaccard_sum",
    "large_gap_sum",
    "greedy_more_items_sum",
    "selected_count_difference_sum",
    "within_1_percent_sum",
    "within_5_percent_sum",
    "within_10_percent_sum",
]


THRESHOLD_V2_METRIC_NAMES = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "greedy_more_items_percent",
    "mean_selected_count_difference",
    "within_1_percent",
    "within_5_percent",
    "within_10_percent",
]


def calculate_threshold_v2_metrics(
    totals: np.ndarray,
) -> np.ndarray:
    """
    Convert feasible matched-group sufficient-statistic
    totals into threshold-specific quality metrics.
    """

    comparable_rows = totals[0]

    return np.array(
        [
            totals[1]
            / comparable_rows,

            100.0
            * totals[2]
            / comparable_rows,

            totals[3]
            / comparable_rows,

            100.0
            * totals[4]
            / comparable_rows,

            100.0
            * totals[5]
            / comparable_rows,

            totals[6]
            / comparable_rows,

            100.0
            * totals[7]
            / comparable_rows,

            100.0
            * totals[8]
            / comparable_rows,

            100.0
            * totals[9]
            / comparable_rows,
        ],
        dtype=float,
    )


# --------------------------------------------------
# Bootstrap separately for each threshold
# --------------------------------------------------

threshold_v2_rng = np.random.default_rng(
    THRESHOLD_V2_BOOTSTRAP_SEED
)


threshold_v2_bootstrap_summary_rows = []
threshold_v2_bootstrap_replicate_frames = []


for tau in sorted(
    threshold_v2_cluster_df[
        "tau"
    ]
    .unique()
):

    current_tau_df = (
        threshold_v2_cluster_df[
            threshold_v2_cluster_df[
                "tau"
            ]
            == tau
        ]
        .copy()
    )


    observed_totals = (
        current_tau_df[
            THRESHOLD_V2_AGGREGATE_COLUMNS
        ]
        .sum()
        .to_numpy(
            dtype=float
        )
    )


    observed_metrics = (
        calculate_threshold_v2_metrics(
            observed_totals
        )
    )


    # Preserve candidate-set-size strata.
    size_stratum_arrays = []
    size_stratum_counts = []


    for n_items, size_df in (
        current_tau_df
        .groupby(
            "n_items",
            sort=True,
        )
    ):

        size_array = (
            size_df[
                THRESHOLD_V2_AGGREGATE_COLUMNS
            ]
            .to_numpy(
                dtype=float
            )
        )


        if len(
            size_array
        ) == 0:

            continue


        size_stratum_arrays.append(
            size_array
        )


        size_stratum_counts.append(
            {
                "n_items":
                    n_items,

                "feasible_groups":
                    len(
                        size_array
                    ),
            }
        )


    bootstrap_estimates = np.empty(
        (
            THRESHOLD_V2_BOOTSTRAP_REPLICATES,
            len(
                THRESHOLD_V2_METRIC_NAMES
            ),
        ),
        dtype=float,
    )


    for bootstrap_index in range(
        THRESHOLD_V2_BOOTSTRAP_REPLICATES
    ):

        resampled_totals = np.zeros(
            len(
                THRESHOLD_V2_AGGREGATE_COLUMNS
            ),
            dtype=float,
        )


        for size_array in size_stratum_arrays:

            number_of_feasible_groups = (
                len(
                    size_array
                )
            )


            sampled_indices = (
                threshold_v2_rng.integers(
                    low=0,
                    high=number_of_feasible_groups,
                    size=number_of_feasible_groups,
                )
            )


            resampled_totals += (
                size_array[
                    sampled_indices
                ]
                .sum(
                    axis=0
                )
            )


        bootstrap_estimates[
            bootstrap_index
        ] = calculate_threshold_v2_metrics(
            resampled_totals
        )


    standard_errors = (
        bootstrap_estimates.std(
            axis=0,
            ddof=1,
        )
    )


    confidence_lower = np.quantile(
        bootstrap_estimates,
        0.025,
        axis=0,
    )


    confidence_upper = np.quantile(
        bootstrap_estimates,
        0.975,
        axis=0,
    )


    for metric_index, metric_name in enumerate(
        THRESHOLD_V2_METRIC_NAMES
    ):

        threshold_v2_bootstrap_summary_rows.append(
            {
                "tau":
                    tau,

                "metric":
                    metric_name,

                "estimate":
                    observed_metrics[
                        metric_index
                    ],

                "bootstrap_standard_error":
                    standard_errors[
                        metric_index
                    ],

                "ci_95_lower":
                    confidence_lower[
                        metric_index
                    ],

                "ci_95_upper":
                    confidence_upper[
                        metric_index
                    ],

                "feasible_matched_groups":
                    current_tau_df[
                        "matched_group_id"
                    ]
                    .nunique(),

                "minimum_feasible_groups_per_size":
                    min(
                        item[
                            "feasible_groups"
                        ]
                        for item
                        in size_stratum_counts
                    ),

                "maximum_feasible_groups_per_size":
                    max(
                        item[
                            "feasible_groups"
                        ]
                        for item
                        in size_stratum_counts
                    ),

                "bootstrap_replicates":
                    THRESHOLD_V2_BOOTSTRAP_REPLICATES,

                "resampling_unit":
                    "feasible_matched_size_replicate_group",

                "resampling_design":
                    "stratified_by_candidate_set_size",

                "estimand":
                    "quality_conditional_on_feasibility",
            }
        )


    replicate_df = pd.DataFrame(
        bootstrap_estimates,
        columns=THRESHOLD_V2_METRIC_NAMES,
    )


    replicate_df.insert(
        0,
        "bootstrap_replicate",
        np.arange(
            THRESHOLD_V2_BOOTSTRAP_REPLICATES
        ),
    )


    replicate_df.insert(
        0,
        "tau",
        tau,
    )


    threshold_v2_bootstrap_replicate_frames.append(
        replicate_df
    )


    print(
        "Completed corrected threshold:",
        tau
    )


# --------------------------------------------------
# Save bootstrap outputs
# --------------------------------------------------

threshold_v2_bootstrap_summary_df = pd.DataFrame(
    threshold_v2_bootstrap_summary_rows
)


threshold_v2_bootstrap_replicates_df = pd.concat(
    threshold_v2_bootstrap_replicate_frames,
    ignore_index=True,
)


threshold_v2_bootstrap_summary_df.to_csv(
    THRESHOLD_V2_BOOTSTRAP_SUMMARY_PATH,
    index=False,
)


threshold_v2_bootstrap_replicates_df.to_csv(
    THRESHOLD_V2_BOOTSTRAP_REPLICATES_PATH,
    index=False,
)


# --------------------------------------------------
# Compact display
# --------------------------------------------------

key_threshold_metrics = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "greedy_more_items_percent",
]


threshold_v2_display_df = (
    threshold_v2_bootstrap_summary_df[
        threshold_v2_bootstrap_summary_df[
            "metric"
        ].isin(
            key_threshold_metrics
        )
    ]
    .copy()
)


threshold_v2_display_df[
    "estimate_and_ci"
] = (
    threshold_v2_display_df[
        "estimate"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    threshold_v2_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    threshold_v2_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


threshold_v2_pivot_df = (
    threshold_v2_display_df
    .pivot(
        index="tau",
        columns="metric",
        values="estimate_and_ci",
    )
    .reset_index()
    .sort_values(
        "tau"
    )
)


print(
    "\n======================================"
)

print(
    "CORRECTED V2 THRESHOLD DESCRIPTIVE RESULTS"
)

print(
    "======================================"
)


print(
    threshold_v2_descriptive_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.4f}",
    )
)


print(
    "\n======================================"
)

print(
    "CORRECTED V2 THRESHOLD MATCHED-GROUP 95% CIs"
)

print(
    "======================================"
)


print(
    threshold_v2_pivot_df.to_string(
        index=False
    )
)


print(
    "\nFeasible matched groups by threshold and size:"
)


print(
    feasible_group_counts_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.4f}",
    )
)


print(
    "\nThreshold descriptive summary saved to:"
)

print(
    THRESHOLD_V2_DESCRIPTIVE_PATH
)


print(
    "\nThreshold feasibility table saved to:"
)

print(
    THRESHOLD_V2_FEASIBILITY_PATH
)


print(
    "\nThreshold confidence intervals saved to:"
)

print(
    THRESHOLD_V2_BOOTSTRAP_SUMMARY_PATH
)


print(
    "\nThreshold bootstrap replicates saved to:"
)

print(
    THRESHOLD_V2_BOOTSTRAP_REPLICATES_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Corrected v2 paired threshold contrasts using only
# matched groups feasible at both thresholds
# --------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd


THRESHOLD_CONTRAST_V2_REPLICATES = 5000
THRESHOLD_CONTRAST_V2_SEED = GLOBAL_SEED + 106


if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


CORRECTED_ANALYSIS_READY_PATH = (
    RESULTS_DIR
    / (
        "exact_vs_greedy_main_benchmark_"
        "v2_marginal_preserving_analysis_ready.csv"
    )
)


THRESHOLD_CONTRAST_V2_PATH = (
    TABLES_DIR
    / (
        "corrected_v2_paired_threshold_"
        "common_support_bootstrap_contrasts.csv"
    )
)


THRESHOLD_CONTRAST_V2_SUPPORT_PATH = (
    TABLES_DIR
    / (
        "corrected_v2_paired_threshold_"
        "common_support_counts.csv"
    )
)


THRESHOLD_CONTRAST_V2_REPLICATES_PATH = (
    RESULTS_DIR
    / (
        "corrected_v2_paired_threshold_"
        "common_support_bootstrap_replicates.csv"
    )
)


# Each contrast is:
# higher threshold minus lower threshold.
THRESHOLD_V2_COMPARISONS = [
    (0.80, 0.70),
    (0.90, 0.80),
    (0.95, 0.90),
    (0.95, 0.70),
]


THRESHOLD_V2_CONTRAST_METRICS = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "greedy_more_items_percent",
    "mean_selected_count_difference",
    "within_1_percent",
    "within_5_percent",
    "within_10_percent",
]


# --------------------------------------------------
# Load and prepare corrected benchmark
# --------------------------------------------------

threshold_contrast_v2_df = pd.read_csv(
    CORRECTED_ANALYSIS_READY_PATH,
    low_memory=False,
)


threshold_contrast_v2_df[
    "is_comparable"
] = (
    (
        threshold_contrast_v2_df[
            "exact_status"
        ]
        == "OPTIMAL"
    )
    &
    (
        threshold_contrast_v2_df[
            "greedy_status"
        ]
        == "FEASIBLE"
    )
)


threshold_contrast_v2_df = (
    threshold_contrast_v2_df[
        threshold_contrast_v2_df[
            "is_comparable"
        ]
    ]
    .copy()
)


for column_name in [
    "n_items",
    "tau",
    "optimality_gap_percent",
    "jaccard_similarity",
    "selected_count_difference",
]:

    threshold_contrast_v2_df[
        column_name
    ] = pd.to_numeric(
        threshold_contrast_v2_df[
            column_name
        ],
        errors="coerce",
    )


# --------------------------------------------------
# Reporting indicators
# --------------------------------------------------

OPTIMAL_COST_TOLERANCE_PERCENT = 1e-10


threshold_contrast_v2_df[
    "greedy_optimal_cost_numeric"
] = (
    threshold_contrast_v2_df[
        "optimality_gap_percent"
    ]
    .abs()
    <= OPTIMAL_COST_TOLERANCE_PERCENT
).astype(int)


threshold_contrast_v2_df[
    "large_gap_numeric"
] = (
    threshold_contrast_v2_df[
        "optimality_gap_percent"
    ]
    > 10.0
).astype(int)


threshold_contrast_v2_df[
    "greedy_more_items_numeric"
] = (
    threshold_contrast_v2_df[
        "selected_count_difference"
    ]
    > 0
).astype(int)


threshold_contrast_v2_df[
    "within_1_percent_numeric"
] = (
    threshold_contrast_v2_df[
        "optimality_gap_percent"
    ]
    <= 1.0
).astype(int)


threshold_contrast_v2_df[
    "within_5_percent_numeric"
] = (
    threshold_contrast_v2_df[
        "optimality_gap_percent"
    ]
    <= 5.0
).astype(int)


threshold_contrast_v2_df[
    "within_10_percent_numeric"
] = (
    threshold_contrast_v2_df[
        "optimality_gap_percent"
    ]
    <= 10.0
).astype(int)


# --------------------------------------------------
# One sufficient-statistics row per matched group
# and threshold
#
# A feasible matched group contains:
# 3 regimes × 4 weight profiles = 12 rows.
# --------------------------------------------------

threshold_contrast_v2_cluster_df = (
    threshold_contrast_v2_df
    .groupby(
        [
            "matched_group_id",
            "n_items",
            "tau",
        ],
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "size",
        ),

        gap_sum=(
            "optimality_gap_percent",
            "sum",
        ),

        optimal_cost_sum=(
            "greedy_optimal_cost_numeric",
            "sum",
        ),

        jaccard_sum=(
            "jaccard_similarity",
            "sum",
        ),

        large_gap_sum=(
            "large_gap_numeric",
            "sum",
        ),

        greedy_more_items_sum=(
            "greedy_more_items_numeric",
            "sum",
        ),

        selected_count_difference_sum=(
            "selected_count_difference",
            "sum",
        ),

        within_1_percent_sum=(
            "within_1_percent_numeric",
            "sum",
        ),

        within_5_percent_sum=(
            "within_5_percent_numeric",
            "sum",
        ),

        within_10_percent_sum=(
            "within_10_percent_numeric",
            "sum",
        ),
    )
)


assert (
    threshold_contrast_v2_cluster_df[
        "comparable_rows"
    ]
    == 12
).all()


THRESHOLD_V2_AGGREGATE_COLUMNS = [
    "comparable_rows",
    "gap_sum",
    "optimal_cost_sum",
    "jaccard_sum",
    "large_gap_sum",
    "greedy_more_items_sum",
    "selected_count_difference_sum",
    "within_1_percent_sum",
    "within_5_percent_sum",
    "within_10_percent_sum",
]


def calculate_threshold_v2_contrast_metrics(
    totals: np.ndarray,
) -> np.ndarray:
    """
    Convert common-support matched-group totals into
    threshold-specific quality metrics.
    """

    comparable_rows = totals[0]

    return np.array(
        [
            totals[1]
            / comparable_rows,

            100.0
            * totals[2]
            / comparable_rows,

            totals[3]
            / comparable_rows,

            100.0
            * totals[4]
            / comparable_rows,

            100.0
            * totals[5]
            / comparable_rows,

            totals[6]
            / comparable_rows,

            100.0
            * totals[7]
            / comparable_rows,

            100.0
            * totals[8]
            / comparable_rows,

            100.0
            * totals[9]
            / comparable_rows,
        ],
        dtype=float,
    )


# --------------------------------------------------
# Paired common-support bootstrap
# --------------------------------------------------

threshold_contrast_v2_rng = np.random.default_rng(
    THRESHOLD_CONTRAST_V2_SEED
)


threshold_contrast_v2_rows = []
threshold_contrast_v2_support_rows = []
threshold_contrast_v2_replicate_frames = []


candidate_size_levels = sorted(
    threshold_contrast_v2_cluster_df[
        "n_items"
    ]
    .unique()
)


for higher_tau, lower_tau in (
    THRESHOLD_V2_COMPARISONS
):

    paired_size_strata = []
    total_common_groups = 0


    # ----------------------------------------------
    # Pair identical feasible matched groups within
    # each candidate-set-size stratum
    # ----------------------------------------------

    for n_items in candidate_size_levels:

        higher_size_df = (
            threshold_contrast_v2_cluster_df[
                (
                    threshold_contrast_v2_cluster_df[
                        "tau"
                    ]
                    == higher_tau
                )
                &
                (
                    threshold_contrast_v2_cluster_df[
                        "n_items"
                    ]
                    == n_items
                )
            ]
            .copy()
        )


        lower_size_df = (
            threshold_contrast_v2_cluster_df[
                (
                    threshold_contrast_v2_cluster_df[
                        "tau"
                    ]
                    == lower_tau
                )
                &
                (
                    threshold_contrast_v2_cluster_df[
                        "n_items"
                    ]
                    == n_items
                )
            ]
            .copy()
        )


        common_group_ids = sorted(
            set(
                higher_size_df[
                    "matched_group_id"
                ]
            )
            &
            set(
                lower_size_df[
                    "matched_group_id"
                ]
            )
        )


        if len(
            common_group_ids
        ) == 0:

            continue


        higher_common_df = (
            higher_size_df[
                higher_size_df[
                    "matched_group_id"
                ].isin(
                    common_group_ids
                )
            ]
            .sort_values(
                "matched_group_id"
            )
            .reset_index(
                drop=True
            )
        )


        lower_common_df = (
            lower_size_df[
                lower_size_df[
                    "matched_group_id"
                ].isin(
                    common_group_ids
                )
            ]
            .sort_values(
                "matched_group_id"
            )
            .reset_index(
                drop=True
            )
        )


        assert (
            higher_common_df[
                "matched_group_id"
            ].tolist()
            ==
            lower_common_df[
                "matched_group_id"
            ].tolist()
        )


        higher_array = (
            higher_common_df[
                THRESHOLD_V2_AGGREGATE_COLUMNS
            ]
            .to_numpy(
                dtype=float
            )
        )


        lower_array = (
            lower_common_df[
                THRESHOLD_V2_AGGREGATE_COLUMNS
            ]
            .to_numpy(
                dtype=float
            )
        )


        paired_size_strata.append(
            {
                "n_items":
                    n_items,

                "higher_array":
                    higher_array,

                "lower_array":
                    lower_array,
            }
        )


        common_count = len(
            common_group_ids
        )


        total_common_groups += (
            common_count
        )


        threshold_contrast_v2_support_rows.append(
            {
                "contrast":
                    (
                        f"tau={higher_tau:.2f}"
                        f" minus "
                        f"tau={lower_tau:.2f}"
                    ),

                "higher_tau":
                    higher_tau,

                "lower_tau":
                    lower_tau,

                "n_items":
                    n_items,

                "common_feasible_matched_groups":
                    common_count,

                "total_possible_matched_groups":
                    100,

                "common_support_percent":
                    100.0
                    * common_count
                    / 100.0,
            }
        )


    # ----------------------------------------------
    # Observed common-support estimates
    # ----------------------------------------------

    observed_higher_totals = np.zeros(
        len(
            THRESHOLD_V2_AGGREGATE_COLUMNS
        ),
        dtype=float,
    )


    observed_lower_totals = np.zeros(
        len(
            THRESHOLD_V2_AGGREGATE_COLUMNS
        ),
        dtype=float,
    )


    for size_stratum in paired_size_strata:

        observed_higher_totals += (
            size_stratum[
                "higher_array"
            ]
            .sum(
                axis=0
            )
        )


        observed_lower_totals += (
            size_stratum[
                "lower_array"
            ]
            .sum(
                axis=0
            )
        )


    observed_higher_metrics = (
        calculate_threshold_v2_contrast_metrics(
            observed_higher_totals
        )
    )


    observed_lower_metrics = (
        calculate_threshold_v2_contrast_metrics(
            observed_lower_totals
        )
    )


    observed_difference = (
        observed_higher_metrics
        -
        observed_lower_metrics
    )


    # ----------------------------------------------
    # Paired stratified bootstrap
    # ----------------------------------------------

    bootstrap_differences = np.empty(
        (
            THRESHOLD_CONTRAST_V2_REPLICATES,
            len(
                THRESHOLD_V2_CONTRAST_METRICS
            ),
        ),
        dtype=float,
    )


    for bootstrap_index in range(
        THRESHOLD_CONTRAST_V2_REPLICATES
    ):

        resampled_higher_totals = np.zeros(
            len(
                THRESHOLD_V2_AGGREGATE_COLUMNS
            ),
            dtype=float,
        )


        resampled_lower_totals = np.zeros(
            len(
                THRESHOLD_V2_AGGREGATE_COLUMNS
            ),
            dtype=float,
        )


        for size_stratum in paired_size_strata:

            higher_array = (
                size_stratum[
                    "higher_array"
                ]
            )


            lower_array = (
                size_stratum[
                    "lower_array"
                ]
            )


            number_of_common_groups = (
                len(
                    higher_array
                )
            )


            # Shared indices preserve the pairing
            # between higher and lower thresholds.
            sampled_indices = (
                threshold_contrast_v2_rng.integers(
                    low=0,
                    high=number_of_common_groups,
                    size=number_of_common_groups,
                )
            )


            resampled_higher_totals += (
                higher_array[
                    sampled_indices
                ]
                .sum(
                    axis=0
                )
            )


            resampled_lower_totals += (
                lower_array[
                    sampled_indices
                ]
                .sum(
                    axis=0
                )
            )


        bootstrap_differences[
            bootstrap_index
        ] = (
            calculate_threshold_v2_contrast_metrics(
                resampled_higher_totals
            )
            -
            calculate_threshold_v2_contrast_metrics(
                resampled_lower_totals
            )
        )


    # ----------------------------------------------
    # Confidence intervals
    # ----------------------------------------------

    bootstrap_standard_errors = (
        bootstrap_differences.std(
            axis=0,
            ddof=1,
        )
    )


    confidence_lower = np.quantile(
        bootstrap_differences,
        0.025,
        axis=0,
    )


    confidence_upper = np.quantile(
        bootstrap_differences,
        0.975,
        axis=0,
    )


    contrast_name = (
        f"tau={higher_tau:.2f}"
        f" minus "
        f"tau={lower_tau:.2f}"
    )


    for metric_index, metric_name in enumerate(
        THRESHOLD_V2_CONTRAST_METRICS
    ):

        threshold_contrast_v2_rows.append(
            {
                "higher_tau":
                    higher_tau,

                "lower_tau":
                    lower_tau,

                "contrast":
                    contrast_name,

                "metric":
                    metric_name,

                "estimated_difference":
                    observed_difference[
                        metric_index
                    ],

                "bootstrap_standard_error":
                    bootstrap_standard_errors[
                        metric_index
                    ],

                "ci_95_lower":
                    confidence_lower[
                        metric_index
                    ],

                "ci_95_upper":
                    confidence_upper[
                        metric_index
                    ],

                "ci_excludes_zero":
                    (
                        confidence_lower[
                            metric_index
                        ]
                        > 0.0
                    )
                    or
                    (
                        confidence_upper[
                            metric_index
                        ]
                        < 0.0
                    ),

                "common_feasible_matched_groups":
                    total_common_groups,

                "candidate_size_strata":
                    len(
                        paired_size_strata
                    ),

                "bootstrap_replicates":
                    THRESHOLD_CONTRAST_V2_REPLICATES,

                "resampling_unit":
                    "common_feasible_matched_group",

                "resampling_design":
                    (
                        "paired_across_thresholds_"
                        "stratified_by_candidate_set_size"
                    ),

                "estimand":
                    (
                        "threshold_difference_"
                        "conditional_on_common_feasibility"
                    ),

                "multiple_comparison_adjustment":
                    "none",
            }
        )


    replicate_df = pd.DataFrame(
        bootstrap_differences,
        columns=THRESHOLD_V2_CONTRAST_METRICS,
    )


    replicate_df.insert(
        0,
        "bootstrap_replicate",
        np.arange(
            THRESHOLD_CONTRAST_V2_REPLICATES
        ),
    )


    replicate_df.insert(
        0,
        "contrast",
        contrast_name,
    )


    threshold_contrast_v2_replicate_frames.append(
        replicate_df
    )


    print(
        "Completed corrected paired threshold contrast:",
        contrast_name,
        "| common matched groups:",
        total_common_groups,
    )


# --------------------------------------------------
# Save outputs
# --------------------------------------------------

threshold_contrast_v2_results_df = pd.DataFrame(
    threshold_contrast_v2_rows
)


threshold_contrast_v2_support_df = pd.DataFrame(
    threshold_contrast_v2_support_rows
)


threshold_contrast_v2_replicates_df = pd.concat(
    threshold_contrast_v2_replicate_frames,
    ignore_index=True,
)


threshold_contrast_v2_results_df.to_csv(
    THRESHOLD_CONTRAST_V2_PATH,
    index=False,
)


threshold_contrast_v2_support_df.to_csv(
    THRESHOLD_CONTRAST_V2_SUPPORT_PATH,
    index=False,
)


threshold_contrast_v2_replicates_df.to_csv(
    THRESHOLD_CONTRAST_V2_REPLICATES_PATH,
    index=False,
)


# --------------------------------------------------
# Readable display
# --------------------------------------------------

key_threshold_contrast_metrics = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "greedy_more_items_percent",
    "mean_selected_count_difference",
]


threshold_contrast_v2_display_df = (
    threshold_contrast_v2_results_df[
        threshold_contrast_v2_results_df[
            "metric"
        ].isin(
            key_threshold_contrast_metrics
        )
    ]
    .copy()
)


threshold_contrast_v2_display_df[
    "difference_and_ci"
] = (
    threshold_contrast_v2_display_df[
        "estimated_difference"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    threshold_contrast_v2_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    threshold_contrast_v2_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


print(
    "\n======================================"
)

print(
    "CORRECTED V2 PAIRED THRESHOLD CONTRASTS"
)

print(
    "======================================"
)


print(
    threshold_contrast_v2_display_df[
        [
            "contrast",
            "metric",
            "difference_and_ci",
            "ci_excludes_zero",
            "common_feasible_matched_groups",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\nCommon-support counts by size:"
)


print(
    threshold_contrast_v2_support_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.4f}",
    )
)


print(
    "\nPaired threshold contrasts saved to:"
)

print(
    THRESHOLD_CONTRAST_V2_PATH
)


print(
    "\nCommon-support counts saved to:"
)

print(
    THRESHOLD_CONTRAST_V2_SUPPORT_PATH
)


print(
    "\nBootstrap replicates saved to:"
)

print(
    THRESHOLD_CONTRAST_V2_REPLICATES_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Corrected v2 weight-profile descriptive results
# and matched-group clustered confidence intervals
# --------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd


WEIGHT_V2_BOOTSTRAP_REPLICATES = 5000
WEIGHT_V2_BOOTSTRAP_SEED = GLOBAL_SEED + 107


if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


CORRECTED_ANALYSIS_READY_PATH = (
    RESULTS_DIR
    / (
        "exact_vs_greedy_main_benchmark_"
        "v2_marginal_preserving_analysis_ready.csv"
    )
)


WEIGHT_V2_DESCRIPTIVE_PATH = (
    TABLES_DIR
    / "corrected_v2_weight_profile_descriptive_summary.csv"
)


WEIGHT_V2_BOOTSTRAP_SUMMARY_PATH = (
    TABLES_DIR
    / "corrected_v2_weight_profile_matched_group_bootstrap_ci.csv"
)


WEIGHT_V2_BOOTSTRAP_REPLICATES_PATH = (
    RESULTS_DIR
    / "corrected_v2_weight_profile_matched_group_bootstrap_replicates.csv"
)


WEIGHT_PROFILE_ORDER = [
    "balanced",
    "privacy_priority",
    "interpretation_priority",
    "parsimony_priority",
]


# --------------------------------------------------
# Load corrected analysis-ready benchmark
# --------------------------------------------------

weight_v2_df = pd.read_csv(
    CORRECTED_ANALYSIS_READY_PATH,
    low_memory=False,
)


weight_v2_df[
    "is_comparable"
] = (
    (
        weight_v2_df[
            "exact_status"
        ]
        == "OPTIMAL"
    )
    &
    (
        weight_v2_df[
            "greedy_status"
        ]
        == "FEASIBLE"
    )
)


for column_name in [
    "n_items",
    "tau",
    "optimality_gap_percent",
    "jaccard_similarity",
    "selected_count_difference",
]:

    weight_v2_df[
        column_name
    ] = pd.to_numeric(
        weight_v2_df[
            column_name
        ],
        errors="coerce",
    )


weight_v2_comparable_df = (
    weight_v2_df[
        weight_v2_df[
            "is_comparable"
        ]
    ]
    .copy()
)


# --------------------------------------------------
# Reporting indicators
# --------------------------------------------------

OPTIMAL_COST_TOLERANCE_PERCENT = 1e-10


weight_v2_comparable_df[
    "greedy_optimal_cost_numeric"
] = (
    weight_v2_comparable_df[
        "optimality_gap_percent"
    ]
    .abs()
    <=
    OPTIMAL_COST_TOLERANCE_PERCENT
).astype(int)


weight_v2_comparable_df[
    "large_gap_numeric"
] = (
    weight_v2_comparable_df[
        "optimality_gap_percent"
    ]
    > 10.0
).astype(int)


weight_v2_comparable_df[
    "greedy_more_items_numeric"
] = (
    weight_v2_comparable_df[
        "selected_count_difference"
    ]
    > 0
).astype(int)


weight_v2_comparable_df[
    "within_1_percent_numeric"
] = (
    weight_v2_comparable_df[
        "optimality_gap_percent"
    ]
    <= 1.0
).astype(int)


weight_v2_comparable_df[
    "within_5_percent_numeric"
] = (
    weight_v2_comparable_df[
        "optimality_gap_percent"
    ]
    <= 5.0
).astype(int)


weight_v2_comparable_df[
    "within_10_percent_numeric"
] = (
    weight_v2_comparable_df[
        "optimality_gap_percent"
    ]
    <= 10.0
).astype(int)


# --------------------------------------------------
# Weight-profile descriptive summaries
# --------------------------------------------------

weight_v2_descriptive_rows = []


for weight_profile in WEIGHT_PROFILE_ORDER:

    all_profile_df = (
        weight_v2_df[
            weight_v2_df[
                "weight_profile"
            ]
            == weight_profile
        ]
    )


    comparable_profile_df = (
        weight_v2_comparable_df[
            weight_v2_comparable_df[
                "weight_profile"
            ]
            == weight_profile
        ]
    )


    gap_values = (
        comparable_profile_df[
            "optimality_gap_percent"
        ]
        .to_numpy(
            dtype=float
        )
    )


    weight_v2_descriptive_rows.append(
        {
            "weight_profile":
                weight_profile,

            "matched_groups":
                all_profile_df[
                    "matched_group_id"
                ]
                .nunique(),

            "total_rows":
                len(
                    all_profile_df
                ),

            "comparable_rows":
                len(
                    comparable_profile_df
                ),

            "feasibility_percent":
                100.0
                *
                len(
                    comparable_profile_df
                )
                /
                len(
                    all_profile_df
                ),

            "mean_optimality_gap_percent":
                gap_values.mean(),

            "median_optimality_gap_percent":
                np.median(
                    gap_values
                ),

            "p90_optimality_gap_percent":
                np.quantile(
                    gap_values,
                    0.90,
                ),

            "p95_optimality_gap_percent":
                np.quantile(
                    gap_values,
                    0.95,
                ),

            "maximum_optimality_gap_percent":
                gap_values.max(),

            "greedy_optimal_cost_percent":
                100.0
                *
                comparable_profile_df[
                    "greedy_optimal_cost_numeric"
                ].mean(),

            "mean_jaccard_similarity":
                comparable_profile_df[
                    "jaccard_similarity"
                ].mean(),

            "large_gap_percent":
                100.0
                *
                comparable_profile_df[
                    "large_gap_numeric"
                ].mean(),

            "greedy_more_items_percent":
                100.0
                *
                comparable_profile_df[
                    "greedy_more_items_numeric"
                ].mean(),

            "mean_selected_count_difference":
                comparable_profile_df[
                    "selected_count_difference"
                ].mean(),

            "within_1_percent":
                100.0
                *
                comparable_profile_df[
                    "within_1_percent_numeric"
                ].mean(),

            "within_5_percent":
                100.0
                *
                comparable_profile_df[
                    "within_5_percent_numeric"
                ].mean(),

            "within_10_percent":
                100.0
                *
                comparable_profile_df[
                    "within_10_percent_numeric"
                ].mean(),
        }
    )


weight_v2_descriptive_df = pd.DataFrame(
    weight_v2_descriptive_rows
)


weight_v2_descriptive_df.to_csv(
    WEIGHT_V2_DESCRIPTIVE_PATH,
    index=False,
)


# --------------------------------------------------
# One sufficient-statistics row per matched group
# and weight profile
# --------------------------------------------------

weight_v2_cluster_df = (
    weight_v2_comparable_df
    .groupby(
        [
            "matched_group_id",
            "n_items",
            "weight_profile",
        ],
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "size",
        ),

        gap_sum=(
            "optimality_gap_percent",
            "sum",
        ),

        optimal_cost_sum=(
            "greedy_optimal_cost_numeric",
            "sum",
        ),

        jaccard_sum=(
            "jaccard_similarity",
            "sum",
        ),

        large_gap_sum=(
            "large_gap_numeric",
            "sum",
        ),

        greedy_more_items_sum=(
            "greedy_more_items_numeric",
            "sum",
        ),

        selected_count_difference_sum=(
            "selected_count_difference",
            "sum",
        ),

        within_1_percent_sum=(
            "within_1_percent_numeric",
            "sum",
        ),

        within_5_percent_sum=(
            "within_5_percent_numeric",
            "sum",
        ),

        within_10_percent_sum=(
            "within_10_percent_numeric",
            "sum",
        ),
    )
)


WEIGHT_V2_AGGREGATE_COLUMNS = [
    "comparable_rows",
    "gap_sum",
    "optimal_cost_sum",
    "jaccard_sum",
    "large_gap_sum",
    "greedy_more_items_sum",
    "selected_count_difference_sum",
    "within_1_percent_sum",
    "within_5_percent_sum",
    "within_10_percent_sum",
]


WEIGHT_V2_METRIC_NAMES = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "greedy_more_items_percent",
    "mean_selected_count_difference",
    "within_1_percent",
    "within_5_percent",
    "within_10_percent",
]


def calculate_weight_v2_metrics(
    totals: np.ndarray,
) -> np.ndarray:
    """
    Convert matched-group sufficient-statistic totals
    into weight-profile quality metrics.
    """

    comparable_rows = totals[0]

    return np.array(
        [
            totals[1]
            / comparable_rows,

            100.0
            * totals[2]
            / comparable_rows,

            totals[3]
            / comparable_rows,

            100.0
            * totals[4]
            / comparable_rows,

            100.0
            * totals[5]
            / comparable_rows,

            totals[6]
            / comparable_rows,

            100.0
            * totals[7]
            / comparable_rows,

            100.0
            * totals[8]
            / comparable_rows,

            100.0
            * totals[9]
            / comparable_rows,
        ],
        dtype=float,
    )


# --------------------------------------------------
# Validate matched-group structure
# --------------------------------------------------

groups_by_profile_size = (
    weight_v2_cluster_df
    .groupby(
        [
            "weight_profile",
            "n_items",
        ]
    )[
        "matched_group_id"
    ]
    .nunique()
)


assert (
    groups_by_profile_size
    == 100
).all()


# Comparable-row counts must match across profiles
# for each matched group because feasibility depends
# only on the utility threshold.
profile_comparable_count_pivot = (
    weight_v2_cluster_df
    .pivot(
        index="matched_group_id",
        columns="weight_profile",
        values="comparable_rows",
    )
)


assert (
    profile_comparable_count_pivot.nunique(
        axis=1
    )
    == 1
).all()


# --------------------------------------------------
# Bootstrap separately for each weight profile
# --------------------------------------------------

weight_v2_rng = np.random.default_rng(
    WEIGHT_V2_BOOTSTRAP_SEED
)


weight_v2_bootstrap_summary_rows = []
weight_v2_bootstrap_replicate_frames = []


for weight_profile in WEIGHT_PROFILE_ORDER:

    current_profile_df = (
        weight_v2_cluster_df[
            weight_v2_cluster_df[
                "weight_profile"
            ]
            == weight_profile
        ]
        .copy()
    )


    observed_totals = (
        current_profile_df[
            WEIGHT_V2_AGGREGATE_COLUMNS
        ]
        .sum()
        .to_numpy(
            dtype=float
        )
    )


    observed_metrics = (
        calculate_weight_v2_metrics(
            observed_totals
        )
    )


    size_stratum_arrays = []


    for n_items, size_df in (
        current_profile_df
        .groupby(
            "n_items",
            sort=True,
        )
    ):

        size_array = (
            size_df
            .sort_values(
                "matched_group_id"
            )[
                WEIGHT_V2_AGGREGATE_COLUMNS
            ]
            .to_numpy(
                dtype=float
            )
        )


        assert (
            size_array.shape[0]
            == 100
        )


        size_stratum_arrays.append(
            size_array
        )


    bootstrap_estimates = np.empty(
        (
            WEIGHT_V2_BOOTSTRAP_REPLICATES,
            len(
                WEIGHT_V2_METRIC_NAMES
            ),
        ),
        dtype=float,
    )


    for bootstrap_index in range(
        WEIGHT_V2_BOOTSTRAP_REPLICATES
    ):

        resampled_totals = np.zeros(
            len(
                WEIGHT_V2_AGGREGATE_COLUMNS
            ),
            dtype=float,
        )


        for size_array in size_stratum_arrays:

            number_of_groups = (
                size_array.shape[0]
            )


            sampled_indices = (
                weight_v2_rng.integers(
                    low=0,
                    high=number_of_groups,
                    size=number_of_groups,
                )
            )


            resampled_totals += (
                size_array[
                    sampled_indices
                ]
                .sum(
                    axis=0
                )
            )


        bootstrap_estimates[
            bootstrap_index
        ] = calculate_weight_v2_metrics(
            resampled_totals
        )


    standard_errors = (
        bootstrap_estimates.std(
            axis=0,
            ddof=1,
        )
    )


    confidence_lower = np.quantile(
        bootstrap_estimates,
        0.025,
        axis=0,
    )


    confidence_upper = np.quantile(
        bootstrap_estimates,
        0.975,
        axis=0,
    )


    for metric_index, metric_name in enumerate(
        WEIGHT_V2_METRIC_NAMES
    ):

        weight_v2_bootstrap_summary_rows.append(
            {
                "weight_profile":
                    weight_profile,

                "metric":
                    metric_name,

                "estimate":
                    observed_metrics[
                        metric_index
                    ],

                "bootstrap_standard_error":
                    standard_errors[
                        metric_index
                    ],

                "ci_95_lower":
                    confidence_lower[
                        metric_index
                    ],

                "ci_95_upper":
                    confidence_upper[
                        metric_index
                    ],

                "matched_groups":
                    500,

                "candidate_size_strata":
                    5,

                "bootstrap_replicates":
                    WEIGHT_V2_BOOTSTRAP_REPLICATES,

                "resampling_unit":
                    "matched_size_replicate_group",

                "resampling_design":
                    "stratified_by_candidate_set_size",

                "estimand":
                    "quality_conditional_on_feasibility",
            }
        )


    replicate_df = pd.DataFrame(
        bootstrap_estimates,
        columns=WEIGHT_V2_METRIC_NAMES,
    )


    replicate_df.insert(
        0,
        "bootstrap_replicate",
        np.arange(
            WEIGHT_V2_BOOTSTRAP_REPLICATES
        ),
    )


    replicate_df.insert(
        0,
        "weight_profile",
        weight_profile,
    )


    weight_v2_bootstrap_replicate_frames.append(
        replicate_df
    )


    print(
        "Completed corrected weight profile:",
        weight_profile
    )


# --------------------------------------------------
# Save outputs
# --------------------------------------------------

weight_v2_bootstrap_summary_df = pd.DataFrame(
    weight_v2_bootstrap_summary_rows
)


weight_v2_bootstrap_replicates_df = pd.concat(
    weight_v2_bootstrap_replicate_frames,
    ignore_index=True,
)


weight_v2_bootstrap_summary_df.to_csv(
    WEIGHT_V2_BOOTSTRAP_SUMMARY_PATH,
    index=False,
)


weight_v2_bootstrap_replicates_df.to_csv(
    WEIGHT_V2_BOOTSTRAP_REPLICATES_PATH,
    index=False,
)


# --------------------------------------------------
# Compact display
# --------------------------------------------------

key_weight_metrics = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "greedy_more_items_percent",
]


weight_v2_display_df = (
    weight_v2_bootstrap_summary_df[
        weight_v2_bootstrap_summary_df[
            "metric"
        ].isin(
            key_weight_metrics
        )
    ]
    .copy()
)


weight_v2_display_df[
    "estimate_and_ci"
] = (
    weight_v2_display_df[
        "estimate"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    weight_v2_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    weight_v2_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


weight_v2_pivot_df = (
    weight_v2_display_df
    .pivot(
        index="weight_profile",
        columns="metric",
        values="estimate_and_ci",
    )
    .reset_index()
)


weight_v2_pivot_df[
    "weight_profile"
] = pd.Categorical(
    weight_v2_pivot_df[
        "weight_profile"
    ],
    categories=WEIGHT_PROFILE_ORDER,
    ordered=True,
)


weight_v2_pivot_df = (
    weight_v2_pivot_df
    .sort_values(
        "weight_profile"
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n======================================"
)

print(
    "CORRECTED V2 WEIGHT-PROFILE DESCRIPTIVE RESULTS"
)

print(
    "======================================"
)


print(
    weight_v2_descriptive_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.4f}",
    )
)


print(
    "\n======================================"
)

print(
    "CORRECTED V2 WEIGHT-PROFILE MATCHED-GROUP 95% CIs"
)

print(
    "======================================"
)


print(
    weight_v2_pivot_df.to_string(
        index=False
    )
)


print(
    "\nWeight-profile descriptive summary saved to:"
)

print(
    WEIGHT_V2_DESCRIPTIVE_PATH
)


print(
    "\nWeight-profile confidence intervals saved to:"
)

print(
    WEIGHT_V2_BOOTSTRAP_SUMMARY_PATH
)


print(
    "\nWeight-profile bootstrap replicates saved to:"
)

print(
    WEIGHT_V2_BOOTSTRAP_REPLICATES_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Corrected v2 paired weight-profile contrasts
# using matched size–replicate groups
# --------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd


WEIGHT_CONTRAST_V2_REPLICATES = 5000
WEIGHT_CONTRAST_V2_SEED = GLOBAL_SEED + 108


if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


CORRECTED_ANALYSIS_READY_PATH = (
    RESULTS_DIR
    / (
        "exact_vs_greedy_main_benchmark_"
        "v2_marginal_preserving_analysis_ready.csv"
    )
)


WEIGHT_CONTRAST_V2_PATH = (
    TABLES_DIR
    / "corrected_v2_paired_weight_profile_bootstrap_contrasts.csv"
)


WEIGHT_CONTRAST_V2_REPLICATES_PATH = (
    RESULTS_DIR
    / (
        "corrected_v2_paired_weight_profile_"
        "bootstrap_contrast_replicates.csv"
    )
)


WEIGHT_PROFILE_ORDER = [
    "balanced",
    "privacy_priority",
    "interpretation_priority",
    "parsimony_priority",
]


# Each contrast is:
# first profile minus second profile.
WEIGHT_PROFILE_V2_COMPARISONS = [
    (
        "privacy_priority",
        "balanced",
    ),
    (
        "interpretation_priority",
        "balanced",
    ),
    (
        "parsimony_priority",
        "balanced",
    ),
    (
        "privacy_priority",
        "interpretation_priority",
    ),
    (
        "privacy_priority",
        "parsimony_priority",
    ),
    (
        "interpretation_priority",
        "parsimony_priority",
    ),
]


WEIGHT_CONTRAST_V2_METRICS = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "greedy_more_items_percent",
    "mean_selected_count_difference",
    "within_1_percent",
    "within_5_percent",
    "within_10_percent",
]


# --------------------------------------------------
# Load and prepare corrected benchmark
# --------------------------------------------------

weight_contrast_v2_df = pd.read_csv(
    CORRECTED_ANALYSIS_READY_PATH,
    low_memory=False,
)


weight_contrast_v2_df[
    "is_comparable"
] = (
    (
        weight_contrast_v2_df[
            "exact_status"
        ]
        == "OPTIMAL"
    )
    &
    (
        weight_contrast_v2_df[
            "greedy_status"
        ]
        == "FEASIBLE"
    )
)


weight_contrast_v2_df = (
    weight_contrast_v2_df[
        weight_contrast_v2_df[
            "is_comparable"
        ]
    ]
    .copy()
)


for column_name in [
    "n_items",
    "tau",
    "optimality_gap_percent",
    "jaccard_similarity",
    "selected_count_difference",
]:

    weight_contrast_v2_df[
        column_name
    ] = pd.to_numeric(
        weight_contrast_v2_df[
            column_name
        ],
        errors="coerce",
    )


# --------------------------------------------------
# Reporting indicators
# --------------------------------------------------

OPTIMAL_COST_TOLERANCE_PERCENT = 1e-10


weight_contrast_v2_df[
    "greedy_optimal_cost_numeric"
] = (
    weight_contrast_v2_df[
        "optimality_gap_percent"
    ]
    .abs()
    <= OPTIMAL_COST_TOLERANCE_PERCENT
).astype(int)


weight_contrast_v2_df[
    "large_gap_numeric"
] = (
    weight_contrast_v2_df[
        "optimality_gap_percent"
    ]
    > 10.0
).astype(int)


weight_contrast_v2_df[
    "greedy_more_items_numeric"
] = (
    weight_contrast_v2_df[
        "selected_count_difference"
    ]
    > 0
).astype(int)


weight_contrast_v2_df[
    "within_1_percent_numeric"
] = (
    weight_contrast_v2_df[
        "optimality_gap_percent"
    ]
    <= 1.0
).astype(int)


weight_contrast_v2_df[
    "within_5_percent_numeric"
] = (
    weight_contrast_v2_df[
        "optimality_gap_percent"
    ]
    <= 5.0
).astype(int)


weight_contrast_v2_df[
    "within_10_percent_numeric"
] = (
    weight_contrast_v2_df[
        "optimality_gap_percent"
    ]
    <= 10.0
).astype(int)


# --------------------------------------------------
# One sufficient-statistics row per matched group
# and weight profile
# --------------------------------------------------

weight_contrast_v2_cluster_df = (
    weight_contrast_v2_df
    .groupby(
        [
            "matched_group_id",
            "n_items",
            "weight_profile",
        ],
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "size",
        ),

        gap_sum=(
            "optimality_gap_percent",
            "sum",
        ),

        optimal_cost_sum=(
            "greedy_optimal_cost_numeric",
            "sum",
        ),

        jaccard_sum=(
            "jaccard_similarity",
            "sum",
        ),

        large_gap_sum=(
            "large_gap_numeric",
            "sum",
        ),

        greedy_more_items_sum=(
            "greedy_more_items_numeric",
            "sum",
        ),

        selected_count_difference_sum=(
            "selected_count_difference",
            "sum",
        ),

        within_1_percent_sum=(
            "within_1_percent_numeric",
            "sum",
        ),

        within_5_percent_sum=(
            "within_5_percent_numeric",
            "sum",
        ),

        within_10_percent_sum=(
            "within_10_percent_numeric",
            "sum",
        ),
    )
)


WEIGHT_CONTRAST_V2_AGGREGATE_COLUMNS = [
    "comparable_rows",
    "gap_sum",
    "optimal_cost_sum",
    "jaccard_sum",
    "large_gap_sum",
    "greedy_more_items_sum",
    "selected_count_difference_sum",
    "within_1_percent_sum",
    "within_5_percent_sum",
    "within_10_percent_sum",
]


def calculate_weight_contrast_v2_metrics(
    totals: np.ndarray,
) -> np.ndarray:
    """
    Convert matched-group sufficient-statistic totals
    into weight-profile quality metrics.
    """

    comparable_rows = totals[0]

    return np.array(
        [
            totals[1]
            / comparable_rows,

            100.0
            * totals[2]
            / comparable_rows,

            totals[3]
            / comparable_rows,

            100.0
            * totals[4]
            / comparable_rows,

            100.0
            * totals[5]
            / comparable_rows,

            totals[6]
            / comparable_rows,

            100.0
            * totals[7]
            / comparable_rows,

            100.0
            * totals[8]
            / comparable_rows,

            100.0
            * totals[9]
            / comparable_rows,
        ],
        dtype=float,
    )


# --------------------------------------------------
# Validate complete pairing across profiles
# --------------------------------------------------

profiles_per_matched_group = (
    weight_contrast_v2_cluster_df
    .groupby(
        "matched_group_id"
    )[
        "weight_profile"
    ]
    .apply(set)
)


expected_profiles = set(
    WEIGHT_PROFILE_ORDER
)


assert (
    profiles_per_matched_group
    .apply(
        lambda profile_set:
            profile_set
            == expected_profiles
    )
    .all()
)


assert (
    weight_contrast_v2_cluster_df[
        "matched_group_id"
    ]
    .nunique()
    == 500
)


# Feasibility depends on utility and tau, not weights.
# Therefore, each matched group must have the same
# comparable-row count across all weight profiles.
comparable_count_pivot = (
    weight_contrast_v2_cluster_df
    .pivot(
        index="matched_group_id",
        columns="weight_profile",
        values="comparable_rows",
    )
)


assert (
    comparable_count_pivot.nunique(
        axis=1
    )
    == 1
).all()


# --------------------------------------------------
# Create aligned profile arrays within each
# candidate-set-size stratum
# --------------------------------------------------

paired_weight_size_strata = []


for n_items in sorted(
    weight_contrast_v2_cluster_df[
        "n_items"
    ]
    .unique()
):

    profile_arrays = {}
    reference_group_ids = None


    for weight_profile in WEIGHT_PROFILE_ORDER:

        current_profile_size_df = (
            weight_contrast_v2_cluster_df[
                (
                    weight_contrast_v2_cluster_df[
                        "n_items"
                    ]
                    == n_items
                )
                &
                (
                    weight_contrast_v2_cluster_df[
                        "weight_profile"
                    ]
                    == weight_profile
                )
            ]
            .sort_values(
                "matched_group_id"
            )
            .reset_index(
                drop=True
            )
        )


        assert len(
            current_profile_size_df
        ) == 100


        current_group_ids = (
            current_profile_size_df[
                "matched_group_id"
            ]
            .tolist()
        )


        if reference_group_ids is None:

            reference_group_ids = (
                current_group_ids
            )

        else:

            assert (
                current_group_ids
                ==
                reference_group_ids
            )


        profile_arrays[
            weight_profile
        ] = (
            current_profile_size_df[
                WEIGHT_CONTRAST_V2_AGGREGATE_COLUMNS
            ]
            .to_numpy(
                dtype=float
            )
        )


    paired_weight_size_strata.append(
        {
            "n_items":
                n_items,

            "matched_group_ids":
                reference_group_ids,

            "profile_arrays":
                profile_arrays,
        }
    )


# --------------------------------------------------
# Observed weight-profile metrics
# --------------------------------------------------

observed_weight_v2_metrics = {}


for weight_profile in WEIGHT_PROFILE_ORDER:

    profile_totals = (
        weight_contrast_v2_cluster_df[
            weight_contrast_v2_cluster_df[
                "weight_profile"
            ]
            == weight_profile
        ][
            WEIGHT_CONTRAST_V2_AGGREGATE_COLUMNS
        ]
        .sum()
        .to_numpy(
            dtype=float
        )
    )


    observed_weight_v2_metrics[
        weight_profile
    ] = calculate_weight_contrast_v2_metrics(
        profile_totals
    )


# --------------------------------------------------
# Shared paired matched-group bootstrap
# --------------------------------------------------

weight_contrast_v2_rng = np.random.default_rng(
    WEIGHT_CONTRAST_V2_SEED
)


bootstrap_weight_v2_metrics = {
    weight_profile: np.empty(
        (
            WEIGHT_CONTRAST_V2_REPLICATES,
            len(
                WEIGHT_CONTRAST_V2_METRICS
            ),
        ),
        dtype=float,
    )
    for weight_profile in WEIGHT_PROFILE_ORDER
}


for bootstrap_index in range(
    WEIGHT_CONTRAST_V2_REPLICATES
):

    resampled_totals_by_profile = {
        weight_profile: np.zeros(
            len(
                WEIGHT_CONTRAST_V2_AGGREGATE_COLUMNS
            ),
            dtype=float,
        )
        for weight_profile in WEIGHT_PROFILE_ORDER
    }


    for size_stratum in paired_weight_size_strata:

        number_of_groups = len(
            size_stratum[
                "matched_group_ids"
            ]
        )


        # Shared indices preserve pairing across all
        # four objective-weight profiles.
        sampled_indices = (
            weight_contrast_v2_rng.integers(
                low=0,
                high=number_of_groups,
                size=number_of_groups,
            )
        )


        for weight_profile in WEIGHT_PROFILE_ORDER:

            profile_array = (
                size_stratum[
                    "profile_arrays"
                ][
                    weight_profile
                ]
            )


            resampled_totals_by_profile[
                weight_profile
            ] += (
                profile_array[
                    sampled_indices
                ]
                .sum(
                    axis=0
                )
            )


    for weight_profile in WEIGHT_PROFILE_ORDER:

        bootstrap_weight_v2_metrics[
            weight_profile
        ][
            bootstrap_index
        ] = calculate_weight_contrast_v2_metrics(
            resampled_totals_by_profile[
                weight_profile
            ]
        )


# --------------------------------------------------
# Calculate paired profile contrasts
# --------------------------------------------------

weight_contrast_v2_rows = []
weight_contrast_v2_replicate_frames = []


for first_profile, second_profile in (
    WEIGHT_PROFILE_V2_COMPARISONS
):

    contrast_name = (
        f"{first_profile}"
        f" minus "
        f"{second_profile}"
    )


    observed_difference = (
        observed_weight_v2_metrics[
            first_profile
        ]
        -
        observed_weight_v2_metrics[
            second_profile
        ]
    )


    bootstrap_differences = (
        bootstrap_weight_v2_metrics[
            first_profile
        ]
        -
        bootstrap_weight_v2_metrics[
            second_profile
        ]
    )


    bootstrap_standard_errors = (
        bootstrap_differences.std(
            axis=0,
            ddof=1,
        )
    )


    confidence_lower = np.quantile(
        bootstrap_differences,
        0.025,
        axis=0,
    )


    confidence_upper = np.quantile(
        bootstrap_differences,
        0.975,
        axis=0,
    )


    for metric_index, metric_name in enumerate(
        WEIGHT_CONTRAST_V2_METRICS
    ):

        weight_contrast_v2_rows.append(
            {
                "first_profile":
                    first_profile,

                "second_profile":
                    second_profile,

                "contrast":
                    contrast_name,

                "metric":
                    metric_name,

                "estimated_difference":
                    observed_difference[
                        metric_index
                    ],

                "bootstrap_standard_error":
                    bootstrap_standard_errors[
                        metric_index
                    ],

                "ci_95_lower":
                    confidence_lower[
                        metric_index
                    ],

                "ci_95_upper":
                    confidence_upper[
                        metric_index
                    ],

                "ci_excludes_zero":
                    (
                        confidence_lower[
                            metric_index
                        ]
                        > 0.0
                    )
                    or
                    (
                        confidence_upper[
                            metric_index
                        ]
                        < 0.0
                    ),

                "matched_groups":
                    500,

                "candidate_size_strata":
                    5,

                "bootstrap_replicates":
                    WEIGHT_CONTRAST_V2_REPLICATES,

                "resampling_unit":
                    "matched_size_replicate_group",

                "resampling_design":
                    (
                        "paired_across_weight_profiles_"
                        "stratified_by_candidate_set_size"
                    ),

                "estimand":
                    "quality_conditional_on_feasibility",

                "multiple_comparison_adjustment":
                    "none",
            }
        )


    contrast_replicate_df = pd.DataFrame(
        bootstrap_differences,
        columns=WEIGHT_CONTRAST_V2_METRICS,
    )


    contrast_replicate_df.insert(
        0,
        "bootstrap_replicate",
        np.arange(
            WEIGHT_CONTRAST_V2_REPLICATES
        ),
    )


    contrast_replicate_df.insert(
        0,
        "contrast",
        contrast_name,
    )


    weight_contrast_v2_replicate_frames.append(
        contrast_replicate_df
    )


    print(
        "Completed corrected paired weight contrast:",
        contrast_name
    )


# --------------------------------------------------
# Save outputs
# --------------------------------------------------

weight_contrast_v2_results_df = pd.DataFrame(
    weight_contrast_v2_rows
)


weight_contrast_v2_replicates_df = pd.concat(
    weight_contrast_v2_replicate_frames,
    ignore_index=True,
)


weight_contrast_v2_results_df.to_csv(
    WEIGHT_CONTRAST_V2_PATH,
    index=False,
)


weight_contrast_v2_replicates_df.to_csv(
    WEIGHT_CONTRAST_V2_REPLICATES_PATH,
    index=False,
)


# --------------------------------------------------
# Readable display
# --------------------------------------------------

key_weight_contrast_metrics = [
    "mean_optimality_gap_percent",
    "greedy_optimal_cost_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "greedy_more_items_percent",
    "mean_selected_count_difference",
]


weight_contrast_v2_display_df = (
    weight_contrast_v2_results_df[
        weight_contrast_v2_results_df[
            "metric"
        ].isin(
            key_weight_contrast_metrics
        )
    ]
    .copy()
)


weight_contrast_v2_display_df[
    "difference_and_ci"
] = (
    weight_contrast_v2_display_df[
        "estimated_difference"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    weight_contrast_v2_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    weight_contrast_v2_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


print(
    "\n======================================"
)

print(
    "CORRECTED V2 PAIRED WEIGHT-PROFILE CONTRASTS"
)

print(
    "======================================"
)


print(
    weight_contrast_v2_display_df[
        [
            "contrast",
            "metric",
            "difference_and_ci",
            "ci_excludes_zero",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\nPaired weight-profile contrasts saved to:"
)

print(
    WEIGHT_CONTRAST_V2_PATH
)


print(
    "\nPaired contrast replicates saved to:"
)

print(
    WEIGHT_CONTRAST_V2_REPLICATES_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Corrected v2 Greedy failure-mechanism decomposition
# --------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd


if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


CORRECTED_ANALYSIS_READY_PATH = (
    RESULTS_DIR
    / (
        "exact_vs_greedy_main_benchmark_"
        "v2_marginal_preserving_analysis_ready.csv"
    )
)


MECHANISM_V2_SUMMARY_PATH = (
    TABLES_DIR
    / "corrected_v2_greedy_failure_mechanism_summary.csv"
)


MECHANISM_V2_COMPONENT_PATH = (
    TABLES_DIR
    / "corrected_v2_greedy_failure_component_differences.csv"
)


MECHANISM_V2_ROW_LEVEL_PATH = (
    RESULTS_DIR
    / "corrected_v2_greedy_failure_mechanism_rows.csv"
)


# --------------------------------------------------
# Load corrected analysis-ready benchmark
# --------------------------------------------------

mechanism_v2_df = pd.read_csv(
    CORRECTED_ANALYSIS_READY_PATH,
    low_memory=False,
)


mechanism_v2_df[
    "is_comparable"
] = (
    (
        mechanism_v2_df[
            "exact_status"
        ]
        == "OPTIMAL"
    )
    &
    (
        mechanism_v2_df[
            "greedy_status"
        ]
        == "FEASIBLE"
    )
)


mechanism_v2_df = (
    mechanism_v2_df[
        mechanism_v2_df[
            "is_comparable"
        ]
    ]
    .copy()
)


# --------------------------------------------------
# Standardize required numerical columns
# --------------------------------------------------

required_numeric_columns = [
    "optimality_gap_percent",
    "jaccard_similarity",
    "selected_count_difference",
]


for column_name in required_numeric_columns:

    if column_name not in mechanism_v2_df.columns:

        raise KeyError(
            f"Required column missing: {column_name}"
        )


    mechanism_v2_df[
        column_name
    ] = pd.to_numeric(
        mechanism_v2_df[
            column_name
        ],
        errors="coerce",
    )


# --------------------------------------------------
# Classify the Greedy–Exact relationship
# --------------------------------------------------

SUBSET_EQUALITY_TOLERANCE = 1e-12


identical_subset_mask = (
    mechanism_v2_df[
        "jaccard_similarity"
    ]
    >=
    1.0
    -
    SUBSET_EQUALITY_TOLERANCE
)


greedy_more_items_mask = (
    mechanism_v2_df[
        "selected_count_difference"
    ]
    > 0
)


greedy_fewer_items_mask = (
    mechanism_v2_df[
        "selected_count_difference"
    ]
    < 0
)


same_count_different_subset_mask = (
    ~identical_subset_mask
    &
    (
        mechanism_v2_df[
            "selected_count_difference"
        ]
        == 0
    )
)


mechanism_v2_df[
    "failure_mechanism"
] = np.select(
    [
        identical_subset_mask,
        same_count_different_subset_mask,
        greedy_more_items_mask,
        greedy_fewer_items_mask,
    ],
    [
        "identical_subset",
        "same_count_different_items",
        "greedy_more_items",
        "greedy_fewer_items",
    ],
    default="unclassified",
)


MECHANISM_ORDER = [
    "identical_subset",
    "same_count_different_items",
    "greedy_more_items",
    "greedy_fewer_items",
]


mechanism_v2_df[
    "failure_mechanism"
] = pd.Categorical(
    mechanism_v2_df[
        "failure_mechanism"
    ],
    categories=MECHANISM_ORDER,
    ordered=True,
)


# --------------------------------------------------
# Validate exhaustive and mutually exclusive classes
# --------------------------------------------------

assert (
    mechanism_v2_df[
        "failure_mechanism"
    ]
    .notna()
    .all()
)


assert (
    mechanism_v2_df[
        "failure_mechanism"
    ]
    .astype(str)
    != "unclassified"
).all()


assert (
    mechanism_v2_df[
        "failure_mechanism"
    ]
    .value_counts()
    .sum()
    ==
    len(
        mechanism_v2_df
    )
)


# --------------------------------------------------
# Indicators used in summaries
# --------------------------------------------------

OPTIMAL_COST_TOLERANCE_PERCENT = 1e-10


mechanism_v2_df[
    "greedy_optimal_cost_numeric"
] = (
    mechanism_v2_df[
        "optimality_gap_percent"
    ]
    .abs()
    <=
    OPTIMAL_COST_TOLERANCE_PERCENT
).astype(int)


mechanism_v2_df[
    "large_gap_numeric"
] = (
    mechanism_v2_df[
        "optimality_gap_percent"
    ]
    > 10.0
).astype(int)


mechanism_v2_df[
    "within_1_percent_numeric"
] = (
    mechanism_v2_df[
        "optimality_gap_percent"
    ]
    <= 1.0
).astype(int)


mechanism_v2_df[
    "within_5_percent_numeric"
] = (
    mechanism_v2_df[
        "optimality_gap_percent"
    ]
    <= 5.0
).astype(int)


mechanism_v2_df[
    "within_10_percent_numeric"
] = (
    mechanism_v2_df[
        "optimality_gap_percent"
    ]
    <= 10.0
).astype(int)


# --------------------------------------------------
# Failure-mechanism descriptive summary
# --------------------------------------------------

mechanism_summary_rows = []


for mechanism in MECHANISM_ORDER:

    current_df = (
        mechanism_v2_df[
            mechanism_v2_df[
                "failure_mechanism"
            ]
            == mechanism
        ]
    )


    if len(
        current_df
    ) == 0:

        mechanism_summary_rows.append(
            {
                "failure_mechanism":
                    mechanism,

                "rows":
                    0,

                "percent_of_comparable_rows":
                    0.0,
            }
        )

        continue


    gap_values = (
        current_df[
            "optimality_gap_percent"
        ]
        .to_numpy(
            dtype=float
        )
    )


    mechanism_summary_rows.append(
        {
            "failure_mechanism":
                mechanism,

            "rows":
                len(
                    current_df
                ),

            "percent_of_comparable_rows":
                100.0
                *
                len(
                    current_df
                )
                /
                len(
                    mechanism_v2_df
                ),

            "greedy_optimal_cost_percent":
                100.0
                *
                current_df[
                    "greedy_optimal_cost_numeric"
                ].mean(),

            "mean_optimality_gap_percent":
                gap_values.mean(),

            "median_optimality_gap_percent":
                np.median(
                    gap_values
                ),

            "p90_optimality_gap_percent":
                np.quantile(
                    gap_values,
                    0.90,
                ),

            "p95_optimality_gap_percent":
                np.quantile(
                    gap_values,
                    0.95,
                ),

            "maximum_optimality_gap_percent":
                gap_values.max(),

            "large_gap_percent":
                100.0
                *
                current_df[
                    "large_gap_numeric"
                ].mean(),

            "within_1_percent":
                100.0
                *
                current_df[
                    "within_1_percent_numeric"
                ].mean(),

            "within_5_percent":
                100.0
                *
                current_df[
                    "within_5_percent_numeric"
                ].mean(),

            "within_10_percent":
                100.0
                *
                current_df[
                    "within_10_percent_numeric"
                ].mean(),

            "mean_jaccard_similarity":
                current_df[
                    "jaccard_similarity"
                ].mean(),

            "mean_selected_count_difference":
                current_df[
                    "selected_count_difference"
                ].mean(),

            "minimum_selected_count_difference":
                current_df[
                    "selected_count_difference"
                ].min(),

            "maximum_selected_count_difference":
                current_df[
                    "selected_count_difference"
                ].max(),

            "matched_groups":
                current_df[
                    "matched_group_id"
                ]
                .nunique(),

            "regime_specific_instances":
                current_df[
                    "instance_id"
                ]
                .nunique(),
        }
    )


mechanism_v2_summary_df = pd.DataFrame(
    mechanism_summary_rows
)


mechanism_v2_summary_df[
    "failure_mechanism"
] = pd.Categorical(
    mechanism_v2_summary_df[
        "failure_mechanism"
    ],
    categories=MECHANISM_ORDER,
    ordered=True,
)


mechanism_v2_summary_df = (
    mechanism_v2_summary_df
    .sort_values(
        "failure_mechanism"
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------
# Detect objective-component columns
# --------------------------------------------------

def first_available_column(
    dataframe,
    candidates,
):

    for candidate in candidates:

        if candidate in dataframe.columns:

            return candidate

    return None


exact_privacy_column = first_available_column(
    mechanism_v2_df,
    [
        "exact_privacy",
        "exact_P",
    ],
)


greedy_privacy_column = first_available_column(
    mechanism_v2_df,
    [
        "greedy_privacy",
        "greedy_P",
    ],
)


exact_cognitive_column = first_available_column(
    mechanism_v2_df,
    [
        "exact_normalized_cognitive_cost",
        "exact_cognitive_cost_normalized",
        "exact_cognitive_cost",
        "exact_C",
    ],
)


greedy_cognitive_column = first_available_column(
    mechanism_v2_df,
    [
        "greedy_normalized_cognitive_cost",
        "greedy_cognitive_cost_normalized",
        "greedy_cognitive_cost",
        "greedy_C",
    ],
)


exact_cost_column = first_available_column(
    mechanism_v2_df,
    [
        "exact_J",
        "exact_cost",
        "exact_disclosure_cost",
    ],
)


greedy_cost_column = first_available_column(
    mechanism_v2_df,
    [
        "greedy_J",
        "greedy_cost",
        "greedy_disclosure_cost",
    ],
)


# --------------------------------------------------
# Create raw component differences where available
# --------------------------------------------------

component_difference_definitions = []


if (
    exact_privacy_column is not None
    and
    greedy_privacy_column is not None
):

    mechanism_v2_df[
        "privacy_difference"
    ] = (
        pd.to_numeric(
            mechanism_v2_df[
                greedy_privacy_column
            ],
            errors="coerce",
        )
        -
        pd.to_numeric(
            mechanism_v2_df[
                exact_privacy_column
            ],
            errors="coerce",
        )
    )


    component_difference_definitions.append(
        (
            "privacy_difference",
            greedy_privacy_column,
            exact_privacy_column,
        )
    )


if (
    exact_cognitive_column is not None
    and
    greedy_cognitive_column is not None
):

    mechanism_v2_df[
        "cognitive_difference"
    ] = (
        pd.to_numeric(
            mechanism_v2_df[
                greedy_cognitive_column
            ],
            errors="coerce",
        )
        -
        pd.to_numeric(
            mechanism_v2_df[
                exact_cognitive_column
            ],
            errors="coerce",
        )
    )


    component_difference_definitions.append(
        (
            "cognitive_difference",
            greedy_cognitive_column,
            exact_cognitive_column,
        )
    )


mechanism_v2_df[
    "cardinality_fraction_difference"
] = (
    mechanism_v2_df[
        "selected_count_difference"
    ]
    /
    pd.to_numeric(
        mechanism_v2_df[
            "n_items"
        ],
        errors="coerce",
    )
)


component_difference_definitions.append(
    (
        "cardinality_fraction_difference",
        "greedy_selected_count / n_items",
        "exact_selected_count / n_items",
    )
)


if (
    exact_cost_column is not None
    and
    greedy_cost_column is not None
):

    mechanism_v2_df[
        "absolute_objective_difference"
    ] = (
        pd.to_numeric(
            mechanism_v2_df[
                greedy_cost_column
            ],
            errors="coerce",
        )
        -
        pd.to_numeric(
            mechanism_v2_df[
                exact_cost_column
            ],
            errors="coerce",
        )
    )


    component_difference_definitions.append(
        (
            "absolute_objective_difference",
            greedy_cost_column,
            exact_cost_column,
        )
    )


# --------------------------------------------------
# Summarize component differences by mechanism
# --------------------------------------------------

component_summary_rows = []


for mechanism in MECHANISM_ORDER:

    current_df = (
        mechanism_v2_df[
            mechanism_v2_df[
                "failure_mechanism"
            ]
            == mechanism
        ]
    )


    for (
        difference_column,
        greedy_source,
        exact_source,
    ) in component_difference_definitions:

        valid_values = (
            current_df[
                difference_column
            ]
            .dropna()
            .to_numpy(
                dtype=float
            )
        )


        if len(
            valid_values
        ) == 0:

            continue


        component_summary_rows.append(
            {
                "failure_mechanism":
                    mechanism,

                "difference_measure":
                    difference_column,

                "definition":
                    (
                        f"{greedy_source}"
                        f" minus "
                        f"{exact_source}"
                    ),

                "rows":
                    len(
                        valid_values
                    ),

                "mean_difference":
                    valid_values.mean(),

                "median_difference":
                    np.median(
                        valid_values
                    ),

                "p90_difference":
                    np.quantile(
                        valid_values,
                        0.90,
                    ),

                "minimum_difference":
                    valid_values.min(),

                "maximum_difference":
                    valid_values.max(),
            }
        )


mechanism_v2_component_df = pd.DataFrame(
    component_summary_rows
)


# --------------------------------------------------
# Integrity checks
# --------------------------------------------------

assert len(
    mechanism_v2_df
) == 23784


assert (
    mechanism_v2_summary_df[
        "rows"
    ].sum()
    == 23784
)


identical_subset_rows = int(
    (
        mechanism_v2_df[
            "failure_mechanism"
        ]
        == "identical_subset"
    ).sum()
)


optimal_cost_rows = int(
    mechanism_v2_df[
        "greedy_optimal_cost_numeric"
    ].sum()
)


assert (
    identical_subset_rows
    ==
    optimal_cost_rows
)


assert (
    mechanism_v2_df.loc[
        mechanism_v2_df[
            "failure_mechanism"
        ]
        == "identical_subset",
        "optimality_gap_percent",
    ]
    .abs()
    .max()
    <= 1e-10
)


# --------------------------------------------------
# Save outputs
# --------------------------------------------------

mechanism_v2_summary_df.to_csv(
    MECHANISM_V2_SUMMARY_PATH,
    index=False,
)


mechanism_v2_component_df.to_csv(
    MECHANISM_V2_COMPONENT_PATH,
    index=False,
)


mechanism_v2_df.to_csv(
    MECHANISM_V2_ROW_LEVEL_PATH,
    index=False,
)


# --------------------------------------------------
# Display results
# --------------------------------------------------

mechanism_display_columns = [
    "failure_mechanism",
    "rows",
    "percent_of_comparable_rows",
    "greedy_optimal_cost_percent",
    "mean_optimality_gap_percent",
    "median_optimality_gap_percent",
    "p90_optimality_gap_percent",
    "maximum_optimality_gap_percent",
    "large_gap_percent",
    "mean_jaccard_similarity",
    "mean_selected_count_difference",
]


print(
    "======================================"
)

print(
    "CORRECTED V2 GREEDY FAILURE MECHANISMS"
)

print(
    "======================================"
)


print(
    mechanism_v2_summary_df[
        mechanism_display_columns
    ].to_string(
        index=False,
        float_format=lambda value:
            f"{value:.4f}",
    )
)


print(
    "\n======================================"
)

print(
    "OBJECTIVE-COMPONENT DIFFERENCES"
)

print(
    "======================================"
)


if len(
    mechanism_v2_component_df
) > 0:

    print(
        mechanism_v2_component_df.to_string(
            index=False,
            float_format=lambda value:
                f"{value:.6f}",
        )
    )

else:

    print(
        "No compatible objective-component columns were found."
    )


print(
    "\nFailure-mechanism summary saved to:"
)

print(
    MECHANISM_V2_SUMMARY_PATH
)


print(
    "\nComponent-difference summary saved to:"
)

print(
    MECHANISM_V2_COMPONENT_PATH
)


print(
    "\nRow-level mechanism dataset saved to:"
)

print(
    MECHANISM_V2_ROW_LEVEL_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Corrected v2 failure mechanisms by structural regime
# with matched-group clustered confidence intervals
# --------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd


MECHANISM_REGIME_BOOTSTRAP_REPLICATES = 5000
MECHANISM_REGIME_BOOTSTRAP_SEED = GLOBAL_SEED + 110


if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


MECHANISM_ROW_LEVEL_PATH = (
    RESULTS_DIR
    / "corrected_v2_greedy_failure_mechanism_rows.csv"
)


MECHANISM_REGIME_SUMMARY_PATH = (
    TABLES_DIR
    / "corrected_v2_failure_mechanism_by_regime_summary.csv"
)


MECHANISM_REGIME_BOOTSTRAP_PATH = (
    TABLES_DIR
    / "corrected_v2_failure_mechanism_by_regime_bootstrap_ci.csv"
)


MECHANISM_REGIME_REPLICATES_PATH = (
    RESULTS_DIR
    / "corrected_v2_failure_mechanism_by_regime_bootstrap_replicates.csv"
)


MECHANISM_ORDER = [
    "identical_subset",
    "same_count_different_items",
    "greedy_more_items",
    "greedy_fewer_items",
]


REGIME_ORDER = [
    "independent",
    "utility_burden_alignment",
    "utility_burden_conflict",
]


# --------------------------------------------------
# Load row-level mechanism dataset
# --------------------------------------------------

mechanism_regime_df = pd.read_csv(
    MECHANISM_ROW_LEVEL_PATH,
    low_memory=False,
)


for column_name in [
    "n_items",
    "optimality_gap_percent",
]:

    mechanism_regime_df[
        column_name
    ] = pd.to_numeric(
        mechanism_regime_df[
            column_name
        ],
        errors="coerce",
    )


mechanism_regime_df[
    "large_gap_numeric"
] = (
    mechanism_regime_df[
        "optimality_gap_percent"
    ]
    > 10.0
).astype(int)


# --------------------------------------------------
# Overall large-gap attribution
# --------------------------------------------------

overall_mechanism_attribution_df = (
    mechanism_regime_df
    .groupby(
        "failure_mechanism",
        as_index=False,
        observed=False,
    )
    .agg(
        mechanism_rows=(
            "optimality_gap_percent",
            "size",
        ),

        large_gap_rows=(
            "large_gap_numeric",
            "sum",
        ),

        mean_optimality_gap_percent=(
            "optimality_gap_percent",
            "mean",
        ),
    )
)


overall_mechanism_attribution_df[
    "percent_of_comparable_rows"
] = (
    100.0
    *
    overall_mechanism_attribution_df[
        "mechanism_rows"
    ]
    /
    len(
        mechanism_regime_df
    )
)


total_large_gap_rows = int(
    mechanism_regime_df[
        "large_gap_numeric"
    ].sum()
)


overall_mechanism_attribution_df[
    "percent_of_all_large_gap_rows"
] = (
    100.0
    *
    overall_mechanism_attribution_df[
        "large_gap_rows"
    ]
    /
    total_large_gap_rows
)


overall_mechanism_attribution_df[
    "failure_mechanism"
] = pd.Categorical(
    overall_mechanism_attribution_df[
        "failure_mechanism"
    ],
    categories=MECHANISM_ORDER,
    ordered=True,
)


overall_mechanism_attribution_df = (
    overall_mechanism_attribution_df
    .sort_values(
        "failure_mechanism"
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------
# One row per matched group and regime containing
# total counts and mechanism-specific sufficient
# statistics
# --------------------------------------------------

group_total_df = (
    mechanism_regime_df
    .groupby(
        [
            "matched_group_id",
            "n_items",
            "regime",
        ],
        as_index=False,
    )
    .agg(
        total_comparable_rows=(
            "optimality_gap_percent",
            "size",
        ),

        total_large_gap_rows=(
            "large_gap_numeric",
            "sum",
        ),
    )
)


group_mechanism_frames = []


for mechanism in MECHANISM_ORDER:

    mechanism_subset_df = (
        mechanism_regime_df[
            mechanism_regime_df[
                "failure_mechanism"
            ]
            == mechanism
        ]
    )


    mechanism_statistics_df = (
        mechanism_subset_df
        .groupby(
            [
                "matched_group_id",
                "n_items",
                "regime",
            ],
            as_index=False,
        )
        .agg(
            mechanism_rows=(
                "optimality_gap_percent",
                "size",
            ),

            mechanism_large_gap_rows=(
                "large_gap_numeric",
                "sum",
            ),

            mechanism_gap_sum=(
                "optimality_gap_percent",
                "sum",
            ),
        )
    )


    complete_mechanism_df = (
        group_total_df
        .merge(
            mechanism_statistics_df,
            on=[
                "matched_group_id",
                "n_items",
                "regime",
            ],
            how="left",
        )
    )


    for column_name in [
        "mechanism_rows",
        "mechanism_large_gap_rows",
        "mechanism_gap_sum",
    ]:

        complete_mechanism_df[
            column_name
        ] = (
            complete_mechanism_df[
                column_name
            ]
            .fillna(0.0)
        )


    complete_mechanism_df[
        "failure_mechanism"
    ] = mechanism


    group_mechanism_frames.append(
        complete_mechanism_df
    )


group_mechanism_df = pd.concat(
    group_mechanism_frames,
    ignore_index=True,
)


# --------------------------------------------------
# Regime-level descriptive summary
# --------------------------------------------------

regime_mechanism_summary_df = (
    group_mechanism_df
    .groupby(
        [
            "regime",
            "failure_mechanism",
        ],
        as_index=False,
        observed=False,
    )
    .agg(
        total_comparable_rows=(
            "total_comparable_rows",
            "sum",
        ),

        total_large_gap_rows=(
            "total_large_gap_rows",
            "sum",
        ),

        mechanism_rows=(
            "mechanism_rows",
            "sum",
        ),

        mechanism_large_gap_rows=(
            "mechanism_large_gap_rows",
            "sum",
        ),

        mechanism_gap_sum=(
            "mechanism_gap_sum",
            "sum",
        ),
    )
)


regime_mechanism_summary_df[
    "mechanism_prevalence_percent"
] = (
    100.0
    *
    regime_mechanism_summary_df[
        "mechanism_rows"
    ]
    /
    regime_mechanism_summary_df[
        "total_comparable_rows"
    ]
)


regime_mechanism_summary_df[
    "large_gap_attribution_percent"
] = np.where(
    regime_mechanism_summary_df[
        "total_large_gap_rows"
    ]
    > 0,

    100.0
    *
    regime_mechanism_summary_df[
        "mechanism_large_gap_rows"
    ]
    /
    regime_mechanism_summary_df[
        "total_large_gap_rows"
    ],

    np.nan,
)


regime_mechanism_summary_df[
    "mean_gap_within_mechanism"
] = np.where(
    regime_mechanism_summary_df[
        "mechanism_rows"
    ]
    > 0,

    regime_mechanism_summary_df[
        "mechanism_gap_sum"
    ]
    /
    regime_mechanism_summary_df[
        "mechanism_rows"
    ],

    np.nan,
)


# --------------------------------------------------
# Convert group-mechanism data to one wide row per
# matched group and regime
# --------------------------------------------------

mechanism_wide_df = (
    group_mechanism_df
    .pivot(
        index=[
            "matched_group_id",
            "n_items",
            "regime",
            "total_comparable_rows",
            "total_large_gap_rows",
        ],
        columns="failure_mechanism",
        values=[
            "mechanism_rows",
            "mechanism_large_gap_rows",
            "mechanism_gap_sum",
        ],
    )
    .fillna(0.0)
    .reset_index()
)


mechanism_wide_df.columns = [
    (
        column
        if isinstance(
            column,
            str,
        )
        else
        "_".join(
            [
                str(part)
                for part in column
                if str(part) != ""
            ]
        )
    )
    for column in mechanism_wide_df.columns
]


# --------------------------------------------------
# Define bootstrap array columns
# --------------------------------------------------

MECHANISM_BOOTSTRAP_COLUMNS = [
    "total_comparable_rows",
    "total_large_gap_rows",
]


for mechanism in MECHANISM_ORDER:

    MECHANISM_BOOTSTRAP_COLUMNS.extend(
        [
            f"mechanism_rows_{mechanism}",
            f"mechanism_large_gap_rows_{mechanism}",
            f"mechanism_gap_sum_{mechanism}",
        ]
    )


for column_name in MECHANISM_BOOTSTRAP_COLUMNS:

    if column_name not in mechanism_wide_df.columns:

        mechanism_wide_df[
            column_name
        ] = 0.0


# --------------------------------------------------
# Convert totals into reporting metrics
# --------------------------------------------------

MECHANISM_BOOTSTRAP_METRIC_NAMES = []


for mechanism in MECHANISM_ORDER:

    MECHANISM_BOOTSTRAP_METRIC_NAMES.extend(
        [
            (
                f"{mechanism}"
                "_prevalence_percent"
            ),
            (
                f"{mechanism}"
                "_large_gap_attribution_percent"
            ),
            (
                f"{mechanism}"
                "_mean_gap_percent"
            ),
        ]
    )


def calculate_mechanism_regime_metrics(
    totals: np.ndarray,
) -> np.ndarray:
    """
    Convert matched-group totals into mechanism
    prevalence, large-gap attribution, and mean gap.
    """

    total_rows = totals[0]
    total_large_gaps = totals[1]

    calculated_metrics = []

    position = 2


    for mechanism in MECHANISM_ORDER:

        mechanism_rows = totals[
            position
        ]

        mechanism_large_gaps = totals[
            position + 1
        ]

        mechanism_gap_sum = totals[
            position + 2
        ]


        prevalence_percent = (
            100.0
            * mechanism_rows
            / total_rows
        )


        if total_large_gaps > 0:

            attribution_percent = (
                100.0
                * mechanism_large_gaps
                / total_large_gaps
            )

        else:

            attribution_percent = np.nan


        if mechanism_rows > 0:

            mean_gap_percent = (
                mechanism_gap_sum
                / mechanism_rows
            )

        else:

            mean_gap_percent = np.nan


        calculated_metrics.extend(
            [
                prevalence_percent,
                attribution_percent,
                mean_gap_percent,
            ]
        )


        position += 3


    return np.array(
        calculated_metrics,
        dtype=float,
    )


# --------------------------------------------------
# Validate matched-group structure
# --------------------------------------------------

groups_by_regime_size = (
    mechanism_wide_df
    .groupby(
        [
            "regime",
            "n_items",
        ]
    )[
        "matched_group_id"
    ]
    .nunique()
)


assert (
    groups_by_regime_size
    == 100
).all()


assert (
    mechanism_wide_df[
        "matched_group_id"
    ].nunique()
    == 500
)


# --------------------------------------------------
# Bootstrap separately for each regime, preserving
# candidate-set-size strata
# --------------------------------------------------

mechanism_regime_rng = np.random.default_rng(
    MECHANISM_REGIME_BOOTSTRAP_SEED
)


mechanism_bootstrap_summary_rows = []
mechanism_bootstrap_replicate_frames = []


for regime in REGIME_ORDER:

    regime_group_df = (
        mechanism_wide_df[
            mechanism_wide_df[
                "regime"
            ]
            == regime
        ]
        .copy()
    )


    observed_totals = (
        regime_group_df[
            MECHANISM_BOOTSTRAP_COLUMNS
        ]
        .sum()
        .to_numpy(
            dtype=float
        )
    )


    observed_metrics = (
        calculate_mechanism_regime_metrics(
            observed_totals
        )
    )


    size_stratum_arrays = []


    for n_items, size_df in (
        regime_group_df
        .groupby(
            "n_items",
            sort=True,
        )
    ):

        size_array = (
            size_df
            .sort_values(
                "matched_group_id"
            )[
                MECHANISM_BOOTSTRAP_COLUMNS
            ]
            .to_numpy(
                dtype=float
            )
        )


        assert (
            size_array.shape[0]
            == 100
        )


        size_stratum_arrays.append(
            size_array
        )


    bootstrap_estimates = np.empty(
        (
            MECHANISM_REGIME_BOOTSTRAP_REPLICATES,
            len(
                MECHANISM_BOOTSTRAP_METRIC_NAMES
            ),
        ),
        dtype=float,
    )


    for bootstrap_index in range(
        MECHANISM_REGIME_BOOTSTRAP_REPLICATES
    ):

        resampled_totals = np.zeros(
            len(
                MECHANISM_BOOTSTRAP_COLUMNS
            ),
            dtype=float,
        )


        for size_array in size_stratum_arrays:

            number_of_groups = (
                size_array.shape[0]
            )


            sampled_indices = (
                mechanism_regime_rng.integers(
                    low=0,
                    high=number_of_groups,
                    size=number_of_groups,
                )
            )


            resampled_totals += (
                size_array[
                    sampled_indices
                ]
                .sum(
                    axis=0
                )
            )


        bootstrap_estimates[
            bootstrap_index
        ] = calculate_mechanism_regime_metrics(
            resampled_totals
        )


    standard_errors = (
        bootstrap_estimates.std(
            axis=0,
            ddof=1,
        )
    )


    confidence_lower = np.quantile(
        bootstrap_estimates,
        0.025,
        axis=0,
    )


    confidence_upper = np.quantile(
        bootstrap_estimates,
        0.975,
        axis=0,
    )


    metric_index = 0


    for mechanism in MECHANISM_ORDER:

        for metric_name in [
            "mechanism_prevalence_percent",
            "large_gap_attribution_percent",
            "mean_gap_within_mechanism",
        ]:

            mechanism_bootstrap_summary_rows.append(
                {
                    "regime":
                        regime,

                    "failure_mechanism":
                        mechanism,

                    "metric":
                        metric_name,

                    "estimate":
                        observed_metrics[
                            metric_index
                        ],

                    "bootstrap_standard_error":
                        standard_errors[
                            metric_index
                        ],

                    "ci_95_lower":
                        confidence_lower[
                            metric_index
                        ],

                    "ci_95_upper":
                        confidence_upper[
                            metric_index
                        ],

                    "matched_groups":
                        500,

                    "candidate_size_strata":
                        5,

                    "bootstrap_replicates":
                        MECHANISM_REGIME_BOOTSTRAP_REPLICATES,

                    "resampling_unit":
                        "matched_size_replicate_group",

                    "resampling_design":
                        "stratified_by_candidate_set_size",
                }
            )


            metric_index += 1


    replicate_df = pd.DataFrame(
        bootstrap_estimates,
        columns=MECHANISM_BOOTSTRAP_METRIC_NAMES,
    )


    replicate_df.insert(
        0,
        "bootstrap_replicate",
        np.arange(
            MECHANISM_REGIME_BOOTSTRAP_REPLICATES
        ),
    )


    replicate_df.insert(
        0,
        "regime",
        regime,
    )


    mechanism_bootstrap_replicate_frames.append(
        replicate_df
    )


    print(
        "Completed failure-mechanism bootstrap:",
        regime
    )


# --------------------------------------------------
# Save outputs
# --------------------------------------------------

mechanism_regime_bootstrap_df = pd.DataFrame(
    mechanism_bootstrap_summary_rows
)


mechanism_regime_replicates_df = pd.concat(
    mechanism_bootstrap_replicate_frames,
    ignore_index=True,
)


regime_mechanism_summary_df.to_csv(
    MECHANISM_REGIME_SUMMARY_PATH,
    index=False,
)


mechanism_regime_bootstrap_df.to_csv(
    MECHANISM_REGIME_BOOTSTRAP_PATH,
    index=False,
)


mechanism_regime_replicates_df.to_csv(
    MECHANISM_REGIME_REPLICATES_PATH,
    index=False,
)


# --------------------------------------------------
# Compact bootstrap display for the two substantive
# suboptimal mechanisms
# --------------------------------------------------

key_mechanisms = [
    "same_count_different_items",
    "greedy_more_items",
]


key_metrics = [
    "mechanism_prevalence_percent",
    "large_gap_attribution_percent",
    "mean_gap_within_mechanism",
]


mechanism_bootstrap_display_df = (
    mechanism_regime_bootstrap_df[
        (
            mechanism_regime_bootstrap_df[
                "failure_mechanism"
            ].isin(
                key_mechanisms
            )
        )
        &
        (
            mechanism_regime_bootstrap_df[
                "metric"
            ].isin(
                key_metrics
            )
        )
    ]
    .copy()
)


mechanism_bootstrap_display_df[
    "estimate_and_ci"
] = (
    mechanism_bootstrap_display_df[
        "estimate"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    mechanism_bootstrap_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    mechanism_bootstrap_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


# --------------------------------------------------
# Display
# --------------------------------------------------

print(
    "\n======================================"
)

print(
    "OVERALL LARGE-GAP ATTRIBUTION"
)

print(
    "======================================"
)


print(
    overall_mechanism_attribution_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.4f}",
    )
)


print(
    "\n======================================"
)

print(
    "FAILURE MECHANISMS BY REGIME"
)

print(
    "======================================"
)


print(
    regime_mechanism_summary_df[
        [
            "regime",
            "failure_mechanism",
            "mechanism_rows",
            "mechanism_prevalence_percent",
            "mechanism_large_gap_rows",
            "large_gap_attribution_percent",
            "mean_gap_within_mechanism",
        ]
    ].to_string(
        index=False,
        float_format=lambda value:
            f"{value:.4f}",
    )
)


print(
    "\n======================================"
)

print(
    "MATCHED-GROUP BOOTSTRAP 95% CIs"
)

print(
    "======================================"
)


print(
    mechanism_bootstrap_display_df[
        [
            "regime",
            "failure_mechanism",
            "metric",
            "estimate_and_ci",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\nRegime mechanism summary saved to:"
)

print(
    MECHANISM_REGIME_SUMMARY_PATH
)


print(
    "\nMechanism confidence intervals saved to:"
)

print(
    MECHANISM_REGIME_BOOTSTRAP_PATH
)


print(
    "\nBootstrap replicates saved to:"
)

print(
    MECHANISM_REGIME_REPLICATES_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Corrected v2 empirical runtime-scaling analysis
# --------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd


RUNTIME_BOOTSTRAP_REPLICATES = 5000
RUNTIME_BOOTSTRAP_SEED = GLOBAL_SEED + 111


if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


CORRECTED_RAW_BENCHMARK_PATH = (
    RESULTS_DIR
    / "exact_vs_greedy_main_benchmark_v2_marginal_preserving.csv"
)


RUNTIME_INSTANCE_PATH = (
    RESULTS_DIR
    / "corrected_v2_runtime_instance_level.csv"
)


RUNTIME_MATCHED_GROUP_PATH = (
    RESULTS_DIR
    / "corrected_v2_runtime_matched_group_level.csv"
)


RUNTIME_SIZE_SUMMARY_PATH = (
    TABLES_DIR
    / "corrected_v2_runtime_scaling_by_candidate_size.csv"
)


RUNTIME_MODEL_FIT_PATH = (
    TABLES_DIR
    / "corrected_v2_runtime_scaling_model_fits.csv"
)


RUNTIME_BOOTSTRAP_REPLICATES_PATH = (
    RESULTS_DIR
    / "corrected_v2_runtime_scaling_bootstrap_replicates.csv"
)


# --------------------------------------------------
# Load corrected raw benchmark
# --------------------------------------------------

runtime_raw_df = pd.read_csv(
    CORRECTED_RAW_BENCHMARK_PATH,
    low_memory=False,
)


required_runtime_columns = [
    "instance_id",
    "matched_group_id",
    "n_items",
    "regime",
    "subset_precomputation_runtime_seconds",
    "exact_cached_solve_runtime_seconds",
    "greedy_runtime_seconds",
]


missing_runtime_columns = [
    column_name
    for column_name in required_runtime_columns
    if column_name not in runtime_raw_df.columns
]


if missing_runtime_columns:

    raise KeyError(
        "Missing required runtime columns: "
        + ", ".join(
            missing_runtime_columns
        )
    )


for column_name in [
    "n_items",
    "subset_precomputation_runtime_seconds",
    "exact_cached_solve_runtime_seconds",
    "greedy_runtime_seconds",
]:

    runtime_raw_df[
        column_name
    ] = pd.to_numeric(
        runtime_raw_df[
            column_name
        ],
        errors="coerce",
    )


# --------------------------------------------------
# Build one runtime row per regime-specific instance
# --------------------------------------------------

runtime_instance_df = (
    runtime_raw_df
    .groupby(
        [
            "instance_id",
            "matched_group_id",
            "n_items",
            "regime",
        ],
        as_index=False,
    )
    .agg(
        conditions=(
            "tau",
            "size",
        ),

        subset_precomputation_seconds=(
            "subset_precomputation_runtime_seconds",
            "first",
        ),

        cached_exact_total_seconds=(
            "exact_cached_solve_runtime_seconds",
            "sum",
        ),

        cached_exact_mean_seconds_per_condition=(
            "exact_cached_solve_runtime_seconds",
            "mean",
        ),

        cached_exact_median_seconds_per_condition=(
            "exact_cached_solve_runtime_seconds",
            "median",
        ),

        greedy_total_seconds=(
            "greedy_runtime_seconds",
            "sum",
        ),

        greedy_mean_seconds_per_condition=(
            "greedy_runtime_seconds",
            "mean",
        ),

        greedy_median_seconds_per_condition=(
            "greedy_runtime_seconds",
            "median",
        ),
    )
)


runtime_instance_df[
    "total_exact_seconds"
] = (
    runtime_instance_df[
        "subset_precomputation_seconds"
    ]
    +
    runtime_instance_df[
        "cached_exact_total_seconds"
    ]
)


runtime_instance_df[
    "cached_exact_to_greedy_condition_ratio"
] = (
    runtime_instance_df[
        "cached_exact_mean_seconds_per_condition"
    ]
    /
    runtime_instance_df[
        "greedy_mean_seconds_per_condition"
    ]
)


runtime_instance_df[
    "total_exact_to_greedy_total_ratio"
] = (
    runtime_instance_df[
        "total_exact_seconds"
    ]
    /
    runtime_instance_df[
        "greedy_total_seconds"
    ]
)


# --------------------------------------------------
# Integrity checks at the instance level
# --------------------------------------------------

assert len(
    runtime_instance_df
) == 1500


assert (
    runtime_instance_df[
        "conditions"
    ]
    == 16
).all()


positive_runtime_columns = [
    "subset_precomputation_seconds",
    "cached_exact_mean_seconds_per_condition",
    "greedy_mean_seconds_per_condition",
    "total_exact_seconds",
]


for column_name in positive_runtime_columns:

    assert (
        runtime_instance_df[
            column_name
        ]
        > 0.0
    ).all()


runtime_instance_df.to_csv(
    RUNTIME_INSTANCE_PATH,
    index=False,
)


# --------------------------------------------------
# Aggregate matched regimes into the 500 independent
# size–replicate groups
# --------------------------------------------------

runtime_matched_group_df = (
    runtime_instance_df
    .groupby(
        [
            "matched_group_id",
            "n_items",
        ],
        as_index=False,
    )
    .agg(
        regime_specific_instances=(
            "instance_id",
            "nunique",
        ),

        subset_precomputation_seconds=(
            "subset_precomputation_seconds",
            "mean",
        ),

        cached_exact_total_seconds=(
            "cached_exact_total_seconds",
            "mean",
        ),

        cached_exact_mean_seconds_per_condition=(
            "cached_exact_mean_seconds_per_condition",
            "mean",
        ),

        greedy_total_seconds=(
            "greedy_total_seconds",
            "mean",
        ),

        greedy_mean_seconds_per_condition=(
            "greedy_mean_seconds_per_condition",
            "mean",
        ),

        total_exact_seconds=(
            "total_exact_seconds",
            "mean",
        ),

        cached_exact_to_greedy_condition_ratio=(
            "cached_exact_to_greedy_condition_ratio",
            "mean",
        ),

        total_exact_to_greedy_total_ratio=(
            "total_exact_to_greedy_total_ratio",
            "mean",
        ),
    )
)


assert len(
    runtime_matched_group_df
) == 500


assert (
    runtime_matched_group_df[
        "regime_specific_instances"
    ]
    == 3
).all()


groups_per_runtime_size = (
    runtime_matched_group_df
    .groupby(
        "n_items"
    )[
        "matched_group_id"
    ]
    .nunique()
)


assert (
    groups_per_runtime_size
    == 100
).all()


runtime_matched_group_df.to_csv(
    RUNTIME_MATCHED_GROUP_PATH,
    index=False,
)


# --------------------------------------------------
# Candidate-size runtime summaries
# --------------------------------------------------

runtime_size_summary_rows = []


for n_items in sorted(
    runtime_matched_group_df[
        "n_items"
    ].unique()
):

    size_df = (
        runtime_matched_group_df[
            runtime_matched_group_df[
                "n_items"
            ]
            == n_items
        ]
    )


    runtime_size_summary_rows.append(
        {
            "n_items":
                int(
                    n_items
                ),

            "matched_groups":
                len(
                    size_df
                ),

            "median_subset_precomputation_seconds":
                size_df[
                    "subset_precomputation_seconds"
                ].median(),

            "q25_subset_precomputation_seconds":
                size_df[
                    "subset_precomputation_seconds"
                ].quantile(
                    0.25
                ),

            "q75_subset_precomputation_seconds":
                size_df[
                    "subset_precomputation_seconds"
                ].quantile(
                    0.75
                ),

            "median_total_exact_seconds_per_instance":
                size_df[
                    "total_exact_seconds"
                ].median(),

            "q25_total_exact_seconds_per_instance":
                size_df[
                    "total_exact_seconds"
                ].quantile(
                    0.25
                ),

            "q75_total_exact_seconds_per_instance":
                size_df[
                    "total_exact_seconds"
                ].quantile(
                    0.75
                ),

            "median_cached_exact_milliseconds_per_condition":
                1000.0
                *
                size_df[
                    "cached_exact_mean_seconds_per_condition"
                ].median(),

            "q25_cached_exact_milliseconds_per_condition":
                1000.0
                *
                size_df[
                    "cached_exact_mean_seconds_per_condition"
                ].quantile(
                    0.25
                ),

            "q75_cached_exact_milliseconds_per_condition":
                1000.0
                *
                size_df[
                    "cached_exact_mean_seconds_per_condition"
                ].quantile(
                    0.75
                ),

            "median_greedy_milliseconds_per_condition":
                1000.0
                *
                size_df[
                    "greedy_mean_seconds_per_condition"
                ].median(),

            "q25_greedy_milliseconds_per_condition":
                1000.0
                *
                size_df[
                    "greedy_mean_seconds_per_condition"
                ].quantile(
                    0.25
                ),

            "q75_greedy_milliseconds_per_condition":
                1000.0
                *
                size_df[
                    "greedy_mean_seconds_per_condition"
                ].quantile(
                    0.75
                ),

            "median_cached_exact_to_greedy_ratio":
                size_df[
                    "cached_exact_to_greedy_condition_ratio"
                ].median(),

            "median_total_exact_to_greedy_total_ratio":
                size_df[
                    "total_exact_to_greedy_total_ratio"
                ].median(),
        }
    )


runtime_size_summary_df = pd.DataFrame(
    runtime_size_summary_rows
)


runtime_size_summary_df.to_csv(
    RUNTIME_SIZE_SUMMARY_PATH,
    index=False,
)


# --------------------------------------------------
# Runtime scaling model functions
# --------------------------------------------------

def fit_linear_model(
    x_values,
    y_values,
):
    """
    Fit y = intercept + slope*x and return the
    coefficient of determination.
    """

    x_values = np.asarray(
        x_values,
        dtype=float,
    )

    y_values = np.asarray(
        y_values,
        dtype=float,
    )


    slope, intercept = np.polyfit(
        x_values,
        y_values,
        deg=1,
    )


    predicted_values = (
        intercept
        +
        slope
        * x_values
    )


    residual_sum_of_squares = np.sum(
        (
            y_values
            -
            predicted_values
        )
        ** 2
    )


    total_sum_of_squares = np.sum(
        (
            y_values
            -
            y_values.mean()
        )
        ** 2
    )


    if total_sum_of_squares > 0.0:

        r_squared = (
            1.0
            -
            residual_sum_of_squares
            /
            total_sum_of_squares
        )

    else:

        r_squared = np.nan


    return {
        "slope":
            slope,

        "intercept":
            intercept,

        "r_squared":
            r_squared,
    }



def fit_exponential_runtime_model(
    dataframe,
    runtime_column,
):
    """
    Fit log2(runtime) = intercept + slope*M.
    """

    fit_result = fit_linear_model(
        x_values=
            dataframe[
                "n_items"
            ].to_numpy(
                dtype=float
            ),

        y_values=
            np.log2(
                dataframe[
                    runtime_column
                ].to_numpy(
                    dtype=float
                )
            ),
    )


    fit_result[
        "factor_per_added_item"
    ] = (
        2.0
        **
        fit_result[
            "slope"
        ]
    )


    fit_result[
        "factor_per_two_added_items"
    ] = (
        2.0
        **
        (
            2.0
            *
            fit_result[
                "slope"
            ]
        )
    )


    fit_result[
        "power_law_exponent"
    ] = np.nan


    fit_result[
        "factor_when_candidate_size_doubles"
    ] = np.nan


    return fit_result



def fit_power_runtime_model(
    dataframe,
    runtime_column,
):
    """
    Fit log2(runtime) = intercept + exponent*log2(M).
    """

    fit_result = fit_linear_model(
        x_values=
            np.log2(
                dataframe[
                    "n_items"
                ].to_numpy(
                    dtype=float
                )
            ),

        y_values=
            np.log2(
                dataframe[
                    runtime_column
                ].to_numpy(
                    dtype=float
                )
            ),
    )


    fit_result[
        "power_law_exponent"
    ] = (
        fit_result[
            "slope"
        ]
    )


    fit_result[
        "factor_when_candidate_size_doubles"
    ] = (
        2.0
        **
        fit_result[
            "slope"
        ]
    )


    fit_result[
        "factor_per_added_item"
    ] = np.nan


    fit_result[
        "factor_per_two_added_items"
    ] = np.nan


    return fit_result


# --------------------------------------------------
# Model specifications
# --------------------------------------------------

RUNTIME_MODEL_SPECIFICATIONS = [
    {
        "runtime_component":
            "subset_precomputation",

        "runtime_column":
            "subset_precomputation_seconds",

        "model_type":
            "exponential_log2_runtime_on_candidate_count",
    },
    {
        "runtime_component":
            "total_exact_per_instance",

        "runtime_column":
            "total_exact_seconds",

        "model_type":
            "exponential_log2_runtime_on_candidate_count",
    },
    {
        "runtime_component":
            "cached_exact_per_condition",

        "runtime_column":
            "cached_exact_mean_seconds_per_condition",

        "model_type":
            "exponential_log2_runtime_on_candidate_count",
    },
    {
        "runtime_component":
            "greedy_per_condition",

        "runtime_column":
            "greedy_mean_seconds_per_condition",

        "model_type":
            "power_law_log2_runtime_on_log2_candidate_count",
    },
]


# --------------------------------------------------
# Observed scaling fits
# --------------------------------------------------

observed_runtime_fits = {}


for specification in RUNTIME_MODEL_SPECIFICATIONS:

    runtime_component = (
        specification[
            "runtime_component"
        ]
    )

    runtime_column = (
        specification[
            "runtime_column"
        ]
    )

    model_type = (
        specification[
            "model_type"
        ]
    )


    if model_type.startswith(
        "exponential"
    ):

        fit_result = (
            fit_exponential_runtime_model(
                runtime_matched_group_df,
                runtime_column,
            )
        )

    else:

        fit_result = (
            fit_power_runtime_model(
                runtime_matched_group_df,
                runtime_column,
            )
        )


    observed_runtime_fits[
        runtime_component
    ] = fit_result


# --------------------------------------------------
# Stratified matched-group bootstrap for model slopes
# --------------------------------------------------

runtime_rng = np.random.default_rng(
    RUNTIME_BOOTSTRAP_SEED
)


runtime_size_strata = {
    int(
        n_items
    ): (
        size_df
        .sort_values(
            "matched_group_id"
        )
        .reset_index(
            drop=True
        )
    )

    for n_items, size_df in (
        runtime_matched_group_df
        .groupby(
            "n_items",
            sort=True,
        )
    )
}


runtime_bootstrap_rows = []


for bootstrap_index in range(
    RUNTIME_BOOTSTRAP_REPLICATES
):

    resampled_size_frames = []


    for n_items, size_df in (
        runtime_size_strata.items()
    ):

        number_of_groups = len(
            size_df
        )


        sampled_indices = (
            runtime_rng.integers(
                low=0,
                high=number_of_groups,
                size=number_of_groups,
            )
        )


        resampled_size_frames.append(
            size_df.iloc[
                sampled_indices
            ]
        )


    resampled_runtime_df = pd.concat(
        resampled_size_frames,
        ignore_index=True,
    )


    for specification in (
        RUNTIME_MODEL_SPECIFICATIONS
    ):

        runtime_component = (
            specification[
                "runtime_component"
            ]
        )

        runtime_column = (
            specification[
                "runtime_column"
            ]
        )

        model_type = (
            specification[
                "model_type"
            ]
        )


        if model_type.startswith(
            "exponential"
        ):

            bootstrap_fit = (
                fit_exponential_runtime_model(
                    resampled_runtime_df,
                    runtime_column,
                )
            )

        else:

            bootstrap_fit = (
                fit_power_runtime_model(
                    resampled_runtime_df,
                    runtime_column,
                )
            )


        runtime_bootstrap_rows.append(
            {
                "bootstrap_replicate":
                    bootstrap_index,

                "runtime_component":
                    runtime_component,

                "model_type":
                    model_type,

                "slope":
                    bootstrap_fit[
                        "slope"
                    ],

                "factor_per_added_item":
                    bootstrap_fit[
                        "factor_per_added_item"
                    ],

                "factor_per_two_added_items":
                    bootstrap_fit[
                        "factor_per_two_added_items"
                    ],

                "power_law_exponent":
                    bootstrap_fit[
                        "power_law_exponent"
                    ],

                "factor_when_candidate_size_doubles":
                    bootstrap_fit[
                        "factor_when_candidate_size_doubles"
                    ],
            }
        )


runtime_bootstrap_df = pd.DataFrame(
    runtime_bootstrap_rows
)


runtime_bootstrap_df.to_csv(
    RUNTIME_BOOTSTRAP_REPLICATES_PATH,
    index=False,
)


# --------------------------------------------------
# Bootstrap confidence intervals for scaling fits
# --------------------------------------------------

runtime_model_summary_rows = []


for specification in RUNTIME_MODEL_SPECIFICATIONS:

    runtime_component = (
        specification[
            "runtime_component"
        ]
    )

    runtime_column = (
        specification[
            "runtime_column"
        ]
    )

    model_type = (
        specification[
            "model_type"
        ]
    )


    observed_fit = (
        observed_runtime_fits[
            runtime_component
        ]
    )


    component_bootstrap_df = (
        runtime_bootstrap_df[
            runtime_bootstrap_df[
                "runtime_component"
            ]
            == runtime_component
        ]
    )


    if model_type.startswith(
        "exponential"
    ):

        reported_parameter = (
            "factor_per_added_item"
        )

        parameter_estimate = (
            observed_fit[
                "factor_per_added_item"
            ]
        )

        parameter_values = (
            component_bootstrap_df[
                "factor_per_added_item"
            ]
            .dropna()
            .to_numpy(
                dtype=float
            )
        )


        secondary_parameter = (
            "factor_per_two_added_items"
        )

        secondary_estimate = (
            observed_fit[
                "factor_per_two_added_items"
            ]
        )

        secondary_values = (
            component_bootstrap_df[
                "factor_per_two_added_items"
            ]
            .dropna()
            .to_numpy(
                dtype=float
            )
        )

    else:

        reported_parameter = (
            "power_law_exponent"
        )

        parameter_estimate = (
            observed_fit[
                "power_law_exponent"
            ]
        )

        parameter_values = (
            component_bootstrap_df[
                "power_law_exponent"
            ]
            .dropna()
            .to_numpy(
                dtype=float
            )
        )


        secondary_parameter = (
            "factor_when_candidate_size_doubles"
        )

        secondary_estimate = (
            observed_fit[
                "factor_when_candidate_size_doubles"
            ]
        )

        secondary_values = (
            component_bootstrap_df[
                "factor_when_candidate_size_doubles"
            ]
            .dropna()
            .to_numpy(
                dtype=float
            )
        )


    runtime_model_summary_rows.append(
        {
            "runtime_component":
                runtime_component,

            "runtime_column":
                runtime_column,

            "model_type":
                model_type,

            "observed_slope":
                observed_fit[
                    "slope"
                ],

            "observed_intercept":
                observed_fit[
                    "intercept"
                ],

            "observed_r_squared":
                observed_fit[
                    "r_squared"
                ],

            "reported_parameter":
                reported_parameter,

            "parameter_estimate":
                parameter_estimate,

            "parameter_ci_95_lower":
                np.quantile(
                    parameter_values,
                    0.025,
                ),

            "parameter_ci_95_upper":
                np.quantile(
                    parameter_values,
                    0.975,
                ),

            "secondary_parameter":
                secondary_parameter,

            "secondary_estimate":
                secondary_estimate,

            "secondary_ci_95_lower":
                np.quantile(
                    secondary_values,
                    0.025,
                ),

            "secondary_ci_95_upper":
                np.quantile(
                    secondary_values,
                    0.975,
                ),

            "matched_groups":
                500,

            "candidate_size_range":
                "8-16",

            "candidate_size_strata":
                5,

            "bootstrap_replicates":
                RUNTIME_BOOTSTRAP_REPLICATES,

            "resampling_unit":
                "matched_size_replicate_group",

            "resampling_design":
                "stratified_by_candidate_set_size",

            "interpretation_scope":
                (
                    "empirical implementation scaling "
                    "on current hardware and benchmark"
                ),
        }
    )


runtime_model_summary_df = pd.DataFrame(
    runtime_model_summary_rows
)


runtime_model_summary_df.to_csv(
    RUNTIME_MODEL_FIT_PATH,
    index=False,
)


# --------------------------------------------------
# Readable scaling-model display
# --------------------------------------------------

runtime_model_display_df = (
    runtime_model_summary_df.copy()
)


runtime_model_display_df[
    "parameter_estimate_and_ci"
] = (
    runtime_model_display_df[
        "parameter_estimate"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    runtime_model_display_df[
        "parameter_ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    runtime_model_display_df[
        "parameter_ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


runtime_model_display_df[
    "secondary_estimate_and_ci"
] = (
    runtime_model_display_df[
        "secondary_estimate"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    runtime_model_display_df[
        "secondary_ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    runtime_model_display_df[
        "secondary_ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


# --------------------------------------------------
# Print results
# --------------------------------------------------

print(
    "======================================"
)

print(
    "CORRECTED V2 RUNTIME BY CANDIDATE-SET SIZE"
)

print(
    "======================================"
)


print(
    runtime_size_summary_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.6f}",
    )
)


print(
    "\n======================================"
)

print(
    "CORRECTED V2 EMPIRICAL RUNTIME-SCALING MODELS"
)

print(
    "======================================"
)


print(
    runtime_model_display_df[
        [
            "runtime_component",
            "model_type",
            "observed_r_squared",
            "reported_parameter",
            "parameter_estimate_and_ci",
            "secondary_parameter",
            "secondary_estimate_and_ci",
        ]
    ].to_string(
        index=False,
        float_format=lambda value:
            f"{value:.4f}",
    )
)


print(
    "\nInstance-level runtime data saved to:"
)

print(
    RUNTIME_INSTANCE_PATH
)


print(
    "\nMatched-group runtime data saved to:"
)

print(
    RUNTIME_MATCHED_GROUP_PATH
)


print(
    "\nRuntime-by-size summary saved to:"
)

print(
    RUNTIME_SIZE_SUMMARY_PATH
)


print(
    "\nRuntime model fits saved to:"
)

print(
    RUNTIME_MODEL_FIT_PATH
)


print(
    "\nRuntime bootstrap replicates saved to:"
)

print(
    RUNTIME_BOOTSTRAP_REPLICATES_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Large-scale corrected-v2 Greedy scalability benchmark
# --------------------------------------------------

from pathlib import Path
import ast
import inspect
import time

import numpy as np
import pandas as pd


# --------------------------------------------------
# Configuration
# --------------------------------------------------

GREEDY_SCALABILITY_REPLICATES = 100

GREEDY_SCALABILITY_VERSION = (
    "greedy_scalability_marginal_preserving_v2"
)


if "SCALABILITY_SIZES" not in globals():

    SCALABILITY_SIZES = [
        16,
        32,
        64,
        128,
        256,
    ]


if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


GREEDY_SCALABILITY_CHECKPOINT_PATH = (
    RESULTS_DIR
    / "greedy_scalability_v2_checkpoint.csv"
)


GREEDY_SCALABILITY_FINAL_PATH = (
    RESULTS_DIR
    / "greedy_scalability_v2.csv"
)


GREEDY_SCALABILITY_SIZE_SUMMARY_PATH = (
    TABLES_DIR
    / "greedy_scalability_v2_runtime_by_candidate_size.csv"
)


CORRECTED_EXACT_REFERENCE_PATH = (
    RESULTS_DIR
    / "exact_vs_greedy_main_benchmark_v2_marginal_preserving.csv"
)


CONDITIONS_PER_INSTANCE = (
    len(TAU_LEVELS)
    *
    len(WEIGHT_PROFILES)
)


MATCHED_GROUPS_EXPECTED = (
    len(SCALABILITY_SIZES)
    *
    GREEDY_SCALABILITY_REPLICATES
)


INSTANCES_EXPECTED = (
    MATCHED_GROUPS_EXPECTED
    *
    len(INSTANCE_REGIMES)
)


ROWS_EXPECTED = (
    INSTANCES_EXPECTED
    *
    CONDITIONS_PER_INSTANCE
)


# --------------------------------------------------
# Result-extraction helpers
# --------------------------------------------------

def result_to_mapping(result):
    """
    Convert common solver-result representations to a
    dictionary without assuming one specific result class.
    """

    if isinstance(result, dict):

        return result


    if hasattr(result, "_asdict"):

        return dict(
            result._asdict()
        )


    if hasattr(result, "__dict__"):

        return dict(
            vars(result)
        )


    return {}



def first_mapping_value(
    mapping,
    candidate_keys,
):
    """
    Return the first non-None value found under one of
    the supplied candidate keys.
    """

    for key in candidate_keys:

        if (
            key in mapping
            and
            mapping[key] is not None
        ):

            return mapping[key]

    return None



def normalize_selected_values(
    selected_value,
):
    """
    Convert a solver's selected-subset representation
    into an ordered list of item identifiers.
    """

    if selected_value is None:

        return []


    if isinstance(
        selected_value,
        str,
    ):

        text = selected_value.strip()


        if text in {
            "",
            "[]",
            "()",
            "{}",
            "None",
            "nan",
        }:

            return []


        try:

            parsed_value = ast.literal_eval(
                text
            )

        except (
            ValueError,
            SyntaxError,
        ):

            parsed_value = [
                token.strip()
                for token in text.split(",")
                if token.strip()
            ]


        selected_value = parsed_value


    if not isinstance(
        selected_value,
        (
            list,
            tuple,
            set,
            np.ndarray,
        ),
    ):

        selected_value = [
            selected_value
        ]


    selected_ids = []


    for selected_item in selected_value:

        if hasattr(
            selected_item,
            "item_id",
        ):

            selected_id = (
                selected_item.item_id
            )

        elif hasattr(
            selected_item,
            "id",
        ):

            selected_id = (
                selected_item.id
            )

        else:

            selected_id = str(
                selected_item
            )


        selected_ids.append(
            str(
                selected_id
            )
        )


    # Preserve the solver ordering while removing
    # accidental duplicate identifiers.
    return list(
        dict.fromkeys(
            selected_ids
        )
    )



def extract_selected_ids_from_result(
    solver_result,
):
    """
    Extract selected item identifiers from a Greedy
    solver result.
    """

    result_mapping = result_to_mapping(
        solver_result
    )


    selected_value = first_mapping_value(
        result_mapping,
        [
            "selected_ids",
            "selected_item_ids",
            "selected_items",
            "selected_subset",
            "selected",
            "subset",
            "items",
        ],
    )


    if selected_value is None:

        if isinstance(
            solver_result,
            (
                list,
                tuple,
                set,
                np.ndarray,
            ),
        ):

            selected_value = (
                solver_result
            )


    return normalize_selected_values(
        selected_value
    )


# --------------------------------------------------
# Flexible call to the current mst_core Greedy solver
# --------------------------------------------------

def call_current_greedy_solver(
    items,
    tau,
    weights,
):
    """
    Call mst_core.greedy_mst while supporting the
    parameter names used by common module versions.
    """

    solver_function = (
        mst_core.greedy_mst
    )


    parameter_values = {
        "items":
            items,

        "information_items":
            items,

        "candidate_items":
            items,

        "candidates":
            items,

        "tau":
            tau,

        "threshold":
            tau,

        "sufficiency_threshold":
            tau,

        "lambda_privacy":
            weights[
                "lambda_privacy"
            ],

        "lambda_p":
            weights[
                "lambda_privacy"
            ],

        "privacy_weight":
            weights[
                "lambda_privacy"
            ],

        "lambda_cognitive":
            weights[
                "lambda_cognitive"
            ],

        "lambda_c":
            weights[
                "lambda_cognitive"
            ],

        "cognitive_weight":
            weights[
                "lambda_cognitive"
            ],

        "lambda_cardinality":
            weights[
                "lambda_cardinality"
            ],

        "lambda_n":
            weights[
                "lambda_cardinality"
            ],

        "cardinality_weight":
            weights[
                "lambda_cardinality"
            ],
    }


    solver_signature = inspect.signature(
        solver_function
    )


    solver_kwargs = {}
    unresolved_required_parameters = []


    for (
        parameter_name,
        parameter,
    ) in solver_signature.parameters.items():

        if parameter.kind in {
            inspect.Parameter.VAR_POSITIONAL,
            inspect.Parameter.VAR_KEYWORD,
        }:

            continue


        if parameter_name in parameter_values:

            solver_kwargs[
                parameter_name
            ] = parameter_values[
                parameter_name
            ]


        elif (
            parameter.default
            is
            inspect.Parameter.empty
        ):

            unresolved_required_parameters.append(
                parameter_name
            )


    if not unresolved_required_parameters:

        return solver_function(
            **solver_kwargs
        )


    # Fallback for the original positional interface.
    try:

        return solver_function(
            items,
            tau,
            weights[
                "lambda_privacy"
            ],
            weights[
                "lambda_cognitive"
            ],
            weights[
                "lambda_cardinality"
            ],
        )


    except TypeError as positional_error:

        raise TypeError(
            "Unable to call mst_core.greedy_mst. "
            "Unresolved required parameters: "
            f"{unresolved_required_parameters}. "
            "Current signature: "
            f"{solver_signature}"
        ) from positional_error


# --------------------------------------------------
# Recompute solution metrics independently
# --------------------------------------------------

def evaluate_selected_ids(
    all_items,
    selected_ids,
    weights,
):
    """
    Independently evaluate a selected subset using the
    official product-aggregation benchmark equations.
    """

    item_by_id = {
        str(
            item.item_id
        ): item
        for item in all_items
    }


    unknown_selected_ids = [
        selected_id
        for selected_id in selected_ids
        if selected_id not in item_by_id
    ]


    if unknown_selected_ids:

        raise ValueError(
            "Greedy returned identifiers not present "
            "in the candidate set: "
            f"{unknown_selected_ids}"
        )


    selected_items = [
        item_by_id[
            selected_id
        ]
        for selected_id in selected_ids
    ]


    if selected_items:

        utility = (
            1.0
            -
            np.prod(
                [
                    1.0
                    -
                    float(
                        item.utility
                    )
                    for item in selected_items
                ]
            )
        )


        privacy = (
            1.0
            -
            np.prod(
                [
                    1.0
                    -
                    float(
                        item.privacy
                    )
                    for item in selected_items
                ]
            )
        )


        cognitive_cost = sum(
            float(
                item.cognitive_cost
            )
            for item in selected_items
        )


    else:

        utility = 0.0
        privacy = 0.0
        cognitive_cost = 0.0


    full_cognitive_cost = sum(
        float(
            item.cognitive_cost
        )
        for item in all_items
    )


    if full_cognitive_cost > 0.0:

        normalized_cognitive_cost = (
            cognitive_cost
            /
            full_cognitive_cost
        )

    else:

        normalized_cognitive_cost = 0.0


    cardinality_fraction = (
        len(
            selected_items
        )
        /
        len(
            all_items
        )
    )


    disclosure_cost = (
        weights[
            "lambda_privacy"
        ]
        *
        privacy
        +
        weights[
            "lambda_cognitive"
        ]
        *
        normalized_cognitive_cost
        +
        weights[
            "lambda_cardinality"
        ]
        *
        cardinality_fraction
    )


    return {
        "selected_count":
            len(
                selected_items
            ),

        "utility":
            float(
                utility
            ),

        "privacy":
            float(
                privacy
            ),

        "cognitive_cost":
            float(
                cognitive_cost
            ),

        "normalized_cognitive_cost":
            float(
                normalized_cognitive_cost
            ),

        "cardinality_fraction":
            float(
                cardinality_fraction
            ),

        "disclosure_cost":
            float(
                disclosure_cost
            ),
    }


# --------------------------------------------------
# Run all 16 conditions for one generated instance
# --------------------------------------------------

def run_greedy_scalability_instance(
    n_items,
    regime,
    matched_seed,
    matched_group_id,
    instance_id,
    replicate,
):
    """
    Run Greedy under all threshold and objective-weight
    conditions for one generated candidate set.
    """

    items = generate_mst_instance(
        n_items=n_items,
        regime=regime,
        seed=matched_seed,
    )


    full_set_utility = (
        1.0
        -
        np.prod(
            [
                1.0
                -
                float(
                    item.utility
                )
                for item in items
            ]
        )
    )


    instance_rows = []


    for tau in TAU_LEVELS:

        for (
            weight_profile,
            weights,
        ) in WEIGHT_PROFILES.items():

            solver_start = (
                time.perf_counter()
            )


            solver_result = (
                call_current_greedy_solver(
                    items=items,
                    tau=tau,
                    weights=weights,
                )
            )


            solver_runtime_seconds = (
                time.perf_counter()
                -
                solver_start
            )


            selected_ids = (
                extract_selected_ids_from_result(
                    solver_result
                )
            )


            evaluation = (
                evaluate_selected_ids(
                    all_items=items,
                    selected_ids=selected_ids,
                    weights=weights,
                )
            )


            if (
                evaluation[
                    "utility"
                ]
                +
                1e-12
                >= tau
            ):

                greedy_status = (
                    "FEASIBLE"
                )

            else:

                greedy_status = (
                    "INFEASIBLE"
                )


            instance_rows.append(
                {
                    "benchmark_version":
                        GREEDY_SCALABILITY_VERSION,

                    "generator_version":
                        "marginal_preserving_v2",

                    "regime_strength":
                        REGIME_STRENGTH,

                    "matched_group_id":
                        matched_group_id,

                    "instance_id":
                        instance_id,

                    "n_items":
                        n_items,

                    "replicate":
                        replicate,

                    "regime":
                        regime,

                    "seed":
                        matched_seed,

                    "matched_seed":
                        matched_seed,

                    "tau":
                        tau,

                    "weight_profile":
                        weight_profile,

                    "lambda_privacy":
                        weights[
                            "lambda_privacy"
                        ],

                    "lambda_cognitive":
                        weights[
                            "lambda_cognitive"
                        ],

                    "lambda_cardinality":
                        weights[
                            "lambda_cardinality"
                        ],

                    "full_set_utility":
                        float(
                            full_set_utility
                        ),

                    "greedy_status":
                        greedy_status,

                    "greedy_selected_ids":
                        repr(
                            tuple(
                                selected_ids
                            )
                        ),

                    "greedy_selected_count":
                        evaluation[
                            "selected_count"
                        ],

                    "greedy_utility":
                        evaluation[
                            "utility"
                        ],

                    "greedy_privacy":
                        evaluation[
                            "privacy"
                        ],

                    "greedy_cognitive_cost":
                        evaluation[
                            "cognitive_cost"
                        ],

                    "greedy_normalized_cognitive_cost":
                        evaluation[
                            "normalized_cognitive_cost"
                        ],

                    "greedy_cardinality_fraction":
                        evaluation[
                            "cardinality_fraction"
                        ],

                    "greedy_disclosure_cost":
                        evaluation[
                            "disclosure_cost"
                        ],

                    "greedy_runtime_seconds":
                        solver_runtime_seconds,
                }
            )


    return instance_rows


# --------------------------------------------------
# Resume from a complete checkpoint if present
# --------------------------------------------------

if GREEDY_SCALABILITY_FINAL_PATH.exists():

    loading_path = (
        GREEDY_SCALABILITY_FINAL_PATH
    )

    print(
        "Existing Greedy scalability final file found."
    )


elif GREEDY_SCALABILITY_CHECKPOINT_PATH.exists():

    loading_path = (
        GREEDY_SCALABILITY_CHECKPOINT_PATH
    )

    print(
        "Existing Greedy scalability checkpoint found."
    )


else:

    loading_path = None

    print(
        "No Greedy scalability checkpoint found."
    )

    print(
        "Starting a new scalability benchmark."
    )


if loading_path is not None:

    loaded_scalability_df = pd.read_csv(
        loading_path,
        low_memory=False,
    )


    loaded_instance_counts = (
        loaded_scalability_df
        .groupby(
            "instance_id"
        )
        .size()
    )


    complete_instance_ids = set(
        loaded_instance_counts[
            loaded_instance_counts
            ==
            CONDITIONS_PER_INSTANCE
        ].index
    )


    scalability_rows = (
        loaded_scalability_df[
            loaded_scalability_df[
                "instance_id"
            ].isin(
                complete_instance_ids
            )
        ]
        .to_dict(
            "records"
        )
    )


    completed_instance_ids = (
        complete_instance_ids
    )


    print(
        "Complete instances loaded:",
        len(
            completed_instance_ids
        )
    )


else:

    scalability_rows = []

    completed_instance_ids = set()


# --------------------------------------------------
# Benchmark execution
# --------------------------------------------------

print(
    "\nScalability sizes:",
    SCALABILITY_SIZES
)

print(
    "Expected matched groups:",
    MATCHED_GROUPS_EXPECTED
)

print(
    "Expected regime-specific instances:",
    INSTANCES_EXPECTED
)

print(
    "Expected condition rows:",
    ROWS_EXPECTED
)


scalability_session_start = (
    time.perf_counter()
)


for size_index, n_items in enumerate(
    SCALABILITY_SIZES
):

    print(
        "\nStarting scalability size:",
        n_items
    )


    for replicate in range(
        GREEDY_SCALABILITY_REPLICATES
    ):


        # Reuse corrected Exact-benchmark seeds at M=16
        # so the Greedy outputs can be cross-validated.
        if n_items == 16:

            matched_seed = (
                GLOBAL_SEED
                +
                50_000_000
                +
                EXACT_BENCHMARK_SIZES.index(
                    16
                )
                *
                1_000_000
                +
                replicate
            )


        else:

            matched_seed = (
                GLOBAL_SEED
                +
                70_000_000
                +
                size_index
                *
                1_000_000
                +
                replicate
            )


        matched_group_id = (
            f"SCAL_V2_M{n_items}"
            f"_R{replicate:03d}"
        )


        for regime in INSTANCE_REGIMES:

            instance_id = (
                f"{matched_group_id}"
                f"_{regime}"
            )


            if (
                instance_id
                in completed_instance_ids
            ):

                continue


            instance_rows = (
                run_greedy_scalability_instance(
                    n_items=n_items,
                    regime=regime,
                    matched_seed=matched_seed,
                    matched_group_id=
                        matched_group_id,
                    instance_id=instance_id,
                    replicate=replicate,
                )
            )


            scalability_rows.extend(
                instance_rows
            )


            completed_instance_ids.add(
                instance_id
            )


        # Save after every five matched groups.
        completed_group_number = (
            size_index
            *
            GREEDY_SCALABILITY_REPLICATES
            +
            replicate
            +
            1
        )


        if (
            completed_group_number % 5
            == 0
        ):

            checkpoint_df = pd.DataFrame(
                scalability_rows
            )


            checkpoint_df.to_csv(
                GREEDY_SCALABILITY_CHECKPOINT_PATH,
                index=False,
            )


            elapsed_seconds = (
                time.perf_counter()
                -
                scalability_session_start
            )


            print(
                f"Completed matched groups: "
                f"{completed_group_number}/"
                f"{MATCHED_GROUPS_EXPECTED}"
                f" | instances="
                f"{len(completed_instance_ids)}"
                f" | rows="
                f"{len(checkpoint_df)}"
                f" | elapsed="
                f"{elapsed_seconds / 60:.2f} min"
            )


# --------------------------------------------------
# Construct and save final dataset
# --------------------------------------------------

greedy_scalability_df = pd.DataFrame(
    scalability_rows
)


greedy_scalability_df = (
    greedy_scalability_df
    .drop_duplicates(
        subset=[
            "instance_id",
            "tau",
            "weight_profile",
        ],
        keep="last",
    )
    .sort_values(
        [
            "n_items",
            "replicate",
            "regime",
            "tau",
            "weight_profile",
        ]
    )
    .reset_index(
        drop=True
    )
)


greedy_scalability_df.to_csv(
    GREEDY_SCALABILITY_FINAL_PATH,
    index=False,
)


# --------------------------------------------------
# Structural and feasibility checks
# --------------------------------------------------

rows_per_instance = (
    greedy_scalability_df
    .groupby(
        "instance_id"
    )
    .size()
)


rows_per_matched_group = (
    greedy_scalability_df
    .groupby(
        "matched_group_id"
    )
    .size()
)


regimes_per_matched_group = (
    greedy_scalability_df
    .groupby(
        "matched_group_id"
    )[
        "regime"
    ]
    .nunique()
)


seeds_per_matched_group = (
    greedy_scalability_df
    .groupby(
        "matched_group_id"
    )[
        "matched_seed"
    ]
    .nunique()
)


duplicate_condition_rows = int(
    greedy_scalability_df
    .duplicated(
        subset=[
            "instance_id",
            "tau",
            "weight_profile",
        ],
        keep=False,
    )
    .sum()
)


threshold_failures = int(
    (
        (
            greedy_scalability_df[
                "greedy_status"
            ]
            == "FEASIBLE"
        )
        &
        (
            greedy_scalability_df[
                "greedy_utility"
            ]
            +
            1e-12
            <
            greedy_scalability_df[
                "tau"
            ]
        )
    ).sum()
)


false_infeasible_rows = int(
    (
        (
            greedy_scalability_df[
                "greedy_status"
            ]
            == "INFEASIBLE"
        )
        &
        (
            greedy_scalability_df[
                "full_set_utility"
            ]
            +
            1e-12
            >=
            greedy_scalability_df[
                "tau"
            ]
        )
    ).sum()
)


assert len(
    greedy_scalability_df
) == ROWS_EXPECTED


assert (
    greedy_scalability_df[
        "instance_id"
    ]
    .nunique()
    ==
    INSTANCES_EXPECTED
)


assert (
    greedy_scalability_df[
        "matched_group_id"
    ]
    .nunique()
    ==
    MATCHED_GROUPS_EXPECTED
)


assert (
    rows_per_instance
    ==
    CONDITIONS_PER_INSTANCE
).all()


assert (
    rows_per_matched_group
    ==
    (
        CONDITIONS_PER_INSTANCE
        *
        len(
            INSTANCE_REGIMES
        )
    )
).all()


assert (
    regimes_per_matched_group
    ==
    len(
        INSTANCE_REGIMES
    )
).all()


assert (
    seeds_per_matched_group
    == 1
).all()


assert duplicate_condition_rows == 0
assert threshold_failures == 0
assert false_infeasible_rows == 0


# --------------------------------------------------
# Cross-check M=16 against corrected Exact benchmark
# --------------------------------------------------

reference_v2_df = pd.read_csv(
    CORRECTED_EXACT_REFERENCE_PATH,
    low_memory=False,
)


reference_m16_df = (
    reference_v2_df[
        pd.to_numeric(
            reference_v2_df[
                "n_items"
            ],
            errors="coerce",
        )
        == 16
    ]
    .copy()
)


scalability_m16_df = (
    greedy_scalability_df[
        greedy_scalability_df[
            "n_items"
        ]
        == 16
    ]
    .copy()
)


comparison_keys = [
    "replicate",
    "regime",
    "tau",
    "weight_profile",
]


m16_comparison_df = (
    scalability_m16_df
    .merge(
        reference_m16_df[
            comparison_keys
            +
            [
                "greedy_status",
                "greedy_selected_count",
                "greedy_utility",
                "greedy_privacy",
                "greedy_normalized_cognitive_cost",
                "greedy_disclosure_cost",
            ]
        ],
        on=comparison_keys,
        how="inner",
        suffixes=(
            "_scalability",
            "_reference",
        ),
    )
)


assert len(
    m16_comparison_df
) == 4800


m16_status_mismatches = int(
    (
        m16_comparison_df[
            "greedy_status_scalability"
        ]
        !=
        m16_comparison_df[
            "greedy_status_reference"
        ]
    ).sum()
)


m16_count_mismatches = int(
    (
        pd.to_numeric(
            m16_comparison_df[
                "greedy_selected_count_scalability"
            ],
            errors="coerce",
        )
        !=
        pd.to_numeric(
            m16_comparison_df[
                "greedy_selected_count_reference"
            ],
            errors="coerce",
        )
    ).sum()
)


M16_NUMERICAL_TOLERANCE = 1e-10


m16_numerical_mismatch_rows = np.zeros(
    len(
        m16_comparison_df
    ),
    dtype=bool,
)


for metric_name in [
    "greedy_utility",
    "greedy_privacy",
    "greedy_normalized_cognitive_cost",
    "greedy_disclosure_cost",
]:

    scalability_values = pd.to_numeric(
        m16_comparison_df[
            f"{metric_name}_scalability"
        ],
        errors="coerce",
    )


    reference_values = pd.to_numeric(
        m16_comparison_df[
            f"{metric_name}_reference"
        ],
        errors="coerce",
    )


    m16_numerical_mismatch_rows |= (
        ~np.isclose(
            scalability_values,
            reference_values,
            atol=M16_NUMERICAL_TOLERANCE,
            rtol=M16_NUMERICAL_TOLERANCE,
            equal_nan=True,
        )
    )


m16_numerical_mismatches = int(
    m16_numerical_mismatch_rows.sum()
)


assert m16_status_mismatches == 0
assert m16_count_mismatches == 0
assert m16_numerical_mismatches == 0


# --------------------------------------------------
# Runtime summary by candidate-set size
# --------------------------------------------------

greedy_scalability_size_summary_df = (
    greedy_scalability_df
    .groupby(
        "n_items",
        as_index=False,
    )
    .agg(
        matched_groups=(
            "matched_group_id",
            "nunique",
        ),

        regime_specific_instances=(
            "instance_id",
            "nunique",
        ),

        condition_rows=(
            "greedy_runtime_seconds",
            "size",
        ),

        feasible_rows=(
            "greedy_status",
            lambda values:
                int(
                    (
                        values
                        ==
                        "FEASIBLE"
                    ).sum()
                ),
        ),

        median_runtime_seconds=(
            "greedy_runtime_seconds",
            "median",
        ),

        q25_runtime_seconds=(
            "greedy_runtime_seconds",
            lambda values:
                values.quantile(
                    0.25
                ),
        ),

        q75_runtime_seconds=(
            "greedy_runtime_seconds",
            lambda values:
                values.quantile(
                    0.75
                ),
        ),

        p90_runtime_seconds=(
            "greedy_runtime_seconds",
            lambda values:
                values.quantile(
                    0.90
                ),
        ),

        p95_runtime_seconds=(
            "greedy_runtime_seconds",
            lambda values:
                values.quantile(
                    0.95
                ),
        ),

        mean_selected_count=(
            "greedy_selected_count",
            "mean",
        ),
    )
)


for column_name in [
    "median_runtime_seconds",
    "q25_runtime_seconds",
    "q75_runtime_seconds",
    "p90_runtime_seconds",
    "p95_runtime_seconds",
]:

    millisecond_column = (
        column_name.replace(
            "_seconds",
            "_milliseconds",
        )
    )


    greedy_scalability_size_summary_df[
        millisecond_column
    ] = (
        1000.0
        *
        greedy_scalability_size_summary_df[
            column_name
        ]
    )


greedy_scalability_size_summary_df[
    "feasibility_percent"
] = (
    100.0
    *
    greedy_scalability_size_summary_df[
        "feasible_rows"
    ]
    /
    greedy_scalability_size_summary_df[
        "condition_rows"
    ]
)


greedy_scalability_size_summary_df.to_csv(
    GREEDY_SCALABILITY_SIZE_SUMMARY_PATH,
    index=False,
)


# --------------------------------------------------
# Final report
# --------------------------------------------------

scalability_session_seconds = (
    time.perf_counter()
    -
    scalability_session_start
)


print(
    "\n======================================"
)

print(
    "GREEDY SCALABILITY BENCHMARK COMPLETED"
)

print(
    "======================================"
)


print(
    "Benchmark version:",
    GREEDY_SCALABILITY_VERSION
)

print(
    "Candidate-set sizes:",
    SCALABILITY_SIZES
)

print(
    "Matched groups:",
    greedy_scalability_df[
        "matched_group_id"
    ].nunique()
)

print(
    "Regime-specific instances:",
    greedy_scalability_df[
        "instance_id"
    ].nunique()
)

print(
    "Condition rows:",
    len(
        greedy_scalability_df
    )
)

print(
    "Duplicate condition rows:",
    duplicate_condition_rows
)

print(
    "Threshold failures:",
    threshold_failures
)

print(
    "False infeasible rows:",
    false_infeasible_rows
)


print(
    "\nM=16 cross-validation against Exact benchmark:"
)

print(
    "Compared rows:",
    len(
        m16_comparison_df
    )
)

print(
    "Status mismatches:",
    m16_status_mismatches
)

print(
    "Selected-count mismatches:",
    m16_count_mismatches
)

print(
    "Numerical metric mismatches:",
    m16_numerical_mismatches
)


print(
    "\nRuntime by candidate-set size:"
)


print(
    greedy_scalability_size_summary_df[
        [
            "n_items",
            "matched_groups",
            "condition_rows",
            "feasibility_percent",
            "median_runtime_milliseconds",
            "q25_runtime_milliseconds",
            "q75_runtime_milliseconds",
            "p90_runtime_milliseconds",
            "p95_runtime_milliseconds",
            "mean_selected_count",
        ]
    ].to_string(
        index=False,
        float_format=lambda value:
            f"{value:.6f}",
    )
)


print(
    "\nExecution-session time:",
    f"{scalability_session_seconds / 60:.2f} minutes"
)


print(
    "\nFinal scalability dataset saved to:"
)

print(
    GREEDY_SCALABILITY_FINAL_PATH
)


print(
    "\nRuntime summary saved to:"
)

print(
    GREEDY_SCALABILITY_SIZE_SUMMARY_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Large-scale Greedy empirical scaling model
# using matched-group bootstrap resampling
# --------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd


# --------------------------------------------------
# Configuration
# --------------------------------------------------

GREEDY_LARGE_SCALE_BOOTSTRAP_REPLICATES = 5000

GREEDY_LARGE_SCALE_BOOTSTRAP_SEED = (
    globals().get(
        "GLOBAL_SEED",
        2026,
    )
    + 113
)


if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


GREEDY_SCALABILITY_PATH = (
    RESULTS_DIR
    / "greedy_scalability_v2.csv"
)


GREEDY_GROUP_RUNTIME_PATH = (
    RESULTS_DIR
    / "greedy_scalability_v2_matched_group_runtime.csv"
)


GREEDY_SIZE_RUNTIME_PATH = (
    TABLES_DIR
    / "greedy_scalability_v2_matched_group_runtime_by_size.csv"
)


GREEDY_POWER_MODEL_PATH = (
    TABLES_DIR
    / "greedy_scalability_v2_power_law_model.csv"
)


GREEDY_LOCAL_RATIO_PATH = (
    TABLES_DIR
    / "greedy_scalability_v2_local_doubling_ratios.csv"
)


GREEDY_MODEL_REPLICATES_PATH = (
    RESULTS_DIR
    / "greedy_scalability_v2_power_law_bootstrap_replicates.csv"
)


EXPECTED_SCALABILITY_SIZES = [
    16,
    32,
    64,
    128,
    256,
]


# --------------------------------------------------
# Load benchmark
# --------------------------------------------------

greedy_large_scale_df = pd.read_csv(
    GREEDY_SCALABILITY_PATH,
    low_memory=False,
)


for column_name in [
    "n_items",
    "greedy_runtime_seconds",
    "greedy_selected_count",
]:

    greedy_large_scale_df[
        column_name
    ] = pd.to_numeric(
        greedy_large_scale_df[
            column_name
        ],
        errors="coerce",
    )


required_columns = [
    "matched_group_id",
    "instance_id",
    "n_items",
    "regime",
    "tau",
    "weight_profile",
    "greedy_status",
    "greedy_runtime_seconds",
    "greedy_selected_count",
]


missing_columns = [
    column_name
    for column_name in required_columns
    if column_name
    not in greedy_large_scale_df.columns
]


if missing_columns:

    raise KeyError(
        "Missing required columns: "
        + ", ".join(
            missing_columns
        )
    )


# --------------------------------------------------
# Integrity checks
# --------------------------------------------------

assert (
    sorted(
        greedy_large_scale_df[
            "n_items"
        ]
        .dropna()
        .unique()
        .astype(int)
        .tolist()
    )
    ==
    EXPECTED_SCALABILITY_SIZES
)


assert len(
    greedy_large_scale_df
) == 24000


assert (
    greedy_large_scale_df[
        "matched_group_id"
    ]
    .nunique()
    == 500
)


assert (
    greedy_large_scale_df[
        "instance_id"
    ]
    .nunique()
    == 1500
)


assert (
    greedy_large_scale_df[
        "greedy_runtime_seconds"
    ]
    > 0.0
).all()


assert (
    greedy_large_scale_df[
        "greedy_status"
    ]
    == "FEASIBLE"
).all()


rows_per_group = (
    greedy_large_scale_df
    .groupby(
        "matched_group_id"
    )
    .size()
)


assert (
    rows_per_group
    == 48
).all()


groups_per_size = (
    greedy_large_scale_df
    .groupby(
        "n_items"
    )[
        "matched_group_id"
    ]
    .nunique()
)


assert (
    groups_per_size
    == 100
).all()


# --------------------------------------------------
# One independent runtime row per matched group
#
# Each matched group contains:
# 3 regimes × 4 thresholds × 4 weight profiles
# = 48 Greedy conditions.
# --------------------------------------------------

greedy_group_runtime_df = (
    greedy_large_scale_df
    .groupby(
        [
            "matched_group_id",
            "n_items",
        ],
        as_index=False,
    )
    .agg(
        condition_rows=(
            "greedy_runtime_seconds",
            "size",
        ),

        regime_specific_instances=(
            "instance_id",
            "nunique",
        ),

        median_runtime_seconds=(
            "greedy_runtime_seconds",
            "median",
        ),

        mean_runtime_seconds=(
            "greedy_runtime_seconds",
            "mean",
        ),

        p90_runtime_seconds=(
            "greedy_runtime_seconds",
            lambda values:
                values.quantile(
                    0.90
                ),
        ),

        p95_runtime_seconds=(
            "greedy_runtime_seconds",
            lambda values:
                values.quantile(
                    0.95
                ),
        ),

        mean_selected_count=(
            "greedy_selected_count",
            "mean",
        ),

        median_selected_count=(
            "greedy_selected_count",
            "median",
        ),

        maximum_selected_count=(
            "greedy_selected_count",
            "max",
        ),
    )
)


assert len(
    greedy_group_runtime_df
) == 500


assert (
    greedy_group_runtime_df[
        "condition_rows"
    ]
    == 48
).all()


assert (
    greedy_group_runtime_df[
        "regime_specific_instances"
    ]
    == 3
).all()


greedy_group_runtime_df.to_csv(
    GREEDY_GROUP_RUNTIME_PATH,
    index=False,
)


# --------------------------------------------------
# Matched-group summaries by candidate-set size
# --------------------------------------------------

greedy_size_summary_rows = []


for n_items in EXPECTED_SCALABILITY_SIZES:

    current_size_df = (
        greedy_group_runtime_df[
            greedy_group_runtime_df[
                "n_items"
            ]
            == n_items
        ]
    )


    assert len(
        current_size_df
    ) == 100


    greedy_size_summary_rows.append(
        {
            "n_items":
                n_items,

            "matched_groups":
                len(
                    current_size_df
                ),

            "median_group_median_runtime_milliseconds":
                1000.0
                *
                current_size_df[
                    "median_runtime_seconds"
                ].median(),

            "q25_group_median_runtime_milliseconds":
                1000.0
                *
                current_size_df[
                    "median_runtime_seconds"
                ].quantile(
                    0.25
                ),

            "q75_group_median_runtime_milliseconds":
                1000.0
                *
                current_size_df[
                    "median_runtime_seconds"
                ].quantile(
                    0.75
                ),

            "median_group_mean_runtime_milliseconds":
                1000.0
                *
                current_size_df[
                    "mean_runtime_seconds"
                ].median(),

            "median_group_p95_runtime_milliseconds":
                1000.0
                *
                current_size_df[
                    "p95_runtime_seconds"
                ].median(),

            "mean_selected_count":
                current_size_df[
                    "mean_selected_count"
                ].mean(),

            "q25_group_mean_selected_count":
                current_size_df[
                    "mean_selected_count"
                ].quantile(
                    0.25
                ),

            "q75_group_mean_selected_count":
                current_size_df[
                    "mean_selected_count"
                ].quantile(
                    0.75
                ),

            "median_maximum_selected_count":
                current_size_df[
                    "maximum_selected_count"
                ].median(),
        }
    )


greedy_size_summary_df = pd.DataFrame(
    greedy_size_summary_rows
)


greedy_size_summary_df.to_csv(
    GREEDY_SIZE_RUNTIME_PATH,
    index=False,
)


# --------------------------------------------------
# Regression helper
# --------------------------------------------------

def fit_linear_regression(
    x_values,
    y_values,
):
    """
    Fit y = intercept + slope*x.
    """

    x_values = np.asarray(
        x_values,
        dtype=float,
    )

    y_values = np.asarray(
        y_values,
        dtype=float,
    )


    slope, intercept = np.polyfit(
        x_values,
        y_values,
        deg=1,
    )


    predicted_values = (
        intercept
        +
        slope
        *
        x_values
    )


    residual_sum_of_squares = np.sum(
        (
            y_values
            -
            predicted_values
        )
        ** 2
    )


    total_sum_of_squares = np.sum(
        (
            y_values
            -
            y_values.mean()
        )
        ** 2
    )


    if total_sum_of_squares > 0.0:

        r_squared = (
            1.0
            -
            residual_sum_of_squares
            /
            total_sum_of_squares
        )

    else:

        r_squared = np.nan


    return {
        "slope":
            float(
                slope
            ),

        "intercept":
            float(
                intercept
            ),

        "r_squared":
            float(
                r_squared
            ),
    }


# --------------------------------------------------
# Observed size-level values
# --------------------------------------------------

observed_sizes = (
    greedy_size_summary_df[
        "n_items"
    ]
    .to_numpy(
        dtype=float
    )
)


observed_runtime_seconds = (
    greedy_size_summary_df[
        "median_group_median_runtime_milliseconds"
    ]
    .to_numpy(
        dtype=float
    )
    /
    1000.0
)


observed_selected_counts = (
    greedy_size_summary_df[
        "mean_selected_count"
    ]
    .to_numpy(
        dtype=float
    )
)


# --------------------------------------------------
# Observed runtime power-law fit
#
# log2(runtime) = intercept + exponent*log2(M)
# --------------------------------------------------

observed_runtime_fit = fit_linear_regression(
    x_values=
        np.log2(
            observed_sizes
        ),

    y_values=
        np.log2(
            observed_runtime_seconds
        ),
)


observed_runtime_exponent = (
    observed_runtime_fit[
        "slope"
    ]
)


observed_runtime_doubling_factor = (
    2.0
    **
    observed_runtime_exponent
)


# --------------------------------------------------
# Selected-count fit
#
# count = intercept + slope*log2(M)
#
# The slope is the average selected-item increase
# whenever candidate-set size doubles.
# --------------------------------------------------

observed_count_fit = fit_linear_regression(
    x_values=
        np.log2(
            observed_sizes
        ),

    y_values=
        observed_selected_counts,
)


observed_selected_count_increase_per_doubling = (
    observed_count_fit[
        "slope"
    ]
)


# --------------------------------------------------
# Observed local runtime ratios for each doubling
# --------------------------------------------------

observed_local_ratio_rows = []


for index in range(
    1,
    len(
        EXPECTED_SCALABILITY_SIZES
    ),
):

    smaller_size = (
        EXPECTED_SCALABILITY_SIZES[
            index - 1
        ]
    )

    larger_size = (
        EXPECTED_SCALABILITY_SIZES[
            index
        ]
    )


    smaller_runtime = (
        observed_runtime_seconds[
            index - 1
        ]
    )

    larger_runtime = (
        observed_runtime_seconds[
            index
        ]
    )


    observed_local_ratio_rows.append(
        {
            "smaller_size":
                smaller_size,

            "larger_size":
                larger_size,

            "doubling_contrast":
                (
                    f"M={larger_size}"
                    f" / "
                    f"M={smaller_size}"
                ),

            "smaller_median_runtime_milliseconds":
                1000.0
                *
                smaller_runtime,

            "larger_median_runtime_milliseconds":
                1000.0
                *
                larger_runtime,

            "observed_runtime_ratio":
                larger_runtime
                /
                smaller_runtime,
        }
    )


# --------------------------------------------------
# Prepare matched-group size strata
# --------------------------------------------------

group_arrays_by_size = {}


for n_items in EXPECTED_SCALABILITY_SIZES:

    current_size_df = (
        greedy_group_runtime_df[
            greedy_group_runtime_df[
                "n_items"
            ]
            == n_items
        ]
        .sort_values(
            "matched_group_id"
        )
        .reset_index(
            drop=True
        )
    )


    assert len(
        current_size_df
    ) == 100


    group_arrays_by_size[
        n_items
    ] = {
        "runtime":
            current_size_df[
                "median_runtime_seconds"
            ]
            .to_numpy(
                dtype=float
            ),

        "selected_count":
            current_size_df[
                "mean_selected_count"
            ]
            .to_numpy(
                dtype=float
            ),
    }


# --------------------------------------------------
# Stratified matched-group bootstrap
#
# Candidate sizes use independent matched groups, so
# each size stratum is resampled independently.
# --------------------------------------------------

greedy_scaling_rng = np.random.default_rng(
    GREEDY_LARGE_SCALE_BOOTSTRAP_SEED
)


bootstrap_rows = []


for bootstrap_index in range(
    GREEDY_LARGE_SCALE_BOOTSTRAP_REPLICATES
):

    bootstrap_size_runtime = []
    bootstrap_size_selected_count = []


    for n_items in EXPECTED_SCALABILITY_SIZES:

        runtime_array = (
            group_arrays_by_size[
                n_items
            ][
                "runtime"
            ]
        )


        selected_count_array = (
            group_arrays_by_size[
                n_items
            ][
                "selected_count"
            ]
        )


        number_of_groups = len(
            runtime_array
        )


        sampled_indices = (
            greedy_scaling_rng.integers(
                low=0,
                high=number_of_groups,
                size=number_of_groups,
            )
        )


        bootstrap_size_runtime.append(
            np.median(
                runtime_array[
                    sampled_indices
                ]
            )
        )


        bootstrap_size_selected_count.append(
            np.mean(
                selected_count_array[
                    sampled_indices
                ]
            )
        )


    bootstrap_size_runtime = np.asarray(
        bootstrap_size_runtime,
        dtype=float,
    )


    bootstrap_size_selected_count = np.asarray(
        bootstrap_size_selected_count,
        dtype=float,
    )


    runtime_fit = fit_linear_regression(
        x_values=
            np.log2(
                observed_sizes
            ),

        y_values=
            np.log2(
                bootstrap_size_runtime
            ),
    )


    runtime_exponent = (
        runtime_fit[
            "slope"
        ]
    )


    selected_count_fit = fit_linear_regression(
        x_values=
            np.log2(
                observed_sizes
            ),

        y_values=
            bootstrap_size_selected_count,
    )


    bootstrap_row = {
        "bootstrap_replicate":
            bootstrap_index,

        "runtime_power_law_exponent":
            runtime_exponent,

        "runtime_factor_per_doubling":
            2.0
            **
            runtime_exponent,

        "runtime_model_r_squared":
            runtime_fit[
                "r_squared"
            ],

        "selected_count_increase_per_doubling":
            selected_count_fit[
                "slope"
            ],

        "selected_count_model_r_squared":
            selected_count_fit[
                "r_squared"
            ],
    }


    for index in range(
        1,
        len(
            EXPECTED_SCALABILITY_SIZES
        ),
    ):

        smaller_size = (
            EXPECTED_SCALABILITY_SIZES[
                index - 1
            ]
        )

        larger_size = (
            EXPECTED_SCALABILITY_SIZES[
                index
            ]
        )


        ratio_column = (
            f"runtime_ratio_"
            f"M{larger_size}_to_M{smaller_size}"
        )


        bootstrap_row[
            ratio_column
        ] = (
            bootstrap_size_runtime[
                index
            ]
            /
            bootstrap_size_runtime[
                index - 1
            ]
        )


    bootstrap_rows.append(
        bootstrap_row
    )


greedy_scaling_bootstrap_df = pd.DataFrame(
    bootstrap_rows
)


greedy_scaling_bootstrap_df.to_csv(
    GREEDY_MODEL_REPLICATES_PATH,
    index=False,
)


# --------------------------------------------------
# Scaling-model confidence intervals
# --------------------------------------------------

def percentile_interval(
    values,
):
    """
    Return the 2.5th and 97.5th percentiles.
    """

    valid_values = (
        pd.Series(
            values
        )
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    return (
        float(
            np.quantile(
                valid_values,
                0.025,
            )
        ),
        float(
            np.quantile(
                valid_values,
                0.975,
            )
        ),
    )


runtime_exponent_ci = percentile_interval(
    greedy_scaling_bootstrap_df[
        "runtime_power_law_exponent"
    ]
)


runtime_doubling_ci = percentile_interval(
    greedy_scaling_bootstrap_df[
        "runtime_factor_per_doubling"
    ]
)


runtime_r_squared_ci = percentile_interval(
    greedy_scaling_bootstrap_df[
        "runtime_model_r_squared"
    ]
)


selected_count_slope_ci = percentile_interval(
    greedy_scaling_bootstrap_df[
        "selected_count_increase_per_doubling"
    ]
)


selected_count_r_squared_ci = percentile_interval(
    greedy_scaling_bootstrap_df[
        "selected_count_model_r_squared"
    ]
)


greedy_model_summary_df = pd.DataFrame(
    [
        {
            "model":
                "Greedy runtime power law",

            "equation":
                (
                    "log2(runtime_seconds) = "
                    "intercept + exponent*log2(n_items)"
                ),

            "parameter":
                "runtime_power_law_exponent",

            "estimate":
                observed_runtime_exponent,

            "ci_95_lower":
                runtime_exponent_ci[
                    0
                ],

            "ci_95_upper":
                runtime_exponent_ci[
                    1
                ],

            "secondary_parameter":
                "runtime_factor_per_doubling",

            "secondary_estimate":
                observed_runtime_doubling_factor,

            "secondary_ci_95_lower":
                runtime_doubling_ci[
                    0
                ],

            "secondary_ci_95_upper":
                runtime_doubling_ci[
                    1
                ],

            "observed_r_squared":
                observed_runtime_fit[
                    "r_squared"
                ],

            "r_squared_ci_95_lower":
                runtime_r_squared_ci[
                    0
                ],

            "r_squared_ci_95_upper":
                runtime_r_squared_ci[
                    1
                ],

            "candidate_size_range":
                "16-256",

            "matched_groups":
                500,

            "bootstrap_replicates":
                GREEDY_LARGE_SCALE_BOOTSTRAP_REPLICATES,

            "resampling_design":
                (
                    "independent matched-group "
                    "resampling within candidate size"
                ),

            "interpretation_scope":
                (
                    "empirical implementation scaling "
                    "for the current benchmark and hardware"
                ),
        },
        {
            "model":
                "Greedy selected-count growth",

            "equation":
                (
                    "mean_selected_count = "
                    "intercept + slope*log2(n_items)"
                ),

            "parameter":
                "selected_count_increase_per_doubling",

            "estimate":
                observed_selected_count_increase_per_doubling,

            "ci_95_lower":
                selected_count_slope_ci[
                    0
                ],

            "ci_95_upper":
                selected_count_slope_ci[
                    1
                ],

            "secondary_parameter":
                np.nan,

            "secondary_estimate":
                np.nan,

            "secondary_ci_95_lower":
                np.nan,

            "secondary_ci_95_upper":
                np.nan,

            "observed_r_squared":
                observed_count_fit[
                    "r_squared"
                ],

            "r_squared_ci_95_lower":
                selected_count_r_squared_ci[
                    0
                ],

            "r_squared_ci_95_upper":
                selected_count_r_squared_ci[
                    1
                ],

            "candidate_size_range":
                "16-256",

            "matched_groups":
                500,

            "bootstrap_replicates":
                GREEDY_LARGE_SCALE_BOOTSTRAP_REPLICATES,

            "resampling_design":
                (
                    "independent matched-group "
                    "resampling within candidate size"
                ),

            "interpretation_scope":
                (
                    "empirical selected-set growth "
                    "for the current benchmark"
                ),
        },
    ]
)


greedy_model_summary_df.to_csv(
    GREEDY_POWER_MODEL_PATH,
    index=False,
)


# --------------------------------------------------
# Local doubling-ratio confidence intervals
# --------------------------------------------------

local_ratio_rows = []


for observed_row in observed_local_ratio_rows:

    smaller_size = (
        observed_row[
            "smaller_size"
        ]
    )

    larger_size = (
        observed_row[
            "larger_size"
        ]
    )


    ratio_column = (
        f"runtime_ratio_"
        f"M{larger_size}_to_M{smaller_size}"
    )


    ratio_ci = percentile_interval(
        greedy_scaling_bootstrap_df[
            ratio_column
        ]
    )


    local_ratio_rows.append(
        {
            **observed_row,

            "ci_95_lower":
                ratio_ci[
                    0
                ],

            "ci_95_upper":
                ratio_ci[
                    1
                ],

            "bootstrap_replicates":
                GREEDY_LARGE_SCALE_BOOTSTRAP_REPLICATES,

            "resampling_design":
                (
                    "independent matched-group "
                    "resampling within candidate size"
                ),
        }
    )


greedy_local_ratio_df = pd.DataFrame(
    local_ratio_rows
)


greedy_local_ratio_df.to_csv(
    GREEDY_LOCAL_RATIO_PATH,
    index=False,
)


# --------------------------------------------------
# Display formatting
# --------------------------------------------------

model_display_df = (
    greedy_model_summary_df.copy()
)


model_display_df[
    "estimate_and_ci"
] = (
    model_display_df[
        "estimate"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    model_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    model_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


model_display_df[
    "secondary_estimate_and_ci"
] = np.where(
    model_display_df[
        "secondary_estimate"
    ].notna(),

    model_display_df[
        "secondary_estimate"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    model_display_df[
        "secondary_ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    model_display_df[
        "secondary_ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]",

    "",
)


greedy_local_ratio_df[
    "ratio_and_ci"
] = (
    greedy_local_ratio_df[
        "observed_runtime_ratio"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    greedy_local_ratio_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    greedy_local_ratio_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


# --------------------------------------------------
# Print results
# --------------------------------------------------

print(
    "======================================"
)

print(
    "GREEDY LARGE-SCALE MATCHED-GROUP RUNTIME"
)

print(
    "======================================"
)


print(
    greedy_size_summary_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.6f}",
    )
)


print(
    "\n======================================"
)

print(
    "GREEDY LARGE-SCALE SCALING MODELS"
)

print(
    "======================================"
)


print(
    model_display_df[
        [
            "model",
            "parameter",
            "estimate_and_ci",
            "secondary_parameter",
            "secondary_estimate_and_ci",
            "observed_r_squared",
        ]
    ].to_string(
        index=False,
        float_format=lambda value:
            f"{value:.4f}",
    )
)


print(
    "\n======================================"
)

print(
    "LOCAL RUNTIME MULTIPLIER PER SIZE DOUBLING"
)

print(
    "======================================"
)


print(
    greedy_local_ratio_df[
        [
            "doubling_contrast",
            "smaller_median_runtime_milliseconds",
            "larger_median_runtime_milliseconds",
            "ratio_and_ci",
        ]
    ].to_string(
        index=False,
        float_format=lambda value:
            f"{value:.6f}",
    )
)


print(
    "\nMatched-group runtime data saved to:"
)

print(
    GREEDY_GROUP_RUNTIME_PATH
)


print(
    "\nRuntime-by-size summary saved to:"
)

print(
    GREEDY_SIZE_RUNTIME_PATH
)


print(
    "\nPower-law model saved to:"
)

print(
    GREEDY_POWER_MODEL_PATH
)


print(
    "\nLocal doubling ratios saved to:"
)

print(
    GREEDY_LOCAL_RATIO_PATH
)


print(
    "\nBootstrap replicates saved to:"
)

print(
    GREEDY_MODEL_REPLICATES_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Large-scale Greedy solution compactness and
# disclosure profile with matched-group bootstrap CIs
# --------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd


# --------------------------------------------------
# Configuration
# --------------------------------------------------

COMPACTNESS_BOOTSTRAP_REPLICATES = 5000

COMPACTNESS_BOOTSTRAP_SEED = (
    globals().get(
        "GLOBAL_SEED",
        2026,
    )
    + 114
)


if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


GREEDY_SCALABILITY_PATH = (
    RESULTS_DIR
    / "greedy_scalability_v2.csv"
)


COMPACTNESS_DESCRIPTIVE_PATH = (
    TABLES_DIR
    / "greedy_scalability_v2_solution_compactness_by_size.csv"
)


COMPACTNESS_BOOTSTRAP_PATH = (
    TABLES_DIR
    / "greedy_scalability_v2_solution_compactness_bootstrap_ci.csv"
)


COMPACTNESS_REPLICATES_PATH = (
    RESULTS_DIR
    / "greedy_scalability_v2_solution_compactness_bootstrap_replicates.csv"
)


SCALABILITY_SIZE_LEVELS = [
    16,
    32,
    64,
    128,
    256,
]


# --------------------------------------------------
# Load benchmark
# --------------------------------------------------

compactness_df = pd.read_csv(
    GREEDY_SCALABILITY_PATH,
    low_memory=False,
)


required_columns = [
    "matched_group_id",
    "instance_id",
    "n_items",
    "regime",
    "tau",
    "weight_profile",
    "greedy_status",
    "greedy_selected_count",
    "greedy_utility",
    "greedy_privacy",
    "greedy_normalized_cognitive_cost",
    "greedy_cardinality_fraction",
    "greedy_disclosure_cost",
]


missing_columns = [
    column_name
    for column_name in required_columns
    if column_name not in compactness_df.columns
]


if missing_columns:

    raise KeyError(
        "Missing required columns: "
        + ", ".join(
            missing_columns
        )
    )


numeric_columns = [
    "n_items",
    "tau",
    "greedy_selected_count",
    "greedy_utility",
    "greedy_privacy",
    "greedy_normalized_cognitive_cost",
    "greedy_cardinality_fraction",
    "greedy_disclosure_cost",
]


for column_name in numeric_columns:

    compactness_df[
        column_name
    ] = pd.to_numeric(
        compactness_df[
            column_name
        ],
        errors="coerce",
    )


# --------------------------------------------------
# Integrity checks
# --------------------------------------------------

assert len(
    compactness_df
) == 24000


assert (
    compactness_df[
        "matched_group_id"
    ]
    .nunique()
    == 500
)


assert (
    compactness_df[
        "instance_id"
    ]
    .nunique()
    == 1500
)


assert (
    compactness_df[
        "greedy_status"
    ]
    == "FEASIBLE"
).all()


assert (
    compactness_df[
        "greedy_selected_count"
    ]
    >= 0
).all()


assert (
    compactness_df[
        "greedy_selected_count"
    ]
    <=
    compactness_df[
        "n_items"
    ]
).all()


assert (
    compactness_df[
        "greedy_utility"
    ]
    +
    1e-12
    >=
    compactness_df[
        "tau"
    ]
).all()


groups_per_size = (
    compactness_df
    .groupby(
        "n_items"
    )[
        "matched_group_id"
    ]
    .nunique()
)


assert (
    groups_per_size
    == 100
).all()


# --------------------------------------------------
# Derived compactness measures
# --------------------------------------------------

compactness_df[
    "selected_fraction_percent"
] = (
    100.0
    *
    compactness_df[
        "greedy_cardinality_fraction"
    ]
)


compactness_df[
    "candidate_pool_reduction_percent"
] = (
    100.0
    *
    (
        1.0
        -
        compactness_df[
            "greedy_cardinality_fraction"
        ]
    )
)


compactness_df[
    "utility_surplus"
] = (
    compactness_df[
        "greedy_utility"
    ]
    -
    compactness_df[
        "tau"
    ]
)


compactness_df[
    "selected_at_most_5_numeric"
] = (
    compactness_df[
        "greedy_selected_count"
    ]
    <= 5
).astype(int)


compactness_df[
    "selected_at_most_10_numeric"
] = (
    compactness_df[
        "greedy_selected_count"
    ]
    <= 10
).astype(int)


# Numerical safety checks
assert (
    compactness_df[
        "utility_surplus"
    ]
    >= -1e-12
).all()


assert (
    compactness_df[
        "selected_fraction_percent"
    ]
    >= 0.0
).all()


assert (
    compactness_df[
        "selected_fraction_percent"
    ]
    <= 100.0
).all()


# --------------------------------------------------
# Descriptive summaries by candidate-set size
# --------------------------------------------------

descriptive_rows = []


for n_items in SCALABILITY_SIZE_LEVELS:

    current_size_df = (
        compactness_df[
            compactness_df[
                "n_items"
            ]
            == n_items
        ]
    )


    selected_count_values = (
        current_size_df[
            "greedy_selected_count"
        ]
        .to_numpy(
            dtype=float
        )
    )


    descriptive_rows.append(
        {
            "n_items":
                n_items,

            "matched_groups":
                current_size_df[
                    "matched_group_id"
                ]
                .nunique(),

            "regime_specific_instances":
                current_size_df[
                    "instance_id"
                ]
                .nunique(),

            "condition_rows":
                len(
                    current_size_df
                ),

            "mean_selected_count":
                selected_count_values.mean(),

            "median_selected_count":
                np.median(
                    selected_count_values
                ),

            "p90_selected_count":
                np.quantile(
                    selected_count_values,
                    0.90,
                ),

            "p95_selected_count":
                np.quantile(
                    selected_count_values,
                    0.95,
                ),

            "maximum_selected_count":
                selected_count_values.max(),

            "selected_at_most_5_percent":
                100.0
                *
                current_size_df[
                    "selected_at_most_5_numeric"
                ].mean(),

            "selected_at_most_10_percent":
                100.0
                *
                current_size_df[
                    "selected_at_most_10_numeric"
                ].mean(),

            "mean_selected_fraction_percent":
                current_size_df[
                    "selected_fraction_percent"
                ].mean(),

            "mean_candidate_pool_reduction_percent":
                current_size_df[
                    "candidate_pool_reduction_percent"
                ].mean(),

            "mean_utility_surplus":
                current_size_df[
                    "utility_surplus"
                ].mean(),

            "median_utility_surplus":
                current_size_df[
                    "utility_surplus"
                ].median(),

            "mean_privacy_exposure":
                current_size_df[
                    "greedy_privacy"
                ].mean(),

            "mean_normalized_cognitive_cost":
                current_size_df[
                    "greedy_normalized_cognitive_cost"
                ].mean(),

            "mean_disclosure_cost":
                current_size_df[
                    "greedy_disclosure_cost"
                ].mean(),
        }
    )


compactness_descriptive_df = pd.DataFrame(
    descriptive_rows
)


compactness_descriptive_df.to_csv(
    COMPACTNESS_DESCRIPTIVE_PATH,
    index=False,
)


# --------------------------------------------------
# One sufficient-statistics row per independent
# matched group
#
# Each matched group contains:
# 3 regimes × 4 thresholds × 4 weight profiles
# = 48 conditions.
# --------------------------------------------------

compactness_group_df = (
    compactness_df
    .groupby(
        [
            "matched_group_id",
            "n_items",
        ],
        as_index=False,
    )
    .agg(
        condition_rows=(
            "greedy_selected_count",
            "size",
        ),

        selected_count_sum=(
            "greedy_selected_count",
            "sum",
        ),

        selected_at_most_5_sum=(
            "selected_at_most_5_numeric",
            "sum",
        ),

        selected_at_most_10_sum=(
            "selected_at_most_10_numeric",
            "sum",
        ),

        selected_fraction_percent_sum=(
            "selected_fraction_percent",
            "sum",
        ),

        candidate_pool_reduction_percent_sum=(
            "candidate_pool_reduction_percent",
            "sum",
        ),

        utility_surplus_sum=(
            "utility_surplus",
            "sum",
        ),

        privacy_exposure_sum=(
            "greedy_privacy",
            "sum",
        ),

        normalized_cognitive_cost_sum=(
            "greedy_normalized_cognitive_cost",
            "sum",
        ),

        disclosure_cost_sum=(
            "greedy_disclosure_cost",
            "sum",
        ),
    )
)


assert len(
    compactness_group_df
) == 500


assert (
    compactness_group_df[
        "condition_rows"
    ]
    == 48
).all()


COMPACTNESS_AGGREGATE_COLUMNS = [
    "condition_rows",
    "selected_count_sum",
    "selected_at_most_5_sum",
    "selected_at_most_10_sum",
    "selected_fraction_percent_sum",
    "candidate_pool_reduction_percent_sum",
    "utility_surplus_sum",
    "privacy_exposure_sum",
    "normalized_cognitive_cost_sum",
    "disclosure_cost_sum",
]


COMPACTNESS_METRIC_NAMES = [
    "mean_selected_count",
    "selected_at_most_5_percent",
    "selected_at_most_10_percent",
    "mean_selected_fraction_percent",
    "mean_candidate_pool_reduction_percent",
    "mean_utility_surplus",
    "mean_privacy_exposure",
    "mean_normalized_cognitive_cost",
    "mean_disclosure_cost",
]


def calculate_compactness_metrics(
    totals: np.ndarray,
) -> np.ndarray:
    """
    Convert matched-group sufficient-statistic totals
    into solution compactness metrics.
    """

    condition_rows = totals[0]


    return np.array(
        [
            totals[1]
            /
            condition_rows,

            100.0
            *
            totals[2]
            /
            condition_rows,

            100.0
            *
            totals[3]
            /
            condition_rows,

            totals[4]
            /
            condition_rows,

            totals[5]
            /
            condition_rows,

            totals[6]
            /
            condition_rows,

            totals[7]
            /
            condition_rows,

            totals[8]
            /
            condition_rows,

            totals[9]
            /
            condition_rows,
        ],
        dtype=float,
    )


# --------------------------------------------------
# Bootstrap independently within each size
# --------------------------------------------------

compactness_rng = np.random.default_rng(
    COMPACTNESS_BOOTSTRAP_SEED
)


bootstrap_summary_rows = []
bootstrap_replicate_frames = []


for n_items in SCALABILITY_SIZE_LEVELS:

    current_group_df = (
        compactness_group_df[
            compactness_group_df[
                "n_items"
            ]
            == n_items
        ]
        .sort_values(
            "matched_group_id"
        )
        .reset_index(
            drop=True
        )
    )


    assert len(
        current_group_df
    ) == 100


    group_array = (
        current_group_df[
            COMPACTNESS_AGGREGATE_COLUMNS
        ]
        .to_numpy(
            dtype=float
        )
    )


    observed_metrics = (
        calculate_compactness_metrics(
            group_array.sum(
                axis=0
            )
        )
    )


    bootstrap_estimates = np.empty(
        (
            COMPACTNESS_BOOTSTRAP_REPLICATES,
            len(
                COMPACTNESS_METRIC_NAMES
            ),
        ),
        dtype=float,
    )


    for bootstrap_index in range(
        COMPACTNESS_BOOTSTRAP_REPLICATES
    ):

        sampled_indices = (
            compactness_rng.integers(
                low=0,
                high=len(
                    group_array
                ),
                size=len(
                    group_array
                ),
            )
        )


        resampled_totals = (
            group_array[
                sampled_indices
            ]
            .sum(
                axis=0
            )
        )


        bootstrap_estimates[
            bootstrap_index
        ] = calculate_compactness_metrics(
            resampled_totals
        )


    standard_errors = (
        bootstrap_estimates.std(
            axis=0,
            ddof=1,
        )
    )


    confidence_lower = np.quantile(
        bootstrap_estimates,
        0.025,
        axis=0,
    )


    confidence_upper = np.quantile(
        bootstrap_estimates,
        0.975,
        axis=0,
    )


    for metric_index, metric_name in enumerate(
        COMPACTNESS_METRIC_NAMES
    ):

        bootstrap_summary_rows.append(
            {
                "n_items":
                    n_items,

                "metric":
                    metric_name,

                "estimate":
                    observed_metrics[
                        metric_index
                    ],

                "bootstrap_standard_error":
                    standard_errors[
                        metric_index
                    ],

                "ci_95_lower":
                    confidence_lower[
                        metric_index
                    ],

                "ci_95_upper":
                    confidence_upper[
                        metric_index
                    ],

                "matched_groups":
                    100,

                "condition_rows":
                    4800,

                "bootstrap_replicates":
                    COMPACTNESS_BOOTSTRAP_REPLICATES,

                "resampling_unit":
                    "matched_size_replicate_group",

                "resampling_design":
                    "within_candidate_set_size",

                "interpretation_scope":
                    (
                        "descriptive properties of "
                        "Greedy feasible solutions"
                    ),
            }
        )


    replicate_df = pd.DataFrame(
        bootstrap_estimates,
        columns=COMPACTNESS_METRIC_NAMES,
    )


    replicate_df.insert(
        0,
        "bootstrap_replicate",
        np.arange(
            COMPACTNESS_BOOTSTRAP_REPLICATES
        ),
    )


    replicate_df.insert(
        0,
        "n_items",
        n_items,
    )


    bootstrap_replicate_frames.append(
        replicate_df
    )


    print(
        "Completed solution compactness size:",
        n_items
    )


# --------------------------------------------------
# Save bootstrap outputs
# --------------------------------------------------

compactness_bootstrap_df = pd.DataFrame(
    bootstrap_summary_rows
)


compactness_replicates_df = pd.concat(
    bootstrap_replicate_frames,
    ignore_index=True,
)


compactness_bootstrap_df.to_csv(
    COMPACTNESS_BOOTSTRAP_PATH,
    index=False,
)


compactness_replicates_df.to_csv(
    COMPACTNESS_REPLICATES_PATH,
    index=False,
)


# --------------------------------------------------
# Compact confidence-interval display
# --------------------------------------------------

key_metrics = [
    "mean_selected_count",
    "selected_at_most_5_percent",
    "mean_selected_fraction_percent",
    "mean_candidate_pool_reduction_percent",
    "mean_utility_surplus",
    "mean_privacy_exposure",
    "mean_normalized_cognitive_cost",
]


compactness_display_df = (
    compactness_bootstrap_df[
        compactness_bootstrap_df[
            "metric"
        ].isin(
            key_metrics
        )
    ]
    .copy()
)


compactness_display_df[
    "estimate_and_ci"
] = (
    compactness_display_df[
        "estimate"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    compactness_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    compactness_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


compactness_pivot_df = (
    compactness_display_df
    .pivot(
        index="n_items",
        columns="metric",
        values="estimate_and_ci",
    )
    .reset_index()
    .sort_values(
        "n_items"
    )
)


# --------------------------------------------------
# Print results
# --------------------------------------------------

print(
    "\n======================================"
)

print(
    "GREEDY LARGE-SCALE SOLUTION COMPACTNESS"
)

print(
    "======================================"
)


print(
    compactness_descriptive_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.6f}",
    )
)


print(
    "\n======================================"
)

print(
    "MATCHED-GROUP BOOTSTRAP 95% CIs"
)

print(
    "======================================"
)


print(
    compactness_pivot_df.to_string(
        index=False
    )
)


print(
    "\nDescriptive compactness summary saved to:"
)

print(
    COMPACTNESS_DESCRIPTIVE_PATH
)


print(
    "\nCompactness confidence intervals saved to:"
)

print(
    COMPACTNESS_BOOTSTRAP_PATH
)


print(
    "\nBootstrap replicates saved to:"
)

print(
    COMPACTNESS_REPLICATES_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Large-scale Greedy compactness contrasts across
# candidate-set sizes
# --------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------
# Configuration
# --------------------------------------------------

COMPACTNESS_CONTRAST_BOOTSTRAP_REPLICATES = 5000

COMPACTNESS_CONTRAST_BOOTSTRAP_SEED = (
    globals().get(
        "GLOBAL_SEED",
        2026,
    )
    + 115
)


if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


GREEDY_SCALABILITY_PATH = (
    RESULTS_DIR
    / "greedy_scalability_v2.csv"
)


COMPACTNESS_CONTRAST_PATH = (
    TABLES_DIR
    / "greedy_scalability_v2_compactness_size_contrasts.csv"
)


COMPACTNESS_CONTRAST_REPLICATES_PATH = (
    RESULTS_DIR
    / "greedy_scalability_v2_compactness_size_contrast_replicates.csv"
)


SCALABILITY_SIZE_LEVELS = [
    16,
    32,
    64,
    128,
    256,
]


# Each comparison is:
# larger candidate set minus smaller candidate set.
COMPACTNESS_SIZE_COMPARISONS = [
    (32, 16),
    (64, 32),
    (128, 64),
    (256, 128),
    (256, 16),
]


COMPACTNESS_CONTRAST_METRICS = [
    "mean_selected_count",
    "selected_at_most_5_percent",
    "selected_at_most_10_percent",
    "mean_selected_fraction_percent",
    "mean_candidate_pool_reduction_percent",
    "mean_utility_surplus",
    "mean_privacy_exposure",
    "mean_normalized_cognitive_cost",
    "mean_disclosure_cost",
]


# --------------------------------------------------
# Load benchmark
# --------------------------------------------------

compactness_contrast_df = pd.read_csv(
    GREEDY_SCALABILITY_PATH,
    low_memory=False,
)


required_columns = [
    "matched_group_id",
    "n_items",
    "greedy_status",
    "tau",
    "greedy_selected_count",
    "greedy_utility",
    "greedy_privacy",
    "greedy_normalized_cognitive_cost",
    "greedy_cardinality_fraction",
    "greedy_disclosure_cost",
]


missing_columns = [
    column_name
    for column_name in required_columns
    if column_name not in compactness_contrast_df.columns
]


if missing_columns:

    raise KeyError(
        "Missing required columns: "
        + ", ".join(
            missing_columns
        )
    )


numeric_columns = [
    "n_items",
    "tau",
    "greedy_selected_count",
    "greedy_utility",
    "greedy_privacy",
    "greedy_normalized_cognitive_cost",
    "greedy_cardinality_fraction",
    "greedy_disclosure_cost",
]


for column_name in numeric_columns:

    compactness_contrast_df[
        column_name
    ] = pd.to_numeric(
        compactness_contrast_df[
            column_name
        ],
        errors="coerce",
    )


# --------------------------------------------------
# Integrity checks
# --------------------------------------------------

assert len(
    compactness_contrast_df
) == 24000


assert (
    compactness_contrast_df[
        "matched_group_id"
    ].nunique()
    == 500
)


assert (
    compactness_contrast_df[
        "greedy_status"
    ]
    == "FEASIBLE"
).all()


assert (
    sorted(
        compactness_contrast_df[
            "n_items"
        ]
        .dropna()
        .unique()
        .astype(int)
        .tolist()
    )
    ==
    SCALABILITY_SIZE_LEVELS
)


groups_per_size = (
    compactness_contrast_df
    .groupby(
        "n_items"
    )[
        "matched_group_id"
    ]
    .nunique()
)


assert (
    groups_per_size
    == 100
).all()


rows_per_group = (
    compactness_contrast_df
    .groupby(
        "matched_group_id"
    )
    .size()
)


assert (
    rows_per_group
    == 48
).all()


# --------------------------------------------------
# Derived compactness measures
# --------------------------------------------------

compactness_contrast_df[
    "selected_at_most_5_numeric"
] = (
    compactness_contrast_df[
        "greedy_selected_count"
    ]
    <= 5
).astype(int)


compactness_contrast_df[
    "selected_at_most_10_numeric"
] = (
    compactness_contrast_df[
        "greedy_selected_count"
    ]
    <= 10
).astype(int)


compactness_contrast_df[
    "selected_fraction_percent"
] = (
    100.0
    *
    compactness_contrast_df[
        "greedy_cardinality_fraction"
    ]
)


compactness_contrast_df[
    "candidate_pool_reduction_percent"
] = (
    100.0
    *
    (
        1.0
        -
        compactness_contrast_df[
            "greedy_cardinality_fraction"
        ]
    )
)


compactness_contrast_df[
    "utility_surplus"
] = (
    compactness_contrast_df[
        "greedy_utility"
    ]
    -
    compactness_contrast_df[
        "tau"
    ]
)


assert (
    compactness_contrast_df[
        "utility_surplus"
    ]
    >= -1e-12
).all()


# --------------------------------------------------
# One sufficient-statistics row per independent
# matched group
# --------------------------------------------------

compactness_contrast_group_df = (
    compactness_contrast_df
    .groupby(
        [
            "matched_group_id",
            "n_items",
        ],
        as_index=False,
    )
    .agg(
        condition_rows=(
            "greedy_selected_count",
            "size",
        ),

        selected_count_sum=(
            "greedy_selected_count",
            "sum",
        ),

        selected_at_most_5_sum=(
            "selected_at_most_5_numeric",
            "sum",
        ),

        selected_at_most_10_sum=(
            "selected_at_most_10_numeric",
            "sum",
        ),

        selected_fraction_percent_sum=(
            "selected_fraction_percent",
            "sum",
        ),

        candidate_pool_reduction_percent_sum=(
            "candidate_pool_reduction_percent",
            "sum",
        ),

        utility_surplus_sum=(
            "utility_surplus",
            "sum",
        ),

        privacy_exposure_sum=(
            "greedy_privacy",
            "sum",
        ),

        normalized_cognitive_cost_sum=(
            "greedy_normalized_cognitive_cost",
            "sum",
        ),

        disclosure_cost_sum=(
            "greedy_disclosure_cost",
            "sum",
        ),
    )
)


assert len(
    compactness_contrast_group_df
) == 500


assert (
    compactness_contrast_group_df[
        "condition_rows"
    ]
    == 48
).all()


COMPACTNESS_AGGREGATE_COLUMNS = [
    "condition_rows",
    "selected_count_sum",
    "selected_at_most_5_sum",
    "selected_at_most_10_sum",
    "selected_fraction_percent_sum",
    "candidate_pool_reduction_percent_sum",
    "utility_surplus_sum",
    "privacy_exposure_sum",
    "normalized_cognitive_cost_sum",
    "disclosure_cost_sum",
]


def calculate_compactness_contrast_metrics(
    totals: np.ndarray,
) -> np.ndarray:
    """
    Convert sufficient-statistic totals into
    large-scale Greedy compactness metrics.
    """

    condition_rows = totals[0]


    return np.array(
        [
            totals[1]
            /
            condition_rows,

            100.0
            *
            totals[2]
            /
            condition_rows,

            100.0
            *
            totals[3]
            /
            condition_rows,

            totals[4]
            /
            condition_rows,

            totals[5]
            /
            condition_rows,

            totals[6]
            /
            condition_rows,

            totals[7]
            /
            condition_rows,

            totals[8]
            /
            condition_rows,

            totals[9]
            /
            condition_rows,
        ],
        dtype=float,
    )


# --------------------------------------------------
# Prepare one aligned array for each size
# --------------------------------------------------

compactness_arrays_by_size = {}
observed_metrics_by_size = {}


for n_items in SCALABILITY_SIZE_LEVELS:

    current_size_group_df = (
        compactness_contrast_group_df[
            compactness_contrast_group_df[
                "n_items"
            ]
            == n_items
        ]
        .sort_values(
            "matched_group_id"
        )
        .reset_index(
            drop=True
        )
    )


    assert len(
        current_size_group_df
    ) == 100


    current_size_array = (
        current_size_group_df[
            COMPACTNESS_AGGREGATE_COLUMNS
        ]
        .to_numpy(
            dtype=float
        )
    )


    compactness_arrays_by_size[
        n_items
    ] = current_size_array


    observed_metrics_by_size[
        n_items
    ] = calculate_compactness_contrast_metrics(
        current_size_array.sum(
            axis=0
        )
    )


# --------------------------------------------------
# Independent bootstrap contrasts across sizes
# --------------------------------------------------

compactness_contrast_rng = np.random.default_rng(
    COMPACTNESS_CONTRAST_BOOTSTRAP_SEED
)


compactness_contrast_rows = []
compactness_contrast_replicate_frames = []


for larger_size, smaller_size in (
    COMPACTNESS_SIZE_COMPARISONS
):

    larger_array = (
        compactness_arrays_by_size[
            larger_size
        ]
    )


    smaller_array = (
        compactness_arrays_by_size[
            smaller_size
        ]
    )


    observed_difference = (
        observed_metrics_by_size[
            larger_size
        ]
        -
        observed_metrics_by_size[
            smaller_size
        ]
    )


    bootstrap_differences = np.empty(
        (
            COMPACTNESS_CONTRAST_BOOTSTRAP_REPLICATES,
            len(
                COMPACTNESS_CONTRAST_METRICS
            ),
        ),
        dtype=float,
    )


    for bootstrap_index in range(
        COMPACTNESS_CONTRAST_BOOTSTRAP_REPLICATES
    ):

        # Candidate-set sizes use independently
        # generated matched groups.
        larger_sample_indices = (
            compactness_contrast_rng.integers(
                low=0,
                high=len(
                    larger_array
                ),
                size=len(
                    larger_array
                ),
            )
        )


        smaller_sample_indices = (
            compactness_contrast_rng.integers(
                low=0,
                high=len(
                    smaller_array
                ),
                size=len(
                    smaller_array
                ),
            )
        )


        larger_totals = (
            larger_array[
                larger_sample_indices
            ]
            .sum(
                axis=0
            )
        )


        smaller_totals = (
            smaller_array[
                smaller_sample_indices
            ]
            .sum(
                axis=0
            )
        )


        bootstrap_differences[
            bootstrap_index
        ] = (
            calculate_compactness_contrast_metrics(
                larger_totals
            )
            -
            calculate_compactness_contrast_metrics(
                smaller_totals
            )
        )


    bootstrap_standard_errors = (
        bootstrap_differences.std(
            axis=0,
            ddof=1,
        )
    )


    confidence_lower = np.quantile(
        bootstrap_differences,
        0.025,
        axis=0,
    )


    confidence_upper = np.quantile(
        bootstrap_differences,
        0.975,
        axis=0,
    )


    contrast_name = (
        f"M={larger_size}"
        f" minus "
        f"M={smaller_size}"
    )


    for metric_index, metric_name in enumerate(
        COMPACTNESS_CONTRAST_METRICS
    ):

        compactness_contrast_rows.append(
            {
                "larger_size":
                    larger_size,

                "smaller_size":
                    smaller_size,

                "contrast":
                    contrast_name,

                "metric":
                    metric_name,

                "estimated_difference":
                    observed_difference[
                        metric_index
                    ],

                "bootstrap_standard_error":
                    bootstrap_standard_errors[
                        metric_index
                    ],

                "ci_95_lower":
                    confidence_lower[
                        metric_index
                    ],

                "ci_95_upper":
                    confidence_upper[
                        metric_index
                    ],

                "ci_excludes_zero":
                    (
                        confidence_lower[
                            metric_index
                        ]
                        > 0.0
                    )
                    or
                    (
                        confidence_upper[
                            metric_index
                        ]
                        < 0.0
                    ),

                "matched_groups_per_size":
                    100,

                "bootstrap_replicates":
                    COMPACTNESS_CONTRAST_BOOTSTRAP_REPLICATES,

                "resampling_unit":
                    "matched_size_replicate_group",

                "resampling_design":
                    (
                        "independent_resampling_"
                        "within_each_candidate_size"
                    ),

                "interpretation_scope":
                    (
                        "descriptive properties of "
                        "Greedy feasible solutions"
                    ),

                "multiple_comparison_adjustment":
                    "none",
            }
        )


    contrast_replicate_df = pd.DataFrame(
        bootstrap_differences,
        columns=COMPACTNESS_CONTRAST_METRICS,
    )


    contrast_replicate_df.insert(
        0,
        "bootstrap_replicate",
        np.arange(
            COMPACTNESS_CONTRAST_BOOTSTRAP_REPLICATES
        ),
    )


    contrast_replicate_df.insert(
        0,
        "contrast",
        contrast_name,
    )


    compactness_contrast_replicate_frames.append(
        contrast_replicate_df
    )


    print(
        "Completed compactness size contrast:",
        contrast_name
    )


# --------------------------------------------------
# Save results
# --------------------------------------------------

compactness_contrast_results_df = pd.DataFrame(
    compactness_contrast_rows
)


compactness_contrast_replicates_df = pd.concat(
    compactness_contrast_replicate_frames,
    ignore_index=True,
)


compactness_contrast_results_df.to_csv(
    COMPACTNESS_CONTRAST_PATH,
    index=False,
)


compactness_contrast_replicates_df.to_csv(
    COMPACTNESS_CONTRAST_REPLICATES_PATH,
    index=False,
)


# --------------------------------------------------
# Readable display
# --------------------------------------------------

compactness_contrast_display_df = (
    compactness_contrast_results_df.copy()
)


compactness_contrast_display_df[
    "difference_and_ci"
] = (
    compactness_contrast_display_df[
        "estimated_difference"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    compactness_contrast_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    compactness_contrast_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


print(
    "\n======================================"
)

print(
    "GREEDY LARGE-SCALE COMPACTNESS CONTRASTS"
)

print(
    "======================================"
)


print(
    compactness_contrast_display_df[
        [
            "contrast",
            "metric",
            "difference_and_ci",
            "ci_excludes_zero",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\nCompactness contrasts saved to:"
)

print(
    COMPACTNESS_CONTRAST_PATH
)


print(
    "\nCompactness contrast replicates saved to:"
)

print(
    COMPACTNESS_CONTRAST_REPLICATES_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Step 116A — Inspect corrected-v2 benchmark columns
# related to Exact, Greedy, subsets, and identifiers
# --------------------------------------------------

from pathlib import Path
import pandas as pd


if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


CORRECTED_REFERENCE_PATH = (
    RESULTS_DIR
    / "exact_vs_greedy_main_benchmark_v2_marginal_preserving.csv"
)


reference_schema_df = pd.read_csv(
    CORRECTED_REFERENCE_PATH,
    low_memory=False,
)


SEARCH_TERMS = [
    "exact",
    "greedy",
    "select",
    "subset",
    "item",
    "id",
    "count",
    "cost",
    "utility",
    "privacy",
    "cognitive",
]


relevant_columns = [
    column_name
    for column_name in reference_schema_df.columns
    if any(
        search_term in column_name.lower()
        for search_term in SEARCH_TERMS
    )
]


schema_rows = []


for column_name in relevant_columns:

    non_null_values = (
        reference_schema_df[
            column_name
        ]
        .dropna()
    )


    if len(non_null_values) > 0:

        example_value = repr(
            non_null_values.iloc[0]
        )

    else:

        example_value = "<all values missing>"


    if len(example_value) > 180:

        example_value = (
            example_value[:177]
            + "..."
        )


    schema_rows.append(
        {
            "column_name":
                column_name,

            "dtype":
                str(
                    reference_schema_df[
                        column_name
                    ].dtype
                ),

            "non_null_rows":
                int(
                    reference_schema_df[
                        column_name
                    ]
                    .notna()
                    .sum()
                ),

            "unique_values":
                int(
                    reference_schema_df[
                        column_name
                    ]
                    .nunique(
                        dropna=True
                    )
                ),

            "example_value":
                example_value,
        }
    )


schema_summary_df = pd.DataFrame(
    schema_rows
)


print(
    "======================================"
)

print(
    "CORRECTED V2 REFERENCE FILE SCHEMA"
)

print(
    "======================================"
)


print(
    "File:"
)

print(
    CORRECTED_REFERENCE_PATH
)


print(
    "\nRows:",
    len(
        reference_schema_df
    )
)


print(
    "Total columns:",
    len(
        reference_schema_df.columns
    )
)


print(
    "\nAll column names:"
)


for column_index, column_name in enumerate(
    reference_schema_df.columns,
    start=1,
):

    print(
        f"{column_index:02d}. {column_name}"
    )


print(
    "\n======================================"
)

print(
    "EXACT / GREEDY / SUBSET-RELATED COLUMNS"
)

print(
    "======================================"
)


if len(
    schema_summary_df
) > 0:

    print(
        schema_summary_df.to_string(
            index=False
        )
    )

else:

    print(
        "No matching columns were found."
    )


print(
    "\n======================================"
)

print(
    "FIRST REFERENCE ROW"
)

print(
    "======================================"
)


print(
    reference_schema_df.iloc[
        0
    ].to_string()
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Step 116B — Corrected-v2 deterministic baselines
# --------------------------------------------------

from pathlib import Path
import ast
import time

import numpy as np
import pandas as pd


# --------------------------------------------------
# Configuration
# --------------------------------------------------

BASELINE_BOOTSTRAP_REPLICATES = 5000

BASELINE_BOOTSTRAP_SEED = (
    globals().get("GLOBAL_SEED", 2026)
    + 116
)


if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


if "generate_mst_instance" not in globals():

    raise NameError(
        "generate_mst_instance is unavailable. "
        "Run the corrected marginal-preserving-v2 "
        "generator cell first."
    )


CORRECTED_REFERENCE_PATH = (
    RESULTS_DIR
    / "exact_vs_greedy_main_benchmark_v2_marginal_preserving.csv"
)


BASELINE_CHECKPOINT_PATH = (
    RESULTS_DIR
    / "corrected_v2_deterministic_baseline_checkpoint.csv"
)


BASELINE_ONLY_PATH = (
    RESULTS_DIR
    / "corrected_v2_deterministic_baseline_results.csv"
)


BASELINE_LONG_PATH = (
    RESULTS_DIR
    / "corrected_v2_algorithm_comparison_long.csv"
)


BASELINE_DESCRIPTIVE_PATH = (
    TABLES_DIR
    / "corrected_v2_algorithm_baseline_descriptive_summary.csv"
)


BASELINE_BOOTSTRAP_PATH = (
    TABLES_DIR
    / "corrected_v2_algorithm_baseline_bootstrap_ci.csv"
)


BASELINE_BOOTSTRAP_REPLICATES_PATH = (
    RESULTS_DIR
    / "corrected_v2_algorithm_baseline_bootstrap_replicates.csv"
)


ALGORITHM_ORDER = [
    "dynamic_greedy",
    "utility_first",
    "lowest_cost_first",
    "static_utility_cost_ratio",
]


BASELINE_ALGORITHMS = [
    "utility_first",
    "lowest_cost_first",
    "static_utility_cost_ratio",
]


EXPECTED_REFERENCE_ROWS = 24000
EXPECTED_INSTANCES = 1500
CONDITIONS_PER_INSTANCE = 16

BASELINE_ROWS_PER_INSTANCE = (
    CONDITIONS_PER_INSTANCE
    * len(BASELINE_ALGORITHMS)
)

EXPECTED_BASELINE_ROWS = (
    EXPECTED_INSTANCES
    * BASELINE_ROWS_PER_INSTANCE
)

EXPECTED_COMBINED_ROWS = (
    EXPECTED_REFERENCE_ROWS
    * len(ALGORITHM_ORDER)
)


# --------------------------------------------------
# Subset parsing
# --------------------------------------------------

def normalize_selected_ids(selected_value):
    """
    Convert all stored subset formats into an ordered
    list of string item identifiers.

    Supported examples:
        z003|z006
        z003
        ('z003', 'z006')
        ['z003', 'z006']
    """

    if selected_value is None:

        return []


    try:

        if pd.isna(selected_value):

            return []

    except (TypeError, ValueError):

        pass


    if isinstance(
        selected_value,
        str,
    ):

        text = selected_value.strip()


        if text in {
            "",
            "[]",
            "()",
            "{}",
            "None",
            "nan",
        }:

            return []


        # Corrected-v2 benchmark representation.
        if "|" in text:

            selected_value = [
                token.strip().strip("'\"")
                for token in text.split("|")
                if token.strip()
            ]


        else:

            try:

                selected_value = ast.literal_eval(
                    text
                )

            except (
                ValueError,
                SyntaxError,
            ):

                selected_value = [
                    token.strip().strip("'\"")
                    for token in text.split(",")
                    if token.strip()
                ]


    if not isinstance(
        selected_value,
        (
            list,
            tuple,
            set,
            np.ndarray,
        ),
    ):

        selected_value = [
            selected_value
        ]


    normalized_ids = []


    for selected_item in selected_value:

        if hasattr(
            selected_item,
            "item_id",
        ):

            selected_id = selected_item.item_id


        elif hasattr(
            selected_item,
            "id",
        ):

            selected_id = selected_item.id


        else:

            selected_id = selected_item


        selected_text = (
            str(selected_id)
            .strip()
            .strip("'\"")
        )


        if selected_text:

            normalized_ids.append(
                selected_text
            )


    return list(
        dict.fromkeys(
            normalized_ids
        )
    )



def calculate_jaccard(
    first_selected,
    second_selected,
):
    """
    Calculate subset Jaccard similarity.
    """

    first_set = set(
        normalize_selected_ids(
            first_selected
        )
    )

    second_set = set(
        normalize_selected_ids(
            second_selected
        )
    )


    union_set = (
        first_set
        |
        second_set
    )


    if not union_set:

        return 1.0


    return (
        len(
            first_set
            &
            second_set
        )
        /
        len(
            union_set
        )
    )


# --------------------------------------------------
# Subset evaluation
# --------------------------------------------------

def evaluate_item_subset(
    all_items,
    selected_items,
    weights,
):
    """
    Evaluate a selected subset under the corrected-v2
    product utility and disclosure-cost equations.
    """

    if selected_items:

        utility = (
            1.0
            -
            np.prod(
                [
                    1.0
                    -
                    float(item.utility)
                    for item in selected_items
                ]
            )
        )


        privacy = (
            1.0
            -
            np.prod(
                [
                    1.0
                    -
                    float(item.privacy)
                    for item in selected_items
                ]
            )
        )


        cognitive_cost = sum(
            float(item.cognitive_cost)
            for item in selected_items
        )


    else:

        utility = 0.0
        privacy = 0.0
        cognitive_cost = 0.0


    full_cognitive_cost = sum(
        float(item.cognitive_cost)
        for item in all_items
    )


    if full_cognitive_cost > 0.0:

        normalized_cognitive_cost = (
            cognitive_cost
            /
            full_cognitive_cost
        )

    else:

        normalized_cognitive_cost = 0.0


    cardinality_fraction = (
        len(selected_items)
        /
        len(all_items)
    )


    disclosure_cost = (
        weights["lambda_privacy"]
        * privacy
        +
        weights["lambda_cognitive"]
        * normalized_cognitive_cost
        +
        weights["lambda_cardinality"]
        * cardinality_fraction
    )


    return {
        "selected_ids": [
            str(item.item_id)
            for item in selected_items
        ],

        "selected_count":
            len(selected_items),

        "utility":
            float(utility),

        "privacy":
            float(privacy),

        "normalized_cognitive_cost":
            float(
                normalized_cognitive_cost
            ),

        "cardinality_fraction":
            float(
                cardinality_fraction
            ),

        "disclosure_cost":
            float(disclosure_cost),
    }


# --------------------------------------------------
# Deterministic static-order baselines
# --------------------------------------------------

def run_deterministic_baseline(
    items,
    tau,
    weights,
    algorithm,
):
    """
    Rank candidates once and select the shortest prefix
    whose modeled utility reaches tau.
    """

    full_cognitive_cost = sum(
        float(item.cognitive_cost)
        for item in items
    )


    item_rows = []


    for item in items:

        item_id = str(item.item_id)
        utility = float(item.utility)
        privacy = float(item.privacy)
        cognitive = float(
            item.cognitive_cost
        )


        if full_cognitive_cost > 0.0:

            normalized_cognitive = (
                cognitive
                /
                full_cognitive_cost
            )

        else:

            normalized_cognitive = 0.0


        singleton_cost = (
            weights["lambda_privacy"]
            * privacy
            +
            weights["lambda_cognitive"]
            * normalized_cognitive
            +
            weights["lambda_cardinality"]
            * (
                1.0
                /
                len(items)
            )
        )


        if singleton_cost > 0.0:

            utility_cost_ratio = (
                utility
                /
                singleton_cost
            )

        elif utility > 0.0:

            utility_cost_ratio = np.inf

        else:

            utility_cost_ratio = 0.0


        item_rows.append(
            {
                "item":
                    item,

                "item_id":
                    item_id,

                "utility":
                    utility,

                "singleton_cost":
                    singleton_cost,

                "utility_cost_ratio":
                    utility_cost_ratio,
            }
        )


    if algorithm == "utility_first":

        ordered_rows = sorted(
            item_rows,
            key=lambda row: (
                -row["utility"],
                row["singleton_cost"],
                row["item_id"],
            ),
        )


    elif algorithm == "lowest_cost_first":

        ordered_rows = sorted(
            item_rows,
            key=lambda row: (
                row["singleton_cost"],
                -row["utility"],
                row["item_id"],
            ),
        )


    elif algorithm == "static_utility_cost_ratio":

        ordered_rows = sorted(
            item_rows,
            key=lambda row: (
                -row["utility_cost_ratio"],
                -row["utility"],
                row["singleton_cost"],
                row["item_id"],
            ),
        )


    else:

        raise ValueError(
            f"Unsupported algorithm: {algorithm}"
        )


    selected_items = []
    residual_failure_probability = 1.0


    for item_row in ordered_rows:

        selected_items.append(
            item_row["item"]
        )


        residual_failure_probability *= (
            1.0
            -
            item_row["utility"]
        )


        current_utility = (
            1.0
            -
            residual_failure_probability
        )


        if (
            current_utility
            +
            1e-12
            >= tau
        ):

            break


    evaluation = evaluate_item_subset(
        all_items=items,
        selected_items=selected_items,
        weights=weights,
    )


    if (
        evaluation["utility"]
        +
        1e-12
        >= tau
    ):

        evaluation["status"] = "FEASIBLE"

    else:

        evaluation["status"] = "INFEASIBLE"


    return evaluation


# --------------------------------------------------
# Load corrected-v2 reference benchmark
# --------------------------------------------------

reference_df = pd.read_csv(
    CORRECTED_REFERENCE_PATH,
    low_memory=False,
)


required_reference_columns = [
    "seed",
    "matched_seed",
    "matched_group_id",
    "instance_id",
    "n_items",
    "replicate",
    "regime",
    "tau",
    "weight_profile",
    "lambda_privacy",
    "lambda_cognitive",
    "lambda_cardinality",
    "exact_status",
    "exact_selected_count",
    "exact_selected_items",
    "exact_disclosure_cost",
    "greedy_status",
    "greedy_selected_count",
    "greedy_selected_items",
    "greedy_utility",
    "greedy_privacy",
    "greedy_normalized_cognitive_cost",
    "greedy_disclosure_cost",
    "greedy_runtime_seconds",
]


missing_reference_columns = [
    column_name
    for column_name in required_reference_columns
    if column_name not in reference_df.columns
]


if missing_reference_columns:

    raise KeyError(
        "Missing reference columns: "
        + ", ".join(
            missing_reference_columns
        )
    )


assert len(reference_df) == EXPECTED_REFERENCE_ROWS

assert (
    reference_df["instance_id"].nunique()
    == EXPECTED_INSTANCES
)


numeric_reference_columns = [
    "matched_seed",
    "n_items",
    "replicate",
    "tau",
    "lambda_privacy",
    "lambda_cognitive",
    "lambda_cardinality",
    "exact_selected_count",
    "exact_disclosure_cost",
    "greedy_selected_count",
    "greedy_utility",
    "greedy_privacy",
    "greedy_normalized_cognitive_cost",
    "greedy_disclosure_cost",
    "greedy_runtime_seconds",
]


for column_name in numeric_reference_columns:

    reference_df[column_name] = pd.to_numeric(
        reference_df[column_name],
        errors="coerce",
    )


# Confirm that pipe-separated subsets are parsed.
example_exact_ids = normalize_selected_ids(
    reference_df.loc[
        reference_df["exact_selected_items"].notna(),
        "exact_selected_items",
    ].iloc[0]
)


example_greedy_ids = normalize_selected_ids(
    reference_df.loc[
        reference_df["greedy_selected_items"].notna(),
        "greedy_selected_items",
    ].iloc[0]
)


assert len(example_exact_ids) >= 1
assert len(example_greedy_ids) >= 1


# --------------------------------------------------
# Load valid checkpoint, when available
# --------------------------------------------------

baseline_rows = []
complete_instance_ids = set()


if BASELINE_ONLY_PATH.exists():

    baseline_loading_path = (
        BASELINE_ONLY_PATH
    )

    print(
        "Existing completed baseline file found."
    )


elif BASELINE_CHECKPOINT_PATH.exists():

    baseline_loading_path = (
        BASELINE_CHECKPOINT_PATH
    )

    print(
        "Existing baseline checkpoint found."
    )


else:

    baseline_loading_path = None

    print(
        "No baseline checkpoint found."
    )


if baseline_loading_path is not None:

    loaded_baseline_df = pd.read_csv(
        baseline_loading_path,
        low_memory=False,
    )


    required_checkpoint_columns = {
        "instance_id",
        "tau",
        "weight_profile",
        "algorithm",
    }


    if required_checkpoint_columns.issubset(
        loaded_baseline_df.columns
    ):

        loaded_baseline_df = (
            loaded_baseline_df
            .drop_duplicates(
                subset=[
                    "instance_id",
                    "tau",
                    "weight_profile",
                    "algorithm",
                ],
                keep="last",
            )
        )


        instance_counts = (
            loaded_baseline_df
            .groupby("instance_id")
            .size()
        )


        complete_instance_ids = set(
            instance_counts[
                instance_counts
                ==
                BASELINE_ROWS_PER_INSTANCE
            ].index
        )


        baseline_rows = (
            loaded_baseline_df[
                loaded_baseline_df[
                    "instance_id"
                ].isin(
                    complete_instance_ids
                )
            ]
            .to_dict("records")
        )


        print(
            "Complete baseline instances loaded:",
            len(complete_instance_ids)
        )


    else:

        print(
            "Existing checkpoint schema is incompatible. "
            "Starting a new baseline run."
        )


# --------------------------------------------------
# Generate baseline results
# --------------------------------------------------

instance_metadata_df = (
    reference_df[
        [
            "matched_group_id",
            "instance_id",
            "n_items",
            "replicate",
            "regime",
            "matched_seed",
        ]
    ]
    .drop_duplicates(
        subset=["instance_id"]
    )
    .sort_values(
        [
            "n_items",
            "replicate",
            "regime",
        ]
    )
    .reset_index(drop=True)
)


assert len(instance_metadata_df) == EXPECTED_INSTANCES


baseline_session_start = time.perf_counter()


for _, metadata_row in (
    instance_metadata_df.iterrows()
):

    instance_id = metadata_row[
        "instance_id"
    ]


    if instance_id in complete_instance_ids:

        continue


    n_items = int(
        metadata_row["n_items"]
    )

    matched_seed = int(
        metadata_row["matched_seed"]
    )

    regime = metadata_row[
        "regime"
    ]


    items = generate_mst_instance(
        n_items=n_items,
        regime=regime,
        seed=matched_seed,
    )


    condition_df = (
        reference_df[
            reference_df["instance_id"]
            ==
            instance_id
        ]
        .sort_values(
            [
                "tau",
                "weight_profile",
            ]
        )
    )


    assert len(condition_df) == CONDITIONS_PER_INSTANCE


    for _, condition_row in (
        condition_df.iterrows()
    ):

        tau = float(
            condition_row["tau"]
        )


        weights = {
            "lambda_privacy":
                float(
                    condition_row[
                        "lambda_privacy"
                    ]
                ),

            "lambda_cognitive":
                float(
                    condition_row[
                        "lambda_cognitive"
                    ]
                ),

            "lambda_cardinality":
                float(
                    condition_row[
                        "lambda_cardinality"
                    ]
                ),
        }


        for algorithm in BASELINE_ALGORITHMS:

            start_time = time.perf_counter()


            result = run_deterministic_baseline(
                items=items,
                tau=tau,
                weights=weights,
                algorithm=algorithm,
            )


            runtime_seconds = (
                time.perf_counter()
                -
                start_time
            )


            baseline_rows.append(
                {
                    "benchmark_version":
                        "corrected_v2_deterministic_baselines",

                    "matched_group_id":
                        condition_row[
                            "matched_group_id"
                        ],

                    "instance_id":
                        instance_id,

                    "n_items":
                        n_items,

                    "replicate":
                        int(
                            condition_row[
                                "replicate"
                            ]
                        ),

                    "regime":
                        regime,

                    "matched_seed":
                        matched_seed,

                    "tau":
                        tau,

                    "weight_profile":
                        condition_row[
                            "weight_profile"
                        ],

                    "lambda_privacy":
                        weights[
                            "lambda_privacy"
                        ],

                    "lambda_cognitive":
                        weights[
                            "lambda_cognitive"
                        ],

                    "lambda_cardinality":
                        weights[
                            "lambda_cardinality"
                        ],

                    "algorithm":
                        algorithm,

                    "algorithm_status":
                        result["status"],

                    "algorithm_selected_ids":
                        repr(
                            tuple(
                                result[
                                    "selected_ids"
                                ]
                            )
                        ),

                    "algorithm_selected_count":
                        result[
                            "selected_count"
                        ],

                    "algorithm_utility":
                        result["utility"],

                    "algorithm_privacy":
                        result["privacy"],

                    "algorithm_normalized_cognitive_cost":
                        result[
                            "normalized_cognitive_cost"
                        ],

                    "algorithm_cardinality_fraction":
                        result[
                            "cardinality_fraction"
                        ],

                    "algorithm_disclosure_cost":
                        result[
                            "disclosure_cost"
                        ],

                    "algorithm_runtime_seconds":
                        runtime_seconds,

                    "exact_status":
                        condition_row[
                            "exact_status"
                        ],

                    "exact_selected_ids":
                        condition_row[
                            "exact_selected_items"
                        ],

                    "exact_selected_count":
                        condition_row[
                            "exact_selected_count"
                        ],

                    "exact_disclosure_cost":
                        condition_row[
                            "exact_disclosure_cost"
                        ],
                }
            )


    complete_instance_ids.add(
        instance_id
    )


    if (
        len(complete_instance_ids)
        % 25
        == 0
    ):

        checkpoint_df = pd.DataFrame(
            baseline_rows
        )


        checkpoint_df.to_csv(
            BASELINE_CHECKPOINT_PATH,
            index=False,
        )


        elapsed_minutes = (
            time.perf_counter()
            -
            baseline_session_start
        ) / 60.0


        print(
            "Completed baseline instances: "
            f"{len(complete_instance_ids)}/"
            f"{EXPECTED_INSTANCES}"
            f" | rows={len(checkpoint_df)}"
            f" | elapsed={elapsed_minutes:.2f} min"
        )


# --------------------------------------------------
# Final baseline-only dataset
# --------------------------------------------------

baseline_df = pd.DataFrame(
    baseline_rows
)


baseline_df = (
    baseline_df
    .drop_duplicates(
        subset=[
            "instance_id",
            "tau",
            "weight_profile",
            "algorithm",
        ],
        keep="last",
    )
    .sort_values(
        [
            "n_items",
            "replicate",
            "regime",
            "tau",
            "weight_profile",
            "algorithm",
        ]
    )
    .reset_index(drop=True)
)


assert len(baseline_df) == EXPECTED_BASELINE_ROWS

assert (
    baseline_df["instance_id"].nunique()
    == EXPECTED_INSTANCES
)

assert (
    baseline_df
    .groupby("instance_id")
    .size()
    ==
    BASELINE_ROWS_PER_INSTANCE
).all()


baseline_df.to_csv(
    BASELINE_ONLY_PATH,
    index=False,
)


# --------------------------------------------------
# Add existing Dynamic Greedy results
# --------------------------------------------------

dynamic_greedy_df = pd.DataFrame(
    {
        "benchmark_version":
            "corrected_v2_existing_dynamic_greedy",

        "matched_group_id":
            reference_df[
                "matched_group_id"
            ],

        "instance_id":
            reference_df[
                "instance_id"
            ],

        "n_items":
            reference_df[
                "n_items"
            ],

        "replicate":
            reference_df[
                "replicate"
            ],

        "regime":
            reference_df[
                "regime"
            ],

        "matched_seed":
            reference_df[
                "matched_seed"
            ],

        "tau":
            reference_df[
                "tau"
            ],

        "weight_profile":
            reference_df[
                "weight_profile"
            ],

        "lambda_privacy":
            reference_df[
                "lambda_privacy"
            ],

        "lambda_cognitive":
            reference_df[
                "lambda_cognitive"
            ],

        "lambda_cardinality":
            reference_df[
                "lambda_cardinality"
            ],

        "algorithm":
            "dynamic_greedy",

        "algorithm_status":
            reference_df[
                "greedy_status"
            ],

        "algorithm_selected_ids":
            reference_df[
                "greedy_selected_items"
            ],

        "algorithm_selected_count":
            reference_df[
                "greedy_selected_count"
            ],

        "algorithm_utility":
            reference_df[
                "greedy_utility"
            ],

        "algorithm_privacy":
            reference_df[
                "greedy_privacy"
            ],

        "algorithm_normalized_cognitive_cost":
            reference_df[
                "greedy_normalized_cognitive_cost"
            ],

        "algorithm_cardinality_fraction":
            (
                reference_df[
                    "greedy_selected_count"
                ]
                /
                reference_df[
                    "n_items"
                ]
            ),

        "algorithm_disclosure_cost":
            reference_df[
                "greedy_disclosure_cost"
            ],

        "algorithm_runtime_seconds":
            reference_df[
                "greedy_runtime_seconds"
            ],

        "exact_status":
            reference_df[
                "exact_status"
            ],

        "exact_selected_ids":
            reference_df[
                "exact_selected_items"
            ],

        "exact_selected_count":
            reference_df[
                "exact_selected_count"
            ],

        "exact_disclosure_cost":
            reference_df[
                "exact_disclosure_cost"
            ],
    }
)


algorithm_long_df = pd.concat(
    [
        dynamic_greedy_df,
        baseline_df,
    ],
    ignore_index=True,
)


numeric_long_columns = [
    "n_items",
    "tau",
    "algorithm_selected_count",
    "algorithm_utility",
    "algorithm_privacy",
    "algorithm_normalized_cognitive_cost",
    "algorithm_cardinality_fraction",
    "algorithm_disclosure_cost",
    "algorithm_runtime_seconds",
    "exact_selected_count",
    "exact_disclosure_cost",
]


for column_name in numeric_long_columns:

    algorithm_long_df[column_name] = pd.to_numeric(
        algorithm_long_df[column_name],
        errors="coerce",
    )


# --------------------------------------------------
# Combined integrity audit
# --------------------------------------------------

assert len(
    algorithm_long_df
) == EXPECTED_COMBINED_ROWS


assert (
    algorithm_long_df[
        "matched_group_id"
    ].nunique()
    == 500
)


assert (
    algorithm_long_df[
        "instance_id"
    ].nunique()
    == EXPECTED_INSTANCES
)


algorithms_per_condition = (
    algorithm_long_df
    .groupby(
        [
            "instance_id",
            "tau",
            "weight_profile",
        ]
    )[
        "algorithm"
    ]
    .nunique()
)


assert (
    algorithms_per_condition
    ==
    len(ALGORITHM_ORDER)
).all()


duplicate_algorithm_rows = int(
    algorithm_long_df
    .duplicated(
        subset=[
            "instance_id",
            "tau",
            "weight_profile",
            "algorithm",
        ],
        keep=False,
    )
    .sum()
)


threshold_failures = int(
    (
        (
            algorithm_long_df[
                "algorithm_status"
            ]
            ==
            "FEASIBLE"
        )
        &
        (
            algorithm_long_df[
                "algorithm_utility"
            ]
            +
            1e-12
            <
            algorithm_long_df[
                "tau"
            ]
        )
    ).sum()
)


false_infeasible_rows = int(
    (
        (
            algorithm_long_df[
                "exact_status"
            ]
            ==
            "OPTIMAL"
        )
        &
        (
            algorithm_long_df[
                "algorithm_status"
            ]
            ==
            "INFEASIBLE"
        )
    ).sum()
)


false_feasible_rows = int(
    (
        (
            algorithm_long_df[
                "exact_status"
            ]
            ==
            "INFEASIBLE"
        )
        &
        (
            algorithm_long_df[
                "algorithm_status"
            ]
            ==
            "FEASIBLE"
        )
    ).sum()
)


assert duplicate_algorithm_rows == 0
assert threshold_failures == 0
assert false_infeasible_rows == 0
assert false_feasible_rows == 0


# --------------------------------------------------
# Exact-relative quality metrics
# --------------------------------------------------

algorithm_long_df[
    "is_comparable"
] = (
    (
        algorithm_long_df[
            "exact_status"
        ]
        ==
        "OPTIMAL"
    )
    &
    (
        algorithm_long_df[
            "algorithm_status"
        ]
        ==
        "FEASIBLE"
    )
)


comparable_mask = (
    algorithm_long_df[
        "is_comparable"
    ]
)


assert (
    algorithm_long_df.loc[
        comparable_mask,
        "exact_disclosure_cost",
    ]
    > 0.0
).all()


algorithm_long_df[
    "optimality_gap_percent"
] = np.nan


algorithm_long_df.loc[
    comparable_mask,
    "optimality_gap_percent",
] = (
    100.0
    *
    (
        algorithm_long_df.loc[
            comparable_mask,
            "algorithm_disclosure_cost",
        ]
        -
        algorithm_long_df.loc[
            comparable_mask,
            "exact_disclosure_cost",
        ]
    )
    /
    algorithm_long_df.loc[
        comparable_mask,
        "exact_disclosure_cost",
    ]
)


algorithm_long_df.loc[
    (
        algorithm_long_df[
            "optimality_gap_percent"
        ].abs()
        <
        1e-10
    ),
    "optimality_gap_percent",
] = 0.0


minimum_gap = float(
    algorithm_long_df.loc[
        comparable_mask,
        "optimality_gap_percent",
    ].min()
)


assert minimum_gap >= -1e-8


algorithm_long_df[
    "selected_count_difference"
] = (
    algorithm_long_df[
        "algorithm_selected_count"
    ]
    -
    algorithm_long_df[
        "exact_selected_count"
    ]
)


algorithm_long_df[
    "jaccard_similarity"
] = np.nan


comparable_indices = (
    algorithm_long_df.index[
        comparable_mask
    ]
)


algorithm_long_df.loc[
    comparable_indices,
    "jaccard_similarity",
] = [
    calculate_jaccard(
        algorithm_long_df.at[
            row_index,
            "algorithm_selected_ids",
        ],
        algorithm_long_df.at[
            row_index,
            "exact_selected_ids",
        ],
    )
    for row_index in comparable_indices
]


OPTIMAL_COST_TOLERANCE_PERCENT = 1e-10


algorithm_long_df[
    "algorithm_optimal_cost_numeric"
] = (
    algorithm_long_df[
        "optimality_gap_percent"
    ].abs()
    <=
    OPTIMAL_COST_TOLERANCE_PERCENT
).astype(int)


algorithm_long_df[
    "large_gap_numeric"
] = (
    algorithm_long_df[
        "optimality_gap_percent"
    ]
    > 10.0
).astype(int)


algorithm_long_df[
    "algorithm_more_items_numeric"
] = (
    algorithm_long_df[
        "selected_count_difference"
    ]
    > 0
).astype(int)


algorithm_long_df[
    "within_1_percent_numeric"
] = (
    algorithm_long_df[
        "optimality_gap_percent"
    ]
    <= 1.0
).astype(int)


algorithm_long_df[
    "within_5_percent_numeric"
] = (
    algorithm_long_df[
        "optimality_gap_percent"
    ]
    <= 5.0
).astype(int)


algorithm_long_df[
    "within_10_percent_numeric"
] = (
    algorithm_long_df[
        "optimality_gap_percent"
    ]
    <= 10.0
).astype(int)


algorithm_long_df.to_csv(
    BASELINE_LONG_PATH,
    index=False,
)


# --------------------------------------------------
# Descriptive algorithm summary
# --------------------------------------------------

comparable_algorithm_df = (
    algorithm_long_df[
        algorithm_long_df[
            "is_comparable"
        ]
    ]
    .copy()
)


descriptive_rows = []


for algorithm in ALGORITHM_ORDER:

    all_algorithm_df = (
        algorithm_long_df[
            algorithm_long_df[
                "algorithm"
            ]
            ==
            algorithm
        ]
    )


    current_df = (
        comparable_algorithm_df[
            comparable_algorithm_df[
                "algorithm"
            ]
            ==
            algorithm
        ]
    )


    gap_values = (
        current_df[
            "optimality_gap_percent"
        ]
        .to_numpy(dtype=float)
    )


    descriptive_rows.append(
        {
            "algorithm":
                algorithm,

            "total_rows":
                len(all_algorithm_df),

            "comparable_rows":
                len(current_df),

            "feasibility_percent":
                100.0
                *
                (
                    all_algorithm_df[
                        "algorithm_status"
                    ]
                    ==
                    "FEASIBLE"
                ).mean(),

            "mean_optimality_gap_percent":
                gap_values.mean(),

            "median_optimality_gap_percent":
                np.median(gap_values),

            "p90_optimality_gap_percent":
                np.quantile(
                    gap_values,
                    0.90,
                ),

            "p95_optimality_gap_percent":
                np.quantile(
                    gap_values,
                    0.95,
                ),

            "maximum_optimality_gap_percent":
                gap_values.max(),

            "exact_cost_recovery_percent":
                100.0
                *
                current_df[
                    "algorithm_optimal_cost_numeric"
                ].mean(),

            "mean_jaccard_similarity":
                current_df[
                    "jaccard_similarity"
                ].mean(),

            "large_gap_percent":
                100.0
                *
                current_df[
                    "large_gap_numeric"
                ].mean(),

            "more_items_percent":
                100.0
                *
                current_df[
                    "algorithm_more_items_numeric"
                ].mean(),

            "mean_selected_count_difference":
                current_df[
                    "selected_count_difference"
                ].mean(),

            "within_1_percent":
                100.0
                *
                current_df[
                    "within_1_percent_numeric"
                ].mean(),

            "within_5_percent":
                100.0
                *
                current_df[
                    "within_5_percent_numeric"
                ].mean(),

            "within_10_percent":
                100.0
                *
                current_df[
                    "within_10_percent_numeric"
                ].mean(),

            "median_runtime_milliseconds":
                1000.0
                *
                current_df[
                    "algorithm_runtime_seconds"
                ].median(),
        }
    )


baseline_descriptive_df = pd.DataFrame(
    descriptive_rows
)


baseline_descriptive_df[
    "algorithm"
] = pd.Categorical(
    baseline_descriptive_df[
        "algorithm"
    ],
    categories=ALGORITHM_ORDER,
    ordered=True,
)


baseline_descriptive_df = (
    baseline_descriptive_df
    .sort_values("algorithm")
    .reset_index(drop=True)
)


baseline_descriptive_df.to_csv(
    BASELINE_DESCRIPTIVE_PATH,
    index=False,
)


# --------------------------------------------------
# Matched-group bootstrap confidence intervals
# --------------------------------------------------

algorithm_cluster_df = (
    comparable_algorithm_df
    .groupby(
        [
            "matched_group_id",
            "n_items",
            "algorithm",
        ],
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "size",
        ),

        gap_sum=(
            "optimality_gap_percent",
            "sum",
        ),

        optimal_cost_sum=(
            "algorithm_optimal_cost_numeric",
            "sum",
        ),

        jaccard_sum=(
            "jaccard_similarity",
            "sum",
        ),

        large_gap_sum=(
            "large_gap_numeric",
            "sum",
        ),

        more_items_sum=(
            "algorithm_more_items_numeric",
            "sum",
        ),

        selected_count_difference_sum=(
            "selected_count_difference",
            "sum",
        ),

        within_1_percent_sum=(
            "within_1_percent_numeric",
            "sum",
        ),

        within_5_percent_sum=(
            "within_5_percent_numeric",
            "sum",
        ),

        within_10_percent_sum=(
            "within_10_percent_numeric",
            "sum",
        ),
    )
)


ALGORITHM_AGGREGATE_COLUMNS = [
    "comparable_rows",
    "gap_sum",
    "optimal_cost_sum",
    "jaccard_sum",
    "large_gap_sum",
    "more_items_sum",
    "selected_count_difference_sum",
    "within_1_percent_sum",
    "within_5_percent_sum",
    "within_10_percent_sum",
]


ALGORITHM_METRIC_NAMES = [
    "mean_optimality_gap_percent",
    "exact_cost_recovery_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "more_items_percent",
    "mean_selected_count_difference",
    "within_1_percent",
    "within_5_percent",
    "within_10_percent",
]



def calculate_algorithm_metrics(
    totals,
):

    comparable_rows = totals[0]


    return np.array(
        [
            totals[1]
            /
            comparable_rows,

            100.0
            *
            totals[2]
            /
            comparable_rows,

            totals[3]
            /
            comparable_rows,

            100.0
            *
            totals[4]
            /
            comparable_rows,

            100.0
            *
            totals[5]
            /
            comparable_rows,

            totals[6]
            /
            comparable_rows,

            100.0
            *
            totals[7]
            /
            comparable_rows,

            100.0
            *
            totals[8]
            /
            comparable_rows,

            100.0
            *
            totals[9]
            /
            comparable_rows,
        ],
        dtype=float,
    )


comparable_count_pivot = (
    algorithm_cluster_df
    .pivot(
        index="matched_group_id",
        columns="algorithm",
        values="comparable_rows",
    )
)


assert (
    comparable_count_pivot.nunique(
        axis=1
    )
    ==
    1
).all()


baseline_rng = np.random.default_rng(
    BASELINE_BOOTSTRAP_SEED
)


bootstrap_summary_rows = []
bootstrap_replicate_frames = []


for algorithm in ALGORITHM_ORDER:

    current_algorithm_df = (
        algorithm_cluster_df[
            algorithm_cluster_df[
                "algorithm"
            ]
            ==
            algorithm
        ]
    )


    observed_totals = (
        current_algorithm_df[
            ALGORITHM_AGGREGATE_COLUMNS
        ]
        .sum()
        .to_numpy(dtype=float)
    )


    observed_metrics = (
        calculate_algorithm_metrics(
            observed_totals
        )
    )


    size_arrays = []


    for _, current_size_df in (
        current_algorithm_df.groupby(
            "n_items",
            sort=True,
        )
    ):

        size_array = (
            current_size_df
            .sort_values(
                "matched_group_id"
            )[
                ALGORITHM_AGGREGATE_COLUMNS
            ]
            .to_numpy(dtype=float)
        )


        assert len(size_array) == 100


        size_arrays.append(
            size_array
        )


    bootstrap_estimates = np.empty(
        (
            BASELINE_BOOTSTRAP_REPLICATES,
            len(ALGORITHM_METRIC_NAMES),
        ),
        dtype=float,
    )


    for bootstrap_index in range(
        BASELINE_BOOTSTRAP_REPLICATES
    ):

        resampled_totals = np.zeros(
            len(
                ALGORITHM_AGGREGATE_COLUMNS
            ),
            dtype=float,
        )


        for size_array in size_arrays:

            sampled_indices = (
                baseline_rng.integers(
                    low=0,
                    high=len(size_array),
                    size=len(size_array),
                )
            )


            resampled_totals += (
                size_array[
                    sampled_indices
                ]
                .sum(axis=0)
            )


        bootstrap_estimates[
            bootstrap_index
        ] = calculate_algorithm_metrics(
            resampled_totals
        )


    confidence_lower = np.quantile(
        bootstrap_estimates,
        0.025,
        axis=0,
    )


    confidence_upper = np.quantile(
        bootstrap_estimates,
        0.975,
        axis=0,
    )


    standard_errors = (
        bootstrap_estimates.std(
            axis=0,
            ddof=1,
        )
    )


    for metric_index, metric_name in enumerate(
        ALGORITHM_METRIC_NAMES
    ):

        bootstrap_summary_rows.append(
            {
                "algorithm":
                    algorithm,

                "metric":
                    metric_name,

                "estimate":
                    observed_metrics[
                        metric_index
                    ],

                "bootstrap_standard_error":
                    standard_errors[
                        metric_index
                    ],

                "ci_95_lower":
                    confidence_lower[
                        metric_index
                    ],

                "ci_95_upper":
                    confidence_upper[
                        metric_index
                    ],

                "matched_groups":
                    500,

                "candidate_size_strata":
                    5,

                "bootstrap_replicates":
                    BASELINE_BOOTSTRAP_REPLICATES,

                "resampling_unit":
                    "matched_size_replicate_group",

                "resampling_design":
                    "stratified_by_candidate_set_size",

                "estimand":
                    "quality_conditional_on_feasibility",
            }
        )


    replicate_df = pd.DataFrame(
        bootstrap_estimates,
        columns=ALGORITHM_METRIC_NAMES,
    )


    replicate_df.insert(
        0,
        "bootstrap_replicate",
        np.arange(
            BASELINE_BOOTSTRAP_REPLICATES
        ),
    )


    replicate_df.insert(
        0,
        "algorithm",
        algorithm,
    )


    bootstrap_replicate_frames.append(
        replicate_df
    )


    print(
        "Completed algorithm bootstrap:",
        algorithm
    )


baseline_bootstrap_df = pd.DataFrame(
    bootstrap_summary_rows
)


baseline_bootstrap_replicates_df = pd.concat(
    bootstrap_replicate_frames,
    ignore_index=True,
)


baseline_bootstrap_df.to_csv(
    BASELINE_BOOTSTRAP_PATH,
    index=False,
)


baseline_bootstrap_replicates_df.to_csv(
    BASELINE_BOOTSTRAP_REPLICATES_PATH,
    index=False,
)


# --------------------------------------------------
# Compact CI display
# --------------------------------------------------

key_metrics = [
    "mean_optimality_gap_percent",
    "exact_cost_recovery_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "more_items_percent",
]


baseline_display_df = (
    baseline_bootstrap_df[
        baseline_bootstrap_df[
            "metric"
        ].isin(key_metrics)
    ]
    .copy()
)


baseline_display_df[
    "estimate_and_ci"
] = (
    baseline_display_df[
        "estimate"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    baseline_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    baseline_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


baseline_pivot_df = (
    baseline_display_df
    .pivot(
        index="algorithm",
        columns="metric",
        values="estimate_and_ci",
    )
    .reset_index()
)


baseline_pivot_df[
    "algorithm"
] = pd.Categorical(
    baseline_pivot_df[
        "algorithm"
    ],
    categories=ALGORITHM_ORDER,
    ordered=True,
)


baseline_pivot_df = (
    baseline_pivot_df
    .sort_values("algorithm")
    .reset_index(drop=True)
)


# --------------------------------------------------
# Print outputs
# --------------------------------------------------

execution_minutes = (
    time.perf_counter()
    -
    baseline_session_start
) / 60.0


print(
    "\n======================================"
)

print(
    "CORRECTED V2 BASELINE INTEGRITY AUDIT"
)

print(
    "======================================"
)


print(
    "Baseline rows:",
    len(baseline_df)
)

print(
    "Combined algorithm rows:",
    len(algorithm_long_df)
)

print(
    "Matched groups:",
    algorithm_long_df[
        "matched_group_id"
    ].nunique()
)

print(
    "Regime-specific instances:",
    algorithm_long_df[
        "instance_id"
    ].nunique()
)

print(
    "Duplicate algorithm-condition rows:",
    duplicate_algorithm_rows
)

print(
    "Threshold failures:",
    threshold_failures
)

print(
    "False infeasible rows:",
    false_infeasible_rows
)

print(
    "False feasible rows:",
    false_feasible_rows
)

print(
    "Minimum observed optimality gap:",
    f"{minimum_gap:.12f}"
)


print(
    "\n======================================"
)

print(
    "CORRECTED V2 ALGORITHM BASELINE RESULTS"
)

print(
    "======================================"
)


print(
    baseline_descriptive_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.4f}",
    )
)


print(
    "\n======================================"
)

print(
    "MATCHED-GROUP BOOTSTRAP 95% CIs"
)

print(
    "======================================"
)


print(
    baseline_pivot_df.to_string(
        index=False
    )
)


print(
    "\nExecution-session time:",
    f"{execution_minutes:.2f} minutes"
)


print(
    "\nBaseline-only results saved to:"
)

print(
    BASELINE_ONLY_PATH
)


print(
    "\nCombined algorithm dataset saved to:"
)

print(
    BASELINE_LONG_PATH
)


print(
    "\nDescriptive summary saved to:"
)

print(
    BASELINE_DESCRIPTIVE_PATH
)


print(
    "\nConfidence intervals saved to:"
)

print(
    BASELINE_BOOTSTRAP_PATH
)


print(
    "\nBootstrap replicates saved to:"
)

print(
    BASELINE_BOOTSTRAP_REPLICATES_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Step 117 — Corrected-v2 paired algorithm contrasts
# --------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------
# Configuration
# --------------------------------------------------

ALGORITHM_CONTRAST_BOOTSTRAP_REPLICATES = 5000

ALGORITHM_CONTRAST_BOOTSTRAP_SEED = (
    globals().get(
        "GLOBAL_SEED",
        2026,
    )
    + 117
)


if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


ALGORITHM_LONG_PATH = (
    RESULTS_DIR
    / "corrected_v2_algorithm_comparison_long.csv"
)


ALGORITHM_CONTRAST_PATH = (
    TABLES_DIR
    / "corrected_v2_paired_algorithm_bootstrap_contrasts.csv"
)


ALGORITHM_CONTRAST_REPLICATES_PATH = (
    RESULTS_DIR
    / "corrected_v2_paired_algorithm_bootstrap_contrast_replicates.csv"
)


ALGORITHM_ORDER = [
    "dynamic_greedy",
    "utility_first",
    "lowest_cost_first",
    "static_utility_cost_ratio",
]


# Each comparison is:
# Dynamic Greedy minus deterministic baseline.
ALGORITHM_COMPARISONS = [
    (
        "dynamic_greedy",
        "utility_first",
    ),
    (
        "dynamic_greedy",
        "lowest_cost_first",
    ),
    (
        "dynamic_greedy",
        "static_utility_cost_ratio",
    ),
]


ALGORITHM_CONTRAST_METRICS = [
    "mean_optimality_gap_percent",
    "exact_cost_recovery_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "more_items_percent",
    "mean_selected_count_difference",
    "within_1_percent",
    "within_5_percent",
    "within_10_percent",
]


# --------------------------------------------------
# Load combined algorithm dataset
# --------------------------------------------------

algorithm_contrast_df = pd.read_csv(
    ALGORITHM_LONG_PATH,
    low_memory=False,
)


required_columns = [
    "matched_group_id",
    "instance_id",
    "n_items",
    "algorithm",
    "exact_status",
    "algorithm_status",
    "optimality_gap_percent",
    "jaccard_similarity",
    "selected_count_difference",
]


missing_columns = [
    column_name
    for column_name in required_columns
    if column_name
    not in algorithm_contrast_df.columns
]


if missing_columns:

    raise KeyError(
        "Missing required columns: "
        + ", ".join(
            missing_columns
        )
    )


for column_name in [
    "n_items",
    "optimality_gap_percent",
    "jaccard_similarity",
    "selected_count_difference",
]:

    algorithm_contrast_df[
        column_name
    ] = pd.to_numeric(
        algorithm_contrast_df[
            column_name
        ],
        errors="coerce",
    )


# --------------------------------------------------
# Restrict to comparable feasible conditions
# --------------------------------------------------

algorithm_contrast_df[
    "is_comparable"
] = (
    (
        algorithm_contrast_df[
            "exact_status"
        ]
        ==
        "OPTIMAL"
    )
    &
    (
        algorithm_contrast_df[
            "algorithm_status"
        ]
        ==
        "FEASIBLE"
    )
)


comparable_algorithm_df = (
    algorithm_contrast_df[
        algorithm_contrast_df[
            "is_comparable"
        ]
    ]
    .copy()
)


assert len(
    comparable_algorithm_df
) == (
    23784
    *
    len(
        ALGORITHM_ORDER
    )
)


# --------------------------------------------------
# Reporting indicators
# --------------------------------------------------

OPTIMAL_COST_TOLERANCE_PERCENT = 1e-10


comparable_algorithm_df[
    "exact_cost_recovery_numeric"
] = (
    comparable_algorithm_df[
        "optimality_gap_percent"
    ]
    .abs()
    <=
    OPTIMAL_COST_TOLERANCE_PERCENT
).astype(int)


comparable_algorithm_df[
    "large_gap_numeric"
] = (
    comparable_algorithm_df[
        "optimality_gap_percent"
    ]
    > 10.0
).astype(int)


comparable_algorithm_df[
    "more_items_numeric"
] = (
    comparable_algorithm_df[
        "selected_count_difference"
    ]
    > 0
).astype(int)


comparable_algorithm_df[
    "within_1_percent_numeric"
] = (
    comparable_algorithm_df[
        "optimality_gap_percent"
    ]
    <= 1.0
).astype(int)


comparable_algorithm_df[
    "within_5_percent_numeric"
] = (
    comparable_algorithm_df[
        "optimality_gap_percent"
    ]
    <= 5.0
).astype(int)


comparable_algorithm_df[
    "within_10_percent_numeric"
] = (
    comparable_algorithm_df[
        "optimality_gap_percent"
    ]
    <= 10.0
).astype(int)


# --------------------------------------------------
# One sufficient-statistics row per matched group,
# candidate size, and algorithm
# --------------------------------------------------

algorithm_cluster_df = (
    comparable_algorithm_df
    .groupby(
        [
            "matched_group_id",
            "n_items",
            "algorithm",
        ],
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "size",
        ),

        gap_sum=(
            "optimality_gap_percent",
            "sum",
        ),

        exact_cost_recovery_sum=(
            "exact_cost_recovery_numeric",
            "sum",
        ),

        jaccard_sum=(
            "jaccard_similarity",
            "sum",
        ),

        large_gap_sum=(
            "large_gap_numeric",
            "sum",
        ),

        more_items_sum=(
            "more_items_numeric",
            "sum",
        ),

        selected_count_difference_sum=(
            "selected_count_difference",
            "sum",
        ),

        within_1_percent_sum=(
            "within_1_percent_numeric",
            "sum",
        ),

        within_5_percent_sum=(
            "within_5_percent_numeric",
            "sum",
        ),

        within_10_percent_sum=(
            "within_10_percent_numeric",
            "sum",
        ),
    )
)


ALGORITHM_AGGREGATE_COLUMNS = [
    "comparable_rows",
    "gap_sum",
    "exact_cost_recovery_sum",
    "jaccard_sum",
    "large_gap_sum",
    "more_items_sum",
    "selected_count_difference_sum",
    "within_1_percent_sum",
    "within_5_percent_sum",
    "within_10_percent_sum",
]


def calculate_algorithm_metrics(
    totals: np.ndarray,
) -> np.ndarray:
    """
    Convert matched-group sufficient statistics into
    Exact-relative algorithm-quality metrics.
    """

    comparable_rows = totals[0]


    return np.array(
        [
            totals[1]
            /
            comparable_rows,

            100.0
            *
            totals[2]
            /
            comparable_rows,

            totals[3]
            /
            comparable_rows,

            100.0
            *
            totals[4]
            /
            comparable_rows,

            100.0
            *
            totals[5]
            /
            comparable_rows,

            totals[6]
            /
            comparable_rows,

            100.0
            *
            totals[7]
            /
            comparable_rows,

            100.0
            *
            totals[8]
            /
            comparable_rows,

            100.0
            *
            totals[9]
            /
            comparable_rows,
        ],
        dtype=float,
    )


# --------------------------------------------------
# Validate matched pairing
# --------------------------------------------------

assert (
    algorithm_cluster_df[
        "matched_group_id"
    ].nunique()
    == 500
)


algorithms_per_group = (
    algorithm_cluster_df
    .groupby(
        "matched_group_id"
    )[
        "algorithm"
    ]
    .apply(set)
)


expected_algorithm_set = set(
    ALGORITHM_ORDER
)


assert (
    algorithms_per_group
    .apply(
        lambda value:
            value
            ==
            expected_algorithm_set
    )
    .all()
)


comparable_count_pivot = (
    algorithm_cluster_df
    .pivot(
        index="matched_group_id",
        columns="algorithm",
        values="comparable_rows",
    )
)


assert (
    comparable_count_pivot.nunique(
        axis=1
    )
    == 1
).all()


# --------------------------------------------------
# Construct aligned arrays within each candidate size
# --------------------------------------------------

paired_size_strata = []


for n_items in sorted(
    algorithm_cluster_df[
        "n_items"
    ].unique()
):

    algorithm_arrays = {}
    reference_group_ids = None


    for algorithm in ALGORITHM_ORDER:

        current_df = (
            algorithm_cluster_df[
                (
                    algorithm_cluster_df[
                        "n_items"
                    ]
                    ==
                    n_items
                )
                &
                (
                    algorithm_cluster_df[
                        "algorithm"
                    ]
                    ==
                    algorithm
                )
            ]
            .sort_values(
                "matched_group_id"
            )
            .reset_index(
                drop=True
            )
        )


        assert len(
            current_df
        ) == 100


        current_group_ids = (
            current_df[
                "matched_group_id"
            ].tolist()
        )


        if reference_group_ids is None:

            reference_group_ids = (
                current_group_ids
            )

        else:

            assert (
                current_group_ids
                ==
                reference_group_ids
            )


        algorithm_arrays[
            algorithm
        ] = (
            current_df[
                ALGORITHM_AGGREGATE_COLUMNS
            ]
            .to_numpy(
                dtype=float
            )
        )


    paired_size_strata.append(
        {
            "n_items":
                int(
                    n_items
                ),

            "matched_group_ids":
                reference_group_ids,

            "algorithm_arrays":
                algorithm_arrays,
        }
    )


# --------------------------------------------------
# Observed metrics
# --------------------------------------------------

observed_metrics_by_algorithm = {}


for algorithm in ALGORITHM_ORDER:

    algorithm_totals = (
        algorithm_cluster_df[
            algorithm_cluster_df[
                "algorithm"
            ]
            ==
            algorithm
        ][
            ALGORITHM_AGGREGATE_COLUMNS
        ]
        .sum()
        .to_numpy(
            dtype=float
        )
    )


    observed_metrics_by_algorithm[
        algorithm
    ] = calculate_algorithm_metrics(
        algorithm_totals
    )


# --------------------------------------------------
# Shared paired bootstrap across all algorithms
# --------------------------------------------------

algorithm_contrast_rng = np.random.default_rng(
    ALGORITHM_CONTRAST_BOOTSTRAP_SEED
)


bootstrap_metrics_by_algorithm = {
    algorithm: np.empty(
        (
            ALGORITHM_CONTRAST_BOOTSTRAP_REPLICATES,
            len(
                ALGORITHM_CONTRAST_METRICS
            ),
        ),
        dtype=float,
    )
    for algorithm in ALGORITHM_ORDER
}


for bootstrap_index in range(
    ALGORITHM_CONTRAST_BOOTSTRAP_REPLICATES
):

    totals_by_algorithm = {
        algorithm: np.zeros(
            len(
                ALGORITHM_AGGREGATE_COLUMNS
            ),
            dtype=float,
        )
        for algorithm in ALGORITHM_ORDER
    }


    for size_stratum in paired_size_strata:

        number_of_groups = len(
            size_stratum[
                "matched_group_ids"
            ]
        )


        # Shared indices preserve pairing among all
        # algorithms on the same generated instances.
        sampled_indices = (
            algorithm_contrast_rng.integers(
                low=0,
                high=number_of_groups,
                size=number_of_groups,
            )
        )


        for algorithm in ALGORITHM_ORDER:

            algorithm_array = (
                size_stratum[
                    "algorithm_arrays"
                ][
                    algorithm
                ]
            )


            totals_by_algorithm[
                algorithm
            ] += (
                algorithm_array[
                    sampled_indices
                ]
                .sum(
                    axis=0
                )
            )


    for algorithm in ALGORITHM_ORDER:

        bootstrap_metrics_by_algorithm[
            algorithm
        ][
            bootstrap_index
        ] = calculate_algorithm_metrics(
            totals_by_algorithm[
                algorithm
            ]
        )


# --------------------------------------------------
# Calculate Dynamic-Greedy contrasts
# --------------------------------------------------

contrast_rows = []
contrast_replicate_frames = []


for first_algorithm, second_algorithm in (
    ALGORITHM_COMPARISONS
):

    contrast_name = (
        f"{first_algorithm}"
        f" minus "
        f"{second_algorithm}"
    )


    observed_difference = (
        observed_metrics_by_algorithm[
            first_algorithm
        ]
        -
        observed_metrics_by_algorithm[
            second_algorithm
        ]
    )


    bootstrap_differences = (
        bootstrap_metrics_by_algorithm[
            first_algorithm
        ]
        -
        bootstrap_metrics_by_algorithm[
            second_algorithm
        ]
    )


    standard_errors = (
        bootstrap_differences.std(
            axis=0,
            ddof=1,
        )
    )


    confidence_lower = np.quantile(
        bootstrap_differences,
        0.025,
        axis=0,
    )


    confidence_upper = np.quantile(
        bootstrap_differences,
        0.975,
        axis=0,
    )


    for metric_index, metric_name in enumerate(
        ALGORITHM_CONTRAST_METRICS
    ):

        contrast_rows.append(
            {
                "first_algorithm":
                    first_algorithm,

                "second_algorithm":
                    second_algorithm,

                "contrast":
                    contrast_name,

                "metric":
                    metric_name,

                "estimated_difference":
                    observed_difference[
                        metric_index
                    ],

                "bootstrap_standard_error":
                    standard_errors[
                        metric_index
                    ],

                "ci_95_lower":
                    confidence_lower[
                        metric_index
                    ],

                "ci_95_upper":
                    confidence_upper[
                        metric_index
                    ],

                "ci_excludes_zero":
                    (
                        confidence_lower[
                            metric_index
                        ]
                        > 0.0
                    )
                    or
                    (
                        confidence_upper[
                            metric_index
                        ]
                        < 0.0
                    ),

                "matched_groups":
                    500,

                "candidate_size_strata":
                    5,

                "bootstrap_replicates":
                    ALGORITHM_CONTRAST_BOOTSTRAP_REPLICATES,

                "resampling_unit":
                    "matched_size_replicate_group",

                "resampling_design":
                    (
                        "paired_across_algorithms_"
                        "stratified_by_candidate_set_size"
                    ),

                "estimand":
                    "quality_conditional_on_feasibility",

                "multiple_comparison_adjustment":
                    "none",
            }
        )


    replicate_df = pd.DataFrame(
        bootstrap_differences,
        columns=ALGORITHM_CONTRAST_METRICS,
    )


    replicate_df.insert(
        0,
        "bootstrap_replicate",
        np.arange(
            ALGORITHM_CONTRAST_BOOTSTRAP_REPLICATES
        ),
    )


    replicate_df.insert(
        0,
        "contrast",
        contrast_name,
    )


    contrast_replicate_frames.append(
        replicate_df
    )


    print(
        "Completed paired algorithm contrast:",
        contrast_name
    )


# --------------------------------------------------
# Save outputs
# --------------------------------------------------

algorithm_contrast_results_df = pd.DataFrame(
    contrast_rows
)


algorithm_contrast_replicates_df = pd.concat(
    contrast_replicate_frames,
    ignore_index=True,
)


algorithm_contrast_results_df.to_csv(
    ALGORITHM_CONTRAST_PATH,
    index=False,
)


algorithm_contrast_replicates_df.to_csv(
    ALGORITHM_CONTRAST_REPLICATES_PATH,
    index=False,
)


# --------------------------------------------------
# Readable display
# --------------------------------------------------

key_metrics = [
    "mean_optimality_gap_percent",
    "exact_cost_recovery_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "more_items_percent",
    "mean_selected_count_difference",
]


algorithm_contrast_display_df = (
    algorithm_contrast_results_df[
        algorithm_contrast_results_df[
            "metric"
        ].isin(
            key_metrics
        )
    ]
    .copy()
)


algorithm_contrast_display_df[
    "difference_and_ci"
] = (
    algorithm_contrast_display_df[
        "estimated_difference"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    algorithm_contrast_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    algorithm_contrast_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


print(
    "\n======================================"
)

print(
    "CORRECTED V2 PAIRED ALGORITHM CONTRASTS"
)

print(
    "======================================"
)


print(
    algorithm_contrast_display_df[
        [
            "contrast",
            "metric",
            "difference_and_ci",
            "ci_excludes_zero",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\nPaired algorithm contrasts saved to:"
)

print(
    ALGORITHM_CONTRAST_PATH
)


print(
    "\nPaired contrast replicates saved to:"
)

print(
    ALGORITHM_CONTRAST_REPLICATES_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Step 118 — Dynamic Greedy versus static utility-cost
# ratio baseline within each structural regime
# --------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------
# Configuration
# --------------------------------------------------

REGIME_ALGORITHM_BOOTSTRAP_REPLICATES = 5000

REGIME_ALGORITHM_BOOTSTRAP_SEED = (
    globals().get(
        "GLOBAL_SEED",
        2026,
    )
    + 118
)


if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


ALGORITHM_LONG_PATH = (
    RESULTS_DIR
    / "corrected_v2_algorithm_comparison_long.csv"
)


REGIME_ALGORITHM_DESCRIPTIVE_PATH = (
    TABLES_DIR
    / "corrected_v2_algorithm_by_regime_descriptive_summary.csv"
)


REGIME_ALGORITHM_CONTRAST_PATH = (
    TABLES_DIR
    / (
        "corrected_v2_dynamic_vs_static_ratio_"
        "by_regime_bootstrap_contrasts.csv"
    )
)


REGIME_ALGORITHM_REPLICATES_PATH = (
    RESULTS_DIR
    / (
        "corrected_v2_dynamic_vs_static_ratio_"
        "by_regime_bootstrap_replicates.csv"
    )
)


REGIME_ORDER = [
    "independent",
    "utility_burden_alignment",
    "utility_burden_conflict",
]


ALGORITHM_ORDER = [
    "dynamic_greedy",
    "utility_first",
    "lowest_cost_first",
    "static_utility_cost_ratio",
]


FIRST_ALGORITHM = "dynamic_greedy"
SECOND_ALGORITHM = "static_utility_cost_ratio"


REGIME_ALGORITHM_METRICS = [
    "mean_optimality_gap_percent",
    "exact_cost_recovery_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "more_items_percent",
    "mean_selected_count_difference",
    "within_1_percent",
    "within_5_percent",
    "within_10_percent",
]


# --------------------------------------------------
# Load combined algorithm dataset
# --------------------------------------------------

regime_algorithm_df = pd.read_csv(
    ALGORITHM_LONG_PATH,
    low_memory=False,
)


required_columns = [
    "matched_group_id",
    "instance_id",
    "n_items",
    "regime",
    "algorithm",
    "exact_status",
    "algorithm_status",
    "optimality_gap_percent",
    "jaccard_similarity",
    "selected_count_difference",
]


missing_columns = [
    column_name
    for column_name in required_columns
    if column_name not in regime_algorithm_df.columns
]


if missing_columns:

    raise KeyError(
        "Missing required columns: "
        + ", ".join(
            missing_columns
        )
    )


for column_name in [
    "n_items",
    "optimality_gap_percent",
    "jaccard_similarity",
    "selected_count_difference",
]:

    regime_algorithm_df[
        column_name
    ] = pd.to_numeric(
        regime_algorithm_df[
            column_name
        ],
        errors="coerce",
    )


# --------------------------------------------------
# Restrict analysis to comparable feasible rows
# --------------------------------------------------

regime_algorithm_df[
    "is_comparable"
] = (
    (
        regime_algorithm_df[
            "exact_status"
        ]
        == "OPTIMAL"
    )
    &
    (
        regime_algorithm_df[
            "algorithm_status"
        ]
        == "FEASIBLE"
    )
)


comparable_df = (
    regime_algorithm_df[
        regime_algorithm_df[
            "is_comparable"
        ]
    ]
    .copy()
)


assert len(
    comparable_df
) == (
    23784
    *
    len(
        ALGORITHM_ORDER
    )
)


# --------------------------------------------------
# Reporting indicators
# --------------------------------------------------

OPTIMAL_COST_TOLERANCE_PERCENT = 1e-10


comparable_df[
    "exact_cost_recovery_numeric"
] = (
    comparable_df[
        "optimality_gap_percent"
    ]
    .abs()
    <= OPTIMAL_COST_TOLERANCE_PERCENT
).astype(int)


comparable_df[
    "large_gap_numeric"
] = (
    comparable_df[
        "optimality_gap_percent"
    ]
    > 10.0
).astype(int)


comparable_df[
    "more_items_numeric"
] = (
    comparable_df[
        "selected_count_difference"
    ]
    > 0
).astype(int)


comparable_df[
    "within_1_percent_numeric"
] = (
    comparable_df[
        "optimality_gap_percent"
    ]
    <= 1.0
).astype(int)


comparable_df[
    "within_5_percent_numeric"
] = (
    comparable_df[
        "optimality_gap_percent"
    ]
    <= 5.0
).astype(int)


comparable_df[
    "within_10_percent_numeric"
] = (
    comparable_df[
        "optimality_gap_percent"
    ]
    <= 10.0
).astype(int)


# --------------------------------------------------
# Descriptive algorithm results within each regime
# --------------------------------------------------

descriptive_rows = []


for regime in REGIME_ORDER:

    for algorithm in ALGORITHM_ORDER:

        current_df = (
            comparable_df[
                (
                    comparable_df[
                        "regime"
                    ]
                    == regime
                )
                &
                (
                    comparable_df[
                        "algorithm"
                    ]
                    == algorithm
                )
            ]
        )


        gap_values = (
            current_df[
                "optimality_gap_percent"
            ]
            .to_numpy(
                dtype=float
            )
        )


        descriptive_rows.append(
            {
                "regime":
                    regime,

                "algorithm":
                    algorithm,

                "matched_groups":
                    current_df[
                        "matched_group_id"
                    ].nunique(),

                "comparable_rows":
                    len(
                        current_df
                    ),

                "mean_optimality_gap_percent":
                    gap_values.mean(),

                "median_optimality_gap_percent":
                    np.median(
                        gap_values
                    ),

                "p90_optimality_gap_percent":
                    np.quantile(
                        gap_values,
                        0.90,
                    ),

                "p95_optimality_gap_percent":
                    np.quantile(
                        gap_values,
                        0.95,
                    ),

                "exact_cost_recovery_percent":
                    100.0
                    *
                    current_df[
                        "exact_cost_recovery_numeric"
                    ].mean(),

                "mean_jaccard_similarity":
                    current_df[
                        "jaccard_similarity"
                    ].mean(),

                "large_gap_percent":
                    100.0
                    *
                    current_df[
                        "large_gap_numeric"
                    ].mean(),

                "more_items_percent":
                    100.0
                    *
                    current_df[
                        "more_items_numeric"
                    ].mean(),

                "mean_selected_count_difference":
                    current_df[
                        "selected_count_difference"
                    ].mean(),

                "within_1_percent":
                    100.0
                    *
                    current_df[
                        "within_1_percent_numeric"
                    ].mean(),

                "within_5_percent":
                    100.0
                    *
                    current_df[
                        "within_5_percent_numeric"
                    ].mean(),

                "within_10_percent":
                    100.0
                    *
                    current_df[
                        "within_10_percent_numeric"
                    ].mean(),
            }
        )


regime_algorithm_descriptive_df = pd.DataFrame(
    descriptive_rows
)


regime_algorithm_descriptive_df[
    "regime"
] = pd.Categorical(
    regime_algorithm_descriptive_df[
        "regime"
    ],
    categories=REGIME_ORDER,
    ordered=True,
)


regime_algorithm_descriptive_df[
    "algorithm"
] = pd.Categorical(
    regime_algorithm_descriptive_df[
        "algorithm"
    ],
    categories=ALGORITHM_ORDER,
    ordered=True,
)


regime_algorithm_descriptive_df = (
    regime_algorithm_descriptive_df
    .sort_values(
        [
            "regime",
            "algorithm",
        ]
    )
    .reset_index(
        drop=True
    )
)


regime_algorithm_descriptive_df.to_csv(
    REGIME_ALGORITHM_DESCRIPTIVE_PATH,
    index=False,
)


# --------------------------------------------------
# Sufficient statistics per matched group, regime,
# candidate-set size, and algorithm
# --------------------------------------------------

cluster_df = (
    comparable_df
    .groupby(
        [
            "matched_group_id",
            "n_items",
            "regime",
            "algorithm",
        ],
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "size",
        ),

        gap_sum=(
            "optimality_gap_percent",
            "sum",
        ),

        exact_cost_recovery_sum=(
            "exact_cost_recovery_numeric",
            "sum",
        ),

        jaccard_sum=(
            "jaccard_similarity",
            "sum",
        ),

        large_gap_sum=(
            "large_gap_numeric",
            "sum",
        ),

        more_items_sum=(
            "more_items_numeric",
            "sum",
        ),

        selected_count_difference_sum=(
            "selected_count_difference",
            "sum",
        ),

        within_1_percent_sum=(
            "within_1_percent_numeric",
            "sum",
        ),

        within_5_percent_sum=(
            "within_5_percent_numeric",
            "sum",
        ),

        within_10_percent_sum=(
            "within_10_percent_numeric",
            "sum",
        ),
    )
)


AGGREGATE_COLUMNS = [
    "comparable_rows",
    "gap_sum",
    "exact_cost_recovery_sum",
    "jaccard_sum",
    "large_gap_sum",
    "more_items_sum",
    "selected_count_difference_sum",
    "within_1_percent_sum",
    "within_5_percent_sum",
    "within_10_percent_sum",
]


def calculate_quality_metrics(
    totals: np.ndarray,
) -> np.ndarray:
    """
    Convert sufficient-statistic totals into
    Exact-relative algorithm-quality metrics.
    """

    comparable_rows = totals[0]


    return np.array(
        [
            totals[1]
            /
            comparable_rows,

            100.0
            *
            totals[2]
            /
            comparable_rows,

            totals[3]
            /
            comparable_rows,

            100.0
            *
            totals[4]
            /
            comparable_rows,

            100.0
            *
            totals[5]
            /
            comparable_rows,

            totals[6]
            /
            comparable_rows,

            100.0
            *
            totals[7]
            /
            comparable_rows,

            100.0
            *
            totals[8]
            /
            comparable_rows,

            100.0
            *
            totals[9]
            /
            comparable_rows,
        ],
        dtype=float,
    )


# --------------------------------------------------
# Validate matched algorithm support
# --------------------------------------------------

groups_by_regime_size_algorithm = (
    cluster_df
    .groupby(
        [
            "regime",
            "n_items",
            "algorithm",
        ]
    )[
        "matched_group_id"
    ]
    .nunique()
)


assert (
    groups_by_regime_size_algorithm
    == 100
).all()


comparison_cluster_df = (
    cluster_df[
        cluster_df[
            "algorithm"
        ].isin(
            [
                FIRST_ALGORITHM,
                SECOND_ALGORITHM,
            ]
        )
    ]
    .copy()
)


comparable_count_pivot = (
    comparison_cluster_df
    .pivot(
        index=[
            "matched_group_id",
            "regime",
        ],
        columns="algorithm",
        values="comparable_rows",
    )
)


assert (
    comparable_count_pivot.nunique(
        axis=1
    )
    == 1
).all()


# --------------------------------------------------
# Paired bootstrap within each structural regime
# --------------------------------------------------

bootstrap_rng = np.random.default_rng(
    REGIME_ALGORITHM_BOOTSTRAP_SEED
)


contrast_rows = []
replicate_frames = []


for regime in REGIME_ORDER:

    paired_size_strata = []


    for n_items in sorted(
        comparison_cluster_df[
            "n_items"
        ].unique()
    ):

        first_df = (
            comparison_cluster_df[
                (
                    comparison_cluster_df[
                        "regime"
                    ]
                    == regime
                )
                &
                (
                    comparison_cluster_df[
                        "n_items"
                    ]
                    == n_items
                )
                &
                (
                    comparison_cluster_df[
                        "algorithm"
                    ]
                    == FIRST_ALGORITHM
                )
            ]
            .sort_values(
                "matched_group_id"
            )
            .reset_index(
                drop=True
            )
        )


        second_df = (
            comparison_cluster_df[
                (
                    comparison_cluster_df[
                        "regime"
                    ]
                    == regime
                )
                &
                (
                    comparison_cluster_df[
                        "n_items"
                    ]
                    == n_items
                )
                &
                (
                    comparison_cluster_df[
                        "algorithm"
                    ]
                    == SECOND_ALGORITHM
                )
            ]
            .sort_values(
                "matched_group_id"
            )
            .reset_index(
                drop=True
            )
        )


        assert len(
            first_df
        ) == 100


        assert len(
            second_df
        ) == 100


        assert (
            first_df[
                "matched_group_id"
            ].tolist()
            ==
            second_df[
                "matched_group_id"
            ].tolist()
        )


        paired_size_strata.append(
            {
                "n_items":
                    int(
                        n_items
                    ),

                "matched_group_ids":
                    first_df[
                        "matched_group_id"
                    ].tolist(),

                "first_array":
                    first_df[
                        AGGREGATE_COLUMNS
                    ]
                    .to_numpy(
                        dtype=float
                    ),

                "second_array":
                    second_df[
                        AGGREGATE_COLUMNS
                    ]
                    .to_numpy(
                        dtype=float
                    ),
            }
        )


    observed_first_totals = np.zeros(
        len(
            AGGREGATE_COLUMNS
        ),
        dtype=float,
    )


    observed_second_totals = np.zeros(
        len(
            AGGREGATE_COLUMNS
        ),
        dtype=float,
    )


    for size_stratum in paired_size_strata:

        observed_first_totals += (
            size_stratum[
                "first_array"
            ]
            .sum(
                axis=0
            )
        )


        observed_second_totals += (
            size_stratum[
                "second_array"
            ]
            .sum(
                axis=0
            )
        )


    observed_difference = (
        calculate_quality_metrics(
            observed_first_totals
        )
        -
        calculate_quality_metrics(
            observed_second_totals
        )
    )


    bootstrap_differences = np.empty(
        (
            REGIME_ALGORITHM_BOOTSTRAP_REPLICATES,
            len(
                REGIME_ALGORITHM_METRICS
            ),
        ),
        dtype=float,
    )


    for bootstrap_index in range(
        REGIME_ALGORITHM_BOOTSTRAP_REPLICATES
    ):

        resampled_first_totals = np.zeros(
            len(
                AGGREGATE_COLUMNS
            ),
            dtype=float,
        )


        resampled_second_totals = np.zeros(
            len(
                AGGREGATE_COLUMNS
            ),
            dtype=float,
        )


        for size_stratum in paired_size_strata:

            number_of_groups = len(
                size_stratum[
                    "matched_group_ids"
                ]
            )


            # Shared indices preserve pairing between
            # Dynamic Greedy and the static baseline.
            sampled_indices = (
                bootstrap_rng.integers(
                    low=0,
                    high=number_of_groups,
                    size=number_of_groups,
                )
            )


            resampled_first_totals += (
                size_stratum[
                    "first_array"
                ][
                    sampled_indices
                ]
                .sum(
                    axis=0
                )
            )


            resampled_second_totals += (
                size_stratum[
                    "second_array"
                ][
                    sampled_indices
                ]
                .sum(
                    axis=0
                )
            )


        bootstrap_differences[
            bootstrap_index
        ] = (
            calculate_quality_metrics(
                resampled_first_totals
            )
            -
            calculate_quality_metrics(
                resampled_second_totals
            )
        )


    standard_errors = (
        bootstrap_differences.std(
            axis=0,
            ddof=1,
        )
    )


    confidence_lower = np.quantile(
        bootstrap_differences,
        0.025,
        axis=0,
    )


    confidence_upper = np.quantile(
        bootstrap_differences,
        0.975,
        axis=0,
    )


    contrast_name = (
        f"{FIRST_ALGORITHM}"
        f" minus "
        f"{SECOND_ALGORITHM}"
    )


    for metric_index, metric_name in enumerate(
        REGIME_ALGORITHM_METRICS
    ):

        contrast_rows.append(
            {
                "regime":
                    regime,

                "first_algorithm":
                    FIRST_ALGORITHM,

                "second_algorithm":
                    SECOND_ALGORITHM,

                "contrast":
                    contrast_name,

                "metric":
                    metric_name,

                "estimated_difference":
                    observed_difference[
                        metric_index
                    ],

                "bootstrap_standard_error":
                    standard_errors[
                        metric_index
                    ],

                "ci_95_lower":
                    confidence_lower[
                        metric_index
                    ],

                "ci_95_upper":
                    confidence_upper[
                        metric_index
                    ],

                "ci_excludes_zero":
                    (
                        confidence_lower[
                            metric_index
                        ]
                        > 0.0
                    )
                    or
                    (
                        confidence_upper[
                            metric_index
                        ]
                        < 0.0
                    ),

                "matched_groups":
                    500,

                "matched_groups_per_size":
                    100,

                "candidate_size_strata":
                    5,

                "bootstrap_replicates":
                    REGIME_ALGORITHM_BOOTSTRAP_REPLICATES,

                "resampling_unit":
                    "matched_size_replicate_group",

                "resampling_design":
                    (
                        "paired_across_algorithms_"
                        "stratified_by_candidate_set_size"
                    ),

                "estimand":
                    (
                        "algorithm_difference_"
                        "conditional_on_feasibility"
                    ),

                "multiple_comparison_adjustment":
                    "none",
            }
        )


    replicate_df = pd.DataFrame(
        bootstrap_differences,
        columns=REGIME_ALGORITHM_METRICS,
    )


    replicate_df.insert(
        0,
        "bootstrap_replicate",
        np.arange(
            REGIME_ALGORITHM_BOOTSTRAP_REPLICATES
        ),
    )


    replicate_df.insert(
        0,
        "regime",
        regime,
    )


    replicate_frames.append(
        replicate_df
    )


    print(
        "Completed Dynamic-versus-static contrast:",
        regime
    )


# --------------------------------------------------
# Save outputs
# --------------------------------------------------

regime_algorithm_contrast_df = pd.DataFrame(
    contrast_rows
)


regime_algorithm_replicates_df = pd.concat(
    replicate_frames,
    ignore_index=True,
)


regime_algorithm_contrast_df.to_csv(
    REGIME_ALGORITHM_CONTRAST_PATH,
    index=False,
)


regime_algorithm_replicates_df.to_csv(
    REGIME_ALGORITHM_REPLICATES_PATH,
    index=False,
)


# --------------------------------------------------
# Readable displays
# --------------------------------------------------

descriptive_display_df = (
    regime_algorithm_descriptive_df[
        regime_algorithm_descriptive_df[
            "algorithm"
        ].isin(
            [
                FIRST_ALGORITHM,
                SECOND_ALGORITHM,
            ]
        )
    ]
    .copy()
)


key_metrics = [
    "mean_optimality_gap_percent",
    "exact_cost_recovery_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "more_items_percent",
    "mean_selected_count_difference",
]


contrast_display_df = (
    regime_algorithm_contrast_df[
        regime_algorithm_contrast_df[
            "metric"
        ].isin(
            key_metrics
        )
    ]
    .copy()
)


contrast_display_df[
    "difference_and_ci"
] = (
    contrast_display_df[
        "estimated_difference"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    contrast_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    contrast_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


print(
    "\n======================================"
)

print(
    "DYNAMIC AND STATIC-RATIO RESULTS BY REGIME"
)

print(
    "======================================"
)


print(
    descriptive_display_df[
        [
            "regime",
            "algorithm",
            "comparable_rows",
            "mean_optimality_gap_percent",
            "exact_cost_recovery_percent",
            "mean_jaccard_similarity",
            "large_gap_percent",
            "more_items_percent",
            "mean_selected_count_difference",
        ]
    ].to_string(
        index=False,
        float_format=lambda value:
            f"{value:.4f}",
    )
)


print(
    "\n======================================"
)

print(
    "PAIRED DYNAMIC-MINUS-STATIC CONTRASTS BY REGIME"
)

print(
    "======================================"
)


print(
    contrast_display_df[
        [
            "regime",
            "metric",
            "difference_and_ci",
            "ci_excludes_zero",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\nAlgorithm-by-regime descriptive summary saved to:"
)

print(
    REGIME_ALGORITHM_DESCRIPTIVE_PATH
)


print(
    "\nRegime-specific paired contrasts saved to:"
)

print(
    REGIME_ALGORITHM_CONTRAST_PATH
)


print(
    "\nBootstrap replicates saved to:"
)

print(
    REGIME_ALGORITHM_REPLICATES_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Step 119 — Dynamic Greedy versus static utility-cost
# ratio baseline within each sufficiency threshold
# --------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------
# Configuration
# --------------------------------------------------

THRESHOLD_ALGORITHM_BOOTSTRAP_REPLICATES = 5000

THRESHOLD_ALGORITHM_BOOTSTRAP_SEED = (
    globals().get(
        "GLOBAL_SEED",
        2026,
    )
    + 119
)


if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


ALGORITHM_LONG_PATH = (
    RESULTS_DIR
    / "corrected_v2_algorithm_comparison_long.csv"
)


THRESHOLD_ALGORITHM_DESCRIPTIVE_PATH = (
    TABLES_DIR
    / "corrected_v2_dynamic_static_ratio_by_threshold_descriptive.csv"
)


THRESHOLD_ALGORITHM_SUPPORT_PATH = (
    TABLES_DIR
    / "corrected_v2_dynamic_static_ratio_threshold_support_counts.csv"
)


THRESHOLD_ALGORITHM_CONTRAST_PATH = (
    TABLES_DIR
    / "corrected_v2_dynamic_vs_static_ratio_by_threshold_bootstrap_contrasts.csv"
)


THRESHOLD_ALGORITHM_REPLICATES_PATH = (
    RESULTS_DIR
    / "corrected_v2_dynamic_vs_static_ratio_by_threshold_bootstrap_replicates.csv"
)


THRESHOLD_ORDER = [
    0.70,
    0.80,
    0.90,
    0.95,
]


FIRST_ALGORITHM = "dynamic_greedy"
SECOND_ALGORITHM = "static_utility_cost_ratio"


THRESHOLD_METRIC_NAMES = [
    "mean_optimality_gap_percent",
    "exact_cost_recovery_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "more_items_percent",
    "mean_selected_count_difference",
    "within_1_percent",
    "within_5_percent",
    "within_10_percent",
]


AGGREGATE_COLUMNS = [
    "comparable_rows",
    "gap_sum",
    "exact_cost_recovery_sum",
    "jaccard_sum",
    "large_gap_sum",
    "more_items_sum",
    "selected_count_difference_sum",
    "within_1_percent_sum",
    "within_5_percent_sum",
    "within_10_percent_sum",
]


# --------------------------------------------------
# Load combined algorithm benchmark
# --------------------------------------------------

threshold_algorithm_df = pd.read_csv(
    ALGORITHM_LONG_PATH,
    low_memory=False,
)


required_columns = [
    "matched_group_id",
    "instance_id",
    "n_items",
    "tau",
    "algorithm",
    "exact_status",
    "algorithm_status",
    "optimality_gap_percent",
    "jaccard_similarity",
    "selected_count_difference",
]


missing_columns = [
    column_name
    for column_name in required_columns
    if column_name not in threshold_algorithm_df.columns
]


if missing_columns:

    raise KeyError(
        "Missing required columns: "
        + ", ".join(
            missing_columns
        )
    )


for column_name in [
    "n_items",
    "tau",
    "optimality_gap_percent",
    "jaccard_similarity",
    "selected_count_difference",
]:

    threshold_algorithm_df[
        column_name
    ] = pd.to_numeric(
        threshold_algorithm_df[
            column_name
        ],
        errors="coerce",
    )


threshold_algorithm_df = (
    threshold_algorithm_df[
        threshold_algorithm_df[
            "algorithm"
        ].isin(
            [
                FIRST_ALGORITHM,
                SECOND_ALGORITHM,
            ]
        )
    ]
    .copy()
)


assert (
    sorted(
        threshold_algorithm_df[
            "tau"
        ]
        .dropna()
        .unique()
        .tolist()
    )
    ==
    THRESHOLD_ORDER
)


# --------------------------------------------------
# Restrict to Exact-feasible comparable conditions
# --------------------------------------------------

threshold_algorithm_df[
    "is_comparable"
] = (
    (
        threshold_algorithm_df[
            "exact_status"
        ]
        == "OPTIMAL"
    )
    &
    (
        threshold_algorithm_df[
            "algorithm_status"
        ]
        == "FEASIBLE"
    )
)


comparable_df = (
    threshold_algorithm_df[
        threshold_algorithm_df[
            "is_comparable"
        ]
    ]
    .copy()
)


# Both algorithms must have the same feasible support.
support_per_condition = (
    comparable_df
    .groupby(
        [
            "instance_id",
            "tau",
        ]
    )[
        "algorithm"
    ]
    .nunique()
)


assert (
    support_per_condition
    == 2
).all()


# --------------------------------------------------
# Reporting indicators
# --------------------------------------------------

OPTIMAL_COST_TOLERANCE_PERCENT = 1e-10


comparable_df[
    "exact_cost_recovery_numeric"
] = (
    comparable_df[
        "optimality_gap_percent"
    ]
    .abs()
    <=
    OPTIMAL_COST_TOLERANCE_PERCENT
).astype(int)


comparable_df[
    "large_gap_numeric"
] = (
    comparable_df[
        "optimality_gap_percent"
    ]
    > 10.0
).astype(int)


comparable_df[
    "more_items_numeric"
] = (
    comparable_df[
        "selected_count_difference"
    ]
    > 0
).astype(int)


comparable_df[
    "within_1_percent_numeric"
] = (
    comparable_df[
        "optimality_gap_percent"
    ]
    <= 1.0
).astype(int)


comparable_df[
    "within_5_percent_numeric"
] = (
    comparable_df[
        "optimality_gap_percent"
    ]
    <= 5.0
).astype(int)


comparable_df[
    "within_10_percent_numeric"
] = (
    comparable_df[
        "optimality_gap_percent"
    ]
    <= 10.0
).astype(int)


# --------------------------------------------------
# Descriptive results by threshold and algorithm
# --------------------------------------------------

descriptive_rows = []


for tau in THRESHOLD_ORDER:

    for algorithm in [
        FIRST_ALGORITHM,
        SECOND_ALGORITHM,
    ]:

        current_df = (
            comparable_df[
                (
                    np.isclose(
                        comparable_df[
                            "tau"
                        ],
                        tau,
                    )
                )
                &
                (
                    comparable_df[
                        "algorithm"
                    ]
                    == algorithm
                )
            ]
        )


        gap_values = (
            current_df[
                "optimality_gap_percent"
            ]
            .to_numpy(
                dtype=float
            )
        )


        descriptive_rows.append(
            {
                "tau":
                    tau,

                "algorithm":
                    algorithm,

                "matched_groups":
                    current_df[
                        "matched_group_id"
                    ].nunique(),

                "comparable_rows":
                    len(
                        current_df
                    ),

                "mean_optimality_gap_percent":
                    gap_values.mean(),

                "median_optimality_gap_percent":
                    np.median(
                        gap_values
                    ),

                "p90_optimality_gap_percent":
                    np.quantile(
                        gap_values,
                        0.90,
                    ),

                "p95_optimality_gap_percent":
                    np.quantile(
                        gap_values,
                        0.95,
                    ),

                "exact_cost_recovery_percent":
                    100.0
                    *
                    current_df[
                        "exact_cost_recovery_numeric"
                    ].mean(),

                "mean_jaccard_similarity":
                    current_df[
                        "jaccard_similarity"
                    ].mean(),

                "large_gap_percent":
                    100.0
                    *
                    current_df[
                        "large_gap_numeric"
                    ].mean(),

                "more_items_percent":
                    100.0
                    *
                    current_df[
                        "more_items_numeric"
                    ].mean(),

                "mean_selected_count_difference":
                    current_df[
                        "selected_count_difference"
                    ].mean(),

                "within_1_percent":
                    100.0
                    *
                    current_df[
                        "within_1_percent_numeric"
                    ].mean(),

                "within_5_percent":
                    100.0
                    *
                    current_df[
                        "within_5_percent_numeric"
                    ].mean(),

                "within_10_percent":
                    100.0
                    *
                    current_df[
                        "within_10_percent_numeric"
                    ].mean(),
            }
        )


threshold_descriptive_df = pd.DataFrame(
    descriptive_rows
)


threshold_descriptive_df.to_csv(
    THRESHOLD_ALGORITHM_DESCRIPTIVE_PATH,
    index=False,
)


# --------------------------------------------------
# Sufficient statistics per matched group, size,
# threshold, and algorithm
# --------------------------------------------------

cluster_df = (
    comparable_df
    .groupby(
        [
            "matched_group_id",
            "n_items",
            "tau",
            "algorithm",
        ],
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "size",
        ),

        gap_sum=(
            "optimality_gap_percent",
            "sum",
        ),

        exact_cost_recovery_sum=(
            "exact_cost_recovery_numeric",
            "sum",
        ),

        jaccard_sum=(
            "jaccard_similarity",
            "sum",
        ),

        large_gap_sum=(
            "large_gap_numeric",
            "sum",
        ),

        more_items_sum=(
            "more_items_numeric",
            "sum",
        ),

        selected_count_difference_sum=(
            "selected_count_difference",
            "sum",
        ),

        within_1_percent_sum=(
            "within_1_percent_numeric",
            "sum",
        ),

        within_5_percent_sum=(
            "within_5_percent_numeric",
            "sum",
        ),

        within_10_percent_sum=(
            "within_10_percent_numeric",
            "sum",
        ),
    )
)


def calculate_threshold_metrics(
    totals: np.ndarray,
) -> np.ndarray:
    """
    Convert matched-group sufficient statistics into
    Exact-relative algorithm-quality metrics.
    """

    comparable_rows = totals[0]


    if comparable_rows <= 0:

        return np.full(
            len(
                THRESHOLD_METRIC_NAMES
            ),
            np.nan,
        )


    return np.array(
        [
            totals[1]
            /
            comparable_rows,

            100.0
            *
            totals[2]
            /
            comparable_rows,

            totals[3]
            /
            comparable_rows,

            100.0
            *
            totals[4]
            /
            comparable_rows,

            100.0
            *
            totals[5]
            /
            comparable_rows,

            totals[6]
            /
            comparable_rows,

            100.0
            *
            totals[7]
            /
            comparable_rows,

            100.0
            *
            totals[8]
            /
            comparable_rows,

            100.0
            *
            totals[9]
            /
            comparable_rows,
        ],
        dtype=float,
    )


# --------------------------------------------------
# Validate and record threshold-specific support
# --------------------------------------------------

support_rows = []


for tau in THRESHOLD_ORDER:

    for n_items in sorted(
        cluster_df[
            "n_items"
        ].unique()
    ):

        first_support_df = (
            cluster_df[
                (
                    np.isclose(
                        cluster_df[
                            "tau"
                        ],
                        tau,
                    )
                )
                &
                (
                    cluster_df[
                        "n_items"
                    ]
                    == n_items
                )
                &
                (
                    cluster_df[
                        "algorithm"
                    ]
                    == FIRST_ALGORITHM
                )
            ]
            .sort_values(
                "matched_group_id"
            )
        )


        second_support_df = (
            cluster_df[
                (
                    np.isclose(
                        cluster_df[
                            "tau"
                        ],
                        tau,
                    )
                )
                &
                (
                    cluster_df[
                        "n_items"
                    ]
                    == n_items
                )
                &
                (
                    cluster_df[
                        "algorithm"
                    ]
                    == SECOND_ALGORITHM
                )
            ]
            .sort_values(
                "matched_group_id"
            )
        )


        assert (
            first_support_df[
                "matched_group_id"
            ].tolist()
            ==
            second_support_df[
                "matched_group_id"
            ].tolist()
        )


        support_rows.append(
            {
                "tau":
                    tau,

                "n_items":
                    int(
                        n_items
                    ),

                "feasible_matched_groups":
                    len(
                        first_support_df
                    ),

                "comparable_rows_per_algorithm":
                    int(
                        first_support_df[
                            "comparable_rows"
                        ].sum()
                    ),
            }
        )


threshold_support_df = pd.DataFrame(
    support_rows
)


threshold_support_df.to_csv(
    THRESHOLD_ALGORITHM_SUPPORT_PATH,
    index=False,
)


# --------------------------------------------------
# Paired bootstrap independently within each
# threshold and candidate-size stratum
# --------------------------------------------------

bootstrap_rng = np.random.default_rng(
    THRESHOLD_ALGORITHM_BOOTSTRAP_SEED
)


contrast_rows = []
replicate_frames = []


for tau in THRESHOLD_ORDER:

    paired_size_strata = []


    for n_items in sorted(
        cluster_df[
            "n_items"
        ].unique()
    ):

        first_df = (
            cluster_df[
                (
                    np.isclose(
                        cluster_df[
                            "tau"
                        ],
                        tau,
                    )
                )
                &
                (
                    cluster_df[
                        "n_items"
                    ]
                    == n_items
                )
                &
                (
                    cluster_df[
                        "algorithm"
                    ]
                    == FIRST_ALGORITHM
                )
            ]
            .sort_values(
                "matched_group_id"
            )
            .reset_index(
                drop=True
            )
        )


        second_df = (
            cluster_df[
                (
                    np.isclose(
                        cluster_df[
                            "tau"
                        ],
                        tau,
                    )
                )
                &
                (
                    cluster_df[
                        "n_items"
                    ]
                    == n_items
                )
                &
                (
                    cluster_df[
                        "algorithm"
                    ]
                    == SECOND_ALGORITHM
                )
            ]
            .sort_values(
                "matched_group_id"
            )
            .reset_index(
                drop=True
            )
        )


        assert len(
            first_df
        ) == len(
            second_df
        )


        assert len(
            first_df
        ) > 0


        assert (
            first_df[
                "matched_group_id"
            ].tolist()
            ==
            second_df[
                "matched_group_id"
            ].tolist()
        )


        assert np.array_equal(
            first_df[
                "comparable_rows"
            ].to_numpy(),
            second_df[
                "comparable_rows"
            ].to_numpy(),
        )


        paired_size_strata.append(
            {
                "n_items":
                    int(
                        n_items
                    ),

                "matched_group_ids":
                    first_df[
                        "matched_group_id"
                    ].tolist(),

                "first_array":
                    first_df[
                        AGGREGATE_COLUMNS
                    ]
                    .to_numpy(
                        dtype=float
                    ),

                "second_array":
                    second_df[
                        AGGREGATE_COLUMNS
                    ]
                    .to_numpy(
                        dtype=float
                    ),
            }
        )


    observed_first_totals = np.zeros(
        len(
            AGGREGATE_COLUMNS
        ),
        dtype=float,
    )


    observed_second_totals = np.zeros(
        len(
            AGGREGATE_COLUMNS
        ),
        dtype=float,
    )


    for size_stratum in paired_size_strata:

        observed_first_totals += (
            size_stratum[
                "first_array"
            ]
            .sum(
                axis=0
            )
        )


        observed_second_totals += (
            size_stratum[
                "second_array"
            ]
            .sum(
                axis=0
            )
        )


    observed_difference = (
        calculate_threshold_metrics(
            observed_first_totals
        )
        -
        calculate_threshold_metrics(
            observed_second_totals
        )
    )


    bootstrap_differences = np.empty(
        (
            THRESHOLD_ALGORITHM_BOOTSTRAP_REPLICATES,
            len(
                THRESHOLD_METRIC_NAMES
            ),
        ),
        dtype=float,
    )


    for bootstrap_index in range(
        THRESHOLD_ALGORITHM_BOOTSTRAP_REPLICATES
    ):

        first_totals = np.zeros(
            len(
                AGGREGATE_COLUMNS
            ),
            dtype=float,
        )


        second_totals = np.zeros(
            len(
                AGGREGATE_COLUMNS
            ),
            dtype=float,
        )


        for size_stratum in paired_size_strata:

            number_of_groups = len(
                size_stratum[
                    "matched_group_ids"
                ]
            )


            # Shared indices preserve the paired
            # algorithm comparison.
            sampled_indices = (
                bootstrap_rng.integers(
                    low=0,
                    high=number_of_groups,
                    size=number_of_groups,
                )
            )


            first_totals += (
                size_stratum[
                    "first_array"
                ][
                    sampled_indices
                ]
                .sum(
                    axis=0
                )
            )


            second_totals += (
                size_stratum[
                    "second_array"
                ][
                    sampled_indices
                ]
                .sum(
                    axis=0
                )
            )


        bootstrap_differences[
            bootstrap_index
        ] = (
            calculate_threshold_metrics(
                first_totals
            )
            -
            calculate_threshold_metrics(
                second_totals
            )
        )


    standard_errors = (
        bootstrap_differences.std(
            axis=0,
            ddof=1,
        )
    )


    confidence_lower = np.quantile(
        bootstrap_differences,
        0.025,
        axis=0,
    )


    confidence_upper = np.quantile(
        bootstrap_differences,
        0.975,
        axis=0,
    )


    total_supported_groups = sum(
        len(
            size_stratum[
                "matched_group_ids"
            ]
        )
        for size_stratum in paired_size_strata
    )


    for metric_index, metric_name in enumerate(
        THRESHOLD_METRIC_NAMES
    ):

        contrast_rows.append(
            {
                "tau":
                    tau,

                "first_algorithm":
                    FIRST_ALGORITHM,

                "second_algorithm":
                    SECOND_ALGORITHM,

                "contrast":
                    (
                        f"{FIRST_ALGORITHM}"
                        f" minus "
                        f"{SECOND_ALGORITHM}"
                    ),

                "metric":
                    metric_name,

                "estimated_difference":
                    observed_difference[
                        metric_index
                    ],

                "bootstrap_standard_error":
                    standard_errors[
                        metric_index
                    ],

                "ci_95_lower":
                    confidence_lower[
                        metric_index
                    ],

                "ci_95_upper":
                    confidence_upper[
                        metric_index
                    ],

                "ci_excludes_zero":
                    (
                        confidence_lower[
                            metric_index
                        ]
                        > 0.0
                    )
                    or
                    (
                        confidence_upper[
                            metric_index
                        ]
                        < 0.0
                    ),

                "supported_matched_groups":
                    total_supported_groups,

                "candidate_size_strata":
                    len(
                        paired_size_strata
                    ),

                "bootstrap_replicates":
                    THRESHOLD_ALGORITHM_BOOTSTRAP_REPLICATES,

                "resampling_unit":
                    "matched_size_replicate_group",

                "resampling_design":
                    (
                        "paired_across_algorithms_"
                        "within_threshold_and_candidate_size"
                    ),

                "estimand":
                    (
                        "algorithm_difference_"
                        "conditional_on_threshold_feasibility"
                    ),

                "multiple_comparison_adjustment":
                    "none",
            }
        )


    replicate_df = pd.DataFrame(
        bootstrap_differences,
        columns=THRESHOLD_METRIC_NAMES,
    )


    replicate_df.insert(
        0,
        "bootstrap_replicate",
        np.arange(
            THRESHOLD_ALGORITHM_BOOTSTRAP_REPLICATES
        ),
    )


    replicate_df.insert(
        0,
        "tau",
        tau,
    )


    replicate_frames.append(
        replicate_df
    )


    print(
        "Completed Dynamic-versus-static threshold:",
        tau
    )


# --------------------------------------------------
# Save outputs
# --------------------------------------------------

threshold_contrast_df = pd.DataFrame(
    contrast_rows
)


threshold_replicates_df = pd.concat(
    replicate_frames,
    ignore_index=True,
)


threshold_contrast_df.to_csv(
    THRESHOLD_ALGORITHM_CONTRAST_PATH,
    index=False,
)


threshold_replicates_df.to_csv(
    THRESHOLD_ALGORITHM_REPLICATES_PATH,
    index=False,
)


# --------------------------------------------------
# Readable displays
# --------------------------------------------------

key_metrics = [
    "mean_optimality_gap_percent",
    "exact_cost_recovery_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "more_items_percent",
    "mean_selected_count_difference",
]


contrast_display_df = (
    threshold_contrast_df[
        threshold_contrast_df[
            "metric"
        ].isin(
            key_metrics
        )
    ]
    .copy()
)


contrast_display_df[
    "difference_and_ci"
] = (
    contrast_display_df[
        "estimated_difference"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    contrast_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    contrast_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


print(
    "\n======================================"
)

print(
    "DYNAMIC AND STATIC-RATIO RESULTS BY THRESHOLD"
)

print(
    "======================================"
)


print(
    threshold_descriptive_df[
        [
            "tau",
            "algorithm",
            "matched_groups",
            "comparable_rows",
            "mean_optimality_gap_percent",
            "exact_cost_recovery_percent",
            "mean_jaccard_similarity",
            "large_gap_percent",
            "more_items_percent",
            "mean_selected_count_difference",
        ]
    ].to_string(
        index=False,
        float_format=lambda value:
            f"{value:.4f}",
    )
)


print(
    "\n======================================"
)

print(
    "THRESHOLD FEASIBLE MATCHED-GROUP SUPPORT"
)

print(
    "======================================"
)


print(
    threshold_support_df.to_string(
        index=False
    )
)


print(
    "\n======================================"
)

print(
    "PAIRED DYNAMIC-MINUS-STATIC CONTRASTS BY THRESHOLD"
)

print(
    "======================================"
)


print(
    contrast_display_df[
        [
            "tau",
            "metric",
            "difference_and_ci",
            "ci_excludes_zero",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\nThreshold descriptive results saved to:"
)

print(
    THRESHOLD_ALGORITHM_DESCRIPTIVE_PATH
)


print(
    "\nThreshold support counts saved to:"
)

print(
    THRESHOLD_ALGORITHM_SUPPORT_PATH
)


print(
    "\nThreshold paired contrasts saved to:"
)

print(
    THRESHOLD_ALGORITHM_CONTRAST_PATH
)


print(
    "\nBootstrap replicates saved to:"
)

print(
    THRESHOLD_ALGORITHM_REPLICATES_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Step 120 — Dynamic Greedy versus static utility-cost
# ratio baseline within each candidate-set size
# --------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------
# Configuration
# --------------------------------------------------

SIZE_ALGORITHM_BOOTSTRAP_REPLICATES = 5000

SIZE_ALGORITHM_BOOTSTRAP_SEED = (
    globals().get(
        "GLOBAL_SEED",
        2026,
    )
    + 120
)


if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


ALGORITHM_LONG_PATH = (
    RESULTS_DIR
    / "corrected_v2_algorithm_comparison_long.csv"
)


SIZE_ALGORITHM_DESCRIPTIVE_PATH = (
    TABLES_DIR
    / "corrected_v2_dynamic_static_ratio_by_candidate_size_descriptive.csv"
)


SIZE_ALGORITHM_SUPPORT_PATH = (
    TABLES_DIR
    / "corrected_v2_dynamic_static_ratio_candidate_size_support.csv"
)


SIZE_ALGORITHM_CONTRAST_PATH = (
    TABLES_DIR
    / "corrected_v2_dynamic_vs_static_ratio_by_candidate_size_contrasts.csv"
)


SIZE_ALGORITHM_REPLICATES_PATH = (
    RESULTS_DIR
    / "corrected_v2_dynamic_vs_static_ratio_by_candidate_size_bootstrap_replicates.csv"
)


CANDIDATE_SIZE_ORDER = [
    8,
    10,
    12,
    14,
    16,
]


FIRST_ALGORITHM = "dynamic_greedy"

SECOND_ALGORITHM = (
    "static_utility_cost_ratio"
)


SIZE_METRIC_NAMES = [
    "mean_optimality_gap_percent",
    "exact_cost_recovery_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "more_items_percent",
    "mean_selected_count_difference",
    "within_1_percent",
    "within_5_percent",
    "within_10_percent",
]


AGGREGATE_COLUMNS = [
    "comparable_rows",
    "gap_sum",
    "exact_cost_recovery_sum",
    "jaccard_sum",
    "large_gap_sum",
    "more_items_sum",
    "selected_count_difference_sum",
    "within_1_percent_sum",
    "within_5_percent_sum",
    "within_10_percent_sum",
]


# --------------------------------------------------
# Load combined algorithm benchmark
# --------------------------------------------------

size_algorithm_df = pd.read_csv(
    ALGORITHM_LONG_PATH,
    low_memory=False,
)


required_columns = [
    "matched_group_id",
    "instance_id",
    "n_items",
    "algorithm",
    "exact_status",
    "algorithm_status",
    "optimality_gap_percent",
    "jaccard_similarity",
    "selected_count_difference",
]


missing_columns = [
    column_name
    for column_name in required_columns
    if column_name not in size_algorithm_df.columns
]


if missing_columns:

    raise KeyError(
        "Missing required columns: "
        + ", ".join(
            missing_columns
        )
    )


for column_name in [
    "n_items",
    "optimality_gap_percent",
    "jaccard_similarity",
    "selected_count_difference",
]:

    size_algorithm_df[
        column_name
    ] = pd.to_numeric(
        size_algorithm_df[
            column_name
        ],
        errors="coerce",
    )


size_algorithm_df = (
    size_algorithm_df[
        size_algorithm_df[
            "algorithm"
        ].isin(
            [
                FIRST_ALGORITHM,
                SECOND_ALGORITHM,
            ]
        )
    ]
    .copy()
)


assert (
    sorted(
        size_algorithm_df[
            "n_items"
        ]
        .dropna()
        .unique()
        .astype(int)
        .tolist()
    )
    ==
    CANDIDATE_SIZE_ORDER
)


# --------------------------------------------------
# Restrict to Exact-feasible comparable conditions
# --------------------------------------------------

size_algorithm_df[
    "is_comparable"
] = (
    (
        size_algorithm_df[
            "exact_status"
        ]
        == "OPTIMAL"
    )
    &
    (
        size_algorithm_df[
            "algorithm_status"
        ]
        == "FEASIBLE"
    )
)


comparable_df = (
    size_algorithm_df[
        size_algorithm_df[
            "is_comparable"
        ]
    ]
    .copy()
)


assert len(
    comparable_df
) == (
    23784
    *
    2
)


# --------------------------------------------------
# Reporting indicators
# --------------------------------------------------

OPTIMAL_COST_TOLERANCE_PERCENT = 1e-10


comparable_df[
    "exact_cost_recovery_numeric"
] = (
    comparable_df[
        "optimality_gap_percent"
    ]
    .abs()
    <=
    OPTIMAL_COST_TOLERANCE_PERCENT
).astype(int)


comparable_df[
    "large_gap_numeric"
] = (
    comparable_df[
        "optimality_gap_percent"
    ]
    > 10.0
).astype(int)


comparable_df[
    "more_items_numeric"
] = (
    comparable_df[
        "selected_count_difference"
    ]
    > 0
).astype(int)


comparable_df[
    "within_1_percent_numeric"
] = (
    comparable_df[
        "optimality_gap_percent"
    ]
    <= 1.0
).astype(int)


comparable_df[
    "within_5_percent_numeric"
] = (
    comparable_df[
        "optimality_gap_percent"
    ]
    <= 5.0
).astype(int)


comparable_df[
    "within_10_percent_numeric"
] = (
    comparable_df[
        "optimality_gap_percent"
    ]
    <= 10.0
).astype(int)


# --------------------------------------------------
# Descriptive results by candidate size and algorithm
# --------------------------------------------------

descriptive_rows = []


for n_items in CANDIDATE_SIZE_ORDER:

    for algorithm in [
        FIRST_ALGORITHM,
        SECOND_ALGORITHM,
    ]:

        current_df = (
            comparable_df[
                (
                    comparable_df[
                        "n_items"
                    ]
                    == n_items
                )
                &
                (
                    comparable_df[
                        "algorithm"
                    ]
                    == algorithm
                )
            ]
        )


        gap_values = (
            current_df[
                "optimality_gap_percent"
            ]
            .to_numpy(
                dtype=float
            )
        )


        descriptive_rows.append(
            {
                "n_items":
                    n_items,

                "algorithm":
                    algorithm,

                "matched_groups":
                    current_df[
                        "matched_group_id"
                    ].nunique(),

                "comparable_rows":
                    len(
                        current_df
                    ),

                "mean_optimality_gap_percent":
                    gap_values.mean(),

                "median_optimality_gap_percent":
                    np.median(
                        gap_values
                    ),

                "p90_optimality_gap_percent":
                    np.quantile(
                        gap_values,
                        0.90,
                    ),

                "p95_optimality_gap_percent":
                    np.quantile(
                        gap_values,
                        0.95,
                    ),

                "exact_cost_recovery_percent":
                    100.0
                    *
                    current_df[
                        "exact_cost_recovery_numeric"
                    ].mean(),

                "mean_jaccard_similarity":
                    current_df[
                        "jaccard_similarity"
                    ].mean(),

                "large_gap_percent":
                    100.0
                    *
                    current_df[
                        "large_gap_numeric"
                    ].mean(),

                "more_items_percent":
                    100.0
                    *
                    current_df[
                        "more_items_numeric"
                    ].mean(),

                "mean_selected_count_difference":
                    current_df[
                        "selected_count_difference"
                    ].mean(),

                "within_1_percent":
                    100.0
                    *
                    current_df[
                        "within_1_percent_numeric"
                    ].mean(),

                "within_5_percent":
                    100.0
                    *
                    current_df[
                        "within_5_percent_numeric"
                    ].mean(),

                "within_10_percent":
                    100.0
                    *
                    current_df[
                        "within_10_percent_numeric"
                    ].mean(),
            }
        )


size_descriptive_df = pd.DataFrame(
    descriptive_rows
)


size_descriptive_df.to_csv(
    SIZE_ALGORITHM_DESCRIPTIVE_PATH,
    index=False,
)


# --------------------------------------------------
# Sufficient statistics per matched group, size,
# and algorithm
# --------------------------------------------------

cluster_df = (
    comparable_df
    .groupby(
        [
            "matched_group_id",
            "n_items",
            "algorithm",
        ],
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "size",
        ),

        gap_sum=(
            "optimality_gap_percent",
            "sum",
        ),

        exact_cost_recovery_sum=(
            "exact_cost_recovery_numeric",
            "sum",
        ),

        jaccard_sum=(
            "jaccard_similarity",
            "sum",
        ),

        large_gap_sum=(
            "large_gap_numeric",
            "sum",
        ),

        more_items_sum=(
            "more_items_numeric",
            "sum",
        ),

        selected_count_difference_sum=(
            "selected_count_difference",
            "sum",
        ),

        within_1_percent_sum=(
            "within_1_percent_numeric",
            "sum",
        ),

        within_5_percent_sum=(
            "within_5_percent_numeric",
            "sum",
        ),

        within_10_percent_sum=(
            "within_10_percent_numeric",
            "sum",
        ),
    )
)


def calculate_size_metrics(
    totals: np.ndarray,
) -> np.ndarray:
    """
    Convert matched-group sufficient statistics into
    Exact-relative algorithm-quality metrics.
    """

    comparable_rows = totals[0]


    if comparable_rows <= 0:

        return np.full(
            len(
                SIZE_METRIC_NAMES
            ),
            np.nan,
        )


    return np.array(
        [
            totals[1]
            /
            comparable_rows,

            100.0
            *
            totals[2]
            /
            comparable_rows,

            totals[3]
            /
            comparable_rows,

            100.0
            *
            totals[4]
            /
            comparable_rows,

            100.0
            *
            totals[5]
            /
            comparable_rows,

            totals[6]
            /
            comparable_rows,

            100.0
            *
            totals[7]
            /
            comparable_rows,

            100.0
            *
            totals[8]
            /
            comparable_rows,

            100.0
            *
            totals[9]
            /
            comparable_rows,
        ],
        dtype=float,
    )


# --------------------------------------------------
# Validate paired support and create support table
# --------------------------------------------------

support_rows = []


for n_items in CANDIDATE_SIZE_ORDER:

    first_support_df = (
        cluster_df[
            (
                cluster_df[
                    "n_items"
                ]
                == n_items
            )
            &
            (
                cluster_df[
                    "algorithm"
                ]
                == FIRST_ALGORITHM
            )
        ]
        .sort_values(
            "matched_group_id"
        )
        .reset_index(
            drop=True
        )
    )


    second_support_df = (
        cluster_df[
            (
                cluster_df[
                    "n_items"
                ]
                == n_items
            )
            &
            (
                cluster_df[
                    "algorithm"
                ]
                == SECOND_ALGORITHM
            )
        ]
        .sort_values(
            "matched_group_id"
        )
        .reset_index(
            drop=True
        )
    )


    assert len(
        first_support_df
    ) == 100


    assert len(
        second_support_df
    ) == 100


    assert (
        first_support_df[
            "matched_group_id"
        ].tolist()
        ==
        second_support_df[
            "matched_group_id"
        ].tolist()
    )


    assert np.array_equal(
        first_support_df[
            "comparable_rows"
        ].to_numpy(),
        second_support_df[
            "comparable_rows"
        ].to_numpy(),
    )


    support_rows.append(
        {
            "n_items":
                n_items,

            "matched_groups":
                len(
                    first_support_df
                ),

            "comparable_rows_per_algorithm":
                int(
                    first_support_df[
                        "comparable_rows"
                    ].sum()
                ),

            "minimum_comparable_rows_per_group":
                int(
                    first_support_df[
                        "comparable_rows"
                    ].min()
                ),

            "maximum_comparable_rows_per_group":
                int(
                    first_support_df[
                        "comparable_rows"
                    ].max()
                ),
        }
    )


size_support_df = pd.DataFrame(
    support_rows
)


size_support_df.to_csv(
    SIZE_ALGORITHM_SUPPORT_PATH,
    index=False,
)


# --------------------------------------------------
# Paired matched-group bootstrap within each size
# --------------------------------------------------

bootstrap_rng = np.random.default_rng(
    SIZE_ALGORITHM_BOOTSTRAP_SEED
)


contrast_rows = []
replicate_frames = []


for n_items in CANDIDATE_SIZE_ORDER:

    first_df = (
        cluster_df[
            (
                cluster_df[
                    "n_items"
                ]
                == n_items
            )
            &
            (
                cluster_df[
                    "algorithm"
                ]
                == FIRST_ALGORITHM
            )
        ]
        .sort_values(
            "matched_group_id"
        )
        .reset_index(
            drop=True
        )
    )


    second_df = (
        cluster_df[
            (
                cluster_df[
                    "n_items"
                ]
                == n_items
            )
            &
            (
                cluster_df[
                    "algorithm"
                ]
                == SECOND_ALGORITHM
            )
        ]
        .sort_values(
            "matched_group_id"
        )
        .reset_index(
            drop=True
        )
    )


    assert len(
        first_df
    ) == 100


    assert (
        first_df[
            "matched_group_id"
        ].tolist()
        ==
        second_df[
            "matched_group_id"
        ].tolist()
    )


    first_array = (
        first_df[
            AGGREGATE_COLUMNS
        ]
        .to_numpy(
            dtype=float
        )
    )


    second_array = (
        second_df[
            AGGREGATE_COLUMNS
        ]
        .to_numpy(
            dtype=float
        )
    )


    observed_first_metrics = (
        calculate_size_metrics(
            first_array.sum(
                axis=0
            )
        )
    )


    observed_second_metrics = (
        calculate_size_metrics(
            second_array.sum(
                axis=0
            )
        )
    )


    observed_difference = (
        observed_first_metrics
        -
        observed_second_metrics
    )


    bootstrap_differences = np.empty(
        (
            SIZE_ALGORITHM_BOOTSTRAP_REPLICATES,
            len(
                SIZE_METRIC_NAMES
            ),
        ),
        dtype=float,
    )


    for bootstrap_index in range(
        SIZE_ALGORITHM_BOOTSTRAP_REPLICATES
    ):

        # Shared indices preserve the paired comparison
        # between algorithms on each matched group.
        sampled_indices = (
            bootstrap_rng.integers(
                low=0,
                high=len(
                    first_array
                ),
                size=len(
                    first_array
                ),
            )
        )


        first_totals = (
            first_array[
                sampled_indices
            ]
            .sum(
                axis=0
            )
        )


        second_totals = (
            second_array[
                sampled_indices
            ]
            .sum(
                axis=0
            )
        )


        bootstrap_differences[
            bootstrap_index
        ] = (
            calculate_size_metrics(
                first_totals
            )
            -
            calculate_size_metrics(
                second_totals
            )
        )


    standard_errors = (
        bootstrap_differences.std(
            axis=0,
            ddof=1,
        )
    )


    confidence_lower = np.quantile(
        bootstrap_differences,
        0.025,
        axis=0,
    )


    confidence_upper = np.quantile(
        bootstrap_differences,
        0.975,
        axis=0,
    )


    for metric_index, metric_name in enumerate(
        SIZE_METRIC_NAMES
    ):

        contrast_rows.append(
            {
                "n_items":
                    n_items,

                "first_algorithm":
                    FIRST_ALGORITHM,

                "second_algorithm":
                    SECOND_ALGORITHM,

                "contrast":
                    (
                        f"{FIRST_ALGORITHM}"
                        f" minus "
                        f"{SECOND_ALGORITHM}"
                    ),

                "metric":
                    metric_name,

                "estimated_difference":
                    observed_difference[
                        metric_index
                    ],

                "bootstrap_standard_error":
                    standard_errors[
                        metric_index
                    ],

                "ci_95_lower":
                    confidence_lower[
                        metric_index
                    ],

                "ci_95_upper":
                    confidence_upper[
                        metric_index
                    ],

                "ci_excludes_zero":
                    (
                        confidence_lower[
                            metric_index
                        ]
                        > 0.0
                    )
                    or
                    (
                        confidence_upper[
                            metric_index
                        ]
                        < 0.0
                    ),

                "matched_groups":
                    100,

                "bootstrap_replicates":
                    SIZE_ALGORITHM_BOOTSTRAP_REPLICATES,

                "resampling_unit":
                    "matched_size_replicate_group",

                "resampling_design":
                    (
                        "paired_across_algorithms_"
                        "within_candidate_set_size"
                    ),

                "estimand":
                    (
                        "algorithm_difference_"
                        "conditional_on_feasibility"
                    ),

                "multiple_comparison_adjustment":
                    "none",
            }
        )


    replicate_df = pd.DataFrame(
        bootstrap_differences,
        columns=SIZE_METRIC_NAMES,
    )


    replicate_df.insert(
        0,
        "bootstrap_replicate",
        np.arange(
            SIZE_ALGORITHM_BOOTSTRAP_REPLICATES
        ),
    )


    replicate_df.insert(
        0,
        "n_items",
        n_items,
    )


    replicate_frames.append(
        replicate_df
    )


    print(
        "Completed Dynamic-versus-static candidate size:",
        n_items
    )


# --------------------------------------------------
# Save outputs
# --------------------------------------------------

size_contrast_df = pd.DataFrame(
    contrast_rows
)


size_replicates_df = pd.concat(
    replicate_frames,
    ignore_index=True,
)


size_contrast_df.to_csv(
    SIZE_ALGORITHM_CONTRAST_PATH,
    index=False,
)


size_replicates_df.to_csv(
    SIZE_ALGORITHM_REPLICATES_PATH,
    index=False,
)


# --------------------------------------------------
# Readable displays
# --------------------------------------------------

key_metrics = [
    "mean_optimality_gap_percent",
    "exact_cost_recovery_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "more_items_percent",
    "mean_selected_count_difference",
]


contrast_display_df = (
    size_contrast_df[
        size_contrast_df[
            "metric"
        ].isin(
            key_metrics
        )
    ]
    .copy()
)


contrast_display_df[
    "difference_and_ci"
] = (
    contrast_display_df[
        "estimated_difference"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    contrast_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    contrast_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


print(
    "\n======================================"
)

print(
    "DYNAMIC AND STATIC-RATIO RESULTS BY CANDIDATE SIZE"
)

print(
    "======================================"
)


print(
    size_descriptive_df[
        [
            "n_items",
            "algorithm",
            "matched_groups",
            "comparable_rows",
            "mean_optimality_gap_percent",
            "exact_cost_recovery_percent",
            "mean_jaccard_similarity",
            "large_gap_percent",
            "more_items_percent",
            "mean_selected_count_difference",
        ]
    ].to_string(
        index=False,
        float_format=lambda value:
            f"{value:.4f}",
    )
)


print(
    "\n======================================"
)

print(
    "CANDIDATE-SIZE COMPARABLE SUPPORT"
)

print(
    "======================================"
)


print(
    size_support_df.to_string(
        index=False
    )
)


print(
    "\n======================================"
)

print(
    "PAIRED DYNAMIC-MINUS-STATIC CONTRASTS BY CANDIDATE SIZE"
)

print(
    "======================================"
)


print(
    contrast_display_df[
        [
            "n_items",
            "metric",
            "difference_and_ci",
            "ci_excludes_zero",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\nCandidate-size descriptive results saved to:"
)

print(
    SIZE_ALGORITHM_DESCRIPTIVE_PATH
)


print(
    "\nCandidate-size support counts saved to:"
)

print(
    SIZE_ALGORITHM_SUPPORT_PATH
)


print(
    "\nCandidate-size paired contrasts saved to:"
)

print(
    SIZE_ALGORITHM_CONTRAST_PATH
)


print(
    "\nBootstrap replicates saved to:"
)

print(
    SIZE_ALGORITHM_REPLICATES_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Step 121 — Dynamic Greedy versus static utility-cost
# ratio baseline within each objective-weight profile
# --------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------
# Configuration
# --------------------------------------------------

WEIGHT_ALGORITHM_BOOTSTRAP_REPLICATES = 5000

WEIGHT_ALGORITHM_BOOTSTRAP_SEED = (
    globals().get(
        "GLOBAL_SEED",
        2026,
    )
    + 121
)


if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


ALGORITHM_LONG_PATH = (
    RESULTS_DIR
    / "corrected_v2_algorithm_comparison_long.csv"
)


WEIGHT_ALGORITHM_DESCRIPTIVE_PATH = (
    TABLES_DIR
    / "corrected_v2_dynamic_static_ratio_by_weight_profile_descriptive.csv"
)


WEIGHT_ALGORITHM_SUPPORT_PATH = (
    TABLES_DIR
    / "corrected_v2_dynamic_static_ratio_weight_profile_support.csv"
)


WEIGHT_ALGORITHM_CONTRAST_PATH = (
    TABLES_DIR
    / "corrected_v2_dynamic_vs_static_ratio_by_weight_profile_contrasts.csv"
)


WEIGHT_ALGORITHM_REPLICATES_PATH = (
    RESULTS_DIR
    / "corrected_v2_dynamic_vs_static_ratio_by_weight_profile_bootstrap_replicates.csv"
)


WEIGHT_PROFILE_ORDER = [
    "balanced",
    "privacy_priority",
    "interpretation_priority",
    "parsimony_priority",
]


FIRST_ALGORITHM = "dynamic_greedy"

SECOND_ALGORITHM = (
    "static_utility_cost_ratio"
)


WEIGHT_METRIC_NAMES = [
    "mean_optimality_gap_percent",
    "exact_cost_recovery_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "more_items_percent",
    "mean_selected_count_difference",
    "within_1_percent",
    "within_5_percent",
    "within_10_percent",
]


AGGREGATE_COLUMNS = [
    "comparable_rows",
    "gap_sum",
    "exact_cost_recovery_sum",
    "jaccard_sum",
    "large_gap_sum",
    "more_items_sum",
    "selected_count_difference_sum",
    "within_1_percent_sum",
    "within_5_percent_sum",
    "within_10_percent_sum",
]


# --------------------------------------------------
# Load combined algorithm benchmark
# --------------------------------------------------

weight_algorithm_df = pd.read_csv(
    ALGORITHM_LONG_PATH,
    low_memory=False,
)


required_columns = [
    "matched_group_id",
    "instance_id",
    "n_items",
    "weight_profile",
    "algorithm",
    "exact_status",
    "algorithm_status",
    "optimality_gap_percent",
    "jaccard_similarity",
    "selected_count_difference",
]


missing_columns = [
    column_name
    for column_name in required_columns
    if column_name not in weight_algorithm_df.columns
]


if missing_columns:

    raise KeyError(
        "Missing required columns: "
        + ", ".join(
            missing_columns
        )
    )


for column_name in [
    "n_items",
    "optimality_gap_percent",
    "jaccard_similarity",
    "selected_count_difference",
]:

    weight_algorithm_df[
        column_name
    ] = pd.to_numeric(
        weight_algorithm_df[
            column_name
        ],
        errors="coerce",
    )


weight_algorithm_df = (
    weight_algorithm_df[
        weight_algorithm_df[
            "algorithm"
        ].isin(
            [
                FIRST_ALGORITHM,
                SECOND_ALGORITHM,
            ]
        )
    ]
    .copy()
)


observed_weight_profiles = set(
    weight_algorithm_df[
        "weight_profile"
    ]
    .dropna()
    .unique()
)


expected_weight_profiles = set(
    WEIGHT_PROFILE_ORDER
)


if observed_weight_profiles != expected_weight_profiles:

    raise ValueError(
        "Unexpected weight-profile values.\n"
        f"Observed: {sorted(observed_weight_profiles)}\n"
        f"Expected: {sorted(expected_weight_profiles)}"
    )


# --------------------------------------------------
# Restrict to Exact-feasible comparable conditions
# --------------------------------------------------

weight_algorithm_df[
    "is_comparable"
] = (
    (
        weight_algorithm_df[
            "exact_status"
        ]
        == "OPTIMAL"
    )
    &
    (
        weight_algorithm_df[
            "algorithm_status"
        ]
        == "FEASIBLE"
    )
)


comparable_df = (
    weight_algorithm_df[
        weight_algorithm_df[
            "is_comparable"
        ]
    ]
    .copy()
)


assert len(
    comparable_df
) == (
    23784
    *
    2
)


# --------------------------------------------------
# Reporting indicators
# --------------------------------------------------

OPTIMAL_COST_TOLERANCE_PERCENT = 1e-10


comparable_df[
    "exact_cost_recovery_numeric"
] = (
    comparable_df[
        "optimality_gap_percent"
    ]
    .abs()
    <=
    OPTIMAL_COST_TOLERANCE_PERCENT
).astype(int)


comparable_df[
    "large_gap_numeric"
] = (
    comparable_df[
        "optimality_gap_percent"
    ]
    > 10.0
).astype(int)


comparable_df[
    "more_items_numeric"
] = (
    comparable_df[
        "selected_count_difference"
    ]
    > 0
).astype(int)


comparable_df[
    "within_1_percent_numeric"
] = (
    comparable_df[
        "optimality_gap_percent"
    ]
    <= 1.0
).astype(int)


comparable_df[
    "within_5_percent_numeric"
] = (
    comparable_df[
        "optimality_gap_percent"
    ]
    <= 5.0
).astype(int)


comparable_df[
    "within_10_percent_numeric"
] = (
    comparable_df[
        "optimality_gap_percent"
    ]
    <= 10.0
).astype(int)


# --------------------------------------------------
# Descriptive results by weight profile and algorithm
# --------------------------------------------------

descriptive_rows = []


for weight_profile in WEIGHT_PROFILE_ORDER:

    for algorithm in [
        FIRST_ALGORITHM,
        SECOND_ALGORITHM,
    ]:

        current_df = (
            comparable_df[
                (
                    comparable_df[
                        "weight_profile"
                    ]
                    == weight_profile
                )
                &
                (
                    comparable_df[
                        "algorithm"
                    ]
                    == algorithm
                )
            ]
        )


        gap_values = (
            current_df[
                "optimality_gap_percent"
            ]
            .to_numpy(
                dtype=float
            )
        )


        descriptive_rows.append(
            {
                "weight_profile":
                    weight_profile,

                "algorithm":
                    algorithm,

                "matched_groups":
                    current_df[
                        "matched_group_id"
                    ].nunique(),

                "comparable_rows":
                    len(
                        current_df
                    ),

                "mean_optimality_gap_percent":
                    gap_values.mean(),

                "median_optimality_gap_percent":
                    np.median(
                        gap_values
                    ),

                "p90_optimality_gap_percent":
                    np.quantile(
                        gap_values,
                        0.90,
                    ),

                "p95_optimality_gap_percent":
                    np.quantile(
                        gap_values,
                        0.95,
                    ),

                "exact_cost_recovery_percent":
                    100.0
                    *
                    current_df[
                        "exact_cost_recovery_numeric"
                    ].mean(),

                "mean_jaccard_similarity":
                    current_df[
                        "jaccard_similarity"
                    ].mean(),

                "large_gap_percent":
                    100.0
                    *
                    current_df[
                        "large_gap_numeric"
                    ].mean(),

                "more_items_percent":
                    100.0
                    *
                    current_df[
                        "more_items_numeric"
                    ].mean(),

                "mean_selected_count_difference":
                    current_df[
                        "selected_count_difference"
                    ].mean(),

                "within_1_percent":
                    100.0
                    *
                    current_df[
                        "within_1_percent_numeric"
                    ].mean(),

                "within_5_percent":
                    100.0
                    *
                    current_df[
                        "within_5_percent_numeric"
                    ].mean(),

                "within_10_percent":
                    100.0
                    *
                    current_df[
                        "within_10_percent_numeric"
                    ].mean(),
            }
        )


weight_descriptive_df = pd.DataFrame(
    descriptive_rows
)


weight_descriptive_df[
    "weight_profile"
] = pd.Categorical(
    weight_descriptive_df[
        "weight_profile"
    ],
    categories=WEIGHT_PROFILE_ORDER,
    ordered=True,
)


weight_descriptive_df = (
    weight_descriptive_df
    .sort_values(
        [
            "weight_profile",
            "algorithm",
        ]
    )
    .reset_index(
        drop=True
    )
)


weight_descriptive_df.to_csv(
    WEIGHT_ALGORITHM_DESCRIPTIVE_PATH,
    index=False,
)


# --------------------------------------------------
# Sufficient statistics per matched group, size,
# weight profile, and algorithm
# --------------------------------------------------

cluster_df = (
    comparable_df
    .groupby(
        [
            "matched_group_id",
            "n_items",
            "weight_profile",
            "algorithm",
        ],
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "size",
        ),

        gap_sum=(
            "optimality_gap_percent",
            "sum",
        ),

        exact_cost_recovery_sum=(
            "exact_cost_recovery_numeric",
            "sum",
        ),

        jaccard_sum=(
            "jaccard_similarity",
            "sum",
        ),

        large_gap_sum=(
            "large_gap_numeric",
            "sum",
        ),

        more_items_sum=(
            "more_items_numeric",
            "sum",
        ),

        selected_count_difference_sum=(
            "selected_count_difference",
            "sum",
        ),

        within_1_percent_sum=(
            "within_1_percent_numeric",
            "sum",
        ),

        within_5_percent_sum=(
            "within_5_percent_numeric",
            "sum",
        ),

        within_10_percent_sum=(
            "within_10_percent_numeric",
            "sum",
        ),
    )
)


def calculate_weight_metrics(
    totals: np.ndarray,
) -> np.ndarray:
    """
    Convert matched-group sufficient statistics into
    Exact-relative algorithm-quality metrics.
    """

    comparable_rows = totals[0]


    if comparable_rows <= 0:

        return np.full(
            len(
                WEIGHT_METRIC_NAMES
            ),
            np.nan,
        )


    return np.array(
        [
            totals[1]
            /
            comparable_rows,

            100.0
            *
            totals[2]
            /
            comparable_rows,

            totals[3]
            /
            comparable_rows,

            100.0
            *
            totals[4]
            /
            comparable_rows,

            100.0
            *
            totals[5]
            /
            comparable_rows,

            totals[6]
            /
            comparable_rows,

            100.0
            *
            totals[7]
            /
            comparable_rows,

            100.0
            *
            totals[8]
            /
            comparable_rows,

            100.0
            *
            totals[9]
            /
            comparable_rows,
        ],
        dtype=float,
    )


# --------------------------------------------------
# Validate paired support and create support table
# --------------------------------------------------

support_rows = []


for weight_profile in WEIGHT_PROFILE_ORDER:

    for n_items in sorted(
        cluster_df[
            "n_items"
        ].unique()
    ):

        first_support_df = (
            cluster_df[
                (
                    cluster_df[
                        "weight_profile"
                    ]
                    == weight_profile
                )
                &
                (
                    cluster_df[
                        "n_items"
                    ]
                    == n_items
                )
                &
                (
                    cluster_df[
                        "algorithm"
                    ]
                    == FIRST_ALGORITHM
                )
            ]
            .sort_values(
                "matched_group_id"
            )
            .reset_index(
                drop=True
            )
        )


        second_support_df = (
            cluster_df[
                (
                    cluster_df[
                        "weight_profile"
                    ]
                    == weight_profile
                )
                &
                (
                    cluster_df[
                        "n_items"
                    ]
                    == n_items
                )
                &
                (
                    cluster_df[
                        "algorithm"
                    ]
                    == SECOND_ALGORITHM
                )
            ]
            .sort_values(
                "matched_group_id"
            )
            .reset_index(
                drop=True
            )
        )


        assert len(
            first_support_df
        ) == 100


        assert len(
            second_support_df
        ) == 100


        assert (
            first_support_df[
                "matched_group_id"
            ].tolist()
            ==
            second_support_df[
                "matched_group_id"
            ].tolist()
        )


        assert np.array_equal(
            first_support_df[
                "comparable_rows"
            ].to_numpy(),
            second_support_df[
                "comparable_rows"
            ].to_numpy(),
        )


        support_rows.append(
            {
                "weight_profile":
                    weight_profile,

                "n_items":
                    int(
                        n_items
                    ),

                "matched_groups":
                    len(
                        first_support_df
                    ),

                "comparable_rows_per_algorithm":
                    int(
                        first_support_df[
                            "comparable_rows"
                        ].sum()
                    ),

                "minimum_comparable_rows_per_group":
                    int(
                        first_support_df[
                            "comparable_rows"
                        ].min()
                    ),

                "maximum_comparable_rows_per_group":
                    int(
                        first_support_df[
                            "comparable_rows"
                        ].max()
                    ),
            }
        )


weight_support_df = pd.DataFrame(
    support_rows
)


weight_support_df.to_csv(
    WEIGHT_ALGORITHM_SUPPORT_PATH,
    index=False,
)


# --------------------------------------------------
# Paired bootstrap within weight profile, stratified
# by candidate-set size
# --------------------------------------------------

bootstrap_rng = np.random.default_rng(
    WEIGHT_ALGORITHM_BOOTSTRAP_SEED
)


contrast_rows = []
replicate_frames = []


for weight_profile in WEIGHT_PROFILE_ORDER:

    paired_size_strata = []


    for n_items in sorted(
        cluster_df[
            "n_items"
        ].unique()
    ):

        first_df = (
            cluster_df[
                (
                    cluster_df[
                        "weight_profile"
                    ]
                    == weight_profile
                )
                &
                (
                    cluster_df[
                        "n_items"
                    ]
                    == n_items
                )
                &
                (
                    cluster_df[
                        "algorithm"
                    ]
                    == FIRST_ALGORITHM
                )
            ]
            .sort_values(
                "matched_group_id"
            )
            .reset_index(
                drop=True
            )
        )


        second_df = (
            cluster_df[
                (
                    cluster_df[
                        "weight_profile"
                    ]
                    == weight_profile
                )
                &
                (
                    cluster_df[
                        "n_items"
                    ]
                    == n_items
                )
                &
                (
                    cluster_df[
                        "algorithm"
                    ]
                    == SECOND_ALGORITHM
                )
            ]
            .sort_values(
                "matched_group_id"
            )
            .reset_index(
                drop=True
            )
        )


        assert len(
            first_df
        ) == 100


        assert len(
            second_df
        ) == 100


        assert (
            first_df[
                "matched_group_id"
            ].tolist()
            ==
            second_df[
                "matched_group_id"
            ].tolist()
        )


        assert np.array_equal(
            first_df[
                "comparable_rows"
            ].to_numpy(),
            second_df[
                "comparable_rows"
            ].to_numpy(),
        )


        paired_size_strata.append(
            {
                "n_items":
                    int(
                        n_items
                    ),

                "matched_group_ids":
                    first_df[
                        "matched_group_id"
                    ].tolist(),

                "first_array":
                    first_df[
                        AGGREGATE_COLUMNS
                    ]
                    .to_numpy(
                        dtype=float
                    ),

                "second_array":
                    second_df[
                        AGGREGATE_COLUMNS
                    ]
                    .to_numpy(
                        dtype=float
                    ),
            }
        )


    observed_first_totals = np.zeros(
        len(
            AGGREGATE_COLUMNS
        ),
        dtype=float,
    )


    observed_second_totals = np.zeros(
        len(
            AGGREGATE_COLUMNS
        ),
        dtype=float,
    )


    for size_stratum in paired_size_strata:

        observed_first_totals += (
            size_stratum[
                "first_array"
            ]
            .sum(
                axis=0
            )
        )


        observed_second_totals += (
            size_stratum[
                "second_array"
            ]
            .sum(
                axis=0
            )
        )


    observed_difference = (
        calculate_weight_metrics(
            observed_first_totals
        )
        -
        calculate_weight_metrics(
            observed_second_totals
        )
    )


    bootstrap_differences = np.empty(
        (
            WEIGHT_ALGORITHM_BOOTSTRAP_REPLICATES,
            len(
                WEIGHT_METRIC_NAMES
            ),
        ),
        dtype=float,
    )


    for bootstrap_index in range(
        WEIGHT_ALGORITHM_BOOTSTRAP_REPLICATES
    ):

        first_totals = np.zeros(
            len(
                AGGREGATE_COLUMNS
            ),
            dtype=float,
        )


        second_totals = np.zeros(
            len(
                AGGREGATE_COLUMNS
            ),
            dtype=float,
        )


        for size_stratum in paired_size_strata:

            number_of_groups = len(
                size_stratum[
                    "matched_group_ids"
                ]
            )


            # Shared indices retain pairing between
            # Dynamic Greedy and the static baseline.
            sampled_indices = (
                bootstrap_rng.integers(
                    low=0,
                    high=number_of_groups,
                    size=number_of_groups,
                )
            )


            first_totals += (
                size_stratum[
                    "first_array"
                ][
                    sampled_indices
                ]
                .sum(
                    axis=0
                )
            )


            second_totals += (
                size_stratum[
                    "second_array"
                ][
                    sampled_indices
                ]
                .sum(
                    axis=0
                )
            )


        bootstrap_differences[
            bootstrap_index
        ] = (
            calculate_weight_metrics(
                first_totals
            )
            -
            calculate_weight_metrics(
                second_totals
            )
        )


    standard_errors = (
        bootstrap_differences.std(
            axis=0,
            ddof=1,
        )
    )


    confidence_lower = np.quantile(
        bootstrap_differences,
        0.025,
        axis=0,
    )


    confidence_upper = np.quantile(
        bootstrap_differences,
        0.975,
        axis=0,
    )


    supported_groups = sum(
        len(
            size_stratum[
                "matched_group_ids"
            ]
        )
        for size_stratum in paired_size_strata
    )


    for metric_index, metric_name in enumerate(
        WEIGHT_METRIC_NAMES
    ):

        contrast_rows.append(
            {
                "weight_profile":
                    weight_profile,

                "first_algorithm":
                    FIRST_ALGORITHM,

                "second_algorithm":
                    SECOND_ALGORITHM,

                "contrast":
                    (
                        f"{FIRST_ALGORITHM}"
                        f" minus "
                        f"{SECOND_ALGORITHM}"
                    ),

                "metric":
                    metric_name,

                "estimated_difference":
                    observed_difference[
                        metric_index
                    ],

                "bootstrap_standard_error":
                    standard_errors[
                        metric_index
                    ],

                "ci_95_lower":
                    confidence_lower[
                        metric_index
                    ],

                "ci_95_upper":
                    confidence_upper[
                        metric_index
                    ],

                "ci_excludes_zero":
                    (
                        confidence_lower[
                            metric_index
                        ]
                        > 0.0
                    )
                    or
                    (
                        confidence_upper[
                            metric_index
                        ]
                        < 0.0
                    ),

                "supported_matched_groups":
                    supported_groups,

                "matched_groups_per_size":
                    100,

                "candidate_size_strata":
                    len(
                        paired_size_strata
                    ),

                "bootstrap_replicates":
                    WEIGHT_ALGORITHM_BOOTSTRAP_REPLICATES,

                "resampling_unit":
                    "matched_size_replicate_group",

                "resampling_design":
                    (
                        "paired_across_algorithms_"
                        "stratified_by_candidate_set_size"
                    ),

                "estimand":
                    (
                        "algorithm_difference_"
                        "conditional_on_feasibility"
                    ),

                "multiple_comparison_adjustment":
                    "none",
            }
        )


    replicate_df = pd.DataFrame(
        bootstrap_differences,
        columns=WEIGHT_METRIC_NAMES,
    )


    replicate_df.insert(
        0,
        "bootstrap_replicate",
        np.arange(
            WEIGHT_ALGORITHM_BOOTSTRAP_REPLICATES
        ),
    )


    replicate_df.insert(
        0,
        "weight_profile",
        weight_profile,
    )


    replicate_frames.append(
        replicate_df
    )


    print(
        "Completed Dynamic-versus-static weight profile:",
        weight_profile
    )


# --------------------------------------------------
# Save outputs
# --------------------------------------------------

weight_contrast_df = pd.DataFrame(
    contrast_rows
)


weight_replicates_df = pd.concat(
    replicate_frames,
    ignore_index=True,
)


weight_contrast_df.to_csv(
    WEIGHT_ALGORITHM_CONTRAST_PATH,
    index=False,
)


weight_replicates_df.to_csv(
    WEIGHT_ALGORITHM_REPLICATES_PATH,
    index=False,
)


# --------------------------------------------------
# Readable displays
# --------------------------------------------------

key_metrics = [
    "mean_optimality_gap_percent",
    "exact_cost_recovery_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "more_items_percent",
    "mean_selected_count_difference",
]


contrast_display_df = (
    weight_contrast_df[
        weight_contrast_df[
            "metric"
        ].isin(
            key_metrics
        )
    ]
    .copy()
)


contrast_display_df[
    "difference_and_ci"
] = (
    contrast_display_df[
        "estimated_difference"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    contrast_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    contrast_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


print(
    "\n======================================"
)

print(
    "DYNAMIC AND STATIC-RATIO RESULTS BY WEIGHT PROFILE"
)

print(
    "======================================"
)


print(
    weight_descriptive_df[
        [
            "weight_profile",
            "algorithm",
            "matched_groups",
            "comparable_rows",
            "mean_optimality_gap_percent",
            "exact_cost_recovery_percent",
            "mean_jaccard_similarity",
            "large_gap_percent",
            "more_items_percent",
            "mean_selected_count_difference",
        ]
    ].to_string(
        index=False,
        float_format=lambda value:
            f"{value:.4f}",
    )
)


print(
    "\n======================================"
)

print(
    "WEIGHT-PROFILE COMPARABLE SUPPORT"
)

print(
    "======================================"
)


print(
    weight_support_df.to_string(
        index=False
    )
)


print(
    "\n======================================"
)

print(
    "PAIRED DYNAMIC-MINUS-STATIC CONTRASTS BY WEIGHT PROFILE"
)

print(
    "======================================"
)


print(
    contrast_display_df[
        [
            "weight_profile",
            "metric",
            "difference_and_ci",
            "ci_excludes_zero",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\nWeight-profile descriptive results saved to:"
)

print(
    WEIGHT_ALGORITHM_DESCRIPTIVE_PATH
)


print(
    "\nWeight-profile support counts saved to:"
)

print(
    WEIGHT_ALGORITHM_SUPPORT_PATH
)


print(
    "\nWeight-profile paired contrasts saved to:"
)

print(
    WEIGHT_ALGORITHM_CONTRAST_PATH
)


print(
    "\nBootstrap replicates saved to:"
)

print(
    WEIGHT_ALGORITHM_REPLICATES_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Step 122 — Dynamic Greedy versus static ratio by
# structural regime × sufficiency threshold
# --------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------
# Configuration
# --------------------------------------------------

INTERACTION_BOOTSTRAP_REPLICATES = 5000

INTERACTION_BOOTSTRAP_SEED = (
    globals().get(
        "GLOBAL_SEED",
        2026,
    )
    + 122
)


if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


ALGORITHM_LONG_PATH = (
    RESULTS_DIR
    / "corrected_v2_algorithm_comparison_long.csv"
)


INTERACTION_DESCRIPTIVE_PATH = (
    TABLES_DIR
    / (
        "corrected_v2_dynamic_static_ratio_"
        "regime_threshold_descriptive.csv"
    )
)


INTERACTION_SUPPORT_PATH = (
    TABLES_DIR
    / (
        "corrected_v2_dynamic_static_ratio_"
        "regime_threshold_support.csv"
    )
)


INTERACTION_CONTRAST_PATH = (
    TABLES_DIR
    / (
        "corrected_v2_dynamic_vs_static_ratio_"
        "regime_threshold_contrasts.csv"
    )
)


INTERACTION_REPLICATES_PATH = (
    RESULTS_DIR
    / (
        "corrected_v2_dynamic_vs_static_ratio_"
        "regime_threshold_bootstrap_replicates.csv"
    )
)


REGIME_ORDER = [
    "independent",
    "utility_burden_alignment",
    "utility_burden_conflict",
]


THRESHOLD_ORDER = [
    0.70,
    0.80,
    0.90,
    0.95,
]


CANDIDATE_SIZE_ORDER = [
    8,
    10,
    12,
    14,
    16,
]


FIRST_ALGORITHM = "dynamic_greedy"

SECOND_ALGORITHM = (
    "static_utility_cost_ratio"
)


INTERACTION_METRIC_NAMES = [
    "mean_optimality_gap_percent",
    "exact_cost_recovery_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "more_items_percent",
    "mean_selected_count_difference",
    "within_1_percent",
    "within_5_percent",
    "within_10_percent",
]


AGGREGATE_COLUMNS = [
    "comparable_rows",
    "gap_sum",
    "exact_cost_recovery_sum",
    "jaccard_sum",
    "large_gap_sum",
    "more_items_sum",
    "selected_count_difference_sum",
    "within_1_percent_sum",
    "within_5_percent_sum",
    "within_10_percent_sum",
]


# --------------------------------------------------
# Load combined algorithm benchmark
# --------------------------------------------------

interaction_df = pd.read_csv(
    ALGORITHM_LONG_PATH,
    low_memory=False,
)


required_columns = [
    "matched_group_id",
    "instance_id",
    "n_items",
    "regime",
    "tau",
    "weight_profile",
    "algorithm",
    "exact_status",
    "algorithm_status",
    "optimality_gap_percent",
    "jaccard_similarity",
    "selected_count_difference",
]


missing_columns = [
    column_name
    for column_name in required_columns
    if column_name not in interaction_df.columns
]


if missing_columns:

    raise KeyError(
        "Missing required columns: "
        + ", ".join(
            missing_columns
        )
    )


for column_name in [
    "n_items",
    "tau",
    "optimality_gap_percent",
    "jaccard_similarity",
    "selected_count_difference",
]:

    interaction_df[
        column_name
    ] = pd.to_numeric(
        interaction_df[
            column_name
        ],
        errors="coerce",
    )


interaction_df = (
    interaction_df[
        interaction_df[
            "algorithm"
        ].isin(
            [
                FIRST_ALGORITHM,
                SECOND_ALGORITHM,
            ]
        )
    ]
    .copy()
)


assert set(
    interaction_df[
        "regime"
    ].dropna().unique()
) == set(
    REGIME_ORDER
)


assert np.allclose(
    sorted(
        interaction_df[
            "tau"
        ]
        .dropna()
        .unique()
    ),
    THRESHOLD_ORDER,
)


assert (
    sorted(
        interaction_df[
            "n_items"
        ]
        .dropna()
        .unique()
        .astype(int)
        .tolist()
    )
    ==
    CANDIDATE_SIZE_ORDER
)


# --------------------------------------------------
# Comparable Exact-feasible conditions
# --------------------------------------------------

interaction_df[
    "is_comparable"
] = (
    (
        interaction_df[
            "exact_status"
        ]
        == "OPTIMAL"
    )
    &
    (
        interaction_df[
            "algorithm_status"
        ]
        == "FEASIBLE"
    )
)


comparable_df = (
    interaction_df[
        interaction_df[
            "is_comparable"
        ]
    ]
    .copy()
)


assert len(
    comparable_df
) == (
    23784
    *
    2
)


# --------------------------------------------------
# Reporting indicators
# --------------------------------------------------

OPTIMAL_COST_TOLERANCE_PERCENT = 1e-10


comparable_df[
    "exact_cost_recovery_numeric"
] = (
    comparable_df[
        "optimality_gap_percent"
    ]
    .abs()
    <=
    OPTIMAL_COST_TOLERANCE_PERCENT
).astype(int)


comparable_df[
    "large_gap_numeric"
] = (
    comparable_df[
        "optimality_gap_percent"
    ]
    > 10.0
).astype(int)


comparable_df[
    "more_items_numeric"
] = (
    comparable_df[
        "selected_count_difference"
    ]
    > 0
).astype(int)


comparable_df[
    "within_1_percent_numeric"
] = (
    comparable_df[
        "optimality_gap_percent"
    ]
    <= 1.0
).astype(int)


comparable_df[
    "within_5_percent_numeric"
] = (
    comparable_df[
        "optimality_gap_percent"
    ]
    <= 5.0
).astype(int)


comparable_df[
    "within_10_percent_numeric"
] = (
    comparable_df[
        "optimality_gap_percent"
    ]
    <= 10.0
).astype(int)


# --------------------------------------------------
# Descriptive results by regime, threshold,
# and algorithm
# --------------------------------------------------

descriptive_rows = []


for regime in REGIME_ORDER:

    for tau in THRESHOLD_ORDER:

        for algorithm in [
            FIRST_ALGORITHM,
            SECOND_ALGORITHM,
        ]:

            current_df = (
                comparable_df[
                    (
                        comparable_df[
                            "regime"
                        ]
                        == regime
                    )
                    &
                    (
                        np.isclose(
                            comparable_df[
                                "tau"
                            ],
                            tau,
                        )
                    )
                    &
                    (
                        comparable_df[
                            "algorithm"
                        ]
                        == algorithm
                    )
                ]
            )


            gap_values = (
                current_df[
                    "optimality_gap_percent"
                ]
                .to_numpy(
                    dtype=float
                )
            )


            descriptive_rows.append(
                {
                    "regime":
                        regime,

                    "tau":
                        tau,

                    "algorithm":
                        algorithm,

                    "matched_groups":
                        current_df[
                            "matched_group_id"
                        ].nunique(),

                    "comparable_rows":
                        len(
                            current_df
                        ),

                    "mean_optimality_gap_percent":
                        gap_values.mean(),

                    "median_optimality_gap_percent":
                        np.median(
                            gap_values
                        ),

                    "p90_optimality_gap_percent":
                        np.quantile(
                            gap_values,
                            0.90,
                        ),

                    "p95_optimality_gap_percent":
                        np.quantile(
                            gap_values,
                            0.95,
                        ),

                    "exact_cost_recovery_percent":
                        100.0
                        *
                        current_df[
                            "exact_cost_recovery_numeric"
                        ].mean(),

                    "mean_jaccard_similarity":
                        current_df[
                            "jaccard_similarity"
                        ].mean(),

                    "large_gap_percent":
                        100.0
                        *
                        current_df[
                            "large_gap_numeric"
                        ].mean(),

                    "more_items_percent":
                        100.0
                        *
                        current_df[
                            "more_items_numeric"
                        ].mean(),

                    "mean_selected_count_difference":
                        current_df[
                            "selected_count_difference"
                        ].mean(),

                    "within_1_percent":
                        100.0
                        *
                        current_df[
                            "within_1_percent_numeric"
                        ].mean(),

                    "within_5_percent":
                        100.0
                        *
                        current_df[
                            "within_5_percent_numeric"
                        ].mean(),

                    "within_10_percent":
                        100.0
                        *
                        current_df[
                            "within_10_percent_numeric"
                        ].mean(),
                }
            )


interaction_descriptive_df = pd.DataFrame(
    descriptive_rows
)


interaction_descriptive_df[
    "regime"
] = pd.Categorical(
    interaction_descriptive_df[
        "regime"
    ],
    categories=REGIME_ORDER,
    ordered=True,
)


interaction_descriptive_df = (
    interaction_descriptive_df
    .sort_values(
        [
            "regime",
            "tau",
            "algorithm",
        ]
    )
    .reset_index(
        drop=True
    )
)


interaction_descriptive_df.to_csv(
    INTERACTION_DESCRIPTIVE_PATH,
    index=False,
)


# --------------------------------------------------
# Sufficient statistics per matched group, size,
# regime, threshold, and algorithm
# --------------------------------------------------

cluster_df = (
    comparable_df
    .groupby(
        [
            "matched_group_id",
            "n_items",
            "regime",
            "tau",
            "algorithm",
        ],
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "size",
        ),

        gap_sum=(
            "optimality_gap_percent",
            "sum",
        ),

        exact_cost_recovery_sum=(
            "exact_cost_recovery_numeric",
            "sum",
        ),

        jaccard_sum=(
            "jaccard_similarity",
            "sum",
        ),

        large_gap_sum=(
            "large_gap_numeric",
            "sum",
        ),

        more_items_sum=(
            "more_items_numeric",
            "sum",
        ),

        selected_count_difference_sum=(
            "selected_count_difference",
            "sum",
        ),

        within_1_percent_sum=(
            "within_1_percent_numeric",
            "sum",
        ),

        within_5_percent_sum=(
            "within_5_percent_numeric",
            "sum",
        ),

        within_10_percent_sum=(
            "within_10_percent_numeric",
            "sum",
        ),
    )
)


def calculate_interaction_metrics(
    totals: np.ndarray,
) -> np.ndarray:
    """
    Convert sufficient-statistic totals into
    Exact-relative algorithm-quality metrics.
    """

    comparable_rows = totals[0]


    if comparable_rows <= 0:

        return np.full(
            len(
                INTERACTION_METRIC_NAMES
            ),
            np.nan,
        )


    return np.array(
        [
            totals[1]
            /
            comparable_rows,

            100.0
            *
            totals[2]
            /
            comparable_rows,

            totals[3]
            /
            comparable_rows,

            100.0
            *
            totals[4]
            /
            comparable_rows,

            100.0
            *
            totals[5]
            /
            comparable_rows,

            totals[6]
            /
            comparable_rows,

            100.0
            *
            totals[7]
            /
            comparable_rows,

            100.0
            *
            totals[8]
            /
            comparable_rows,

            100.0
            *
            totals[9]
            /
            comparable_rows,
        ],
        dtype=float,
    )


# --------------------------------------------------
# Validate paired support and create support table
# --------------------------------------------------

support_rows = []


for regime in REGIME_ORDER:

    for tau in THRESHOLD_ORDER:

        for n_items in CANDIDATE_SIZE_ORDER:

            first_support_df = (
                cluster_df[
                    (
                        cluster_df[
                            "regime"
                        ]
                        == regime
                    )
                    &
                    (
                        np.isclose(
                            cluster_df[
                                "tau"
                            ],
                            tau,
                        )
                    )
                    &
                    (
                        cluster_df[
                            "n_items"
                        ]
                        == n_items
                    )
                    &
                    (
                        cluster_df[
                            "algorithm"
                        ]
                        == FIRST_ALGORITHM
                    )
                ]
                .sort_values(
                    "matched_group_id"
                )
                .reset_index(
                    drop=True
                )
            )


            second_support_df = (
                cluster_df[
                    (
                        cluster_df[
                            "regime"
                        ]
                        == regime
                    )
                    &
                    (
                        np.isclose(
                            cluster_df[
                                "tau"
                            ],
                            tau,
                        )
                    )
                    &
                    (
                        cluster_df[
                            "n_items"
                        ]
                        == n_items
                    )
                    &
                    (
                        cluster_df[
                            "algorithm"
                        ]
                        == SECOND_ALGORITHM
                    )
                ]
                .sort_values(
                    "matched_group_id"
                )
                .reset_index(
                    drop=True
                )
            )


            assert len(
                first_support_df
            ) == len(
                second_support_df
            )


            assert len(
                first_support_df
            ) > 0


            assert (
                first_support_df[
                    "matched_group_id"
                ].tolist()
                ==
                second_support_df[
                    "matched_group_id"
                ].tolist()
            )


            assert np.array_equal(
                first_support_df[
                    "comparable_rows"
                ].to_numpy(),
                second_support_df[
                    "comparable_rows"
                ].to_numpy(),
            )


            support_rows.append(
                {
                    "regime":
                        regime,

                    "tau":
                        tau,

                    "n_items":
                        n_items,

                    "feasible_matched_groups":
                        len(
                            first_support_df
                        ),

                    "comparable_rows_per_algorithm":
                        int(
                            first_support_df[
                                "comparable_rows"
                            ].sum()
                        ),

                    "minimum_rows_per_group":
                        int(
                            first_support_df[
                                "comparable_rows"
                            ].min()
                        ),

                    "maximum_rows_per_group":
                        int(
                            first_support_df[
                                "comparable_rows"
                            ].max()
                        ),
                }
            )


interaction_support_df = pd.DataFrame(
    support_rows
)


interaction_support_df.to_csv(
    INTERACTION_SUPPORT_PATH,
    index=False,
)


# --------------------------------------------------
# Paired bootstrap within each regime × threshold,
# stratified by candidate-set size
# --------------------------------------------------

bootstrap_rng = np.random.default_rng(
    INTERACTION_BOOTSTRAP_SEED
)


contrast_rows = []
replicate_frames = []


for regime in REGIME_ORDER:

    for tau in THRESHOLD_ORDER:

        paired_size_strata = []


        for n_items in CANDIDATE_SIZE_ORDER:

            first_df = (
                cluster_df[
                    (
                        cluster_df[
                            "regime"
                        ]
                        == regime
                    )
                    &
                    (
                        np.isclose(
                            cluster_df[
                                "tau"
                            ],
                            tau,
                        )
                    )
                    &
                    (
                        cluster_df[
                            "n_items"
                        ]
                        == n_items
                    )
                    &
                    (
                        cluster_df[
                            "algorithm"
                        ]
                        == FIRST_ALGORITHM
                    )
                ]
                .sort_values(
                    "matched_group_id"
                )
                .reset_index(
                    drop=True
                )
            )


            second_df = (
                cluster_df[
                    (
                        cluster_df[
                            "regime"
                        ]
                        == regime
                    )
                    &
                    (
                        np.isclose(
                            cluster_df[
                                "tau"
                            ],
                            tau,
                        )
                    )
                    &
                    (
                        cluster_df[
                            "n_items"
                        ]
                        == n_items
                    )
                    &
                    (
                        cluster_df[
                            "algorithm"
                        ]
                        == SECOND_ALGORITHM
                    )
                ]
                .sort_values(
                    "matched_group_id"
                )
                .reset_index(
                    drop=True
                )
            )


            assert len(
                first_df
            ) == len(
                second_df
            )


            assert len(
                first_df
            ) > 0


            assert (
                first_df[
                    "matched_group_id"
                ].tolist()
                ==
                second_df[
                    "matched_group_id"
                ].tolist()
            )


            paired_size_strata.append(
                {
                    "n_items":
                        n_items,

                    "matched_group_ids":
                        first_df[
                            "matched_group_id"
                        ].tolist(),

                    "first_array":
                        first_df[
                            AGGREGATE_COLUMNS
                        ]
                        .to_numpy(
                            dtype=float
                        ),

                    "second_array":
                        second_df[
                            AGGREGATE_COLUMNS
                        ]
                        .to_numpy(
                            dtype=float
                        ),
                }
            )


        observed_first_totals = np.zeros(
            len(
                AGGREGATE_COLUMNS
            ),
            dtype=float,
        )


        observed_second_totals = np.zeros(
            len(
                AGGREGATE_COLUMNS
            ),
            dtype=float,
        )


        for size_stratum in paired_size_strata:

            observed_first_totals += (
                size_stratum[
                    "first_array"
                ]
                .sum(
                    axis=0
                )
            )


            observed_second_totals += (
                size_stratum[
                    "second_array"
                ]
                .sum(
                    axis=0
                )
            )


        observed_difference = (
            calculate_interaction_metrics(
                observed_first_totals
            )
            -
            calculate_interaction_metrics(
                observed_second_totals
            )
        )


        bootstrap_differences = np.empty(
            (
                INTERACTION_BOOTSTRAP_REPLICATES,
                len(
                    INTERACTION_METRIC_NAMES
                ),
            ),
            dtype=float,
        )


        for bootstrap_index in range(
            INTERACTION_BOOTSTRAP_REPLICATES
        ):

            first_totals = np.zeros(
                len(
                    AGGREGATE_COLUMNS
                ),
                dtype=float,
            )


            second_totals = np.zeros(
                len(
                    AGGREGATE_COLUMNS
                ),
                dtype=float,
            )


            for size_stratum in paired_size_strata:

                number_of_groups = len(
                    size_stratum[
                        "matched_group_ids"
                    ]
                )


                sampled_indices = (
                    bootstrap_rng.integers(
                        low=0,
                        high=number_of_groups,
                        size=number_of_groups,
                    )
                )


                first_totals += (
                    size_stratum[
                        "first_array"
                    ][
                        sampled_indices
                    ]
                    .sum(
                        axis=0
                    )
                )


                second_totals += (
                    size_stratum[
                        "second_array"
                    ][
                        sampled_indices
                    ]
                    .sum(
                        axis=0
                    )
                )


            bootstrap_differences[
                bootstrap_index
            ] = (
                calculate_interaction_metrics(
                    first_totals
                )
                -
                calculate_interaction_metrics(
                    second_totals
                )
            )


        standard_errors = (
            bootstrap_differences.std(
                axis=0,
                ddof=1,
            )
        )


        confidence_lower = np.quantile(
            bootstrap_differences,
            0.025,
            axis=0,
        )


        confidence_upper = np.quantile(
            bootstrap_differences,
            0.975,
            axis=0,
        )


        supported_groups = sum(
            len(
                size_stratum[
                    "matched_group_ids"
                ]
            )
            for size_stratum in paired_size_strata
        )


        for metric_index, metric_name in enumerate(
            INTERACTION_METRIC_NAMES
        ):

            contrast_rows.append(
                {
                    "regime":
                        regime,

                    "tau":
                        tau,

                    "first_algorithm":
                        FIRST_ALGORITHM,

                    "second_algorithm":
                        SECOND_ALGORITHM,

                    "contrast":
                        (
                            f"{FIRST_ALGORITHM}"
                            f" minus "
                            f"{SECOND_ALGORITHM}"
                        ),

                    "metric":
                        metric_name,

                    "estimated_difference":
                        observed_difference[
                            metric_index
                        ],

                    "bootstrap_standard_error":
                        standard_errors[
                            metric_index
                        ],

                    "ci_95_lower":
                        confidence_lower[
                            metric_index
                        ],

                    "ci_95_upper":
                        confidence_upper[
                            metric_index
                        ],

                    "ci_excludes_zero":
                        (
                            confidence_lower[
                                metric_index
                            ]
                            > 0.0
                        )
                        or
                        (
                            confidence_upper[
                                metric_index
                            ]
                            < 0.0
                        ),

                    "supported_matched_groups":
                        supported_groups,

                    "candidate_size_strata":
                        len(
                            paired_size_strata
                        ),

                    "bootstrap_replicates":
                        INTERACTION_BOOTSTRAP_REPLICATES,

                    "resampling_unit":
                        "matched_size_replicate_group",

                    "resampling_design":
                        (
                            "paired_across_algorithms_"
                            "within_regime_threshold_"
                            "stratified_by_candidate_size"
                        ),

                    "estimand":
                        (
                            "algorithm_difference_"
                            "conditional_on_feasibility"
                        ),

                    "multiple_comparison_adjustment":
                        "none",
                }
            )


        replicate_df = pd.DataFrame(
            bootstrap_differences,
            columns=INTERACTION_METRIC_NAMES,
        )


        replicate_df.insert(
            0,
            "bootstrap_replicate",
            np.arange(
                INTERACTION_BOOTSTRAP_REPLICATES
            ),
        )


        replicate_df.insert(
            0,
            "tau",
            tau,
        )


        replicate_df.insert(
            0,
            "regime",
            regime,
        )


        replicate_frames.append(
            replicate_df
        )


        print(
            "Completed regime-threshold contrast:",
            regime,
            tau
        )


# --------------------------------------------------
# Save outputs
# --------------------------------------------------

interaction_contrast_df = pd.DataFrame(
    contrast_rows
)


interaction_replicates_df = pd.concat(
    replicate_frames,
    ignore_index=True,
)


interaction_contrast_df.to_csv(
    INTERACTION_CONTRAST_PATH,
    index=False,
)


interaction_replicates_df.to_csv(
    INTERACTION_REPLICATES_PATH,
    index=False,
)


# --------------------------------------------------
# Readable displays
# --------------------------------------------------

key_metrics = [
    "mean_optimality_gap_percent",
    "exact_cost_recovery_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "more_items_percent",
    "mean_selected_count_difference",
]


contrast_display_df = (
    interaction_contrast_df[
        interaction_contrast_df[
            "metric"
        ].isin(
            key_metrics
        )
    ]
    .copy()
)


contrast_display_df[
    "difference_and_ci"
] = (
    contrast_display_df[
        "estimated_difference"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    contrast_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    contrast_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


summary_metrics = [
    "mean_optimality_gap_percent",
    "large_gap_percent",
    "more_items_percent",
]


interaction_map_df = (
    contrast_display_df[
        contrast_display_df[
            "metric"
        ].isin(
            summary_metrics
        )
    ]
    .pivot(
        index=[
            "regime",
            "tau",
        ],
        columns="metric",
        values="difference_and_ci",
    )
    .reset_index()
)


print(
    "\n======================================"
)

print(
    "DYNAMIC AND STATIC-RATIO RESULTS BY REGIME × THRESHOLD"
)

print(
    "======================================"
)


print(
    interaction_descriptive_df[
        [
            "regime",
            "tau",
            "algorithm",
            "matched_groups",
            "comparable_rows",
            "mean_optimality_gap_percent",
            "exact_cost_recovery_percent",
            "mean_jaccard_similarity",
            "large_gap_percent",
            "more_items_percent",
            "mean_selected_count_difference",
        ]
    ].to_string(
        index=False,
        float_format=lambda value:
            f"{value:.4f}",
    )
)


print(
    "\n======================================"
)

print(
    "REGIME × THRESHOLD COMPARABLE SUPPORT"
)

print(
    "======================================"
)


print(
    interaction_support_df.to_string(
        index=False
    )
)


print(
    "\n======================================"
)

print(
    "PAIRED DYNAMIC-MINUS-STATIC INTERACTION CONTRASTS"
)

print(
    "======================================"
)


print(
    contrast_display_df[
        [
            "regime",
            "tau",
            "metric",
            "difference_and_ci",
            "ci_excludes_zero",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\n======================================"
)

print(
    "COMPACT INTERACTION MAP"
)

print(
    "======================================"
)


print(
    interaction_map_df.to_string(
        index=False
    )
)


print(
    "\nInteraction descriptive results saved to:"
)

print(
    INTERACTION_DESCRIPTIVE_PATH
)


print(
    "\nInteraction support counts saved to:"
)

print(
    INTERACTION_SUPPORT_PATH
)


print(
    "\nInteraction paired contrasts saved to:"
)

print(
    INTERACTION_CONTRAST_PATH
)


print(
    "\nBootstrap replicates saved to:"
)

print(
    INTERACTION_REPLICATES_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Step 123 — Simultaneous bootstrap confidence
# intervals for regime × threshold interaction cells
# --------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------
# Configuration
# --------------------------------------------------

SIMULTANEOUS_BOOTSTRAP_REPLICATES = 5000

SIMULTANEOUS_BOOTSTRAP_SEED = (
    globals().get(
        "GLOBAL_SEED",
        2026,
    )
    + 123
)


if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


ALGORITHM_LONG_PATH = (
    RESULTS_DIR
    / "corrected_v2_algorithm_comparison_long.csv"
)


SIMULTANEOUS_RESULTS_PATH = (
    TABLES_DIR
    / (
        "corrected_v2_dynamic_static_ratio_"
        "regime_threshold_simultaneous_ci.csv"
    )
)


SIMULTANEOUS_REPLICATES_PATH = (
    RESULTS_DIR
    / (
        "corrected_v2_dynamic_static_ratio_"
        "regime_threshold_shared_bootstrap_replicates.csv"
    )
)


REGIME_ORDER = [
    "independent",
    "utility_burden_alignment",
    "utility_burden_conflict",
]


THRESHOLD_ORDER = [
    0.70,
    0.80,
    0.90,
    0.95,
]


CANDIDATE_SIZE_ORDER = [
    8,
    10,
    12,
    14,
    16,
]


FIRST_ALGORITHM = "dynamic_greedy"

SECOND_ALGORITHM = (
    "static_utility_cost_ratio"
)


FOCAL_METRICS = [
    "mean_optimality_gap_percent",
    "large_gap_percent",
    "more_items_percent",
]


AGGREGATE_COLUMNS = [
    "comparable_rows",
    "gap_sum",
    "large_gap_sum",
    "more_items_sum",
]


CELL_ORDER = [
    (
        regime,
        tau,
    )
    for regime in REGIME_ORDER
    for tau in THRESHOLD_ORDER
]


# --------------------------------------------------
# Load combined algorithm data
# --------------------------------------------------

simultaneous_df = pd.read_csv(
    ALGORITHM_LONG_PATH,
    low_memory=False,
)


required_columns = [
    "matched_group_id",
    "n_items",
    "regime",
    "tau",
    "algorithm",
    "exact_status",
    "algorithm_status",
    "optimality_gap_percent",
    "selected_count_difference",
]


missing_columns = [
    column_name
    for column_name in required_columns
    if column_name not in simultaneous_df.columns
]


if missing_columns:

    raise KeyError(
        "Missing required columns: "
        + ", ".join(
            missing_columns
        )
    )


for column_name in [
    "n_items",
    "tau",
    "optimality_gap_percent",
    "selected_count_difference",
]:

    simultaneous_df[
        column_name
    ] = pd.to_numeric(
        simultaneous_df[
            column_name
        ],
        errors="coerce",
    )


simultaneous_df = (
    simultaneous_df[
        simultaneous_df[
            "algorithm"
        ].isin(
            [
                FIRST_ALGORITHM,
                SECOND_ALGORITHM,
            ]
        )
    ]
    .copy()
)


# --------------------------------------------------
# Comparable-condition indicators
# --------------------------------------------------

simultaneous_df[
    "is_comparable"
] = (
    (
        simultaneous_df[
            "exact_status"
        ]
        == "OPTIMAL"
    )
    &
    (
        simultaneous_df[
            "algorithm_status"
        ]
        == "FEASIBLE"
    )
)


comparable_df = (
    simultaneous_df[
        simultaneous_df[
            "is_comparable"
        ]
    ]
    .copy()
)


assert len(
    comparable_df
) == (
    23784
    *
    2
)


comparable_df[
    "large_gap_numeric"
] = (
    comparable_df[
        "optimality_gap_percent"
    ]
    > 10.0
).astype(int)


comparable_df[
    "more_items_numeric"
] = (
    comparable_df[
        "selected_count_difference"
    ]
    > 0
).astype(int)


# --------------------------------------------------
# Sufficient statistics per matched group and cell
# --------------------------------------------------

observed_cluster_df = (
    comparable_df
    .groupby(
        [
            "matched_group_id",
            "n_items",
            "regime",
            "tau",
            "algorithm",
        ],
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "size",
        ),

        gap_sum=(
            "optimality_gap_percent",
            "sum",
        ),

        large_gap_sum=(
            "large_gap_numeric",
            "sum",
        ),

        more_items_sum=(
            "more_items_numeric",
            "sum",
        ),
    )
)


# --------------------------------------------------
# Full grid including groups with zero comparable
# rows at infeasible threshold conditions
# --------------------------------------------------

matched_group_base_df = (
    simultaneous_df[
        [
            "matched_group_id",
            "n_items",
        ]
    ]
    .drop_duplicates()
)


assert len(
    matched_group_base_df
) == 500


cell_df = pd.DataFrame(
    [
        {
            "regime":
                regime,

            "tau":
                tau,
        }
        for regime, tau in CELL_ORDER
    ]
)


algorithm_df = pd.DataFrame(
    {
        "algorithm": [
            FIRST_ALGORITHM,
            SECOND_ALGORITHM,
        ]
    }
)


full_grid_df = (
    matched_group_base_df
    .merge(
        cell_df,
        how="cross",
    )
    .merge(
        algorithm_df,
        how="cross",
    )
)


full_cluster_df = (
    full_grid_df
    .merge(
        observed_cluster_df,
        on=[
            "matched_group_id",
            "n_items",
            "regime",
            "tau",
            "algorithm",
        ],
        how="left",
    )
)


for column_name in AGGREGATE_COLUMNS:

    full_cluster_df[
        column_name
    ] = (
        full_cluster_df[
            column_name
        ]
        .fillna(0.0)
    )


assert len(
    full_cluster_df
) == (
    500
    *
    len(
        CELL_ORDER
    )
    *
    2
)


# --------------------------------------------------
# Metric conversion
# --------------------------------------------------

def calculate_focal_metrics(
    totals,
):
    """
    Convert sufficient-statistic totals into the
    three focal Exact-relative metrics.
    """

    comparable_rows = totals[
        ...,
        0
    ]


    gap_sum = totals[
        ...,
        1
    ]


    large_gap_sum = totals[
        ...,
        2
    ]


    more_items_sum = totals[
        ...,
        3
    ]


    with np.errstate(
        divide="ignore",
        invalid="ignore",
    ):

        mean_gap = (
            gap_sum
            /
            comparable_rows
        )


        large_gap_percent = (
            100.0
            *
            large_gap_sum
            /
            comparable_rows
        )


        more_items_percent = (
            100.0
            *
            more_items_sum
            /
            comparable_rows
        )


    return np.stack(
        [
            mean_gap,
            large_gap_percent,
            more_items_percent,
        ],
        axis=-1,
    )


# --------------------------------------------------
# Construct aligned group arrays
#
# Shape for each candidate size and algorithm:
#     groups × cells × aggregate statistics
# --------------------------------------------------

arrays_by_size_algorithm = {}


for n_items in CANDIDATE_SIZE_ORDER:

    arrays_by_size_algorithm[
        n_items
    ] = {}


    reference_group_ids = None


    for algorithm in [
        FIRST_ALGORITHM,
        SECOND_ALGORITHM,
    ]:

        current_df = (
            full_cluster_df[
                (
                    full_cluster_df[
                        "n_items"
                    ]
                    == n_items
                )
                &
                (
                    full_cluster_df[
                        "algorithm"
                    ]
                    == algorithm
                )
            ]
            .copy()
        )


        current_df[
            "cell_index"
        ] = current_df.apply(
            lambda row:
                CELL_ORDER.index(
                    (
                        row[
                            "regime"
                        ],
                        float(
                            row[
                                "tau"
                            ]
                        ),
                    )
                ),
            axis=1,
        )


        current_df = (
            current_df
            .sort_values(
                [
                    "matched_group_id",
                    "cell_index",
                ]
            )
            .reset_index(
                drop=True
            )
        )


        current_group_ids = (
            current_df[
                "matched_group_id"
            ]
            .drop_duplicates()
            .tolist()
        )


        assert len(
            current_group_ids
        ) == 100


        if reference_group_ids is None:

            reference_group_ids = (
                current_group_ids
            )

        else:

            assert (
                current_group_ids
                ==
                reference_group_ids
            )


        current_array = (
            current_df[
                AGGREGATE_COLUMNS
            ]
            .to_numpy(
                dtype=float
            )
            .reshape(
                100,
                len(
                    CELL_ORDER
                ),
                len(
                    AGGREGATE_COLUMNS
                ),
            )
        )


        arrays_by_size_algorithm[
            n_items
        ][
            algorithm
        ] = current_array


# --------------------------------------------------
# Observed cell contrasts
# --------------------------------------------------

observed_totals = {
    algorithm: np.zeros(
        (
            len(
                CELL_ORDER
            ),
            len(
                AGGREGATE_COLUMNS
            ),
        ),
        dtype=float,
    )
    for algorithm in [
        FIRST_ALGORITHM,
        SECOND_ALGORITHM,
    ]
}


for n_items in CANDIDATE_SIZE_ORDER:

    for algorithm in [
        FIRST_ALGORITHM,
        SECOND_ALGORITHM,
    ]:

        observed_totals[
            algorithm
        ] += (
            arrays_by_size_algorithm[
                n_items
            ][
                algorithm
            ]
            .sum(
                axis=0
            )
        )


observed_metrics = {
    algorithm:
        calculate_focal_metrics(
            observed_totals[
                algorithm
            ]
        )
    for algorithm in [
        FIRST_ALGORITHM,
        SECOND_ALGORITHM,
    ]
}


observed_difference = (
    observed_metrics[
        FIRST_ALGORITHM
    ]
    -
    observed_metrics[
        SECOND_ALGORITHM
    ]
)


assert observed_difference.shape == (
    12,
    3,
)


# --------------------------------------------------
# Shared bootstrap across all interaction cells
#
# The same resampled matched-group indices are used
# across regimes, thresholds, and both algorithms.
# --------------------------------------------------

bootstrap_rng = np.random.default_rng(
    SIMULTANEOUS_BOOTSTRAP_SEED
)


bootstrap_differences = np.empty(
    (
        SIMULTANEOUS_BOOTSTRAP_REPLICATES,
        len(
            CELL_ORDER
        ),
        len(
            FOCAL_METRICS
        ),
    ),
    dtype=float,
)


for bootstrap_index in range(
    SIMULTANEOUS_BOOTSTRAP_REPLICATES
):

    resampled_totals = {
        algorithm: np.zeros(
            (
                len(
                    CELL_ORDER
                ),
                len(
                    AGGREGATE_COLUMNS
                ),
            ),
            dtype=float,
        )
        for algorithm in [
            FIRST_ALGORITHM,
            SECOND_ALGORITHM,
        ]
    }


    for n_items in CANDIDATE_SIZE_ORDER:

        number_of_groups = (
            arrays_by_size_algorithm[
                n_items
            ][
                FIRST_ALGORITHM
            ].shape[0]
        )


        sampled_indices = (
            bootstrap_rng.integers(
                low=0,
                high=number_of_groups,
                size=number_of_groups,
            )
        )


        for algorithm in [
            FIRST_ALGORITHM,
            SECOND_ALGORITHM,
        ]:

            resampled_totals[
                algorithm
            ] += (
                arrays_by_size_algorithm[
                    n_items
                ][
                    algorithm
                ][
                    sampled_indices
                ]
                .sum(
                    axis=0
                )
            )


    first_metrics = calculate_focal_metrics(
        resampled_totals[
            FIRST_ALGORITHM
        ]
    )


    second_metrics = calculate_focal_metrics(
        resampled_totals[
            SECOND_ALGORITHM
        ]
    )


    bootstrap_differences[
        bootstrap_index
    ] = (
        first_metrics
        -
        second_metrics
    )


# --------------------------------------------------
# Simultaneous max-t intervals
#
# Family-wise error is controlled across the
# 12 regime × threshold cells separately for each
# focal metric.
# --------------------------------------------------

result_rows = []


for metric_index, metric_name in enumerate(
    FOCAL_METRICS
):

    metric_bootstrap = (
        bootstrap_differences[
            :,
            :,
            metric_index
        ]
    )


    metric_observed = (
        observed_difference[
            :,
            metric_index
        ]
    )


    bootstrap_standard_errors = (
        metric_bootstrap.std(
            axis=0,
            ddof=1,
        )
    )


    unadjusted_lower = np.quantile(
        metric_bootstrap,
        0.025,
        axis=0,
    )


    unadjusted_upper = np.quantile(
        metric_bootstrap,
        0.975,
        axis=0,
    )


    centered_deviations = (
        metric_bootstrap
        -
        metric_observed[
            None,
            :
        ]
    )


    standardized_deviations = np.zeros_like(
        centered_deviations
    )


    positive_se_mask = (
        bootstrap_standard_errors
        > 0.0
    )


    standardized_deviations[
        :,
        positive_se_mask
    ] = (
        centered_deviations[
            :,
            positive_se_mask
        ]
        /
        bootstrap_standard_errors[
            positive_se_mask
        ][
            None,
            :
        ]
    )


    maximum_absolute_t = np.max(
        np.abs(
            standardized_deviations
        ),
        axis=1,
    )


    max_t_critical_value = float(
        np.quantile(
            maximum_absolute_t,
            0.95,
        )
    )


    simultaneous_lower = (
        metric_observed
        -
        max_t_critical_value
        *
        bootstrap_standard_errors
    )


    simultaneous_upper = (
        metric_observed
        +
        max_t_critical_value
        *
        bootstrap_standard_errors
    )


    for cell_index, (
        regime,
        tau,
    ) in enumerate(
        CELL_ORDER
    ):

        result_rows.append(
            {
                "regime":
                    regime,

                "tau":
                    tau,

                "metric":
                    metric_name,

                "estimated_difference":
                    metric_observed[
                        cell_index
                    ],

                "bootstrap_standard_error":
                    bootstrap_standard_errors[
                        cell_index
                    ],

                "unadjusted_ci_95_lower":
                    unadjusted_lower[
                        cell_index
                    ],

                "unadjusted_ci_95_upper":
                    unadjusted_upper[
                        cell_index
                    ],

                "unadjusted_ci_excludes_zero":
                    (
                        unadjusted_lower[
                            cell_index
                        ]
                        > 0.0
                    )
                    or
                    (
                        unadjusted_upper[
                            cell_index
                        ]
                        < 0.0
                    ),

                "simultaneous_ci_95_lower":
                    simultaneous_lower[
                        cell_index
                    ],

                "simultaneous_ci_95_upper":
                    simultaneous_upper[
                        cell_index
                    ],

                "simultaneous_ci_excludes_zero":
                    (
                        simultaneous_lower[
                            cell_index
                        ]
                        > 0.0
                    )
                    or
                    (
                        simultaneous_upper[
                            cell_index
                        ]
                        < 0.0
                    ),

                "max_t_critical_value":
                    max_t_critical_value,

                "familywise_cells":
                    len(
                        CELL_ORDER
                    ),

                "familywise_scope":
                    (
                        "12 regime-threshold cells "
                        "within this metric"
                    ),

                "bootstrap_replicates":
                    SIMULTANEOUS_BOOTSTRAP_REPLICATES,

                "resampling_unit":
                    "matched_size_replicate_group",

                "resampling_design":
                    (
                        "shared_across_regimes_"
                        "thresholds_and_algorithms_"
                        "stratified_by_candidate_size"
                    ),
            }
        )


# --------------------------------------------------
# Save result table
# --------------------------------------------------

simultaneous_results_df = pd.DataFrame(
    result_rows
)


simultaneous_results_df.to_csv(
    SIMULTANEOUS_RESULTS_PATH,
    index=False,
)


# --------------------------------------------------
# Save bootstrap replicates in long form
# --------------------------------------------------

replicate_rows = []


for bootstrap_index in range(
    SIMULTANEOUS_BOOTSTRAP_REPLICATES
):

    for cell_index, (
        regime,
        tau,
    ) in enumerate(
        CELL_ORDER
    ):

        replicate_rows.append(
            {
                "bootstrap_replicate":
                    bootstrap_index,

                "regime":
                    regime,

                "tau":
                    tau,

                "mean_optimality_gap_percent":
                    bootstrap_differences[
                        bootstrap_index,
                        cell_index,
                        0,
                    ],

                "large_gap_percent":
                    bootstrap_differences[
                        bootstrap_index,
                        cell_index,
                        1,
                    ],

                "more_items_percent":
                    bootstrap_differences[
                        bootstrap_index,
                        cell_index,
                        2,
                    ],
            }
        )


simultaneous_replicates_df = pd.DataFrame(
    replicate_rows
)


simultaneous_replicates_df.to_csv(
    SIMULTANEOUS_REPLICATES_PATH,
    index=False,
)


# --------------------------------------------------
# Compact display
# --------------------------------------------------

display_df = simultaneous_results_df.copy()


display_df[
    "unadjusted_ci"
] = (
    display_df[
        "estimated_difference"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    display_df[
        "unadjusted_ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    display_df[
        "unadjusted_ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


display_df[
    "simultaneous_ci"
] = (
    display_df[
        "estimated_difference"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    display_df[
        "simultaneous_ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    display_df[
        "simultaneous_ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


compact_map_df = (
    display_df
    .pivot(
        index=[
            "regime",
            "tau",
        ],
        columns="metric",
        values="simultaneous_ci",
    )
    .reset_index()
)


print(
    "======================================"
)

print(
    "SIMULTANEOUS REGIME × THRESHOLD INTERVALS"
)

print(
    "======================================"
)


print(
    display_df[
        [
            "regime",
            "tau",
            "metric",
            "unadjusted_ci",
            "simultaneous_ci",
            "unadjusted_ci_excludes_zero",
            "simultaneous_ci_excludes_zero",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\n======================================"
)

print(
    "SIMULTANEOUS COMPACT INTERACTION MAP"
)

print(
    "======================================"
)


print(
    compact_map_df.to_string(
        index=False
    )
)


print(
    "\nSimultaneous confidence intervals saved to:"
)

print(
    SIMULTANEOUS_RESULTS_PATH
)


print(
    "\nShared bootstrap replicates saved to:"
)

print(
    SIMULTANEOUS_REPLICATES_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Step 124 — Coverage-based submodular utility:
# matched marginal-preserving generator validation
# --------------------------------------------------

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------
# Configuration
# --------------------------------------------------

COVERAGE_GENERATOR_VERSION = (
    "coverage_submodular_marginal_preserving_v1"
)

COVERAGE_VALIDATION_SEED = (
    globals().get(
        "GLOBAL_SEED",
        2026,
    )
    + 124
)

COVERAGE_VALIDATION_INSTANCES = 500
COVERAGE_VALIDATION_N_ITEMS = 64
COVERAGE_FEATURES = 12

# Probability that an item provides nonzero coverage
# for a particular decision-relevant feature.
COVERAGE_DENSITY = 0.35

# Controls the noisy rank association between
# singleton utility and burden.
COVERAGE_REGIME_STRENGTH = 0.75


if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "generate_mst_instance" not in globals():

    raise NameError(
        "generate_mst_instance is unavailable. "
        "Run the corrected marginal-preserving-v2 "
        "generator cell first."
    )


COVERAGE_VALIDATION_RAW_PATH = (
    RESULTS_DIR
    / "coverage_generator_validation_raw.csv"
)


COVERAGE_VALIDATION_SUMMARY_PATH = (
    RESULTS_DIR
    / "coverage_generator_validation_summary.csv"
)


COVERAGE_VALIDATION_METADATA_PATH = (
    RESULTS_DIR
    / "coverage_generator_metadata.csv"
)


COVERAGE_REGIME_ORDER = [
    "independent",
    "utility_burden_alignment",
    "utility_burden_conflict",
]


# --------------------------------------------------
# Coverage item representation
# --------------------------------------------------

@dataclass(frozen=True)
class CoverageInformationItem:
    """
    Information item for the coverage-based
    submodular benchmark.
    """

    item_id: str

    privacy: float

    cognitive_cost: float

    feature_coverage: tuple


# --------------------------------------------------
# Coverage-based utility
# --------------------------------------------------

def coverage_utility(
    selected_items,
    feature_weights,
):
    """
    Weighted maximum coverage:

        U(S) = sum_f w_f max_{z in S} a_zf / sum_f w_f

    This function is normalized, monotone, and
    submodular for nonnegative feature weights and
    coverage values.
    """

    feature_weights = np.asarray(
        feature_weights,
        dtype=float,
    )


    if feature_weights.ndim != 1:

        raise ValueError(
            "feature_weights must be one-dimensional."
        )


    if np.any(
        feature_weights < 0.0
    ):

        raise ValueError(
            "feature_weights must be nonnegative."
        )


    total_feature_weight = (
        feature_weights.sum()
    )


    if total_feature_weight <= 0.0:

        raise ValueError(
            "At least one feature weight must be positive."
        )


    if len(
        selected_items
    ) == 0:

        return 0.0


    coverage_matrix = np.asarray(
        [
            item.feature_coverage
            for item in selected_items
        ],
        dtype=float,
    )


    maximum_feature_coverage = (
        coverage_matrix.max(
            axis=0
        )
    )


    return float(
        np.dot(
            feature_weights,
            maximum_feature_coverage,
        )
        /
        total_feature_weight
    )


# --------------------------------------------------
# Coverage matrix generation
# --------------------------------------------------

def generate_coverage_structure(
    n_items,
    n_features,
    seed,
    coverage_density,
):
    """
    Generate matched feature weights and an
    item-by-feature coverage matrix.

    Each item covers at least one feature, and each
    feature is covered by at least one item.
    """

    rng = np.random.default_rng(
        seed
    )


    # Unequal but bounded decision-feature importance.
    feature_weights = rng.lognormal(
        mean=0.0,
        sigma=0.45,
        size=n_features,
    )


    feature_weights = (
        feature_weights
        /
        feature_weights.sum()
    )


    active_mask = (
        rng.random(
            (
                n_items,
                n_features,
            )
        )
        <
        coverage_density
    )


    # Ensure every item carries some decision-relevant
    # information.
    empty_item_indices = np.where(
        active_mask.sum(
            axis=1
        )
        == 0
    )[0]


    for item_index in empty_item_indices:

        selected_feature = rng.integers(
            low=0,
            high=n_features,
        )


        active_mask[
            item_index,
            selected_feature,
        ] = True


    # Ensure every feature appears in the candidate
    # information pool.
    empty_feature_indices = np.where(
        active_mask.sum(
            axis=0
        )
        == 0
    )[0]


    for feature_index in empty_feature_indices:

        selected_item = rng.integers(
            low=0,
            high=n_items,
        )


        active_mask[
            selected_item,
            feature_index,
        ] = True


    coverage_strengths = rng.beta(
        a=2.5,
        b=1.5,
        size=(
            n_items,
            n_features,
        ),
    )


    coverage_matrix = np.where(
        active_mask,
        coverage_strengths,
        0.0,
    )


    return (
        feature_weights,
        coverage_matrix,
    )


# --------------------------------------------------
# Marginal-preserving burden assignment
# --------------------------------------------------

def noisy_rank_assignment(
    burden_values,
    utility_proxy,
    association_direction,
    strength,
    rng,
):
    """
    Reassign an unchanged burden-value multiset using
    a noisy utility-rank score.

    association_direction:
         0 -> independent
        +1 -> utility-burden conflict
        -1 -> utility-burden alignment
    """

    burden_values = np.asarray(
        burden_values,
        dtype=float,
    )


    utility_proxy = np.asarray(
        utility_proxy,
        dtype=float,
    )


    if len(
        burden_values
    ) != len(
        utility_proxy
    ):

        raise ValueError(
            "Burden and utility arrays must have "
            "the same length."
        )


    n_items = len(
        burden_values
    )


    sorted_burden_values = np.sort(
        burden_values
    )


    if association_direction == 0:

        assignment_order = rng.permutation(
            n_items
        )


    else:

        utility_ranks = pd.Series(
            utility_proxy
        ).rank(
            method="average"
        ).to_numpy(
            dtype=float
        )


        utility_rank_z = (
            utility_ranks
            -
            utility_ranks.mean()
        )


        utility_rank_sd = (
            utility_rank_z.std(
                ddof=0
            )
        )


        if utility_rank_sd > 0.0:

            utility_rank_z = (
                utility_rank_z
                /
                utility_rank_sd
            )


        random_noise = rng.normal(
            loc=0.0,
            scale=1.0,
            size=n_items,
        )


        latent_score = (
            association_direction
            *
            strength
            *
            utility_rank_z
            +
            np.sqrt(
                max(
                    0.0,
                    1.0
                    -
                    strength ** 2,
                )
            )
            *
            random_noise
        )


        assignment_order = np.argsort(
            latent_score,
            kind="mergesort",
        )


    assigned_values = np.empty(
        n_items,
        dtype=float,
    )


    assigned_values[
        assignment_order
    ] = sorted_burden_values


    return assigned_values


# --------------------------------------------------
# Matched coverage instance generator
# --------------------------------------------------

def generate_matched_coverage_instances(
    n_items,
    n_features,
    seed,
    coverage_density=
        COVERAGE_DENSITY,
    regime_strength=
        COVERAGE_REGIME_STRENGTH,
):
    """
    Generate three matched structural regimes.

    Across regimes, the following are identical:
    - feature weights;
    - item-feature coverage values;
    - singleton utility values;
    - sorted privacy values;
    - sorted cognitive-cost values.

    Only the association between singleton utility and
    burden changes.
    """

    feature_weights, coverage_matrix = (
        generate_coverage_structure(
            n_items=n_items,
            n_features=n_features,
            seed=seed,
            coverage_density=
                coverage_density,
        )
    )


    singleton_utilities = (
        coverage_matrix
        @
        feature_weights
    )


    # Reuse the corrected-v2 burden distributions so
    # that the two utility benchmark families remain
    # comparable in their burden marginals.
    burden_source_items = (
        generate_mst_instance(
            n_items=n_items,
            regime="independent",
            seed=seed,
        )
    )


    base_privacy_values = np.asarray(
        [
            float(
                item.privacy
            )
            for item in burden_source_items
        ],
        dtype=float,
    )


    base_cognitive_values = np.asarray(
        [
            float(
                item.cognitive_cost
            )
            for item in burden_source_items
        ],
        dtype=float,
    )


    regime_directions = {
        "independent":
            0,

        "utility_burden_alignment":
            -1,

        "utility_burden_conflict":
            1,
    }


    matched_instances = {}


    for regime_index, regime in enumerate(
        COVERAGE_REGIME_ORDER
    ):

        direction = regime_directions[
            regime
        ]


        privacy_rng = np.random.default_rng(
            seed
            +
            10_000
            +
            100
            *
            regime_index
        )


        cognitive_rng = np.random.default_rng(
            seed
            +
            20_000
            +
            100
            *
            regime_index
        )


        assigned_privacy = (
            noisy_rank_assignment(
                burden_values=
                    base_privacy_values,

                utility_proxy=
                    singleton_utilities,

                association_direction=
                    direction,

                strength=
                    regime_strength,

                rng=
                    privacy_rng,
            )
        )


        assigned_cognitive = (
            noisy_rank_assignment(
                burden_values=
                    base_cognitive_values,

                utility_proxy=
                    singleton_utilities,

                association_direction=
                    direction,

                strength=
                    regime_strength,

                rng=
                    cognitive_rng,
            )
        )


        items = [
            CoverageInformationItem(
                item_id=
                    f"c{item_index + 1:03d}",

                privacy=
                    float(
                        assigned_privacy[
                            item_index
                        ]
                    ),

                cognitive_cost=
                    float(
                        assigned_cognitive[
                            item_index
                        ]
                    ),

                feature_coverage=
                    tuple(
                        float(value)
                        for value in coverage_matrix[
                            item_index
                        ]
                    ),
            )
            for item_index in range(
                n_items
            )
        ]


        matched_instances[
            regime
        ] = {
            "items":
                items,

            "feature_weights":
                feature_weights.copy(),

            "coverage_matrix":
                coverage_matrix.copy(),

            "singleton_utilities":
                singleton_utilities.copy(),
        }


    return matched_instances


# --------------------------------------------------
# Validation run
# --------------------------------------------------

validation_rows = []
marginal_audit_rows = []


for replicate in range(
    COVERAGE_VALIDATION_INSTANCES
):

    matched_seed = (
        COVERAGE_VALIDATION_SEED
        +
        1_000_000
        +
        replicate
    )


    matched_instances = (
        generate_matched_coverage_instances(
            n_items=
                COVERAGE_VALIDATION_N_ITEMS,

            n_features=
                COVERAGE_FEATURES,

            seed=
                matched_seed,
        )
    )


    reference_regime_data = (
        matched_instances[
            "independent"
        ]
    )


    reference_coverage_matrix = (
        reference_regime_data[
            "coverage_matrix"
        ]
    )


    reference_feature_weights = (
        reference_regime_data[
            "feature_weights"
        ]
    )


    reference_singleton_utility = (
        reference_regime_data[
            "singleton_utilities"
        ]
    )


    full_set_utility = coverage_utility(
        selected_items=
            reference_regime_data[
                "items"
            ],

        feature_weights=
            reference_feature_weights,
    )


    nonzero_coverage_fraction = (
        reference_coverage_matrix
        >
        0.0
    ).mean()


    privacy_values_by_regime = {}
    cognitive_values_by_regime = {}


    for regime in COVERAGE_REGIME_ORDER:

        regime_data = matched_instances[
            regime
        ]


        regime_items = regime_data[
            "items"
        ]


        privacy_values = np.asarray(
            [
                item.privacy
                for item in regime_items
            ],
            dtype=float,
        )


        cognitive_values = np.asarray(
            [
                item.cognitive_cost
                for item in regime_items
            ],
            dtype=float,
        )


        privacy_values_by_regime[
            regime
        ] = privacy_values


        cognitive_values_by_regime[
            regime
        ] = cognitive_values


        utility_series = pd.Series(
            reference_singleton_utility
        )


        privacy_series = pd.Series(
            privacy_values
        )


        cognitive_series = pd.Series(
            cognitive_values
        )


        validation_rows.append(
            {
                "generator_version":
                    COVERAGE_GENERATOR_VERSION,

                "replicate":
                    replicate,

                "matched_seed":
                    matched_seed,

                "n_items":
                    COVERAGE_VALIDATION_N_ITEMS,

                "n_features":
                    COVERAGE_FEATURES,

                "regime":
                    regime,

                "regime_strength":
                    COVERAGE_REGIME_STRENGTH,

                "coverage_density_parameter":
                    COVERAGE_DENSITY,

                "observed_nonzero_coverage_fraction":
                    nonzero_coverage_fraction,

                "full_set_utility":
                    full_set_utility,

                "mean_singleton_utility":
                    reference_singleton_utility.mean(),

                "sd_singleton_utility":
                    reference_singleton_utility.std(
                        ddof=1
                    ),

                "utility_privacy_pearson":
                    utility_series.corr(
                        privacy_series,
                        method="pearson",
                    ),

                "utility_privacy_spearman":
                    utility_series.corr(
                        privacy_series,
                        method="spearman",
                    ),

                "utility_cognitive_pearson":
                    utility_series.corr(
                        cognitive_series,
                        method="pearson",
                    ),

                "utility_cognitive_spearman":
                    utility_series.corr(
                        cognitive_series,
                        method="spearman",
                    ),

                "mean_privacy":
                    privacy_values.mean(),

                "sd_privacy":
                    privacy_values.std(
                        ddof=1
                    ),

                "mean_cognitive":
                    cognitive_values.mean(),

                "sd_cognitive":
                    cognitive_values.std(
                        ddof=1
                    ),
            }
        )


    independent_privacy = np.sort(
        privacy_values_by_regime[
            "independent"
        ]
    )


    independent_cognitive = np.sort(
        cognitive_values_by_regime[
            "independent"
        ]
    )


    maximum_privacy_marginal_difference = 0.0
    maximum_cognitive_marginal_difference = 0.0


    for regime in COVERAGE_REGIME_ORDER:

        maximum_privacy_marginal_difference = max(
            maximum_privacy_marginal_difference,

            float(
                np.max(
                    np.abs(
                        np.sort(
                            privacy_values_by_regime[
                                regime
                            ]
                        )
                        -
                        independent_privacy
                    )
                )
            ),
        )


        maximum_cognitive_marginal_difference = max(
            maximum_cognitive_marginal_difference,

            float(
                np.max(
                    np.abs(
                        np.sort(
                            cognitive_values_by_regime[
                                regime
                            ]
                        )
                        -
                        independent_cognitive
                    )
                )
            ),
        )


    marginal_audit_rows.append(
        {
            "replicate":
                replicate,

            "matched_seed":
                matched_seed,

            "maximum_sorted_privacy_difference":
                maximum_privacy_marginal_difference,

            "maximum_sorted_cognitive_difference":
                maximum_cognitive_marginal_difference,
        }
    )


# --------------------------------------------------
# Summaries
# --------------------------------------------------

coverage_validation_raw_df = pd.DataFrame(
    validation_rows
)


coverage_marginal_audit_df = pd.DataFrame(
    marginal_audit_rows
)


coverage_validation_summary_df = (
    coverage_validation_raw_df
    .groupby(
        "regime",
        as_index=False,
    )
    .agg(
        instances=(
            "replicate",
            "nunique",
        ),

        mean_utility_privacy_pearson=(
            "utility_privacy_pearson",
            "mean",
        ),

        mean_utility_privacy_spearman=(
            "utility_privacy_spearman",
            "mean",
        ),

        mean_utility_cognitive_pearson=(
            "utility_cognitive_pearson",
            "mean",
        ),

        mean_utility_cognitive_spearman=(
            "utility_cognitive_spearman",
            "mean",
        ),

        mean_singleton_utility=(
            "mean_singleton_utility",
            "mean",
        ),

        mean_full_set_utility=(
            "full_set_utility",
            "mean",
        ),

        minimum_full_set_utility=(
            "full_set_utility",
            "min",
        ),

        mean_privacy=(
            "mean_privacy",
            "mean",
        ),

        mean_privacy_sd=(
            "sd_privacy",
            "mean",
        ),

        mean_cognitive=(
            "mean_cognitive",
            "mean",
        ),

        mean_cognitive_sd=(
            "sd_cognitive",
            "mean",
        ),

        mean_observed_coverage_density=(
            "observed_nonzero_coverage_fraction",
            "mean",
        ),
    )
)


coverage_metadata_df = pd.DataFrame(
    [
        {
            "generator_version":
                COVERAGE_GENERATOR_VERSION,

            "validation_seed":
                COVERAGE_VALIDATION_SEED,

            "validation_instances":
                COVERAGE_VALIDATION_INSTANCES,

            "validation_n_items":
                COVERAGE_VALIDATION_N_ITEMS,

            "n_features":
                COVERAGE_FEATURES,

            "coverage_density":
                COVERAGE_DENSITY,

            "regime_strength":
                COVERAGE_REGIME_STRENGTH,

            "utility_function":
                (
                    "weighted maximum feature coverage"
                ),

            "burden_distribution_source":
                (
                    "corrected marginal-preserving-v2 "
                    "independent generator"
                ),

            "privacy_marginals_preserved":
                True,

            "cognitive_marginals_preserved":
                True,

            "regime_association_proxy":
                "singleton coverage utility",
        }
    ]
)


# --------------------------------------------------
# Integrity assertions
# --------------------------------------------------

assert len(
    coverage_validation_raw_df
) == (
    COVERAGE_VALIDATION_INSTANCES
    *
    len(
        COVERAGE_REGIME_ORDER
    )
)


assert (
    coverage_validation_summary_df[
        "instances"
    ]
    ==
    COVERAGE_VALIDATION_INSTANCES
).all()


assert (
    coverage_marginal_audit_df[
        "maximum_sorted_privacy_difference"
    ].max()
    <= 1e-15
)


assert (
    coverage_marginal_audit_df[
        "maximum_sorted_cognitive_difference"
    ].max()
    <= 1e-15
)


assert (
    coverage_validation_raw_df[
        "full_set_utility"
    ]
    >= 0.0
).all()


assert (
    coverage_validation_raw_df[
        "full_set_utility"
    ]
    <= 1.0
    +
    1e-12
).all()


# --------------------------------------------------
# Save outputs
# --------------------------------------------------

coverage_validation_raw_df.to_csv(
    COVERAGE_VALIDATION_RAW_PATH,
    index=False,
)


coverage_validation_summary_df.to_csv(
    COVERAGE_VALIDATION_SUMMARY_PATH,
    index=False,
)


coverage_metadata_df.to_csv(
    COVERAGE_VALIDATION_METADATA_PATH,
    index=False,
)


# --------------------------------------------------
# Print results
# --------------------------------------------------

print(
    "======================================"
)

print(
    "COVERAGE-BASED GENERATOR VALIDATION"
)

print(
    "======================================"
)


print(
    coverage_validation_summary_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.6f}",
    )
)


print(
    "\n======================================"
)

print(
    "MATCHED-MARGINAL AUDIT"
)

print(
    "======================================"
)


print(
    "Maximum sorted privacy difference:"
)

print(
    f"{coverage_marginal_audit_df['maximum_sorted_privacy_difference'].max():.16f}"
)


print(
    "\nMaximum sorted cognitive difference:"
)

print(
    f"{coverage_marginal_audit_df['maximum_sorted_cognitive_difference'].max():.16f}"
)


print(
    "\nCoverage validation raw data saved to:"
)

print(
    COVERAGE_VALIDATION_RAW_PATH
)


print(
    "\nCoverage validation summary saved to:"
)

print(
    COVERAGE_VALIDATION_SUMMARY_PATH
)


print(
    "\nCoverage generator metadata saved to:"
)

print(
    COVERAGE_VALIDATION_METADATA_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Step 125 — Coverage-based threshold-feasibility
# calibration across benchmark candidate-set sizes
# --------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------
# Configuration
# --------------------------------------------------

COVERAGE_FEASIBILITY_REPLICATES = 100

COVERAGE_FEASIBILITY_BASE_SEED = (
    globals().get(
        "GLOBAL_SEED",
        2026,
    )
    + 125
)


COVERAGE_FEASIBILITY_SIZES = [
    8,
    10,
    12,
    14,
    16,
]


COVERAGE_FEASIBILITY_THRESHOLDS = [
    0.70,
    0.80,
    0.90,
    0.95,
]


COVERAGE_FEASIBILITY_FEATURES = (
    globals().get(
        "COVERAGE_FEATURES",
        12,
    )
)


COVERAGE_FEASIBILITY_DENSITY = (
    globals().get(
        "COVERAGE_DENSITY",
        0.35,
    )
)


COVERAGE_FEASIBILITY_REGIME_STRENGTH = (
    globals().get(
        "COVERAGE_REGIME_STRENGTH",
        0.75,
    )
)


COVERAGE_FEASIBILITY_REGIMES = (
    globals().get(
        "COVERAGE_REGIME_ORDER",
        [
            "independent",
            "utility_burden_alignment",
            "utility_burden_conflict",
        ],
    )
)


if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


if "generate_matched_coverage_instances" not in globals():

    raise NameError(
        "generate_matched_coverage_instances is unavailable. "
        "Run Step 124 first."
    )


if "coverage_utility" not in globals():

    raise NameError(
        "coverage_utility is unavailable. "
        "Run Step 124 first."
    )


# --------------------------------------------------
# Output paths
# --------------------------------------------------

COVERAGE_FEASIBILITY_RAW_PATH = (
    RESULTS_DIR
    / "coverage_feasibility_calibration_raw.csv"
)


COVERAGE_FEASIBILITY_LONG_PATH = (
    RESULTS_DIR
    / "coverage_feasibility_calibration_threshold_rows.csv"
)


COVERAGE_UTILITY_SUMMARY_PATH = (
    TABLES_DIR
    / "coverage_full_set_utility_by_candidate_size.csv"
)


COVERAGE_FEASIBILITY_SUMMARY_PATH = (
    TABLES_DIR
    / "coverage_feasibility_by_candidate_size_and_threshold.csv"
)


COVERAGE_FEASIBILITY_OVERALL_PATH = (
    TABLES_DIR
    / "coverage_feasibility_overall_by_threshold.csv"
)


COVERAGE_FEASIBILITY_AUDIT_PATH = (
    RESULTS_DIR
    / "coverage_feasibility_generator_integrity_audit.csv"
)


COVERAGE_FEASIBILITY_METADATA_PATH = (
    RESULTS_DIR
    / "coverage_feasibility_calibration_metadata.csv"
)


# --------------------------------------------------
# Containers
# --------------------------------------------------

group_rows = []
threshold_rows = []
audit_rows = []


# --------------------------------------------------
# Generate matched coverage instances
# --------------------------------------------------

for n_items in COVERAGE_FEASIBILITY_SIZES:

    for replicate in range(
        COVERAGE_FEASIBILITY_REPLICATES
    ):

        matched_seed = (
            72_000_000
            +
            100_000
            *
            n_items
            +
            COVERAGE_FEASIBILITY_BASE_SEED
            +
            replicate
        )


        matched_instances = (
            generate_matched_coverage_instances(
                n_items=n_items,

                n_features=
                    COVERAGE_FEASIBILITY_FEATURES,

                seed=
                    matched_seed,

                coverage_density=
                    COVERAGE_FEASIBILITY_DENSITY,

                regime_strength=
                    COVERAGE_FEASIBILITY_REGIME_STRENGTH,
            )
        )


        reference_data = (
            matched_instances[
                "independent"
            ]
        )


        reference_feature_weights = np.asarray(
            reference_data[
                "feature_weights"
            ],
            dtype=float,
        )


        reference_coverage_matrix = np.asarray(
            reference_data[
                "coverage_matrix"
            ],
            dtype=float,
        )


        reference_singleton_utilities = np.asarray(
            reference_data[
                "singleton_utilities"
            ],
            dtype=float,
        )


        reference_items = (
            reference_data[
                "items"
            ]
        )


        full_set_utility = coverage_utility(
            selected_items=
                reference_items,

            feature_weights=
                reference_feature_weights,
        )


        observed_coverage_density = float(
            (
                reference_coverage_matrix
                >
                0.0
            ).mean()
        )


        maximum_feature_coverage = (
            reference_coverage_matrix.max(
                axis=0
            )
        )


        uncovered_feature_count = int(
            (
                maximum_feature_coverage
                <= 0.0
            ).sum()
        )


        empty_item_count = int(
            (
                reference_coverage_matrix
                .sum(
                    axis=1
                )
                <= 0.0
            ).sum()
        )


        # ------------------------------------------
        # Matched-regime integrity audit
        # ------------------------------------------

        maximum_feature_weight_difference = 0.0
        maximum_coverage_matrix_difference = 0.0
        maximum_singleton_utility_difference = 0.0
        maximum_full_set_utility_difference = 0.0
        maximum_sorted_privacy_difference = 0.0
        maximum_sorted_cognitive_difference = 0.0


        reference_privacy = np.sort(
            np.asarray(
                [
                    item.privacy
                    for item in reference_items
                ],
                dtype=float,
            )
        )


        reference_cognitive = np.sort(
            np.asarray(
                [
                    item.cognitive_cost
                    for item in reference_items
                ],
                dtype=float,
            )
        )


        for regime in COVERAGE_FEASIBILITY_REGIMES:

            regime_data = (
                matched_instances[
                    regime
                ]
            )


            regime_feature_weights = np.asarray(
                regime_data[
                    "feature_weights"
                ],
                dtype=float,
            )


            regime_coverage_matrix = np.asarray(
                regime_data[
                    "coverage_matrix"
                ],
                dtype=float,
            )


            regime_singleton_utilities = np.asarray(
                regime_data[
                    "singleton_utilities"
                ],
                dtype=float,
            )


            regime_items = (
                regime_data[
                    "items"
                ]
            )


            regime_full_set_utility = coverage_utility(
                selected_items=
                    regime_items,

                feature_weights=
                    regime_feature_weights,
            )


            regime_privacy = np.sort(
                np.asarray(
                    [
                        item.privacy
                        for item in regime_items
                    ],
                    dtype=float,
                )
            )


            regime_cognitive = np.sort(
                np.asarray(
                    [
                        item.cognitive_cost
                        for item in regime_items
                    ],
                    dtype=float,
                )
            )


            maximum_feature_weight_difference = max(
                maximum_feature_weight_difference,

                float(
                    np.max(
                        np.abs(
                            regime_feature_weights
                            -
                            reference_feature_weights
                        )
                    )
                ),
            )


            maximum_coverage_matrix_difference = max(
                maximum_coverage_matrix_difference,

                float(
                    np.max(
                        np.abs(
                            regime_coverage_matrix
                            -
                            reference_coverage_matrix
                        )
                    )
                ),
            )


            maximum_singleton_utility_difference = max(
                maximum_singleton_utility_difference,

                float(
                    np.max(
                        np.abs(
                            regime_singleton_utilities
                            -
                            reference_singleton_utilities
                        )
                    )
                ),
            )


            maximum_full_set_utility_difference = max(
                maximum_full_set_utility_difference,

                float(
                    abs(
                        regime_full_set_utility
                        -
                        full_set_utility
                    )
                ),
            )


            maximum_sorted_privacy_difference = max(
                maximum_sorted_privacy_difference,

                float(
                    np.max(
                        np.abs(
                            regime_privacy
                            -
                            reference_privacy
                        )
                    )
                ),
            )


            maximum_sorted_cognitive_difference = max(
                maximum_sorted_cognitive_difference,

                float(
                    np.max(
                        np.abs(
                            regime_cognitive
                            -
                            reference_cognitive
                        )
                    )
                ),
            )


        audit_rows.append(
            {
                "n_items":
                    n_items,

                "replicate":
                    replicate,

                "matched_seed":
                    matched_seed,

                "maximum_feature_weight_difference":
                    maximum_feature_weight_difference,

                "maximum_coverage_matrix_difference":
                    maximum_coverage_matrix_difference,

                "maximum_singleton_utility_difference":
                    maximum_singleton_utility_difference,

                "maximum_full_set_utility_difference":
                    maximum_full_set_utility_difference,

                "maximum_sorted_privacy_difference":
                    maximum_sorted_privacy_difference,

                "maximum_sorted_cognitive_difference":
                    maximum_sorted_cognitive_difference,

                "uncovered_feature_count":
                    uncovered_feature_count,

                "empty_item_count":
                    empty_item_count,
            }
        )


        # ------------------------------------------
        # One row per matched generated group
        # ------------------------------------------

        group_rows.append(
            {
                "generator_version":
                    globals().get(
                        "COVERAGE_GENERATOR_VERSION",
                        (
                            "coverage_submodular_"
                            "marginal_preserving_v1"
                        ),
                    ),

                "matched_group_id":
                    (
                        f"COV_M{n_items}_"
                        f"R{replicate:03d}"
                    ),

                "n_items":
                    n_items,

                "replicate":
                    replicate,

                "matched_seed":
                    matched_seed,

                "n_features":
                    COVERAGE_FEASIBILITY_FEATURES,

                "coverage_density_parameter":
                    COVERAGE_FEASIBILITY_DENSITY,

                "regime_strength":
                    COVERAGE_FEASIBILITY_REGIME_STRENGTH,

                "full_set_utility":
                    full_set_utility,

                "mean_singleton_utility":
                    float(
                        reference_singleton_utilities.mean()
                    ),

                "maximum_singleton_utility":
                    float(
                        reference_singleton_utilities.max()
                    ),

                "minimum_singleton_utility":
                    float(
                        reference_singleton_utilities.min()
                    ),

                "observed_coverage_density":
                    observed_coverage_density,

                "minimum_feature_maximum_coverage":
                    float(
                        maximum_feature_coverage.min()
                    ),

                "mean_feature_maximum_coverage":
                    float(
                        maximum_feature_coverage.mean()
                    ),

                "uncovered_feature_count":
                    uncovered_feature_count,

                "empty_item_count":
                    empty_item_count,
            }
        )


        # ------------------------------------------
        # Feasibility under every threshold
        # ------------------------------------------

        for tau in COVERAGE_FEASIBILITY_THRESHOLDS:

            threshold_rows.append(
                {
                    "matched_group_id":
                        (
                            f"COV_M{n_items}_"
                            f"R{replicate:03d}"
                        ),

                    "n_items":
                        n_items,

                    "replicate":
                        replicate,

                    "matched_seed":
                        matched_seed,

                    "tau":
                        tau,

                    "full_set_utility":
                        full_set_utility,

                    "utility_shortfall":
                        max(
                            0.0,
                            tau
                            -
                            full_set_utility,
                        ),

                    "is_feasible":
                        int(
                            full_set_utility
                            +
                            1e-12
                            >= tau
                        ),
                }
            )


    print(
        "Completed coverage feasibility size:",
        n_items
    )


# --------------------------------------------------
# Data frames
# --------------------------------------------------

coverage_group_df = pd.DataFrame(
    group_rows
)


coverage_threshold_df = pd.DataFrame(
    threshold_rows
)


coverage_audit_df = pd.DataFrame(
    audit_rows
)


# --------------------------------------------------
# Full-set utility summary by candidate-set size
# --------------------------------------------------

coverage_utility_summary_df = (
    coverage_group_df
    .groupby(
        "n_items",
        as_index=False,
    )
    .agg(
        matched_groups=(
            "matched_group_id",
            "nunique",
        ),

        mean_full_set_utility=(
            "full_set_utility",
            "mean",
        ),

        sd_full_set_utility=(
            "full_set_utility",
            "std",
        ),

        minimum_full_set_utility=(
            "full_set_utility",
            "min",
        ),

        p05_full_set_utility=(
            "full_set_utility",
            lambda values:
                values.quantile(
                    0.05
                ),
        ),

        median_full_set_utility=(
            "full_set_utility",
            "median",
        ),

        p95_full_set_utility=(
            "full_set_utility",
            lambda values:
                values.quantile(
                    0.95
                ),
        ),

        maximum_full_set_utility=(
            "full_set_utility",
            "max",
        ),

        mean_singleton_utility=(
            "mean_singleton_utility",
            "mean",
        ),

        mean_observed_coverage_density=(
            "observed_coverage_density",
            "mean",
        ),

        minimum_feature_maximum_coverage=(
            "minimum_feature_maximum_coverage",
            "min",
        ),
    )
)


# --------------------------------------------------
# Feasibility by size and threshold
# --------------------------------------------------

coverage_feasibility_summary_df = (
    coverage_threshold_df
    .groupby(
        [
            "n_items",
            "tau",
        ],
        as_index=False,
    )
    .agg(
        matched_groups=(
            "matched_group_id",
            "nunique",
        ),

        feasible_groups=(
            "is_feasible",
            "sum",
        ),

        mean_full_set_utility=(
            "full_set_utility",
            "mean",
        ),

        minimum_full_set_utility=(
            "full_set_utility",
            "min",
        ),

        mean_utility_shortfall=(
            "utility_shortfall",
            "mean",
        ),

        maximum_utility_shortfall=(
            "utility_shortfall",
            "max",
        ),
    )
)


coverage_feasibility_summary_df[
    "feasibility_percent"
] = (
    100.0
    *
    coverage_feasibility_summary_df[
        "feasible_groups"
    ]
    /
    coverage_feasibility_summary_df[
        "matched_groups"
    ]
)


# --------------------------------------------------
# Overall feasibility by threshold
# --------------------------------------------------

coverage_feasibility_overall_df = (
    coverage_threshold_df
    .groupby(
        "tau",
        as_index=False,
    )
    .agg(
        matched_groups=(
            "matched_group_id",
            "nunique",
        ),

        feasible_groups=(
            "is_feasible",
            "sum",
        ),

        mean_full_set_utility=(
            "full_set_utility",
            "mean",
        ),

        mean_utility_shortfall=(
            "utility_shortfall",
            "mean",
        ),

        maximum_utility_shortfall=(
            "utility_shortfall",
            "max",
        ),
    )
)


coverage_feasibility_overall_df[
    "feasibility_percent"
] = (
    100.0
    *
    coverage_feasibility_overall_df[
        "feasible_groups"
    ]
    /
    coverage_feasibility_overall_df[
        "matched_groups"
    ]
)


# --------------------------------------------------
# Integrity assertions
# --------------------------------------------------

expected_groups = (
    len(
        COVERAGE_FEASIBILITY_SIZES
    )
    *
    COVERAGE_FEASIBILITY_REPLICATES
)


expected_threshold_rows = (
    expected_groups
    *
    len(
        COVERAGE_FEASIBILITY_THRESHOLDS
    )
)


assert len(
    coverage_group_df
) == expected_groups


assert len(
    coverage_threshold_df
) == expected_threshold_rows


assert (
    coverage_group_df[
        "matched_group_id"
    ].nunique()
    == expected_groups
)


assert (
    coverage_group_df[
        "full_set_utility"
    ]
    >= 0.0
).all()


assert (
    coverage_group_df[
        "full_set_utility"
    ]
    <= 1.0
    +
    1e-12
).all()


assert (
    coverage_audit_df[
        "maximum_feature_weight_difference"
    ].max()
    <= 1e-15
)


assert (
    coverage_audit_df[
        "maximum_coverage_matrix_difference"
    ].max()
    <= 1e-15
)


assert (
    coverage_audit_df[
        "maximum_singleton_utility_difference"
    ].max()
    <= 1e-15
)


assert (
    coverage_audit_df[
        "maximum_full_set_utility_difference"
    ].max()
    <= 1e-15
)


assert (
    coverage_audit_df[
        "maximum_sorted_privacy_difference"
    ].max()
    <= 1e-15
)


assert (
    coverage_audit_df[
        "maximum_sorted_cognitive_difference"
    ].max()
    <= 1e-15
)


assert (
    coverage_audit_df[
        "uncovered_feature_count"
    ].max()
    == 0
)


assert (
    coverage_audit_df[
        "empty_item_count"
    ].max()
    == 0
)


# --------------------------------------------------
# Metadata
# --------------------------------------------------

coverage_feasibility_metadata_df = pd.DataFrame(
    [
        {
            "analysis":
                "coverage_threshold_feasibility_calibration",

            "base_seed":
                COVERAGE_FEASIBILITY_BASE_SEED,

            "candidate_sizes":
                repr(
                    COVERAGE_FEASIBILITY_SIZES
                ),

            "replicates_per_size":
                COVERAGE_FEASIBILITY_REPLICATES,

            "total_matched_groups":
                expected_groups,

            "thresholds":
                repr(
                    COVERAGE_FEASIBILITY_THRESHOLDS
                ),

            "n_features":
                COVERAGE_FEASIBILITY_FEATURES,

            "coverage_density":
                COVERAGE_FEASIBILITY_DENSITY,

            "regime_strength":
                COVERAGE_FEASIBILITY_REGIME_STRENGTH,

            "utility_function":
                "weighted maximum feature coverage",

            "feasibility_definition":
                "full_set_utility >= tau",
        }
    ]
)


# --------------------------------------------------
# Save outputs
# --------------------------------------------------

coverage_group_df.to_csv(
    COVERAGE_FEASIBILITY_RAW_PATH,
    index=False,
)


coverage_threshold_df.to_csv(
    COVERAGE_FEASIBILITY_LONG_PATH,
    index=False,
)


coverage_utility_summary_df.to_csv(
    COVERAGE_UTILITY_SUMMARY_PATH,
    index=False,
)


coverage_feasibility_summary_df.to_csv(
    COVERAGE_FEASIBILITY_SUMMARY_PATH,
    index=False,
)


coverage_feasibility_overall_df.to_csv(
    COVERAGE_FEASIBILITY_OVERALL_PATH,
    index=False,
)


coverage_audit_df.to_csv(
    COVERAGE_FEASIBILITY_AUDIT_PATH,
    index=False,
)


coverage_feasibility_metadata_df.to_csv(
    COVERAGE_FEASIBILITY_METADATA_PATH,
    index=False,
)


# --------------------------------------------------
# Printed report
# --------------------------------------------------

print(
    "\n======================================"
)

print(
    "COVERAGE FULL-SET UTILITY BY CANDIDATE SIZE"
)

print(
    "======================================"
)


print(
    coverage_utility_summary_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.6f}",
    )
)


print(
    "\n======================================"
)

print(
    "COVERAGE FEASIBILITY BY SIZE AND THRESHOLD"
)

print(
    "======================================"
)


print(
    coverage_feasibility_summary_df[
        [
            "n_items",
            "tau",
            "matched_groups",
            "feasible_groups",
            "feasibility_percent",
            "mean_full_set_utility",
            "minimum_full_set_utility",
            "mean_utility_shortfall",
            "maximum_utility_shortfall",
        ]
    ].to_string(
        index=False,
        float_format=lambda value:
            f"{value:.6f}",
    )
)


print(
    "\n======================================"
)

print(
    "COVERAGE MATCHED-GENERATOR INTEGRITY AUDIT"
)

print(
    "======================================"
)


audit_maximums = pd.DataFrame(
    [
        {
            "maximum_feature_weight_difference":
                coverage_audit_df[
                    "maximum_feature_weight_difference"
                ].max(),

            "maximum_coverage_matrix_difference":
                coverage_audit_df[
                    "maximum_coverage_matrix_difference"
                ].max(),

            "maximum_singleton_utility_difference":
                coverage_audit_df[
                    "maximum_singleton_utility_difference"
                ].max(),

            "maximum_full_set_utility_difference":
                coverage_audit_df[
                    "maximum_full_set_utility_difference"
                ].max(),

            "maximum_sorted_privacy_difference":
                coverage_audit_df[
                    "maximum_sorted_privacy_difference"
                ].max(),

            "maximum_sorted_cognitive_difference":
                coverage_audit_df[
                    "maximum_sorted_cognitive_difference"
                ].max(),

            "maximum_uncovered_features":
                coverage_audit_df[
                    "uncovered_feature_count"
                ].max(),

            "maximum_empty_items":
                coverage_audit_df[
                    "empty_item_count"
                ].max(),
        }
    ]
)


print(
    audit_maximums.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.16f}",
    )
)


print(
    "\nRaw group results saved to:"
)

print(
    COVERAGE_FEASIBILITY_RAW_PATH
)


print(
    "\nFeasibility summary saved to:"
)

print(
    COVERAGE_FEASIBILITY_SUMMARY_PATH
)


print(
    "\nFull-set utility summary saved to:"
)

print(
    COVERAGE_UTILITY_SUMMARY_PATH
)


print(
    "\nIntegrity audit saved to:"
)

print(
    COVERAGE_FEASIBILITY_AUDIT_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Step 126 — Candidate-pool-normalized coverage
# utility calibration and submodularity audit
# --------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------
# Configuration
# --------------------------------------------------

COVERAGE_POOL_NORMALIZED_VERSION = (
    "coverage_submodular_pool_normalized_v2"
)


COVERAGE_POOL_NORMALIZED_BASE_SEED = (
    globals().get(
        "COVERAGE_FEASIBILITY_BASE_SEED",
        2026 + 125,
    )
)


COVERAGE_POOL_NORMALIZED_SIZES = [
    8,
    10,
    12,
    14,
    16,
]


COVERAGE_POOL_NORMALIZED_THRESHOLDS = [
    0.70,
    0.80,
    0.90,
    0.95,
]


COVERAGE_POOL_NORMALIZED_REPLICATES = 100

COVERAGE_PROPERTY_TESTS_PER_GROUP = 25

COVERAGE_PROPERTY_TOLERANCE = 1e-12


COVERAGE_POOL_NORMALIZED_FEATURES = (
    globals().get(
        "COVERAGE_FEATURES",
        12,
    )
)


COVERAGE_POOL_NORMALIZED_DENSITY = (
    globals().get(
        "COVERAGE_DENSITY",
        0.35,
    )
)


COVERAGE_POOL_NORMALIZED_REGIME_STRENGTH = (
    globals().get(
        "COVERAGE_REGIME_STRENGTH",
        0.75,
    )
)


if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


if "generate_matched_coverage_instances" not in globals():

    raise NameError(
        "generate_matched_coverage_instances is unavailable. "
        "Run Step 124 first."
    )


# --------------------------------------------------
# Output paths
# --------------------------------------------------

POOL_NORMALIZED_RAW_PATH = (
    RESULTS_DIR
    / "coverage_pool_normalized_calibration_raw.csv"
)


POOL_NORMALIZED_UTILITY_SUMMARY_PATH = (
    TABLES_DIR
    / "coverage_pool_normalized_utility_by_candidate_size.csv"
)


POOL_NORMALIZED_FEASIBILITY_PATH = (
    TABLES_DIR
    / "coverage_pool_normalized_feasibility_by_size_threshold.csv"
)


POOL_NORMALIZED_PROPERTY_AUDIT_PATH = (
    RESULTS_DIR
    / "coverage_pool_normalized_submodularity_audit.csv"
)


POOL_NORMALIZED_PROPERTY_SUMMARY_PATH = (
    TABLES_DIR
    / "coverage_pool_normalized_property_audit_summary.csv"
)


POOL_NORMALIZED_METADATA_PATH = (
    RESULTS_DIR
    / "coverage_pool_normalized_metadata.csv"
)


# --------------------------------------------------
# Candidate-pool-normalized coverage utility
# --------------------------------------------------

def pool_normalized_coverage_utility(
    selected_items,
    all_items,
    feature_weights,
):
    """
    Candidate-pool-normalized weighted maximum coverage:

        U_D(S) =
            sum_f w_f max_{z in S} a_zf
            --------------------------------
            sum_f w_f max_{z in D} a_zf

    where D is the complete candidate information pool.

    Properties:
    - U_D(empty) = 0
    - U_D(D) = 1
    - monotone
    - submodular
    """

    feature_weights = np.asarray(
        feature_weights,
        dtype=float,
    )


    if feature_weights.ndim != 1:

        raise ValueError(
            "feature_weights must be one-dimensional."
        )


    if np.any(
        feature_weights < 0.0
    ):

        raise ValueError(
            "feature_weights must be nonnegative."
        )


    if len(
        all_items
    ) == 0:

        raise ValueError(
            "all_items must contain at least one item."
        )


    all_coverage_matrix = np.asarray(
        [
            item.feature_coverage
            for item in all_items
        ],
        dtype=float,
    )


    if (
        all_coverage_matrix.ndim
        != 2
    ):

        raise ValueError(
            "The candidate-pool coverage matrix must "
            "be two-dimensional."
        )


    if (
        all_coverage_matrix.shape[1]
        != len(
            feature_weights
        )
    ):

        raise ValueError(
            "Coverage dimensions do not match "
            "feature_weights."
        )


    available_feature_coverage = (
        all_coverage_matrix.max(
            axis=0
        )
    )


    available_weighted_coverage = float(
        np.dot(
            feature_weights,
            available_feature_coverage,
        )
    )


    if available_weighted_coverage <= 0.0:

        raise ValueError(
            "The candidate pool provides no positive "
            "weighted feature coverage."
        )


    if len(
        selected_items
    ) == 0:

        return 0.0


    selected_coverage_matrix = np.asarray(
        [
            item.feature_coverage
            for item in selected_items
        ],
        dtype=float,
    )


    selected_feature_coverage = (
        selected_coverage_matrix.max(
            axis=0
        )
    )


    selected_weighted_coverage = float(
        np.dot(
            feature_weights,
            selected_feature_coverage,
        )
    )


    utility = (
        selected_weighted_coverage
        /
        available_weighted_coverage
    )


    # Remove negligible floating-point excursions.
    if (
        utility < 0.0
        and
        utility >= -1e-12
    ):

        utility = 0.0


    if (
        utility > 1.0
        and
        utility <= 1.0 + 1e-12
    ):

        utility = 1.0


    return float(
        utility
    )


# --------------------------------------------------
# Vectorized singleton utility calculation
# --------------------------------------------------

def pool_normalized_singleton_utilities(
    all_items,
    feature_weights,
):
    """
    Return U_D({z}) for every candidate item.
    """

    feature_weights = np.asarray(
        feature_weights,
        dtype=float,
    )


    coverage_matrix = np.asarray(
        [
            item.feature_coverage
            for item in all_items
        ],
        dtype=float,
    )


    available_feature_coverage = (
        coverage_matrix.max(
            axis=0
        )
    )


    denominator = float(
        np.dot(
            feature_weights,
            available_feature_coverage,
        )
    )


    if denominator <= 0.0:

        raise ValueError(
            "The candidate pool has zero attainable "
            "weighted coverage."
        )


    singleton_numerators = (
        coverage_matrix
        @
        feature_weights
    )


    return (
        singleton_numerators
        /
        denominator
    )


# --------------------------------------------------
# Raw ideal-denominator utility for comparison
# --------------------------------------------------

def ideal_normalized_coverage_utility(
    selected_items,
    feature_weights,
):
    """
    Original coverage utility normalized against
    ideal unit coverage of every feature.
    """

    feature_weights = np.asarray(
        feature_weights,
        dtype=float,
    )


    if len(
        selected_items
    ) == 0:

        return 0.0


    selected_matrix = np.asarray(
        [
            item.feature_coverage
            for item in selected_items
        ],
        dtype=float,
    )


    selected_maximum = (
        selected_matrix.max(
            axis=0
        )
    )


    return float(
        np.dot(
            feature_weights,
            selected_maximum,
        )
        /
        feature_weights.sum()
    )


# --------------------------------------------------
# Calibration and property-audit containers
# --------------------------------------------------

calibration_rows = []
feasibility_rows = []
property_rows = []


# --------------------------------------------------
# Generate matched benchmark groups
# --------------------------------------------------

for n_items in COVERAGE_POOL_NORMALIZED_SIZES:

    for replicate in range(
        COVERAGE_POOL_NORMALIZED_REPLICATES
    ):

        matched_seed = (
            72_000_000
            +
            100_000
            *
            n_items
            +
            COVERAGE_POOL_NORMALIZED_BASE_SEED
            +
            replicate
        )


        matched_instances = (
            generate_matched_coverage_instances(
                n_items=n_items,

                n_features=
                    COVERAGE_POOL_NORMALIZED_FEATURES,

                seed=
                    matched_seed,

                coverage_density=
                    COVERAGE_POOL_NORMALIZED_DENSITY,

                regime_strength=
                    COVERAGE_POOL_NORMALIZED_REGIME_STRENGTH,
            )
        )


        # Utility structure is identical across the
        # three matched burden-association regimes.
        reference_data = (
            matched_instances[
                "independent"
            ]
        )


        all_items = reference_data[
            "items"
        ]


        feature_weights = np.asarray(
            reference_data[
                "feature_weights"
            ],
            dtype=float,
        )


        raw_full_set_utility = (
            ideal_normalized_coverage_utility(
                selected_items=
                    all_items,

                feature_weights=
                    feature_weights,
            )
        )


        normalized_full_set_utility = (
            pool_normalized_coverage_utility(
                selected_items=
                    all_items,

                all_items=
                    all_items,

                feature_weights=
                    feature_weights,
            )
        )


        singleton_utilities = (
            pool_normalized_singleton_utilities(
                all_items=
                    all_items,

                feature_weights=
                    feature_weights,
            )
        )


        calibration_rows.append(
            {
                "generator_version":
                    COVERAGE_POOL_NORMALIZED_VERSION,

                "matched_group_id":
                    (
                        f"COVN_M{n_items}_"
                        f"R{replicate:03d}"
                    ),

                "n_items":
                    n_items,

                "replicate":
                    replicate,

                "matched_seed":
                    matched_seed,

                "n_features":
                    COVERAGE_POOL_NORMALIZED_FEATURES,

                "coverage_density":
                    COVERAGE_POOL_NORMALIZED_DENSITY,

                "raw_full_set_utility":
                    raw_full_set_utility,

                "pool_normalized_full_set_utility":
                    normalized_full_set_utility,

                "mean_pool_normalized_singleton_utility":
                    float(
                        singleton_utilities.mean()
                    ),

                "minimum_pool_normalized_singleton_utility":
                    float(
                        singleton_utilities.min()
                    ),

                "maximum_pool_normalized_singleton_utility":
                    float(
                        singleton_utilities.max()
                    ),
            }
        )


        for tau in (
            COVERAGE_POOL_NORMALIZED_THRESHOLDS
        ):

            feasibility_rows.append(
                {
                    "matched_group_id":
                        (
                            f"COVN_M{n_items}_"
                            f"R{replicate:03d}"
                        ),

                    "n_items":
                        n_items,

                    "replicate":
                        replicate,

                    "matched_seed":
                        matched_seed,

                    "tau":
                        tau,

                    "pool_normalized_full_set_utility":
                        normalized_full_set_utility,

                    "is_feasible":
                        int(
                            normalized_full_set_utility
                            +
                            COVERAGE_PROPERTY_TOLERANCE
                            >= tau
                        ),
                }
            )


        # ------------------------------------------
        # Random monotonicity and diminishing-returns
        # checks
        # ------------------------------------------

        property_rng = np.random.default_rng(
            matched_seed
            +
            50_000_000
        )


        for property_test in range(
            COVERAGE_PROPERTY_TESTS_PER_GROUP
        ):

            permutation = (
                property_rng.permutation(
                    n_items
                )
            )


            # A has at most n-2 elements so B can
            # retain at least one candidate x outside.
            a_size = int(
                property_rng.integers(
                    low=0,
                    high=n_items - 1,
                )
            )


            b_size = int(
                property_rng.integers(
                    low=a_size,
                    high=n_items,
                )
            )


            a_indices = permutation[
                :a_size
            ]


            b_indices = permutation[
                :b_size
            ]


            x_index = int(
                permutation[
                    b_size
                ]
            )


            a_items = [
                all_items[
                    item_index
                ]
                for item_index in a_indices
            ]


            b_items = [
                all_items[
                    item_index
                ]
                for item_index in b_indices
            ]


            x_item = all_items[
                x_index
            ]


            utility_a = (
                pool_normalized_coverage_utility(
                    selected_items=
                        a_items,

                    all_items=
                        all_items,

                    feature_weights=
                        feature_weights,
                )
            )


            utility_b = (
                pool_normalized_coverage_utility(
                    selected_items=
                        b_items,

                    all_items=
                        all_items,

                    feature_weights=
                        feature_weights,
                )
            )


            utility_a_plus_x = (
                pool_normalized_coverage_utility(
                    selected_items=
                        a_items
                        +
                        [
                            x_item
                        ],

                    all_items=
                        all_items,

                    feature_weights=
                        feature_weights,
                )
            )


            utility_b_plus_x = (
                pool_normalized_coverage_utility(
                    selected_items=
                        b_items
                        +
                        [
                            x_item
                        ],

                    all_items=
                        all_items,

                    feature_weights=
                        feature_weights,
                )
            )


            marginal_gain_a = (
                utility_a_plus_x
                -
                utility_a
            )


            marginal_gain_b = (
                utility_b_plus_x
                -
                utility_b
            )


            monotonicity_violation = max(
                0.0,
                utility_a
                -
                utility_b,
            )


            diminishing_returns_violation = max(
                0.0,
                marginal_gain_b
                -
                marginal_gain_a,
            )


            range_violation = max(
                0.0,
                -utility_a,
                utility_a
                -
                1.0,
                -utility_b,
                utility_b
                -
                1.0,
                -utility_a_plus_x,
                utility_a_plus_x
                -
                1.0,
                -utility_b_plus_x,
                utility_b_plus_x
                -
                1.0,
            )


            property_rows.append(
                {
                    "matched_group_id":
                        (
                            f"COVN_M{n_items}_"
                            f"R{replicate:03d}"
                        ),

                    "n_items":
                        n_items,

                    "replicate":
                        replicate,

                    "matched_seed":
                        matched_seed,

                    "property_test":
                        property_test,

                    "a_size":
                        a_size,

                    "b_size":
                        b_size,

                    "utility_a":
                        utility_a,

                    "utility_b":
                        utility_b,

                    "marginal_gain_a":
                        marginal_gain_a,

                    "marginal_gain_b":
                        marginal_gain_b,

                    "monotonicity_violation":
                        monotonicity_violation,

                    "diminishing_returns_violation":
                        diminishing_returns_violation,

                    "range_violation":
                        range_violation,
                }
            )


    print(
        "Completed pool-normalized coverage size:",
        n_items
    )


# --------------------------------------------------
# Construct data frames
# --------------------------------------------------

calibration_df = pd.DataFrame(
    calibration_rows
)


feasibility_df = pd.DataFrame(
    feasibility_rows
)


property_audit_df = pd.DataFrame(
    property_rows
)


# --------------------------------------------------
# Candidate-size utility summary
# --------------------------------------------------

utility_summary_df = (
    calibration_df
    .groupby(
        "n_items",
        as_index=False,
    )
    .agg(
        matched_groups=(
            "matched_group_id",
            "nunique",
        ),

        mean_raw_full_set_utility=(
            "raw_full_set_utility",
            "mean",
        ),

        minimum_raw_full_set_utility=(
            "raw_full_set_utility",
            "min",
        ),

        mean_pool_normalized_full_set_utility=(
            "pool_normalized_full_set_utility",
            "mean",
        ),

        minimum_pool_normalized_full_set_utility=(
            "pool_normalized_full_set_utility",
            "min",
        ),

        maximum_pool_normalized_full_set_utility=(
            "pool_normalized_full_set_utility",
            "max",
        ),

        mean_singleton_utility=(
            "mean_pool_normalized_singleton_utility",
            "mean",
        ),

        minimum_singleton_utility=(
            "minimum_pool_normalized_singleton_utility",
            "min",
        ),

        maximum_singleton_utility=(
            "maximum_pool_normalized_singleton_utility",
            "max",
        ),
    )
)


# --------------------------------------------------
# Feasibility summary
# --------------------------------------------------

feasibility_summary_df = (
    feasibility_df
    .groupby(
        [
            "n_items",
            "tau",
        ],
        as_index=False,
    )
    .agg(
        matched_groups=(
            "matched_group_id",
            "nunique",
        ),

        feasible_groups=(
            "is_feasible",
            "sum",
        ),

        minimum_full_set_utility=(
            "pool_normalized_full_set_utility",
            "min",
        ),

        maximum_full_set_utility=(
            "pool_normalized_full_set_utility",
            "max",
        ),
    )
)


feasibility_summary_df[
    "feasibility_percent"
] = (
    100.0
    *
    feasibility_summary_df[
        "feasible_groups"
    ]
    /
    feasibility_summary_df[
        "matched_groups"
    ]
)


# --------------------------------------------------
# Property-audit summary
# --------------------------------------------------

property_summary_df = pd.DataFrame(
    [
        {
            "matched_groups":
                property_audit_df[
                    "matched_group_id"
                ].nunique(),

            "random_property_tests":
                len(
                    property_audit_df
                ),

            "maximum_monotonicity_violation":
                property_audit_df[
                    "monotonicity_violation"
                ].max(),

            "maximum_diminishing_returns_violation":
                property_audit_df[
                    "diminishing_returns_violation"
                ].max(),

            "maximum_range_violation":
                property_audit_df[
                    "range_violation"
                ].max(),

            "maximum_full_set_deviation_from_one":
                (
                    calibration_df[
                        "pool_normalized_full_set_utility"
                    ]
                    -
                    1.0
                )
                .abs()
                .max(),

            "monotonicity_failures_above_tolerance":
                int(
                    (
                        property_audit_df[
                            "monotonicity_violation"
                        ]
                        >
                        COVERAGE_PROPERTY_TOLERANCE
                    ).sum()
                ),

            "diminishing_returns_failures_above_tolerance":
                int(
                    (
                        property_audit_df[
                            "diminishing_returns_violation"
                        ]
                        >
                        COVERAGE_PROPERTY_TOLERANCE
                    ).sum()
                ),

            "range_failures_above_tolerance":
                int(
                    (
                        property_audit_df[
                            "range_violation"
                        ]
                        >
                        COVERAGE_PROPERTY_TOLERANCE
                    ).sum()
                ),
        }
    ]
)


# --------------------------------------------------
# Integrity assertions
# --------------------------------------------------

expected_groups = (
    len(
        COVERAGE_POOL_NORMALIZED_SIZES
    )
    *
    COVERAGE_POOL_NORMALIZED_REPLICATES
)


expected_property_tests = (
    expected_groups
    *
    COVERAGE_PROPERTY_TESTS_PER_GROUP
)


assert len(
    calibration_df
) == expected_groups


assert len(
    property_audit_df
) == expected_property_tests


assert (
    calibration_df[
        "matched_group_id"
    ].nunique()
    == expected_groups
)


assert (
    calibration_df[
        "pool_normalized_full_set_utility"
    ]
    .sub(1.0)
    .abs()
    .max()
    <=
    COVERAGE_PROPERTY_TOLERANCE
)


assert (
    feasibility_summary_df[
        "feasibility_percent"
    ]
    ==
    100.0
).all()


assert (
    property_audit_df[
        "monotonicity_violation"
    ].max()
    <=
    COVERAGE_PROPERTY_TOLERANCE
)


assert (
    property_audit_df[
        "diminishing_returns_violation"
    ].max()
    <=
    COVERAGE_PROPERTY_TOLERANCE
)


assert (
    property_audit_df[
        "range_violation"
    ].max()
    <=
    COVERAGE_PROPERTY_TOLERANCE
)


# --------------------------------------------------
# Metadata
# --------------------------------------------------

metadata_df = pd.DataFrame(
    [
        {
            "utility_version":
                COVERAGE_POOL_NORMALIZED_VERSION,

            "candidate_sizes":
                repr(
                    COVERAGE_POOL_NORMALIZED_SIZES
                ),

            "replicates_per_size":
                COVERAGE_POOL_NORMALIZED_REPLICATES,

            "total_matched_groups":
                expected_groups,

            "thresholds":
                repr(
                    COVERAGE_POOL_NORMALIZED_THRESHOLDS
                ),

            "n_features":
                COVERAGE_POOL_NORMALIZED_FEATURES,

            "coverage_density":
                COVERAGE_POOL_NORMALIZED_DENSITY,

            "regime_strength":
                COVERAGE_POOL_NORMALIZED_REGIME_STRENGTH,

            "normalization_denominator":
                (
                    "weighted maximum feature coverage "
                    "available in the candidate pool"
                ),

            "threshold_interpretation":
                (
                    "fraction of available modeled "
                    "decision-support utility"
                ),

            "full_set_utility":
                1.0,

            "submodularity_preserved":
                True,

            "monotonicity_preserved":
                True,

            "property_tests_per_group":
                COVERAGE_PROPERTY_TESTS_PER_GROUP,

            "property_tolerance":
                COVERAGE_PROPERTY_TOLERANCE,
        }
    ]
)


# --------------------------------------------------
# Save outputs
# --------------------------------------------------

calibration_df.to_csv(
    POOL_NORMALIZED_RAW_PATH,
    index=False,
)


utility_summary_df.to_csv(
    POOL_NORMALIZED_UTILITY_SUMMARY_PATH,
    index=False,
)


feasibility_summary_df.to_csv(
    POOL_NORMALIZED_FEASIBILITY_PATH,
    index=False,
)


property_audit_df.to_csv(
    POOL_NORMALIZED_PROPERTY_AUDIT_PATH,
    index=False,
)


property_summary_df.to_csv(
    POOL_NORMALIZED_PROPERTY_SUMMARY_PATH,
    index=False,
)


metadata_df.to_csv(
    POOL_NORMALIZED_METADATA_PATH,
    index=False,
)


# --------------------------------------------------
# Printed report
# --------------------------------------------------

print(
    "\n======================================"
)

print(
    "POOL-NORMALIZED COVERAGE UTILITY BY CANDIDATE SIZE"
)

print(
    "======================================"
)


print(
    utility_summary_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.6f}",
    )
)


print(
    "\n======================================"
)

print(
    "POOL-NORMALIZED COVERAGE FEASIBILITY"
)

print(
    "======================================"
)


print(
    feasibility_summary_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.6f}",
    )
)


print(
    "\n======================================"
)

print(
    "POOL-NORMALIZED COVERAGE PROPERTY AUDIT"
)

print(
    "======================================"
)


print(
    property_summary_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.16f}",
    )
)


print(
    "\nCalibration rows saved to:"
)

print(
    POOL_NORMALIZED_RAW_PATH
)


print(
    "\nUtility summary saved to:"
)

print(
    POOL_NORMALIZED_UTILITY_SUMMARY_PATH
)


print(
    "\nFeasibility summary saved to:"
)

print(
    POOL_NORMALIZED_FEASIBILITY_PATH
)


print(
    "\nProperty audit saved to:"
)

print(
    POOL_NORMALIZED_PROPERTY_AUDIT_PATH
)


print(
    "\nMetadata saved to:"
)

print(
    POOL_NORMALIZED_METADATA_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Step 127 — Pool-normalized coverage Exact-versus-
# Greedy matched-group smoke test
# --------------------------------------------------

from pathlib import Path
import itertools
import time

import numpy as np
import pandas as pd


# --------------------------------------------------
# Configuration
# --------------------------------------------------

COVERAGE_SMOKE_N_ITEMS = 12

COVERAGE_SMOKE_SEED = (
    globals().get(
        "GLOBAL_SEED",
        2026,
    )
    + 127
)


COVERAGE_SMOKE_REGIMES = [
    "independent",
    "utility_burden_alignment",
    "utility_burden_conflict",
]


COVERAGE_SMOKE_THRESHOLDS = [
    0.70,
    0.80,
    0.90,
    0.95,
]


COVERAGE_SMOKE_WEIGHT_PROFILES = {
    "balanced": {
        "lambda_privacy": 1 / 3,
        "lambda_cognitive": 1 / 3,
        "lambda_cardinality": 1 / 3,
    },

    "privacy_priority": {
        "lambda_privacy": 0.60,
        "lambda_cognitive": 0.20,
        "lambda_cardinality": 0.20,
    },

    "interpretation_priority": {
        "lambda_privacy": 0.20,
        "lambda_cognitive": 0.60,
        "lambda_cardinality": 0.20,
    },

    "parsimony_priority": {
        "lambda_privacy": 0.20,
        "lambda_cognitive": 0.20,
        "lambda_cardinality": 0.60,
    },
}


COVERAGE_SMOKE_FEATURES = globals().get(
    "COVERAGE_FEATURES",
    12,
)


COVERAGE_SMOKE_DENSITY = globals().get(
    "COVERAGE_DENSITY",
    0.35,
)


COVERAGE_SMOKE_REGIME_STRENGTH = globals().get(
    "COVERAGE_REGIME_STRENGTH",
    0.75,
)


COVERAGE_NUMERICAL_TOLERANCE = 1e-12


if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


if "generate_matched_coverage_instances" not in globals():

    raise NameError(
        "generate_matched_coverage_instances is unavailable. "
        "Run Step 124 first."
    )


if "pool_normalized_coverage_utility" not in globals():

    raise NameError(
        "pool_normalized_coverage_utility is unavailable. "
        "Run Step 126 first."
    )


COVERAGE_SMOKE_RESULTS_PATH = (
    RESULTS_DIR
    / "coverage_exact_vs_greedy_smoke_test.csv"
)


COVERAGE_SMOKE_AUDIT_PATH = (
    RESULTS_DIR
    / "coverage_exact_vs_greedy_smoke_test_audit.csv"
)


# --------------------------------------------------
# Subset evaluation
# --------------------------------------------------

def evaluate_coverage_subset(
    all_items,
    selected_indices,
    feature_weights,
    weights,
):
    """
    Evaluate one coverage-based information subset.
    """

    selected_indices = tuple(
        sorted(
            int(index)
            for index in selected_indices
        )
    )


    selected_items = [
        all_items[index]
        for index in selected_indices
    ]


    utility = pool_normalized_coverage_utility(
        selected_items=selected_items,
        all_items=all_items,
        feature_weights=feature_weights,
    )


    if selected_items:

        privacy = (
            1.0
            -
            np.prod(
                [
                    1.0
                    -
                    float(item.privacy)
                    for item in selected_items
                ]
            )
        )


        cognitive_cost = sum(
            float(item.cognitive_cost)
            for item in selected_items
        )


    else:

        privacy = 0.0
        cognitive_cost = 0.0


    full_cognitive_cost = sum(
        float(item.cognitive_cost)
        for item in all_items
    )


    if full_cognitive_cost > 0.0:

        normalized_cognitive_cost = (
            cognitive_cost
            /
            full_cognitive_cost
        )

    else:

        normalized_cognitive_cost = 0.0


    cardinality_fraction = (
        len(selected_indices)
        /
        len(all_items)
    )


    disclosure_cost = (
        weights["lambda_privacy"]
        *
        privacy
        +
        weights["lambda_cognitive"]
        *
        normalized_cognitive_cost
        +
        weights["lambda_cardinality"]
        *
        cardinality_fraction
    )


    selected_ids = tuple(
        str(
            all_items[index].item_id
        )
        for index in selected_indices
    )


    return {
        "selected_indices":
            selected_indices,

        "selected_ids":
            selected_ids,

        "selected_count":
            len(selected_indices),

        "utility":
            float(utility),

        "privacy":
            float(privacy),

        "normalized_cognitive_cost":
            float(
                normalized_cognitive_cost
            ),

        "cardinality_fraction":
            float(
                cardinality_fraction
            ),

        "disclosure_cost":
            float(disclosure_cost),
    }


# --------------------------------------------------
# Exact exhaustive solver
# --------------------------------------------------

def exact_coverage_mst(
    all_items,
    feature_weights,
    tau,
    weights,
):
    """
    Exhaustively select the minimum-disclosure-cost
    subset satisfying pool-normalized coverage utility.
    """

    start_time = time.perf_counter()

    best_result = None
    evaluated_subsets = 0


    for subset_size in range(
        len(all_items) + 1
    ):

        for selected_indices in itertools.combinations(
            range(
                len(all_items)
            ),
            subset_size,
        ):

            evaluated_subsets += 1


            result = evaluate_coverage_subset(
                all_items=all_items,
                selected_indices=
                    selected_indices,
                feature_weights=
                    feature_weights,
                weights=weights,
            )


            if (
                result["utility"]
                +
                COVERAGE_NUMERICAL_TOLERANCE
                <
                tau
            ):

                continue


            if best_result is None:

                best_result = result
                continue


            candidate_key = (
                result["disclosure_cost"],
                result["selected_count"],
                result["selected_ids"],
            )


            incumbent_key = (
                best_result[
                    "disclosure_cost"
                ],
                best_result[
                    "selected_count"
                ],
                best_result[
                    "selected_ids"
                ],
            )


            if candidate_key < incumbent_key:

                best_result = result


    runtime_seconds = (
        time.perf_counter()
        -
        start_time
    )


    if best_result is None:

        return {
            "status":
                "INFEASIBLE",

            "evaluated_subsets":
                evaluated_subsets,

            "runtime_seconds":
                runtime_seconds,
        }


    return {
        "status":
            "OPTIMAL",

        **best_result,

        "evaluated_subsets":
            evaluated_subsets,

        "runtime_seconds":
            runtime_seconds,
    }


# --------------------------------------------------
# Dynamic marginal Greedy solver
# --------------------------------------------------

def greedy_coverage_mst(
    all_items,
    feature_weights,
    tau,
    weights,
):
    """
    Dynamic Greedy selection using current marginal
    utility divided by current marginal disclosure cost.
    """

    start_time = time.perf_counter()

    selected_indices = []
    remaining_indices = set(
        range(
            len(all_items)
        )
    )


    current_result = evaluate_coverage_subset(
        all_items=all_items,
        selected_indices=
            selected_indices,
        feature_weights=
            feature_weights,
        weights=weights,
    )


    while (
        current_result["utility"]
        +
        COVERAGE_NUMERICAL_TOLERANCE
        <
        tau
        and
        remaining_indices
    ):

        candidate_rows = []


        for candidate_index in sorted(
            remaining_indices
        ):

            candidate_indices = (
                selected_indices
                +
                [
                    candidate_index
                ]
            )


            candidate_result = (
                evaluate_coverage_subset(
                    all_items=all_items,
                    selected_indices=
                        candidate_indices,
                    feature_weights=
                        feature_weights,
                    weights=weights,
                )
            )


            marginal_utility = (
                candidate_result["utility"]
                -
                current_result["utility"]
            )


            marginal_cost = (
                candidate_result[
                    "disclosure_cost"
                ]
                -
                current_result[
                    "disclosure_cost"
                ]
            )


            if marginal_cost > 0.0:

                score = (
                    marginal_utility
                    /
                    marginal_cost
                )


            elif marginal_utility > 0.0:

                score = np.inf


            else:

                score = -np.inf


            candidate_rows.append(
                {
                    "candidate_index":
                        candidate_index,

                    "candidate_id":
                        str(
                            all_items[
                                candidate_index
                            ].item_id
                        ),

                    "score":
                        float(score),

                    "marginal_utility":
                        float(
                            marginal_utility
                        ),

                    "marginal_cost":
                        float(
                            marginal_cost
                        ),

                    "result":
                        candidate_result,
                }
            )


        best_candidate = min(
            candidate_rows,
            key=lambda row: (
                -row["score"],
                -row["marginal_utility"],
                row["marginal_cost"],
                row["candidate_id"],
            ),
        )


        if (
            best_candidate[
                "marginal_utility"
            ]
            <=
            COVERAGE_NUMERICAL_TOLERANCE
        ):

            break


        selected_indices.append(
            best_candidate[
                "candidate_index"
            ]
        )


        selected_indices.sort()


        remaining_indices.remove(
            best_candidate[
                "candidate_index"
            ]
        )


        current_result = (
            best_candidate[
                "result"
            ]
        )


    runtime_seconds = (
        time.perf_counter()
        -
        start_time
    )


    if (
        current_result["utility"]
        +
        COVERAGE_NUMERICAL_TOLERANCE
        >=
        tau
    ):

        status = "FEASIBLE"

    else:

        status = "INFEASIBLE"


    return {
        "status":
            status,

        **current_result,

        "runtime_seconds":
            runtime_seconds,
    }


# --------------------------------------------------
# Generate one matched coverage group
# --------------------------------------------------

matched_instances = (
    generate_matched_coverage_instances(
        n_items=
            COVERAGE_SMOKE_N_ITEMS,

        n_features=
            COVERAGE_SMOKE_FEATURES,

        seed=
            COVERAGE_SMOKE_SEED,

        coverage_density=
            COVERAGE_SMOKE_DENSITY,

        regime_strength=
            COVERAGE_SMOKE_REGIME_STRENGTH,
    )
)


# --------------------------------------------------
# Matched-generator integrity checks
# --------------------------------------------------

reference_data = matched_instances[
    "independent"
]


reference_feature_weights = np.asarray(
    reference_data[
        "feature_weights"
    ],
    dtype=float,
)


reference_coverage_matrix = np.asarray(
    reference_data[
        "coverage_matrix"
    ],
    dtype=float,
)


reference_privacy = np.sort(
    np.asarray(
        [
            item.privacy
            for item in reference_data[
                "items"
            ]
        ],
        dtype=float,
    )
)


reference_cognitive = np.sort(
    np.asarray(
        [
            item.cognitive_cost
            for item in reference_data[
                "items"
            ]
        ],
        dtype=float,
    )
)


maximum_feature_weight_difference = 0.0
maximum_coverage_matrix_difference = 0.0
maximum_sorted_privacy_difference = 0.0
maximum_sorted_cognitive_difference = 0.0


for regime in COVERAGE_SMOKE_REGIMES:

    regime_data = matched_instances[
        regime
    ]


    maximum_feature_weight_difference = max(
        maximum_feature_weight_difference,

        float(
            np.max(
                np.abs(
                    np.asarray(
                        regime_data[
                            "feature_weights"
                        ],
                        dtype=float,
                    )
                    -
                    reference_feature_weights
                )
            )
        ),
    )


    maximum_coverage_matrix_difference = max(
        maximum_coverage_matrix_difference,

        float(
            np.max(
                np.abs(
                    np.asarray(
                        regime_data[
                            "coverage_matrix"
                        ],
                        dtype=float,
                    )
                    -
                    reference_coverage_matrix
                )
            )
        ),
    )


    regime_privacy = np.sort(
        np.asarray(
            [
                item.privacy
                for item in regime_data[
                    "items"
                ]
            ],
            dtype=float,
        )
    )


    regime_cognitive = np.sort(
        np.asarray(
            [
                item.cognitive_cost
                for item in regime_data[
                    "items"
                ]
            ],
            dtype=float,
        )
    )


    maximum_sorted_privacy_difference = max(
        maximum_sorted_privacy_difference,

        float(
            np.max(
                np.abs(
                    regime_privacy
                    -
                    reference_privacy
                )
            )
        ),
    )


    maximum_sorted_cognitive_difference = max(
        maximum_sorted_cognitive_difference,

        float(
            np.max(
                np.abs(
                    regime_cognitive
                    -
                    reference_cognitive
                )
            )
        ),
    )


assert maximum_feature_weight_difference <= 1e-15
assert maximum_coverage_matrix_difference <= 1e-15
assert maximum_sorted_privacy_difference <= 1e-15
assert maximum_sorted_cognitive_difference <= 1e-15


# --------------------------------------------------
# Run all 48 matched conditions
# --------------------------------------------------

result_rows = []


for regime in COVERAGE_SMOKE_REGIMES:

    regime_data = matched_instances[
        regime
    ]


    all_items = regime_data[
        "items"
    ]


    feature_weights = np.asarray(
        regime_data[
            "feature_weights"
        ],
        dtype=float,
    )


    full_set_utility = (
        pool_normalized_coverage_utility(
            selected_items=
                all_items,

            all_items=
                all_items,

            feature_weights=
                feature_weights,
        )
    )


    assert abs(
        full_set_utility
        -
        1.0
    ) <= COVERAGE_NUMERICAL_TOLERANCE


    for tau in COVERAGE_SMOKE_THRESHOLDS:

        for weight_profile, weights in (
            COVERAGE_SMOKE_WEIGHT_PROFILES.items()
        ):

            exact_result = exact_coverage_mst(
                all_items=all_items,
                feature_weights=
                    feature_weights,
                tau=tau,
                weights=weights,
            )


            greedy_result = greedy_coverage_mst(
                all_items=all_items,
                feature_weights=
                    feature_weights,
                tau=tau,
                weights=weights,
            )


            if (
                exact_result["status"]
                ==
                "OPTIMAL"
                and
                greedy_result["status"]
                ==
                "FEASIBLE"
            ):

                optimality_gap = (
                    greedy_result[
                        "disclosure_cost"
                    ]
                    -
                    exact_result[
                        "disclosure_cost"
                    ]
                )


                optimality_gap_percent = (
                    100.0
                    *
                    optimality_gap
                    /
                    exact_result[
                        "disclosure_cost"
                    ]
                )


                exact_set = set(
                    exact_result[
                        "selected_ids"
                    ]
                )


                greedy_set = set(
                    greedy_result[
                        "selected_ids"
                    ]
                )


                union_set = (
                    exact_set
                    |
                    greedy_set
                )


                if union_set:

                    jaccard_similarity = (
                        len(
                            exact_set
                            &
                            greedy_set
                        )
                        /
                        len(
                            union_set
                        )
                    )

                else:

                    jaccard_similarity = 1.0


                same_selected_subset = (
                    exact_set
                    ==
                    greedy_set
                )


            else:

                optimality_gap = np.nan
                optimality_gap_percent = np.nan
                jaccard_similarity = np.nan
                same_selected_subset = np.nan


            result_rows.append(
                {
                    "benchmark_version":
                        (
                            "coverage_pool_normalized_"
                            "smoke_test_v1"
                        ),

                    "matched_seed":
                        COVERAGE_SMOKE_SEED,

                    "n_items":
                        COVERAGE_SMOKE_N_ITEMS,

                    "regime":
                        regime,

                    "tau":
                        tau,

                    "weight_profile":
                        weight_profile,

                    "lambda_privacy":
                        weights[
                            "lambda_privacy"
                        ],

                    "lambda_cognitive":
                        weights[
                            "lambda_cognitive"
                        ],

                    "lambda_cardinality":
                        weights[
                            "lambda_cardinality"
                        ],

                    "full_set_utility":
                        full_set_utility,

                    "exact_status":
                        exact_result[
                            "status"
                        ],

                    "exact_selected_count":
                        exact_result.get(
                            "selected_count",
                            np.nan,
                        ),

                    "exact_selected_items":
                        "|".join(
                            exact_result.get(
                                "selected_ids",
                                (),
                            )
                        ),

                    "exact_utility":
                        exact_result.get(
                            "utility",
                            np.nan,
                        ),

                    "exact_privacy":
                        exact_result.get(
                            "privacy",
                            np.nan,
                        ),

                    "exact_normalized_cognitive_cost":
                        exact_result.get(
                            "normalized_cognitive_cost",
                            np.nan,
                        ),

                    "exact_disclosure_cost":
                        exact_result.get(
                            "disclosure_cost",
                            np.nan,
                        ),

                    "exact_runtime_seconds":
                        exact_result[
                            "runtime_seconds"
                        ],

                    "evaluated_subsets":
                        exact_result[
                            "evaluated_subsets"
                        ],

                    "greedy_status":
                        greedy_result[
                            "status"
                        ],

                    "greedy_selected_count":
                        greedy_result.get(
                            "selected_count",
                            np.nan,
                        ),

                    "greedy_selected_items":
                        "|".join(
                            greedy_result.get(
                                "selected_ids",
                                (),
                            )
                        ),

                    "greedy_utility":
                        greedy_result.get(
                            "utility",
                            np.nan,
                        ),

                    "greedy_privacy":
                        greedy_result.get(
                            "privacy",
                            np.nan,
                        ),

                    "greedy_normalized_cognitive_cost":
                        greedy_result.get(
                            "normalized_cognitive_cost",
                            np.nan,
                        ),

                    "greedy_disclosure_cost":
                        greedy_result.get(
                            "disclosure_cost",
                            np.nan,
                        ),

                    "greedy_runtime_seconds":
                        greedy_result[
                            "runtime_seconds"
                        ],

                    "optimality_gap":
                        optimality_gap,

                    "optimality_gap_percent":
                        optimality_gap_percent,

                    "same_selected_subset":
                        same_selected_subset,

                    "jaccard_similarity":
                        jaccard_similarity,
                }
            )


            print(
                "Completed coverage smoke condition:",
                regime,
                tau,
                weight_profile
            )


# --------------------------------------------------
# Results and integrity audit
# --------------------------------------------------

coverage_smoke_df = pd.DataFrame(
    result_rows
)


negative_gap_rows = int(
    (
        coverage_smoke_df[
            "optimality_gap_percent"
        ]
        <
        -1e-8
    ).sum()
)


exact_threshold_failures = int(
    (
        (
            coverage_smoke_df[
                "exact_status"
            ]
            ==
            "OPTIMAL"
        )
        &
        (
            coverage_smoke_df[
                "exact_utility"
            ]
            +
            COVERAGE_NUMERICAL_TOLERANCE
            <
            coverage_smoke_df[
                "tau"
            ]
        )
    ).sum()
)


greedy_threshold_failures = int(
    (
        (
            coverage_smoke_df[
                "greedy_status"
            ]
            ==
            "FEASIBLE"
        )
        &
        (
            coverage_smoke_df[
                "greedy_utility"
            ]
            +
            COVERAGE_NUMERICAL_TOLERANCE
            <
            coverage_smoke_df[
                "tau"
            ]
        )
    ).sum()
)


status_disagreements = int(
    (
        (
            coverage_smoke_df[
                "exact_status"
            ]
            ==
            "OPTIMAL"
        )
        &
        (
            coverage_smoke_df[
                "greedy_status"
            ]
            !=
            "FEASIBLE"
        )
    ).sum()
)


duplicate_conditions = int(
    coverage_smoke_df
    .duplicated(
        subset=[
            "regime",
            "tau",
            "weight_profile",
        ],
        keep=False,
    )
    .sum()
)


coverage_smoke_audit_df = pd.DataFrame(
    [
        {
            "rows":
                len(
                    coverage_smoke_df
                ),

            "expected_rows":
                48,

            "regimes":
                coverage_smoke_df[
                    "regime"
                ].nunique(),

            "thresholds":
                coverage_smoke_df[
                    "tau"
                ].nunique(),

            "weight_profiles":
                coverage_smoke_df[
                    "weight_profile"
                ].nunique(),

            "exact_optimal_rows":
                int(
                    (
                        coverage_smoke_df[
                            "exact_status"
                        ]
                        ==
                        "OPTIMAL"
                    ).sum()
                ),

            "greedy_feasible_rows":
                int(
                    (
                        coverage_smoke_df[
                            "greedy_status"
                        ]
                        ==
                        "FEASIBLE"
                    ).sum()
                ),

            "duplicate_conditions":
                duplicate_conditions,

            "negative_gap_rows":
                negative_gap_rows,

            "exact_threshold_failures":
                exact_threshold_failures,

            "greedy_threshold_failures":
                greedy_threshold_failures,

            "status_disagreements":
                status_disagreements,

            "maximum_feature_weight_difference":
                maximum_feature_weight_difference,

            "maximum_coverage_matrix_difference":
                maximum_coverage_matrix_difference,

            "maximum_sorted_privacy_difference":
                maximum_sorted_privacy_difference,

            "maximum_sorted_cognitive_difference":
                maximum_sorted_cognitive_difference,

            "minimum_optimality_gap_percent":
                coverage_smoke_df[
                    "optimality_gap_percent"
                ].min(),

            "maximum_optimality_gap_percent":
                coverage_smoke_df[
                    "optimality_gap_percent"
                ].max(),

            "mean_optimality_gap_percent":
                coverage_smoke_df[
                    "optimality_gap_percent"
                ].mean(),

            "exact_subset_space_size":
                int(
                    coverage_smoke_df[
                        "evaluated_subsets"
                    ].max()
                ),
        }
    ]
)


assert len(
    coverage_smoke_df
) == 48


assert duplicate_conditions == 0
assert negative_gap_rows == 0
assert exact_threshold_failures == 0
assert greedy_threshold_failures == 0
assert status_disagreements == 0


assert (
    coverage_smoke_df[
        "exact_status"
    ]
    ==
    "OPTIMAL"
).all()


assert (
    coverage_smoke_df[
        "greedy_status"
    ]
    ==
    "FEASIBLE"
).all()


assert (
    coverage_smoke_df[
        "evaluated_subsets"
    ]
    ==
    2
    **
    COVERAGE_SMOKE_N_ITEMS
).all()


# --------------------------------------------------
# Save outputs
# --------------------------------------------------

coverage_smoke_df.to_csv(
    COVERAGE_SMOKE_RESULTS_PATH,
    index=False,
)


coverage_smoke_audit_df.to_csv(
    COVERAGE_SMOKE_AUDIT_PATH,
    index=False,
)


# --------------------------------------------------
# Printed report
# --------------------------------------------------

coverage_smoke_summary_df = (
    coverage_smoke_df
    .groupby(
        "regime",
        as_index=False,
    )
    .agg(
        conditions=(
            "optimality_gap_percent",
            "size",
        ),

        mean_optimality_gap_percent=(
            "optimality_gap_percent",
            "mean",
        ),

        median_optimality_gap_percent=(
            "optimality_gap_percent",
            "median",
        ),

        maximum_optimality_gap_percent=(
            "optimality_gap_percent",
            "max",
        ),

        exact_subset_recovery_percent=(
            "same_selected_subset",
            lambda values:
                100.0
                *
                values.astype(
                    float
                ).mean(),
        ),

        mean_jaccard_similarity=(
            "jaccard_similarity",
            "mean",
        ),

        mean_exact_selected_count=(
            "exact_selected_count",
            "mean",
        ),

        mean_greedy_selected_count=(
            "greedy_selected_count",
            "mean",
        ),

        mean_exact_runtime_seconds=(
            "exact_runtime_seconds",
            "mean",
        ),

        mean_greedy_runtime_seconds=(
            "greedy_runtime_seconds",
            "mean",
        ),
    )
)


print(
    "\n======================================"
)

print(
    "COVERAGE EXACT-VERSUS-GREEDY SMOKE AUDIT"
)

print(
    "======================================"
)


print(
    coverage_smoke_audit_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.12f}",
    )
)


print(
    "\n======================================"
)

print(
    "COVERAGE SMOKE RESULTS BY REGIME"
)

print(
    "======================================"
)


print(
    coverage_smoke_summary_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.6f}",
    )
)


print(
    "\nSmoke-test results saved to:"
)

print(
    COVERAGE_SMOKE_RESULTS_PATH
)


print(
    "\nSmoke-test audit saved to:"
)

print(
    COVERAGE_SMOKE_AUDIT_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Step 128 — Pool-normalized coverage cached-Exact
# solver validation against the brute-force smoke test
# --------------------------------------------------

from pathlib import Path
import time

import numpy as np
import pandas as pd


# --------------------------------------------------
# Configuration
# --------------------------------------------------

CACHED_EXACT_TOLERANCE = 1e-10


if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


SMOKE_REFERENCE_PATH = (
    RESULTS_DIR
    / "coverage_exact_vs_greedy_smoke_test.csv"
)


CACHED_VALIDATION_PATH = (
    RESULTS_DIR
    / "coverage_cached_exact_smoke_validation.csv"
)


CACHED_VALIDATION_AUDIT_PATH = (
    RESULTS_DIR
    / "coverage_cached_exact_smoke_validation_audit.csv"
)


if "generate_matched_coverage_instances" not in globals():

    raise NameError(
        "generate_matched_coverage_instances is unavailable. "
        "Run Step 124 first."
    )


# --------------------------------------------------
# Helpers
# --------------------------------------------------

def parse_pipe_subset(
    stored_subset,
):
    """
    Parse strings such as:
        c001|c004|c009
    """

    if stored_subset is None:

        return tuple()


    try:

        if pd.isna(
            stored_subset
        ):

            return tuple()

    except (
        TypeError,
        ValueError,
    ):

        pass


    text = str(
        stored_subset
    ).strip()


    if not text:

        return tuple()


    return tuple(
        token.strip()
        for token in text.split("|")
        if token.strip()
    )



def subset_ids_from_mask(
    subset_mask,
    item_ids,
):
    """
    Return selected item identifiers in increasing
    candidate-index order.
    """

    return tuple(
        item_ids[
            item_index
        ]
        for item_index in range(
            len(
                item_ids
            )
        )
        if (
            subset_mask
            >>
            item_index
        )
        &
        1
    )


# --------------------------------------------------
# Cached subset precomputation
# --------------------------------------------------

def precompute_coverage_subset_cache(
    all_items,
    feature_weights,
):
    """
    Precompute utility and disclosure components for
    all 2^M subsets.

    Utility:
        candidate-pool-normalized weighted coverage

    Privacy:
        1 - product(1 - p_i)

    Cognitive:
        sum(c_i) / sum(all candidate cognitive costs)

    Cardinality:
        |S| / M
    """

    precomputation_start = time.perf_counter()


    n_items = len(
        all_items
    )


    if n_items <= 0:

        raise ValueError(
            "The candidate pool must not be empty."
        )


    item_ids = tuple(
        str(
            item.item_id
        )
        for item in all_items
    )


    if len(
        set(
            item_ids
        )
    ) != n_items:

        raise ValueError(
            "Candidate item identifiers must be unique."
        )


    feature_weights = np.asarray(
        feature_weights,
        dtype=float,
    )


    coverage_matrix = np.asarray(
        [
            item.feature_coverage
            for item in all_items
        ],
        dtype=float,
    )


    privacy_values = np.asarray(
        [
            float(
                item.privacy
            )
            for item in all_items
        ],
        dtype=float,
    )


    cognitive_values = np.asarray(
        [
            float(
                item.cognitive_cost
            )
            for item in all_items
        ],
        dtype=float,
    )


    if coverage_matrix.ndim != 2:

        raise ValueError(
            "The coverage matrix must be two-dimensional."
        )


    if (
        coverage_matrix.shape[0]
        != n_items
    ):

        raise ValueError(
            "Coverage rows must equal the number of items."
        )


    if (
        coverage_matrix.shape[1]
        != len(
            feature_weights
        )
    ):

        raise ValueError(
            "Coverage columns must match feature weights."
        )


    if np.any(
        feature_weights < 0.0
    ):

        raise ValueError(
            "Feature weights must be nonnegative."
        )


    full_feature_coverage = (
        coverage_matrix.max(
            axis=0
        )
    )


    utility_denominator = float(
        np.dot(
            feature_weights,
            full_feature_coverage,
        )
    )


    if utility_denominator <= 0.0:

        raise ValueError(
            "The candidate pool provides no positive "
            "weighted feature coverage."
        )


    full_cognitive_cost = float(
        cognitive_values.sum()
    )


    if full_cognitive_cost <= 0.0:

        raise ValueError(
            "The candidate pool must have positive "
            "total cognitive cost."
        )


    number_of_subsets = (
        1
        <<
        n_items
    )


    number_of_features = (
        coverage_matrix.shape[1]
    )


    maximum_coverage = np.zeros(
        (
            number_of_subsets,
            number_of_features,
        ),
        dtype=float,
    )


    privacy_failure_probability = np.ones(
        number_of_subsets,
        dtype=float,
    )


    cognitive_sum = np.zeros(
        number_of_subsets,
        dtype=float,
    )


    selected_count = np.zeros(
        number_of_subsets,
        dtype=np.int16,
    )


    # Add the highest-index selected item to a subset
    # containing only lower-index items. This preserves
    # increasing item order in floating-point summation.
    for subset_mask in range(
        1,
        number_of_subsets,
    ):

        highest_item_index = (
            subset_mask.bit_length()
            -
            1
        )


        highest_item_bit = (
            1
            <<
            highest_item_index
        )


        previous_mask = (
            subset_mask
            ^
            highest_item_bit
        )


        maximum_coverage[
            subset_mask
        ] = np.maximum(
            maximum_coverage[
                previous_mask
            ],
            coverage_matrix[
                highest_item_index
            ],
        )


        privacy_failure_probability[
            subset_mask
        ] = (
            privacy_failure_probability[
                previous_mask
            ]
            *
            (
                1.0
                -
                privacy_values[
                    highest_item_index
                ]
            )
        )


        cognitive_sum[
            subset_mask
        ] = (
            cognitive_sum[
                previous_mask
            ]
            +
            cognitive_values[
                highest_item_index
            ]
        )


        selected_count[
            subset_mask
        ] = (
            selected_count[
                previous_mask
            ]
            +
            1
        )


    utility = (
        maximum_coverage
        @
        feature_weights
    ) / utility_denominator


    privacy = (
        1.0
        -
        privacy_failure_probability
    )


    normalized_cognitive_cost = (
        cognitive_sum
        /
        full_cognitive_cost
    )


    cardinality_fraction = (
        selected_count.astype(
            float
        )
        /
        n_items
    )


    # Remove negligible floating-point excursions.
    utility = np.clip(
        utility,
        0.0,
        1.0,
    )


    precomputation_runtime_seconds = (
        time.perf_counter()
        -
        precomputation_start
    )


    return {
        "n_items":
            n_items,

        "number_of_subsets":
            number_of_subsets,

        "item_ids":
            item_ids,

        "utility":
            utility,

        "privacy":
            privacy,

        "normalized_cognitive_cost":
            normalized_cognitive_cost,

        "cardinality_fraction":
            cardinality_fraction,

        "selected_count":
            selected_count,

        "precomputation_runtime_seconds":
            precomputation_runtime_seconds,

        "full_set_utility":
            float(
                utility[
                    number_of_subsets
                    -
                    1
                ]
            ),
    }


# --------------------------------------------------
# Cached Exact solve
# --------------------------------------------------

def cached_exact_coverage_mst(
    subset_cache,
    tau,
    lambda_privacy,
    lambda_cognitive,
    lambda_cardinality,
):
    """
    Find the minimum-disclosure-cost sufficient subset
    using precomputed subset quantities.
    """

    solve_start = time.perf_counter()


    utility = subset_cache[
        "utility"
    ]


    feasible_indices = np.flatnonzero(
        utility
        +
        CACHED_EXACT_TOLERANCE
        >=
        tau
    )


    if len(
        feasible_indices
    ) == 0:

        return {
            "status":
                "INFEASIBLE",

            "cached_solve_runtime_seconds":
                (
                    time.perf_counter()
                    -
                    solve_start
                ),
        }


    disclosure_cost = (
        lambda_privacy
        *
        subset_cache[
            "privacy"
        ]
        +
        lambda_cognitive
        *
        subset_cache[
            "normalized_cognitive_cost"
        ]
        +
        lambda_cardinality
        *
        subset_cache[
            "cardinality_fraction"
        ]
    )


    feasible_costs = (
        disclosure_cost[
            feasible_indices
        ]
    )


    minimum_cost = float(
        feasible_costs.min()
    )


    # Exact floating-point ties only. No tolerance is
    # used here because the brute-force reference used
    # the raw tuple cost as its primary comparison key.
    tied_indices = feasible_indices[
        feasible_costs
        ==
        minimum_cost
    ]


    if len(
        tied_indices
    ) == 1:

        best_mask = int(
            tied_indices[0]
        )


    else:

        best_mask = min(
            (
                int(
                    subset_mask
                )
                for subset_mask in tied_indices
            ),
            key=lambda subset_mask: (
                int(
                    subset_cache[
                        "selected_count"
                    ][
                        subset_mask
                    ]
                ),

                subset_ids_from_mask(
                    subset_mask=
                        subset_mask,

                    item_ids=
                        subset_cache[
                            "item_ids"
                        ],
                ),
            ),
        )


    selected_ids = subset_ids_from_mask(
        subset_mask=
            best_mask,

        item_ids=
            subset_cache[
                "item_ids"
            ],
    )


    cached_solve_runtime_seconds = (
        time.perf_counter()
        -
        solve_start
    )


    return {
        "status":
            "OPTIMAL",

        "selected_mask":
            best_mask,

        "selected_ids":
            selected_ids,

        "selected_count":
            int(
                subset_cache[
                    "selected_count"
                ][
                    best_mask
                ]
            ),

        "utility":
            float(
                subset_cache[
                    "utility"
                ][
                    best_mask
                ]
            ),

        "privacy":
            float(
                subset_cache[
                    "privacy"
                ][
                    best_mask
                ]
            ),

        "normalized_cognitive_cost":
            float(
                subset_cache[
                    "normalized_cognitive_cost"
                ][
                    best_mask
                ]
            ),

        "cardinality_fraction":
            float(
                subset_cache[
                    "cardinality_fraction"
                ][
                    best_mask
                ]
            ),

        "disclosure_cost":
            float(
                disclosure_cost[
                    best_mask
                ]
            ),

        "cached_solve_runtime_seconds":
            cached_solve_runtime_seconds,
    }


# --------------------------------------------------
# Load Step-127 brute-force reference
# --------------------------------------------------

smoke_reference_df = pd.read_csv(
    SMOKE_REFERENCE_PATH,
    low_memory=False,
)


required_reference_columns = [
    "matched_seed",
    "n_items",
    "regime",
    "tau",
    "weight_profile",
    "lambda_privacy",
    "lambda_cognitive",
    "lambda_cardinality",
    "exact_status",
    "exact_selected_count",
    "exact_selected_items",
    "exact_utility",
    "exact_privacy",
    "exact_normalized_cognitive_cost",
    "exact_disclosure_cost",
]


missing_reference_columns = [
    column_name
    for column_name in required_reference_columns
    if column_name not in smoke_reference_df.columns
]


if missing_reference_columns:

    raise KeyError(
        "Missing smoke-reference columns: "
        + ", ".join(
            missing_reference_columns
        )
    )


assert len(
    smoke_reference_df
) == 48


unique_seeds = (
    smoke_reference_df[
        "matched_seed"
    ]
    .dropna()
    .unique()
)


unique_sizes = (
    smoke_reference_df[
        "n_items"
    ]
    .dropna()
    .unique()
)


assert len(
    unique_seeds
) == 1


assert len(
    unique_sizes
) == 1


matched_seed = int(
    unique_seeds[0]
)


n_items = int(
    unique_sizes[0]
)


# --------------------------------------------------
# Recreate the identical matched group
# --------------------------------------------------

matched_instances = (
    generate_matched_coverage_instances(
        n_items=n_items,

        n_features=globals().get(
            "COVERAGE_SMOKE_FEATURES",
            globals().get(
                "COVERAGE_FEATURES",
                12,
            ),
        ),

        seed=matched_seed,

        coverage_density=globals().get(
            "COVERAGE_SMOKE_DENSITY",
            globals().get(
                "COVERAGE_DENSITY",
                0.35,
            ),
        ),

        regime_strength=globals().get(
            "COVERAGE_SMOKE_REGIME_STRENGTH",
            globals().get(
                "COVERAGE_REGIME_STRENGTH",
                0.75,
            ),
        ),
    )
)


# --------------------------------------------------
# Precompute one cache per regime
# --------------------------------------------------

subset_caches = {}


for regime in sorted(
    smoke_reference_df[
        "regime"
    ].unique()
):

    regime_data = (
        matched_instances[
            regime
        ]
    )


    subset_caches[
        regime
    ] = (
        precompute_coverage_subset_cache(
            all_items=
                regime_data[
                    "items"
                ],

            feature_weights=
                regime_data[
                    "feature_weights"
                ],
        )
    )


    print(
        "Completed cached subset precomputation:",
        regime,
        "| subsets =",
        subset_caches[
            regime
        ][
            "number_of_subsets"
        ],
        "| seconds =",
        f"{subset_caches[regime]['precomputation_runtime_seconds']:.4f}",
    )


# --------------------------------------------------
# Re-solve all 48 Exact conditions
# --------------------------------------------------

validation_rows = []


for _, reference_row in (
    smoke_reference_df
    .sort_values(
        [
            "regime",
            "tau",
            "weight_profile",
        ]
    )
    .iterrows()
):

    regime = reference_row[
        "regime"
    ]


    cached_result = (
        cached_exact_coverage_mst(
            subset_cache=
                subset_caches[
                    regime
                ],

            tau=
                float(
                    reference_row[
                        "tau"
                    ]
                ),

            lambda_privacy=
                float(
                    reference_row[
                        "lambda_privacy"
                    ]
                ),

            lambda_cognitive=
                float(
                    reference_row[
                        "lambda_cognitive"
                    ]
                ),

            lambda_cardinality=
                float(
                    reference_row[
                        "lambda_cardinality"
                    ]
                ),
        )
    )


    reference_selected_ids = (
        parse_pipe_subset(
            reference_row[
                "exact_selected_items"
            ]
        )
    )


    cached_selected_ids = tuple(
        cached_result.get(
            "selected_ids",
            tuple(),
        )
    )


    validation_rows.append(
        {
            "regime":
                regime,

            "tau":
                float(
                    reference_row[
                        "tau"
                    ]
                ),

            "weight_profile":
                reference_row[
                    "weight_profile"
                ],

            "reference_status":
                reference_row[
                    "exact_status"
                ],

            "cached_status":
                cached_result[
                    "status"
                ],

            "status_match":
                (
                    reference_row[
                        "exact_status"
                    ]
                    ==
                    cached_result[
                        "status"
                    ]
                ),

            "reference_selected_items":
                "|".join(
                    reference_selected_ids
                ),

            "cached_selected_items":
                "|".join(
                    cached_selected_ids
                ),

            "selected_subset_match":
                (
                    reference_selected_ids
                    ==
                    cached_selected_ids
                ),

            "reference_selected_count":
                int(
                    reference_row[
                        "exact_selected_count"
                    ]
                ),

            "cached_selected_count":
                cached_result.get(
                    "selected_count",
                    np.nan,
                ),

            "selected_count_match":
                (
                    int(
                        reference_row[
                            "exact_selected_count"
                        ]
                    )
                    ==
                    cached_result.get(
                        "selected_count",
                        -1,
                    )
                ),

            "absolute_utility_difference":
                abs(
                    float(
                        reference_row[
                            "exact_utility"
                        ]
                    )
                    -
                    cached_result.get(
                        "utility",
                        np.nan,
                    )
                ),

            "absolute_privacy_difference":
                abs(
                    float(
                        reference_row[
                            "exact_privacy"
                        ]
                    )
                    -
                    cached_result.get(
                        "privacy",
                        np.nan,
                    )
                ),

            "absolute_normalized_cognitive_difference":
                abs(
                    float(
                        reference_row[
                            "exact_normalized_cognitive_cost"
                        ]
                    )
                    -
                    cached_result.get(
                        "normalized_cognitive_cost",
                        np.nan,
                    )
                ),

            "absolute_disclosure_cost_difference":
                abs(
                    float(
                        reference_row[
                            "exact_disclosure_cost"
                        ]
                    )
                    -
                    cached_result.get(
                        "disclosure_cost",
                        np.nan,
                    )
                ),

            "cached_solve_runtime_seconds":
                cached_result[
                    "cached_solve_runtime_seconds"
                ],
        }
    )


validation_df = pd.DataFrame(
    validation_rows
)


# --------------------------------------------------
# Validation audit
# --------------------------------------------------

audit_df = pd.DataFrame(
    [
        {
            "validation_rows":
                len(
                    validation_df
                ),

            "expected_rows":
                48,

            "status_matches":
                int(
                    validation_df[
                        "status_match"
                    ].sum()
                ),

            "selected_subset_matches":
                int(
                    validation_df[
                        "selected_subset_match"
                    ].sum()
                ),

            "selected_count_matches":
                int(
                    validation_df[
                        "selected_count_match"
                    ].sum()
                ),

            "maximum_absolute_utility_difference":
                validation_df[
                    "absolute_utility_difference"
                ].max(),

            "maximum_absolute_privacy_difference":
                validation_df[
                    "absolute_privacy_difference"
                ].max(),

            "maximum_absolute_normalized_cognitive_difference":
                validation_df[
                    "absolute_normalized_cognitive_difference"
                ].max(),

            "maximum_absolute_disclosure_cost_difference":
                validation_df[
                    "absolute_disclosure_cost_difference"
                ].max(),

            "maximum_full_set_deviation_from_one":
                max(
                    abs(
                        subset_cache[
                            "full_set_utility"
                        ]
                        -
                        1.0
                    )
                    for subset_cache in (
                        subset_caches.values()
                    )
                ),

            "total_subset_precomputation_seconds":
                sum(
                    subset_cache[
                        "precomputation_runtime_seconds"
                    ]
                    for subset_cache in (
                        subset_caches.values()
                    )
                ),

            "mean_cached_solve_milliseconds":
                1000.0
                *
                validation_df[
                    "cached_solve_runtime_seconds"
                ].mean(),

            "maximum_cached_solve_milliseconds":
                1000.0
                *
                validation_df[
                    "cached_solve_runtime_seconds"
                ].max(),

            "subset_space_size_per_regime":
                2
                **
                n_items,
        }
    ]
)


# --------------------------------------------------
# Integrity assertions
# --------------------------------------------------

assert len(
    validation_df
) == 48


assert validation_df[
    "status_match"
].all()


assert validation_df[
    "selected_subset_match"
].all()


assert validation_df[
    "selected_count_match"
].all()


assert (
    validation_df[
        "absolute_utility_difference"
    ].max()
    <=
    CACHED_EXACT_TOLERANCE
)


assert (
    validation_df[
        "absolute_privacy_difference"
    ].max()
    <=
    CACHED_EXACT_TOLERANCE
)


assert (
    validation_df[
        "absolute_normalized_cognitive_difference"
    ].max()
    <=
    CACHED_EXACT_TOLERANCE
)


assert (
    validation_df[
        "absolute_disclosure_cost_difference"
    ].max()
    <=
    CACHED_EXACT_TOLERANCE
)


assert (
    audit_df.loc[
        0,
        "maximum_full_set_deviation_from_one",
    ]
    <=
    CACHED_EXACT_TOLERANCE
)


# --------------------------------------------------
# Save results
# --------------------------------------------------

validation_df.to_csv(
    CACHED_VALIDATION_PATH,
    index=False,
)


audit_df.to_csv(
    CACHED_VALIDATION_AUDIT_PATH,
    index=False,
)


# --------------------------------------------------
# Printed report
# --------------------------------------------------

print(
    "\n======================================"
)

print(
    "COVERAGE CACHED-EXACT VALIDATION AUDIT"
)

print(
    "======================================"
)


print(
    audit_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.12f}",
    )
)


print(
    "\n======================================"
)

print(
    "CACHED-EXACT VALIDATION BY REGIME"
)

print(
    "======================================"
)


regime_summary_df = (
    validation_df
    .groupby(
        "regime",
        as_index=False,
    )
    .agg(
        conditions=(
            "status_match",
            "size",
        ),

        status_matches=(
            "status_match",
            "sum",
        ),

        selected_subset_matches=(
            "selected_subset_match",
            "sum",
        ),

        selected_count_matches=(
            "selected_count_match",
            "sum",
        ),

        maximum_utility_difference=(
            "absolute_utility_difference",
            "max",
        ),

        maximum_privacy_difference=(
            "absolute_privacy_difference",
            "max",
        ),

        maximum_cognitive_difference=(
            "absolute_normalized_cognitive_difference",
            "max",
        ),

        maximum_disclosure_cost_difference=(
            "absolute_disclosure_cost_difference",
            "max",
        ),

        mean_cached_solve_milliseconds=(
            "cached_solve_runtime_seconds",
            lambda values:
                1000.0
                *
                values.mean(),
        ),
    )
)


print(
    regime_summary_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.12f}",
    )
)


print(
    "\nValidation rows saved to:"
)

print(
    CACHED_VALIDATION_PATH
)


print(
    "\nValidation audit saved to:"
)

print(
    CACHED_VALIDATION_AUDIT_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Step 129 — Full pool-normalized coverage benchmark:
# Cached Exact versus Dynamic Greedy
# --------------------------------------------------

from pathlib import Path
import time

import numpy as np
import pandas as pd


# --------------------------------------------------
# Configuration
# --------------------------------------------------

COVERAGE_MAIN_BENCHMARK_VERSION = (
    "coverage_pool_normalized_exact_greedy_v1"
)


COVERAGE_MAIN_BASE_SEED = (
    globals().get(
        "GLOBAL_SEED",
        2026,
    )
    + 129
)


COVERAGE_MAIN_SIZES = [
    8,
    10,
    12,
    14,
    16,
]


COVERAGE_MAIN_REPLICATES_PER_SIZE = 100


COVERAGE_MAIN_REGIMES = [
    "independent",
    "utility_burden_alignment",
    "utility_burden_conflict",
]


COVERAGE_MAIN_THRESHOLDS = [
    0.70,
    0.80,
    0.90,
    0.95,
]


COVERAGE_MAIN_WEIGHT_PROFILES = {
    "balanced": {
        "lambda_privacy": 1 / 3,
        "lambda_cognitive": 1 / 3,
        "lambda_cardinality": 1 / 3,
    },

    "privacy_priority": {
        "lambda_privacy": 0.60,
        "lambda_cognitive": 0.20,
        "lambda_cardinality": 0.20,
    },

    "interpretation_priority": {
        "lambda_privacy": 0.20,
        "lambda_cognitive": 0.60,
        "lambda_cardinality": 0.20,
    },

    "parsimony_priority": {
        "lambda_privacy": 0.20,
        "lambda_cognitive": 0.20,
        "lambda_cardinality": 0.60,
    },
}


COVERAGE_MAIN_FEATURES = globals().get(
    "COVERAGE_FEATURES",
    12,
)


COVERAGE_MAIN_DENSITY = globals().get(
    "COVERAGE_DENSITY",
    0.35,
)


COVERAGE_MAIN_REGIME_STRENGTH = globals().get(
    "COVERAGE_REGIME_STRENGTH",
    0.75,
)


COVERAGE_MAIN_TOLERANCE = 1e-12

COVERAGE_EXACT_RECOVERY_TOLERANCE_PERCENT = 1e-10

COVERAGE_MAIN_RESUME = True


EXPECTED_CONDITIONS_PER_MATCHED_GROUP = (
    len(
        COVERAGE_MAIN_REGIMES
    )
    *
    len(
        COVERAGE_MAIN_THRESHOLDS
    )
    *
    len(
        COVERAGE_MAIN_WEIGHT_PROFILES
    )
)


EXPECTED_MATCHED_GROUPS = (
    len(
        COVERAGE_MAIN_SIZES
    )
    *
    COVERAGE_MAIN_REPLICATES_PER_SIZE
)


EXPECTED_BENCHMARK_ROWS = (
    EXPECTED_MATCHED_GROUPS
    *
    EXPECTED_CONDITIONS_PER_MATCHED_GROUP
)


# --------------------------------------------------
# Project paths
# --------------------------------------------------

if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


if "generate_matched_coverage_instances" not in globals():

    raise NameError(
        "generate_matched_coverage_instances is unavailable. "
        "Run Step 124 first."
    )


# --------------------------------------------------
# Output paths
# --------------------------------------------------

COVERAGE_MAIN_RESULTS_PATH = (
    RESULTS_DIR
    / (
        "coverage_exact_vs_greedy_"
        "main_benchmark_v1_pool_normalized.csv"
    )
)


COVERAGE_MAIN_GROUP_AUDIT_PATH = (
    RESULTS_DIR
    / (
        "coverage_exact_vs_greedy_"
        "main_benchmark_v1_group_audit.csv"
    )
)


COVERAGE_MAIN_OVERALL_SUMMARY_PATH = (
    TABLES_DIR
    / (
        "coverage_exact_vs_greedy_"
        "main_overall_summary.csv"
    )
)


COVERAGE_MAIN_REGIME_SUMMARY_PATH = (
    TABLES_DIR
    / (
        "coverage_exact_vs_greedy_"
        "main_by_regime_summary.csv"
    )
)


COVERAGE_MAIN_RUNTIME_SUMMARY_PATH = (
    TABLES_DIR
    / (
        "coverage_exact_vs_greedy_"
        "runtime_by_candidate_size.csv"
    )
)


COVERAGE_MAIN_METADATA_PATH = (
    RESULTS_DIR
    / (
        "coverage_exact_vs_greedy_"
        "main_benchmark_metadata.csv"
    )
)


# --------------------------------------------------
# Utility helpers
# --------------------------------------------------

def coverage_subset_ids_from_mask(
    subset_mask,
    item_ids,
):
    """
    Return selected identifiers in candidate-index
    order.
    """

    return tuple(
        item_ids[
            item_index
        ]
        for item_index in range(
            len(
                item_ids
            )
        )
        if (
            subset_mask
            >>
            item_index
        )
        &
        1
    )


def coverage_jaccard(
    first_ids,
    second_ids,
):
    """
    Jaccard similarity between two selected subsets.
    """

    first_set = set(
        first_ids
    )


    second_set = set(
        second_ids
    )


    union_set = (
        first_set
        |
        second_set
    )


    if not union_set:

        return 1.0


    return (
        len(
            first_set
            &
            second_set
        )
        /
        len(
            union_set
        )
    )


# --------------------------------------------------
# Shared utility-cache precomputation
# --------------------------------------------------

def precompute_shared_coverage_utility_cache(
    all_items,
    feature_weights,
):
    """
    Precompute pool-normalized coverage utility for all
    2^M subsets.

    The utility structure is shared across the three
    matched burden-association regimes.
    """

    start_time = time.perf_counter()


    n_items = len(
        all_items
    )


    if n_items <= 0:

        raise ValueError(
            "The candidate pool cannot be empty."
        )


    item_ids = tuple(
        str(
            item.item_id
        )
        for item in all_items
    )


    if len(
        set(
            item_ids
        )
    ) != n_items:

        raise ValueError(
            "Candidate item identifiers must be unique."
        )


    feature_weights = np.asarray(
        feature_weights,
        dtype=float,
    )


    coverage_matrix = np.asarray(
        [
            item.feature_coverage
            for item in all_items
        ],
        dtype=float,
    )


    if coverage_matrix.ndim != 2:

        raise ValueError(
            "The coverage matrix must be two-dimensional."
        )


    if (
        coverage_matrix.shape[1]
        != len(
            feature_weights
        )
    ):

        raise ValueError(
            "Coverage dimensions do not match "
            "feature weights."
        )


    available_feature_coverage = (
        coverage_matrix.max(
            axis=0
        )
    )


    utility_denominator = float(
        np.dot(
            feature_weights,
            available_feature_coverage,
        )
    )


    if utility_denominator <= 0.0:

        raise ValueError(
            "The candidate pool has zero attainable "
            "weighted coverage."
        )


    number_of_subsets = (
        1
        <<
        n_items
    )


    number_of_features = (
        coverage_matrix.shape[1]
    )


    maximum_coverage = np.zeros(
        (
            number_of_subsets,
            number_of_features,
        ),
        dtype=float,
    )


    selected_count = np.zeros(
        number_of_subsets,
        dtype=np.int16,
    )


    for subset_mask in range(
        1,
        number_of_subsets,
    ):

        highest_item_index = (
            subset_mask.bit_length()
            -
            1
        )


        highest_item_bit = (
            1
            <<
            highest_item_index
        )


        previous_mask = (
            subset_mask
            ^
            highest_item_bit
        )


        maximum_coverage[
            subset_mask
        ] = np.maximum(
            maximum_coverage[
                previous_mask
            ],
            coverage_matrix[
                highest_item_index
            ],
        )


        selected_count[
            subset_mask
        ] = (
            selected_count[
                previous_mask
            ]
            +
            1
        )


    utility = (
        maximum_coverage
        @
        feature_weights
    ) / utility_denominator


    utility = np.clip(
        utility,
        0.0,
        1.0,
    )


    cardinality_fraction = (
        selected_count.astype(
            float
        )
        /
        n_items
    )


    runtime_seconds = (
        time.perf_counter()
        -
        start_time
    )


    return {
        "n_items":
            n_items,

        "number_of_subsets":
            number_of_subsets,

        "item_ids":
            item_ids,

        "feature_weights":
            feature_weights,

        "coverage_matrix":
            coverage_matrix,

        "utility_denominator":
            utility_denominator,

        "utility":
            utility,

        "selected_count":
            selected_count,

        "cardinality_fraction":
            cardinality_fraction,

        "full_set_utility":
            float(
                utility[
                    number_of_subsets
                    -
                    1
                ]
            ),

        "runtime_seconds":
            runtime_seconds,
    }


# --------------------------------------------------
# Regime-specific burden cache
# --------------------------------------------------

def precompute_coverage_burden_cache(
    all_items,
    shared_cache,
):
    """
    Precompute privacy exposure and normalized modeled
    interpretation cost for every subset.
    """

    start_time = time.perf_counter()


    n_items = shared_cache[
        "n_items"
    ]


    number_of_subsets = shared_cache[
        "number_of_subsets"
    ]


    if len(
        all_items
    ) != n_items:

        raise ValueError(
            "Burden items do not match the shared "
            "candidate pool size."
        )


    privacy_values = np.asarray(
        [
            float(
                item.privacy
            )
            for item in all_items
        ],
        dtype=float,
    )


    cognitive_values = np.asarray(
        [
            float(
                item.cognitive_cost
            )
            for item in all_items
        ],
        dtype=float,
    )


    if np.any(
        privacy_values < 0.0
    ) or np.any(
        privacy_values > 1.0
    ):

        raise ValueError(
            "Privacy values must lie in [0, 1]."
        )


    if np.any(
        cognitive_values < 0.0
    ):

        raise ValueError(
            "Modeled interpretation costs must be "
            "nonnegative."
        )


    total_cognitive_cost = float(
        cognitive_values.sum()
    )


    if total_cognitive_cost <= 0.0:

        raise ValueError(
            "The candidate pool must have positive "
            "total modeled interpretation cost."
        )


    privacy_failure_probability = np.ones(
        number_of_subsets,
        dtype=float,
    )


    cognitive_sum = np.zeros(
        number_of_subsets,
        dtype=float,
    )


    for subset_mask in range(
        1,
        number_of_subsets,
    ):

        highest_item_index = (
            subset_mask.bit_length()
            -
            1
        )


        highest_item_bit = (
            1
            <<
            highest_item_index
        )


        previous_mask = (
            subset_mask
            ^
            highest_item_bit
        )


        privacy_failure_probability[
            subset_mask
        ] = (
            privacy_failure_probability[
                previous_mask
            ]
            *
            (
                1.0
                -
                privacy_values[
                    highest_item_index
                ]
            )
        )


        cognitive_sum[
            subset_mask
        ] = (
            cognitive_sum[
                previous_mask
            ]
            +
            cognitive_values[
                highest_item_index
            ]
        )


    privacy = (
        1.0
        -
        privacy_failure_probability
    )


    normalized_cognitive_cost = (
        cognitive_sum
        /
        total_cognitive_cost
    )


    runtime_seconds = (
        time.perf_counter()
        -
        start_time
    )


    return {
        "privacy_values":
            privacy_values,

        "cognitive_values":
            cognitive_values,

        "total_cognitive_cost":
            total_cognitive_cost,

        "privacy":
            privacy,

        "normalized_cognitive_cost":
            normalized_cognitive_cost,

        "runtime_seconds":
            runtime_seconds,
    }


# --------------------------------------------------
# Cached Exact solver
# --------------------------------------------------

def solve_cached_exact_coverage(
    shared_cache,
    burden_cache,
    tau,
    weights,
):
    """
    Select the minimum-disclosure-cost subset satisfying
    the modeled utility threshold.
    """

    start_time = time.perf_counter()


    utility = shared_cache[
        "utility"
    ]


    feasible_masks = np.flatnonzero(
        utility
        +
        COVERAGE_MAIN_TOLERANCE
        >=
        tau
    )


    if len(
        feasible_masks
    ) == 0:

        return {
            "status":
                "INFEASIBLE",

            "runtime_seconds":
                (
                    time.perf_counter()
                    -
                    start_time
                ),
        }


    disclosure_cost = (
        weights[
            "lambda_privacy"
        ]
        *
        burden_cache[
            "privacy"
        ]
        +
        weights[
            "lambda_cognitive"
        ]
        *
        burden_cache[
            "normalized_cognitive_cost"
        ]
        +
        weights[
            "lambda_cardinality"
        ]
        *
        shared_cache[
            "cardinality_fraction"
        ]
    )


    feasible_costs = (
        disclosure_cost[
            feasible_masks
        ]
    )


    minimum_cost = float(
        feasible_costs.min()
    )


    tied_masks = feasible_masks[
        feasible_costs
        ==
        minimum_cost
    ]


    if len(
        tied_masks
    ) == 1:

        best_mask = int(
            tied_masks[0]
        )


    else:

        best_mask = min(
            (
                int(
                    subset_mask
                )
                for subset_mask in tied_masks
            ),
            key=lambda subset_mask: (
                int(
                    shared_cache[
                        "selected_count"
                    ][
                        subset_mask
                    ]
                ),

                coverage_subset_ids_from_mask(
                    subset_mask,
                    shared_cache[
                        "item_ids"
                    ],
                ),
            ),
        )


    selected_ids = coverage_subset_ids_from_mask(
        best_mask,
        shared_cache[
            "item_ids"
        ],
    )


    return {
        "status":
            "OPTIMAL",

        "selected_mask":
            best_mask,

        "selected_ids":
            selected_ids,

        "selected_count":
            int(
                shared_cache[
                    "selected_count"
                ][
                    best_mask
                ]
            ),

        "utility":
            float(
                shared_cache[
                    "utility"
                ][
                    best_mask
                ]
            ),

        "privacy":
            float(
                burden_cache[
                    "privacy"
                ][
                    best_mask
                ]
            ),

        "normalized_cognitive_cost":
            float(
                burden_cache[
                    "normalized_cognitive_cost"
                ][
                    best_mask
                ]
            ),

        "cardinality_fraction":
            float(
                shared_cache[
                    "cardinality_fraction"
                ][
                    best_mask
                ]
            ),

        "disclosure_cost":
            float(
                disclosure_cost[
                    best_mask
                ]
            ),

        "runtime_seconds":
            (
                time.perf_counter()
                -
                start_time
            ),
    }


# --------------------------------------------------
# Dynamic marginal Greedy solver
# --------------------------------------------------

def solve_dynamic_greedy_coverage(
    all_items,
    shared_cache,
    burden_cache,
    tau,
    weights,
):
    """
    Dynamic Greedy selection based on current marginal
    modeled utility divided by current marginal
    disclosure cost.
    """

    start_time = time.perf_counter()


    n_items = shared_cache[
        "n_items"
    ]


    feature_weights = shared_cache[
        "feature_weights"
    ]


    coverage_matrix = shared_cache[
        "coverage_matrix"
    ]


    utility_denominator = shared_cache[
        "utility_denominator"
    ]


    privacy_values = burden_cache[
        "privacy_values"
    ]


    cognitive_values = burden_cache[
        "cognitive_values"
    ]


    total_cognitive_cost = burden_cache[
        "total_cognitive_cost"
    ]


    selected_indices = []
    remaining_indices = set(
        range(
            n_items
        )
    )


    current_feature_coverage = np.zeros(
        coverage_matrix.shape[1],
        dtype=float,
    )


    current_utility = 0.0
    current_privacy = 0.0
    current_normalized_cognitive = 0.0
    current_cardinality_fraction = 0.0


    current_disclosure_cost = 0.0


    while (
        current_utility
        +
        COVERAGE_MAIN_TOLERANCE
        <
        tau
        and
        remaining_indices
    ):

        candidate_rows = []


        for candidate_index in sorted(
            remaining_indices
        ):

            candidate_feature_coverage = np.maximum(
                current_feature_coverage,
                coverage_matrix[
                    candidate_index
                ],
            )


            candidate_utility = float(
                np.dot(
                    feature_weights,
                    candidate_feature_coverage,
                )
                /
                utility_denominator
            )


            candidate_privacy = (
                1.0
                -
                (
                    1.0
                    -
                    current_privacy
                )
                *
                (
                    1.0
                    -
                    privacy_values[
                        candidate_index
                    ]
                )
            )


            candidate_normalized_cognitive = (
                current_normalized_cognitive
                +
                cognitive_values[
                    candidate_index
                ]
                /
                total_cognitive_cost
            )


            candidate_cardinality_fraction = (
                current_cardinality_fraction
                +
                1.0
                /
                n_items
            )


            candidate_disclosure_cost = (
                weights[
                    "lambda_privacy"
                ]
                *
                candidate_privacy
                +
                weights[
                    "lambda_cognitive"
                ]
                *
                candidate_normalized_cognitive
                +
                weights[
                    "lambda_cardinality"
                ]
                *
                candidate_cardinality_fraction
            )


            marginal_utility = (
                candidate_utility
                -
                current_utility
            )


            marginal_cost = (
                candidate_disclosure_cost
                -
                current_disclosure_cost
            )


            if marginal_cost > 0.0:

                score = (
                    marginal_utility
                    /
                    marginal_cost
                )


            elif marginal_utility > 0.0:

                score = np.inf


            else:

                score = -np.inf


            candidate_rows.append(
                {
                    "candidate_index":
                        candidate_index,

                    "candidate_id":
                        str(
                            all_items[
                                candidate_index
                            ].item_id
                        ),

                    "candidate_feature_coverage":
                        candidate_feature_coverage,

                    "candidate_utility":
                        candidate_utility,

                    "candidate_privacy":
                        candidate_privacy,

                    "candidate_normalized_cognitive":
                        candidate_normalized_cognitive,

                    "candidate_cardinality_fraction":
                        candidate_cardinality_fraction,

                    "candidate_disclosure_cost":
                        candidate_disclosure_cost,

                    "marginal_utility":
                        marginal_utility,

                    "marginal_cost":
                        marginal_cost,

                    "score":
                        score,
                }
            )


        best_candidate = min(
            candidate_rows,
            key=lambda row: (
                -row[
                    "score"
                ],

                -row[
                    "marginal_utility"
                ],

                row[
                    "marginal_cost"
                ],

                row[
                    "candidate_id"
                ],
            ),
        )


        if (
            best_candidate[
                "marginal_utility"
            ]
            <=
            COVERAGE_MAIN_TOLERANCE
        ):

            break


        selected_indices.append(
            best_candidate[
                "candidate_index"
            ]
        )


        selected_indices.sort()


        remaining_indices.remove(
            best_candidate[
                "candidate_index"
            ]
        )


        current_feature_coverage = (
            best_candidate[
                "candidate_feature_coverage"
            ]
        )


        current_utility = (
            best_candidate[
                "candidate_utility"
            ]
        )


        current_privacy = (
            best_candidate[
                "candidate_privacy"
            ]
        )


        current_normalized_cognitive = (
            best_candidate[
                "candidate_normalized_cognitive"
            ]
        )


        current_cardinality_fraction = (
            best_candidate[
                "candidate_cardinality_fraction"
            ]
        )


        current_disclosure_cost = (
            best_candidate[
                "candidate_disclosure_cost"
            ]
        )


    selected_ids = tuple(
        str(
            all_items[
                item_index
            ].item_id
        )
        for item_index in selected_indices
    )


    status = (
        "FEASIBLE"
        if (
            current_utility
            +
            COVERAGE_MAIN_TOLERANCE
            >=
            tau
        )
        else
        "INFEASIBLE"
    )


    return {
        "status":
            status,

        "selected_ids":
            selected_ids,

        "selected_count":
            len(
                selected_ids
            ),

        "utility":
            float(
                current_utility
            ),

        "privacy":
            float(
                current_privacy
            ),

        "normalized_cognitive_cost":
            float(
                current_normalized_cognitive
            ),

        "cardinality_fraction":
            float(
                current_cardinality_fraction
            ),

        "disclosure_cost":
            float(
                current_disclosure_cost
            ),

        "runtime_seconds":
            (
                time.perf_counter()
                -
                start_time
            ),
    }


# --------------------------------------------------
# Resume completed checkpoint groups
# --------------------------------------------------

if (
    COVERAGE_MAIN_RESUME
    and
    COVERAGE_MAIN_RESULTS_PATH.exists()
):

    existing_results_df = pd.read_csv(
        COVERAGE_MAIN_RESULTS_PATH,
        low_memory=False,
    )


    if (
        "benchmark_version"
        in existing_results_df.columns
    ):

        existing_results_df = (
            existing_results_df[
                existing_results_df[
                    "benchmark_version"
                ]
                ==
                COVERAGE_MAIN_BENCHMARK_VERSION
            ]
            .copy()
        )


    existing_group_counts = (
        existing_results_df
        .groupby(
            "matched_group_id"
        )
        .size()
    )


    completed_group_ids = set(
        existing_group_counts[
            existing_group_counts
            ==
            EXPECTED_CONDITIONS_PER_MATCHED_GROUP
        ].index
    )


    existing_results_df = (
        existing_results_df[
            existing_results_df[
                "matched_group_id"
            ].isin(
                completed_group_ids
            )
        ]
        .copy()
    )


else:

    existing_results_df = pd.DataFrame()

    completed_group_ids = set()


if (
    COVERAGE_MAIN_RESUME
    and
    COVERAGE_MAIN_GROUP_AUDIT_PATH.exists()
):

    existing_audit_df = pd.read_csv(
        COVERAGE_MAIN_GROUP_AUDIT_PATH,
        low_memory=False,
    )


    existing_audit_df = (
        existing_audit_df[
            existing_audit_df[
                "matched_group_id"
            ].isin(
                completed_group_ids
            )
        ]
        .copy()
    )


else:

    existing_audit_df = pd.DataFrame()


print(
    "Previously completed matched groups:",
    len(
        completed_group_ids
    )
)


# --------------------------------------------------
# Run benchmark
# --------------------------------------------------

new_result_rows = []
new_audit_rows = []

benchmark_start_time = time.perf_counter()


for n_items in COVERAGE_MAIN_SIZES:

    for replicate in range(
        COVERAGE_MAIN_REPLICATES_PER_SIZE
    ):

        matched_group_id = (
            f"COVB_M{n_items}_"
            f"R{replicate:03d}"
        )


        if matched_group_id in completed_group_ids:

            continue


        matched_seed = (
            83_000_000
            +
            100_000
            *
            n_items
            +
            COVERAGE_MAIN_BASE_SEED
            +
            replicate
        )


        matched_instances = (
            generate_matched_coverage_instances(
                n_items=
                    n_items,

                n_features=
                    COVERAGE_MAIN_FEATURES,

                seed=
                    matched_seed,

                coverage_density=
                    COVERAGE_MAIN_DENSITY,

                regime_strength=
                    COVERAGE_MAIN_REGIME_STRENGTH,
            )
        )


        reference_data = (
            matched_instances[
                "independent"
            ]
        )


        shared_cache = (
            precompute_shared_coverage_utility_cache(
                all_items=
                    reference_data[
                        "items"
                    ],

                feature_weights=
                    reference_data[
                        "feature_weights"
                    ],
            )
        )


        # ------------------------------------------
        # Matched-generator audit
        # ------------------------------------------

        reference_feature_weights = np.asarray(
            reference_data[
                "feature_weights"
            ],
            dtype=float,
        )


        reference_coverage_matrix = np.asarray(
            reference_data[
                "coverage_matrix"
            ],
            dtype=float,
        )


        reference_privacy = np.sort(
            np.asarray(
                [
                    item.privacy
                    for item in reference_data[
                        "items"
                    ]
                ],
                dtype=float,
            )
        )


        reference_cognitive = np.sort(
            np.asarray(
                [
                    item.cognitive_cost
                    for item in reference_data[
                        "items"
                    ]
                ],
                dtype=float,
            )
        )


        maximum_feature_weight_difference = 0.0
        maximum_coverage_matrix_difference = 0.0
        maximum_sorted_privacy_difference = 0.0
        maximum_sorted_cognitive_difference = 0.0


        for regime in COVERAGE_MAIN_REGIMES:

            regime_data = (
                matched_instances[
                    regime
                ]
            )


            maximum_feature_weight_difference = max(
                maximum_feature_weight_difference,

                float(
                    np.max(
                        np.abs(
                            np.asarray(
                                regime_data[
                                    "feature_weights"
                                ],
                                dtype=float,
                            )
                            -
                            reference_feature_weights
                        )
                    )
                ),
            )


            maximum_coverage_matrix_difference = max(
                maximum_coverage_matrix_difference,

                float(
                    np.max(
                        np.abs(
                            np.asarray(
                                regime_data[
                                    "coverage_matrix"
                                ],
                                dtype=float,
                            )
                            -
                            reference_coverage_matrix
                        )
                    )
                ),
            )


            regime_privacy = np.sort(
                np.asarray(
                    [
                        item.privacy
                        for item in regime_data[
                            "items"
                        ]
                    ],
                    dtype=float,
                )
            )


            regime_cognitive = np.sort(
                np.asarray(
                    [
                        item.cognitive_cost
                        for item in regime_data[
                            "items"
                        ]
                    ],
                    dtype=float,
                )
            )


            maximum_sorted_privacy_difference = max(
                maximum_sorted_privacy_difference,

                float(
                    np.max(
                        np.abs(
                            regime_privacy
                            -
                            reference_privacy
                        )
                    )
                ),
            )


            maximum_sorted_cognitive_difference = max(
                maximum_sorted_cognitive_difference,

                float(
                    np.max(
                        np.abs(
                            regime_cognitive
                            -
                            reference_cognitive
                        )
                    )
                ),
            )


        new_audit_rows.append(
            {
                "benchmark_version":
                    COVERAGE_MAIN_BENCHMARK_VERSION,

                "matched_group_id":
                    matched_group_id,

                "n_items":
                    n_items,

                "replicate":
                    replicate,

                "matched_seed":
                    matched_seed,

                "number_of_subsets":
                    shared_cache[
                        "number_of_subsets"
                    ],

                "full_set_utility":
                    shared_cache[
                        "full_set_utility"
                    ],

                "shared_utility_precomputation_seconds":
                    shared_cache[
                        "runtime_seconds"
                    ],

                "maximum_feature_weight_difference":
                    maximum_feature_weight_difference,

                "maximum_coverage_matrix_difference":
                    maximum_coverage_matrix_difference,

                "maximum_sorted_privacy_difference":
                    maximum_sorted_privacy_difference,

                "maximum_sorted_cognitive_difference":
                    maximum_sorted_cognitive_difference,
            }
        )


        for regime in COVERAGE_MAIN_REGIMES:

            regime_data = (
                matched_instances[
                    regime
                ]
            )


            all_items = (
                regime_data[
                    "items"
                ]
            )


            burden_cache = (
                precompute_coverage_burden_cache(
                    all_items=
                        all_items,

                    shared_cache=
                        shared_cache,
                )
            )


            for tau in COVERAGE_MAIN_THRESHOLDS:

                for (
                    weight_profile,
                    weights,
                ) in (
                    COVERAGE_MAIN_WEIGHT_PROFILES.items()
                ):

                    exact_result = (
                        solve_cached_exact_coverage(
                            shared_cache=
                                shared_cache,

                            burden_cache=
                                burden_cache,

                            tau=
                                tau,

                            weights=
                                weights,
                        )
                    )


                    greedy_result = (
                        solve_dynamic_greedy_coverage(
                            all_items=
                                all_items,

                            shared_cache=
                                shared_cache,

                            burden_cache=
                                burden_cache,

                            tau=
                                tau,

                            weights=
                                weights,
                        )
                    )


                    if (
                        exact_result[
                            "status"
                        ]
                        ==
                        "OPTIMAL"
                        and
                        greedy_result[
                            "status"
                        ]
                        ==
                        "FEASIBLE"
                    ):

                        optimality_gap = (
                            greedy_result[
                                "disclosure_cost"
                            ]
                            -
                            exact_result[
                                "disclosure_cost"
                            ]
                        )


                        if (
                            abs(
                                optimality_gap
                            )
                            <=
                            COVERAGE_MAIN_TOLERANCE
                        ):

                            optimality_gap = 0.0


                        optimality_gap_percent = (
                            100.0
                            *
                            optimality_gap
                            /
                            exact_result[
                                "disclosure_cost"
                            ]
                        )


                        exact_ids = (
                            exact_result[
                                "selected_ids"
                            ]
                        )


                        greedy_ids = (
                            greedy_result[
                                "selected_ids"
                            ]
                        )


                        same_selected_subset = (
                            set(
                                exact_ids
                            )
                            ==
                            set(
                                greedy_ids
                            )
                        )


                        jaccard_similarity = (
                            coverage_jaccard(
                                exact_ids,
                                greedy_ids,
                            )
                        )


                        selected_count_difference = (
                            greedy_result[
                                "selected_count"
                            ]
                            -
                            exact_result[
                                "selected_count"
                            ]
                        )


                    else:

                        optimality_gap = np.nan
                        optimality_gap_percent = np.nan
                        same_selected_subset = np.nan
                        jaccard_similarity = np.nan
                        selected_count_difference = np.nan


                    new_result_rows.append(
                        {
                            "benchmark_version":
                                COVERAGE_MAIN_BENCHMARK_VERSION,

                            "utility_family":
                                (
                                    "pool_normalized_"
                                    "weighted_maximum_coverage"
                                ),

                            "matched_group_id":
                                matched_group_id,

                            "instance_id":
                                (
                                    f"{matched_group_id}_"
                                    f"{regime}"
                                ),

                            "n_items":
                                n_items,

                            "replicate":
                                replicate,

                            "matched_seed":
                                matched_seed,

                            "regime":
                                regime,

                            "tau":
                                tau,

                            "weight_profile":
                                weight_profile,

                            "lambda_privacy":
                                weights[
                                    "lambda_privacy"
                                ],

                            "lambda_cognitive":
                                weights[
                                    "lambda_cognitive"
                                ],

                            "lambda_cardinality":
                                weights[
                                    "lambda_cardinality"
                                ],

                            "full_set_utility":
                                shared_cache[
                                    "full_set_utility"
                                ],

                            "number_of_subsets":
                                shared_cache[
                                    "number_of_subsets"
                                ],

                            "shared_utility_precomputation_seconds":
                                shared_cache[
                                    "runtime_seconds"
                                ],

                            "burden_precomputation_seconds":
                                burden_cache[
                                    "runtime_seconds"
                                ],

                            "exact_status":
                                exact_result[
                                    "status"
                                ],

                            "exact_selected_count":
                                exact_result.get(
                                    "selected_count",
                                    np.nan,
                                ),

                            "exact_selected_items":
                                "|".join(
                                    exact_result.get(
                                        "selected_ids",
                                        (),
                                    )
                                ),

                            "exact_utility":
                                exact_result.get(
                                    "utility",
                                    np.nan,
                                ),

                            "exact_privacy":
                                exact_result.get(
                                    "privacy",
                                    np.nan,
                                ),

                            "exact_normalized_cognitive_cost":
                                exact_result.get(
                                    "normalized_cognitive_cost",
                                    np.nan,
                                ),

                            "exact_cardinality_fraction":
                                exact_result.get(
                                    "cardinality_fraction",
                                    np.nan,
                                ),

                            "exact_disclosure_cost":
                                exact_result.get(
                                    "disclosure_cost",
                                    np.nan,
                                ),

                            "exact_runtime_seconds":
                                exact_result[
                                    "runtime_seconds"
                                ],

                            "greedy_status":
                                greedy_result[
                                    "status"
                                ],

                            "greedy_selected_count":
                                greedy_result.get(
                                    "selected_count",
                                    np.nan,
                                ),

                            "greedy_selected_items":
                                "|".join(
                                    greedy_result.get(
                                        "selected_ids",
                                        (),
                                    )
                                ),

                            "greedy_utility":
                                greedy_result.get(
                                    "utility",
                                    np.nan,
                                ),

                            "greedy_privacy":
                                greedy_result.get(
                                    "privacy",
                                    np.nan,
                                ),

                            "greedy_normalized_cognitive_cost":
                                greedy_result.get(
                                    "normalized_cognitive_cost",
                                    np.nan,
                                ),

                            "greedy_cardinality_fraction":
                                greedy_result.get(
                                    "cardinality_fraction",
                                    np.nan,
                                ),

                            "greedy_disclosure_cost":
                                greedy_result.get(
                                    "disclosure_cost",
                                    np.nan,
                                ),

                            "greedy_runtime_seconds":
                                greedy_result[
                                    "runtime_seconds"
                                ],

                            "optimality_gap":
                                optimality_gap,

                            "optimality_gap_percent":
                                optimality_gap_percent,

                            "same_selected_subset":
                                same_selected_subset,

                            "jaccard_similarity":
                                jaccard_similarity,

                            "selected_count_difference":
                                selected_count_difference,
                        }
                    )


        if (
            (
                replicate
                +
                1
            )
            %
            10
            ==
            0
        ):

            print(
                "Completed coverage benchmark:",
                f"M={n_items}",
                f"replicate={replicate + 1}/"
                f"{COVERAGE_MAIN_REPLICATES_PER_SIZE}",
            )


    # ----------------------------------------------
    # Size-level checkpoint
    # ----------------------------------------------

    new_results_df = pd.DataFrame(
        new_result_rows
    )


    checkpoint_results_df = pd.concat(
        [
            existing_results_df,
            new_results_df,
        ],
        ignore_index=True,
    )


    checkpoint_results_df = (
        checkpoint_results_df
        .drop_duplicates(
            subset=[
                "matched_group_id",
                "regime",
                "tau",
                "weight_profile",
            ],
            keep="last",
        )
        .sort_values(
            [
                "n_items",
                "replicate",
                "regime",
                "tau",
                "weight_profile",
            ]
        )
        .reset_index(
            drop=True
        )
    )


    checkpoint_results_df.to_csv(
        COVERAGE_MAIN_RESULTS_PATH,
        index=False,
    )


    new_audit_df = pd.DataFrame(
        new_audit_rows
    )


    checkpoint_audit_df = pd.concat(
        [
            existing_audit_df,
            new_audit_df,
        ],
        ignore_index=True,
    )


    checkpoint_audit_df = (
        checkpoint_audit_df
        .drop_duplicates(
            subset=[
                "matched_group_id",
            ],
            keep="last",
        )
        .sort_values(
            [
                "n_items",
                "replicate",
            ]
        )
        .reset_index(
            drop=True
        )
    )


    checkpoint_audit_df.to_csv(
        COVERAGE_MAIN_GROUP_AUDIT_PATH,
        index=False,
    )


    print(
        "Checkpoint saved after candidate size:",
        n_items,
        "| current rows:",
        len(
            checkpoint_results_df
        ),
    )


# --------------------------------------------------
# Final benchmark data
# --------------------------------------------------

coverage_main_df = pd.read_csv(
    COVERAGE_MAIN_RESULTS_PATH,
    low_memory=False,
)


coverage_group_audit_df = pd.read_csv(
    COVERAGE_MAIN_GROUP_AUDIT_PATH,
    low_memory=False,
)


for column_name in [
    "n_items",
    "tau",
    "exact_selected_count",
    "greedy_selected_count",
    "exact_utility",
    "greedy_utility",
    "exact_disclosure_cost",
    "greedy_disclosure_cost",
    "optimality_gap_percent",
    "jaccard_similarity",
    "selected_count_difference",
]:

    coverage_main_df[
        column_name
    ] = pd.to_numeric(
        coverage_main_df[
            column_name
        ],
        errors="coerce",
    )


# --------------------------------------------------
# Reporting indicators
# --------------------------------------------------

coverage_main_df[
    "exact_cost_recovery_numeric"
] = (
    coverage_main_df[
        "optimality_gap_percent"
    ]
    .abs()
    <=
    COVERAGE_EXACT_RECOVERY_TOLERANCE_PERCENT
).astype(int)


coverage_main_df[
    "same_selected_subset_numeric"
] = (
    coverage_main_df[
        "same_selected_subset"
    ]
    .astype(str)
    .str.lower()
    .eq(
        "true"
    )
    .astype(int)
)


coverage_main_df[
    "large_gap_numeric"
] = (
    coverage_main_df[
        "optimality_gap_percent"
    ]
    >
    10.0
).astype(int)


coverage_main_df[
    "more_items_numeric"
] = (
    coverage_main_df[
        "selected_count_difference"
    ]
    >
    0
).astype(int)


coverage_main_df[
    "within_1_percent_numeric"
] = (
    coverage_main_df[
        "optimality_gap_percent"
    ]
    <=
    1.0
).astype(int)


coverage_main_df[
    "within_5_percent_numeric"
] = (
    coverage_main_df[
        "optimality_gap_percent"
    ]
    <=
    5.0
).astype(int)


coverage_main_df[
    "within_10_percent_numeric"
] = (
    coverage_main_df[
        "optimality_gap_percent"
    ]
    <=
    10.0
).astype(int)


# --------------------------------------------------
# Integrity audit
# --------------------------------------------------

duplicate_conditions = int(
    coverage_main_df
    .duplicated(
        subset=[
            "matched_group_id",
            "regime",
            "tau",
            "weight_profile",
        ],
        keep=False,
    )
    .sum()
)


negative_gap_rows = int(
    (
        coverage_main_df[
            "optimality_gap_percent"
        ]
        <
        -1e-8
    ).sum()
)


exact_threshold_failures = int(
    (
        (
            coverage_main_df[
                "exact_status"
            ]
            ==
            "OPTIMAL"
        )
        &
        (
            coverage_main_df[
                "exact_utility"
            ]
            +
            COVERAGE_MAIN_TOLERANCE
            <
            coverage_main_df[
                "tau"
            ]
        )
    ).sum()
)


greedy_threshold_failures = int(
    (
        (
            coverage_main_df[
                "greedy_status"
            ]
            ==
            "FEASIBLE"
        )
        &
        (
            coverage_main_df[
                "greedy_utility"
            ]
            +
            COVERAGE_MAIN_TOLERANCE
            <
            coverage_main_df[
                "tau"
            ]
        )
    ).sum()
)


status_disagreements = int(
    (
        (
            coverage_main_df[
                "exact_status"
            ]
            ==
            "OPTIMAL"
        )
        &
        (
            coverage_main_df[
                "greedy_status"
            ]
            !=
            "FEASIBLE"
        )
    ).sum()
)


group_condition_counts = (
    coverage_main_df
    .groupby(
        "matched_group_id"
    )
    .size()
)


coverage_integrity_audit_df = pd.DataFrame(
    [
        {
            "benchmark_rows":
                len(
                    coverage_main_df
                ),

            "expected_benchmark_rows":
                EXPECTED_BENCHMARK_ROWS,

            "matched_groups":
                coverage_main_df[
                    "matched_group_id"
                ].nunique(),

            "expected_matched_groups":
                EXPECTED_MATCHED_GROUPS,

            "minimum_conditions_per_group":
                int(
                    group_condition_counts.min()
                ),

            "maximum_conditions_per_group":
                int(
                    group_condition_counts.max()
                ),

            "exact_optimal_rows":
                int(
                    (
                        coverage_main_df[
                            "exact_status"
                        ]
                        ==
                        "OPTIMAL"
                    ).sum()
                ),

            "greedy_feasible_rows":
                int(
                    (
                        coverage_main_df[
                            "greedy_status"
                        ]
                        ==
                        "FEASIBLE"
                    ).sum()
                ),

            "duplicate_conditions":
                duplicate_conditions,

            "negative_gap_rows":
                negative_gap_rows,

            "exact_threshold_failures":
                exact_threshold_failures,

            "greedy_threshold_failures":
                greedy_threshold_failures,

            "status_disagreements":
                status_disagreements,

            "maximum_full_set_deviation_from_one":
                (
                    coverage_group_audit_df[
                        "full_set_utility"
                    ]
                    -
                    1.0
                )
                .abs()
                .max(),

            "maximum_feature_weight_difference":
                coverage_group_audit_df[
                    "maximum_feature_weight_difference"
                ].max(),

            "maximum_coverage_matrix_difference":
                coverage_group_audit_df[
                    "maximum_coverage_matrix_difference"
                ].max(),

            "maximum_sorted_privacy_difference":
                coverage_group_audit_df[
                    "maximum_sorted_privacy_difference"
                ].max(),

            "maximum_sorted_cognitive_difference":
                coverage_group_audit_df[
                    "maximum_sorted_cognitive_difference"
                ].max(),

            "minimum_optimality_gap_percent":
                coverage_main_df[
                    "optimality_gap_percent"
                ].min(),

            "maximum_optimality_gap_percent":
                coverage_main_df[
                    "optimality_gap_percent"
                ].max(),

            "elapsed_seconds_this_run":
                (
                    time.perf_counter()
                    -
                    benchmark_start_time
                ),
        }
    ]
)


assert len(
    coverage_main_df
) == EXPECTED_BENCHMARK_ROWS


assert (
    coverage_main_df[
        "matched_group_id"
    ].nunique()
    ==
    EXPECTED_MATCHED_GROUPS
)


assert (
    group_condition_counts
    ==
    EXPECTED_CONDITIONS_PER_MATCHED_GROUP
).all()


assert duplicate_conditions == 0
assert negative_gap_rows == 0
assert exact_threshold_failures == 0
assert greedy_threshold_failures == 0
assert status_disagreements == 0


assert (
    coverage_main_df[
        "exact_status"
    ]
    ==
    "OPTIMAL"
).all()


assert (
    coverage_main_df[
        "greedy_status"
    ]
    ==
    "FEASIBLE"
).all()


assert (
    coverage_integrity_audit_df.loc[
        0,
        "maximum_full_set_deviation_from_one",
    ]
    <=
    COVERAGE_MAIN_TOLERANCE
)


assert (
    coverage_integrity_audit_df.loc[
        0,
        "maximum_feature_weight_difference",
    ]
    <=
    COVERAGE_MAIN_TOLERANCE
)


assert (
    coverage_integrity_audit_df.loc[
        0,
        "maximum_coverage_matrix_difference",
    ]
    <=
    COVERAGE_MAIN_TOLERANCE
)


assert (
    coverage_integrity_audit_df.loc[
        0,
        "maximum_sorted_privacy_difference",
    ]
    <=
    COVERAGE_MAIN_TOLERANCE
)


assert (
    coverage_integrity_audit_df.loc[
        0,
        "maximum_sorted_cognitive_difference",
    ]
    <=
    COVERAGE_MAIN_TOLERANCE
)


# --------------------------------------------------
# Summary helper
# --------------------------------------------------

def summarize_coverage_algorithm_quality(
    data_frame,
):
    """
    Descriptive Exact-relative Dynamic Greedy quality.
    """

    gap_values = (
        data_frame[
            "optimality_gap_percent"
        ]
        .to_numpy(
            dtype=float
        )
    )


    return pd.Series(
        {
            "comparable_rows":
                len(
                    data_frame
                ),

            "mean_optimality_gap_percent":
                gap_values.mean(),

            "median_optimality_gap_percent":
                np.median(
                    gap_values
                ),

            "p90_optimality_gap_percent":
                np.quantile(
                    gap_values,
                    0.90,
                ),

            "p95_optimality_gap_percent":
                np.quantile(
                    gap_values,
                    0.95,
                ),

            "maximum_optimality_gap_percent":
                gap_values.max(),

            "exact_cost_recovery_percent":
                100.0
                *
                data_frame[
                    "exact_cost_recovery_numeric"
                ].mean(),

            "exact_subset_recovery_percent":
                100.0
                *
                data_frame[
                    "same_selected_subset_numeric"
                ].mean(),

            "mean_jaccard_similarity":
                data_frame[
                    "jaccard_similarity"
                ].mean(),

            "large_gap_percent":
                100.0
                *
                data_frame[
                    "large_gap_numeric"
                ].mean(),

            "more_items_percent":
                100.0
                *
                data_frame[
                    "more_items_numeric"
                ].mean(),

            "mean_selected_count_difference":
                data_frame[
                    "selected_count_difference"
                ].mean(),

            "within_1_percent":
                100.0
                *
                data_frame[
                    "within_1_percent_numeric"
                ].mean(),

            "within_5_percent":
                100.0
                *
                data_frame[
                    "within_5_percent_numeric"
                ].mean(),

            "within_10_percent":
                100.0
                *
                data_frame[
                    "within_10_percent_numeric"
                ].mean(),

            "mean_exact_selected_count":
                data_frame[
                    "exact_selected_count"
                ].mean(),

            "mean_greedy_selected_count":
                data_frame[
                    "greedy_selected_count"
                ].mean(),
        }
    )


# --------------------------------------------------
# Overall and regime summaries
# --------------------------------------------------

coverage_overall_summary_df = (
    summarize_coverage_algorithm_quality(
        coverage_main_df
    )
    .to_frame()
    .T
)


coverage_overall_summary_df.insert(
    0,
    "utility_family",
    (
        "pool_normalized_"
        "weighted_maximum_coverage"
    ),
)


coverage_regime_summary_df = (
    coverage_main_df
    .groupby(
        "regime",
        sort=False,
    )
    .apply(
        summarize_coverage_algorithm_quality,
        include_groups=False,
    )
    .reset_index()
)


# --------------------------------------------------
# Runtime summary by candidate-set size
# --------------------------------------------------

group_runtime_df = (
    coverage_group_audit_df[
        [
            "matched_group_id",
            "n_items",
            "shared_utility_precomputation_seconds",
        ]
    ]
    .drop_duplicates(
        "matched_group_id"
    )
)


burden_runtime_df = (
    coverage_main_df[
        [
            "matched_group_id",
            "n_items",
            "regime",
            "burden_precomputation_seconds",
        ]
    ]
    .drop_duplicates(
        [
            "matched_group_id",
            "regime",
        ]
    )
)


utility_runtime_summary_df = (
    group_runtime_df
    .groupby(
        "n_items",
        as_index=False,
    )
    .agg(
        matched_groups=(
            "matched_group_id",
            "nunique",
        ),

        mean_shared_utility_precomputation_seconds=(
            "shared_utility_precomputation_seconds",
            "mean",
        ),

        median_shared_utility_precomputation_seconds=(
            "shared_utility_precomputation_seconds",
            "median",
        ),
    )
)


burden_runtime_summary_df = (
    burden_runtime_df
    .groupby(
        "n_items",
        as_index=False,
    )
    .agg(
        mean_burden_precomputation_seconds=(
            "burden_precomputation_seconds",
            "mean",
        ),

        median_burden_precomputation_seconds=(
            "burden_precomputation_seconds",
            "median",
        ),
    )
)


solver_runtime_summary_df = (
    coverage_main_df
    .groupby(
        "n_items",
        as_index=False,
    )
    .agg(
        conditions=(
            "optimality_gap_percent",
            "size",
        ),

        mean_exact_solve_milliseconds=(
            "exact_runtime_seconds",
            lambda values:
                1000.0
                *
                values.mean(),
        ),

        median_exact_solve_milliseconds=(
            "exact_runtime_seconds",
            lambda values:
                1000.0
                *
                values.median(),
        ),

        mean_greedy_solve_milliseconds=(
            "greedy_runtime_seconds",
            lambda values:
                1000.0
                *
                values.mean(),
        ),

        median_greedy_solve_milliseconds=(
            "greedy_runtime_seconds",
            lambda values:
                1000.0
                *
                values.median(),
        ),
    )
)


coverage_runtime_summary_df = (
    utility_runtime_summary_df
    .merge(
        burden_runtime_summary_df,
        on="n_items",
        how="left",
    )
    .merge(
        solver_runtime_summary_df,
        on="n_items",
        how="left",
    )
)


# --------------------------------------------------
# Metadata
# --------------------------------------------------

coverage_metadata_df = pd.DataFrame(
    [
        {
            "benchmark_version":
                COVERAGE_MAIN_BENCHMARK_VERSION,

            "utility_family":
                (
                    "pool-normalized weighted "
                    "maximum coverage"
                ),

            "base_seed":
                COVERAGE_MAIN_BASE_SEED,

            "candidate_sizes":
                repr(
                    COVERAGE_MAIN_SIZES
                ),

            "replicates_per_size":
                COVERAGE_MAIN_REPLICATES_PER_SIZE,

            "total_matched_groups":
                EXPECTED_MATCHED_GROUPS,

            "regimes":
                repr(
                    COVERAGE_MAIN_REGIMES
                ),

            "thresholds":
                repr(
                    COVERAGE_MAIN_THRESHOLDS
                ),

            "weight_profiles":
                repr(
                    list(
                        COVERAGE_MAIN_WEIGHT_PROFILES.keys()
                    )
                ),

            "total_conditions":
                EXPECTED_BENCHMARK_ROWS,

            "n_features":
                COVERAGE_MAIN_FEATURES,

            "coverage_density":
                COVERAGE_MAIN_DENSITY,

            "regime_strength":
                COVERAGE_MAIN_REGIME_STRENGTH,

            "exact_method":
                (
                    "complete subset enumeration "
                    "with cached utility and burden arrays"
                ),

            "heuristic":
                (
                    "dynamic marginal utility divided "
                    "by dynamic marginal disclosure cost"
                ),

            "threshold_interpretation":
                (
                    "fraction of available modeled "
                    "decision-support utility"
                ),

            "approximation_ratio_claimed":
                False,
        }
    ]
)


# --------------------------------------------------
# Save final tables
# --------------------------------------------------

coverage_overall_summary_df.to_csv(
    COVERAGE_MAIN_OVERALL_SUMMARY_PATH,
    index=False,
)


coverage_regime_summary_df.to_csv(
    COVERAGE_MAIN_REGIME_SUMMARY_PATH,
    index=False,
)


coverage_runtime_summary_df.to_csv(
    COVERAGE_MAIN_RUNTIME_SUMMARY_PATH,
    index=False,
)


coverage_metadata_df.to_csv(
    COVERAGE_MAIN_METADATA_PATH,
    index=False,
)


# --------------------------------------------------
# Printed report
# --------------------------------------------------

print(
    "\n======================================"
)

print(
    "COVERAGE MAIN BENCHMARK INTEGRITY AUDIT"
)

print(
    "======================================"
)


print(
    coverage_integrity_audit_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.12f}",
    )
)


print(
    "\n======================================"
)

print(
    "COVERAGE EXACT-VERSUS-GREEDY OVERALL RESULTS"
)

print(
    "======================================"
)


print(
    coverage_overall_summary_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.4f}",
    )
)


print(
    "\n======================================"
)

print(
    "COVERAGE EXACT-VERSUS-GREEDY RESULTS BY REGIME"
)

print(
    "======================================"
)


print(
    coverage_regime_summary_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.4f}",
    )
)


print(
    "\n======================================"
)

print(
    "COVERAGE BENCHMARK RUNTIME BY CANDIDATE SIZE"
)

print(
    "======================================"
)


print(
    coverage_runtime_summary_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.6f}",
    )
)


print(
    "\nMain benchmark saved to:"
)

print(
    COVERAGE_MAIN_RESULTS_PATH
)


print(
    "\nGroup-level integrity audit saved to:"
)

print(
    COVERAGE_MAIN_GROUP_AUDIT_PATH
)


print(
    "\nOverall summary saved to:"
)

print(
    COVERAGE_MAIN_OVERALL_SUMMARY_PATH
)


print(
    "\nRegime summary saved to:"
)

print(
    COVERAGE_MAIN_REGIME_SUMMARY_PATH
)


print(
    "\nRuntime summary saved to:"
)

print(
    COVERAGE_MAIN_RUNTIME_SUMMARY_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Step 130 — Coverage benchmark paired structural-
# regime contrasts with matched-group bootstrap
# --------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------
# Configuration
# --------------------------------------------------

COVERAGE_REGIME_BOOTSTRAP_REPLICATES = 5000

COVERAGE_REGIME_BOOTSTRAP_SEED = (
    globals().get(
        "GLOBAL_SEED",
        2026,
    )
    + 130
)


if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


COVERAGE_MAIN_RESULTS_PATH = (
    RESULTS_DIR
    / (
        "coverage_exact_vs_greedy_"
        "main_benchmark_v1_pool_normalized.csv"
    )
)


COVERAGE_REGIME_DESCRIPTIVE_PATH = (
    TABLES_DIR
    / "coverage_exact_vs_greedy_regime_descriptive_summary.csv"
)


COVERAGE_REGIME_CONTRAST_PATH = (
    TABLES_DIR
    / "coverage_exact_vs_greedy_paired_regime_contrasts.csv"
)


COVERAGE_REGIME_REPLICATES_PATH = (
    RESULTS_DIR
    / "coverage_exact_vs_greedy_paired_regime_bootstrap_replicates.csv"
)


COVERAGE_REGIME_AUDIT_PATH = (
    RESULTS_DIR
    / "coverage_exact_vs_greedy_regime_contrast_audit.csv"
)


REGIME_ORDER = [
    "independent",
    "utility_burden_alignment",
    "utility_burden_conflict",
]


CANDIDATE_SIZE_ORDER = [
    8,
    10,
    12,
    14,
    16,
]


REGIME_CONTRASTS = [
    (
        "utility_burden_alignment",
        "independent",
    ),

    (
        "utility_burden_conflict",
        "independent",
    ),

    (
        "utility_burden_conflict",
        "utility_burden_alignment",
    ),
]


METRIC_NAMES = [
    "mean_optimality_gap_percent",
    "exact_cost_recovery_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "more_items_percent",
    "mean_selected_count_difference",
    "within_1_percent",
    "within_5_percent",
    "within_10_percent",
]


AGGREGATE_COLUMNS = [
    "comparable_rows",
    "gap_sum",
    "exact_cost_recovery_sum",
    "jaccard_sum",
    "large_gap_sum",
    "more_items_sum",
    "selected_count_difference_sum",
    "within_1_percent_sum",
    "within_5_percent_sum",
    "within_10_percent_sum",
]


EXACT_COST_RECOVERY_TOLERANCE_PERCENT = 1e-10


# --------------------------------------------------
# Load coverage benchmark
# --------------------------------------------------

coverage_df = pd.read_csv(
    COVERAGE_MAIN_RESULTS_PATH,
    low_memory=False,
)


required_columns = [
    "matched_group_id",
    "n_items",
    "regime",
    "tau",
    "weight_profile",
    "exact_status",
    "greedy_status",
    "optimality_gap_percent",
    "jaccard_similarity",
    "selected_count_difference",
]


missing_columns = [
    column_name
    for column_name in required_columns
    if column_name not in coverage_df.columns
]


if missing_columns:

    raise KeyError(
        "Missing required columns: "
        + ", ".join(
            missing_columns
        )
    )


for column_name in [
    "n_items",
    "tau",
    "optimality_gap_percent",
    "jaccard_similarity",
    "selected_count_difference",
]:

    coverage_df[
        column_name
    ] = pd.to_numeric(
        coverage_df[
            column_name
        ],
        errors="coerce",
    )


# --------------------------------------------------
# Benchmark design validation
# --------------------------------------------------

assert len(
    coverage_df
) == 24000


assert (
    coverage_df[
        "matched_group_id"
    ].nunique()
    == 500
)


assert set(
    coverage_df[
        "regime"
    ].dropna().unique()
) == set(
    REGIME_ORDER
)


assert (
    sorted(
        coverage_df[
            "n_items"
        ]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
    ==
    CANDIDATE_SIZE_ORDER
)


assert (
    coverage_df[
        "exact_status"
    ]
    ==
    "OPTIMAL"
).all()


assert (
    coverage_df[
        "greedy_status"
    ]
    ==
    "FEASIBLE"
).all()


# --------------------------------------------------
# Reporting indicators
# --------------------------------------------------

coverage_df[
    "exact_cost_recovery_numeric"
] = (
    coverage_df[
        "optimality_gap_percent"
    ]
    .abs()
    <=
    EXACT_COST_RECOVERY_TOLERANCE_PERCENT
).astype(int)


coverage_df[
    "large_gap_numeric"
] = (
    coverage_df[
        "optimality_gap_percent"
    ]
    >
    10.0
).astype(int)


coverage_df[
    "more_items_numeric"
] = (
    coverage_df[
        "selected_count_difference"
    ]
    >
    0
).astype(int)


coverage_df[
    "within_1_percent_numeric"
] = (
    coverage_df[
        "optimality_gap_percent"
    ]
    <=
    1.0
).astype(int)


coverage_df[
    "within_5_percent_numeric"
] = (
    coverage_df[
        "optimality_gap_percent"
    ]
    <=
    5.0
).astype(int)


coverage_df[
    "within_10_percent_numeric"
] = (
    coverage_df[
        "optimality_gap_percent"
    ]
    <=
    10.0
).astype(int)


# --------------------------------------------------
# Descriptive regime summary
# --------------------------------------------------

descriptive_rows = []


for regime in REGIME_ORDER:

    current_df = (
        coverage_df[
            coverage_df[
                "regime"
            ]
            ==
            regime
        ]
    )


    gap_values = (
        current_df[
            "optimality_gap_percent"
        ]
        .to_numpy(
            dtype=float
        )
    )


    descriptive_rows.append(
        {
            "regime":
                regime,

            "comparable_rows":
                len(
                    current_df
                ),

            "matched_groups":
                current_df[
                    "matched_group_id"
                ].nunique(),

            "mean_optimality_gap_percent":
                gap_values.mean(),

            "median_optimality_gap_percent":
                np.median(
                    gap_values
                ),

            "p90_optimality_gap_percent":
                np.quantile(
                    gap_values,
                    0.90,
                ),

            "p95_optimality_gap_percent":
                np.quantile(
                    gap_values,
                    0.95,
                ),

            "maximum_optimality_gap_percent":
                gap_values.max(),

            "exact_cost_recovery_percent":
                100.0
                *
                current_df[
                    "exact_cost_recovery_numeric"
                ].mean(),

            "mean_jaccard_similarity":
                current_df[
                    "jaccard_similarity"
                ].mean(),

            "large_gap_percent":
                100.0
                *
                current_df[
                    "large_gap_numeric"
                ].mean(),

            "more_items_percent":
                100.0
                *
                current_df[
                    "more_items_numeric"
                ].mean(),

            "mean_selected_count_difference":
                current_df[
                    "selected_count_difference"
                ].mean(),

            "within_1_percent":
                100.0
                *
                current_df[
                    "within_1_percent_numeric"
                ].mean(),

            "within_5_percent":
                100.0
                *
                current_df[
                    "within_5_percent_numeric"
                ].mean(),

            "within_10_percent":
                100.0
                *
                current_df[
                    "within_10_percent_numeric"
                ].mean(),
        }
    )


coverage_regime_descriptive_df = pd.DataFrame(
    descriptive_rows
)


coverage_regime_descriptive_df.to_csv(
    COVERAGE_REGIME_DESCRIPTIVE_PATH,
    index=False,
)


# --------------------------------------------------
# Matched-group sufficient statistics
# --------------------------------------------------

cluster_df = (
    coverage_df
    .groupby(
        [
            "n_items",
            "matched_group_id",
            "regime",
        ],
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "size",
        ),

        gap_sum=(
            "optimality_gap_percent",
            "sum",
        ),

        exact_cost_recovery_sum=(
            "exact_cost_recovery_numeric",
            "sum",
        ),

        jaccard_sum=(
            "jaccard_similarity",
            "sum",
        ),

        large_gap_sum=(
            "large_gap_numeric",
            "sum",
        ),

        more_items_sum=(
            "more_items_numeric",
            "sum",
        ),

        selected_count_difference_sum=(
            "selected_count_difference",
            "sum",
        ),

        within_1_percent_sum=(
            "within_1_percent_numeric",
            "sum",
        ),

        within_5_percent_sum=(
            "within_5_percent_numeric",
            "sum",
        ),

        within_10_percent_sum=(
            "within_10_percent_numeric",
            "sum",
        ),
    )
)


# Each matched group contains:
# 4 thresholds × 4 weight profiles = 16 rows/regime.
assert (
    cluster_df[
        "comparable_rows"
    ]
    ==
    16
).all()


# --------------------------------------------------
# Metric conversion
# --------------------------------------------------

def calculate_regime_metrics(
    totals,
):
    """
    Convert sufficient-statistic totals into
    Exact-relative Dynamic Greedy metrics.
    """

    comparable_rows = totals[0]


    if comparable_rows <= 0:

        return np.full(
            len(
                METRIC_NAMES
            ),
            np.nan,
        )


    return np.array(
        [
            totals[1]
            /
            comparable_rows,

            100.0
            *
            totals[2]
            /
            comparable_rows,

            totals[3]
            /
            comparable_rows,

            100.0
            *
            totals[4]
            /
            comparable_rows,

            100.0
            *
            totals[5]
            /
            comparable_rows,

            totals[6]
            /
            comparable_rows,

            100.0
            *
            totals[7]
            /
            comparable_rows,

            100.0
            *
            totals[8]
            /
            comparable_rows,

            100.0
            *
            totals[9]
            /
            comparable_rows,
        ],
        dtype=float,
    )


# --------------------------------------------------
# Construct aligned arrays by size and regime
# --------------------------------------------------

arrays_by_size_regime = {}

support_rows = []


for n_items in CANDIDATE_SIZE_ORDER:

    arrays_by_size_regime[
        n_items
    ] = {}


    reference_group_ids = None


    for regime in REGIME_ORDER:

        current_df = (
            cluster_df[
                (
                    cluster_df[
                        "n_items"
                    ]
                    ==
                    n_items
                )
                &
                (
                    cluster_df[
                        "regime"
                    ]
                    ==
                    regime
                )
            ]
            .sort_values(
                "matched_group_id"
            )
            .reset_index(
                drop=True
            )
        )


        current_group_ids = (
            current_df[
                "matched_group_id"
            ].tolist()
        )


        assert len(
            current_group_ids
        ) == 100


        if reference_group_ids is None:

            reference_group_ids = (
                current_group_ids
            )

        else:

            assert (
                current_group_ids
                ==
                reference_group_ids
            )


        arrays_by_size_regime[
            n_items
        ][
            regime
        ] = (
            current_df[
                AGGREGATE_COLUMNS
            ]
            .to_numpy(
                dtype=float
            )
        )


        support_rows.append(
            {
                "n_items":
                    n_items,

                "regime":
                    regime,

                "matched_groups":
                    len(
                        current_df
                    ),

                "conditions":
                    int(
                        current_df[
                            "comparable_rows"
                        ].sum()
                    ),

                "minimum_conditions_per_group":
                    int(
                        current_df[
                            "comparable_rows"
                        ].min()
                    ),

                "maximum_conditions_per_group":
                    int(
                        current_df[
                            "comparable_rows"
                        ].max()
                    ),
            }
        )


support_df = pd.DataFrame(
    support_rows
)


# --------------------------------------------------
# Observed regime metrics
# --------------------------------------------------

observed_totals = {
    regime: np.zeros(
        len(
            AGGREGATE_COLUMNS
        ),
        dtype=float,
    )
    for regime in REGIME_ORDER
}


for n_items in CANDIDATE_SIZE_ORDER:

    for regime in REGIME_ORDER:

        observed_totals[
            regime
        ] += (
            arrays_by_size_regime[
                n_items
            ][
                regime
            ]
            .sum(
                axis=0
            )
        )


observed_metrics = {
    regime:
        calculate_regime_metrics(
            observed_totals[
                regime
            ]
        )
    for regime in REGIME_ORDER
}


# --------------------------------------------------
# Shared matched-group bootstrap
#
# The same resampled group indices are used across
# all three regimes within each candidate-set size.
# --------------------------------------------------

bootstrap_rng = np.random.default_rng(
    COVERAGE_REGIME_BOOTSTRAP_SEED
)


number_of_contrasts = len(
    REGIME_CONTRASTS
)


number_of_metrics = len(
    METRIC_NAMES
)


bootstrap_differences = np.empty(
    (
        COVERAGE_REGIME_BOOTSTRAP_REPLICATES,
        number_of_contrasts,
        number_of_metrics,
    ),
    dtype=float,
)


for bootstrap_index in range(
    COVERAGE_REGIME_BOOTSTRAP_REPLICATES
):

    bootstrap_totals = {
        regime: np.zeros(
            len(
                AGGREGATE_COLUMNS
            ),
            dtype=float,
        )
        for regime in REGIME_ORDER
    }


    for n_items in CANDIDATE_SIZE_ORDER:

        number_of_groups = (
            arrays_by_size_regime[
                n_items
            ][
                REGIME_ORDER[0]
            ].shape[0]
        )


        sampled_indices = (
            bootstrap_rng.integers(
                low=0,
                high=number_of_groups,
                size=number_of_groups,
            )
        )


        for regime in REGIME_ORDER:

            bootstrap_totals[
                regime
            ] += (
                arrays_by_size_regime[
                    n_items
                ][
                    regime
                ][
                    sampled_indices
                ]
                .sum(
                    axis=0
                )
            )


    bootstrap_metrics = {
        regime:
            calculate_regime_metrics(
                bootstrap_totals[
                    regime
                ]
            )
        for regime in REGIME_ORDER
    }


    for contrast_index, (
        first_regime,
        second_regime,
    ) in enumerate(
        REGIME_CONTRASTS
    ):

        bootstrap_differences[
            bootstrap_index,
            contrast_index,
        ] = (
            bootstrap_metrics[
                first_regime
            ]
            -
            bootstrap_metrics[
                second_regime
            ]
        )


# --------------------------------------------------
# Paired contrast confidence intervals
# --------------------------------------------------

contrast_rows = []


for contrast_index, (
    first_regime,
    second_regime,
) in enumerate(
    REGIME_CONTRASTS
):

    observed_difference = (
        observed_metrics[
            first_regime
        ]
        -
        observed_metrics[
            second_regime
        ]
    )


    current_bootstrap = (
        bootstrap_differences[
            :,
            contrast_index,
            :,
        ]
    )


    standard_errors = (
        current_bootstrap.std(
            axis=0,
            ddof=1,
        )
    )


    confidence_lower = np.quantile(
        current_bootstrap,
        0.025,
        axis=0,
    )


    confidence_upper = np.quantile(
        current_bootstrap,
        0.975,
        axis=0,
    )


    for metric_index, metric_name in enumerate(
        METRIC_NAMES
    ):

        contrast_rows.append(
            {
                "first_regime":
                    first_regime,

                "second_regime":
                    second_regime,

                "contrast":
                    (
                        f"{first_regime}"
                        f" minus "
                        f"{second_regime}"
                    ),

                "metric":
                    metric_name,

                "estimated_difference":
                    observed_difference[
                        metric_index
                    ],

                "bootstrap_standard_error":
                    standard_errors[
                        metric_index
                    ],

                "ci_95_lower":
                    confidence_lower[
                        metric_index
                    ],

                "ci_95_upper":
                    confidence_upper[
                        metric_index
                    ],

                "ci_excludes_zero":
                    (
                        confidence_lower[
                            metric_index
                        ]
                        >
                        0.0
                    )
                    or
                    (
                        confidence_upper[
                            metric_index
                        ]
                        <
                        0.0
                    ),

                "matched_groups":
                    500,

                "matched_groups_per_size":
                    100,

                "candidate_size_strata":
                    len(
                        CANDIDATE_SIZE_ORDER
                    ),

                "bootstrap_replicates":
                    COVERAGE_REGIME_BOOTSTRAP_REPLICATES,

                "resampling_unit":
                    "matched_size_replicate_group",

                "resampling_design":
                    (
                        "shared_across_regimes_"
                        "stratified_by_candidate_set_size"
                    ),

                "multiple_comparison_adjustment":
                    "none",
            }
        )


coverage_regime_contrast_df = pd.DataFrame(
    contrast_rows
)


# --------------------------------------------------
# Save bootstrap replicates
# --------------------------------------------------

replicate_rows = []


for bootstrap_index in range(
    COVERAGE_REGIME_BOOTSTRAP_REPLICATES
):

    for contrast_index, (
        first_regime,
        second_regime,
    ) in enumerate(
        REGIME_CONTRASTS
    ):

        replicate_row = {
            "bootstrap_replicate":
                bootstrap_index,

            "first_regime":
                first_regime,

            "second_regime":
                second_regime,

            "contrast":
                (
                    f"{first_regime}"
                    f" minus "
                    f"{second_regime}"
                ),
        }


        for metric_index, metric_name in enumerate(
            METRIC_NAMES
        ):

            replicate_row[
                metric_name
            ] = (
                bootstrap_differences[
                    bootstrap_index,
                    contrast_index,
                    metric_index,
                ]
            )


        replicate_rows.append(
            replicate_row
        )


coverage_regime_replicates_df = pd.DataFrame(
    replicate_rows
)


# --------------------------------------------------
# Audit
# --------------------------------------------------

group_condition_counts = (
    coverage_df
    .groupby(
        [
            "matched_group_id",
            "regime",
        ]
    )
    .size()
)


coverage_regime_audit_df = pd.DataFrame(
    [
        {
            "benchmark_rows":
                len(
                    coverage_df
                ),

            "matched_groups":
                coverage_df[
                    "matched_group_id"
                ].nunique(),

            "regimes":
                coverage_df[
                    "regime"
                ].nunique(),

            "candidate_sizes":
                coverage_df[
                    "n_items"
                ].nunique(),

            "minimum_conditions_per_group_regime":
                int(
                    group_condition_counts.min()
                ),

            "maximum_conditions_per_group_regime":
                int(
                    group_condition_counts.max()
                ),

            "negative_gap_rows":
                int(
                    (
                        coverage_df[
                            "optimality_gap_percent"
                        ]
                        <
                        -1e-8
                    ).sum()
                ),

            "exact_nonoptimal_rows":
                int(
                    (
                        coverage_df[
                            "exact_status"
                        ]
                        !=
                        "OPTIMAL"
                    ).sum()
                ),

            "greedy_infeasible_rows":
                int(
                    (
                        coverage_df[
                            "greedy_status"
                        ]
                        !=
                        "FEASIBLE"
                    ).sum()
                ),

            "bootstrap_replicates":
                COVERAGE_REGIME_BOOTSTRAP_REPLICATES,

            "contrasts":
                len(
                    REGIME_CONTRASTS
                ),

            "confidence_intervals":
                "unadjusted_percentile_95",
        }
    ]
)


assert (
    group_condition_counts
    ==
    16
).all()


assert coverage_regime_audit_df.loc[
    0,
    "negative_gap_rows",
] == 0


assert coverage_regime_audit_df.loc[
    0,
    "exact_nonoptimal_rows",
] == 0


assert coverage_regime_audit_df.loc[
    0,
    "greedy_infeasible_rows",
] == 0


# --------------------------------------------------
# Save outputs
# --------------------------------------------------

coverage_regime_contrast_df.to_csv(
    COVERAGE_REGIME_CONTRAST_PATH,
    index=False,
)


coverage_regime_replicates_df.to_csv(
    COVERAGE_REGIME_REPLICATES_PATH,
    index=False,
)


coverage_regime_audit_df.to_csv(
    COVERAGE_REGIME_AUDIT_PATH,
    index=False,
)


# --------------------------------------------------
# Readable paired-contrast display
# --------------------------------------------------

key_metrics = [
    "mean_optimality_gap_percent",
    "exact_cost_recovery_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "more_items_percent",
    "mean_selected_count_difference",
]


contrast_display_df = (
    coverage_regime_contrast_df[
        coverage_regime_contrast_df[
            "metric"
        ].isin(
            key_metrics
        )
    ]
    .copy()
)


contrast_display_df[
    "difference_and_ci"
] = (
    contrast_display_df[
        "estimated_difference"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    contrast_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    contrast_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


# --------------------------------------------------
# Printed report
# --------------------------------------------------

print(
    "======================================"
)

print(
    "COVERAGE REGIME-CONTRAST AUDIT"
)

print(
    "======================================"
)


print(
    coverage_regime_audit_df.to_string(
        index=False
    )
)


print(
    "\n======================================"
)

print(
    "COVERAGE RESULTS BY STRUCTURAL REGIME"
)

print(
    "======================================"
)


print(
    coverage_regime_descriptive_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.4f}",
    )
)


print(
    "\n======================================"
)

print(
    "PAIRED COVERAGE REGIME CONTRASTS"
)

print(
    "======================================"
)


print(
    contrast_display_df[
        [
            "first_regime",
            "second_regime",
            "metric",
            "difference_and_ci",
            "ci_excludes_zero",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\nRegime descriptive summary saved to:"
)

print(
    COVERAGE_REGIME_DESCRIPTIVE_PATH
)


print(
    "\nPaired regime contrasts saved to:"
)

print(
    COVERAGE_REGIME_CONTRAST_PATH
)


print(
    "\nBootstrap replicates saved to:"
)

print(
    COVERAGE_REGIME_REPLICATES_PATH
)


print(
    "\nRegime-contrast audit saved to:"
)

print(
    COVERAGE_REGIME_AUDIT_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Step 131 — Coverage Dynamic Greedy failure
# mechanisms and extreme-tail audit
# --------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------
# Project paths
# --------------------------------------------------

if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


COVERAGE_MAIN_RESULTS_PATH = (
    RESULTS_DIR
    / (
        "coverage_exact_vs_greedy_"
        "main_benchmark_v1_pool_normalized.csv"
    )
)


COVERAGE_MECHANISM_SUMMARY_PATH = (
    TABLES_DIR
    / "coverage_greedy_failure_mechanism_summary.csv"
)


COVERAGE_MECHANISM_BY_REGIME_PATH = (
    TABLES_DIR
    / "coverage_greedy_failure_mechanism_by_regime.csv"
)


COVERAGE_LARGE_GAP_ATTRIBUTION_PATH = (
    TABLES_DIR
    / "coverage_large_gap_mechanism_attribution.csv"
)


COVERAGE_TAIL_SUMMARY_PATH = (
    TABLES_DIR
    / "coverage_optimality_gap_tail_summary.csv"
)


COVERAGE_EXTREME_ROWS_PATH = (
    RESULTS_DIR
    / "coverage_extreme_optimality_gap_rows.csv"
)


COVERAGE_MECHANISM_AUDIT_PATH = (
    RESULTS_DIR
    / "coverage_failure_mechanism_audit.csv"
)


# --------------------------------------------------
# Load benchmark
# --------------------------------------------------

coverage_df = pd.read_csv(
    COVERAGE_MAIN_RESULTS_PATH,
    low_memory=False,
)


required_columns = [
    "matched_group_id",
    "n_items",
    "regime",
    "tau",
    "weight_profile",
    "exact_status",
    "greedy_status",
    "exact_selected_count",
    "greedy_selected_count",
    "exact_selected_items",
    "greedy_selected_items",
    "exact_utility",
    "greedy_utility",
    "exact_privacy",
    "greedy_privacy",
    "exact_normalized_cognitive_cost",
    "greedy_normalized_cognitive_cost",
    "exact_disclosure_cost",
    "greedy_disclosure_cost",
    "optimality_gap",
    "optimality_gap_percent",
    "jaccard_similarity",
    "selected_count_difference",
]


missing_columns = [
    column_name
    for column_name in required_columns
    if column_name not in coverage_df.columns
]


if missing_columns:

    raise KeyError(
        "Missing required columns: "
        + ", ".join(
            missing_columns
        )
    )


numeric_columns = [
    "n_items",
    "tau",
    "exact_selected_count",
    "greedy_selected_count",
    "exact_utility",
    "greedy_utility",
    "exact_privacy",
    "greedy_privacy",
    "exact_normalized_cognitive_cost",
    "greedy_normalized_cognitive_cost",
    "exact_disclosure_cost",
    "greedy_disclosure_cost",
    "optimality_gap",
    "optimality_gap_percent",
    "jaccard_similarity",
    "selected_count_difference",
]


for column_name in numeric_columns:

    coverage_df[
        column_name
    ] = pd.to_numeric(
        coverage_df[
            column_name
        ],
        errors="coerce",
    )


# --------------------------------------------------
# Integrity checks
# --------------------------------------------------

assert len(
    coverage_df
) == 24000


assert (
    coverage_df[
        "exact_status"
    ]
    ==
    "OPTIMAL"
).all()


assert (
    coverage_df[
        "greedy_status"
    ]
    ==
    "FEASIBLE"
).all()


assert (
    coverage_df[
        "optimality_gap_percent"
    ]
    >=
    -1e-8
).all()


# --------------------------------------------------
# Parse selected subsets
# --------------------------------------------------

def parse_selected_subset(
    stored_value,
):
    """
    Parse a pipe-delimited selected subset.
    """

    if stored_value is None:

        return frozenset()


    try:

        if pd.isna(
            stored_value
        ):

            return frozenset()

    except (
        TypeError,
        ValueError,
    ):

        pass


    text = str(
        stored_value
    ).strip()


    if not text:

        return frozenset()


    return frozenset(
        token.strip()
        for token in text.split("|")
        if token.strip()
    )


coverage_df[
    "exact_subset"
] = coverage_df[
    "exact_selected_items"
].map(
    parse_selected_subset
)


coverage_df[
    "greedy_subset"
] = coverage_df[
    "greedy_selected_items"
].map(
    parse_selected_subset
)


coverage_df[
    "subsets_identical"
] = (
    coverage_df[
        "exact_subset"
    ]
    ==
    coverage_df[
        "greedy_subset"
    ]
)


# --------------------------------------------------
# Failure-mechanism classification
# --------------------------------------------------

def classify_failure_mechanism(
    row,
):
    """
    Partition each condition into one mutually
    exclusive mechanism.
    """

    if row[
        "subsets_identical"
    ]:

        return "identical_subset"


    count_difference = row[
        "selected_count_difference"
    ]


    if count_difference > 0:

        return "greedy_more_items"


    if count_difference < 0:

        return "greedy_fewer_items"


    return "same_count_different_items"


coverage_df[
    "failure_mechanism"
] = coverage_df.apply(
    classify_failure_mechanism,
    axis=1,
)


MECHANISM_ORDER = [
    "identical_subset",
    "same_count_different_items",
    "greedy_more_items",
    "greedy_fewer_items",
]


coverage_df[
    "failure_mechanism"
] = pd.Categorical(
    coverage_df[
        "failure_mechanism"
    ],
    categories=MECHANISM_ORDER,
    ordered=True,
)


coverage_df[
    "large_gap"
] = (
    coverage_df[
        "optimality_gap_percent"
    ]
    >
    10.0
)


coverage_df[
    "absolute_disclosure_cost_gap"
] = (
    coverage_df[
        "greedy_disclosure_cost"
    ]
    -
    coverage_df[
        "exact_disclosure_cost"
    ]
)


coverage_df[
    "utility_surplus_difference"
] = (
    (
        coverage_df[
            "greedy_utility"
        ]
        -
        coverage_df[
            "tau"
        ]
    )
    -
    (
        coverage_df[
            "exact_utility"
        ]
        -
        coverage_df[
            "tau"
        ]
    )
)


coverage_df[
    "privacy_difference"
] = (
    coverage_df[
        "greedy_privacy"
    ]
    -
    coverage_df[
        "exact_privacy"
    ]
)


coverage_df[
    "normalized_cognitive_difference"
] = (
    coverage_df[
        "greedy_normalized_cognitive_cost"
    ]
    -
    coverage_df[
        "exact_normalized_cognitive_cost"
    ]
)


# --------------------------------------------------
# Summary helper
# --------------------------------------------------

def summarize_mechanism(
    data_frame,
    total_rows,
):
    """
    Descriptive summary for one mechanism.
    """

    gap_values = data_frame[
        "optimality_gap_percent"
    ].to_numpy(
        dtype=float
    )


    large_gap_rows = int(
        data_frame[
            "large_gap"
        ].sum()
    )


    return pd.Series(
        {
            "rows":
                len(
                    data_frame
                ),

            "share_of_all_rows_percent":
                100.0
                *
                len(
                    data_frame
                )
                /
                total_rows,

            "mean_optimality_gap_percent":
                gap_values.mean(),

            "median_optimality_gap_percent":
                np.median(
                    gap_values
                ),

            "p90_optimality_gap_percent":
                np.quantile(
                    gap_values,
                    0.90,
                ),

            "p95_optimality_gap_percent":
                np.quantile(
                    gap_values,
                    0.95,
                ),

            "maximum_optimality_gap_percent":
                gap_values.max(),

            "large_gap_rows":
                large_gap_rows,

            "within_mechanism_large_gap_percent":
                100.0
                *
                large_gap_rows
                /
                len(
                    data_frame
                ),

            "mean_absolute_disclosure_cost_gap":
                data_frame[
                    "absolute_disclosure_cost_gap"
                ].mean(),

            "mean_jaccard_similarity":
                data_frame[
                    "jaccard_similarity"
                ].mean(),

            "mean_selected_count_difference":
                data_frame[
                    "selected_count_difference"
                ].mean(),

            "mean_privacy_difference":
                data_frame[
                    "privacy_difference"
                ].mean(),

            "mean_normalized_cognitive_difference":
                data_frame[
                    "normalized_cognitive_difference"
                ].mean(),

            "mean_utility_surplus_difference":
                data_frame[
                    "utility_surplus_difference"
                ].mean(),
        }
    )


# --------------------------------------------------
# Overall mechanism summary
# --------------------------------------------------

overall_mechanism_rows = []


for mechanism in MECHANISM_ORDER:

    current_df = coverage_df[
        coverage_df[
            "failure_mechanism"
        ]
        ==
        mechanism
    ]


    if len(
        current_df
    ) == 0:

        continue


    current_summary = summarize_mechanism(
        data_frame=current_df,
        total_rows=len(
            coverage_df
        ),
    )


    current_summary[
        "failure_mechanism"
    ] = mechanism


    overall_mechanism_rows.append(
        current_summary
    )


coverage_mechanism_summary_df = pd.DataFrame(
    overall_mechanism_rows
)


coverage_mechanism_summary_df = (
    coverage_mechanism_summary_df[
        [
            "failure_mechanism",
            "rows",
            "share_of_all_rows_percent",
            "mean_optimality_gap_percent",
            "median_optimality_gap_percent",
            "p90_optimality_gap_percent",
            "p95_optimality_gap_percent",
            "maximum_optimality_gap_percent",
            "large_gap_rows",
            "within_mechanism_large_gap_percent",
            "mean_absolute_disclosure_cost_gap",
            "mean_jaccard_similarity",
            "mean_selected_count_difference",
            "mean_privacy_difference",
            "mean_normalized_cognitive_difference",
            "mean_utility_surplus_difference",
        ]
    ]
)


# --------------------------------------------------
# Mechanism summary by structural regime
# --------------------------------------------------

regime_mechanism_rows = []


for regime in coverage_df[
    "regime"
].drop_duplicates():

    regime_df = coverage_df[
        coverage_df[
            "regime"
        ]
        ==
        regime
    ]


    for mechanism in MECHANISM_ORDER:

        current_df = regime_df[
            regime_df[
                "failure_mechanism"
            ]
            ==
            mechanism
        ]


        if len(
            current_df
        ) == 0:

            continue


        current_summary = summarize_mechanism(
            data_frame=current_df,
            total_rows=len(
                regime_df
            ),
        )


        current_summary[
            "regime"
        ] = regime


        current_summary[
            "failure_mechanism"
        ] = mechanism


        regime_mechanism_rows.append(
            current_summary
        )


coverage_mechanism_by_regime_df = pd.DataFrame(
    regime_mechanism_rows
)


coverage_mechanism_by_regime_df = (
    coverage_mechanism_by_regime_df[
        [
            "regime",
            "failure_mechanism",
            "rows",
            "share_of_all_rows_percent",
            "mean_optimality_gap_percent",
            "median_optimality_gap_percent",
            "p90_optimality_gap_percent",
            "p95_optimality_gap_percent",
            "maximum_optimality_gap_percent",
            "large_gap_rows",
            "within_mechanism_large_gap_percent",
            "mean_absolute_disclosure_cost_gap",
            "mean_jaccard_similarity",
            "mean_selected_count_difference",
            "mean_privacy_difference",
            "mean_normalized_cognitive_difference",
            "mean_utility_surplus_difference",
        ]
    ]
)


# --------------------------------------------------
# Large-gap attribution
# --------------------------------------------------

large_gap_df = coverage_df[
    coverage_df[
        "large_gap"
    ]
].copy()


large_gap_attribution_rows = []


for scope_name, scope_df in [
    (
        "overall",
        large_gap_df,
    ),
    *[
        (
            regime,
            large_gap_df[
                large_gap_df[
                    "regime"
                ]
                ==
                regime
            ],
        )
        for regime in coverage_df[
            "regime"
        ].drop_duplicates()
    ],
]:

    total_large_gap_rows = len(
        scope_df
    )


    for mechanism in MECHANISM_ORDER:

        current_rows = int(
            (
                scope_df[
                    "failure_mechanism"
                ]
                ==
                mechanism
            ).sum()
        )


        large_gap_attribution_rows.append(
            {
                "scope":
                    scope_name,

                "failure_mechanism":
                    mechanism,

                "large_gap_rows":
                    current_rows,

                "share_of_large_gap_rows_percent":
                    (
                        100.0
                        *
                        current_rows
                        /
                        total_large_gap_rows
                        if total_large_gap_rows > 0
                        else np.nan
                    ),
            }
        )


coverage_large_gap_attribution_df = pd.DataFrame(
    large_gap_attribution_rows
)


# --------------------------------------------------
# Extreme-tail analysis
# --------------------------------------------------

gap_p95 = float(
    coverage_df[
        "optimality_gap_percent"
    ].quantile(
        0.95
    )
)


gap_p99 = float(
    coverage_df[
        "optimality_gap_percent"
    ].quantile(
        0.99
    )
)


tail_definitions = [
    (
        "all_rows",
        coverage_df,
        0.0,
    ),

    (
        "gap_above_10_percent",
        coverage_df[
            coverage_df[
                "optimality_gap_percent"
            ]
            >
            10.0
        ],
        10.0,
    ),

    (
        "top_5_percent",
        coverage_df[
            coverage_df[
                "optimality_gap_percent"
            ]
            >=
            gap_p95
        ],
        gap_p95,
    ),

    (
        "top_1_percent",
        coverage_df[
            coverage_df[
                "optimality_gap_percent"
            ]
            >=
            gap_p99
        ],
        gap_p99,
    ),
]


tail_summary_rows = []


for tail_name, tail_df, threshold_value in (
    tail_definitions
):

    tail_summary_rows.append(
        {
            "tail_definition":
                tail_name,

            "gap_threshold_percent":
                threshold_value,

            "rows":
                len(
                    tail_df
                ),

            "share_of_all_rows_percent":
                100.0
                *
                len(
                    tail_df
                )
                /
                len(
                    coverage_df
                ),

            "mean_optimality_gap_percent":
                tail_df[
                    "optimality_gap_percent"
                ].mean(),

            "median_optimality_gap_percent":
                tail_df[
                    "optimality_gap_percent"
                ].median(),

            "maximum_optimality_gap_percent":
                tail_df[
                    "optimality_gap_percent"
                ].max(),

            "mean_absolute_disclosure_cost_gap":
                tail_df[
                    "absolute_disclosure_cost_gap"
                ].mean(),

            "mean_exact_disclosure_cost":
                tail_df[
                    "exact_disclosure_cost"
                ].mean(),

            "median_exact_disclosure_cost":
                tail_df[
                    "exact_disclosure_cost"
                ].median(),

            "same_count_different_items_percent":
                100.0
                *
                (
                    tail_df[
                        "failure_mechanism"
                    ]
                    ==
                    "same_count_different_items"
                ).mean(),

            "greedy_more_items_percent":
                100.0
                *
                (
                    tail_df[
                        "failure_mechanism"
                    ]
                    ==
                    "greedy_more_items"
                ).mean(),

            "greedy_fewer_items_percent":
                100.0
                *
                (
                    tail_df[
                        "failure_mechanism"
                    ]
                    ==
                    "greedy_fewer_items"
                ).mean(),

            "conflict_regime_percent":
                100.0
                *
                (
                    tail_df[
                        "regime"
                    ]
                    ==
                    "utility_burden_conflict"
                ).mean(),

            "mean_selected_count_difference":
                tail_df[
                    "selected_count_difference"
                ].mean(),

            "mean_jaccard_similarity":
                tail_df[
                    "jaccard_similarity"
                ].mean(),
        }
    )


coverage_tail_summary_df = pd.DataFrame(
    tail_summary_rows
)


# --------------------------------------------------
# Highest-gap individual conditions
# --------------------------------------------------

extreme_columns = [
    "matched_group_id",
    "n_items",
    "regime",
    "tau",
    "weight_profile",
    "failure_mechanism",
    "optimality_gap_percent",
    "absolute_disclosure_cost_gap",
    "exact_disclosure_cost",
    "greedy_disclosure_cost",
    "exact_selected_count",
    "greedy_selected_count",
    "selected_count_difference",
    "jaccard_similarity",
    "exact_utility",
    "greedy_utility",
    "exact_privacy",
    "greedy_privacy",
    "exact_normalized_cognitive_cost",
    "greedy_normalized_cognitive_cost",
    "exact_selected_items",
    "greedy_selected_items",
]


coverage_extreme_rows_df = (
    coverage_df
    .sort_values(
        "optimality_gap_percent",
        ascending=False,
    )[
        extreme_columns
    ]
    .head(
        20
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------
# Audit
# --------------------------------------------------

mechanism_counts = (
    coverage_df[
        "failure_mechanism"
    ]
    .value_counts(
        dropna=False
    )
)


large_gap_share_sum = (
    coverage_large_gap_attribution_df[
        coverage_large_gap_attribution_df[
            "scope"
        ]
        ==
        "overall"
    ][
        "share_of_large_gap_rows_percent"
    ]
    .sum()
)


coverage_mechanism_audit_df = pd.DataFrame(
    [
        {
            "benchmark_rows":
                len(
                    coverage_df
                ),

            "classified_rows":
                int(
                    mechanism_counts.sum()
                ),

            "unclassified_rows":
                int(
                    coverage_df[
                        "failure_mechanism"
                    ].isna().sum()
                ),

            "mechanism_categories_observed":
                int(
                    (
                        mechanism_counts
                        >
                        0
                    ).sum()
                ),

            "large_gap_rows":
                len(
                    large_gap_df
                ),

            "large_gap_percent":
                100.0
                *
                len(
                    large_gap_df
                )
                /
                len(
                    coverage_df
                ),

            "large_gap_attribution_share_sum":
                large_gap_share_sum,

            "p95_gap_threshold":
                gap_p95,

            "p99_gap_threshold":
                gap_p99,

            "maximum_gap_percent":
                coverage_df[
                    "optimality_gap_percent"
                ].max(),

            "minimum_absolute_gap":
                coverage_df[
                    "absolute_disclosure_cost_gap"
                ].min(),

            "negative_gap_rows":
                int(
                    (
                        coverage_df[
                            "optimality_gap_percent"
                        ]
                        <
                        -1e-8
                    ).sum()
                ),

            "count_difference_mismatch_rows":
                int(
                    (
                        (
                            coverage_df[
                                "greedy_selected_count"
                            ]
                            -
                            coverage_df[
                                "exact_selected_count"
                            ]
                        )
                        !=
                        coverage_df[
                            "selected_count_difference"
                        ]
                    ).sum()
                ),
        }
    ]
)


# --------------------------------------------------
# Integrity assertions
# --------------------------------------------------

assert coverage_mechanism_audit_df.loc[
    0,
    "classified_rows",
] == 24000


assert coverage_mechanism_audit_df.loc[
    0,
    "unclassified_rows",
] == 0


assert coverage_mechanism_audit_df.loc[
    0,
    "negative_gap_rows",
] == 0


assert coverage_mechanism_audit_df.loc[
    0,
    "count_difference_mismatch_rows",
] == 0


assert np.isclose(
    large_gap_share_sum,
    100.0,
    atol=1e-10,
)


assert (
    coverage_df.loc[
        coverage_df[
            "failure_mechanism"
        ]
        ==
        "identical_subset",
        "selected_count_difference",
    ]
    ==
    0
).all()


assert (
    coverage_df.loc[
        coverage_df[
            "failure_mechanism"
        ]
        ==
        "greedy_more_items",
        "selected_count_difference",
    ]
    >
    0
).all()


assert (
    coverage_df.loc[
        coverage_df[
            "failure_mechanism"
        ]
        ==
        "greedy_fewer_items",
        "selected_count_difference",
    ]
    <
    0
).all()


# --------------------------------------------------
# Save outputs
# --------------------------------------------------

coverage_mechanism_summary_df.to_csv(
    COVERAGE_MECHANISM_SUMMARY_PATH,
    index=False,
)


coverage_mechanism_by_regime_df.to_csv(
    COVERAGE_MECHANISM_BY_REGIME_PATH,
    index=False,
)


coverage_large_gap_attribution_df.to_csv(
    COVERAGE_LARGE_GAP_ATTRIBUTION_PATH,
    index=False,
)


coverage_tail_summary_df.to_csv(
    COVERAGE_TAIL_SUMMARY_PATH,
    index=False,
)


coverage_extreme_rows_df.to_csv(
    COVERAGE_EXTREME_ROWS_PATH,
    index=False,
)


coverage_mechanism_audit_df.to_csv(
    COVERAGE_MECHANISM_AUDIT_PATH,
    index=False,
)


# --------------------------------------------------
# Printed report
# --------------------------------------------------

print(
    "======================================"
)

print(
    "COVERAGE FAILURE-MECHANISM AUDIT"
)

print(
    "======================================"
)


print(
    coverage_mechanism_audit_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.6f}",
    )
)


print(
    "\n======================================"
)

print(
    "COVERAGE FAILURE MECHANISMS — OVERALL"
)

print(
    "======================================"
)


print(
    coverage_mechanism_summary_df[
        [
            "failure_mechanism",
            "rows",
            "share_of_all_rows_percent",
            "mean_optimality_gap_percent",
            "median_optimality_gap_percent",
            "p95_optimality_gap_percent",
            "maximum_optimality_gap_percent",
            "large_gap_rows",
            "within_mechanism_large_gap_percent",
            "mean_absolute_disclosure_cost_gap",
            "mean_jaccard_similarity",
            "mean_selected_count_difference",
        ]
    ].to_string(
        index=False,
        float_format=lambda value:
            f"{value:.4f}",
    )
)


print(
    "\n======================================"
)

print(
    "COVERAGE LARGE-GAP MECHANISM ATTRIBUTION"
)

print(
    "======================================"
)


print(
    coverage_large_gap_attribution_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.4f}",
    )
)


print(
    "\n======================================"
)

print(
    "COVERAGE OPTIMALITY-GAP TAIL SUMMARY"
)

print(
    "======================================"
)


print(
    coverage_tail_summary_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.4f}",
    )
)


print(
    "\n======================================"
)

print(
    "TOP 20 COVERAGE OPTIMALITY-GAP CONDITIONS"
)

print(
    "======================================"
)


print(
    coverage_extreme_rows_df[
        [
            "matched_group_id",
            "n_items",
            "regime",
            "tau",
            "weight_profile",
            "failure_mechanism",
            "optimality_gap_percent",
            "absolute_disclosure_cost_gap",
            "exact_disclosure_cost",
            "greedy_disclosure_cost",
            "exact_selected_count",
            "greedy_selected_count",
            "jaccard_similarity",
        ]
    ].to_string(
        index=False,
        float_format=lambda value:
            f"{value:.6f}",
    )
)


print(
    "\nOverall mechanism summary saved to:"
)

print(
    COVERAGE_MECHANISM_SUMMARY_PATH
)


print(
    "\nMechanism-by-regime summary saved to:"
)

print(
    COVERAGE_MECHANISM_BY_REGIME_PATH
)


print(
    "\nLarge-gap attribution saved to:"
)

print(
    COVERAGE_LARGE_GAP_ATTRIBUTION_PATH
)


print(
    "\nTail summary saved to:"
)

print(
    COVERAGE_TAIL_SUMMARY_PATH
)


print(
    "\nExtreme rows saved to:"
)

print(
    COVERAGE_EXTREME_ROWS_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Step 132 — Coverage threshold effects, relative-gap
# denominator audit, and paired adjacent contrasts
# --------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------
# Configuration
# --------------------------------------------------

THRESHOLD_AUDIT_BOOTSTRAP_REPLICATES = 5000

THRESHOLD_AUDIT_BOOTSTRAP_SEED = (
    globals().get(
        "GLOBAL_SEED",
        2026,
    )
    + 132
)


THRESHOLD_ORDER = [
    0.70,
    0.80,
    0.90,
    0.95,
]


CANDIDATE_SIZE_ORDER = [
    8,
    10,
    12,
    14,
    16,
]


ADJACENT_THRESHOLD_CONTRASTS = [
    (
        0.80,
        0.70,
    ),

    (
        0.90,
        0.80,
    ),

    (
        0.95,
        0.90,
    ),
]


EXACT_RECOVERY_TOLERANCE_PERCENT = 1e-10


METRIC_NAMES = [
    "mean_optimality_gap_percent",
    "mean_absolute_disclosure_cost_gap",
    "mean_exact_disclosure_cost",
    "exact_cost_recovery_percent",
    "large_gap_percent",
    "more_items_percent",
    "same_count_different_items_percent",
    "mean_jaccard_similarity",
    "mean_selected_count_difference",
]


AGGREGATE_COLUMNS = [
    "comparable_rows",
    "gap_sum",
    "absolute_gap_sum",
    "exact_cost_sum",
    "exact_recovery_sum",
    "large_gap_sum",
    "more_items_sum",
    "same_count_different_items_sum",
    "jaccard_sum",
    "selected_count_difference_sum",
]


# --------------------------------------------------
# Paths
# --------------------------------------------------

if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


COVERAGE_MAIN_RESULTS_PATH = (
    RESULTS_DIR
    / (
        "coverage_exact_vs_greedy_"
        "main_benchmark_v1_pool_normalized.csv"
    )
)


THRESHOLD_DESCRIPTIVE_PATH = (
    TABLES_DIR
    / "coverage_threshold_denominator_descriptive_summary.csv"
)


THRESHOLD_TAIL_COMPOSITION_PATH = (
    TABLES_DIR
    / "coverage_threshold_extreme_tail_composition.csv"
)


THRESHOLD_CONTRAST_PATH = (
    TABLES_DIR
    / "coverage_threshold_paired_adjacent_contrasts.csv"
)


THRESHOLD_BOOTSTRAP_REPLICATES_PATH = (
    RESULTS_DIR
    / "coverage_threshold_paired_bootstrap_replicates.csv"
)


THRESHOLD_AUDIT_PATH = (
    RESULTS_DIR
    / "coverage_threshold_denominator_audit.csv"
)


# --------------------------------------------------
# Load benchmark
# --------------------------------------------------

coverage_df = pd.read_csv(
    COVERAGE_MAIN_RESULTS_PATH,
    low_memory=False,
)


required_columns = [
    "matched_group_id",
    "n_items",
    "regime",
    "tau",
    "weight_profile",
    "exact_status",
    "greedy_status",
    "exact_selected_items",
    "greedy_selected_items",
    "exact_selected_count",
    "greedy_selected_count",
    "exact_disclosure_cost",
    "greedy_disclosure_cost",
    "optimality_gap_percent",
    "jaccard_similarity",
    "selected_count_difference",
]


missing_columns = [
    column_name
    for column_name in required_columns
    if column_name not in coverage_df.columns
]


if missing_columns:

    raise KeyError(
        "Missing required columns: "
        + ", ".join(
            missing_columns
        )
    )


numeric_columns = [
    "n_items",
    "tau",
    "exact_selected_count",
    "greedy_selected_count",
    "exact_disclosure_cost",
    "greedy_disclosure_cost",
    "optimality_gap_percent",
    "jaccard_similarity",
    "selected_count_difference",
]


for column_name in numeric_columns:

    coverage_df[
        column_name
    ] = pd.to_numeric(
        coverage_df[
            column_name
        ],
        errors="coerce",
    )


# --------------------------------------------------
# Benchmark integrity
# --------------------------------------------------

assert len(
    coverage_df
) == 24000


assert (
    coverage_df[
        "matched_group_id"
    ].nunique()
    == 500
)


assert (
    sorted(
        coverage_df[
            "tau"
        ]
        .dropna()
        .unique()
        .tolist()
    )
    ==
    THRESHOLD_ORDER
)


assert (
    sorted(
        coverage_df[
            "n_items"
        ]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
    ==
    CANDIDATE_SIZE_ORDER
)


assert (
    coverage_df[
        "exact_status"
    ]
    ==
    "OPTIMAL"
).all()


assert (
    coverage_df[
        "greedy_status"
    ]
    ==
    "FEASIBLE"
).all()


# --------------------------------------------------
# Derived indicators
# --------------------------------------------------

coverage_df[
    "exact_selected_items_clean"
] = (
    coverage_df[
        "exact_selected_items"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
)


coverage_df[
    "greedy_selected_items_clean"
] = (
    coverage_df[
        "greedy_selected_items"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
)


coverage_df[
    "same_selected_subset"
] = (
    coverage_df[
        "exact_selected_items_clean"
    ]
    ==
    coverage_df[
        "greedy_selected_items_clean"
    ]
)


coverage_df[
    "exact_cost_recovery_numeric"
] = (
    coverage_df[
        "optimality_gap_percent"
    ]
    .abs()
    <=
    EXACT_RECOVERY_TOLERANCE_PERCENT
).astype(int)


coverage_df[
    "large_gap_numeric"
] = (
    coverage_df[
        "optimality_gap_percent"
    ]
    >
    10.0
).astype(int)


coverage_df[
    "more_items_numeric"
] = (
    coverage_df[
        "selected_count_difference"
    ]
    >
    0
).astype(int)


coverage_df[
    "same_count_different_items_numeric"
] = (
    (
        coverage_df[
            "selected_count_difference"
        ]
        ==
        0
    )
    &
    (
        ~coverage_df[
            "same_selected_subset"
        ]
    )
).astype(int)


coverage_df[
    "absolute_disclosure_cost_gap"
] = (
    coverage_df[
        "greedy_disclosure_cost"
    ]
    -
    coverage_df[
        "exact_disclosure_cost"
    ]
)


# Remove negligible floating-point deviations.
coverage_df.loc[
    coverage_df[
        "absolute_disclosure_cost_gap"
    ].abs()
    <=
    1e-12,
    "absolute_disclosure_cost_gap",
] = 0.0


# --------------------------------------------------
# Global tail thresholds
# --------------------------------------------------

GLOBAL_P95_GAP = float(
    coverage_df[
        "optimality_gap_percent"
    ].quantile(
        0.95
    )
)


GLOBAL_P99_GAP = float(
    coverage_df[
        "optimality_gap_percent"
    ].quantile(
        0.99
    )
)


coverage_df[
    "global_top_5_percent"
] = (
    coverage_df[
        "optimality_gap_percent"
    ]
    >=
    GLOBAL_P95_GAP
)


coverage_df[
    "global_top_1_percent"
] = (
    coverage_df[
        "optimality_gap_percent"
    ]
    >=
    GLOBAL_P99_GAP
)


# --------------------------------------------------
# Descriptive threshold summary
# --------------------------------------------------

descriptive_rows = []


for tau in THRESHOLD_ORDER:

    current_df = coverage_df[
        np.isclose(
            coverage_df[
                "tau"
            ],
            tau,
        )
    ]


    gap_values = (
        current_df[
            "optimality_gap_percent"
        ]
        .to_numpy(
            dtype=float
        )
    )


    descriptive_rows.append(
        {
            "tau":
                tau,

            "rows":
                len(
                    current_df
                ),

            "matched_groups":
                current_df[
                    "matched_group_id"
                ].nunique(),

            "mean_optimality_gap_percent":
                gap_values.mean(),

            "median_optimality_gap_percent":
                np.median(
                    gap_values
                ),

            "p90_optimality_gap_percent":
                np.quantile(
                    gap_values,
                    0.90,
                ),

            "p95_optimality_gap_percent":
                np.quantile(
                    gap_values,
                    0.95,
                ),

            "maximum_optimality_gap_percent":
                gap_values.max(),

            "mean_absolute_disclosure_cost_gap":
                current_df[
                    "absolute_disclosure_cost_gap"
                ].mean(),

            "median_absolute_disclosure_cost_gap":
                current_df[
                    "absolute_disclosure_cost_gap"
                ].median(),

            "mean_exact_disclosure_cost":
                current_df[
                    "exact_disclosure_cost"
                ].mean(),

            "median_exact_disclosure_cost":
                current_df[
                    "exact_disclosure_cost"
                ].median(),

            "exact_cost_recovery_percent":
                100.0
                *
                current_df[
                    "exact_cost_recovery_numeric"
                ].mean(),

            "large_gap_percent":
                100.0
                *
                current_df[
                    "large_gap_numeric"
                ].mean(),

            "more_items_percent":
                100.0
                *
                current_df[
                    "more_items_numeric"
                ].mean(),

            "same_count_different_items_percent":
                100.0
                *
                current_df[
                    "same_count_different_items_numeric"
                ].mean(),

            "mean_jaccard_similarity":
                current_df[
                    "jaccard_similarity"
                ].mean(),

            "mean_selected_count_difference":
                current_df[
                    "selected_count_difference"
                ].mean(),

            "global_top_5_percent_rows":
                int(
                    current_df[
                        "global_top_5_percent"
                    ].sum()
                ),

            "global_top_1_percent_rows":
                int(
                    current_df[
                        "global_top_1_percent"
                    ].sum()
                ),

            "within_threshold_top_5_percent_rate":
                100.0
                *
                current_df[
                    "global_top_5_percent"
                ].mean(),

            "within_threshold_top_1_percent_rate":
                100.0
                *
                current_df[
                    "global_top_1_percent"
                ].mean(),
        }
    )


threshold_descriptive_df = pd.DataFrame(
    descriptive_rows
)


# --------------------------------------------------
# Tail composition by threshold
# --------------------------------------------------

tail_definitions = [
    (
        "gap_above_10_percent",
        coverage_df[
            coverage_df[
                "optimality_gap_percent"
            ]
            >
            10.0
        ],
        10.0,
    ),

    (
        "global_top_5_percent",
        coverage_df[
            coverage_df[
                "global_top_5_percent"
            ]
        ],
        GLOBAL_P95_GAP,
    ),

    (
        "global_top_1_percent",
        coverage_df[
            coverage_df[
                "global_top_1_percent"
            ]
        ],
        GLOBAL_P99_GAP,
    ),
]


tail_composition_rows = []


for tail_name, tail_df, gap_threshold in (
    tail_definitions
):

    total_tail_rows = len(
        tail_df
    )


    for tau in THRESHOLD_ORDER:

        current_df = tail_df[
            np.isclose(
                tail_df[
                    "tau"
                ],
                tau,
            )
        ]


        tau_total_rows = int(
            np.isclose(
                coverage_df[
                    "tau"
                ],
                tau,
            ).sum()
        )


        tail_composition_rows.append(
            {
                "tail_definition":
                    tail_name,

                "gap_threshold_percent":
                    gap_threshold,

                "tau":
                    tau,

                "tail_rows":
                    len(
                        current_df
                    ),

                "share_of_tail_percent":
                    (
                        100.0
                        *
                        len(
                            current_df
                        )
                        /
                        total_tail_rows
                    ),

                "share_of_threshold_rows_percent":
                    (
                        100.0
                        *
                        len(
                            current_df
                        )
                        /
                        tau_total_rows
                    ),

                "mean_optimality_gap_percent":
                    current_df[
                        "optimality_gap_percent"
                    ].mean(),

                "mean_absolute_disclosure_cost_gap":
                    current_df[
                        "absolute_disclosure_cost_gap"
                    ].mean(),

                "mean_exact_disclosure_cost":
                    current_df[
                        "exact_disclosure_cost"
                    ].mean(),

                "more_items_percent":
                    100.0
                    *
                    current_df[
                        "more_items_numeric"
                    ].mean(),

                "same_count_different_items_percent":
                    100.0
                    *
                    current_df[
                        "same_count_different_items_numeric"
                    ].mean(),

                "mean_selected_count_difference":
                    current_df[
                        "selected_count_difference"
                    ].mean(),
            }
        )


threshold_tail_composition_df = pd.DataFrame(
    tail_composition_rows
)


# --------------------------------------------------
# Matched-group sufficient statistics
#
# Each group × threshold contains:
# 3 regimes × 4 weight profiles = 12 conditions.
# --------------------------------------------------

cluster_df = (
    coverage_df
    .groupby(
        [
            "n_items",
            "matched_group_id",
            "tau",
        ],
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "size",
        ),

        gap_sum=(
            "optimality_gap_percent",
            "sum",
        ),

        absolute_gap_sum=(
            "absolute_disclosure_cost_gap",
            "sum",
        ),

        exact_cost_sum=(
            "exact_disclosure_cost",
            "sum",
        ),

        exact_recovery_sum=(
            "exact_cost_recovery_numeric",
            "sum",
        ),

        large_gap_sum=(
            "large_gap_numeric",
            "sum",
        ),

        more_items_sum=(
            "more_items_numeric",
            "sum",
        ),

        same_count_different_items_sum=(
            "same_count_different_items_numeric",
            "sum",
        ),

        jaccard_sum=(
            "jaccard_similarity",
            "sum",
        ),

        selected_count_difference_sum=(
            "selected_count_difference",
            "sum",
        ),
    )
)


assert (
    cluster_df[
        "comparable_rows"
    ]
    ==
    12
).all()


# --------------------------------------------------
# Metric conversion
# --------------------------------------------------

def calculate_threshold_metrics(
    totals,
):
    """
    Convert sufficient-statistic totals into
    threshold-level metrics.
    """

    comparable_rows = totals[0]


    if comparable_rows <= 0:

        return np.full(
            len(
                METRIC_NAMES
            ),
            np.nan,
        )


    return np.array(
        [
            totals[1]
            /
            comparable_rows,

            totals[2]
            /
            comparable_rows,

            totals[3]
            /
            comparable_rows,

            100.0
            *
            totals[4]
            /
            comparable_rows,

            100.0
            *
            totals[5]
            /
            comparable_rows,

            100.0
            *
            totals[6]
            /
            comparable_rows,

            100.0
            *
            totals[7]
            /
            comparable_rows,

            totals[8]
            /
            comparable_rows,

            totals[9]
            /
            comparable_rows,
        ],
        dtype=float,
    )


# --------------------------------------------------
# Aligned arrays by size and threshold
# --------------------------------------------------

arrays_by_size_threshold = {}


for n_items in CANDIDATE_SIZE_ORDER:

    arrays_by_size_threshold[
        n_items
    ] = {}


    reference_group_ids = None


    for tau in THRESHOLD_ORDER:

        current_df = (
            cluster_df[
                (
                    cluster_df[
                        "n_items"
                    ]
                    ==
                    n_items
                )
                &
                (
                    np.isclose(
                        cluster_df[
                            "tau"
                        ],
                        tau,
                    )
                )
            ]
            .sort_values(
                "matched_group_id"
            )
            .reset_index(
                drop=True
            )
        )


        current_group_ids = (
            current_df[
                "matched_group_id"
            ].tolist()
        )


        assert len(
            current_group_ids
        ) == 100


        if reference_group_ids is None:

            reference_group_ids = (
                current_group_ids
            )

        else:

            assert (
                current_group_ids
                ==
                reference_group_ids
            )


        arrays_by_size_threshold[
            n_items
        ][
            tau
        ] = (
            current_df[
                AGGREGATE_COLUMNS
            ]
            .to_numpy(
                dtype=float
            )
        )


# --------------------------------------------------
# Observed threshold totals
# --------------------------------------------------

observed_totals = {
    tau: np.zeros(
        len(
            AGGREGATE_COLUMNS
        ),
        dtype=float,
    )
    for tau in THRESHOLD_ORDER
}


for n_items in CANDIDATE_SIZE_ORDER:

    for tau in THRESHOLD_ORDER:

        observed_totals[
            tau
        ] += (
            arrays_by_size_threshold[
                n_items
            ][
                tau
            ]
            .sum(
                axis=0
            )
        )


observed_metrics = {
    tau:
        calculate_threshold_metrics(
            observed_totals[
                tau
            ]
        )
    for tau in THRESHOLD_ORDER
}


# --------------------------------------------------
# Shared paired bootstrap across all thresholds
# --------------------------------------------------

bootstrap_rng = np.random.default_rng(
    THRESHOLD_AUDIT_BOOTSTRAP_SEED
)


bootstrap_differences = np.empty(
    (
        THRESHOLD_AUDIT_BOOTSTRAP_REPLICATES,
        len(
            ADJACENT_THRESHOLD_CONTRASTS
        ),
        len(
            METRIC_NAMES
        ),
    ),
    dtype=float,
)


for bootstrap_index in range(
    THRESHOLD_AUDIT_BOOTSTRAP_REPLICATES
):

    bootstrap_totals = {
        tau: np.zeros(
            len(
                AGGREGATE_COLUMNS
            ),
            dtype=float,
        )
        for tau in THRESHOLD_ORDER
    }


    for n_items in CANDIDATE_SIZE_ORDER:

        number_of_groups = (
            arrays_by_size_threshold[
                n_items
            ][
                THRESHOLD_ORDER[0]
            ].shape[0]
        )


        sampled_indices = (
            bootstrap_rng.integers(
                low=0,
                high=number_of_groups,
                size=number_of_groups,
            )
        )


        for tau in THRESHOLD_ORDER:

            bootstrap_totals[
                tau
            ] += (
                arrays_by_size_threshold[
                    n_items
                ][
                    tau
                ][
                    sampled_indices
                ]
                .sum(
                    axis=0
                )
            )


    bootstrap_metrics = {
        tau:
            calculate_threshold_metrics(
                bootstrap_totals[
                    tau
                ]
            )
        for tau in THRESHOLD_ORDER
    }


    for contrast_index, (
        higher_tau,
        lower_tau,
    ) in enumerate(
        ADJACENT_THRESHOLD_CONTRASTS
    ):

        bootstrap_differences[
            bootstrap_index,
            contrast_index,
        ] = (
            bootstrap_metrics[
                higher_tau
            ]
            -
            bootstrap_metrics[
                lower_tau
            ]
        )


# --------------------------------------------------
# Paired adjacent-threshold contrasts
# --------------------------------------------------

contrast_rows = []


for contrast_index, (
    higher_tau,
    lower_tau,
) in enumerate(
    ADJACENT_THRESHOLD_CONTRASTS
):

    observed_difference = (
        observed_metrics[
            higher_tau
        ]
        -
        observed_metrics[
            lower_tau
        ]
    )


    current_bootstrap = (
        bootstrap_differences[
            :,
            contrast_index,
            :,
        ]
    )


    confidence_lower = np.quantile(
        current_bootstrap,
        0.025,
        axis=0,
    )


    confidence_upper = np.quantile(
        current_bootstrap,
        0.975,
        axis=0,
    )


    standard_errors = (
        current_bootstrap.std(
            axis=0,
            ddof=1,
        )
    )


    for metric_index, metric_name in enumerate(
        METRIC_NAMES
    ):

        contrast_rows.append(
            {
                "higher_tau":
                    higher_tau,

                "lower_tau":
                    lower_tau,

                "contrast":
                    (
                        f"{higher_tau:.2f}"
                        f" minus "
                        f"{lower_tau:.2f}"
                    ),

                "metric":
                    metric_name,

                "estimated_difference":
                    observed_difference[
                        metric_index
                    ],

                "bootstrap_standard_error":
                    standard_errors[
                        metric_index
                    ],

                "ci_95_lower":
                    confidence_lower[
                        metric_index
                    ],

                "ci_95_upper":
                    confidence_upper[
                        metric_index
                    ],

                "ci_excludes_zero":
                    (
                        confidence_lower[
                            metric_index
                        ]
                        >
                        0.0
                    )
                    or
                    (
                        confidence_upper[
                            metric_index
                        ]
                        <
                        0.0
                    ),

                "matched_groups":
                    500,

                "matched_groups_per_size":
                    100,

                "bootstrap_replicates":
                    THRESHOLD_AUDIT_BOOTSTRAP_REPLICATES,

                "resampling_design":
                    (
                        "paired_across_thresholds_"
                        "stratified_by_candidate_set_size"
                    ),

                "multiple_comparison_adjustment":
                    "none",
            }
        )


threshold_contrast_df = pd.DataFrame(
    contrast_rows
)


# --------------------------------------------------
# Save bootstrap replicates
# --------------------------------------------------

replicate_rows = []


for bootstrap_index in range(
    THRESHOLD_AUDIT_BOOTSTRAP_REPLICATES
):

    for contrast_index, (
        higher_tau,
        lower_tau,
    ) in enumerate(
        ADJACENT_THRESHOLD_CONTRASTS
    ):

        replicate_row = {
            "bootstrap_replicate":
                bootstrap_index,

            "higher_tau":
                higher_tau,

            "lower_tau":
                lower_tau,

            "contrast":
                (
                    f"{higher_tau:.2f}"
                    f" minus "
                    f"{lower_tau:.2f}"
                ),
        }


        for metric_index, metric_name in enumerate(
            METRIC_NAMES
        ):

            replicate_row[
                metric_name
            ] = (
                bootstrap_differences[
                    bootstrap_index,
                    contrast_index,
                    metric_index,
                ]
            )


        replicate_rows.append(
            replicate_row
        )


threshold_bootstrap_replicates_df = (
    pd.DataFrame(
        replicate_rows
    )
)


# --------------------------------------------------
# Audit
# --------------------------------------------------

group_threshold_counts = (
    coverage_df
    .groupby(
        [
            "matched_group_id",
            "tau",
        ]
    )
    .size()
)


threshold_audit_df = pd.DataFrame(
    [
        {
            "benchmark_rows":
                len(
                    coverage_df
                ),

            "matched_groups":
                coverage_df[
                    "matched_group_id"
                ].nunique(),

            "thresholds":
                coverage_df[
                    "tau"
                ].nunique(),

            "minimum_rows_per_group_threshold":
                int(
                    group_threshold_counts.min()
                ),

            "maximum_rows_per_group_threshold":
                int(
                    group_threshold_counts.max()
                ),

            "global_p95_gap_threshold":
                GLOBAL_P95_GAP,

            "global_p99_gap_threshold":
                GLOBAL_P99_GAP,

            "negative_absolute_gap_rows":
                int(
                    (
                        coverage_df[
                            "absolute_disclosure_cost_gap"
                        ]
                        <
                        -1e-10
                    ).sum()
                ),

            "maximum_reconstruction_error":
                (
                    (
                        coverage_df[
                            "absolute_disclosure_cost_gap"
                        ]
                        /
                        coverage_df[
                            "exact_disclosure_cost"
                        ]
                    )
                    *
                    100.0
                    -
                    coverage_df[
                        "optimality_gap_percent"
                    ]
                )
                .abs()
                .max(),

            "bootstrap_replicates":
                THRESHOLD_AUDIT_BOOTSTRAP_REPLICATES,

            "confidence_intervals":
                "unadjusted_percentile_95",
        }
    ]
)


assert (
    group_threshold_counts
    ==
    12
).all()


assert threshold_audit_df.loc[
    0,
    "negative_absolute_gap_rows",
] == 0


assert threshold_audit_df.loc[
    0,
    "maximum_reconstruction_error",
] <= 1e-8


# --------------------------------------------------
# Save outputs
# --------------------------------------------------

threshold_descriptive_df.to_csv(
    THRESHOLD_DESCRIPTIVE_PATH,
    index=False,
)


threshold_tail_composition_df.to_csv(
    THRESHOLD_TAIL_COMPOSITION_PATH,
    index=False,
)


threshold_contrast_df.to_csv(
    THRESHOLD_CONTRAST_PATH,
    index=False,
)


threshold_bootstrap_replicates_df.to_csv(
    THRESHOLD_BOOTSTRAP_REPLICATES_PATH,
    index=False,
)


threshold_audit_df.to_csv(
    THRESHOLD_AUDIT_PATH,
    index=False,
)


# --------------------------------------------------
# Readable contrast display
# --------------------------------------------------

contrast_display_df = (
    threshold_contrast_df.copy()
)


contrast_display_df[
    "difference_and_ci"
] = (
    contrast_display_df[
        "estimated_difference"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    contrast_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    contrast_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


# --------------------------------------------------
# Printed report
# --------------------------------------------------

print(
    "======================================"
)

print(
    "COVERAGE THRESHOLD-DENOMINATOR AUDIT"
)

print(
    "======================================"
)


print(
    threshold_audit_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.10f}",
    )
)


print(
    "\n======================================"
)

print(
    "COVERAGE RESULTS BY SUFFICIENCY THRESHOLD"
)

print(
    "======================================"
)


print(
    threshold_descriptive_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.4f}",
    )
)


print(
    "\n======================================"
)

print(
    "EXTREME-TAIL COMPOSITION BY THRESHOLD"
)

print(
    "======================================"
)


print(
    threshold_tail_composition_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.4f}",
    )
)


print(
    "\n======================================"
)

print(
    "PAIRED ADJACENT-THRESHOLD CONTRASTS"
)

print(
    "======================================"
)


print(
    contrast_display_df[
        [
            "contrast",
            "metric",
            "difference_and_ci",
            "ci_excludes_zero",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\nThreshold descriptive summary saved to:"
)

print(
    THRESHOLD_DESCRIPTIVE_PATH
)


print(
    "\nTail composition saved to:"
)

print(
    THRESHOLD_TAIL_COMPOSITION_PATH
)


print(
    "\nPaired threshold contrasts saved to:"
)

print(
    THRESHOLD_CONTRAST_PATH
)


print(
    "\nBootstrap replicates saved to:"
)

print(
    THRESHOLD_BOOTSTRAP_REPLICATES_PATH
)


print(
    "\nAudit saved to:"
)

print(
    THRESHOLD_AUDIT_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Step 133 — Coverage benchmark sensitivity by
# candidate-set size and objective-weight profile
# --------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------
# Configuration
# --------------------------------------------------

SENSITIVITY_BOOTSTRAP_REPLICATES = 5000

SENSITIVITY_BOOTSTRAP_SEED = (
    globals().get(
        "GLOBAL_SEED",
        2026,
    )
    + 133
)


CANDIDATE_SIZE_ORDER = [
    8,
    10,
    12,
    14,
    16,
]


WEIGHT_PROFILE_ORDER = [
    "balanced",
    "privacy_priority",
    "interpretation_priority",
    "parsimony_priority",
]


REFERENCE_WEIGHT_PROFILE = "balanced"


WEIGHT_PROFILE_CONTRASTS = [
    (
        "privacy_priority",
        REFERENCE_WEIGHT_PROFILE,
    ),

    (
        "interpretation_priority",
        REFERENCE_WEIGHT_PROFILE,
    ),

    (
        "parsimony_priority",
        REFERENCE_WEIGHT_PROFILE,
    ),
]


METRIC_NAMES = [
    "mean_optimality_gap_percent",
    "exact_cost_recovery_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "more_items_percent",
    "mean_selected_count_difference",
]


AGGREGATE_COLUMNS = [
    "comparable_rows",
    "gap_sum",
    "exact_recovery_sum",
    "jaccard_sum",
    "large_gap_sum",
    "more_items_sum",
    "selected_count_difference_sum",
]


EXACT_RECOVERY_TOLERANCE_PERCENT = 1e-10


# --------------------------------------------------
# Paths
# --------------------------------------------------

if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


COVERAGE_MAIN_RESULTS_PATH = (
    RESULTS_DIR
    / (
        "coverage_exact_vs_greedy_"
        "main_benchmark_v1_pool_normalized.csv"
    )
)


SIZE_SUMMARY_PATH = (
    TABLES_DIR
    / "coverage_candidate_size_bootstrap_summary.csv"
)


SIZE_BOOTSTRAP_REPLICATES_PATH = (
    RESULTS_DIR
    / "coverage_candidate_size_bootstrap_replicates.csv"
)


WEIGHT_DESCRIPTIVE_PATH = (
    TABLES_DIR
    / "coverage_weight_profile_descriptive_summary.csv"
)


WEIGHT_CONTRAST_PATH = (
    TABLES_DIR
    / "coverage_paired_weight_profile_contrasts.csv"
)


WEIGHT_BOOTSTRAP_REPLICATES_PATH = (
    RESULTS_DIR
    / "coverage_weight_profile_bootstrap_replicates.csv"
)


SENSITIVITY_AUDIT_PATH = (
    RESULTS_DIR
    / "coverage_size_weight_sensitivity_audit.csv"
)


# --------------------------------------------------
# Load benchmark
# --------------------------------------------------

coverage_df = pd.read_csv(
    COVERAGE_MAIN_RESULTS_PATH,
    low_memory=False,
)


required_columns = [
    "matched_group_id",
    "n_items",
    "regime",
    "tau",
    "weight_profile",
    "exact_status",
    "greedy_status",
    "optimality_gap_percent",
    "jaccard_similarity",
    "selected_count_difference",
]


missing_columns = [
    column_name
    for column_name in required_columns
    if column_name not in coverage_df.columns
]


if missing_columns:

    raise KeyError(
        "Missing required columns: "
        + ", ".join(
            missing_columns
        )
    )


for column_name in [
    "n_items",
    "tau",
    "optimality_gap_percent",
    "jaccard_similarity",
    "selected_count_difference",
]:

    coverage_df[
        column_name
    ] = pd.to_numeric(
        coverage_df[
            column_name
        ],
        errors="coerce",
    )


# --------------------------------------------------
# Design validation
# --------------------------------------------------

assert len(
    coverage_df
) == 24000


assert (
    coverage_df[
        "matched_group_id"
    ].nunique()
    == 500
)


assert (
    sorted(
        coverage_df[
            "n_items"
        ]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
    ==
    CANDIDATE_SIZE_ORDER
)


assert set(
    coverage_df[
        "weight_profile"
    ].dropna().unique()
) == set(
    WEIGHT_PROFILE_ORDER
)


assert (
    coverage_df[
        "exact_status"
    ]
    ==
    "OPTIMAL"
).all()


assert (
    coverage_df[
        "greedy_status"
    ]
    ==
    "FEASIBLE"
).all()


# --------------------------------------------------
# Reporting indicators
# --------------------------------------------------

coverage_df[
    "exact_recovery_numeric"
] = (
    coverage_df[
        "optimality_gap_percent"
    ]
    .abs()
    <=
    EXACT_RECOVERY_TOLERANCE_PERCENT
).astype(int)


coverage_df[
    "large_gap_numeric"
] = (
    coverage_df[
        "optimality_gap_percent"
    ]
    >
    10.0
).astype(int)


coverage_df[
    "more_items_numeric"
] = (
    coverage_df[
        "selected_count_difference"
    ]
    >
    0
).astype(int)


# --------------------------------------------------
# Metric conversion
# --------------------------------------------------

def calculate_metrics(
    totals,
):
    """
    Convert sufficient-statistic totals into the six
    Exact-relative Dynamic Greedy metrics.
    """

    comparable_rows = totals[0]


    if comparable_rows <= 0:

        return np.full(
            len(
                METRIC_NAMES
            ),
            np.nan,
        )


    return np.array(
        [
            totals[1]
            /
            comparable_rows,

            100.0
            *
            totals[2]
            /
            comparable_rows,

            totals[3]
            /
            comparable_rows,

            100.0
            *
            totals[4]
            /
            comparable_rows,

            100.0
            *
            totals[5]
            /
            comparable_rows,

            totals[6]
            /
            comparable_rows,
        ],
        dtype=float,
    )


# ==================================================
# PART A — Candidate-set size sensitivity
# ==================================================

size_cluster_df = (
    coverage_df
    .groupby(
        [
            "n_items",
            "matched_group_id",
        ],
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "size",
        ),

        gap_sum=(
            "optimality_gap_percent",
            "sum",
        ),

        exact_recovery_sum=(
            "exact_recovery_numeric",
            "sum",
        ),

        jaccard_sum=(
            "jaccard_similarity",
            "sum",
        ),

        large_gap_sum=(
            "large_gap_numeric",
            "sum",
        ),

        more_items_sum=(
            "more_items_numeric",
            "sum",
        ),

        selected_count_difference_sum=(
            "selected_count_difference",
            "sum",
        ),
    )
)


# Each matched group contains:
# 3 regimes × 4 thresholds × 4 profiles = 48 rows.
assert (
    size_cluster_df[
        "comparable_rows"
    ]
    ==
    48
).all()


size_arrays = {}


for n_items in CANDIDATE_SIZE_ORDER:

    current_df = (
        size_cluster_df[
            size_cluster_df[
                "n_items"
            ]
            ==
            n_items
        ]
        .sort_values(
            "matched_group_id"
        )
        .reset_index(
            drop=True
        )
    )


    assert len(
        current_df
    ) == 100


    size_arrays[
        n_items
    ] = (
        current_df[
            AGGREGATE_COLUMNS
        ]
        .to_numpy(
            dtype=float
        )
    )


size_observed_metrics = {}


for n_items in CANDIDATE_SIZE_ORDER:

    size_observed_metrics[
        n_items
    ] = calculate_metrics(
        size_arrays[
            n_items
        ].sum(
            axis=0
        )
    )


size_bootstrap_rng = np.random.default_rng(
    SENSITIVITY_BOOTSTRAP_SEED
)


size_bootstrap_values = np.empty(
    (
        SENSITIVITY_BOOTSTRAP_REPLICATES,
        len(
            CANDIDATE_SIZE_ORDER
        ),
        len(
            METRIC_NAMES
        ),
    ),
    dtype=float,
)


for bootstrap_index in range(
    SENSITIVITY_BOOTSTRAP_REPLICATES
):

    for size_index, n_items in enumerate(
        CANDIDATE_SIZE_ORDER
    ):

        number_of_groups = (
            size_arrays[
                n_items
            ].shape[0]
        )


        sampled_indices = (
            size_bootstrap_rng.integers(
                low=0,
                high=number_of_groups,
                size=number_of_groups,
            )
        )


        resampled_totals = (
            size_arrays[
                n_items
            ][
                sampled_indices
            ]
            .sum(
                axis=0
            )
        )


        size_bootstrap_values[
            bootstrap_index,
            size_index,
        ] = calculate_metrics(
            resampled_totals
        )


size_summary_rows = []


for size_index, n_items in enumerate(
    CANDIDATE_SIZE_ORDER
):

    current_bootstrap = (
        size_bootstrap_values[
            :,
            size_index,
            :,
        ]
    )


    confidence_lower = np.quantile(
        current_bootstrap,
        0.025,
        axis=0,
    )


    confidence_upper = np.quantile(
        current_bootstrap,
        0.975,
        axis=0,
    )


    standard_errors = (
        current_bootstrap.std(
            axis=0,
            ddof=1,
        )
    )


    for metric_index, metric_name in enumerate(
        METRIC_NAMES
    ):

        size_summary_rows.append(
            {
                "n_items":
                    n_items,

                "metric":
                    metric_name,

                "estimate":
                    size_observed_metrics[
                        n_items
                    ][
                        metric_index
                    ],

                "bootstrap_standard_error":
                    standard_errors[
                        metric_index
                    ],

                "ci_95_lower":
                    confidence_lower[
                        metric_index
                    ],

                "ci_95_upper":
                    confidence_upper[
                        metric_index
                    ],

                "matched_groups":
                    100,

                "conditions":
                    4800,

                "bootstrap_replicates":
                    SENSITIVITY_BOOTSTRAP_REPLICATES,

                "resampling_unit":
                    "matched_group_within_candidate_size",
            }
        )


size_summary_df = pd.DataFrame(
    size_summary_rows
)


size_replicate_rows = []


for bootstrap_index in range(
    SENSITIVITY_BOOTSTRAP_REPLICATES
):

    for size_index, n_items in enumerate(
        CANDIDATE_SIZE_ORDER
    ):

        replicate_row = {
            "bootstrap_replicate":
                bootstrap_index,

            "n_items":
                n_items,
        }


        for metric_index, metric_name in enumerate(
            METRIC_NAMES
        ):

            replicate_row[
                metric_name
            ] = (
                size_bootstrap_values[
                    bootstrap_index,
                    size_index,
                    metric_index,
                ]
            )


        size_replicate_rows.append(
            replicate_row
        )


size_bootstrap_replicates_df = pd.DataFrame(
    size_replicate_rows
)


# ==================================================
# PART B — Weight-profile sensitivity
# ==================================================

weight_cluster_df = (
    coverage_df
    .groupby(
        [
            "n_items",
            "matched_group_id",
            "weight_profile",
        ],
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "size",
        ),

        gap_sum=(
            "optimality_gap_percent",
            "sum",
        ),

        exact_recovery_sum=(
            "exact_recovery_numeric",
            "sum",
        ),

        jaccard_sum=(
            "jaccard_similarity",
            "sum",
        ),

        large_gap_sum=(
            "large_gap_numeric",
            "sum",
        ),

        more_items_sum=(
            "more_items_numeric",
            "sum",
        ),

        selected_count_difference_sum=(
            "selected_count_difference",
            "sum",
        ),
    )
)


# Each group × profile contains:
# 3 regimes × 4 thresholds = 12 rows.
assert (
    weight_cluster_df[
        "comparable_rows"
    ]
    ==
    12
).all()


weight_arrays = {}


for n_items in CANDIDATE_SIZE_ORDER:

    weight_arrays[
        n_items
    ] = {}


    reference_group_ids = None


    for weight_profile in WEIGHT_PROFILE_ORDER:

        current_df = (
            weight_cluster_df[
                (
                    weight_cluster_df[
                        "n_items"
                    ]
                    ==
                    n_items
                )
                &
                (
                    weight_cluster_df[
                        "weight_profile"
                    ]
                    ==
                    weight_profile
                )
            ]
            .sort_values(
                "matched_group_id"
            )
            .reset_index(
                drop=True
            )
        )


        current_group_ids = (
            current_df[
                "matched_group_id"
            ].tolist()
        )


        assert len(
            current_group_ids
        ) == 100


        if reference_group_ids is None:

            reference_group_ids = (
                current_group_ids
            )

        else:

            assert (
                current_group_ids
                ==
                reference_group_ids
            )


        weight_arrays[
            n_items
        ][
            weight_profile
        ] = (
            current_df[
                AGGREGATE_COLUMNS
            ]
            .to_numpy(
                dtype=float
            )
        )


weight_observed_totals = {
    weight_profile: np.zeros(
        len(
            AGGREGATE_COLUMNS
        ),
        dtype=float,
    )
    for weight_profile in WEIGHT_PROFILE_ORDER
}


for n_items in CANDIDATE_SIZE_ORDER:

    for weight_profile in WEIGHT_PROFILE_ORDER:

        weight_observed_totals[
            weight_profile
        ] += (
            weight_arrays[
                n_items
            ][
                weight_profile
            ]
            .sum(
                axis=0
            )
        )


weight_observed_metrics = {
    weight_profile:
        calculate_metrics(
            weight_observed_totals[
                weight_profile
            ]
        )
    for weight_profile in WEIGHT_PROFILE_ORDER
}


weight_descriptive_rows = []


for weight_profile in WEIGHT_PROFILE_ORDER:

    current_metrics = (
        weight_observed_metrics[
            weight_profile
        ]
    )


    weight_descriptive_rows.append(
        {
            "weight_profile":
                weight_profile,

            "comparable_rows":
                int(
                    weight_observed_totals[
                        weight_profile
                    ][0]
                ),

            **{
                metric_name:
                    current_metrics[
                        metric_index
                    ]
                for metric_index, metric_name
                in enumerate(
                    METRIC_NAMES
                )
            },
        }
    )


weight_descriptive_df = pd.DataFrame(
    weight_descriptive_rows
)


# --------------------------------------------------
# Shared paired bootstrap across weight profiles
# --------------------------------------------------

weight_bootstrap_rng = np.random.default_rng(
    SENSITIVITY_BOOTSTRAP_SEED
    +
    10_000
)


weight_bootstrap_differences = np.empty(
    (
        SENSITIVITY_BOOTSTRAP_REPLICATES,
        len(
            WEIGHT_PROFILE_CONTRASTS
        ),
        len(
            METRIC_NAMES
        ),
    ),
    dtype=float,
)


for bootstrap_index in range(
    SENSITIVITY_BOOTSTRAP_REPLICATES
):

    bootstrap_totals = {
        weight_profile: np.zeros(
            len(
                AGGREGATE_COLUMNS
            ),
            dtype=float,
        )
        for weight_profile in WEIGHT_PROFILE_ORDER
    }


    for n_items in CANDIDATE_SIZE_ORDER:

        number_of_groups = (
            weight_arrays[
                n_items
            ][
                REFERENCE_WEIGHT_PROFILE
            ].shape[0]
        )


        sampled_indices = (
            weight_bootstrap_rng.integers(
                low=0,
                high=number_of_groups,
                size=number_of_groups,
            )
        )


        for weight_profile in WEIGHT_PROFILE_ORDER:

            bootstrap_totals[
                weight_profile
            ] += (
                weight_arrays[
                    n_items
                ][
                    weight_profile
                ][
                    sampled_indices
                ]
                .sum(
                    axis=0
                )
            )


    bootstrap_metrics = {
        weight_profile:
            calculate_metrics(
                bootstrap_totals[
                    weight_profile
                ]
            )
        for weight_profile in WEIGHT_PROFILE_ORDER
    }


    for contrast_index, (
        first_profile,
        second_profile,
    ) in enumerate(
        WEIGHT_PROFILE_CONTRASTS
    ):

        weight_bootstrap_differences[
            bootstrap_index,
            contrast_index,
        ] = (
            bootstrap_metrics[
                first_profile
            ]
            -
            bootstrap_metrics[
                second_profile
            ]
        )


weight_contrast_rows = []


for contrast_index, (
    first_profile,
    second_profile,
) in enumerate(
    WEIGHT_PROFILE_CONTRASTS
):

    observed_difference = (
        weight_observed_metrics[
            first_profile
        ]
        -
        weight_observed_metrics[
            second_profile
        ]
    )


    current_bootstrap = (
        weight_bootstrap_differences[
            :,
            contrast_index,
            :,
        ]
    )


    confidence_lower = np.quantile(
        current_bootstrap,
        0.025,
        axis=0,
    )


    confidence_upper = np.quantile(
        current_bootstrap,
        0.975,
        axis=0,
    )


    standard_errors = (
        current_bootstrap.std(
            axis=0,
            ddof=1,
        )
    )


    for metric_index, metric_name in enumerate(
        METRIC_NAMES
    ):

        weight_contrast_rows.append(
            {
                "first_profile":
                    first_profile,

                "second_profile":
                    second_profile,

                "contrast":
                    (
                        f"{first_profile}"
                        f" minus "
                        f"{second_profile}"
                    ),

                "metric":
                    metric_name,

                "estimated_difference":
                    observed_difference[
                        metric_index
                    ],

                "bootstrap_standard_error":
                    standard_errors[
                        metric_index
                    ],

                "ci_95_lower":
                    confidence_lower[
                        metric_index
                    ],

                "ci_95_upper":
                    confidence_upper[
                        metric_index
                    ],

                "ci_excludes_zero":
                    (
                        confidence_lower[
                            metric_index
                        ]
                        >
                        0.0
                    )
                    or
                    (
                        confidence_upper[
                            metric_index
                        ]
                        <
                        0.0
                    ),

                "matched_groups":
                    500,

                "bootstrap_replicates":
                    SENSITIVITY_BOOTSTRAP_REPLICATES,

                "resampling_design":
                    (
                        "shared_across_weight_profiles_"
                        "stratified_by_candidate_size"
                    ),

                "multiple_comparison_adjustment":
                    "none",
            }
        )


weight_contrast_df = pd.DataFrame(
    weight_contrast_rows
)


weight_replicate_rows = []


for bootstrap_index in range(
    SENSITIVITY_BOOTSTRAP_REPLICATES
):

    for contrast_index, (
        first_profile,
        second_profile,
    ) in enumerate(
        WEIGHT_PROFILE_CONTRASTS
    ):

        replicate_row = {
            "bootstrap_replicate":
                bootstrap_index,

            "first_profile":
                first_profile,

            "second_profile":
                second_profile,

            "contrast":
                (
                    f"{first_profile}"
                    f" minus "
                    f"{second_profile}"
                ),
        }


        for metric_index, metric_name in enumerate(
            METRIC_NAMES
        ):

            replicate_row[
                metric_name
            ] = (
                weight_bootstrap_differences[
                    bootstrap_index,
                    contrast_index,
                    metric_index,
                ]
            )


        weight_replicate_rows.append(
            replicate_row
        )


weight_bootstrap_replicates_df = pd.DataFrame(
    weight_replicate_rows
)


# --------------------------------------------------
# Audit
# --------------------------------------------------

size_group_counts = (
    coverage_df
    .groupby(
        "matched_group_id"
    )
    .size()
)


weight_group_counts = (
    coverage_df
    .groupby(
        [
            "matched_group_id",
            "weight_profile",
        ]
    )
    .size()
)


sensitivity_audit_df = pd.DataFrame(
    [
        {
            "benchmark_rows":
                len(
                    coverage_df
                ),

            "matched_groups":
                coverage_df[
                    "matched_group_id"
                ].nunique(),

            "candidate_sizes":
                coverage_df[
                    "n_items"
                ].nunique(),

            "weight_profiles":
                coverage_df[
                    "weight_profile"
                ].nunique(),

            "minimum_rows_per_matched_group":
                int(
                    size_group_counts.min()
                ),

            "maximum_rows_per_matched_group":
                int(
                    size_group_counts.max()
                ),

            "minimum_rows_per_group_profile":
                int(
                    weight_group_counts.min()
                ),

            "maximum_rows_per_group_profile":
                int(
                    weight_group_counts.max()
                ),

            "negative_gap_rows":
                int(
                    (
                        coverage_df[
                            "optimality_gap_percent"
                        ]
                        <
                        -1e-8
                    ).sum()
                ),

            "bootstrap_replicates":
                SENSITIVITY_BOOTSTRAP_REPLICATES,

            "candidate_size_design":
                (
                    "independent generated groups "
                    "within each size"
                ),

            "weight_profile_design":
                (
                    "paired within matched generated groups"
                ),

            "confidence_intervals":
                "unadjusted_percentile_95",
        }
    ]
)


assert (
    size_group_counts
    ==
    48
).all()


assert (
    weight_group_counts
    ==
    12
).all()


assert sensitivity_audit_df.loc[
    0,
    "negative_gap_rows",
] == 0


# --------------------------------------------------
# Save new Step-133 outputs
# --------------------------------------------------

size_summary_df.to_csv(
    SIZE_SUMMARY_PATH,
    index=False,
)


size_bootstrap_replicates_df.to_csv(
    SIZE_BOOTSTRAP_REPLICATES_PATH,
    index=False,
)


weight_descriptive_df.to_csv(
    WEIGHT_DESCRIPTIVE_PATH,
    index=False,
)


weight_contrast_df.to_csv(
    WEIGHT_CONTRAST_PATH,
    index=False,
)


weight_bootstrap_replicates_df.to_csv(
    WEIGHT_BOOTSTRAP_REPLICATES_PATH,
    index=False,
)


sensitivity_audit_df.to_csv(
    SENSITIVITY_AUDIT_PATH,
    index=False,
)


# --------------------------------------------------
# Readable displays
# --------------------------------------------------

size_display_df = size_summary_df.copy()


size_display_df[
    "estimate_and_ci"
] = (
    size_display_df[
        "estimate"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    size_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    size_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


weight_display_df = weight_contrast_df.copy()


weight_display_df[
    "difference_and_ci"
] = (
    weight_display_df[
        "estimated_difference"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    weight_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    weight_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


print(
    "======================================"
)

print(
    "COVERAGE SIZE/WEIGHT SENSITIVITY AUDIT"
)

print(
    "======================================"
)


print(
    sensitivity_audit_df.to_string(
        index=False
    )
)


print(
    "\n======================================"
)

print(
    "COVERAGE RESULTS BY CANDIDATE-SET SIZE"
)

print(
    "======================================"
)


print(
    size_display_df[
        [
            "n_items",
            "metric",
            "estimate_and_ci",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\n======================================"
)

print(
    "COVERAGE RESULTS BY WEIGHT PROFILE"
)

print(
    "======================================"
)


print(
    weight_descriptive_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.4f}",
    )
)


print(
    "\n======================================"
)

print(
    "PAIRED WEIGHT-PROFILE CONTRASTS"
)

print(
    "======================================"
)


print(
    weight_display_df[
        [
            "first_profile",
            "second_profile",
            "metric",
            "difference_and_ci",
            "ci_excludes_zero",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\nCandidate-size summary saved to:"
)

print(
    SIZE_SUMMARY_PATH
)


print(
    "\nWeight-profile descriptive summary saved to:"
)

print(
    WEIGHT_DESCRIPTIVE_PATH
)


print(
    "\nPaired weight-profile contrasts saved to:"
)

print(
    WEIGHT_CONTRAST_PATH
)


print(
    "\nSensitivity audit saved to:"
)

print(
    SENSITIVITY_AUDIT_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Step 134 — Corrected product-form versus
# pool-normalized coverage utility-family comparison
# --------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------
# Configuration
# --------------------------------------------------

FAMILY_BOOTSTRAP_REPLICATES = 5000

FAMILY_BOOTSTRAP_SEED = (
    globals().get(
        "GLOBAL_SEED",
        2026,
    )
    + 134
)


COMMON_THRESHOLDS = [
    0.70,
    0.80,
    0.90,
]


CANDIDATE_SIZE_ORDER = [
    8,
    10,
    12,
    14,
    16,
]


REGIME_ORDER = [
    "independent",
    "utility_burden_alignment",
    "utility_burden_conflict",
]


UTILITY_FAMILY_ORDER = [
    "product_form",
    "pool_normalized_coverage",
]


SCOPE_ORDER = [
    "overall",
    *REGIME_ORDER,
]


METRIC_NAMES = [
    "mean_optimality_gap_percent",
    "exact_cost_recovery_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "more_items_percent",
    "mean_selected_count_difference",
    "within_1_percent",
    "within_5_percent",
    "within_10_percent",
]


AGGREGATE_COLUMNS = [
    "comparable_rows",
    "gap_sum",
    "exact_recovery_sum",
    "jaccard_sum",
    "large_gap_sum",
    "more_items_sum",
    "selected_count_difference_sum",
    "within_1_percent_sum",
    "within_5_percent_sum",
    "within_10_percent_sum",
]


EXACT_RECOVERY_TOLERANCE_PERCENT = 1e-10


# --------------------------------------------------
# Paths
# --------------------------------------------------

if "RESULTS_DIR" not in globals():

    RESULTS_DIR = Path(
        os.path.join(PROJECT_ROOT, "05_Results")
    )


if "TABLES_DIR" not in globals():

    TABLES_DIR = Path(
        os.path.join(PROJECT_ROOT, "07_Tables")
    )


RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


PRODUCT_RESULTS_PATH = (
    RESULTS_DIR
    / "exact_vs_greedy_main_benchmark_v2_marginal_preserving.csv"
)


COVERAGE_RESULTS_PATH = (
    RESULTS_DIR
    / (
        "coverage_exact_vs_greedy_"
        "main_benchmark_v1_pool_normalized.csv"
    )
)


# New filenames: earlier results will not be overwritten.
FAMILY_AUDIT_PATH = (
    RESULTS_DIR
    / "utility_family_comparison_audit_fixed.csv"
)


FAMILY_DESCRIPTIVE_PATH = (
    TABLES_DIR
    / "utility_family_common_threshold_descriptive_summary_fixed.csv"
)


FAMILY_CONTRAST_PATH = (
    TABLES_DIR
    / "coverage_minus_product_family_bootstrap_contrasts_fixed.csv"
)


FAMILY_BOOTSTRAP_PATH = (
    RESULTS_DIR
    / "coverage_minus_product_family_bootstrap_replicates_fixed.csv"
)


# --------------------------------------------------
# Safe saving
# --------------------------------------------------

def save_csv_without_overwriting(
    data_frame,
    output_path,
):
    """
    Save a CSV only if the target file does not already
    exist.
    """

    if output_path.exists():

        raise FileExistsError(
            "Output already exists and was not overwritten:\n"
            f"{output_path}"
        )


    data_frame.to_csv(
        output_path,
        index=False,
    )


# --------------------------------------------------
# Load and harmonize one benchmark family
# --------------------------------------------------

def load_benchmark_family(
    file_path,
    utility_family,
):
    """
    Load one benchmark and harmonize its schema.

    The older product-form benchmark does not store
    selected_count_difference, so it is derived from:

        greedy_selected_count - exact_selected_count
    """

    if not file_path.exists():

        raise FileNotFoundError(
            f"Benchmark file not found:\n{file_path}"
        )


    data_frame = pd.read_csv(
        file_path,
        low_memory=False,
    )


    count_columns = [
        "exact_selected_count",
        "greedy_selected_count",
    ]


    missing_count_columns = [
        column_name
        for column_name in count_columns
        if column_name not in data_frame.columns
    ]


    if missing_count_columns:

        raise KeyError(
            f"{utility_family} cannot derive selected-count "
            "difference because these columns are missing: "
            + ", ".join(
                missing_count_columns
            )
        )


    for column_name in count_columns:

        data_frame[
            column_name
        ] = pd.to_numeric(
            data_frame[
                column_name
            ],
            errors="coerce",
        )


    selected_count_difference_was_derived = False


    if (
        "selected_count_difference"
        not in data_frame.columns
    ):

        data_frame[
            "selected_count_difference"
        ] = (
            data_frame[
                "greedy_selected_count"
            ]
            -
            data_frame[
                "exact_selected_count"
            ]
        )


        selected_count_difference_was_derived = True


    required_columns = [
        "matched_group_id",
        "n_items",
        "regime",
        "tau",
        "weight_profile",
        "exact_status",
        "greedy_status",
        "exact_selected_count",
        "greedy_selected_count",
        "optimality_gap_percent",
        "jaccard_similarity",
        "selected_count_difference",
    ]


    missing_columns = [
        column_name
        for column_name in required_columns
        if column_name not in data_frame.columns
    ]


    if missing_columns:

        raise KeyError(
            f"{utility_family} is missing columns: "
            + ", ".join(
                missing_columns
            )
        )


    numeric_columns = [
        "n_items",
        "tau",
        "exact_selected_count",
        "greedy_selected_count",
        "optimality_gap_percent",
        "jaccard_similarity",
        "selected_count_difference",
    ]


    for column_name in numeric_columns:

        data_frame[
            column_name
        ] = pd.to_numeric(
            data_frame[
                column_name
            ],
            errors="coerce",
        )


    if data_frame[
        numeric_columns
    ].isna().any().any():

        problematic_columns = (
            data_frame[
                numeric_columns
            ]
            .columns[
                data_frame[
                    numeric_columns
                ]
                .isna()
                .any()
            ]
            .tolist()
        )


        raise ValueError(
            f"{utility_family} contains missing or "
            "non-numeric values in: "
            + ", ".join(
                problematic_columns
            )
        )


    reconstructed_count_difference = (
        data_frame[
            "greedy_selected_count"
        ]
        -
        data_frame[
            "exact_selected_count"
        ]
    )


    maximum_count_difference_error = float(
        (
            data_frame[
                "selected_count_difference"
            ]
            -
            reconstructed_count_difference
        )
        .abs()
        .max()
    )


    if maximum_count_difference_error > 1e-12:

        raise ValueError(
            f"{utility_family} has inconsistent selected "
            "count differences."
        )


    data_frame[
        "utility_family"
    ] = utility_family


    data_frame[
        "selected_count_difference_was_derived"
    ] = selected_count_difference_was_derived


    data_frame[
        "exact_recovery_numeric"
    ] = (
        data_frame[
            "optimality_gap_percent"
        ]
        .abs()
        <=
        EXACT_RECOVERY_TOLERANCE_PERCENT
    ).astype(int)


    data_frame[
        "large_gap_numeric"
    ] = (
        data_frame[
            "optimality_gap_percent"
        ]
        >
        10.0
    ).astype(int)


    data_frame[
        "more_items_numeric"
    ] = (
        data_frame[
            "selected_count_difference"
        ]
        >
        0
    ).astype(int)


    data_frame[
        "within_1_percent_numeric"
    ] = (
        data_frame[
            "optimality_gap_percent"
        ]
        <=
        1.0
    ).astype(int)


    data_frame[
        "within_5_percent_numeric"
    ] = (
        data_frame[
            "optimality_gap_percent"
        ]
        <=
        5.0
    ).astype(int)


    data_frame[
        "within_10_percent_numeric"
    ] = (
        data_frame[
            "optimality_gap_percent"
        ]
        <=
        10.0
    ).astype(int)


    schema_metadata = {
        "utility_family":
            utility_family,

        "rows":
            len(
                data_frame
            ),

        "selected_count_difference_was_derived":
            selected_count_difference_was_derived,

        "maximum_count_difference_error":
            maximum_count_difference_error,
    }


    return (
        data_frame,
        schema_metadata,
    )


# --------------------------------------------------
# Load both utility families
# --------------------------------------------------

(
    product_df,
    product_schema_metadata,
) = load_benchmark_family(
    file_path=
        PRODUCT_RESULTS_PATH,

    utility_family=
        "product_form",
)


(
    coverage_df,
    coverage_schema_metadata,
) = load_benchmark_family(
    file_path=
        COVERAGE_RESULTS_PATH,

    utility_family=
        "pool_normalized_coverage",
)


# --------------------------------------------------
# Initial integrity checks
# --------------------------------------------------

assert len(
    product_df
) == 24000


assert len(
    coverage_df
) == 24000


assert (
    product_df[
        "matched_group_id"
    ].nunique()
    ==
    500
)


assert (
    coverage_df[
        "matched_group_id"
    ].nunique()
    ==
    500
)


assert set(
    product_df[
        "regime"
    ].unique()
) == set(
    REGIME_ORDER
)


assert set(
    coverage_df[
        "regime"
    ].unique()
) == set(
    REGIME_ORDER
)


# --------------------------------------------------
# Restrict to common thresholds
# --------------------------------------------------

product_common_df = (
    product_df[
        product_df[
            "tau"
        ].isin(
            COMMON_THRESHOLDS
        )
    ]
    .copy()
)


coverage_common_df = (
    coverage_df[
        coverage_df[
            "tau"
        ].isin(
            COMMON_THRESHOLDS
        )
    ]
    .copy()
)


expected_rows_per_family = (
    500
    *
    3
    *
    len(
        COMMON_THRESHOLDS
    )
    *
    4
)


assert len(
    product_common_df
) == expected_rows_per_family


assert len(
    coverage_common_df
) == expected_rows_per_family


product_exact_nonoptimal_rows = int(
    (
        product_common_df[
            "exact_status"
        ]
        !=
        "OPTIMAL"
    ).sum()
)


product_greedy_infeasible_rows = int(
    (
        product_common_df[
            "greedy_status"
        ]
        !=
        "FEASIBLE"
    ).sum()
)


coverage_exact_nonoptimal_rows = int(
    (
        coverage_common_df[
            "exact_status"
        ]
        !=
        "OPTIMAL"
    ).sum()
)


coverage_greedy_infeasible_rows = int(
    (
        coverage_common_df[
            "greedy_status"
        ]
        !=
        "FEASIBLE"
    ).sum()
)


assert product_exact_nonoptimal_rows == 0
assert product_greedy_infeasible_rows == 0
assert coverage_exact_nonoptimal_rows == 0
assert coverage_greedy_infeasible_rows == 0


combined_df = pd.concat(
    [
        product_common_df,
        coverage_common_df,
    ],
    ignore_index=True,
)


# --------------------------------------------------
# Metric calculation
# --------------------------------------------------

def calculate_family_metrics(
    totals,
):
    """
    Convert sufficient-statistic totals into common
    Exact-relative heuristic-quality metrics.
    """

    comparable_rows = totals[0]


    if comparable_rows <= 0:

        return np.full(
            len(
                METRIC_NAMES
            ),
            np.nan,
        )


    return np.array(
        [
            totals[1]
            /
            comparable_rows,

            100.0
            *
            totals[2]
            /
            comparable_rows,

            totals[3]
            /
            comparable_rows,

            100.0
            *
            totals[4]
            /
            comparable_rows,

            100.0
            *
            totals[5]
            /
            comparable_rows,

            totals[6]
            /
            comparable_rows,

            100.0
            *
            totals[7]
            /
            comparable_rows,

            100.0
            *
            totals[8]
            /
            comparable_rows,

            100.0
            *
            totals[9]
            /
            comparable_rows,
        ],
        dtype=float,
    )


# --------------------------------------------------
# Descriptive summaries
# --------------------------------------------------

descriptive_rows = []


for utility_family in UTILITY_FAMILY_ORDER:

    family_df = combined_df[
        combined_df[
            "utility_family"
        ]
        ==
        utility_family
    ]


    for scope in SCOPE_ORDER:

        if scope == "overall":

            current_df = family_df

        else:

            current_df = family_df[
                family_df[
                    "regime"
                ]
                ==
                scope
            ]


        gap_values = (
            current_df[
                "optimality_gap_percent"
            ]
            .to_numpy(
                dtype=float
            )
        )


        descriptive_rows.append(
            {
                "utility_family":
                    utility_family,

                "scope":
                    scope,

                "rows":
                    len(
                        current_df
                    ),

                "matched_groups":
                    current_df[
                        "matched_group_id"
                    ].nunique(),

                "mean_optimality_gap_percent":
                    gap_values.mean(),

                "median_optimality_gap_percent":
                    np.median(
                        gap_values
                    ),

                "p90_optimality_gap_percent":
                    np.quantile(
                        gap_values,
                        0.90,
                    ),

                "p95_optimality_gap_percent":
                    np.quantile(
                        gap_values,
                        0.95,
                    ),

                "exact_cost_recovery_percent":
                    100.0
                    *
                    current_df[
                        "exact_recovery_numeric"
                    ].mean(),

                "mean_jaccard_similarity":
                    current_df[
                        "jaccard_similarity"
                    ].mean(),

                "large_gap_percent":
                    100.0
                    *
                    current_df[
                        "large_gap_numeric"
                    ].mean(),

                "more_items_percent":
                    100.0
                    *
                    current_df[
                        "more_items_numeric"
                    ].mean(),

                "mean_selected_count_difference":
                    current_df[
                        "selected_count_difference"
                    ].mean(),

                "within_1_percent":
                    100.0
                    *
                    current_df[
                        "within_1_percent_numeric"
                    ].mean(),

                "within_5_percent":
                    100.0
                    *
                    current_df[
                        "within_5_percent_numeric"
                    ].mean(),

                "within_10_percent":
                    100.0
                    *
                    current_df[
                        "within_10_percent_numeric"
                    ].mean(),
            }
        )


family_descriptive_df = pd.DataFrame(
    descriptive_rows
)


# --------------------------------------------------
# Group-level sufficient statistics
#
# Each family/group/regime has:
# 3 thresholds × 4 profiles = 12 rows.
# --------------------------------------------------

cluster_df = (
    combined_df
    .groupby(
        [
            "utility_family",
            "n_items",
            "matched_group_id",
            "regime",
        ],
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "size",
        ),

        gap_sum=(
            "optimality_gap_percent",
            "sum",
        ),

        exact_recovery_sum=(
            "exact_recovery_numeric",
            "sum",
        ),

        jaccard_sum=(
            "jaccard_similarity",
            "sum",
        ),

        large_gap_sum=(
            "large_gap_numeric",
            "sum",
        ),

        more_items_sum=(
            "more_items_numeric",
            "sum",
        ),

        selected_count_difference_sum=(
            "selected_count_difference",
            "sum",
        ),

        within_1_percent_sum=(
            "within_1_percent_numeric",
            "sum",
        ),

        within_5_percent_sum=(
            "within_5_percent_numeric",
            "sum",
        ),

        within_10_percent_sum=(
            "within_10_percent_numeric",
            "sum",
        ),
    )
)


assert (
    cluster_df[
        "comparable_rows"
    ]
    ==
    12
).all()


# --------------------------------------------------
# Aligned arrays by family and candidate size
# --------------------------------------------------

arrays_by_family_size = {}


for utility_family in UTILITY_FAMILY_ORDER:

    arrays_by_family_size[
        utility_family
    ] = {}


    for n_items in CANDIDATE_SIZE_ORDER:

        current_size_df = (
            cluster_df[
                (
                    cluster_df[
                        "utility_family"
                    ]
                    ==
                    utility_family
                )
                &
                (
                    cluster_df[
                        "n_items"
                    ]
                    ==
                    n_items
                )
            ]
            .copy()
        )


        group_ids = sorted(
            current_size_df[
                "matched_group_id"
            ].unique()
        )


        assert len(
            group_ids
        ) == 100


        regime_arrays = []


        for regime in REGIME_ORDER:

            regime_df = (
                current_size_df[
                    current_size_df[
                        "regime"
                    ]
                    ==
                    regime
                ]
                .set_index(
                    "matched_group_id"
                )
                .loc[
                    group_ids
                ]
                .reset_index()
            )


            regime_arrays.append(
                regime_df[
                    AGGREGATE_COLUMNS
                ]
                .to_numpy(
                    dtype=float
                )
            )


        arrays_by_family_size[
            utility_family
        ][
            n_items
        ] = np.stack(
            regime_arrays,
            axis=1,
        )


# --------------------------------------------------
# Scope helper
# --------------------------------------------------

def totals_for_scope(
    totals_by_regime,
    scope,
):
    """
    totals_by_regime shape:
        regimes × sufficient statistics.
    """

    if scope == "overall":

        return totals_by_regime.sum(
            axis=0
        )


    regime_index = REGIME_ORDER.index(
        scope
    )


    return totals_by_regime[
        regime_index
    ]


# --------------------------------------------------
# Observed family metrics
# --------------------------------------------------

observed_metrics = {
    utility_family: {}
    for utility_family in UTILITY_FAMILY_ORDER
}


for utility_family in UTILITY_FAMILY_ORDER:

    totals_by_regime = np.zeros(
        (
            len(
                REGIME_ORDER
            ),
            len(
                AGGREGATE_COLUMNS
            ),
        ),
        dtype=float,
    )


    for n_items in CANDIDATE_SIZE_ORDER:

        totals_by_regime += (
            arrays_by_family_size[
                utility_family
            ][
                n_items
            ]
            .sum(
                axis=0
            )
        )


    for scope in SCOPE_ORDER:

        observed_metrics[
            utility_family
        ][
            scope
        ] = calculate_family_metrics(
            totals_for_scope(
                totals_by_regime,
                scope,
            )
        )


# --------------------------------------------------
# Independent bootstrap between utility families
#
# The two benchmark families were independently
# generated. Within each family, the same resampled
# group indices are shared across regimes.
# --------------------------------------------------

bootstrap_rng = np.random.default_rng(
    FAMILY_BOOTSTRAP_SEED
)


bootstrap_differences = np.empty(
    (
        FAMILY_BOOTSTRAP_REPLICATES,
        len(
            SCOPE_ORDER
        ),
        len(
            METRIC_NAMES
        ),
    ),
    dtype=float,
)


for bootstrap_index in range(
    FAMILY_BOOTSTRAP_REPLICATES
):

    bootstrap_family_totals = {
        utility_family: np.zeros(
            (
                len(
                    REGIME_ORDER
                ),
                len(
                    AGGREGATE_COLUMNS
                ),
            ),
            dtype=float,
        )
        for utility_family in UTILITY_FAMILY_ORDER
    }


    for utility_family in UTILITY_FAMILY_ORDER:

        for n_items in CANDIDATE_SIZE_ORDER:

            current_array = (
                arrays_by_family_size[
                    utility_family
                ][
                    n_items
                ]
            )


            number_of_groups = (
                current_array.shape[0]
            )


            sampled_indices = (
                bootstrap_rng.integers(
                    low=0,
                    high=number_of_groups,
                    size=number_of_groups,
                )
            )


            bootstrap_family_totals[
                utility_family
            ] += (
                current_array[
                    sampled_indices
                ]
                .sum(
                    axis=0
                )
            )


    for scope_index, scope in enumerate(
        SCOPE_ORDER
    ):

        coverage_metrics = (
            calculate_family_metrics(
                totals_for_scope(
                    bootstrap_family_totals[
                        "pool_normalized_coverage"
                    ],
                    scope,
                )
            )
        )


        product_metrics = (
            calculate_family_metrics(
                totals_for_scope(
                    bootstrap_family_totals[
                        "product_form"
                    ],
                    scope,
                )
            )
        )


        bootstrap_differences[
            bootstrap_index,
            scope_index,
        ] = (
            coverage_metrics
            -
            product_metrics
        )


# --------------------------------------------------
# Coverage-minus-product contrasts
# --------------------------------------------------

contrast_rows = []


for scope_index, scope in enumerate(
    SCOPE_ORDER
):

    observed_difference = (
        observed_metrics[
            "pool_normalized_coverage"
        ][
            scope
        ]
        -
        observed_metrics[
            "product_form"
        ][
            scope
        ]
    )


    current_bootstrap = (
        bootstrap_differences[
            :,
            scope_index,
            :,
        ]
    )


    confidence_lower = np.quantile(
        current_bootstrap,
        0.025,
        axis=0,
    )


    confidence_upper = np.quantile(
        current_bootstrap,
        0.975,
        axis=0,
    )


    standard_errors = (
        current_bootstrap.std(
            axis=0,
            ddof=1,
        )
    )


    for metric_index, metric_name in enumerate(
        METRIC_NAMES
    ):

        contrast_rows.append(
            {
                "scope":
                    scope,

                "contrast":
                    (
                        "pool_normalized_coverage "
                        "minus product_form"
                    ),

                "metric":
                    metric_name,

                "estimated_difference":
                    observed_difference[
                        metric_index
                    ],

                "bootstrap_standard_error":
                    standard_errors[
                        metric_index
                    ],

                "ci_95_lower":
                    confidence_lower[
                        metric_index
                    ],

                "ci_95_upper":
                    confidence_upper[
                        metric_index
                    ],

                "ci_excludes_zero":
                    (
                        confidence_lower[
                            metric_index
                        ]
                        >
                        0.0
                    )
                    or
                    (
                        confidence_upper[
                            metric_index
                        ]
                        <
                        0.0
                    ),

                "product_matched_groups":
                    500,

                "coverage_matched_groups":
                    500,

                "common_thresholds":
                    repr(
                        COMMON_THRESHOLDS
                    ),

                "bootstrap_replicates":
                    FAMILY_BOOTSTRAP_REPLICATES,

                "resampling_design":
                    (
                        "independent_between_utility_families;"
                        "shared_across_regimes_within_family;"
                        "stratified_by_candidate_size"
                    ),

                "multiple_comparison_adjustment":
                    "none",
            }
        )


family_contrast_df = pd.DataFrame(
    contrast_rows
)


# --------------------------------------------------
# Store bootstrap replicates
# --------------------------------------------------

replicate_rows = []


for bootstrap_index in range(
    FAMILY_BOOTSTRAP_REPLICATES
):

    for scope_index, scope in enumerate(
        SCOPE_ORDER
    ):

        replicate_row = {
            "bootstrap_replicate":
                bootstrap_index,

            "scope":
                scope,
        }


        for metric_index, metric_name in enumerate(
            METRIC_NAMES
        ):

            replicate_row[
                metric_name
            ] = (
                bootstrap_differences[
                    bootstrap_index,
                    scope_index,
                    metric_index,
                ]
            )


        replicate_rows.append(
            replicate_row
        )


family_bootstrap_df = pd.DataFrame(
    replicate_rows
)


# --------------------------------------------------
# Audit
# --------------------------------------------------

family_group_counts = (
    combined_df
    .groupby(
        [
            "utility_family",
            "matched_group_id",
        ]
    )
    .size()
)


family_regime_group_counts = (
    combined_df
    .groupby(
        [
            "utility_family",
            "matched_group_id",
            "regime",
        ]
    )
    .size()
)


family_audit_df = pd.DataFrame(
    [
        {
            "product_original_rows":
                len(
                    product_df
                ),

            "coverage_original_rows":
                len(
                    coverage_df
                ),

            "product_common_rows":
                len(
                    product_common_df
                ),

            "coverage_common_rows":
                len(
                    coverage_common_df
                ),

            "expected_common_rows_per_family":
                expected_rows_per_family,

            "product_matched_groups":
                product_common_df[
                    "matched_group_id"
                ].nunique(),

            "coverage_matched_groups":
                coverage_common_df[
                    "matched_group_id"
                ].nunique(),

            "common_thresholds":
                repr(
                    COMMON_THRESHOLDS
                ),

            "product_selected_count_difference_derived":
                product_schema_metadata[
                    "selected_count_difference_was_derived"
                ],

            "coverage_selected_count_difference_derived":
                coverage_schema_metadata[
                    "selected_count_difference_was_derived"
                ],

            "product_maximum_count_difference_error":
                product_schema_metadata[
                    "maximum_count_difference_error"
                ],

            "coverage_maximum_count_difference_error":
                coverage_schema_metadata[
                    "maximum_count_difference_error"
                ],

            "minimum_rows_per_family_group":
                int(
                    family_group_counts.min()
                ),

            "maximum_rows_per_family_group":
                int(
                    family_group_counts.max()
                ),

            "minimum_rows_per_family_group_regime":
                int(
                    family_regime_group_counts.min()
                ),

            "maximum_rows_per_family_group_regime":
                int(
                    family_regime_group_counts.max()
                ),

            "product_exact_nonoptimal_rows":
                product_exact_nonoptimal_rows,

            "product_greedy_infeasible_rows":
                product_greedy_infeasible_rows,

            "coverage_exact_nonoptimal_rows":
                coverage_exact_nonoptimal_rows,

            "coverage_greedy_infeasible_rows":
                coverage_greedy_infeasible_rows,

            "bootstrap_replicates":
                FAMILY_BOOTSTRAP_REPLICATES,

            "confidence_intervals":
                "unadjusted_percentile_95",
        }
    ]
)


assert (
    family_group_counts
    ==
    36
).all()


assert (
    family_regime_group_counts
    ==
    12
).all()


# --------------------------------------------------
# Save without overwriting
# --------------------------------------------------

save_csv_without_overwriting(
    family_audit_df,
    FAMILY_AUDIT_PATH,
)


save_csv_without_overwriting(
    family_descriptive_df,
    FAMILY_DESCRIPTIVE_PATH,
)


save_csv_without_overwriting(
    family_contrast_df,
    FAMILY_CONTRAST_PATH,
)


save_csv_without_overwriting(
    family_bootstrap_df,
    FAMILY_BOOTSTRAP_PATH,
)


# --------------------------------------------------
# Readable contrast display
# --------------------------------------------------

contrast_display_df = (
    family_contrast_df.copy()
)


contrast_display_df[
    "difference_and_ci"
] = (
    contrast_display_df[
        "estimated_difference"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    contrast_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    contrast_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


# --------------------------------------------------
# Printed report
# --------------------------------------------------

print(
    "======================================"
)

print(
    "UTILITY-FAMILY COMPARISON AUDIT"
)

print(
    "======================================"
)


print(
    family_audit_df.to_string(
        index=False
    )
)


print(
    "\n======================================"
)

print(
    "UTILITY-FAMILY DESCRIPTIVE RESULTS"
)

print(
    "COMMON THRESHOLDS: 0.70, 0.80, 0.90"
)

print(
    "======================================"
)


print(
    family_descriptive_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.4f}",
    )
)


print(
    "\n======================================"
)

print(
    "COVERAGE-MINUS-PRODUCT FAMILY CONTRASTS"
)

print(
    "======================================"
)


print(
    contrast_display_df[
        [
            "scope",
            "metric",
            "difference_and_ci",
            "ci_excludes_zero",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\nComparison audit saved to:"
)

print(
    FAMILY_AUDIT_PATH
)


print(
    "\nDescriptive summary saved to:"
)

print(
    FAMILY_DESCRIPTIVE_PATH
)


print(
    "\nFamily contrasts saved to:"
)

print(
    FAMILY_CONTRAST_PATH
)


print(
    "\nBootstrap replicates saved to:"
)

print(
    FAMILY_BOOTSTRAP_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Step 134A — Product/coverage feasibility and
# missing-value audit before utility-family comparison
# --------------------------------------------------

from pathlib import Path

import pandas as pd


RESULTS_DIR = Path(
    os.path.join(PROJECT_ROOT, "05_Results")
)


PRODUCT_RESULTS_PATH = (
    RESULTS_DIR
    / "exact_vs_greedy_main_benchmark_v2_marginal_preserving.csv"
)


COVERAGE_RESULTS_PATH = (
    RESULTS_DIR
    / "coverage_exact_vs_greedy_main_benchmark_v1_pool_normalized.csv"
)


product_df = pd.read_csv(
    PRODUCT_RESULTS_PATH,
    low_memory=False,
)


coverage_df = pd.read_csv(
    COVERAGE_RESULTS_PATH,
    low_memory=False,
)


for data_frame in [
    product_df,
    coverage_df,
]:

    for column_name in [
        "tau",
        "optimality_gap_percent",
        "jaccard_similarity",
        "exact_selected_count",
        "greedy_selected_count",
    ]:

        if column_name in data_frame.columns:

            data_frame[column_name] = pd.to_numeric(
                data_frame[column_name],
                errors="coerce",
            )


def create_audit(
    data_frame,
    utility_family,
):

    audit_df = (
        data_frame
        .groupby(
            [
                "tau",
                "exact_status",
                "greedy_status",
            ],
            dropna=False,
            as_index=False,
        )
        .agg(
            rows=(
                "tau",
                "size",
            ),

            missing_optimality_gap=(
                "optimality_gap_percent",
                lambda values:
                    int(
                        values.isna().sum()
                    ),
            ),

            missing_jaccard=(
                "jaccard_similarity",
                lambda values:
                    int(
                        values.isna().sum()
                    ),
            ),
        )
    )


    audit_df.insert(
        0,
        "utility_family",
        utility_family,
    )


    return audit_df


audit_df = pd.concat(
    [
        create_audit(
            product_df,
            "product_form",
        ),

        create_audit(
            coverage_df,
            "pool_normalized_coverage",
        ),
    ],
    ignore_index=True,
)


product_comparable_df = product_df[
    (
        product_df["exact_status"]
        ==
        "OPTIMAL"
    )
    &
    (
        product_df["greedy_status"]
        ==
        "FEASIBLE"
    )
    &
    product_df[
        "optimality_gap_percent"
    ].notna()
    &
    product_df[
        "jaccard_similarity"
    ].notna()
].copy()


coverage_comparable_df = coverage_df[
    (
        coverage_df["exact_status"]
        ==
        "OPTIMAL"
    )
    &
    (
        coverage_df["greedy_status"]
        ==
        "FEASIBLE"
    )
    &
    coverage_df[
        "optimality_gap_percent"
    ].notna()
    &
    coverage_df[
        "jaccard_similarity"
    ].notna()
].copy()


comparable_counts_df = pd.concat(
    [
        (
            product_comparable_df
            .groupby(
                "tau",
                as_index=False,
            )
            .size()
            .rename(
                columns={
                    "size":
                        "comparable_rows"
                }
            )
            .assign(
                utility_family=
                    "product_form"
            )
        ),

        (
            coverage_comparable_df
            .groupby(
                "tau",
                as_index=False,
            )
            .size()
            .rename(
                columns={
                    "size":
                        "comparable_rows"
                }
            )
            .assign(
                utility_family=
                    "pool_normalized_coverage"
            )
        ),
    ],
    ignore_index=True,
)


comparable_counts_df = (
    comparable_counts_df[
        [
            "utility_family",
            "tau",
            "comparable_rows",
        ]
    ]
    .sort_values(
        [
            "utility_family",
            "tau",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "======================================"
)

print(
    "UTILITY-FAMILY STATUS AND MISSINGNESS AUDIT"
)

print(
    "======================================"
)


print(
    audit_df.to_string(
        index=False
    )
)


print(
    "\n======================================"
)

print(
    "COMPARABLE ROWS BY THRESHOLD"
)

print(
    "======================================"
)


print(
    comparable_counts_df.to_string(
        index=False
    )
)


print(
    "\nProduct selected_count_difference stored:",
    (
        "selected_count_difference"
        in
        product_df.columns
    ),
)


print(
    "Coverage selected_count_difference stored:",
    (
        "selected_count_difference"
        in
        coverage_df.columns
    ),
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Step 134B — Corrected product-form versus
# pool-normalized coverage comparison on the common
# fully feasible threshold grid
# --------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------
# Configuration
# --------------------------------------------------

BOOTSTRAP_REPLICATES = 5000

BOOTSTRAP_SEED = (
    globals().get(
        "GLOBAL_SEED",
        2026,
    )
    + 134
)


COMMON_THRESHOLDS = [
    0.70,
    0.80,
    0.90,
]


CANDIDATE_SIZES = [
    8,
    10,
    12,
    14,
    16,
]


REGIMES = [
    "independent",
    "utility_burden_alignment",
    "utility_burden_conflict",
]


FAMILIES = [
    "product_form",
    "pool_normalized_coverage",
]


SCOPES = [
    "overall",
    *REGIMES,
]


METRIC_NAMES = [
    "mean_optimality_gap_percent",
    "exact_cost_recovery_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "more_items_percent",
    "mean_selected_count_difference",
    "within_1_percent",
    "within_5_percent",
    "within_10_percent",
]


AGGREGATE_COLUMNS = [
    "comparable_rows",
    "gap_sum",
    "exact_recovery_sum",
    "jaccard_sum",
    "large_gap_sum",
    "more_items_sum",
    "count_difference_sum",
    "within_1_sum",
    "within_5_sum",
    "within_10_sum",
]


EXACT_RECOVERY_TOLERANCE = 1e-10


# --------------------------------------------------
# Paths
# --------------------------------------------------

RESULTS_DIR = Path(
    os.path.join(PROJECT_ROOT, "05_Results")
)


TABLES_DIR = Path(
    os.path.join(PROJECT_ROOT, "07_Tables")
)


RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


PRODUCT_PATH = (
    RESULTS_DIR
    / "exact_vs_greedy_main_benchmark_v2_marginal_preserving.csv"
)


COVERAGE_PATH = (
    RESULTS_DIR
    / "coverage_exact_vs_greedy_main_benchmark_v1_pool_normalized.csv"
)


AUDIT_PATH = (
    RESULTS_DIR
    / "utility_family_comparison_audit_v2.csv"
)


DESCRIPTIVE_PATH = (
    TABLES_DIR
    / "utility_family_common_threshold_descriptive_v2.csv"
)


CONTRAST_PATH = (
    TABLES_DIR
    / "coverage_minus_product_family_contrasts_v2.csv"
)


BOOTSTRAP_PATH = (
    RESULTS_DIR
    / "coverage_minus_product_family_bootstrap_v2.csv"
)


# --------------------------------------------------
# Prevent accidental overwriting
# --------------------------------------------------

for output_path in [
    AUDIT_PATH,
    DESCRIPTIVE_PATH,
    CONTRAST_PATH,
    BOOTSTRAP_PATH,
]:

    if output_path.exists():

        raise FileExistsError(
            "Output already exists and was not overwritten:\n"
            f"{output_path}"
        )


# --------------------------------------------------
# Load and harmonize one family
# --------------------------------------------------

def load_family(
    file_path,
    family_name,
):
    """
    Load one benchmark, derive selected-count
    difference when missing, and retain only the
    common fully comparable threshold grid.
    """

    data_frame = pd.read_csv(
        file_path,
        low_memory=False,
    )


    required_before_derivation = [
        "matched_group_id",
        "n_items",
        "regime",
        "tau",
        "weight_profile",
        "exact_status",
        "greedy_status",
        "exact_selected_count",
        "greedy_selected_count",
        "optimality_gap_percent",
        "jaccard_similarity",
    ]


    missing_columns = [
        column_name
        for column_name in required_before_derivation
        if column_name not in data_frame.columns
    ]


    if missing_columns:

        raise KeyError(
            f"{family_name} is missing: "
            + ", ".join(
                missing_columns
            )
        )


    numeric_columns = [
        "n_items",
        "tau",
        "exact_selected_count",
        "greedy_selected_count",
        "optimality_gap_percent",
        "jaccard_similarity",
    ]


    for column_name in numeric_columns:

        data_frame[column_name] = pd.to_numeric(
            data_frame[column_name],
            errors="coerce",
        )


    count_difference_derived = (
        "selected_count_difference"
        not in data_frame.columns
    )


    if count_difference_derived:

        data_frame[
            "selected_count_difference"
        ] = (
            data_frame[
                "greedy_selected_count"
            ]
            -
            data_frame[
                "exact_selected_count"
            ]
        )

    else:

        data_frame[
            "selected_count_difference"
        ] = pd.to_numeric(
            data_frame[
                "selected_count_difference"
            ],
            errors="coerce",
        )


    # Restrict before missing-value validation.
    comparable_df = (
        data_frame[
            data_frame[
                "tau"
            ].isin(
                COMMON_THRESHOLDS
            )
            &
            (
                data_frame[
                    "exact_status"
                ]
                ==
                "OPTIMAL"
            )
            &
            (
                data_frame[
                    "greedy_status"
                ]
                ==
                "FEASIBLE"
            )
        ]
        .copy()
    )


    required_numeric = [
        "n_items",
        "tau",
        "exact_selected_count",
        "greedy_selected_count",
        "optimality_gap_percent",
        "jaccard_similarity",
        "selected_count_difference",
    ]


    missing_numeric_counts = (
        comparable_df[
            required_numeric
        ]
        .isna()
        .sum()
    )


    problematic_columns = (
        missing_numeric_counts[
            missing_numeric_counts
            >
            0
        ]
        .index
        .tolist()
    )


    if problematic_columns:

        raise ValueError(
            f"{family_name} still contains missing "
            "comparable values in: "
            + ", ".join(
                problematic_columns
            )
        )


    reconstructed_difference = (
        comparable_df[
            "greedy_selected_count"
        ]
        -
        comparable_df[
            "exact_selected_count"
        ]
    )


    maximum_count_difference_error = float(
        (
            comparable_df[
                "selected_count_difference"
            ]
            -
            reconstructed_difference
        )
        .abs()
        .max()
    )


    assert maximum_count_difference_error <= 1e-12


    comparable_df[
        "utility_family"
    ] = family_name


    comparable_df[
        "exact_recovery_numeric"
    ] = (
        comparable_df[
            "optimality_gap_percent"
        ]
        .abs()
        <=
        EXACT_RECOVERY_TOLERANCE
    ).astype(int)


    comparable_df[
        "large_gap_numeric"
    ] = (
        comparable_df[
            "optimality_gap_percent"
        ]
        >
        10.0
    ).astype(int)


    comparable_df[
        "more_items_numeric"
    ] = (
        comparable_df[
            "selected_count_difference"
        ]
        >
        0
    ).astype(int)


    comparable_df[
        "within_1_numeric"
    ] = (
        comparable_df[
            "optimality_gap_percent"
        ]
        <=
        1.0
    ).astype(int)


    comparable_df[
        "within_5_numeric"
    ] = (
        comparable_df[
            "optimality_gap_percent"
        ]
        <=
        5.0
    ).astype(int)


    comparable_df[
        "within_10_numeric"
    ] = (
        comparable_df[
            "optimality_gap_percent"
        ]
        <=
        10.0
    ).astype(int)


    metadata = {
        "utility_family":
            family_name,

        "original_rows":
            len(
                data_frame
            ),

        "common_comparable_rows":
            len(
                comparable_df
            ),

        "selected_count_difference_derived":
            count_difference_derived,

        "maximum_count_difference_error":
            maximum_count_difference_error,
    }


    return comparable_df, metadata


product_df, product_metadata = load_family(
    PRODUCT_PATH,
    "product_form",
)


coverage_df, coverage_metadata = load_family(
    COVERAGE_PATH,
    "pool_normalized_coverage",
)


# --------------------------------------------------
# Design validation
# --------------------------------------------------

EXPECTED_ROWS_PER_FAMILY = 18000


assert len(
    product_df
) == EXPECTED_ROWS_PER_FAMILY


assert len(
    coverage_df
) == EXPECTED_ROWS_PER_FAMILY


assert (
    product_df[
        "matched_group_id"
    ].nunique()
    ==
    500
)


assert (
    coverage_df[
        "matched_group_id"
    ].nunique()
    ==
    500
)


combined_df = pd.concat(
    [
        product_df,
        coverage_df,
    ],
    ignore_index=True,
)


group_counts = (
    combined_df
    .groupby(
        [
            "utility_family",
            "matched_group_id",
        ]
    )
    .size()
)


group_regime_counts = (
    combined_df
    .groupby(
        [
            "utility_family",
            "matched_group_id",
            "regime",
        ]
    )
    .size()
)


assert (
    group_counts
    ==
    36
).all()


assert (
    group_regime_counts
    ==
    12
).all()


# --------------------------------------------------
# Metric conversion
# --------------------------------------------------

def calculate_metrics(
    totals,
):
    """
    Convert aggregate totals into reporting metrics.
    """

    rows = totals[0]


    return np.array(
        [
            totals[1] / rows,

            100.0
            *
            totals[2]
            /
            rows,

            totals[3] / rows,

            100.0
            *
            totals[4]
            /
            rows,

            100.0
            *
            totals[5]
            /
            rows,

            totals[6] / rows,

            100.0
            *
            totals[7]
            /
            rows,

            100.0
            *
            totals[8]
            /
            rows,

            100.0
            *
            totals[9]
            /
            rows,
        ],
        dtype=float,
    )


# --------------------------------------------------
# Descriptive summaries
# --------------------------------------------------

descriptive_rows = []


for family_name in FAMILIES:

    family_df = combined_df[
        combined_df[
            "utility_family"
        ]
        ==
        family_name
    ]


    for scope in SCOPES:

        if scope == "overall":

            current_df = family_df

        else:

            current_df = family_df[
                family_df[
                    "regime"
                ]
                ==
                scope
            ]


        gaps = current_df[
            "optimality_gap_percent"
        ].to_numpy(
            dtype=float
        )


        descriptive_rows.append(
            {
                "utility_family":
                    family_name,

                "scope":
                    scope,

                "rows":
                    len(
                        current_df
                    ),

                "mean_optimality_gap_percent":
                    gaps.mean(),

                "median_optimality_gap_percent":
                    np.median(
                        gaps
                    ),

                "p90_optimality_gap_percent":
                    np.quantile(
                        gaps,
                        0.90,
                    ),

                "p95_optimality_gap_percent":
                    np.quantile(
                        gaps,
                        0.95,
                    ),

                "exact_cost_recovery_percent":
                    100.0
                    *
                    current_df[
                        "exact_recovery_numeric"
                    ].mean(),

                "mean_jaccard_similarity":
                    current_df[
                        "jaccard_similarity"
                    ].mean(),

                "large_gap_percent":
                    100.0
                    *
                    current_df[
                        "large_gap_numeric"
                    ].mean(),

                "more_items_percent":
                    100.0
                    *
                    current_df[
                        "more_items_numeric"
                    ].mean(),

                "mean_selected_count_difference":
                    current_df[
                        "selected_count_difference"
                    ].mean(),

                "within_1_percent":
                    100.0
                    *
                    current_df[
                        "within_1_numeric"
                    ].mean(),

                "within_5_percent":
                    100.0
                    *
                    current_df[
                        "within_5_numeric"
                    ].mean(),

                "within_10_percent":
                    100.0
                    *
                    current_df[
                        "within_10_numeric"
                    ].mean(),
            }
        )


descriptive_df = pd.DataFrame(
    descriptive_rows
)


# --------------------------------------------------
# Cluster-level sufficient statistics
# --------------------------------------------------

cluster_df = (
    combined_df
    .groupby(
        [
            "utility_family",
            "n_items",
            "matched_group_id",
            "regime",
        ],
        as_index=False,
    )
    .agg(
        comparable_rows=(
            "optimality_gap_percent",
            "size",
        ),

        gap_sum=(
            "optimality_gap_percent",
            "sum",
        ),

        exact_recovery_sum=(
            "exact_recovery_numeric",
            "sum",
        ),

        jaccard_sum=(
            "jaccard_similarity",
            "sum",
        ),

        large_gap_sum=(
            "large_gap_numeric",
            "sum",
        ),

        more_items_sum=(
            "more_items_numeric",
            "sum",
        ),

        count_difference_sum=(
            "selected_count_difference",
            "sum",
        ),

        within_1_sum=(
            "within_1_numeric",
            "sum",
        ),

        within_5_sum=(
            "within_5_numeric",
            "sum",
        ),

        within_10_sum=(
            "within_10_numeric",
            "sum",
        ),
    )
)


assert (
    cluster_df[
        "comparable_rows"
    ]
    ==
    12
).all()


# --------------------------------------------------
# Aligned arrays within each family
# --------------------------------------------------

family_size_arrays = {}


for family_name in FAMILIES:

    family_size_arrays[
        family_name
    ] = {}


    for n_items in CANDIDATE_SIZES:

        size_df = cluster_df[
            (
                cluster_df[
                    "utility_family"
                ]
                ==
                family_name
            )
            &
            (
                cluster_df[
                    "n_items"
                ]
                ==
                n_items
            )
        ]


        group_ids = sorted(
            size_df[
                "matched_group_id"
            ].unique()
        )


        assert len(
            group_ids
        ) == 100


        regime_arrays = []


        for regime in REGIMES:

            regime_df = (
                size_df[
                    size_df[
                        "regime"
                    ]
                    ==
                    regime
                ]
                .set_index(
                    "matched_group_id"
                )
                .loc[
                    group_ids
                ]
            )


            regime_arrays.append(
                regime_df[
                    AGGREGATE_COLUMNS
                ]
                .to_numpy(
                    dtype=float
                )
            )


        family_size_arrays[
            family_name
        ][
            n_items
        ] = np.stack(
            regime_arrays,
            axis=1,
        )


# --------------------------------------------------
# Scope helper
# --------------------------------------------------

def select_scope_totals(
    totals_by_regime,
    scope,
):

    if scope == "overall":

        return totals_by_regime.sum(
            axis=0
        )


    return totals_by_regime[
        REGIMES.index(
            scope
        )
    ]


# --------------------------------------------------
# Observed metrics
# --------------------------------------------------

observed_metrics = {
    family_name: {}
    for family_name in FAMILIES
}


for family_name in FAMILIES:

    totals_by_regime = np.zeros(
        (
            len(
                REGIMES
            ),
            len(
                AGGREGATE_COLUMNS
            ),
        ),
        dtype=float,
    )


    for n_items in CANDIDATE_SIZES:

        totals_by_regime += (
            family_size_arrays[
                family_name
            ][
                n_items
            ]
            .sum(
                axis=0
            )
        )


    for scope in SCOPES:

        observed_metrics[
            family_name
        ][
            scope
        ] = calculate_metrics(
            select_scope_totals(
                totals_by_regime,
                scope,
            )
        )


# --------------------------------------------------
# Independent bootstrap between utility families
# --------------------------------------------------

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)


bootstrap_differences = np.empty(
    (
        BOOTSTRAP_REPLICATES,
        len(
            SCOPES
        ),
        len(
            METRIC_NAMES
        ),
    ),
    dtype=float,
)


for bootstrap_index in range(
    BOOTSTRAP_REPLICATES
):

    family_totals = {
        family_name: np.zeros(
            (
                len(
                    REGIMES
                ),
                len(
                    AGGREGATE_COLUMNS
                ),
            ),
            dtype=float,
        )
        for family_name in FAMILIES
    }


    for family_name in FAMILIES:

        for n_items in CANDIDATE_SIZES:

            current_array = (
                family_size_arrays[
                    family_name
                ][
                    n_items
                ]
            )


            sampled_indices = rng.integers(
                0,
                current_array.shape[0],
                size=current_array.shape[0],
            )


            family_totals[
                family_name
            ] += (
                current_array[
                    sampled_indices
                ]
                .sum(
                    axis=0
                )
            )


    for scope_index, scope in enumerate(
        SCOPES
    ):

        coverage_metrics = calculate_metrics(
            select_scope_totals(
                family_totals[
                    "pool_normalized_coverage"
                ],
                scope,
            )
        )


        product_metrics = calculate_metrics(
            select_scope_totals(
                family_totals[
                    "product_form"
                ],
                scope,
            )
        )


        bootstrap_differences[
            bootstrap_index,
            scope_index,
        ] = (
            coverage_metrics
            -
            product_metrics
        )


# --------------------------------------------------
# Confidence intervals
# --------------------------------------------------

contrast_rows = []


for scope_index, scope in enumerate(
    SCOPES
):

    observed_difference = (
        observed_metrics[
            "pool_normalized_coverage"
        ][
            scope
        ]
        -
        observed_metrics[
            "product_form"
        ][
            scope
        ]
    )


    current_bootstrap = (
        bootstrap_differences[
            :,
            scope_index,
            :,
        ]
    )


    lower = np.quantile(
        current_bootstrap,
        0.025,
        axis=0,
    )


    upper = np.quantile(
        current_bootstrap,
        0.975,
        axis=0,
    )


    standard_error = current_bootstrap.std(
        axis=0,
        ddof=1,
    )


    for metric_index, metric_name in enumerate(
        METRIC_NAMES
    ):

        contrast_rows.append(
            {
                "scope":
                    scope,

                "metric":
                    metric_name,

                "estimated_difference":
                    observed_difference[
                        metric_index
                    ],

                "bootstrap_standard_error":
                    standard_error[
                        metric_index
                    ],

                "ci_95_lower":
                    lower[
                        metric_index
                    ],

                "ci_95_upper":
                    upper[
                        metric_index
                    ],

                "ci_excludes_zero":
                    (
                        lower[
                            metric_index
                        ]
                        >
                        0
                    )
                    or
                    (
                        upper[
                            metric_index
                        ]
                        <
                        0
                    ),

                "contrast":
                    (
                        "pool_normalized_coverage "
                        "minus product_form"
                    ),

                "common_thresholds":
                    repr(
                        COMMON_THRESHOLDS
                    ),

                "bootstrap_replicates":
                    BOOTSTRAP_REPLICATES,

                "resampling_design":
                    (
                        "independent_between_families;"
                        "matched_across_regimes_within_family;"
                        "stratified_by_candidate_size"
                    ),

                "multiple_comparison_adjustment":
                    "none",
            }
        )


contrast_df = pd.DataFrame(
    contrast_rows
)


# --------------------------------------------------
# Bootstrap replicate table
# --------------------------------------------------

bootstrap_rows = []


for bootstrap_index in range(
    BOOTSTRAP_REPLICATES
):

    for scope_index, scope in enumerate(
        SCOPES
    ):

        row = {
            "bootstrap_replicate":
                bootstrap_index,

            "scope":
                scope,
        }


        for metric_index, metric_name in enumerate(
            METRIC_NAMES
        ):

            row[
                metric_name
            ] = bootstrap_differences[
                bootstrap_index,
                scope_index,
                metric_index,
            ]


        bootstrap_rows.append(
            row
        )


bootstrap_df = pd.DataFrame(
    bootstrap_rows
)


# --------------------------------------------------
# Audit
# --------------------------------------------------

audit_df = pd.DataFrame(
    [
        {
            "product_original_rows":
                product_metadata[
                    "original_rows"
                ],

            "coverage_original_rows":
                coverage_metadata[
                    "original_rows"
                ],

            "product_common_comparable_rows":
                len(
                    product_df
                ),

            "coverage_common_comparable_rows":
                len(
                    coverage_df
                ),

            "expected_rows_per_family":
                EXPECTED_ROWS_PER_FAMILY,

            "product_matched_groups":
                product_df[
                    "matched_group_id"
                ].nunique(),

            "coverage_matched_groups":
                coverage_df[
                    "matched_group_id"
                ].nunique(),

            "common_thresholds":
                repr(
                    COMMON_THRESHOLDS
                ),

            "product_count_difference_derived":
                product_metadata[
                    "selected_count_difference_derived"
                ],

            "coverage_count_difference_derived":
                coverage_metadata[
                    "selected_count_difference_derived"
                ],

            "product_maximum_count_difference_error":
                product_metadata[
                    "maximum_count_difference_error"
                ],

            "coverage_maximum_count_difference_error":
                coverage_metadata[
                    "maximum_count_difference_error"
                ],

            "minimum_rows_per_family_group":
                int(
                    group_counts.min()
                ),

            "maximum_rows_per_family_group":
                int(
                    group_counts.max()
                ),

            "minimum_rows_per_family_group_regime":
                int(
                    group_regime_counts.min()
                ),

            "maximum_rows_per_family_group_regime":
                int(
                    group_regime_counts.max()
                ),

            "bootstrap_replicates":
                BOOTSTRAP_REPLICATES,

            "confidence_intervals":
                "unadjusted_percentile_95",
        }
    ]
)


# --------------------------------------------------
# Save new outputs
# --------------------------------------------------

audit_df.to_csv(
    AUDIT_PATH,
    index=False,
)


descriptive_df.to_csv(
    DESCRIPTIVE_PATH,
    index=False,
)


contrast_df.to_csv(
    CONTRAST_PATH,
    index=False,
)


bootstrap_df.to_csv(
    BOOTSTRAP_PATH,
    index=False,
)


# --------------------------------------------------
# Printed report
# --------------------------------------------------

contrast_display_df = contrast_df.copy()


contrast_display_df[
    "difference_and_ci"
] = (
    contrast_display_df[
        "estimated_difference"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    contrast_display_df[
        "ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    contrast_display_df[
        "ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


print(
    "======================================"
)

print(
    "UTILITY-FAMILY COMPARISON AUDIT"
)

print(
    "======================================"
)


print(
    audit_df.to_string(
        index=False
    )
)


print(
    "\n======================================"
)

print(
    "UTILITY-FAMILY DESCRIPTIVE RESULTS"
)

print(
    "COMMON THRESHOLDS: 0.70, 0.80, 0.90"
)

print(
    "======================================"
)


print(
    descriptive_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.4f}",
    )
)


print(
    "\n======================================"
)

print(
    "COVERAGE-MINUS-PRODUCT CONTRASTS"
)

print(
    "======================================"
)


print(
    contrast_display_df[
        [
            "scope",
            "metric",
            "difference_and_ci",
            "ci_excludes_zero",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\nAudit saved to:"
)

print(
    AUDIT_PATH
)


print(
    "\nDescriptive results saved to:"
)

print(
    DESCRIPTIVE_PATH
)


print(
    "\nContrasts saved to:"
)

print(
    CONTRAST_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Step 135 — Utility-family × structural-regime
# interaction analysis with simultaneous max-t
# confidence intervals
# --------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------
# Configuration
# --------------------------------------------------

SCOPES = [
    "overall",
    "independent",
    "utility_burden_alignment",
    "utility_burden_conflict",
]


REGIME_SCOPES = [
    "independent",
    "utility_burden_alignment",
    "utility_burden_conflict",
]


KEY_METRICS = [
    "mean_optimality_gap_percent",
    "exact_cost_recovery_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "more_items_percent",
    "mean_selected_count_difference",
]


INTERACTION_CONTRASTS = [
    (
        "utility_burden_alignment",
        "independent",
    ),

    (
        "utility_burden_conflict",
        "independent",
    ),

    (
        "utility_burden_conflict",
        "utility_burden_alignment",
    ),
]


SIMULTANEOUS_CONFIDENCE_LEVEL = 0.95


# --------------------------------------------------
# Paths
# --------------------------------------------------

RESULTS_DIR = Path(
    os.path.join(PROJECT_ROOT, "05_Results")
)


TABLES_DIR = Path(
    os.path.join(PROJECT_ROOT, "07_Tables")
)


RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


OBSERVED_CONTRAST_PATH = (
    TABLES_DIR
    / "coverage_minus_product_family_contrasts_v2.csv"
)


BOOTSTRAP_REPLICATES_PATH = (
    RESULTS_DIR
    / "coverage_minus_product_family_bootstrap_v2.csv"
)


SCOPE_SIMULTANEOUS_PATH = (
    TABLES_DIR
    / "utility_family_scope_simultaneous_ci_v1.csv"
)


INTERACTION_SIMULTANEOUS_PATH = (
    TABLES_DIR
    / "utility_family_regime_interaction_simultaneous_ci_v1.csv"
)


INTERACTION_REPLICATES_PATH = (
    RESULTS_DIR
    / "utility_family_regime_interaction_bootstrap_v1.csv"
)


INFERENCE_AUDIT_PATH = (
    RESULTS_DIR
    / "utility_family_simultaneous_inference_audit_v1.csv"
)


# --------------------------------------------------
# Prevent accidental overwriting
# --------------------------------------------------

output_paths = [
    SCOPE_SIMULTANEOUS_PATH,
    INTERACTION_SIMULTANEOUS_PATH,
    INTERACTION_REPLICATES_PATH,
    INFERENCE_AUDIT_PATH,
]


existing_outputs = [
    output_path
    for output_path in output_paths
    if output_path.exists()
]


if existing_outputs:

    raise FileExistsError(
        "The following Step-135 output already exists. "
        "Nothing was overwritten:\n"
        +
        "\n".join(
            str(output_path)
            for output_path in existing_outputs
        )
    )


# --------------------------------------------------
# Load results from Step 134B
# --------------------------------------------------

observed_df = pd.read_csv(
    OBSERVED_CONTRAST_PATH,
    low_memory=False,
)


bootstrap_df = pd.read_csv(
    BOOTSTRAP_REPLICATES_PATH,
    low_memory=False,
)


required_observed_columns = [
    "scope",
    "metric",
    "estimated_difference",
]


missing_observed_columns = [
    column_name
    for column_name in required_observed_columns
    if column_name not in observed_df.columns
]


if missing_observed_columns:

    raise KeyError(
        "Missing observed-contrast columns: "
        +
        ", ".join(
            missing_observed_columns
        )
    )


required_bootstrap_columns = [
    "bootstrap_replicate",
    "scope",
    *KEY_METRICS,
]


missing_bootstrap_columns = [
    column_name
    for column_name in required_bootstrap_columns
    if column_name not in bootstrap_df.columns
]


if missing_bootstrap_columns:

    raise KeyError(
        "Missing bootstrap columns: "
        +
        ", ".join(
            missing_bootstrap_columns
        )
    )


observed_df[
    "estimated_difference"
] = pd.to_numeric(
    observed_df[
        "estimated_difference"
    ],
    errors="coerce",
)


bootstrap_df[
    "bootstrap_replicate"
] = pd.to_numeric(
    bootstrap_df[
        "bootstrap_replicate"
    ],
    errors="coerce",
)


for metric_name in KEY_METRICS:

    bootstrap_df[
        metric_name
    ] = pd.to_numeric(
        bootstrap_df[
            metric_name
        ],
        errors="coerce",
    )


# --------------------------------------------------
# Design validation
# --------------------------------------------------

observed_key_df = (
    observed_df[
        observed_df[
            "scope"
        ].isin(
            SCOPES
        )
        &
        observed_df[
            "metric"
        ].isin(
            KEY_METRICS
        )
    ]
    .copy()
)


assert len(
    observed_key_df
) == (
    len(
        SCOPES
    )
    *
    len(
        KEY_METRICS
    )
)


assert observed_key_df[
    [
        "scope",
        "metric",
    ]
].duplicated().sum() == 0


assert set(
    observed_key_df[
        "scope"
    ].unique()
) == set(
    SCOPES
)


assert set(
    observed_key_df[
        "metric"
    ].unique()
) == set(
    KEY_METRICS
)


bootstrap_replicates = sorted(
    bootstrap_df[
        "bootstrap_replicate"
    ]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)


number_of_bootstrap_replicates = len(
    bootstrap_replicates
)


assert number_of_bootstrap_replicates == 5000


bootstrap_scope_counts = (
    bootstrap_df
    .groupby(
        "bootstrap_replicate"
    )[
        "scope"
    ]
    .nunique()
)


assert (
    bootstrap_scope_counts
    ==
    len(
        SCOPES
    )
).all()


assert bootstrap_df[
    KEY_METRICS
].isna().sum().sum() == 0


# --------------------------------------------------
# Observed coverage-minus-product estimates
# --------------------------------------------------

observed_pivot = (
    observed_key_df
    .pivot(
        index="scope",
        columns="metric",
        values="estimated_difference",
    )
    .loc[
        SCOPES,
        KEY_METRICS,
    ]
)


# ==================================================
# PART A — Simultaneous intervals across four scopes
# ==================================================

scope_result_rows = []


for metric_name in KEY_METRICS:

    bootstrap_matrix = (
        bootstrap_df
        .pivot(
            index="bootstrap_replicate",
            columns="scope",
            values=metric_name,
        )
        .loc[
            bootstrap_replicates,
            SCOPES,
        ]
        .to_numpy(
            dtype=float
        )
    )


    observed_values = (
        observed_pivot[
            metric_name
        ]
        .to_numpy(
            dtype=float
        )
    )


    standard_errors = bootstrap_matrix.std(
        axis=0,
        ddof=1,
    )


    if np.any(
        standard_errors
        <=
        0.0
    ):

        raise ValueError(
            f"Nonpositive bootstrap standard error "
            f"for metric: {metric_name}"
        )


    standardized_deviations = np.abs(
        (
            bootstrap_matrix
            -
            observed_values
        )
        /
        standard_errors
    )


    maximum_t_per_replicate = (
        standardized_deviations.max(
            axis=1
        )
    )


    critical_value = float(
        np.quantile(
            maximum_t_per_replicate,
            SIMULTANEOUS_CONFIDENCE_LEVEL,
        )
    )


    simultaneous_lower = (
        observed_values
        -
        critical_value
        *
        standard_errors
    )


    simultaneous_upper = (
        observed_values
        +
        critical_value
        *
        standard_errors
    )


    for scope_index, scope in enumerate(
        SCOPES
    ):

        scope_result_rows.append(
            {
                "scope":
                    scope,

                "metric":
                    metric_name,

                "estimated_difference":
                    observed_values[
                        scope_index
                    ],

                "bootstrap_standard_error":
                    standard_errors[
                        scope_index
                    ],

                "max_t_critical_value":
                    critical_value,

                "simultaneous_ci_95_lower":
                    simultaneous_lower[
                        scope_index
                    ],

                "simultaneous_ci_95_upper":
                    simultaneous_upper[
                        scope_index
                    ],

                "simultaneous_ci_excludes_zero":
                    (
                        simultaneous_lower[
                            scope_index
                        ]
                        >
                        0.0
                    )
                    or
                    (
                        simultaneous_upper[
                            scope_index
                        ]
                        <
                        0.0
                    ),

                "family_contrast":
                    (
                        "pool_normalized_coverage "
                        "minus product_form"
                    ),

                "familywise_scope_count":
                    len(
                        SCOPES
                    ),

                "bootstrap_replicates":
                    number_of_bootstrap_replicates,

                "interval_method":
                    (
                        "studentized_max_t_"
                        "simultaneous_within_metric"
                    ),
            }
        )


scope_simultaneous_df = pd.DataFrame(
    scope_result_rows
)


# ==================================================
# PART B — Difference-in-differences interactions
#
# Example:
#   (coverage − product in conflict)
#   minus
#   (coverage − product in independence)
# ==================================================

interaction_result_rows = []
interaction_replicate_rows = []


for metric_name in KEY_METRICS:

    regime_bootstrap_matrix = (
        bootstrap_df
        .pivot(
            index="bootstrap_replicate",
            columns="scope",
            values=metric_name,
        )
        .loc[
            bootstrap_replicates,
            REGIME_SCOPES,
        ]
        .to_numpy(
            dtype=float
        )
    )


    regime_observed_values = (
        observed_pivot
        .loc[
            REGIME_SCOPES,
            metric_name,
        ]
        .to_numpy(
            dtype=float
        )
    )


    bootstrap_interactions = []
    observed_interactions = []


    for first_regime, second_regime in (
        INTERACTION_CONTRASTS
    ):

        first_index = REGIME_SCOPES.index(
            first_regime
        )


        second_index = REGIME_SCOPES.index(
            second_regime
        )


        bootstrap_interactions.append(
            regime_bootstrap_matrix[
                :,
                first_index,
            ]
            -
            regime_bootstrap_matrix[
                :,
                second_index,
            ]
        )


        observed_interactions.append(
            regime_observed_values[
                first_index
            ]
            -
            regime_observed_values[
                second_index
            ]
        )


    bootstrap_interactions = np.column_stack(
        bootstrap_interactions
    )


    observed_interactions = np.asarray(
        observed_interactions,
        dtype=float,
    )


    interaction_standard_errors = (
        bootstrap_interactions.std(
            axis=0,
            ddof=1,
        )
    )


    if np.any(
        interaction_standard_errors
        <=
        0.0
    ):

        raise ValueError(
            f"Nonpositive interaction standard error "
            f"for metric: {metric_name}"
        )


    standardized_interaction_deviations = np.abs(
        (
            bootstrap_interactions
            -
            observed_interactions
        )
        /
        interaction_standard_errors
    )


    maximum_interaction_t = (
        standardized_interaction_deviations.max(
            axis=1
        )
    )


    interaction_critical_value = float(
        np.quantile(
            maximum_interaction_t,
            SIMULTANEOUS_CONFIDENCE_LEVEL,
        )
    )


    interaction_lower = (
        observed_interactions
        -
        interaction_critical_value
        *
        interaction_standard_errors
    )


    interaction_upper = (
        observed_interactions
        +
        interaction_critical_value
        *
        interaction_standard_errors
    )


    for contrast_index, (
        first_regime,
        second_regime,
    ) in enumerate(
        INTERACTION_CONTRASTS
    ):

        interaction_name = (
            f"({first_regime} minus {second_regime}) "
            "difference in coverage-minus-product effect"
        )


        interaction_result_rows.append(
            {
                "first_regime":
                    first_regime,

                "second_regime":
                    second_regime,

                "interaction_contrast":
                    interaction_name,

                "metric":
                    metric_name,

                "estimated_interaction":
                    observed_interactions[
                        contrast_index
                    ],

                "bootstrap_standard_error":
                    interaction_standard_errors[
                        contrast_index
                    ],

                "max_t_critical_value":
                    interaction_critical_value,

                "simultaneous_ci_95_lower":
                    interaction_lower[
                        contrast_index
                    ],

                "simultaneous_ci_95_upper":
                    interaction_upper[
                        contrast_index
                    ],

                "simultaneous_ci_excludes_zero":
                    (
                        interaction_lower[
                            contrast_index
                        ]
                        >
                        0.0
                    )
                    or
                    (
                        interaction_upper[
                            contrast_index
                        ]
                        <
                        0.0
                    ),

                "interaction_interpretation":
                    (
                        "difference_in_coverage_minus_"
                        "product_effect_between_regimes"
                    ),

                "familywise_interaction_count":
                    len(
                        INTERACTION_CONTRASTS
                    ),

                "bootstrap_replicates":
                    number_of_bootstrap_replicates,

                "interval_method":
                    (
                        "studentized_max_t_"
                        "simultaneous_within_metric"
                    ),
            }
        )


        for bootstrap_index, bootstrap_replicate in enumerate(
            bootstrap_replicates
        ):

            interaction_replicate_rows.append(
                {
                    "bootstrap_replicate":
                        bootstrap_replicate,

                    "first_regime":
                        first_regime,

                    "second_regime":
                        second_regime,

                    "interaction_contrast":
                        interaction_name,

                    "metric":
                        metric_name,

                    "interaction_value":
                        bootstrap_interactions[
                            bootstrap_index,
                            contrast_index,
                        ],
                }
            )


interaction_simultaneous_df = pd.DataFrame(
    interaction_result_rows
)


interaction_replicates_df = pd.DataFrame(
    interaction_replicate_rows
)


# --------------------------------------------------
# Audit
# --------------------------------------------------

inference_audit_df = pd.DataFrame(
    [
        {
            "observed_scope_metric_rows":
                len(
                    observed_key_df
                ),

            "expected_observed_scope_metric_rows":
                len(
                    SCOPES
                )
                *
                len(
                    KEY_METRICS
                ),

            "bootstrap_replicates":
                number_of_bootstrap_replicates,

            "scopes":
                len(
                    SCOPES
                ),

            "regime_scopes":
                len(
                    REGIME_SCOPES
                ),

            "metrics":
                len(
                    KEY_METRICS
                ),

            "interaction_contrasts":
                len(
                    INTERACTION_CONTRASTS
                ),

            "scope_simultaneous_rows":
                len(
                    scope_simultaneous_df
                ),

            "interaction_simultaneous_rows":
                len(
                    interaction_simultaneous_df
                ),

            "interaction_bootstrap_rows":
                len(
                    interaction_replicates_df
                ),

            "missing_scope_estimates":
                int(
                    scope_simultaneous_df[
                        "estimated_difference"
                    ].isna().sum()
                ),

            "missing_interaction_estimates":
                int(
                    interaction_simultaneous_df[
                        "estimated_interaction"
                    ].isna().sum()
                ),

            "scope_interval_family":
                (
                    "four scopes controlled separately "
                    "within each metric"
                ),

            "interaction_interval_family":
                (
                    "three interactions controlled "
                    "separately within each metric"
                ),

            "confidence_level":
                SIMULTANEOUS_CONFIDENCE_LEVEL,

            "global_across_metrics_adjustment":
                False,
        }
    ]
)


assert inference_audit_df.loc[
    0,
    "observed_scope_metric_rows",
] == inference_audit_df.loc[
    0,
    "expected_observed_scope_metric_rows",
]


assert inference_audit_df.loc[
    0,
    "scope_simultaneous_rows",
] == (
    len(
        SCOPES
    )
    *
    len(
        KEY_METRICS
    )
)


assert inference_audit_df.loc[
    0,
    "interaction_simultaneous_rows",
] == (
    len(
        INTERACTION_CONTRASTS
    )
    *
    len(
        KEY_METRICS
    )
)


assert inference_audit_df.loc[
    0,
    "missing_scope_estimates",
] == 0


assert inference_audit_df.loc[
    0,
    "missing_interaction_estimates",
] == 0


# --------------------------------------------------
# Save outputs
# --------------------------------------------------

scope_simultaneous_df.to_csv(
    SCOPE_SIMULTANEOUS_PATH,
    index=False,
)


interaction_simultaneous_df.to_csv(
    INTERACTION_SIMULTANEOUS_PATH,
    index=False,
)


interaction_replicates_df.to_csv(
    INTERACTION_REPLICATES_PATH,
    index=False,
)


inference_audit_df.to_csv(
    INFERENCE_AUDIT_PATH,
    index=False,
)


# --------------------------------------------------
# Readable displays
# --------------------------------------------------

scope_display_df = (
    scope_simultaneous_df.copy()
)


scope_display_df[
    "estimate_and_simultaneous_ci"
] = (
    scope_display_df[
        "estimated_difference"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    scope_display_df[
        "simultaneous_ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    scope_display_df[
        "simultaneous_ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


interaction_display_df = (
    interaction_simultaneous_df.copy()
)


interaction_display_df[
    "interaction_and_simultaneous_ci"
] = (
    interaction_display_df[
        "estimated_interaction"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    " ["
    +
    interaction_display_df[
        "simultaneous_ci_95_lower"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    ", "
    +
    interaction_display_df[
        "simultaneous_ci_95_upper"
    ].map(
        lambda value:
            f"{value:.4f}"
    )
    +
    "]"
)


# --------------------------------------------------
# Printed report
# --------------------------------------------------

print(
    "======================================"
)

print(
    "UTILITY-FAMILY SIMULTANEOUS-INFERENCE AUDIT"
)

print(
    "======================================"
)


print(
    inference_audit_df.to_string(
        index=False
    )
)


print(
    "\n======================================"
)

print(
    "SIMULTANEOUS COVERAGE-MINUS-PRODUCT INTERVALS"
)

print(
    "======================================"
)


print(
    scope_display_df[
        [
            "scope",
            "metric",
            "estimate_and_simultaneous_ci",
            "simultaneous_ci_excludes_zero",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\n======================================"
)

print(
    "UTILITY-FAMILY × REGIME INTERACTIONS"
)

print(
    "======================================"
)


print(
    interaction_display_df[
        [
            "first_regime",
            "second_regime",
            "metric",
            "interaction_and_simultaneous_ci",
            "simultaneous_ci_excludes_zero",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\nScope-level simultaneous intervals saved to:"
)

print(
    SCOPE_SIMULTANEOUS_PATH
)


print(
    "\nInteraction intervals saved to:"
)

print(
    INTERACTION_SIMULTANEOUS_PATH
)


print(
    "\nInference audit saved to:"
)

print(
    INFERENCE_AUDIT_PATH
)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
# --------------------------------------------------
# Step 136 — Final computational evidence synthesis,
# integrity closure, and reproducibility freeze
# --------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone
import hashlib

import numpy as np
import pandas as pd


# --------------------------------------------------
# Project paths
# --------------------------------------------------

PROJECT_ROOT = Path(
    PROJECT_ROOT
)


ADMIN_DIR = (
    PROJECT_ROOT
    / "01_Administrative"
)


RESULTS_DIR = (
    PROJECT_ROOT
    / "05_Results"
)


TABLES_DIR = (
    PROJECT_ROOT
    / "07_Tables"
)


for directory in [
    ADMIN_DIR,
    RESULTS_DIR,
    TABLES_DIR,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# --------------------------------------------------
# Required final source files
# --------------------------------------------------

SOURCE_FILES = {
    "product_main_benchmark":
        RESULTS_DIR
        / "exact_vs_greedy_main_benchmark_v2_marginal_preserving.csv",

    "coverage_main_benchmark":
        RESULTS_DIR
        / "coverage_exact_vs_greedy_main_benchmark_v1_pool_normalized.csv",

    "family_descriptive":
        TABLES_DIR
        / "utility_family_common_threshold_descriptive_v2.csv",

    "family_scope_simultaneous":
        TABLES_DIR
        / "utility_family_scope_simultaneous_ci_v1.csv",

    "family_regime_interactions":
        TABLES_DIR
        / "utility_family_regime_interaction_simultaneous_ci_v1.csv",

    "coverage_regime_contrasts":
        TABLES_DIR
        / "coverage_exact_vs_greedy_paired_regime_contrasts.csv",

    "coverage_failure_mechanisms":
        TABLES_DIR
        / "coverage_greedy_failure_mechanism_summary.csv",

    "coverage_large_gap_attribution":
        TABLES_DIR
        / "coverage_large_gap_mechanism_attribution.csv",

    "coverage_tail_summary":
        TABLES_DIR
        / "coverage_optimality_gap_tail_summary.csv",

    "coverage_size_summary":
        TABLES_DIR
        / "coverage_candidate_size_bootstrap_summary.csv",

    "coverage_threshold_summary":
        TABLES_DIR
        / "coverage_threshold_denominator_descriptive_summary.csv",

    "coverage_weight_descriptive":
        TABLES_DIR
        / "coverage_weight_profile_descriptive_summary.csv",

    "coverage_weight_contrasts":
        TABLES_DIR
        / "coverage_paired_weight_profile_contrasts.csv",

    "coverage_runtime_summary":
        TABLES_DIR
        / "coverage_exact_vs_greedy_runtime_by_candidate_size.csv",
}


# --------------------------------------------------
# New Step-136 outputs
# --------------------------------------------------

FINAL_EVIDENCE_PATH = (
    TABLES_DIR
    / "final_computational_evidence_ledger_v1.csv"
)


FINAL_AUDIT_PATH = (
    RESULTS_DIR
    / "final_computational_analysis_audit_v1.csv"
)


FINAL_MANIFEST_PATH = (
    RESULTS_DIR
    / "final_computational_file_manifest_v1.csv"
)


FINAL_FREEZE_NOTE_PATH = (
    ADMIN_DIR
    / "computational_analysis_freeze_notes_v1.txt"
)


FINAL_SNAPSHOT_PATH = (
    TABLES_DIR
    / "final_key_evidence_snapshot_v1.csv"
)


OUTPUT_FILES = [
    FINAL_EVIDENCE_PATH,
    FINAL_AUDIT_PATH,
    FINAL_MANIFEST_PATH,
    FINAL_FREEZE_NOTE_PATH,
    FINAL_SNAPSHOT_PATH,
]


# --------------------------------------------------
# Prevent overwriting
# --------------------------------------------------

existing_outputs = [
    output_path
    for output_path in OUTPUT_FILES
    if output_path.exists()
]


if existing_outputs:

    raise FileExistsError(
        "Step-136 output already exists. "
        "Nothing was overwritten:\n"
        +
        "\n".join(
            str(output_path)
            for output_path in existing_outputs
        )
    )


# --------------------------------------------------
# Confirm all required sources exist
# --------------------------------------------------

missing_sources = [
    source_name
    for source_name, source_path
    in SOURCE_FILES.items()
    if not source_path.exists()
]


if missing_sources:

    raise FileNotFoundError(
        "Missing required final source files:\n"
        +
        "\n".join(
            missing_sources
        )
    )


# --------------------------------------------------
# Load primary raw benchmarks
# --------------------------------------------------

product_df = pd.read_csv(
    SOURCE_FILES[
        "product_main_benchmark"
    ],
    low_memory=False,
)


coverage_df = pd.read_csv(
    SOURCE_FILES[
        "coverage_main_benchmark"
    ],
    low_memory=False,
)


for data_frame in [
    product_df,
    coverage_df,
]:

    for column_name in [
        "tau",
        "n_items",
        "optimality_gap_percent",
        "jaccard_similarity",
        "exact_selected_count",
        "greedy_selected_count",
    ]:

        data_frame[
            column_name
        ] = pd.to_numeric(
            data_frame[
                column_name
            ],
            errors="coerce",
        )


# --------------------------------------------------
# Load final analysis tables
# --------------------------------------------------

family_descriptive_df = pd.read_csv(
    SOURCE_FILES[
        "family_descriptive"
    ],
    low_memory=False,
)


family_scope_df = pd.read_csv(
    SOURCE_FILES[
        "family_scope_simultaneous"
    ],
    low_memory=False,
)


family_interaction_df = pd.read_csv(
    SOURCE_FILES[
        "family_regime_interactions"
    ],
    low_memory=False,
)


coverage_regime_df = pd.read_csv(
    SOURCE_FILES[
        "coverage_regime_contrasts"
    ],
    low_memory=False,
)


mechanism_df = pd.read_csv(
    SOURCE_FILES[
        "coverage_failure_mechanisms"
    ],
    low_memory=False,
)


large_gap_attribution_df = pd.read_csv(
    SOURCE_FILES[
        "coverage_large_gap_attribution"
    ],
    low_memory=False,
)


tail_df = pd.read_csv(
    SOURCE_FILES[
        "coverage_tail_summary"
    ],
    low_memory=False,
)


size_df = pd.read_csv(
    SOURCE_FILES[
        "coverage_size_summary"
    ],
    low_memory=False,
)


threshold_df = pd.read_csv(
    SOURCE_FILES[
        "coverage_threshold_summary"
    ],
    low_memory=False,
)


weight_descriptive_df = pd.read_csv(
    SOURCE_FILES[
        "coverage_weight_descriptive"
    ],
    low_memory=False,
)


weight_contrast_df = pd.read_csv(
    SOURCE_FILES[
        "coverage_weight_contrasts"
    ],
    low_memory=False,
)


runtime_df = pd.read_csv(
    SOURCE_FILES[
        "coverage_runtime_summary"
    ],
    low_memory=False,
)


# --------------------------------------------------
# Structural integrity checks
# --------------------------------------------------

assert len(
    product_df
) == 24000


assert len(
    coverage_df
) == 24000


assert len(
    family_descriptive_df
) == 8


assert len(
    family_scope_df
) == 24


assert len(
    family_interaction_df
) == 18


assert len(
    coverage_regime_df
) == 27


assert len(
    mechanism_df
) == 4


assert len(
    large_gap_attribution_df
) == 16


assert len(
    tail_df
) == 4


assert len(
    size_df
) == 30


assert len(
    threshold_df
) == 4


assert len(
    weight_descriptive_df
) == 4


assert len(
    weight_contrast_df
) == 18


assert len(
    runtime_df
) == 5


# --------------------------------------------------
# Comparable-row audits
# --------------------------------------------------

product_comparable_mask = (
    (
        product_df[
            "exact_status"
        ]
        ==
        "OPTIMAL"
    )
    &
    (
        product_df[
            "greedy_status"
        ]
        ==
        "FEASIBLE"
    )
    &
    product_df[
        "optimality_gap_percent"
    ].notna()
    &
    product_df[
        "jaccard_similarity"
    ].notna()
)


coverage_comparable_mask = (
    (
        coverage_df[
            "exact_status"
        ]
        ==
        "OPTIMAL"
    )
    &
    (
        coverage_df[
            "greedy_status"
        ]
        ==
        "FEASIBLE"
    )
    &
    coverage_df[
        "optimality_gap_percent"
    ].notna()
    &
    coverage_df[
        "jaccard_similarity"
    ].notna()
)


product_comparable_rows = int(
    product_comparable_mask.sum()
)


coverage_comparable_rows = int(
    coverage_comparable_mask.sum()
)


product_infeasible_rows = int(
    (
        product_df[
            "exact_status"
        ]
        ==
        "INFEASIBLE"
    ).sum()
)


coverage_infeasible_rows = int(
    (
        coverage_df[
            "exact_status"
        ]
        ==
        "INFEASIBLE"
    ).sum()
)


product_common_rows = int(
    (
        product_comparable_mask
        &
        product_df[
            "tau"
        ].isin(
            [
                0.70,
                0.80,
                0.90,
            ]
        )
    ).sum()
)


coverage_common_rows = int(
    (
        coverage_comparable_mask
        &
        coverage_df[
            "tau"
        ].isin(
            [
                0.70,
                0.80,
                0.90,
            ]
        )
    ).sum()
)


assert product_comparable_rows == 23784
assert coverage_comparable_rows == 24000
assert product_infeasible_rows == 216
assert coverage_infeasible_rows == 0
assert product_common_rows == 18000
assert coverage_common_rows == 18000


# --------------------------------------------------
# Evidence-ledger helper
# --------------------------------------------------

evidence_rows = []


def add_evidence(
    evidence_family,
    scope,
    comparison,
    metric,
    estimate,
    ci_lower=np.nan,
    ci_upper=np.nan,
    interval_type="descriptive_no_interval",
    source_file="",
    interpretation_guardrail="",
):
    """
    Append one canonical evidence-ledger row.
    """

    evidence_rows.append(
        {
            "evidence_family":
                evidence_family,

            "scope":
                scope,

            "comparison":
                comparison,

            "metric":
                metric,

            "estimate":
                float(
                    estimate
                ),

            "ci_95_lower":
                (
                    float(
                        ci_lower
                    )
                    if pd.notna(
                        ci_lower
                    )
                    else np.nan
                ),

            "ci_95_upper":
                (
                    float(
                        ci_upper
                    )
                    if pd.notna(
                        ci_upper
                    )
                    else np.nan
                ),

            "interval_type":
                interval_type,

            "source_file":
                source_file,

            "interpretation_guardrail":
                interpretation_guardrail,
        }
    )


# --------------------------------------------------
# A. Utility-family descriptive evidence
# --------------------------------------------------

FAMILY_DESCRIPTIVE_METRICS = [
    "mean_optimality_gap_percent",
    "exact_cost_recovery_percent",
    "mean_jaccard_similarity",
    "large_gap_percent",
    "more_items_percent",
    "mean_selected_count_difference",
    "within_1_percent",
    "within_5_percent",
    "within_10_percent",
]


for _, row in family_descriptive_df.iterrows():

    for metric_name in FAMILY_DESCRIPTIVE_METRICS:

        add_evidence(
            evidence_family=
                "utility_family_descriptive",

            scope=
                row[
                    "scope"
                ],

            comparison=
                row[
                    "utility_family"
                ],

            metric=
                metric_name,

            estimate=
                row[
                    metric_name
                ],

            source_file=
                SOURCE_FILES[
                    "family_descriptive"
                ].name,

            interpretation_guardrail=
                (
                    "Common fully feasible thresholds "
                    "0.70, 0.80, and 0.90 only."
                ),
        )


# --------------------------------------------------
# B. Simultaneous coverage-minus-product evidence
# --------------------------------------------------

for _, row in family_scope_df.iterrows():

    add_evidence(
        evidence_family=
            "utility_family_simultaneous",

        scope=
            row[
                "scope"
            ],

        comparison=
            (
                "pool_normalized_coverage "
                "minus product_form"
            ),

        metric=
            row[
                "metric"
            ],

        estimate=
            row[
                "estimated_difference"
            ],

        ci_lower=
            row[
                "simultaneous_ci_95_lower"
            ],

        ci_upper=
            row[
                "simultaneous_ci_95_upper"
            ],

        interval_type=
            (
                "studentized_max_t_"
                "simultaneous_within_metric"
            ),

        source_file=
            SOURCE_FILES[
                "family_scope_simultaneous"
            ].name,

        interpretation_guardrail=
            (
                "Families were independently generated; "
                "this is benchmark-family sensitivity, "
                "not a causal effect."
            ),
    )


# --------------------------------------------------
# C. Utility-family × regime interaction evidence
# --------------------------------------------------

for _, row in family_interaction_df.iterrows():

    add_evidence(
        evidence_family=
            "utility_family_regime_interaction",

        scope=
            (
                f"{row['first_regime']}"
                f" minus "
                f"{row['second_regime']}"
            ),

        comparison=
            (
                "difference in coverage-minus-product "
                "effect between regimes"
            ),

        metric=
            row[
                "metric"
            ],

        estimate=
            row[
                "estimated_interaction"
            ],

        ci_lower=
            row[
                "simultaneous_ci_95_lower"
            ],

        ci_upper=
            row[
                "simultaneous_ci_95_upper"
            ],

        interval_type=
            (
                "studentized_max_t_"
                "simultaneous_within_metric"
            ),

        source_file=
            SOURCE_FILES[
                "family_regime_interactions"
            ].name,

        interpretation_guardrail=
            (
                "Synthetic regime interaction under "
                "matched burden marginals."
            ),
    )


# --------------------------------------------------
# D. Coverage structural-regime evidence
# --------------------------------------------------

for _, row in coverage_regime_df.iterrows():

    add_evidence(
        evidence_family=
            "coverage_regime_contrast",

        scope=
            row[
                "contrast"
            ],

        comparison=
            (
                f"{row['first_regime']}"
                f" minus "
                f"{row['second_regime']}"
            ),

        metric=
            row[
                "metric"
            ],

        estimate=
            row[
                "estimated_difference"
            ],

        ci_lower=
            row[
                "ci_95_lower"
            ],

        ci_upper=
            row[
                "ci_95_upper"
            ],

        interval_type=
            "unadjusted_paired_percentile_95",

        source_file=
            SOURCE_FILES[
                "coverage_regime_contrasts"
            ].name,

        interpretation_guardrail=
            (
                "Matched synthetic regimes with identical "
                "coverage structure and burden marginals."
            ),
    )


# --------------------------------------------------
# E. Coverage failure mechanisms
# --------------------------------------------------

for _, row in mechanism_df.iterrows():

    for metric_name in [
        "share_of_all_rows_percent",
        "mean_optimality_gap_percent",
        "within_mechanism_large_gap_percent",
        "mean_selected_count_difference",
    ]:

        add_evidence(
            evidence_family=
                "coverage_failure_mechanism",

            scope=
                row[
                    "failure_mechanism"
                ],

            comparison=
                "dynamic_greedy_relative_to_exact",

            metric=
                metric_name,

            estimate=
                row[
                    metric_name
                ],

            source_file=
                SOURCE_FILES[
                    "coverage_failure_mechanisms"
                ].name,

            interpretation_guardrail=
                (
                    "Mechanism classification is "
                    "descriptive and mutually exclusive."
                ),
        )


# --------------------------------------------------
# F. Candidate-size endpoint evidence
# --------------------------------------------------

for _, row in size_df[
    size_df[
        "n_items"
    ].isin(
        [
            8,
            16,
        ]
    )
].iterrows():

    add_evidence(
        evidence_family=
            "coverage_candidate_size",

        scope=
            f"M={int(row['n_items'])}",

        comparison=
            "dynamic_greedy_relative_to_exact",

        metric=
            row[
                "metric"
            ],

        estimate=
            row[
                "estimate"
            ],

        ci_lower=
            row[
                "ci_95_lower"
            ],

        ci_upper=
            row[
                "ci_95_upper"
            ],

        interval_type=
            "unadjusted_bootstrap_percentile_95",

        source_file=
            SOURCE_FILES[
                "coverage_size_summary"
            ].name,

        interpretation_guardrail=
            (
                "Candidate sizes use independently "
                "generated groups."
            ),
    )


# --------------------------------------------------
# G. Threshold endpoint evidence
# --------------------------------------------------

THRESHOLD_METRICS = [
    "mean_optimality_gap_percent",
    "mean_absolute_disclosure_cost_gap",
    "mean_exact_disclosure_cost",
    "exact_cost_recovery_percent",
    "large_gap_percent",
    "more_items_percent",
    "mean_selected_count_difference",
]


for _, row in threshold_df[
    threshold_df[
        "tau"
    ].isin(
        [
            0.70,
            0.95,
        ]
    )
].iterrows():

    for metric_name in THRESHOLD_METRICS:

        add_evidence(
            evidence_family=
                "coverage_threshold",

            scope=
                f"tau={row['tau']:.2f}",

            comparison=
                "dynamic_greedy_relative_to_exact",

            metric=
                metric_name,

            estimate=
                row[
                    metric_name
                ],

            source_file=
                SOURCE_FILES[
                    "coverage_threshold_summary"
                ].name,

            interpretation_guardrail=
                (
                    "Percentage gaps depend partly on "
                    "the Exact-cost denominator."
                ),
        )


# --------------------------------------------------
# H. Weight-profile paired contrasts
# --------------------------------------------------

for _, row in weight_contrast_df.iterrows():

    add_evidence(
        evidence_family=
            "coverage_weight_profile",

        scope=
            row[
                "contrast"
            ],

        comparison=
            (
                f"{row['first_profile']}"
                f" minus "
                f"{row['second_profile']}"
            ),

        metric=
            row[
                "metric"
            ],

        estimate=
            row[
                "estimated_difference"
            ],

        ci_lower=
            row[
                "ci_95_lower"
            ],

        ci_upper=
            row[
                "ci_95_upper"
            ],

        interval_type=
            "unadjusted_paired_percentile_95",

        source_file=
            SOURCE_FILES[
                "coverage_weight_contrasts"
            ].name,

        interpretation_guardrail=
            (
                "Weight profiles are benchmark design "
                "points, not measured human preferences."
            ),
    )


# --------------------------------------------------
# I. Coverage runtime endpoint evidence
# --------------------------------------------------

RUNTIME_METRICS = [
    "mean_shared_utility_precomputation_seconds",
    "mean_burden_precomputation_seconds",
    "mean_exact_solve_milliseconds",
    "mean_greedy_solve_milliseconds",
]


for _, row in runtime_df[
    runtime_df[
        "n_items"
    ].isin(
        [
            8,
            16,
        ]
    )
].iterrows():

    for metric_name in RUNTIME_METRICS:

        add_evidence(
            evidence_family=
                "coverage_runtime",

            scope=
                f"M={int(row['n_items'])}",

            comparison=
                "empirical_runtime",

            metric=
                metric_name,

            estimate=
                row[
                    metric_name
                ],

            source_file=
                SOURCE_FILES[
                    "coverage_runtime_summary"
                ].name,

            interpretation_guardrail=
                (
                    "Empirical runtime on the current "
                    "hardware; not an asymptotic proof."
                ),
        )


final_evidence_df = pd.DataFrame(
    evidence_rows
)


assert len(
    final_evidence_df
) > 100


assert final_evidence_df[
    "estimate"
].notna().all()


# --------------------------------------------------
# Final key-evidence snapshot
# --------------------------------------------------

snapshot_rows = []


def add_snapshot(
    finding,
    estimate,
    lower=np.nan,
    upper=np.nan,
    unit="",
):
    """
    Add one compact publication-planning result.
    """

    snapshot_rows.append(
        {
            "finding":
                finding,

            "estimate":
                float(
                    estimate
                ),

            "ci_95_lower":
                (
                    float(
                        lower
                    )
                    if pd.notna(
                        lower
                    )
                    else np.nan
                ),

            "ci_95_upper":
                (
                    float(
                        upper
                    )
                    if pd.notna(
                        upper
                    )
                    else np.nan
                ),

            "unit":
                unit,
        }
    )


# Overall simultaneous family effects
for metric_name in [
    "mean_optimality_gap_percent",
    "exact_cost_recovery_percent",
    "large_gap_percent",
    "more_items_percent",
]:

    row = family_scope_df[
        (
            family_scope_df[
                "scope"
            ]
            ==
            "overall"
        )
        &
        (
            family_scope_df[
                "metric"
            ]
            ==
            metric_name
        )
    ].iloc[0]


    add_snapshot(
        finding=
            (
                "Coverage minus product overall: "
                f"{metric_name}"
            ),

        estimate=
            row[
                "estimated_difference"
            ],

        lower=
            row[
                "simultaneous_ci_95_lower"
            ],

        upper=
            row[
                "simultaneous_ci_95_upper"
            ],

        unit=
            (
                "percentage points"
                if metric_name
                !=
                "mean_jaccard_similarity"
                else
                "proportion"
            ),
    )


# Alignment and conflict mean-gap effects
for scope in [
    "utility_burden_alignment",
    "utility_burden_conflict",
]:

    row = family_scope_df[
        (
            family_scope_df[
                "scope"
            ]
            ==
            scope
        )
        &
        (
            family_scope_df[
                "metric"
            ]
            ==
            "mean_optimality_gap_percent"
        )
    ].iloc[0]


    add_snapshot(
        finding=
            (
                "Coverage minus product mean gap: "
                f"{scope}"
            ),

        estimate=
            row[
                "estimated_difference"
            ],

        lower=
            row[
                "simultaneous_ci_95_lower"
            ],

        upper=
            row[
                "simultaneous_ci_95_upper"
            ],

        unit=
            "percentage points",
    )


# Large-gap mechanism attribution
for mechanism in [
    "same_count_different_items",
    "greedy_more_items",
]:

    row = large_gap_attribution_df[
        (
            large_gap_attribution_df[
                "scope"
            ]
            ==
            "overall"
        )
        &
        (
            large_gap_attribution_df[
                "failure_mechanism"
            ]
            ==
            mechanism
        )
    ].iloc[0]


    add_snapshot(
        finding=
            (
                "Share of coverage large gaps: "
                f"{mechanism}"
            ),

        estimate=
            row[
                "share_of_large_gap_rows_percent"
            ],

        unit=
            "percent",
    )


# Candidate-size endpoints
for n_items in [
    8,
    16,
]:

    for metric_name in [
        "mean_optimality_gap_percent",
        "exact_cost_recovery_percent",
        "more_items_percent",
    ]:

        row = size_df[
            (
                size_df[
                    "n_items"
                ]
                ==
                n_items
            )
            &
            (
                size_df[
                    "metric"
                ]
                ==
                metric_name
            )
        ].iloc[0]


        add_snapshot(
            finding=
                (
                    f"M={n_items}: "
                    f"{metric_name}"
                ),

            estimate=
                row[
                    "estimate"
                ],

            lower=
                row[
                    "ci_95_lower"
                ],

            upper=
                row[
                    "ci_95_upper"
                ],

            unit=
                (
                    "percentage points"
                    if metric_name
                    !=
                    "mean_jaccard_similarity"
                    else
                    "proportion"
                ),
        )


# Threshold endpoints
for tau in [
    0.70,
    0.95,
]:

    row = threshold_df[
        np.isclose(
            threshold_df[
                "tau"
            ],
            tau,
        )
    ].iloc[0]


    for metric_name, unit in [
        (
            "mean_optimality_gap_percent",
            "percent",
        ),
        (
            "mean_absolute_disclosure_cost_gap",
            "modeled disclosure-cost units",
        ),
        (
            "more_items_percent",
            "percent",
        ),
    ]:

        add_snapshot(
            finding=
                (
                    f"tau={tau:.2f}: "
                    f"{metric_name}"
                ),

            estimate=
                row[
                    metric_name
                ],

            unit=
                unit,
        )


# Parsimony versus balanced
for metric_name in [
    "mean_optimality_gap_percent",
    "large_gap_percent",
    "more_items_percent",
]:

    row = weight_contrast_df[
        (
            weight_contrast_df[
                "first_profile"
            ]
            ==
            "parsimony_priority"
        )
        &
        (
            weight_contrast_df[
                "second_profile"
            ]
            ==
            "balanced"
        )
        &
        (
            weight_contrast_df[
                "metric"
            ]
            ==
            metric_name
        )
    ].iloc[0]


    add_snapshot(
        finding=
            (
                "Parsimony minus balanced: "
                f"{metric_name}"
            ),

        estimate=
            row[
                "estimated_difference"
            ],

        lower=
            row[
                "ci_95_lower"
            ],

        upper=
            row[
                "ci_95_upper"
            ],

        unit=
            "percentage points",
    )


final_snapshot_df = pd.DataFrame(
    snapshot_rows
)


# --------------------------------------------------
# SHA-256 reproducibility manifest
# --------------------------------------------------

def calculate_sha256(
    file_path,
    chunk_size=1024 * 1024,
):
    """
    Calculate SHA-256 without loading the entire file
    into memory.
    """

    digest = hashlib.sha256()


    with file_path.open(
        "rb"
    ) as file_handle:

        while True:

            chunk = file_handle.read(
                chunk_size
            )


            if not chunk:

                break


            digest.update(
                chunk
            )


    return digest.hexdigest()


manifest_rows = []


for source_name, source_path in (
    SOURCE_FILES.items()
):

    file_stat = source_path.stat()


    manifest_rows.append(
        {
            "source_name":
                source_name,

            "relative_path":
                str(
                    source_path.relative_to(
                        PROJECT_ROOT
                    )
                ),

            "size_bytes":
                file_stat.st_size,

            "modified_utc":
                datetime.fromtimestamp(
                    file_stat.st_mtime,
                    tz=timezone.utc,
                ).isoformat(),

            "sha256":
                calculate_sha256(
                    source_path
                ),
        }
    )


manifest_df = pd.DataFrame(
    manifest_rows
)


assert len(
    manifest_df
) == len(
    SOURCE_FILES
)


assert manifest_df[
    "sha256"
].str.len().eq(
    64
).all()


# --------------------------------------------------
# Final closure audit
# --------------------------------------------------

final_audit_df = pd.DataFrame(
    [
        {
            "computational_phase_status":
                "COMPLETE",

            "required_source_files":
                len(
                    SOURCE_FILES
                ),

            "verified_source_files":
                len(
                    manifest_df
                ),

            "product_raw_rows":
                len(
                    product_df
                ),

            "coverage_raw_rows":
                len(
                    coverage_df
                ),

            "product_comparable_rows":
                product_comparable_rows,

            "coverage_comparable_rows":
                coverage_comparable_rows,

            "product_infeasible_rows":
                product_infeasible_rows,

            "coverage_infeasible_rows":
                coverage_infeasible_rows,

            "product_common_threshold_rows":
                product_common_rows,

            "coverage_common_threshold_rows":
                coverage_common_rows,

            "family_descriptive_rows":
                len(
                    family_descriptive_df
                ),

            "family_scope_simultaneous_rows":
                len(
                    family_scope_df
                ),

            "family_interaction_rows":
                len(
                    family_interaction_df
                ),

            "coverage_regime_contrast_rows":
                len(
                    coverage_regime_df
                ),

            "failure_mechanism_categories":
                len(
                    mechanism_df
                ),

            "candidate_size_metric_rows":
                len(
                    size_df
                ),

            "threshold_summary_rows":
                len(
                    threshold_df
                ),

            "weight_contrast_rows":
                len(
                    weight_contrast_df
                ),

            "runtime_size_rows":
                len(
                    runtime_df
                ),

            "final_evidence_ledger_rows":
                len(
                    final_evidence_df
                ),

            "final_snapshot_rows":
                len(
                    final_snapshot_df
                ),

            "negative_coverage_gap_rows":
                int(
                    (
                        coverage_df[
                            "optimality_gap_percent"
                        ]
                        <
                        -1e-8
                    ).sum()
                ),

            "missing_final_evidence_estimates":
                int(
                    final_evidence_df[
                        "estimate"
                    ].isna().sum()
                ),

            "manifest_hash_failures":
                int(
                    (
                        manifest_df[
                            "sha256"
                        ].str.len()
                        !=
                        64
                    ).sum()
                ),

            "cross_family_threshold_grid":
                "[0.70, 0.80, 0.90]",

            "cross_family_interval_method":
                (
                    "studentized max-t simultaneous "
                    "within each metric"
                ),

            "global_adjustment_across_metrics":
                False,

            "approximation_ratio_claimed":
                False,
        }
    ]
)


assert final_audit_df.loc[
    0,
    "negative_coverage_gap_rows",
] == 0


assert final_audit_df.loc[
    0,
    "missing_final_evidence_estimates",
] == 0


assert final_audit_df.loc[
    0,
    "manifest_hash_failures",
] == 0


# --------------------------------------------------
# Methodological freeze note
# --------------------------------------------------

freeze_note = f"""
MST COMPUTATIONAL ANALYSIS FREEZE — VERSION 1
Generated: {datetime.now(timezone.utc).isoformat()}

STATUS
The planned computational analysis is complete.

UTILITY SEMANTICS
1. Product-form utility and pool-normalized weighted maximum-coverage
   utility are treated as distinct modeled decision-support utility
   families.
2. Pool-normalized thresholds represent the fraction of modeled
   decision-support utility available in the current candidate pool.
3. Cross-family comparisons use only the common fully feasible
   threshold grid: tau = 0.70, 0.80, and 0.90.
4. The 216 product-form infeasible rows at tau = 0.95 remain valid
   benchmark results but are excluded from cross-family inference.

INTERPRETATION LIMITS
1. Utility is modeled decision-support utility, not measured human
   sufficiency.
2. Cognitive cost is modeled interpretation cost, not directly
   observed human cognitive burden.
3. Privacy is relative modeled privacy exposure, not a deployment
   privacy guarantee.
4. Weight profiles are benchmark design points, not measured human
   preferences.
5. Synthetic regime effects are claims about the benchmark design.
6. No approximation ratio is claimed.
7. Runtime findings are empirical on the current hardware and are not
   asymptotic proofs.

INFERENCE
1. Matched-group resampling is used where benchmark conditions are
   matched.
2. Utility-family benchmarks are independently generated and are
   resampled independently between families.
3. Step-135 max-t intervals control the relevant scopes or
   interactions separately within each metric.
4. There is no global multiplicity correction across different
   metrics.

REPRODUCIBILITY
Number of frozen source files: {len(manifest_df)}
Evidence-ledger rows: {len(final_evidence_df)}
Manifest file: {FINAL_MANIFEST_PATH.name}
""".strip()


# --------------------------------------------------
# Save final frozen outputs
# --------------------------------------------------

final_evidence_df.to_csv(
    FINAL_EVIDENCE_PATH,
    index=False,
)


final_snapshot_df.to_csv(
    FINAL_SNAPSHOT_PATH,
    index=False,
)


final_audit_df.to_csv(
    FINAL_AUDIT_PATH,
    index=False,
)


manifest_df.to_csv(
    FINAL_MANIFEST_PATH,
    index=False,
)


FINAL_FREEZE_NOTE_PATH.write_text(
    freeze_note,
    encoding="utf-8",
)


# --------------------------------------------------
# Printed report
# --------------------------------------------------

print(
    "======================================"
)

print(
    "FINAL COMPUTATIONAL CLOSURE AUDIT"
)

print(
    "======================================"
)


print(
    final_audit_df.to_string(
        index=False
    )
)


print(
    "\n======================================"
)

print(
    "FINAL KEY EVIDENCE SNAPSHOT"
)

print(
    "======================================"
)


print(
    final_snapshot_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.4f}",
    )
)


manifest_summary_df = pd.DataFrame(
    [
        {
            "frozen_source_files":
                len(
                    manifest_df
                ),

            "total_size_megabytes":
                manifest_df[
                    "size_bytes"
                ].sum()
                /
                (
                    1024
                    **
                    2
                ),

            "unique_sha256_hashes":
                manifest_df[
                    "sha256"
                ].nunique(),

            "duplicate_hashes":
                len(
                    manifest_df
                )
                -
                manifest_df[
                    "sha256"
                ].nunique(),

            "freeze_note_created":
                FINAL_FREEZE_NOTE_PATH.exists(),

            "all_outputs_created":
                all(
                    output_path.exists()
                    for output_path in OUTPUT_FILES
                ),
        }
    ]
)


print(
    "\n======================================"
)

print(
    "FINAL REPRODUCIBILITY MANIFEST SUMMARY"
)

print(
    "======================================"
)


print(
    manifest_summary_df.to_string(
        index=False,
        float_format=lambda value:
            f"{value:.6f}",
    )
)


print(
    "\nFinal evidence ledger saved to:"
)

print(
    FINAL_EVIDENCE_PATH
)


print(
    "\nFinal key snapshot saved to:"
)

print(
    FINAL_SNAPSHOT_PATH
)


print(
    "\nFinal audit saved to:"
)

print(
    FINAL_AUDIT_PATH
)


print(
    "\nSHA-256 manifest saved to:"
)

print(
    FINAL_MANIFEST_PATH
)


print(
    "\nMethodological freeze note saved to:"
)

print(
    FINAL_FREEZE_NOTE_PATH
)